# Correr cada inicio de sesión

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/isac2026')
for carpeta in ['datos_crudos', 'datos_parquet', 'salidas', 'codigo']:
    (BASE / carpeta).mkdir(parents=True, exist_ok=True)

print(sorted(p.name for p in BASE.iterdir()))

In [ ]:
from google.colab import userdata

USUARIO = userdata.get('SB_USER')
CLAVE = userdata.get('SB_PASS')
print('Usuario cargado:', bool(USUARIO), '| Clave cargada:', bool(CLAVE))

# Data download

Import competitions

In [ ]:
import requests
import pandas as pd

URL_COMPETITIONS = "https://data.statsbomb.com/api/v4/competitions"

r = requests.get(URL_COMPETITIONS, auth=(USUARIO, CLAVE), timeout=60)
print("Código HTTP:", r.status_code)

if r.status_code == 200:
    comp = pd.DataFrame(r.json())
    print("Filas totales:", len(comp))
    liga_mx = comp[comp['competition_id'] == 73]
    print(liga_mx[['season_id', 'season_name', 'match_available', 'match_available_360']]
          .sort_values('season_name').to_string(index=False))
    comp.to_parquet(BASE / 'datos_parquet' / 'competitions.parquet')
    print("Guardado en Drive.")
else:
    print(r.text[:300])

Import match

In [ ]:
import json, time, requests, pandas as pd
from datetime import date

URL_MATCHES = "https://data.statsbomb.com/api/v6/competitions/{competition_id}/seasons/{season_id}/matches"

SEASONS = {108: '2021/2022', 235: '2022/2023', 281: '2023/2024',
           317: '2024/2025', 318: '2025/2026', 351: '2026/2027'}
hoy = date.today().isoformat()
frames = []

for sid, nombre in SEASONS.items():
    url = URL_MATCHES.format(competition_id=73, season_id=sid)
    r = requests.get(url, auth=(USUARIO, CLAVE), timeout=120)
    print(nombre, '-> HTTP', r.status_code)
    if r.status_code != 200:
        print(r.text[:200])
        continue
    data = r.json()
    with open(BASE / 'datos_crudos' / f'matches_73_{sid}_{hoy}.json', 'w') as f:
        json.dump(data, f)
    df = pd.json_normalize(data)
    df['season_id'] = sid
    df['season_label'] = nombre
    frames.append(df)
    time.sleep(1)

matches = pd.concat(frames, ignore_index=True)

try:
    matches.to_parquet(BASE / 'datos_parquet' / f'matches_73_{hoy}.parquet')
    print('Guardado en parquet.')
except Exception as e:
    matches.to_pickle(BASE / 'datos_parquet' / f'matches_73_{hoy}.pkl')
    print('Parquet falló (', type(e).__name__, '), guardado como pickle.')

print('\nTotal de partidos:', len(matches))
print('match_id duplicados:', matches['match_id'].duplicated().sum()
      if 'match_id' in matches.columns else 'sin columna match_id')
print('\nPartidos por temporada:')
print(matches.groupby('season_label').size().to_string())
print('\nColumnas:')
print(list(matches.columns))
cols_equipo = [c for c in matches.columns if 'team_name' in c]
if cols_equipo:
    for c in cols_equipo:
        print(f'\nEquipos distintos en {c}:', matches[c].nunique())

Diagnostics 1

In [ ]:
import pandas as pd, glob
archivo = sorted(glob.glob(str(BASE / 'datos_parquet' / 'matches_73_*.parquet')))[-1]
matches = pd.read_parquet(archivo)

print("1) Estado de los partidos por temporada")
print(matches.groupby(['season_label', 'match_status']).size().to_string())

print("\n2) Estado 360")
print(matches.groupby(['season_label', 'match_status_360']).size().to_string())

print("\n3) Marcadores nulos por temporada")
print(matches.groupby('season_label')['home_score'].apply(lambda s: s.isna().sum()).to_string())

print("\n4) Rango de fechas por temporada")
print(matches.groupby('season_label')['match_date'].agg(['min', 'max']).to_string())

print("\n5) Etapas de la competición")
print(matches['competition_stage.name'].value_counts().to_string())

print("\n6) Equipos distintos por temporada")
print(matches.groupby('season_label')['home_team.home_team_name'].nunique().to_string())

print("\n7) Todos los equipos")
print(sorted(matches['home_team.home_team_name'].unique()))

print("\n8) Ejemplo de entrenadores (un solo partido)")
print(matches['home_team.managers'].iloc[0])

Diagnostics 2

In [ ]:
import pandas as pd, glob
archivo = sorted(glob.glob(str(BASE / 'datos_parquet' / 'matches_73_*.parquet')))[-1]
matches = pd.read_parquet(archivo)

def n_tec(m):
    try: return len(m)
    except Exception: return 0

def campo(m, k):
    try: return m[0][k] if len(m) > 0 else None
    except Exception: return None

def lado(df, s):
    o = pd.DataFrame({
        'match_id': df['match_id'], 'fecha': df['match_date'],
        'temporada': df['season_label'], 'etapa': df['competition_stage.name'],
        'jugado': df['home_score'].notna(),
        'team': df[f'{s}_team.{s}_team_name'],
        'n_tec': df[f'{s}_team.managers'].apply(n_tec),
        'tec_id': df[f'{s}_team.managers'].apply(lambda m: campo(m, 'id')),
        'tec': df[f'{s}_team.managers'].apply(lambda m: campo(m, 'name')),
    })
    o['lado'] = s
    return o

tm = pd.concat([lado(matches, 'home'), lado(matches, 'away')], ignore_index=True)
tm.to_parquet(BASE / 'datos_parquet' / 'equipo_partido_tecnico.parquet')

print("1) Partidos jugados por temporada")
print(matches.assign(jugado=matches['home_score'].notna()).groupby('season_label')['jugado'].sum().to_string())

print("\n2) Partidos-equipo según número de técnicos registrados")
print(tm['n_tec'].value_counts().to_string())

j = tm[tm['jugado'] & (tm['n_tec'] > 0)]
g = (j.groupby(['team', 'tec']).agg(partidos=('match_id', 'count'),
     desde=('fecha', 'min'), hasta=('fecha', 'max')).reset_index())
multi = g[g.groupby('team')['tec'].transform('nunique') > 1]

print("\n3) Equipos con más de un técnico (jugados):", multi['team'].nunique(), "de 18")
print(multi.sort_values(['team', 'desde']).to_string(index=False))

t = j.groupby('tec')['team'].nunique()
varios = t[t > 1].index
print(g[g['tec'].isin(varios)].sort_values(['tec', 'desde']).to_string(index=False))

Numero de tecnicos posibles

In [ ]:
import pandas as pd, glob

UMBRAL_SUSP = 5   # racha corta entre dos rachas del mismo técnico -> suspensión
UMBRAL_INT = 3    # racha corta entre dos técnicos distintos -> interino

tm = pd.read_parquet(BASE / 'datos_parquet' / 'equipo_partido_tecnico.parquet')
j = tm[tm['jugado'] & (tm['n_tec'] > 0)].copy()

def procesar(g):
    g = g.sort_values(['fecha', 'match_id']).reset_index(drop=True)
    g['tec_id_p'] = g['tec_id']
    g['tec_p'] = g['tec']
    g['rol'] = 'titular'
    while True:
        g['etapa'] = (g['tec_id_p'] != g['tec_id_p'].shift()).cumsum()
        r = g.groupby('etapa').agg(n=('match_id', 'count'), tid=('tec_id_p', 'first'))
        ids = r.index.tolist()
        movido = False
        for k in range(1, len(ids) - 1):
            a, b, c = r.iloc[k - 1], r.iloc[k], r.iloc[k + 1]
            if a['tid'] == c['tid'] and b['tid'] != a['tid'] and b['n'] <= UMBRAL_SUSP:
                m = g['etapa'] == ids[k]
                nombre = g.loc[g['etapa'] == ids[k - 1], 'tec_p'].iloc[0]
                g.loc[m, 'rol'] = 'sustituto_suspension'
                g.loc[m, 'tec_id_p'] = a['tid']
                g.loc[m, 'tec_p'] = nombre
                movido = True
                break
        if not movido:
            break
    tam = g.groupby('etapa')['match_id'].transform('count')
    ultima = g['etapa'].max()
    g.loc[(tam <= UMBRAL_INT) & (g['etapa'] < ultima) & (g['rol'] == 'titular'), 'rol'] = 'interino'
    return g

jj = pd.concat([procesar(g) for _, g in j.groupby('team')], ignore_index=True)
jj.to_parquet(BASE / 'datos_parquet' / 'partido_tecnico_periodo.parquet')

ep = (jj.groupby(['team', 'etapa'])
        .agg(tec=('tec_p', 'first'),
             partidos=('match_id', 'count'),
             de_sustituto=('rol', lambda s: (s == 'sustituto_suspension').sum()),
             desde=('fecha', 'min'), hasta=('fecha', 'max'),
             rol=('rol', lambda s: 'interino' if (s == 'interino').any() else 'titular'))
        .reset_index())
ep.to_parquet(BASE / 'datos_parquet' / 'etapas_tecnico.parquet')

print("1) Resumen")
print("Etapas:", len(ep), "| interinos:", (ep['rol'] == 'interino').sum(),
      "| partidos de sustitutos por suspensión:", (jj['rol'] == 'sustituto_suspension').sum())

archivo = sorted(glob.glob(str(BASE / 'datos_parquet' / 'matches_73_*.parquet')))[-1]
matches = pd.read_parquet(archivo)

print("\n2) Partidos con dos técnicos registrados")
for s in ['home', 'away']:
    col = f'{s}_team.managers'
    dos = matches[matches[col].apply(lambda m: hasattr(m, '__len__') and len(m) == 2)]
    for _, f in dos.iterrows():
        print(f['match_date'], f[f'{s}_team.{s}_team_name'], [x['name'] for x in f[col]])

print("\n3) América, etapas")
print(ep[ep['team'] == 'América'].drop(columns='etapa').to_string(index=False))

print("\n4) Técnicos con más de una etapa titular en el mismo club")
tit = ep[ep['rol'] == 'titular']
reg = tit.groupby(['team', 'tec']).filter(lambda x: len(x) > 1)
print(reg.sort_values(['team', 'tec', 'desde'])
         [['team', 'tec', 'partidos', 'desde', 'hasta']].to_string(index=False))

print("\n5) Interinos")
print(ep[ep['rol'] == 'interino'][['team', 'tec', 'partidos', 'desde']].to_string(index=False))

print("\n6) team_id de Atlante y Mazatlán")
print(matches[matches['home_team.home_team_name'].isin(['Atlante', 'Mazatlán'])]
      .groupby('home_team.home_team_name')['home_team.home_team_id'].unique().to_string())

Import lineups events y 360

In [ ]:
import json, time, shutil, requests, pandas as pd
from collections import Counter
from datetime import date

URL_LINEUPS = "https://data.statsbomb.com/api/v5/lineups/{match_id}"
URL_EVENTOS = "https://data.statsbomb.com/api/v11/events/{match_id}"
URL_360     = "https://data.statsbomb.com/api/v2/360-frames/{match_id}"


hoy = date.today().isoformat()
archivo = sorted((BASE / 'datos_parquet').glob('matches_73_*.parquet'))[-1]
matches = pd.read_parquet(archivo)

jugados = matches[matches['home_score'].notna()]
print("Valores de match_status_360 en partidos jugados:")
print(jugados['match_status_360'].value_counts().to_string())

normal_360 = jugados['match_status_360'].value_counts().index[0]
candidatos = jugados[jugados['match_status_360'] == normal_360]
muestra = pd.concat([
    candidatos[candidatos['season_label'] == '2021/2022'].head(1),
    candidatos[candidatos['season_label'] == '2025/2026'].head(1),
])
print("\nPartidos de prueba:", muestra['match_id'].tolist())

def pedir(nombre, url_tpl, mid):
    t0 = time.time()
    r = requests.get(url_tpl.format(match_id=mid), auth=(USUARIO, CLAVE), timeout=180)
    seg = time.time() - t0
    print(f"\n[{nombre}] match {mid} -> HTTP {r.status_code} | {len(r.content)/1e6:.2f} MB | {seg:.1f} s")
    if r.status_code != 200:
        print(r.text[:200]); return None, 0
    data = r.json()
    with open(BASE / 'datos_crudos' / f'{nombre}_{mid}_{hoy}.json', 'w') as f:
        json.dump(data, f)
    return data, len(r.content)

pesos = {'lineups': [], 'eventos': [], 'frames360': []}

for mid in muestra['match_id']:
    d, b = pedir('lineups', URL_LINEUPS, mid)
    if d is not None:
        pesos['lineups'].append(b)
        print("  tipo:", type(d).__name__, "| elementos:", len(d))
        if isinstance(d, list) and d:
            print("  campos del primer elemento:", list(d[0].keys()))
            for eq in d:
                if 'lineup' in eq:
                    print("  jugadores en", eq.get('team_name'), ":", len(eq['lineup']))

    d, b = pedir('eventos', URL_EVENTOS, mid)
    if d is not None:
        pesos['eventos'].append(b)
        print("  tipo:", type(d).__name__, "| elementos:", len(d))
        if isinstance(d, list) and d:
            print("  campos del primer elemento:", list(d[0].keys()))
            tipos = Counter((e.get('type') or {}).get('name') for e in d)
            print("  tipos de evento (top 12):", tipos.most_common(12))

    d, b = pedir('frames360', URL_360, mid)
    if d is not None:
        pesos['frames360'].append(b)
        print("  tipo:", type(d).__name__, "| elementos:", len(d))
        if isinstance(d, list) and d:
            print("  campos del primer elemento:", list(d[0].keys()))
            print("  frames con jugadores:", sum(1 for x in d if x.get('freeze_frame')))
    time.sleep(1)

n_total = len(jugados)
print("\nESTIMACIÓN DE PESO (JSON crudo)")
tot = 0
for k, v in pesos.items():
    if v:
        prom = sum(v) / len(v)
        est = prom * n_total / 1e9
        tot += est
        print(f"{k}: {prom/1e6:.2f} MB por partido -> ~{est:.1f} GB para {n_total} partidos")
print(f"TOTAL estimado: ~{tot:.1f} GB")

try:
    u = shutil.disk_usage('/content/drive')
    print(f"\nDrive: libre ~{u.free/1e9:.0f} GB de {u.total/1e9:.0f} GB")
except Exception as e:
    print("\nNo pude leer el espacio libre de Drive:", type(e).__name__)

Download 1

In [ ]:
import json, gzip, shutil, pandas as pd
from collections import Counter
from pathlib import Path

CRUDOS = BASE / 'datos_crudos'
TMP = Path('/content/prueba'); TMP.mkdir(exist_ok=True)
mids = [3799476, 4004046]

def ruta(nombre, mid):
    return sorted(CRUDOS.glob(f'{nombre}_{mid}_*.json'))[-1]

def mb(p):
    return Path(p).stat().st_size / 1e6

res = {'gz_ev': [], 'gz_360': [], 'pq_ev': [], 'pq_360': []}

for mid in mids:
    print(f"\n=== Partido {mid} ===")
    ev_raw = json.load(open(ruta('eventos', mid)))
    f_raw = json.load(open(ruta('frames360', mid)))
    lu_raw = json.load(open(ruta('lineups', mid)))

    # 1) Compresión sin pérdida del JSON original
    for nombre, clave in [('eventos', 'gz_ev'), ('frames360', 'gz_360')]:
        orig = ruta(nombre, mid)
        destino = TMP / (orig.name + '.gz')
        with open(orig, 'rb') as a, gzip.open(destino, 'wb') as b:
            shutil.copyfileobj(a, b)
        res[clave].append(mb(destino))
        print(f"{nombre}: {mb(orig):.2f} MB -> {mb(destino):.2f} MB comprimido")

    # 2) Eventos aplanados
    ev = pd.json_normalize(ev_raw)
    print("Eventos: filas", len(ev), "| columnas", ev.shape[1])
    print("Columnas por grupo:", dict(Counter(c.split('.')[0] for c in ev.columns)))
    print("Columnas con xG:", [c for c in ev.columns if 'xg' in c.lower()])
    try:
        p = TMP / f'ev_{mid}.parquet'
        ev.to_parquet(p)
        res['pq_ev'].append(mb(p))
        print(f"Eventos en parquet: {mb(p):.2f} MB")
    except Exception as e:
        print("Parquet de eventos falló:", type(e).__name__, str(e)[:150])

    # 3) Enlace entre 360 y eventos
    ids_ev = set(ev['id'])
    ids_360 = {x['event_uuid'] for x in f_raw}
    print(f"Frames 360 enlazados a un evento: {len(ids_360 & ids_ev)} de {len(ids_360)}")

    # 4) 360: una tabla por frame y otra por jugador-frame
    filas = []
    for fr in f_raw:
        for pl in fr.get('freeze_frame') or []:
            loc = pl.get('location') or [None, None]
            filas.append((fr['event_uuid'], pl.get('teammate'), pl.get('actor'),
                          pl.get('keeper'), loc[0], loc[1]))
    jug = pd.DataFrame(filas, columns=['event_uuid', 'teammate', 'actor', 'keeper', 'x', 'y'])
    primero = next((fr['freeze_frame'][0] for fr in f_raw if fr.get('freeze_frame')), {})
    print("360 campos de un jugador:", list(primero.keys()))
    print("360 filas jugador-frame:", len(jug), "| x nulos:", int(jug['x'].isna().sum()),
          "| rango x:", jug['x'].min(), jug['x'].max(), "| rango y:", jug['y'].min(), jug['y'].max())
    fr_df = pd.DataFrame([{k: v for k, v in fr.items() if k != 'freeze_frame'} for fr in f_raw])
    try:
        p1, p2 = TMP / f'jug_{mid}.parquet', TMP / f'fr_{mid}.parquet'
        jug.to_parquet(p1); fr_df.to_parquet(p2)
        res['pq_360'].append(mb(p1) + mb(p2))
        print(f"360 en parquet: {mb(p1) + mb(p2):.2f} MB")
    except Exception as e:
        print("Parquet de 360 falló:", type(e).__name__, str(e)[:150])

    # 5) Alineaciones
    lu = lu_raw[0]['lineup']
    print("Lineup: campos de un jugador:", list(lu[0].keys()))
    print("Jugadores con campo 'positions' no vacío:", sum(1 for j in lu if j.get('positions')), "de", len(lu))

def prom(l):
    return sum(l) / len(l) if l else float('nan')

n = 1789
gz = prom(res['gz_ev']) + prom(res['gz_360'])
pq = prom(res['pq_ev']) + prom(res['pq_360'])
print("\nRESUMEN (promedio por partido, 2 partidos)")
print(f"JSON comprimido: eventos {prom(res['gz_ev']):.2f} MB + 360 {prom(res['gz_360']):.2f} MB")
print(f"Parquet: eventos {prom(res['pq_ev']):.2f} MB + 360 {prom(res['pq_360']):.2f} MB")
print(f"Extrapolado a {n} partidos: JSON comprimido ~{gz*n/1e3:.1f} GB | parquet ~{pq*n/1e3:.1f} GB")

Download Lineups Events y 360 (FR)

In [ ]:
import gzip, json, os, time, requests, pandas as pd
from concurrent.futures import ThreadPoolExecutor
from datetime import date

URL_LINEUPS = "https://data.statsbomb.com/api/v5/lineups/{match_id}"
URL_EVENTOS = "https://data.statsbomb.com/api/v11/events/{match_id}"
URL_360     = "https://data.statsbomb.com/api/v2/360-frames/{match_id}"

TEMPORADAS = [108, 235, 281, 317]   # etapa 1
TRABAJADORES = 4
TIEMPO_MAX_MIN = 50
TIPOS = {'lineups': URL_LINEUPS, 'eventos': URL_EVENTOS, 'frames360': URL_360}

DEST = BASE / 'datos_crudos' / 'liga'
for t in TIPOS: (DEST / t).mkdir(parents=True, exist_ok=True)

archivo = sorted((BASE / 'datos_parquet').glob('matches_73_*.parquet'))[-1]
matches = pd.read_parquet(archivo)
obj = matches[matches['home_score'].notna() & matches['season_id'].isin(TEMPORADAS)]
obj = obj.sort_values(['season_id', 'match_date'])

existentes = {t: set(os.listdir(DEST / t)) for t in TIPOS}
pendientes = []
for _, f in obj.iterrows():
    mid = f['match_id']
    for t in TIPOS:
        if t == 'frames360' and f['match_status_360'] != 'available':
            continue
        if f'{mid}.json.gz' not in existentes[t]:
            pendientes.append((mid, t))

def descargar(mid, tipo):
    destino = DEST / tipo / f'{mid}.json.gz'
    url = TIPOS[tipo].format(match_id=mid)
    for intento in range(1, 5):
        try:
            r = requests.get(url, auth=(USUARIO, CLAVE), timeout=180)
            if r.status_code == 200:
                r.json()  # valida que sea JSON
                tmp = destino.with_suffix('.tmp')
                with open(tmp, 'wb') as f:
                    f.write(gzip.compress(r.content))
                tmp.replace(destino)
                return (mid, tipo, 'ok')
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(10 * intento)
                continue
            return (mid, tipo, f'HTTP {r.status_code}')
        except Exception as e:
            time.sleep(5 * intento)
    return (mid, tipo, 'fallo_reintentos')

print(f"Partidos objetivo: {len(obj)} | descargas pendientes: {len(pendientes)}")
limite = time.time() + TIEMPO_MAX_MIN * 60
resultados = []
LOTE = 200
for i in range(0, len(pendientes), LOTE):
    if time.time() > limite:
        print("Tiempo máximo alcanzado. Vuelve a correr la celda para continuar.")
        break
    bloque = pendientes[i:i + LOTE]
    with ThreadPoolExecutor(TRABAJADORES) as ex:
        resultados.extend(ex.map(lambda t: descargar(*t), bloque))
    print(f"Avance: {min(i + LOTE, len(pendientes))} de {len(pendientes)} descargas")

res = pd.DataFrame(resultados, columns=['match_id', 'tipo', 'estado'])
if len(res):
    print("\nResultado de esta corrida:")
    print(res['estado'].value_counts().to_string())
    fallos = res[res['estado'] != 'ok']
    if len(fallos):
        fallos.to_csv(BASE / 'salidas' / f'fallos_descarga_{date.today()}.csv', index=False)
        print("Primeros fallos:\n", fallos.head(15).to_string(index=False))

print("\nArchivos en Drive vs esperados:")
esperado = {'lineups': len(obj), 'eventos': len(obj),
            'frames360': int((obj['match_status_360'] == 'available').sum())}
for t in TIPOS:
    n = len([x for x in os.listdir(DEST / t) if x.endswith('.json.gz')])
    gb = sum((DEST / t / x).stat().st_size for x in os.listdir(DEST / t)) / 1e9
    print(f"{t}: {n} de {esperado[t]} | {gb:.2f} GB")

Convertor

In [ ]:
import gzip, json, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

TEMPORADAS = [108, 235, 281, 317]
MAX_PARTIDOS = None
TIEMPO_MAX_MIN = 45

RAW = BASE / 'datos_crudos' / 'liga'
OUT = BASE / 'datos_parquet' / 'liga'
for c in ['eventos', 'frames360', 'jugadores360', 'lineups']:
    (OUT / c).mkdir(parents=True, exist_ok=True)
LOG = BASE / 'salidas' / 'validacion_etapa1.jsonl'

archivo = sorted((BASE / 'datos_parquet').glob('matches_73_*.parquet'))[-1]
matches = pd.read_parquet(archivo)
obj = matches[matches['home_score'].notna() & matches['season_id'].isin(TEMPORADAS)]
obj = obj.sort_values(['season_id', 'match_date'])
partidos = obj.to_dict('records')


def leer(tipo, mid):
    with gzip.open(RAW / tipo / f'{mid}.json.gz', 'rt', encoding='utf-8') as f:
        return json.load(f)


def a_texto(v):
    if v is None or (isinstance(v, float) and v != v):
        return None
    return json.dumps(v, ensure_ascii=False)


def anidado_a_texto(df):
    for c in df.columns:
        if pd.api.types.is_object_dtype(df[c]):
            if df[c].apply(lambda v: isinstance(v, (list, dict))).any():
                df[c] = df[c].apply(a_texto)
    return df


def separar_coordenadas(df, col, pref):
    if col not in df.columns:
        return
    for i, eje in enumerate(['x', 'y', 'z']):
        vals = df[col].apply(lambda a: a[i] if isinstance(a, list) and len(a) > i else np.nan)
        if eje == 'z' and vals.isna().all():
            continue
        df[f'{pref}_{eje}'] = vals.astype('float32')
    df.drop(columns=[col], inplace=True)


def aplanar_eventos(raw, mid):
    df = pd.json_normalize(raw)
    df.insert(0, 'match_id', mid)
    for col, pref in [('location', 'loc'), ('pass.end_location', 'pass_end'),
                      ('carry.end_location', 'carry_end'), ('shot.end_location', 'shot_end')]:
        separar_coordenadas(df, col, pref)
    return anidado_a_texto(df)


def guardar(df, ruta):
    try:
        df.to_parquet(ruta, index=False, compression='zstd')
        return 'ok'
    except Exception:
        d = df.copy()
        for c in d.columns:
            if pd.api.types.is_object_dtype(d[c]):
                d[c] = d[c].apply(lambda v: None if v is None or (isinstance(v, float) and v != v) else str(v))
        d.to_parquet(ruta, index=False, compression='zstd')
        return 'respaldo_texto'


def convertir_360(raw, mapa, mid):
    fr, jg = [], []
    for f in raw:
        uuid = f.get('event_uuid')
        idx = mapa.get(uuid, -1)
        ff = f.get('freeze_frame') or []
        va = f.get('visible_area') or []
        mates = sum(1 for p in ff if p.get('teammate'))
        fr.append({
            'match_id': mid, 'event_index': idx, 'event_uuid': uuid,
            'line_breaking_pass': f.get('line_breaking_pass'),
            'num_defenders_on_goal_side_of_actor': f.get('num_defenders_on_goal_side_of_actor'),
            'distance_to_nearest_defender': f.get('distance_to_nearest_defender'),
            'ball_receipt_in_space': f.get('ball_receipt_in_space'),
            'ball_receipt_exceeds_distance': f.get('ball_receipt_exceeds_distance'),
            'n_visibles': len(ff), 'n_visibles_companeros': mates,
            'n_visibles_rivales': len(ff) - mates,
            'tiene_actor': any(p.get('actor') for p in ff),
            'area_visible_vacia': len(va) == 0,
            'visible_area': a_texto(va) if va else None,
            'visible_player_counts': a_texto(f.get('visible_player_counts')),
        })
        for p in ff:
            loc = p.get('location') or [np.nan, np.nan]
            jg.append((mid, idx, bool(p.get('teammate')), bool(p.get('actor')),
                       bool(p.get('keeper')), loc[0], loc[1]))
    frames = pd.DataFrame(fr)
    jug = pd.DataFrame(jg, columns=['match_id', 'event_index', 'teammate', 'actor', 'keeper', 'x', 'y'])
    jug = jug.astype({'match_id': 'int32', 'event_index': 'int32', 'x': 'float32', 'y': 'float32'})
    return frames, jug


def convertir_lineups(raw, mid):
    filas = []
    for eq in raw:
        for p in eq.get('lineup') or []:
            r = {'match_id': mid, 'team_id': eq.get('team_id'), 'team_name': eq.get('team_name'),
                 'equipo_formations': a_texto(eq.get('formations')),
                 'equipo_events': a_texto(eq.get('events'))}
            r.update(p)
            filas.append(r)
    return anidado_a_texto(pd.DataFrame(filas))


def goles_equipos(ev):
    e = ev[ev['period'] < 5]   # excluye la tanda de penales
    g = {}
    if 'shot.outcome.name' in e.columns:
        s = e[(e['type.name'] == 'Shot') & (e['shot.outcome.name'] == 'Goal')]
        for t, n in s['team.id'].value_counts().items():
            g[int(t)] = g.get(int(t), 0) + int(n)
    og = e[e['type.name'] == 'Own Goal For']
    for t, n in og['team.id'].value_counts().items():
        g[int(t)] = g.get(int(t), 0) + int(n)
    return g


# ---------- Qué falta por convertir ----------
hechos = set()
if LOG.exists():
    with open(LOG) as f:
        for linea in f:
            try:
                hechos.add(json.loads(linea)['match_id'])
            except Exception:
                pass
pend = [m for m in partidos if int(m['match_id']) not in hechos]
if MAX_PARTIDOS:
    paso = max(1, len(pend) // MAX_PARTIDOS)
    pend = pend[::paso][:MAX_PARTIDOS]
print(f"Partidos objetivo: {len(partidos)} | ya convertidos: {len(hechos)} | a convertir ahora: {len(pend)}")

# ---------- Conversión ----------
t0 = time.time()
errores, respaldo, n_proc = [], 0, 0
with open(LOG, 'a') as flog:
    for m in pend:
        if time.time() - t0 > TIEMPO_MAX_MIN * 60:
            print("Tiempo máximo alcanzado. Vuelve a correr la celda para continuar.")
            break
        mid = int(m['match_id'])
        try:
            ev = aplanar_eventos(leer('eventos', mid), mid)
            if guardar(ev, OUT / 'eventos' / f'{mid}.parquet') != 'ok':
                respaldo += 1
            mapa = dict(zip(ev['id'], ev['index']))

            lu = convertir_lineups(leer('lineups', mid), mid)
            guardar(lu, OUT / 'lineups' / f'{mid}.parquet')

            tiene360 = (RAW / 'frames360' / f'{mid}.json.gz').exists()
            n_frames = unm = sin_actor = 0
            prom_vis = None
            if tiene360:
                frames, jug = convertir_360(leer('frames360', mid), mapa, mid)
                guardar(frames, OUT / 'frames360' / f'{mid}.parquet')
                guardar(jug, OUT / 'jugadores360' / f'{mid}.parquet')
                n_frames = len(frames)
                unm = int((frames['event_index'] < 0).sum())
                sin_actor = int((~frames['tiene_actor']).sum())
                prom_vis = float(frames['n_visibles'].mean())

            g = goles_equipos(ev)
            gl = g.get(int(m['home_team.home_team_id']), 0)
            gv = g.get(int(m['away_team.away_team_id']), 0)
            fuera = -1
            if 'loc_x' in ev.columns:
                fuera = int(((ev['loc_x'] < 0) | (ev['loc_x'] > 120) |
                             (ev['loc_y'] < 0) | (ev['loc_y'] > 80)).sum())
            fila = {
                'match_id': mid, 'season_id': int(m['season_id']),
                'n_eventos': int(len(ev)), 'n_columnas': int(ev.shape[1]),
                'n_starting_xi': int((ev['type.name'] == 'Starting XI').sum()),
                'periodo_max': int(ev['period'].max()), 'minuto_max': int(ev['minute'].max()),
                'loc_fuera_campo': fuera,
                'n_lineup_min_equipo': int(lu.groupby('team_id').size().min()) if len(lu) else 0,
                'tiene_360': bool(tiene360), 'n_frames': int(n_frames),
                'frames_sin_evento': unm, 'frames_sin_actor': sin_actor, 'prom_visibles': prom_vis,
                'goles_ev_local': int(gl), 'goles_ev_visita': int(gv),
                'goles_marcador_local': int(m['home_score']),
                'goles_marcador_visita': int(m['away_score']),
            }
            fila['coincide_marcador'] = bool(gl == fila['goles_marcador_local']
                                             and gv == fila['goles_marcador_visita'])
            flog.write(json.dumps(fila) + '\n')
            flog.flush()
            n_proc += 1
            if n_proc % 25 == 0:
                print(f"  {n_proc} partidos | {(time.time() - t0) / 60:.1f} min")
        except Exception as e:
            errores.append((mid, type(e).__name__, str(e)[:150]))

print(f"\nTiempo promedio: {(time.time() - t0) / max(1, n_proc):.1f} s por partido ({n_proc} partidos)")

# ---------- Resumen para pegar ----------
print("\n=== RESUMEN ===")
val = pd.read_json(LOG, lines=True).drop_duplicates('match_id', keep='last')
print("Partidos convertidos en total:", len(val), "de", len(partidos))
print("Errores en esta corrida:", len(errores))
for e in errores[:10]:
    print("  ", e)
print("Eventos guardados con respaldo de texto:", respaldo)

print("\n1) Peso en Drive por carpeta")
for c in ['eventos', 'frames360', 'jugadores360', 'lineups']:
    arch = list((OUT / c).glob('*.parquet'))
    print(f"{c}: {len(arch)} archivos | {sum(a.stat().st_size for a in arch) / 1e9:.3f} GB")

print("\n2) Por temporada")
print(val.groupby('season_id').agg(partidos=('match_id', 'count'),
                                   eventos_mediana=('n_eventos', 'median'),
                                   con_360=('tiene_360', 'sum'),
                                   marcador_ok=('coincide_marcador', 'sum')).to_string())

print("\n3) Chequeos de sanidad")
mal = val[~val['coincide_marcador']]
print("Partidos con goles de eventos distintos al marcador:", len(mal))
if len(mal):
    print(mal[['match_id', 'goles_ev_local', 'goles_ev_visita',
               'goles_marcador_local', 'goles_marcador_visita']].head(10).to_string(index=False))
print("Starting XI distinto de 2:", int((val['n_starting_xi'] != 2).sum()))
print("Mínimo de jugadores por equipo en lineups:", int(val['n_lineup_min_equipo'].min()))
print("Eventos con ubicación fuera de 0-120 / 0-80:", int(val['loc_fuera_campo'].clip(lower=0).sum()))
print("Periodo máximo:", val['periodo_max'].value_counts().to_dict())
v360 = val[val['tiene_360']]
if len(v360):
    print("Frames 360 sin evento enlazado:", int(v360['frames_sin_evento'].sum()),
          "de", int(v360['n_frames'].sum()))
    print("Frames 360 sin actor:", int(v360['frames_sin_actor'].sum()))
    print("Jugadores visibles por frame (promedio):", round(v360['prom_visibles'].mean(), 1))

print("\n4) Esquema de eventos (muestra de hasta 100 archivos)")
arch = sorted((OUT / 'eventos').glob('*.parquet'))
if arch:
    muestra = arch[::max(1, len(arch) // 100)]
    tipos, cols_arch = {}, []
    for a in muestra:
        s = pq.read_schema(a)
        cols_arch.append(set(s.names))
        for n, t in zip(s.names, s.types):
            tipos.setdefault(n, set()).add(str(t))
    print("Columnas: unión", len(set().union(*cols_arch)),
          "| comunes a todos", len(set.intersection(*cols_arch)))
    conf = {n: sorted(t) for n, t in tipos.items() if len(t - {'null'}) > 1}
    print("Columnas con tipos distintos entre partidos:", len(conf))
    for n, t in list(conf.items())[:15]:
        print("  ", n, t)

    print("\n5) Presencia de columnas v11 por temporada (fracción de la muestra)")
    clave = ['control_degree', 'is_transition', 'is_controlled_possession',
             'possession_directness', 'location_bucket', 'obv_total_net',
             'defensive_responsibility.probability_sum']
    temp = dict(zip(val['match_id'], val['season_id']))
    filas = []
    for a, cols in zip(muestra, cols_arch):
        r = {'season_id': temp.get(int(a.stem))}
        for k in clave:
            r[k] = k in cols
        filas.append(r)
    print(pd.DataFrame(filas).groupby('season_id').mean().round(2).to_string())

print("\n6) Versiones de datos por temporada (tabla de partidos)")
sel = matches[matches['season_id'].isin(TEMPORADAS)]
for c in ['metadata.data_version', 'metadata.shot_fidelity_version', 'metadata.xy_fidelity_version']:
    if c in sel.columns:
        print(c)
        print(sel.groupby(['season_label', c]).size().to_string())

Resumen

In [ ]:
import time
import numpy as np, pandas as pd, pyarrow.parquet as pq

OUT = BASE / 'datos_parquet' / 'liga'
LOG = BASE / 'salidas' / 'validacion_etapa1.jsonl'
RES = BASE / 'datos_parquet' / 'resumen_partido_etapa1.parquet'
TIEMPO_MAX_MIN = 40

val = pd.read_json(LOG, lines=True).drop_duplicates('match_id', keep='last')
archivo = sorted((BASE / 'datos_parquet').glob('matches_73_*.parquet'))[-1]
matches = pd.read_parquet(archivo)[['match_id', 'season_label', 'competition_stage.name']]

PEDIR = ['type.name', 'is_transition', 'control_degree', 'possession_directness',
         'obv_total_net', 'under_pressure', 'counterpress', 'set_piece_phase',
         'pass.outcome.name', 'pass.pass_success_probability', 'shot.statsbomb_xg']


def serie(df, c):
    return df[c] if c in df.columns else pd.Series(np.nan, index=df.index)


def resumen_eventos(mid):
    ruta = OUT / 'eventos' / f'{mid}.parquet'
    nombres = pq.read_schema(ruta).names
    ev = pd.read_parquet(ruta, columns=[c for c in PEDIR if c in nombres])
    tipo = serie(ev, 'type.name')
    pases = ev[tipo == 'Pass']
    trans = serie(ev, 'is_transition').dropna()
    return {
        'match_id': mid, 'n_eventos': len(ev),
        'pct_con_control_degree': serie(ev, 'control_degree').notna().mean(),
        'control_degree_medio': serie(ev, 'control_degree').mean(),
        'pct_transicion_sb': trans.eq(True).mean() if len(trans) else np.nan,
        'directness_media': serie(ev, 'possession_directness').mean(),
        'pct_con_obv': serie(ev, 'obv_total_net').notna().mean(),
        'obv_total_medio': serie(ev, 'obv_total_net').mean(),
        'pases': len(pases),
        'pct_pase_sin_outcome': serie(pases, 'pass.outcome.name').isna().mean() if len(pases) else np.nan,
        'prob_exito_pase_media': serie(pases, 'pass.pass_success_probability').mean(),
        'tiros': int((tipo == 'Shot').sum()),
        'xg_total': serie(ev, 'shot.statsbomb_xg').sum(),
        'presiones': int((tipo == 'Pressure').sum()),
        'pct_bajo_presion': serie(ev, 'under_pressure').eq(True).mean(),
        'pct_contrapresion': serie(ev, 'counterpress').eq(True).mean(),
        'pct_balon_parado': serie(ev, 'set_piece_phase').notna().mean(),
    }


def resumen_360(mid, n_eventos):
    f = OUT / 'frames360' / f'{mid}.parquet'
    if not f.exists():
        return {'tiene_360': False}
    fr = pd.read_parquet(f, columns=['n_visibles', 'tiene_actor', 'area_visible_vacia'])
    jg = pd.read_parquet(OUT / 'jugadores360' / f'{mid}.parquet', columns=['x', 'y'])
    fuera = (jg['x'] < 0) | (jg['x'] > 120) | (jg['y'] < 0) | (jg['y'] > 80)
    return {'tiene_360': True,
            'frames_por_evento': len(fr) / n_eventos,
            'visibles_medio': fr['n_visibles'].mean(),
            'pct_sin_actor': (~fr['tiene_actor'].astype(bool)).mean(),
            'pct_area_vacia': fr['area_visible_vacia'].astype(bool).mean(),
            'pct_jugadores_fuera_campo': fuera.mean()}


hechos = set(pd.read_parquet(RES)['match_id']) if RES.exists() else set()
pend = [int(m) for m in val['match_id'] if int(m) not in hechos]
print(f"Partidos: {len(val)} | ya resumidos: {len(hechos)} | por resumir: {len(pend)}")

t0, filas = time.time(), []
for i, mid in enumerate(pend, 1):
    if time.time() - t0 > TIEMPO_MAX_MIN * 60:
        print("Tiempo máximo alcanzado. Vuelve a correr la celda para continuar.")
        break
    r = resumen_eventos(mid)
    r.update(resumen_360(mid, r['n_eventos']))
    filas.append(r)
    if i % 200 == 0:
        print(f"  {i} partidos | {(time.time() - t0) / 60:.1f} min")

if filas:
    nuevo = pd.DataFrame(filas)
    if RES.exists():
        nuevo = pd.concat([pd.read_parquet(RES), nuevo], ignore_index=True)
    nuevo.to_parquet(RES, index=False)

res = pd.read_parquet(RES).merge(matches, on='match_id')
print("\nPartidos en la tabla resumen:", len(res), "de", len(val))

recoleccion = ['pct_con_control_degree', 'pct_con_obv', 'frames_por_evento', 'visibles_medio',
               'pct_sin_actor', 'pct_area_vacia', 'pct_jugadores_fuera_campo']
modelo = ['control_degree_medio', 'pct_transicion_sb', 'directness_media',
          'obv_total_medio', 'prob_exito_pase_media']
juego = ['pases', 'tiros', 'xg_total', 'presiones', 'pct_bajo_presion', 'pct_contrapresion',
         'pct_pase_sin_outcome', 'pct_balon_parado']


def comparar(cols, titulo):
    g = res.groupby('season_label')[cols].mean().T
    desv = ((g.max(axis=1) - g.min(axis=1)) / res[cols].std()).round(2)
    g['desv'] = desv
    g['revisar'] = np.where(desv >= 0.3, '<--', '')
    print(f"\n{titulo}")
    print(g.round(3).to_string())


comparar(recoleccion, "A) RECOLECCIÓN (cobertura y visibilidad)")
comparar(modelo, "B) CAMPOS DEL MODELO DE STATSBOMB")
comparar(juego, "C) CONTEXTO DE JUEGO (pueden cambiar por razones futbolísticas)")

print("\nTamaño de alineación (mínimo por equipo), partidos por valor:")
print(val['n_lineup_min_equipo'].value_counts().sort_index().to_string())

print("\nPartidos con más de 2 periodos, por etapa:")
x = val[val['periodo_max'] > 2].merge(matches, on='match_id')
print(x.groupby(['periodo_max', 'competition_stage.name']).size().to_string())

Coach diagnostics

In [ ]:
import os, glob, time
import numpy as np, pandas as pd
import pyarrow.parquet as pq

t0 = time.time()
BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026')
P = f'{BASE}/datos_parquet'
UMBRALES = [5, 10, 15, 20, 30, 40, 60]
U_CONTEXTO = 15
MIN_ESTADO = 270
ESTADOS = ['pierde2+', 'pierde1', 'empate', 'gana1', 'gana2+']
mcols = [f'min_{k}' for k in ESTADOS]
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 100)

def a_bool(s):
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).astype(float) == 1
    return s.astype(str).str.lower().str.strip().isin(['true', '1', 'si', 'sí', 'local', 'home', 'l', 'h', 'casa'])

def etiqueta(d):
    return 'pierde2+' if d <= -2 else 'pierde1' if d == -1 else 'empate' if d == 0 else 'gana1' if d == 1 else 'gana2+'

# ===== 1. TABLA DE TÉCNICOS =====
ptp = pd.read_parquet(f'{P}/partido_tecnico_periodo.parquet')
print('== 1. TABLA DE TÉCNICOS', ptp.shape)
for c in ['jugado', 'lado', 'rol', 'n_tec']:
    print(f'  {c}:', ptp[c].value_counts(dropna=False).head(8).to_dict())
print('  etapa (muestra):', ptp['etapa'].drop_duplicates().head(5).tolist(), '| tipo:', ptp['etapa'].dtype)
print('  temporadas:', sorted(ptp['temporada'].astype(str).unique()))

ptp = ptp[a_bool(ptp['jugado'])].copy()
ptp['match_id'] = pd.to_numeric(ptp['match_id']).astype('int64')
ptp['_fecha'] = pd.to_datetime(ptp['fecha'])
ptp['_local'] = a_bool(ptp['lado'])
ptp['_temp'] = ptp['temporada'].astype(str)
ptp['_int'] = ptp['rol'].astype(str).str.lower().str.contains('interin')
ptp['etapa_id'] = ptp['team'].astype(str) + '_' + ptp['etapa'].astype(str)
ptp = ptp.sort_values(['team', '_fecha', 'match_id'])
ptp['racha_id'] = ptp['team'].astype(str) + '_r' + (ptp['tec_p'] != ptp.groupby('team')['tec_p'].shift()).groupby(ptp['team']).cumsum().astype(str)
n_tab, n_rach = ptp.etapa_id.nunique(), ptp.racha_id.nunique()
ID = 'etapa_id' if abs(n_tab - 148) <= abs(n_rach - 148) else 'racha_id'
print(f'Filas jugadas: {len(ptp)} (esperado 3,578) | local=True: {int(ptp._local.sum())} (esperado la mitad)')
print(f'Etapas según columna etapa: {n_tab} | según rachas de tec_p: {n_rach} | bitácora: 148 -> uso {ID}')

# ===== 2. EVENTOS: marcador y minutos por estado =====
ev_files = sorted(glob.glob(f'{P}/liga/eventos/*.parquet'))
print(f'\n== 2. EVENTOS: {len(ev_files)} partidos (esperado 1,364)')
CLAVE = ['type.name', 'team.id', 'team.name', 'player.id', 'position.name', 'possession', 'possession_team.id',
         'play_pattern.name', 'location_bucket', 'is_transition', 'control_degree', 'possession_directness',
         'obv_total_net', 'shot.statsbomb_xg', 'shot.outcome.name', 'pass.length', 'pass.type.name', 'pass.cross',
         'pass.outcome.name', 'loc_x', 'loc_y', 'pass_end_x', 'carry_end_x', 'under_pressure', 'counterpress',
         'substitution.replacement.id', 'substitution.outcome.name', 'bad_behaviour.card.name',
         'foul_committed.card.name', 'tactics.formation', 'minute', 'second', 'period']
nom0 = set(pq.read_schema(ev_files[0]).names)
print('Columnas clave ausentes en un partido de muestra:', [c for c in CLAVE if c not in nom0])
print(f'Total de columnas en ese partido: {len(nom0)}')

NEC = ['type.name', 'team.name', 'shot.outcome.name', 'period', 'minute', 'second']
filas, fallos = [], []
for i, f in enumerate(ev_files):
    mid = int(os.path.basename(f).split('.')[0])
    try:
        disp = set(pq.read_schema(f).names)
        e = pd.read_parquet(f, columns=[c for c in NEC if c in disp])
        e = e[e['period'] <= 4].copy()
        e['t'] = e['minute'] * 60 + e['second']
        xi = e[e['type.name'] == 'Starting XI']
        eq = list(dict.fromkeys(xi['team.name']))
        if len(eq) != 2:
            fallos.append(mid); continue
        goles = []
        if 'shot.outcome.name' in e.columns:
            s = e[(e['type.name'] == 'Shot') & (e['shot.outcome.name'] == 'Goal')]
            goles += list(zip(s['period'], s['t'], s['team.name']))
        og = e[e['type.name'] == 'Own Goal For']
        goles += list(zip(og['period'], og['t'], og['team.name']))
        goles.sort()
        a, b_ = eq
        marc = {a: 0, b_: 0}
        mins = {a: dict.fromkeys(ESTADOS, 0.0), b_: dict.fromkeys(ESTADOS, 0.0)}
        for per, ini in [(1, 0), (2, 2700)]:
            ep = e[e['period'] == per]
            if ep.empty:
                continue
            cur = ini
            for gp, gt, gtm in goles:
                if gp != per:
                    continue
                for x, y in [(a, b_), (b_, a)]:
                    mins[x][etiqueta(marc[x] - marc[y])] += max(gt - cur, 0)
                marc[gtm] += 1
                cur = max(cur, gt)
            for x, y in [(a, b_), (b_, a)]:
                mins[x][etiqueta(marc[x] - marc[y])] += max(ep['t'].max() - cur, 0)
        for gp, gt, gtm in goles:
            if gp >= 3:
                marc[gtm] += 1
        for x, y in [(a, b_), (b_, a)]:
            fila = {'match_id': mid, 'team': x, 'rival': y, 'gf': marc[x], 'gc': marc[y]}
            fila.update({f'min_{k}': v / 60 for k, v in mins[x].items()})
            filas.append(fila)
    except Exception:
        fallos.append(mid)
    if (i + 1) % 250 == 0:
        print(f'  {i + 1} partidos leídos ({time.time() - t0:.0f} s)')

tm = pd.DataFrame(filas)
pn = ptp.groupby('match_id')['team'].apply(set).to_dict()
mapa = {}
for mid, g_ in tm.groupby('match_id'):
    p_, s_ = pn.get(mid, set()), set(g_['team'])
    if len(s_ - p_) == 1 and len(p_ - s_) == 1:
        mapa[next(iter(s_ - p_))] = next(iter(p_ - s_))
if mapa:
    print('Nombres de equipo reconciliados (eventos -> tabla):', mapa)
    tm['team'] = tm['team'].replace(mapa); tm['rival'] = tm['rival'].replace(mapa)
print(f'Filas equipo-partido: {len(tm)} (esperado {2 * len(ev_files)}) | partidos con fallo: {len(fallos)} {fallos[:10]}')
print(f'Goles por partido (sin tanda de penales): {tm.gf.sum() / (len(tm) / 2):.2f}')
print(f'Minutos promedio por equipo-partido (periodos 1 y 2): {tm[mcols].sum(axis=1).mean():.1f}')
tot = tm[mcols].sum()
print('Reparto del tiempo por estado de marcador (liga):', (tot / tot.sum()).round(3).to_dict())

# ===== 3. ETAPAS Y UMBRALES =====
b = ptp.merge(tm, on=['match_id', 'team'], how='left')
b['_ev'] = b['gf'].notna()
print('\n== 3. ETAPAS')
print(f'Filas con eventos que casan con la tabla de técnicos: {int(b._ev.sum())} de {len(tm)}')
E = b.groupby(ID).agg(club=('team', 'first'), tecnico=('tec_p', 'first'), ini=('_fecha', 'min'), fin=('_fecha', 'max'),
                      n_total=('match_id', 'size'), n_ev=('_ev', 'sum'), interino=('_int', 'any'),
                      rol=('rol', lambda s: s.astype(str).mode().iat[0])).reset_index().rename(columns={ID: 'etapa_id'})
print(f'Etapas: {len(E)} | rol por etapa: {E.rol.value_counts().to_dict()} | interinas: {int(E.interino.sum())} (bitácora: 20)')
base = E[(E.n_ev > 0) & (~E.interino)].copy()
print(f'Etapas no interinas con partidos en 2021/22-2024/25: {len(base)}')

res = []
for U in UMBRALES:
    ok = base[base.n_ev >= U]
    clubes = ok.groupby('tecnico')['club'].nunique()
    netapas = ok.groupby('tecnico').size()
    pares = 0
    for _, g_ in base.sort_values('ini').groupby('club'):
        n = g_['n_ev'].tolist()
        pares += sum(1 for x, y in zip(n[:-1], n[1:]) if x >= U and y >= U)
    res.append(dict(umbral=U, etapas=len(ok), tecnicos=ok.tecnico.nunique(), tec_2etapas=int((netapas >= 2).sum()),
                    tec_2clubes=int((clubes >= 2).sum()), relevos_club=pares, partidos=int(ok.n_ev.sum())))
print('\nUmbral = partidos mínimos de la etapa dentro de 2021/22-2024/25')
print(pd.DataFrame(res).to_string(index=False))

ok = base[base.n_ev >= U_CONTEXTO]
mov = ok.groupby('tecnico').filter(lambda g_: len(g_) >= 2).sort_values(['tecnico', 'ini'])
print(f'\nTécnicos con 2+ etapas de {U_CONTEXTO}+ partidos ({mov.tecnico.nunique()}):')
for t, g_ in mov.groupby('tecnico'):
    print('  ', t, '|', ' -> '.join(f'{r.club} ({int(r.n_ev)}, {r.ini:%Y-%m})' for r in g_.itertuples()))

# ===== 4. PARTIDOS POR CONTEXTO =====
bm = b[b._ev].copy()
bm['pts'] = np.select([bm.gf > bm.gc, bm.gf == bm.gc], [3, 1], 0)
ppg = bm.groupby(['_temp', 'team'])['pts'].mean().rename('ppg').reset_index()
ppg['pct'] = ppg.groupby('_temp')['ppg'].rank(pct=True)
ppg['nivel'] = np.where(ppg.pct > 2 / 3, 'fuerte', np.where(ppg.pct <= 1 / 3, 'debil', 'medio'))
bm = bm.merge(ppg[['_temp', 'team', 'nivel']].rename(columns={'team': 'rival', 'nivel': 'nivel_rival'}),
              on=['_temp', 'rival'], how='left')
bm['_id'] = bm[ID]
C = bm.groupby('_id').agg(
    local=('_local', lambda s: int((s == True).sum())),
    visita=('_local', lambda s: int((s == False).sum())),
    vs_fuerte=('nivel_rival', lambda s: int((s == 'fuerte').sum())),
    vs_medio=('nivel_rival', lambda s: int((s == 'medio').sum())),
    vs_debil=('nivel_rival', lambda s: int((s == 'debil').sum())),
    **{c: (c, 'sum') for c in mcols}).reset_index().rename(columns={'_id': 'etapa_id'})
K = base[base.n_ev >= U_CONTEXTO].merge(C, on='etapa_id')
cols = ['n_ev', 'local', 'visita', 'vs_fuerte', 'vs_medio', 'vs_debil'] + mcols
print(f'\n== 4. CONTEXTO en las {len(K)} etapas no interinas con {U_CONTEXTO}+ partidos')
print(K[cols].describe(percentiles=[.1, .25, .5]).loc[['min', '10%', '25%', '50%', 'max']].round(0).to_string())
pct = {c: round(100 * (K[c] >= 5).mean()) for c in ['local', 'visita', 'vs_fuerte', 'vs_medio', 'vs_debil']}
pct.update({c: round(100 * (K[c] >= MIN_ESTADO).mean()) for c in mcols})
print(f'% de etapas con 5+ partidos por celda (o {MIN_ESTADO}+ minutos por estado):'); print(pct)

# ===== 5. AMÉRICA (solo conteos) =====
am = E[E.club.astype(str).str.contains('Am.rica', case=False, regex=True)].sort_values('ini').merge(C, on='etapa_id', how='left')
print('\n== 5. AMÉRICA: n_total = toda la tabla (incluye 2025/26 y 2026/27); n_ev = con eventos descargados')
print(am[['tecnico', 'ini', 'fin', 'n_total', 'n_ev', 'rol', 'local', 'visita', 'vs_fuerte',
          'min_pierde1', 'min_gana1']].to_string(index=False))

E.to_parquet(f'{P}/viabilidad_etapas.parquet', index=False)
tm.to_parquet(f'{P}/equipo_partido_marcador.parquet', index=False)
print(f'\nGuardados viabilidad_etapas y equipo_partido_marcador. Tiempo total: {time.time() - t0:.0f} s')

Download etapa 2 lineups eventos y 360

In [ ]:
import gzip, os, time, requests, pandas as pd
from concurrent.futures import ThreadPoolExecutor
from datetime import date

URL_LINEUPS = " https://data.statsbomb.com/api/v5/lineups/{match_id}"
URL_EVENTOS = " https://data.statsbomb.com/api/v11/events/{match_id}"
URL_360     =  "https://data.statsbomb.com/api/v2/360-frames/?{match_id}"
TEMPORADAS = [318]          # etapa 2: 2025/26
TRABAJADORES = 4
TIEMPO_MAX_MIN = 50
TIPOS = {'lineups': URL_LINEUPS, 'eventos': URL_EVENTOS, 'frames360': URL_360}

DEST = BASE / 'datos_crudos' / 'liga'
for t in TIPOS: (DEST / t).mkdir(parents=True, exist_ok=True)

archivo = sorted((BASE / 'datos_parquet').glob('matches_73_*.parquet'))[-1]
matches = pd.read_parquet(archivo)
obj = matches[matches['home_score'].notna() & matches['season_id'].isin(TEMPORADAS)]
obj = obj.sort_values(['season_id', 'match_date'])

existentes = {t: set(os.listdir(DEST / t)) for t in TIPOS}
pendientes = []
for _, f in obj.iterrows():
    mid = f['match_id']
    for t in TIPOS:
        if t == 'frames360' and f['match_status_360'] != 'available':
            continue
        if f'{mid}.json.gz' not in existentes[t]:
            pendientes.append((mid, t))

def descargar(mid, tipo):
    destino = DEST / tipo / f'{mid}.json.gz'
    url = TIPOS[tipo].format(match_id=mid)
    for intento in range(1, 5):
        try:
            r = requests.get(url, auth=(USUARIO, CLAVE), timeout=180)
            if r.status_code == 200:
                r.json()
                tmp = destino.with_suffix('.tmp')
                with open(tmp, 'wb') as f:
                    f.write(gzip.compress(r.content))
                tmp.replace(destino)
                return (mid, tipo, 'ok')
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(10 * intento)
                continue
            return (mid, tipo, f'HTTP {r.status_code}')
        except Exception as e:
            time.sleep(5 * intento)
    return (mid, tipo, 'fallo_reintentos')

print(f"Partidos objetivo: {len(obj)} | descargas pendientes: {len(pendientes)}")
limite = time.time() + TIEMPO_MAX_MIN * 60
resultados = []
LOTE = 200
for i in range(0, len(pendientes), LOTE):
    if time.time() > limite:
        print("Tiempo máximo alcanzado. Vuelve a correr la celda para continuar.")
        break
    bloque = pendientes[i:i + LOTE]
    with ThreadPoolExecutor(TRABAJADORES) as ex:
        resultados.extend(ex.map(lambda t: descargar(*t), bloque))
    print(f"Avance: {min(i + LOTE, len(pendientes))} de {len(pendientes)} descargas")

res = pd.DataFrame(resultados, columns=['match_id', 'tipo', 'estado'])
if len(res):
    print("\nResultado de esta corrida:")
    print(res['estado'].value_counts().to_string())
    fallos = res[res['estado'] != 'ok']
    if len(fallos):
        fallos.to_csv(BASE / 'salidas' / f'fallos_descarga_{date.today()}.csv', index=False)
        print("Primeros fallos:\n", fallos.head(15).to_string(index=False))

print("\nArchivos en Drive vs esperados (todas las etapas descargadas hasta ahora):")
esperado = {'lineups': len(obj), 'eventos': len(obj),
            'frames360': int((obj['match_status_360'] == 'available').sum())}
for t in TIPOS:
    n = len([x for x in os.listdir(DEST / t) if x.endswith('.json.gz')])
    gb = sum((DEST / t / x).stat().st_size for x in os.listdir(DEST / t)) / 1e9
    print(f"{t}: {n} en Drive (etapa 1 + etapa 2) | esperado solo de etapa 2: {esperado[t]} | {gb:.2f} GB")

Conversor etapa 2

In [ ]:
import gzip, json, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

ETAPA = 'etapa2'
TEMPORADAS = [318]
MAX_PARTIDOS = None
TIEMPO_MAX_MIN = 45

RAW = BASE / 'datos_crudos' / 'liga'
OUT = BASE / 'datos_parquet' / 'liga'
for c in ['eventos', 'frames360', 'jugadores360', 'lineups']:
    (OUT / c).mkdir(parents=True, exist_ok=True)
LOG = BASE / 'salidas' / f'validacion_{ETAPA}.jsonl'

archivo = sorted((BASE / 'datos_parquet').glob('matches_73_*.parquet'))[-1]
matches = pd.read_parquet(archivo)
obj = matches[matches['home_score'].notna() & matches['season_id'].isin(TEMPORADAS)]
obj = obj.sort_values(['season_id', 'match_date'])
partidos = obj.to_dict('records')


def leer(tipo, mid):
    with gzip.open(RAW / tipo / f'{mid}.json.gz', 'rt', encoding='utf-8') as f:
        return json.load(f)


def a_texto(v):
    if v is None or (isinstance(v, float) and v != v):
        return None
    return json.dumps(v, ensure_ascii=False)


def anidado_a_texto(df):
    for c in df.columns:
        if pd.api.types.is_object_dtype(df[c]):
            if df[c].apply(lambda v: isinstance(v, (list, dict))).any():
                df[c] = df[c].apply(a_texto)
    return df


def separar_coordenadas(df, col, pref):
    if col not in df.columns:
        return
    for i, eje in enumerate(['x', 'y', 'z']):
        vals = df[col].apply(lambda a: a[i] if isinstance(a, list) and len(a) > i else np.nan)
        if eje == 'z' and vals.isna().all():
            continue
        df[f'{pref}_{eje}'] = vals.astype('float32')
    df.drop(columns=[col], inplace=True)


def aplanar_eventos(raw, mid):
    df = pd.json_normalize(raw)
    df.insert(0, 'match_id', mid)
    for col, pref in [('location', 'loc'), ('pass.end_location', 'pass_end'),
                      ('carry.end_location', 'carry_end'), ('shot.end_location', 'shot_end')]:
        separar_coordenadas(df, col, pref)
    return anidado_a_texto(df)


def guardar(df, ruta):
    try:
        df.to_parquet(ruta, index=False, compression='zstd')
        return 'ok'
    except Exception:
        d = df.copy()
        for c in d.columns:
            if pd.api.types.is_object_dtype(d[c]):
                d[c] = d[c].apply(lambda v: None if v is None or (isinstance(v, float) and v != v) else str(v))
        d.to_parquet(ruta, index=False, compression='zstd')
        return 'respaldo_texto'


def convertir_360(raw, mapa, mid):
    fr, jg = [], []
    for f in raw:
        uuid = f.get('event_uuid')
        idx = mapa.get(uuid, -1)
        ff = f.get('freeze_frame') or []
        va = f.get('visible_area') or []
        mates = sum(1 for p in ff if p.get('teammate'))
        fr.append({
            'match_id': mid, 'event_index': idx, 'event_uuid': uuid,
            'line_breaking_pass': f.get('line_breaking_pass'),
            'num_defenders_on_goal_side_of_actor': f.get('num_defenders_on_goal_side_of_actor'),
            'distance_to_nearest_defender': f.get('distance_to_nearest_defender'),
            'ball_receipt_in_space': f.get('ball_receipt_in_space'),
            'ball_receipt_exceeds_distance': f.get('ball_receipt_exceeds_distance'),
            'n_visibles': len(ff), 'n_visibles_companeros': mates,
            'n_visibles_rivales': len(ff) - mates,
            'tiene_actor': any(p.get('actor') for p in ff),
            'area_visible_vacia': len(va) == 0,
            'visible_area': a_texto(va) if va else None,
            'visible_player_counts': a_texto(f.get('visible_player_counts')),
        })
        for p in ff:
            loc = p.get('location') or [np.nan, np.nan]
            jg.append((mid, idx, bool(p.get('teammate')), bool(p.get('actor')),
                       bool(p.get('keeper')), loc[0], loc[1]))
    frames = pd.DataFrame(fr)
    jug = pd.DataFrame(jg, columns=['match_id', 'event_index', 'teammate', 'actor', 'keeper', 'x', 'y'])
    jug = jug.astype({'match_id': 'int32', 'event_index': 'int32', 'x': 'float32', 'y': 'float32'})
    return frames, jug


def convertir_lineups(raw, mid):
    filas = []
    for eq in raw:
        for p in eq.get('lineup') or []:
            r = {'match_id': mid, 'team_id': eq.get('team_id'), 'team_name': eq.get('team_name'),
                 'equipo_formations': a_texto(eq.get('formations')),
                 'equipo_events': a_texto(eq.get('events'))}
            r.update(p)
            filas.append(r)
    return anidado_a_texto(pd.DataFrame(filas))


def goles_equipos(ev):
    e = ev[ev['period'] < 5]
    g = {}
    if 'shot.outcome.name' in e.columns:
        s = e[(e['type.name'] == 'Shot') & (e['shot.outcome.name'] == 'Goal')]
        for t, n in s['team.id'].value_counts().items():
            g[int(t)] = g.get(int(t), 0) + int(n)
    og = e[e['type.name'] == 'Own Goal For']
    for t, n in og['team.id'].value_counts().items():
        g[int(t)] = g.get(int(t), 0) + int(n)
    return g


hechos = set()
if LOG.exists():
    with open(LOG) as f:
        for linea in f:
            try:
                hechos.add(json.loads(linea)['match_id'])
            except Exception:
                pass
pend = [m for m in partidos if int(m['match_id']) not in hechos]
if MAX_PARTIDOS:
    paso = max(1, len(pend) // MAX_PARTIDOS)
    pend = pend[::paso][:MAX_PARTIDOS]
print(f"Partidos objetivo: {len(partidos)} | ya convertidos: {len(hechos)} | a convertir ahora: {len(pend)}")

t0 = time.time()
errores, respaldo, n_proc = [], 0, 0
with open(LOG, 'a') as flog:
    for m in pend:
        if time.time() - t0 > TIEMPO_MAX_MIN * 60:
            print("Tiempo máximo alcanzado. Vuelve a correr la celda para continuar.")
            break
        mid = int(m['match_id'])
        try:
            ev = aplanar_eventos(leer('eventos', mid), mid)
            if guardar(ev, OUT / 'eventos' / f'{mid}.parquet') != 'ok':
                respaldo += 1
            mapa = dict(zip(ev['id'], ev['index']))

            lu = convertir_lineups(leer('lineups', mid), mid)
            guardar(lu, OUT / 'lineups' / f'{mid}.parquet')

            tiene360 = (RAW / 'frames360' / f'{mid}.json.gz').exists()
            n_frames = unm = sin_actor = 0
            prom_vis = None
            if tiene360:
                frames, jug = convertir_360(leer('frames360', mid), mapa, mid)
                guardar(frames, OUT / 'frames360' / f'{mid}.parquet')
                guardar(jug, OUT / 'jugadores360' / f'{mid}.parquet')
                n_frames = len(frames)
                unm = int((frames['event_index'] < 0).sum())
                sin_actor = int((~frames['tiene_actor']).sum())
                prom_vis = float(frames['n_visibles'].mean())

            g = goles_equipos(ev)
            gl = g.get(int(m['home_team.home_team_id']), 0)
            gv = g.get(int(m['away_team.away_team_id']), 0)
            fuera = -1
            if 'loc_x' in ev.columns:
                fuera = int(((ev['loc_x'] < 0) | (ev['loc_x'] > 120) |
                             (ev['loc_y'] < 0) | (ev['loc_y'] > 80)).sum())
            fila = {
                'match_id': mid, 'season_id': int(m['season_id']),
                'n_eventos': int(len(ev)), 'n_columnas': int(ev.shape[1]),
                'n_starting_xi': int((ev['type.name'] == 'Starting XI').sum()),
                'periodo_max': int(ev['period'].max()), 'minuto_max': int(ev['minute'].max()),
                'loc_fuera_campo': fuera,
                'n_lineup_min_equipo': int(lu.groupby('team_id').size().min()) if len(lu) else 0,
                'tiene_360': bool(tiene360), 'n_frames': int(n_frames),
                'frames_sin_evento': unm, 'frames_sin_actor': sin_actor, 'prom_visibles': prom_vis,
                'goles_ev_local': int(gl), 'goles_ev_visita': int(gv),
                'goles_marcador_local': int(m['home_score']),
                'goles_marcador_visita': int(m['away_score']),
            }
            fila['coincide_marcador'] = bool(gl == fila['goles_marcador_local']
                                             and gv == fila['goles_marcador_visita'])
            flog.write(json.dumps(fila) + '\n')
            flog.flush()
            n_proc += 1
            if n_proc % 25 == 0:
                print(f"  {n_proc} partidos | {(time.time() - t0) / 60:.1f} min")
        except Exception as e:
            errores.append((mid, type(e).__name__, str(e)[:150]))

print(f"\nTiempo promedio: {(time.time() - t0) / max(1, n_proc):.1f} s por partido ({n_proc} partidos)")

print("\n=== RESUMEN ===")
val = pd.read_json(LOG, lines=True).drop_duplicates('match_id', keep='last')
print("Partidos convertidos en total:", len(val), "de", len(partidos))
print("Errores en esta corrida:", len(errores))
for e in errores[:10]:
    print("  ", e)
print("Eventos guardados con respaldo de texto:", respaldo)

print("\n1) Peso en Drive por carpeta (todas las etapas)")
for c in ['eventos', 'frames360', 'jugadores360', 'lineups']:
    arch = list((OUT / c).glob('*.parquet'))
    print(f"{c}: {len(arch)} archivos | {sum(a.stat().st_size for a in arch) / 1e9:.3f} GB")

print("\n2) Por temporada")
print(val.groupby('season_id').agg(partidos=('match_id', 'count'),
                                   eventos_mediana=('n_eventos', 'median'),
                                   con_360=('tiene_360', 'sum'),
                                   marcador_ok=('coincide_marcador', 'sum')).to_string())

print("\n3) Chequeos de sanidad")
mal = val[~val['coincide_marcador']]
print("Partidos con goles de eventos distintos al marcador:", len(mal))
if len(mal):
    print(mal[['match_id', 'goles_ev_local', 'goles_ev_visita',
               'goles_marcador_local', 'goles_marcador_visita']].head(10).to_string(index=False))
print("Starting XI distinto de 2:", int((val['n_starting_xi'] != 2).sum()))
print("Mínimo de jugadores por equipo en lineups:", int(val['n_lineup_min_equipo'].min()))
print("Eventos con ubicación fuera de 0-120 / 0-80:", int(val['loc_fuera_campo'].clip(lower=0).sum()))
print("Periodo máximo:", val['periodo_max'].value_counts().to_dict())
v360 = val[val['tiene_360']]
if len(v360):
    print("Frames 360 sin evento enlazado:", int(v360['frames_sin_evento'].sum()),
          "de", int(v360['n_frames'].sum()))
    print("Frames 360 sin actor:", int(v360['frames_sin_actor'].sum()))
    print("Jugadores visibles por frame (promedio):", round(v360['prom_visibles'].mean(), 1))

print("\n4) Esquema de eventos de la etapa 2 (muestra de hasta 100 archivos)")
ids2 = set(val['match_id'])
arch = [a for a in sorted((OUT / 'eventos').glob('*.parquet')) if int(a.stem) in ids2]
if arch:
    muestra = arch[::max(1, len(arch) // 100)]
    tipos, cols_arch = {}, []
    for a in muestra:
        s = pq.read_schema(a)
        cols_arch.append(set(s.names))
        for n, t in zip(s.names, s.types):
            tipos.setdefault(n, set()).add(str(t))
    print("Columnas: unión", len(set().union(*cols_arch)),
          "| comunes a todos", len(set.intersection(*cols_arch)))
    conf = {n: sorted(t) for n, t in tipos.items() if len(t - {'null'}) > 1}
    print("Columnas con tipos distintos entre partidos:", len(conf))
    for n, t in list(conf.items())[:15]:
        print("  ", n, t)
    clave = ['control_degree', 'is_transition', 'is_controlled_possession',
             'possession_directness', 'location_bucket', 'obv_total_net',
             'defensive_responsibility.probability_sum']
    print("\n5) Columnas v11 presentes en 2025/26 (fracción de la muestra)")
    print({k: round(sum(k in c for c in cols_arch) / len(cols_arch), 2) for k in clave})

print("\n6) Versiones de datos en la tabla de partidos")
sel = matches[matches['season_id'].isin(TEMPORADAS)]
for c in ['metadata.data_version', 'metadata.shot_fidelity_version', 'metadata.xy_fidelity_version']:
    if c in sel.columns:
        print(c, sel[c].value_counts().to_dict())

Contexto

In [ ]:
import time
import numpy as np, pandas as pd, pyarrow.parquet as pq

OUT = BASE / 'datos_parquet' / 'liga'
LOG2 = BASE / 'salidas' / 'validacion_etapa2.jsonl'
RES1 = BASE / 'datos_parquet' / 'resumen_partido_etapa1.parquet'
RES2 = BASE / 'datos_parquet' / 'resumen_partido_etapa2.parquet'

val2 = pd.read_json(LOG2, lines=True).drop_duplicates('match_id', keep='last')
archivo = sorted((BASE / 'datos_parquet').glob('matches_73_*.parquet'))[-1]
matches = pd.read_parquet(archivo)[['match_id', 'season_label', 'competition_stage.name']]

PEDIR = ['type.name', 'is_transition', 'control_degree', 'possession_directness',
         'obv_total_net', 'under_pressure', 'counterpress', 'set_piece_phase',
         'pass.outcome.name', 'pass.pass_success_probability', 'shot.statsbomb_xg']


def serie(df, c):
    return df[c] if c in df.columns else pd.Series(np.nan, index=df.index)


def resumen_eventos(mid):
    ruta = OUT / 'eventos' / f'{mid}.parquet'
    nombres = pq.read_schema(ruta).names
    ev = pd.read_parquet(ruta, columns=[c for c in PEDIR if c in nombres])
    tipo = serie(ev, 'type.name')
    pases = ev[tipo == 'Pass']
    trans = serie(ev, 'is_transition').dropna()
    return {
        'match_id': mid, 'n_eventos': len(ev),
        'pct_con_control_degree': serie(ev, 'control_degree').notna().mean(),
        'control_degree_medio': serie(ev, 'control_degree').mean(),
        'pct_transicion_sb': trans.eq(True).mean() if len(trans) else np.nan,
        'directness_media': serie(ev, 'possession_directness').mean(),
        'pct_con_obv': serie(ev, 'obv_total_net').notna().mean(),
        'obv_total_medio': serie(ev, 'obv_total_net').mean(),
        'pases': len(pases),
        'pct_pase_sin_outcome': serie(pases, 'pass.outcome.name').isna().mean() if len(pases) else np.nan,
        'prob_exito_pase_media': serie(pases, 'pass.pass_success_probability').mean(),
        'tiros': int((tipo == 'Shot').sum()),
        'xg_total': serie(ev, 'shot.statsbomb_xg').sum(),
        'presiones': int((tipo == 'Pressure').sum()),
        'pct_bajo_presion': serie(ev, 'under_pressure').eq(True).mean(),
        'pct_contrapresion': serie(ev, 'counterpress').eq(True).mean(),
        'pct_balon_parado': serie(ev, 'set_piece_phase').notna().mean(),
    }


def resumen_360(mid, n_eventos):
    f = OUT / 'frames360' / f'{mid}.parquet'
    if not f.exists():
        return {'tiene_360': False}
    fr = pd.read_parquet(f, columns=['n_visibles', 'tiene_actor', 'area_visible_vacia'])
    jg = pd.read_parquet(OUT / 'jugadores360' / f'{mid}.parquet', columns=['x', 'y'])
    fuera = (jg['x'] < 0) | (jg['x'] > 120) | (jg['y'] < 0) | (jg['y'] > 80)
    return {'tiene_360': True,
            'frames_por_evento': len(fr) / n_eventos,
            'visibles_medio': fr['n_visibles'].mean(),
            'pct_sin_actor': (~fr['tiene_actor'].astype(bool)).mean(),
            'pct_area_vacia': fr['area_visible_vacia'].astype(bool).mean(),
            'pct_jugadores_fuera_campo': fuera.mean()}


hechos = set(pd.read_parquet(RES2)['match_id']) if RES2.exists() else set()
pend = [int(m) for m in val2['match_id'] if int(m) not in hechos]
print(f"Partidos etapa 2: {len(val2)} | ya resumidos: {len(hechos)} | por resumir: {len(pend)}")

filas = []
for mid in pend:
    r = resumen_eventos(mid)
    r.update(resumen_360(mid, r['n_eventos']))
    filas.append(r)
if filas:
    nuevo = pd.DataFrame(filas)
    if RES2.exists():
        nuevo = pd.concat([pd.read_parquet(RES2), nuevo], ignore_index=True)
    nuevo.to_parquet(RES2, index=False)

res = pd.concat([pd.read_parquet(RES1), pd.read_parquet(RES2)], ignore_index=True)
res = res.merge(matches, on='match_id')
print("\nPartidos por temporada en la comparación:")
print(res.groupby('season_label').size().to_string())

recoleccion = ['pct_con_control_degree', 'pct_con_obv', 'frames_por_evento', 'visibles_medio',
               'pct_sin_actor', 'pct_area_vacia', 'pct_jugadores_fuera_campo']
modelo = ['control_degree_medio', 'pct_transicion_sb', 'directness_media',
          'obv_total_medio', 'prob_exito_pase_media']
juego = ['pases', 'tiros', 'xg_total', 'presiones', 'pct_bajo_presion', 'pct_contrapresion',
         'pct_pase_sin_outcome', 'pct_balon_parado']


def comparar(cols, titulo):
    g = res.groupby('season_label')[cols].mean().T
    sd = res[cols].std()
    orden = list(g.columns)
    d = g[orden].diff(axis=1)
    g['desv'] = ((g.max(axis=1) - g.min(axis=1)) / sd).round(2)
    g['paso_ultimo'] = d[orden[-1]]
    g['paso_prom_previos'] = d[orden[1:-1]].mean(axis=1)
    print(f"\n{titulo}")
    print(g.round(3).to_string())


comparar(recoleccion, "A) RECOLECCIÓN (cobertura y visibilidad)")
comparar(modelo, "B) CAMPOS DEL MODELO DE STATSBOMB")
comparar(juego, "C) CONTEXTO DE JUEGO")

print("\nTamaño de alineación (mínimo por equipo), partidos por valor, etapa 2:")
print(val2['n_lineup_min_equipo'].value_counts().sort_index().to_string())

print("\nPartidos con más de 2 periodos, por etapa, etapa 2:")
x = val2[val2['periodo_max'] > 2].merge(matches, on='match_id')
print(x.groupby(['periodo_max', 'competition_stage.name']).size().to_string()
      if len(x) else "ninguno")

# Todos los datos descargados, chequeos de viabilidad below

In [ ]:
import os, glob, time
import numpy as np, pandas as pd
import pyarrow.parquet as pq

t0 = time.time()
BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026')
P = f'{BASE}/datos_parquet'
VENT, LARGO = 300, 32
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

print('Archivos en datos_parquet:', sorted(os.path.basename(x) for x in glob.glob(f'{P}/*.parquet')))
print('Archivos en salidas:', sorted(os.path.basename(x) for x in glob.glob(f'{BASE}/salidas/*'))[:30])

COLS = ['type.name', 'team.name', 'possession', 'period', 'minute', 'second', 'loc_x', 'pass.length',
        'pass.type.name', 'pass.outcome.name', 'pass.cross', 'shot.statsbomb_xg', 'shot.outcome.name',
        'shot.type.name', 'play_pattern.name', 'possession_directness', 'is_transition', 'counterpress',
        'obv_total_net', 'substitution.outcome.name', 'foul_committed.card.name', 'bad_behaviour.card.name',
        'duel.type.name']
ptp_ = pd.read_parquet(f'{P}/partido_tecnico_periodo.parquet')
ENTRENO = {'2021/2022', '2022/2023', '2023/2024', '2024/2025'}
ids_entreno = set(ptp_.loc[ptp_['temporada'].isin(ENTRENO), 'match_id'].astype(int))
ev_files = sorted(f for f in glob.glob(f'{P}/liga/eventos/*.parquet')
                  if int(os.path.basename(f).split('.')[0]) in ids_entreno)
print(f'Partidos de entrenamiento con eventos: {len(ev_files)} (esperado 1,364)')
tiene = {c: 0 for c in COLS}
for f in ev_files[:50]:
    n = set(pq.read_schema(f).names)
    for c in COLS:
        tiene[c] += c in n
print('Columnas presentes en 50 partidos de muestra (de 50):', tiene)

def div(a, b):
    return a / b if b and b > 0 else np.nan

est, ven, hit, fallos = [], [], [], []
for i, f in enumerate(ev_files):
    mid = int(os.path.basename(f).split('.')[0])
    try:
        disp = set(pq.read_schema(f).names)
        e = pd.read_parquet(f, columns=[c for c in COLS if c in disp])
        e = e[e['period'] <= 2].copy()
        g = lambda c: e[c] if c in e.columns else pd.Series(np.nan, index=e.index)
        e['t'] = e['minute'] * 60 + e['second']
        e['t_rel'] = np.where(e['period'] == 1, e['t'], e['t'] - 2700)
        e['ventana'] = (e['t_rel'] // VENT).clip(lower=0, upper=9).astype(int)
        eq = list(dict.fromkeys(e.loc[e['type.name'] == 'Starting XI', 'team.name']))
        if len(eq) != 2:
            fallos.append(mid); continue
        tipo, x, pt, plen = e['type.name'], g('loc_x'), g('pass.type.name'), g('pass.length')
        es_pase = tipo == 'Pass'
        abierto = es_pase & (pt.isna() | pt.isin(['Recovery', 'Interception']))
        completo = g('pass.outcome.name').isna()
        tiro = tipo == 'Shot'
        stype, pp, xg = g('shot.type.name'), g('play_pattern.name'), g('shot.statsbomb_xg').fillna(0)
        no_pen = tiro & (stype != 'Penalty')
        gol = (tiro & (g('shot.outcome.name') == 'Goal')) | (tipo == 'Own Goal For')
        trans, cp, cruz = g('is_transition') == True, g('counterpress') == True, g('pass.cross') == True
        pres = tipo == 'Pressure'
        defact = ((tipo == 'Duel') & (g('duel.type.name') == 'Tackle')) | tipo.isin(['Interception', 'Foul Committed'])
        obv = g('obv_total_net').fillna(0)
        for T, R in [(eq[0], eq[1]), (eq[1], eq[0])]:
            mT, mR = e['team.name'] == T, e['team.name'] == R
            pT, pR = (abierto & mT).sum(), (abierto & mR).sum()
            utT, utR = (abierto & mT & (x >= 80)).sum(), (abierto & mR & (x >= 80)).sum()
            prT = pres & mT
            gk = es_pase & mT & (pt == 'Goal Kick')
            xgT, xgR = xg[no_pen & mT].sum(), xg[no_pen & mR].sum()
            xg_op = xg[no_pen & mT & (stype == 'Open Play')].sum()
            dT = e.loc[mT & g('possession_directness').notna()].groupby('possession')['possession_directness'].first()
            est.append(dict(
                match_id=mid, team=T, rival=R,
                posesion_pct=div(pT, pT + pR), field_tilt=div(utT, utT + utR),
                ppda=div((abierto & mR & (x <= 72)).sum(), (defact & mT & (x >= 48)).sum()),
                pres_altura=x[prT].mean(), pres_alta_pct=div((prT & (x >= 80)).sum(), prT.sum()),
                contrapresion_pct=div((prT & cp).sum(), prT.sum()), directness=dT.mean(),
                largo_pct=div((abierto & mT & (plen >= LARGO)).sum(), pT),
                pase_ok_pct=div((abierto & mT & completo).sum(), pT),
                saque_meta_corto_pct=div((gk & (plen < LARGO)).sum(), gk.sum()),
                centros_por_ut=div((abierto & mT & cruz).sum(), utT),
                xg_np=xgT, xg_np_contra=xgR, xg_por_tiro=div(xgT, (no_pen & mT).sum()),
                xg_bp_pct=div(xg[no_pen & mT & pp.isin(['From Corner', 'From Free Kick', 'From Throw In'])].sum(), xgT),
                xg_trans_pct=div(xg[no_pen & mT & (stype == 'Open Play') & trans].sum(), xg_op),
                obv=obv[mT].sum(), presiones=int(prT.sum()), pases=int(pT)))
        w = pd.DataFrame({'period': e['period'], 'ventana': e['ventana'], 'team': e['team.name'],
                          'pases': abierto, 'pases_ut': abierto & (x >= 80), 'xg': xg.where(no_pen, 0),
                          'tiros': no_pen, 'goles': gol, 'presiones': pres, 'obv': obv})
        w = w[w.team.isin(eq)].groupby(['period', 'ventana', 'team']).sum(numeric_only=True).reset_index()
        w.insert(0, 'match_id', mid); ven.append(w)
        for idx in e.index[gol]:
            T = e.at[idx, 'team.name']; R = eq[1] if T == eq[0] else eq[0]
            pos = e.at[idx, 'possession']
            err = ((tipo == 'Error') & (e['team.name'] == R) & (e['possession'] == pos)).any()
            hit.append(dict(match_id=mid, period=e.at[idx, 'period'], t=e.at[idx, 't'], team=T, rival=R, hito='gol',
                            detalle=pp.at[idx], xg=xg.at[idx], error_previo=bool(err),
                            autogol=tipo.at[idx] == 'Own Goal For'))
        tarj = g('foul_committed.card.name').fillna(g('bad_behaviour.card.name'))
        for idx in e.index[tarj.notna()]:
            hit.append(dict(match_id=mid, period=e.at[idx, 'period'], t=e.at[idx, 't'], team=e.at[idx, 'team.name'],
                            hito='tarjeta', detalle=tarj.at[idx]))
        for idx in e.index[tipo == 'Substitution']:
            hit.append(dict(match_id=mid, period=e.at[idx, 'period'], t=e.at[idx, 't'], team=e.at[idx, 'team.name'],
                            hito='cambio', detalle=g('substitution.outcome.name').at[idx]))
        for idx in e.index[tipo == 'Tactical Shift']:
            hit.append(dict(match_id=mid, period=e.at[idx, 'period'], t=e.at[idx, 't'], team=e.at[idx, 'team.name'],
                            hito='ajuste_tactico'))
    except Exception as ex:
        fallos.append((mid, str(ex)[:60]))
    if (i + 1) % 250 == 0:
        print(f'  {i + 1} partidos ({time.time() - t0:.0f} s)')

EST, VEN, HIT = pd.DataFrame(est), pd.concat(ven, ignore_index=True), pd.DataFrame(hit)
EST.to_parquet(f'{P}/estilo_equipo_partido_v1.parquet', index=False)
VEN.to_parquet(f'{P}/ventanas_5min_v1.parquet', index=False)
HIT.to_parquet(f'{P}/hitos_v1.parquet', index=False)

nm = len(ev_files)
print(f'\nFallos: {len(fallos)} {fallos[:5]} | estilo: {len(EST)} filas | ventanas: {len(VEN)} | hitos: {len(HIT)}')
print('\nMétricas de estilo (por equipo-partido):')
print(EST.drop(columns=['match_id', 'team', 'rival']).describe(percentiles=[.1, .5, .9])
      .loc[['count', 'mean', '10%', '50%', '90%']].T.round(3).to_string())
print('\nChequeos de consistencia:')
print(f'  posesión media {EST.posesion_pct.mean():.3f} y field tilt medio {EST.field_tilt.mean():.3f} (ambos deben ser 0.500)')
print(f'  xG en ventanas = xG en estilo: {np.isclose(VEN.xg.sum(), EST.xg_np.sum())}')
print(f'  goles por partido (periodos 1 y 2): {VEN.goles.sum() / nm:.2f}')
if len(HIT):
    G, T_, C_ = HIT[HIT.hito == 'gol'], HIT[HIT.hito == 'tarjeta'], HIT[HIT.hito == 'cambio']
    print(f'  tarjetas por partido: {T_.detalle.value_counts().div(nm).round(3).to_dict()}')
    print(f'  cambios por equipo-partido: {len(C_) / (2 * nm):.2f} | por tipo: {C_.detalle.value_counts().to_dict()}')
    print(f'  ajustes tácticos por equipo-partido: {(HIT.hito == "ajuste_tactico").sum() / (2 * nm):.2f}')
    print(f'  goles con error previo del rival: {int(G.error_previo.sum())} de {len(G)} | autogoles: {int(G.autogol.sum())}')
    print(f'  goles por patrón de juego: {G.detalle.value_counts().to_dict()}')
print(f'\nTiempo total: {time.time() - t0:.0f} s')

In [ ]:
import numpy as np, pandas as pd, time, warnings
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
warnings.filterwarnings('ignore')
t0 = time.time()
rng = np.random.default_rng(2026)
BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026')
P = f'{BASE}/datos_parquet'
ENTRENO = ['2021/2022', '2022/2023', '2023/2024', '2024/2025']
U = 15
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

S = ['posesion_pct', 'field_tilt', 'ppda', 'pres_altura', 'pres_alta_pct', 'contrapresion_pct', 'directness',
     'largo_pct', 'pase_ok_pct', 'saque_meta_corto_pct', 'centros_por_ut', 'xg_por_tiro', 'xg_trans_pct']
SC = ['pase_ok_pct', 'largo_pct', 'directness', 'saque_meta_corto_pct', 'pres_altura', 'ppda', 'xg_por_tiro', 'centros_por_ut']

def ajusta(y, df, extra=None):
    X = pd.get_dummies(df['temporada'], drop_first=True, dtype=float)
    X['home'] = df['home'].astype(float); X['elo_dif'] = df['elo_dif']
    if extra is not None:
        X['extra'] = extra
    X.insert(0, 'c', 1.0)
    ok = (y.notna() & X.notna().all(axis=1)).values
    b, *_ = np.linalg.lstsq(X.values[ok], y.values[ok].astype(float), rcond=None)
    out = pd.Series(np.nan, index=y.index); out[ok] = y.values[ok] - X.values[ok] @ b
    return out

def confiab(df, cols, etapas, reps=50, minimo=6):
    sub = df[df.etapa_id.isin(etapas)]; grupos = [g for _, g in sub.groupby('etapa_id')]; filas = []
    for c in cols:
        rs, usados = [], 0
        for _ in range(reps):
            a, b = [], []
            for g in grupos:
                v = g[c].dropna().values
                if len(v) < minimo:
                    continue
                v = rng.permutation(v); h = len(v) // 2
                a.append(v[:h].mean()); b.append(v[h:2 * h].mean())
            usados = len(a); rs.append(np.corrcoef(a, b)[0, 1])
        r = float(np.nanmean(rs))
        nh = np.median([len(g[c].dropna()) // 2 for g in grupos if len(g[c].dropna()) >= minimo])
        rho1 = r / (nh - (nh - 1) * r) if r > 0 else np.nan
        n07 = 0.7 * (1 - rho1) / (rho1 * 0.3) if r > 0 else np.inf
        filas.append(dict(medida=c, etapas=usados, casos_por_mitad=int(nh), r_mitades=round(r, 2),
                          confiab_etapa=round(2 * r / (1 + r), 2) if r > 0 else round(r, 2),
                          casos_para_0_7=round(n07) if np.isfinite(n07) else 'nunca'))
    return pd.DataFrame(filas)

# ===== 0. TABLA BASE =====
ptp = pd.read_parquet(f'{P}/partido_tecnico_periodo.parquet')
ptp = ptp[ptp.temporada.isin(ENTRENO)].copy()
ptp['fecha'] = pd.to_datetime(ptp['fecha'])
D = ptp.merge(ptp[['match_id', 'team']].rename(columns={'team': 'rival'}), on='match_id').query('team != rival')
D = D.merge(pd.read_parquet(f'{P}/equipo_partido_marcador.parquet')[['match_id', 'team', 'gf', 'gc']], on=['match_id', 'team'])
D = D.merge(pd.read_parquet(f'{P}/estilo_equipo_partido_v1.parquet').drop(columns=['rival']), on=['match_id', 'team'], how='left')
D['ppda'] = np.log(D['ppda'])
D['home'] = (D['lado'] == 'home').astype(int)
D['etapa_id'] = D['team'].astype(str) + '_' + D['etapa'].astype(str)
print(f'== 0. Tabla base: {len(D)} filas (esperado 2,728) | sin métricas de estilo: {int(D[S[0]].isna().sum())}')

# ===== 1. ELO =====
M = D[D.home == 1][['match_id', 'fecha', 'temporada', 'team', 'rival', 'gf', 'gc']].rename(
    columns={'team': 'local', 'rival': 'visita', 'gf': 'gl', 'gc': 'gv'}).sort_values(['fecha', 'match_id']).reset_index(drop=True)
M['y'] = np.select([M.gl > M.gv, M.gl == M.gv], [0, 1], 2)

def elo(M, K, H, REV=0.25, base=1500.0):
    r, tprev, pl, pv = {}, None, [], []
    for row in M.itertuples():
        if row.temporada != tprev:
            r = {k: base + (v - base) * (1 - REV) for k, v in r.items()}; tprev = row.temporada
        rl, rv = r.get(row.local, base), r.get(row.visita, base)
        pl.append(rl); pv.append(rv)
        esp = 1 / (1 + 10 ** (-(rl + H - rv) / 400))
        s = 1.0 if row.gl > row.gv else 0.5 if row.gl == row.gv else 0.0
        d = K * (np.log(abs(row.gl - row.gv) + 1) + 1) * (s - esp)
        r[row.local], r[row.visita] = rl + d, rv - d
    return np.array(pl), np.array(pv)

ll = lambda y, p: -np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1))
brier = lambda y, p: ((p - np.eye(3)[y]) ** 2).sum(axis=1)
TR = (M.fecha >= '2021-10-01') & M.temporada.isin(ENTRENO[:3])
TE = M.temporada == ENTRENO[3]
mejor = None
for K in [10, 20, 30, 40]:
    for H in [0, 50, 100]:
        pl, pv = elo(M, K, H); X = ((pl - pv) / 100).reshape(-1, 1)
        m = LogisticRegression(C=10, max_iter=1000).fit(X[TR], M.y[TR])
        v = ll(M.y[TR].values, m.predict_proba(X[TR])).mean()
        if mejor is None or v < mejor[0]:
            mejor = (v, K, H)
_, K, H = mejor
M['elo_l'], M['elo_v'] = elo(M, K, H)
print(f'\n== 1. ELO elegido: K={K}, ventaja local={H}, regresión entre temporadas 25% | log-loss entrenamiento {mejor[0]:.4f}')
el = pd.concat([M[['match_id', 'local', 'elo_l', 'elo_v']].set_axis(['match_id', 'team', 'elo', 'elo_riv'], axis=1),
                M[['match_id', 'visita', 'elo_v', 'elo_l']].set_axis(['match_id', 'team', 'elo', 'elo_riv'], axis=1)])
D = D.merge(el, on=['match_id', 'team']).sort_values(['fecha', 'match_id']).reset_index(drop=True)
D['elo_dif'] = (D['elo'] - D['elo_riv']) / 100
D[['match_id', 'team', 'rival', 'temporada', 'fecha', 'home', 'etapa_id', 'tec_p', 'rol', 'gf', 'gc', 'elo', 'elo_riv',
   'elo_dif']].to_parquet(f'{P}/equipo_partido_elo_v1.parquet', index=False)
n_et = D.groupby('etapa_id').size()
int_et = D.groupby('etapa_id')['rol'].agg(lambda s: (s == 'interino').any())
ETAPAS = n_et[(n_et >= U) & (~int_et)].index
print(f'Etapas analizables (no interinas, {U}+ partidos): {len(ETAPAS)}')

# ===== 2. HUELLA AJUSTADA (cara propia) Y RELOJ =====
R = D[['match_id', 'team', 'etapa_id']].copy()
for m_ in S:
    g = D.groupby(['rival', 'temporada'])[m_]
    concede = (g.transform('sum') - D[m_]) / (g.transform('count') - 1)
    R[m_] = ajusta(D[m_], D, concede)
CA = confiab(R, S, ETAPAS)
print(f'\n== 2. CONFIABILIDAD Y RELOJ: huella ajustada por rival, localía, Elo y temporada (casos = partidos)')
print(CA.to_string(index=False))

# ===== 3. PORTABILIDAD: ¿técnico o club? =====
PERF = R[R.etapa_id.isin(ETAPAS)].groupby('etapa_id')[S].mean()
Z = (PERF - PERF.mean()) / PERF.std()
info = D.groupby('etapa_id').agg(club=('team', 'first'), tecnico=('tec_p', 'first'), tec_id=('tec_id_p', 'first'),
                                 ini=('fecha', 'min'), n=('match_id', 'size'), elo_medio=('elo', 'mean')).loc[ETAPAS]
et = list(ETAPAS); p_tec, p_club, nulos = [], [], []
for i in range(len(et)):
    for j in range(i + 1, len(et)):
        a, b = et[i], et[j]
        mt, mc = info.at[a, 'tec_id'] == info.at[b, 'tec_id'], info.at[a, 'club'] == info.at[b, 'club']
        if mt and not mc: p_tec.append((a, b))
        elif not mt and not mc: nulos.append((a, b))
for _, g in info.sort_values('ini').groupby('club'):
    ids = list(g.index)
    p_club += [(a, b) for a, b in zip(ids[:-1], ids[1:]) if info.at[a, 'tec_id'] != info.at[b, 'tec_id']]
dist = lambda a, b, cols: np.sqrt(np.nanmean((Z.loc[a, cols].values - Z.loc[b, cols].values) ** 2))

def prueba(pares, cols, B=10000):
    obs = np.mean([dist(a, b, cols) for a, b in pares])
    dn = np.array([dist(a, b, cols) for a, b in nulos])
    sims = np.array([rng.choice(dn, len(pares), replace=False).mean() for _ in range(B)])
    return obs, dn.mean(), (sims <= obs).mean()

confiables = CA.loc[CA.confiab_etapa >= 0.5, 'medida'].tolist()
print(f'\n== 3. PORTABILIDAD (distancia: 0 = idénticos; p = probabilidad de que parejas al azar se parezcan igual o más)')
for nombre, cols in [('todas las métricas', S), (f'solo confiables ({len(confiables)})', confiables)]:
    if len(cols) < 2:
        print(f'  [{nombre}] muy pocas métricas'); continue
    ot, mn, pt = prueba(p_tec, cols); oc, _, pc = prueba(p_club, cols)
    print(f'  [{nombre}] mismo técnico, otro club: {ot:.2f} (p={pt:.3f}, {len(p_tec)} pares) | '
          f'mismo club, otro técnico: {oc:.2f} (p={pc:.3f}, {len(p_club)} pares) | al azar: {mn:.2f}')
fil = []
for c in S:
    sim = lambda pares: np.corrcoef([Z.at[x, c] for x, y in pares] + [Z.at[y, c] for x, y in pares],
                                    [Z.at[y, c] for x, y in pares] + [Z.at[x, c] for x, y in pares])[0, 1]
    fil.append(dict(medida=c, viaja_con_tecnico=round(sim(p_tec), 2), se_queda_en_club=round(sim(p_club), 2)))
print('Por métrica (correlación entre las dos etapas de cada pareja):'); print(pd.DataFrame(fil).to_string(index=False))

# ===== 4. IMPUESTO AL RIVAL =====
for c in SC:
    g = D.groupby(['team', 'temporada'])[c]
    D[c + '_norma'] = (g.transform('sum') - D[c]) / (g.transform('count') - 1)
RV = D[['match_id', 'team'] + SC + [c + '_norma' for c in SC]].rename(
    columns={'team': 'rival', **{c: c + '_r' for c in SC}, **{c + '_norma': c + '_rn' for c in SC}})
DI = D[['match_id', 'team', 'rival', 'temporada', 'home', 'elo_dif', 'etapa_id']].merge(RV, on=['match_id', 'rival'])
IMP = DI[['match_id', 'team', 'etapa_id']].copy()
for c in SC:
    IMP[c] = ajusta(DI[c + '_r'] - DI[c + '_rn'], DI)
CI = confiab(IMP, SC, ETAPAS)
PI = IMP[IMP.etapa_id.isin(ETAPAS)].groupby('etapa_id')[SC].mean()
CI['corr_fuerza'] = [round(PI[c].corr(info.loc[PI.index, 'elo_medio']), 2) for c in SC]
CI['corr_cara_propia'] = [round(PI[c].corr(PERF.loc[PI.index, c]), 2) for c in SC]
print('\n== 4. IMPUESTO AL RIVAL (desviación del rival respecto a su propia norma, ajustada por Elo, localía y temporada)')
print(CI.to_string(index=False))

# ===== 5. CHOQUE DE ESTILOS =====
SE = ['posesion_pct', 'field_tilt', 'ppda', 'pres_altura', 'contrapresion_pct', 'directness', 'largo_pct',
      'pase_ok_pct', 'saque_meta_corto_pct', 'xg_trans_pct']
D = D.sort_values(['team', 'fecha'])
prev = lambda c: D.groupby('team')[c].transform(lambda s: s.shift(1).rolling(10, min_periods=5).mean())
for c in SE:
    D[c + '_p'] = prev(c)
D['xgd_p'] = prev('xg_np') - prev('xg_np_contra')
F = [c + '_p' for c in SE] + ['xgd_p']
L_ = D[D.home == 1][['match_id'] + F].rename(columns={c: c + 'l' for c in F})
V_ = D[D.home == 0][['match_id'] + F].rename(columns={c: c + 'v' for c in F})
MM = M.merge(L_, on='match_id').merge(V_, on='match_id')
MM['elo_d'] = (MM.elo_l - MM.elo_v) / 100
MM['xgd_d'] = MM.xgd_pl - MM.xgd_pv
for c in SE:
    MM[c + '_d'] = MM[c + '_pl'] - MM[c + '_pv']
MM['i_presion_vs_salida'] = MM.pres_altura_pl * MM.saque_meta_corto_pct_pv - MM.pres_altura_pv * MM.saque_meta_corto_pct_pl
MM['i_directo_vs_presion'] = MM.directness_pl * MM.pres_altura_pv - MM.directness_pv * MM.pres_altura_pl
MM['i_transicion_vs_posesion'] = MM.xg_trans_pct_pl * MM.posesion_pct_pv - MM.xg_trans_pct_pv * MM.posesion_pct_pl
est_d = [c + '_d' for c in SE]
modelos = {'M0 Elo': ['elo_d'], 'M0x Elo + xG': ['elo_d', 'xgd_d'], 'M1 + estilo': ['elo_d', 'xgd_d'] + est_d,
           'M2 + choques': ['elo_d', 'xgd_d'] + est_d + ['i_presion_vs_salida', 'i_directo_vs_presion', 'i_transicion_vs_posesion']}
todas = sorted(set(sum(modelos.values(), [])))
ok = MM[todas].notna().all(axis=1)
tr = ok & MM.match_id.isin(M.loc[TR, 'match_id']); te = ok & MM.match_id.isin(M.loc[TE, 'match_id'])
ytr, yte = MM.y[tr].values, MM.y[te].values
pb = np.tile(np.bincount(ytr, minlength=3) / len(ytr), (len(yte), 1))
res = {'Frecuencias de la liga': (ll(yte, pb), brier(yte, pb))}
for nom, cols in modelos.items():
    mdl = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000)).fit(MM.loc[tr, cols], ytr)
    p = mdl.predict_proba(MM.loc[te, cols]); res[nom] = (ll(yte, p), brier(yte, p))
ref = res['M0x Elo + xG'][0]; fil = []
for nom, (l_, b_) in res.items():
    d = l_ - ref
    bs = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(2000)]
    fil.append(dict(modelo=nom, log_loss=round(l_.mean(), 4), brier=round(b_.mean(), 4), dif_vs_M0x=round(d.mean(), 4),
                    ic95=f'[{np.percentile(bs, 2.5):.4f}, {np.percentile(bs, 97.5):.4f}]'))
print(f'\n== 5. CHOQUE DE ESTILOS: entrenamiento {int(tr.sum())} partidos (2021/22-2023/24), prueba {int(te.sum())} (2024/25)')
print(pd.DataFrame(fil).to_string(index=False))

# ===== 6. VISTA PREVIA: técnicos de América (z = desviaciones estándar respecto al promedio de etapas) =====
sel = info[info.tecnico.str.contains('Solari|Ortiz|Jardine|Almada', regex=True)].sort_values('ini')
vista = Z.loc[sel.index].T.round(1)
vista.columns = [f'{t.split()[-1] if len(t.split()) < 3 else t.split()[-2]}-{c[:8]} ({n})'
                 for t, c, n in zip(sel.tecnico, sel.club, sel.n)]
print('\n== 6. VISTA PREVIA (no interpretar antes de ver confiabilidad):'); print(vista.to_string())

Z.to_parquet(f'{P}/perfiles_etapa_v1.parquet'); PI.to_parquet(f'{P}/impuesto_etapa_v1.parquet')
print(f'\nTiempo total: {time.time() - t0:.0f} s')

In [ ]:
import numpy as np, pandas as pd, time
t0 = time.time(); rng = np.random.default_rng(2026)
BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026'); P = f'{BASE}/datos_parquet'; U = 15
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
OFF = 100000
key = lambda per, t: (per - 1) * OFF + t

def exceso(y, df, cols):
    X = pd.get_dummies(df[cols].astype(str), drop_first=True, dtype=float); X.insert(0, 'c', 1.0)
    ok = y.notna().values
    b, *_ = np.linalg.lstsq(X.values[ok], y.values[ok].astype(float), rcond=None)
    out = pd.Series(np.nan, index=y.index); out[ok] = y.values[ok] - X.values[ok] @ b
    return out

def confiab(df, cols, etapas, reps=50, minimo=6):
    sub = df[df.etapa_id.isin(etapas)]; grupos = [g for _, g in sub.groupby('etapa_id')]; filas = []
    for c in cols:
        rs, usados = [], 0
        for _ in range(reps):
            a, b = [], []
            for g in grupos:
                v = g[c].dropna().values
                if len(v) < minimo:
                    continue
                v = rng.permutation(v); h = len(v) // 2
                a.append(v[:h].mean()); b.append(v[h:2 * h].mean())
            usados = len(a); rs.append(np.corrcoef(a, b)[0, 1])
        r = float(np.nanmean(rs))
        nh = np.median([len(g[c].dropna()) // 2 for g in grupos if len(g[c].dropna()) >= minimo])
        rho1 = r / (nh - (nh - 1) * r) if r > 0 else np.nan
        n07 = 0.7 * (1 - rho1) / (rho1 * 0.3) if r > 0 else np.inf
        filas.append(dict(medida=c, etapas=usados, casos_por_mitad=int(nh), r_mitades=round(r, 2),
                          confiab_etapa=round(2 * r / (1 + r), 2) if r > 0 else round(r, 2),
                          casos_para_0_7=round(n07) if np.isfinite(n07) else 'nunca'))
    return pd.DataFrame(filas)

VEN = pd.read_parquet(f'{P}/ventanas_5min_v1.parquet')
HIT = pd.read_parquet(f'{P}/hitos_v1.parquet')
D = pd.read_parquet(f'{P}/equipo_partido_elo_v1.parquet')
D = D[D.match_id.isin(set(VEN.match_id))].copy()
D['elo_t'] = pd.qcut(D.elo_dif, 3, labels=['inferior', 'parejo', 'superior']).astype(str)
n_et = D.groupby('etapa_id').size()
int_et = D.groupby('etapa_id')['rol'].agg(lambda s: (s == 'interino').any())
ETAPAS = n_et[(n_et >= U) & (~int_et)].index

# ===== 1. REJILLA DE VENTANAS CON MARCADOR =====
win = pd.DataFrame([(p, v) for p in (1, 2) for v in range(10)], columns=['period', 'ventana'])
G = D[['match_id', 'team', 'rival', 'home', 'elo_t', 'etapa_id']].merge(win, how='cross')
num = ['pases', 'pases_ut', 'xg', 'tiros', 'goles', 'presiones', 'obv']
G = G.merge(VEN, on=['match_id', 'period', 'ventana', 'team'], how='left')
G = G.merge(VEN.rename(columns={'team': 'rival', **{c: c + '_r' for c in num}}), on=['match_id', 'period', 'ventana', 'rival'], how='left')
G[num + [c + '_r' for c in num]] = G[num + [c + '_r' for c in num]].fillna(0)
G['t0'] = np.where(G.period == 1, G.ventana * 300, 2700 + G.ventana * 300)
G['key0'] = key(G.period, G.t0)
G['minuto'] = pd.cut(G.t0 / 60, [-1, 30, 60, 200], labels=['0-30', '30-60', '60+']).astype(str)
G = G.sort_values(['match_id', 'team', 'period', 'ventana']).reset_index(drop=True)
GL = HIT[HIT.hito == 'gol'].copy(); GL['key'] = key(GL.period, GL.t)
gk = {k: np.sort(v.values) for k, v in GL.groupby(['match_id', 'team'])['key']}
n_antes = lambda mid, tm, keys: np.searchsorted(gk[(mid, tm)], keys, side='left') if (mid, tm) in gk else np.zeros(len(keys))
gf_a, gc_a = np.zeros(len(G)), np.zeros(len(G))
for (mid, tm, rv), idx in G.groupby(['match_id', 'team', 'rival']).indices.items():
    kk = G['key0'].values[idx]; gf_a[idx] = n_antes(mid, tm, kk); gc_a[idx] = n_antes(mid, rv, kk)
G['dif'] = gf_a - gc_a
G['estado'] = np.sign(G['dif']).map({-1: 'perdiendo', 0: 'empate', 1: 'ganando'})
den = G.pases_ut + G.pases_ut_r
G['tilt'] = G.pases_ut / den.replace(0, np.nan)
print(f'== 1. Rejilla: {len(G)} ventanas equipo (esperado {len(D) * 20}) | etapas analizables: {len(ETAPAS)}')
print('   Reparto de ventanas por estado al inicio:', G.estado.value_counts(normalize=True).round(3).to_dict())

# ===== 2. RESPUESTA AL GOLPE =====
CON = GL[((GL.period == 1) & (GL.t <= 2100)) | ((GL.period == 2) & (GL.t <= 4800))]
Gi = G.set_index(['match_id', 'team', 'period', 'ventana'])
def tomar(mid, tm, per, ws, c, f):
    vals = [Gi.at[(mid, tm, per, w), c] for w in ws if (mid, tm, per, w) in Gi.index]
    return f(vals) if len(vals) else np.nan
fil = []
for r in CON.itertuples():
    C_, A_ = r.rival, r.team
    arr = gk.get((r.match_id, A_), np.array([]))
    tr = r.t if r.period == 1 else r.t - 2700
    v = min(9, int(tr // 300))
    post = [w for w in (v + 1, v + 2) if w <= 9]; pre = [w for w in (v - 3, v - 2, v - 1) if w >= 0]
    t_antes = tomar(r.match_id, C_, r.period, pre, 'tilt', np.nanmean)
    gfC = n_antes(r.match_id, C_, np.array([r.key + 1]))[0]; gcC = n_antes(r.match_id, A_, np.array([r.key + 1]))[0]
    inesperado = bool(r.error_previo) or ((not r.autogol) and r.xg < 0.05) or r.detalle == 'From Counter' \
                 or (t_antes == t_antes and t_antes >= 0.6)
    fil.append(dict(match_id=r.match_id, team=C_, otro_10=int(((arr > r.key) & (arr <= r.key + 600)).any()),
                    xg_contra_10=tomar(r.match_id, C_, r.period, post, 'xg_r', np.sum),
                    tilt_10=tomar(r.match_id, C_, r.period, post, 'tilt', np.nanmean),
                    tilt_antes=t_antes, dif=int(np.clip(gfC - gcC, -2, 1)), inesperado=inesperado,
                    minuto=('0-30' if r.t < 1800 else '30-60' if r.t < 3600 else '60+')))
SH = pd.DataFrame(fil).merge(D[['match_id', 'team', 'home', 'elo_t', 'etapa_id']], on=['match_id', 'team'])
SH['tilt_antes_t'] = pd.qcut(SH.tilt_antes, 3, labels=False).astype(str)
ctrl = ['dif', 'minuto', 'home', 'elo_t', 'tilt_antes_t']
for c in ['otro_10', 'xg_contra_10', 'tilt_10']:
    SH['exc_' + c] = exceso(SH[c].astype(float), SH, ctrl)
print(f'\n== 2. RESPUESTA AL GOLPE: {len(SH)} goles recibidos con al menos 10 min por delante | inesperados: {int(SH.inesperado.sum())}')
print('   Promedios brutos (liga) según tipo de golpe:')
print(SH.groupby('inesperado')[['otro_10', 'xg_contra_10', 'tilt_10']].mean().round(3).to_string())
cnt = SH[SH.etapa_id.isin(ETAPAS)].groupby('etapa_id').agg(golpes=('otro_10', 'size'), inesperados=('inesperado', 'sum'))
print(f'   Por etapa: golpes mediana {cnt.golpes.median():.0f} (mín {cnt.golpes.min()}), inesperados mediana {cnt.inesperados.median():.0f}')
print(confiab(SH, ['exc_otro_10', 'exc_xg_contra_10', 'exc_tilt_10'], ETAPAS).to_string(index=False))

# ===== 3. DOMINIO DEL RIVAL Y LATENCIA DE REACCIÓN =====
gb = lambda: G.groupby(['match_id', 'team', 'period'], sort=False)
ut2 = G.pases_ut + gb().pases_ut.shift(1); ut2r = G.pases_ut_r + gb().pases_ut_r.shift(1)
G['dom_r'] = ((ut2r / (ut2 + ut2r).replace(0, np.nan)) >= 0.7) & ((ut2 + ut2r) >= 8)
prev = gb()['dom_r'].shift(1)
G['inicio'] = G.dom_r & ~prev.fillna(False).astype(bool)
G['run'] = (G.dom_r.ne(prev) | prev.isna()).cumsum()
G['dur'] = G.groupby('run')['dom_r'].transform('size') * 5
EP = G[G.inicio].copy()
CH = HIT[((HIT.hito == 'cambio') & (HIT.detalle == 'Tactical')) | (HIT.hito == 'ajuste_tactico')].copy()
CH['key'] = key(CH.period, CH.t)
ck = {k: np.sort(v.values) for k, v in CH.groupby(['match_id', 'team'])['key']}
lat = []
for r in EP.itertuples():
    arr = ck.get((r.match_id, r.team), np.array([])); ini = r.key0 - 300
    c = arr[(arr >= ini) & (arr < r.period * OFF)]
    lat.append((c[0] - ini) / 60 if len(c) else np.nan)
EP['latencia'] = lat
EP['reac_10'] = (EP.latencia <= 10).astype(float)
ctrl2 = ['estado', 'minuto', 'home', 'elo_t']
EP['exc_dur'] = exceso(EP.dur.astype(float), EP, ctrl2)
EP['exc_reac_10'] = exceso(EP.reac_10, EP, ctrl2)
ce = EP[EP.etapa_id.isin(ETAPAS)].groupby('etapa_id').size()
print(f'\n== 3. DOMINIO DEL RIVAL (70%+ del último tercio en 10 min): {len(EP)} episodios | por etapa mediana {ce.median():.0f} (mín {ce.min()})')
print(f'   Duración mediana {EP.dur.median():.0f} min | reaccionó en 10 min o menos: {EP.reac_10.mean():.1%} | '
      f'latencia mediana (si reaccionó): {EP.latencia.median():.1f} min')
print(confiab(EP, ['exc_dur', 'exc_reac_10'], ETAPAS).to_string(index=False))

# ===== 4. CAMBIOS FORZADOS (nivel liga) =====
INJ = HIT[(HIT.hito == 'cambio') & (HIT.detalle == 'Injury')]
RED = HIT[(HIT.hito == 'tarjeta') & HIT.detalle.isin(['Red Card', 'Second Yellow'])]
dt = []
for r in RED.itertuples():
    tr = r.t if r.period == 1 else r.t - 2700; v = min(9, int(tr // 300))
    a = tomar(r.match_id, r.team, r.period, [w for w in (v - 3, v - 2, v - 1) if w >= 0], 'tilt', np.nanmean)
    b = tomar(r.match_id, r.team, r.period, [w for w in (v + 1, v + 2, v + 3) if w <= 9], 'tilt', np.nanmean)
    dt.append(b - a)
dt = np.array(dt, dtype=float)
por_et = lambda H_: H_.merge(D[['match_id', 'team', 'etapa_id']], on=['match_id', 'team']).query('etapa_id in @ETAPAS').groupby('etapa_id').size()
print(f'\n== 4. CAMBIOS FORZADOS: lesiones {len(INJ)} (por etapa mediana {por_et(INJ).median():.0f}) | '
      f'expulsiones {len(RED)} (por etapa mediana {por_et(RED).median():.0f})')
print(f'   Cambio medio en field tilt del expulsado (15 min después contra 15 min antes): {np.nanmean(dt):+.3f} (n={np.isfinite(dt).sum()})')

# ===== 5. AJUSTE VOLUNTARIO: perdiendo o empatando al 55' =====
w55 = G[(G.period == 2) & (G.ventana == 2)][['match_id', 'team', 'etapa_id', 'estado']].copy()
TS = HIT[(HIT.hito == 'cambio') & (HIT.detalle == 'Tactical') & (HIT.period == 2) & (HIT.t >= 3300) & (HIT.t <= 4500)]
cc = set(zip(TS.match_id, TS.team))
w55['cambio_55_75'] = [(m, t) in cc for m, t in zip(w55.match_id, w55.team)]
tab = w55[w55.etapa_id.isin(ETAPAS)].groupby(['etapa_id', 'estado']).cambio_55_75.agg(['size', 'mean']).reset_index()
print("\n== 5. CAMBIO TÁCTICO ENTRE 55' Y 75' según marcador al 55' (mediana por etapa)")
print(tab.groupby('estado')[['size', 'mean']].median().round(2).rename(columns={'size': 'partidos', 'mean': 'pct_con_cambio'}).to_string())

SH.to_parquet(f'{P}/golpes_v1.parquet', index=False); EP.to_parquet(f'{P}/dominio_episodios_v1.parquet', index=False)
print(f'\nTiempo total: {time.time() - t0:.0f} s')

# Sesión 3 (definitions y diagnosticos)

In [ ]:
# ===== D1. Diagnóstico del 360 de la etapa 2 (4 peticiones, no descarga en masa) =====
# Requiere haber corrido las celdas 1 (BASE) y 2 (secretos).
import glob, requests, pandas as pd
from google.colab import userdata
AUTH = (userdata.get('SB_USER'), userdata.get('SB_PASS'))

URL_360     = "https://data.statsbomb.com/api/v2/360-frames/{match_id}"
URL_EVENTOS = "https://data.statsbomb.com/api/v11/events/{match_id}"
URL_COMP    = "https://data.statsbomb.com/api/v4/competitions"
# 1) Qué dice el archivo de fallos
f = glob.glob(f"{BASE}/**/fallos_descarga_2026-10-06.csv", recursive=True)
print("1) Archivo de fallos:", f[0] if f else "NO ENCONTRADO")
if f:
    fa = pd.read_csv(f[0])
    print("   filas:", len(fa), "| columnas:", list(fa.columns))
    for c in fa.columns:
        if fa[c].nunique() <= 10:
            print("  ", c, fa[c].value_counts(dropna=False).to_dict())
    print(fa.head(2).to_string())

# 2) Estado 360 según la tabla de partidos del 2026-10-02
m = pd.read_parquet(glob.glob(f"{BASE}/**/matches_73_2026-10-02.parquet", recursive=True)[0])
cs   = [c for c in m.columns if c.endswith('season_id')][0]
c360 = [c for c in m.columns if 'match_status_360' in c][0]
print("\n2) match_status_360 por temporada (tabla del 2026-10-02):")
print(pd.crosstab(m[cs], m[c360].fillna('nulo')))
mid_351 = int(m.loc[(m[cs] == 351) & (m[c360] == 'available'), 'match_id'].iloc[0]) \
          if ((m[cs] == 351) & (m[c360] == 'available')).any() else None

# 3) Prueba en vivo: el mismo URL con un partido de cada etapa
def prueba(nombre, url, mid):
    if mid is None:
        print(f"   {nombre:24s} sin partido disponible"); return
    try:
        r = requests.get(url.format(match_id=mid), auth=AUTH, timeout=90)
        n = len(r.json()) if r.status_code == 200 else '-'
        print(f"   {nombre:24s} {mid}  HTTP {r.status_code}  elementos={n}")
    except Exception as e:
        print(f"   {nombre:24s} {mid}  ERROR {type(e).__name__}")
print("\n3) Prueba en vivo")
prueba("360 etapa 1 (2021/22)", URL_360, 3799476)
prueba("360 etapa 2 (2025/26)", URL_360, 4004046)   # este partido SÍ tenía 360 el 2026-10-02
prueba("360 2026/27",           URL_360, mid_351)
prueba("eventos 2025/26",       URL_EVENTOS, 4004046)

# 4) Competitions hoy (opcional)
if URL_COMP.startswith('http'):
    c = pd.json_normalize(requests.get(URL_COMP, auth=AUTH, timeout=90).json())
    c = c[c['competition_id'] == 73][['season_id', 'season_name', 'match_updated', 'match_updated_360',
                                      'match_available', 'match_available_360']]
    print("\n4) Competitions hoy (Liga MX):"); print(c.sort_values('season_id').to_string(index=False))

# 5) Calendario de América en 2026/27 alrededor del corte
hn = [c for c in m.columns if c.endswith('home_team_name')][0]
an = [c for c in m.columns if c.endswith('away_team_name')][0]
am = m[(m[cs] == 351) & (m[hn].str.contains('América|America') | m[an].str.contains('America|América'))]
print("\n5) América 2026/27 desde el 2026-09-20:")
print(am.loc[am['match_date'] >= '2026-09-20', ['match_date', hn, an, 'home_score', 'away_score']]
        .sort_values('match_date').to_string(index=False))
print("Partidos de América jugados al 2026-10-02:", int(am['home_score'].notna().sum()))


In [ ]:
# ===== D2. Evidencia para los objetos base (solo temporadas de entrenamiento, muestra de 200 partidos) =====
import os, random, numpy as np, pandas as pd, pyarrow.parquet as pq

ptp = pd.read_parquet(f"{BASE}/datos_parquet/partido_tecnico_periodo.parquet")
temps = sorted(ptp['temporada'].dropna().unique())
TRAIN = temps[:4]
print("Temporadas en la tabla:", temps, "| entrenamiento:", TRAIN)
ids = sorted(ptp.loc[ptp['temporada'].isin(TRAIN), 'match_id'].unique())
random.seed(2026)
muestra = random.sample(ids, min(200, len(ids)))

COLS = ['index','period','timestamp','duration','type.name','team.id','possession','possession_team.id',
        'play_pattern.name','loc_x','set_piece_phase','control_degree','is_controlled_possession',
        'counterpress','shot.statsbomb_xg','shot.type.name']
dfs = []
for mid in muestra:
    p = f"{BASE}/datos_parquet/liga/eventos/{mid}.parquet"
    if not os.path.exists(p):
        continue
    cols = [c for c in COLS if c in pq.read_schema(p).names]
    d = pd.read_parquet(p, columns=cols); d['match_id'] = mid
    dfs.append(d)
ev = pd.concat(dfs, ignore_index=True)
for c in COLS:
    if c not in ev.columns: ev[c] = np.nan
ev = ev[ev['period'].isin([1, 2])].sort_values(['match_id', 'index']).reset_index(drop=True)
ev['t'] = pd.to_timedelta(ev['timestamp']).dt.total_seconds()
ev['ctrl_pos'] = ev['is_controlled_possession'].eq(True)
print(f"Partidos leídos: {ev['match_id'].nunique()} | eventos: {len(ev):,}")

def q(s, ps=(.25, .5, .75, .9)):
    s = s.dropna()
    return {f"p{int(p*100)}": round(float(s.quantile(p)), 1) for p in ps} | {"n": int(len(s))}

K = ['match_id', 'possession']
pt = ev[ev['team.id'] == ev['possession_team.id']]   # eventos del equipo en posesión
pos = ev.groupby(K).agg(t0=('t', 'first'), tf=('t', 'last'), patron=('play_pattern.name', 'first'),
                        controlada=('ctrl_pos', 'max'))
pos['n_propios'] = pt.groupby(K).size()
pos['n_propios'] = pos['n_propios'].fillna(0)
pos['t_sp']   = ev[ev['set_piece_phase'].notna()].groupby(K)['t'].max()
pos['t_sp1']  = ev[ev['set_piece_phase'] == 1].groupby(K)['t'].max()
pos['t_ctrl'] = ev[ev['control_degree'] >= 0.999].groupby(K)['t'].min()
pos['t80']    = pt[pt['loc_x'] >= 80].groupby(K)['t'].min()
pos['tsh']    = pt[pt['type.name'] == 'Shot'].groupby(K)['t'].min()
for c, n in [('t_sp','s_sp'), ('t_sp1','s_sp1'), ('t_ctrl','s_ctrl'), ('t80','s_80'), ('tsh','s_sh')]:
    pos[n] = pos[c] - pos['t0']

print("\n[A] BALÓN PARADO Y JUEGO ABIERTO")
print("Valores de set_piece_phase (eventos):", ev['set_piece_phase'].value_counts(dropna=False).to_dict())
print(pd.crosstab(np.where(ev['set_piece_phase'].notna(), 'con set_piece_phase', 'sin set_piece_phase'),
                  np.where(ev['control_degree'].notna(), 'con control_degree', 'sin control_degree')))
tab = pos.groupby('patron').agg(posesiones=('t0', 'size'),
                                pct_con_sp=('t_sp', lambda s: round(100 * s.notna().mean(), 1)),
                                seg_fase1_p50=('s_sp1', 'median'), seg_fase1_p90=('s_sp1', lambda s: s.quantile(.9)),
                                seg_sp_p50=('s_sp', 'median'), seg_sp_p90=('s_sp', lambda s: s.quantile(.9)))
tab['seg_posesion_p50'] = (pos['tf'] - pos['t0']).groupby(pos['patron']).median()
print(tab.round(1).to_string())
sh = ev[ev['type.name'] == 'Shot']; tot = sh['shot.statsbomb_xg'].sum()
print(f"Tiros: {len(sh):,} | xG total: {tot:.1f}")
print(f"% del xG en eventos con set_piece_phase: {100*sh.loc[sh['set_piece_phase'].notna(),'shot.statsbomb_xg'].sum()/tot:.1f}")
print("% del xG por shot.type:", (100*sh.groupby('shot.type.name')['shot.statsbomb_xg'].sum()/tot).round(1).to_dict())
print("% del xG por play_pattern (el error de la sesión 2):",
      (100*sh.groupby('play_pattern.name')['shot.statsbomb_xg'].sum()/tot).round(1).to_dict())

print("\n[B] TRANSICIÓN (posesiones Regular Play / From Counter)")
reg = pos[pos['patron'].isin(['Regular Play', 'From Counter'])]
print(f"Posesiones: {len(reg):,} | % que nunca llega a control_degree = 1: {100*reg['t_ctrl'].isna().mean():.1f}")
print("Segundos hasta que el juego se asienta (control_degree = 1):", q(reg['s_ctrl']))
for w in [5, 10, 15, 20]:
    print(f"  En ≤{w:>2}s tras recuperar: llega a último tercio {100*(reg['s_80']<=w).mean():5.1f}% | tira {100*(reg['s_sh']<=w).mean():4.1f}%")
print(f"  En toda la posesión:     llega a último tercio {100*reg['t80'].notna().mean():5.1f}% | tira {100*reg['tsh'].notna().mean():4.1f}%")

print("\n[C] FASE DE PRESIÓN")
pr = ev[ev['type.name'] == 'Pressure'].copy()
pr['fin'] = pr['t'] + pr['duration'].fillna(0)
pr['pausa'] = pr.groupby(['match_id', 'possession', 'team.id'])['t'].shift(-1) - pr['fin']
print(f"Presiones: {len(pr):,} | % marcadas counterpress: {100*(pr['counterpress']==True).mean():.1f}")
print("Pausa (s) entre una presión y la siguiente del mismo equipo, misma posesión:", q(pr['pausa']))
print("  % de pausas ≤1s, ≤2s, ≤3s, ≤5s:", [round(float(100*(pr["pausa"].dropna() <= x).mean()), 1) for x in [1, 2, 3, 5]])
print("Presiones por posesión rival con al menos una:", q(pr.groupby(['match_id', 'possession']).size()))

print("\n[D] POSESIÓN Y SECUENCIA OFENSIVA")
print("Posesiones por partido (ambos equipos):", q(pos.groupby('match_id').size()))
print(f"% posesiones con 0-1 eventos propios (micro): {100*(pos['n_propios']<=1).mean():.1f}")
print(f"% posesiones controladas (is_controlled_possession): {100*(pos['controlada']==True).mean():.1f}")
print(f"% posesiones que llegan al último tercio: {100*pos['t80'].notna().mean():.1f} | terminan con tiro: {100*pos['tsh'].notna().mean():.1f}")
print(f"% de los tiros que salen de posesiones que pisaron el último tercio: "
      f"{100*pos.loc[pos['tsh'].notna(),'t80'].notna().mean():.1f}")

Download fase 3

In [ ]:
# ===== E3. Descarga: 360 faltante de 2025/26 + etapa 3 (2026/27, corte 2026-10-05) =====
# Requiere celdas 1 (BASE) y 2. Pega las MISMAS URLs de la etapa 1 (las de D1 funcionaron).
import os, gzip, json, time, requests, pandas as pd
from google.colab import userdata
AUTH = (userdata.get('SB_USER'), userdata.get('SB_PASS'))

URL_360     = "https://data.statsbomb.com/api/v2/360-frames/{match_id}"
URL_EVENTOS = "https://data.statsbomb.com/api/v11/events/{match_id}"
URL_MATCHES = "https://data.statsbomb.com/api/v6/competitions/{competition_id}/seasons/{season_id}/matches"
URL_LINEUPS = "https://data.statsbomb.com/api/v5/lineups/{match_id}"

CORTE, HOY = '2026-10-05', '2026-10-07'
CRUDO = f"{BASE}/datos_crudos/liga"

def bajar(url, ruta):
    if os.path.exists(ruta):
        return 'ya existía'
    estado = 'sin intento'
    for _ in range(3):
        try:
            r = requests.get(url, auth=AUTH, timeout=120)
        except Exception as e:
            estado = type(e).__name__; time.sleep(5); continue
        if r.status_code == 200:
            os.makedirs(os.path.dirname(ruta), exist_ok=True)
            with gzip.open(ruta, 'wb') as f:
                f.write(r.content)
            return 'ok'
        estado = f'HTTP {r.status_code}'
        if r.status_code == 404:
            break
        time.sleep(5)
    return estado

def partidos(season_id):
    r = requests.get(URL_MATCHES.format(competition_id=73, season_id=season_id), auth=AUTH, timeout=120)
    r.raise_for_status()
    with gzip.open(f"{BASE}/datos_crudos/matches_73_{season_id}_{HOY}.json.gz", 'wb') as f:
        f.write(r.content)                      # foto congelada de la tabla de partidos
    return pd.json_normalize(r.json())

log = []

# ---- 1) 360 faltante de 2025/26 ----
m318 = partidos(318)
disp318 = set(m318.loc[m318['match_status_360'] == 'available', 'match_id'].astype(int))
fa = pd.read_csv(f"{BASE}/salidas/fallos_descarga_2026-10-06.csv")
ids_360_318 = sorted(set(fa['match_id'].astype(int)) & disp318)
print(f"1) 2025/26: fallos previos {fa['match_id'].nunique()} | con 360 disponible hoy {len(ids_360_318)}")
for i, mid in enumerate(ids_360_318, 1):
    log.append((mid, 'frames360', bajar(URL_360.format(match_id=mid), f"{CRUDO}/frames360/{mid}.json.gz")))
    if i % 50 == 0: print(f"   {i}/{len(ids_360_318)}")

# ---- 2) Etapa 3: 2026/27 hasta el corte ----
m351 = partidos(351)
jug = m351[m351['home_score'].notna()]
e3 = jug[jug['match_date'] <= CORTE].copy()
print(f"\n2) 2026/27: jugados hoy {len(jug)} | hasta el corte {len(e3)} | "
      f"jugados del 2026-10-03 al corte: {int(((e3['match_date'] >= '2026-10-03')).sum())}")
print("   match_status_360 hasta el corte:", e3['match_status_360'].value_counts(dropna=False).to_dict())
es_am = e3['home_team.home_team_name'].str.contains('Am[eé]rica') | e3['away_team.away_team_name'].str.contains('Am[eé]rica')
print(f"   América hasta el corte: {int(es_am.sum())} partidos | sin 360: {int((es_am & (e3['match_status_360'] != 'available')).sum())}")
for i, row in enumerate(e3.itertuples(), 1):
    mid = int(row.match_id)
    log.append((mid, 'lineups', bajar(URL_LINEUPS.format(match_id=mid), f"{CRUDO}/lineups/{mid}.json.gz")))
    log.append((mid, 'eventos', bajar(URL_EVENTOS.format(match_id=mid), f"{CRUDO}/eventos/{mid}.json.gz")))
    if row.match_status_360 == 'available':
        log.append((mid, 'frames360', bajar(URL_360.format(match_id=mid), f"{CRUDO}/frames360/{mid}.json.gz")))
    if i % 25 == 0: print(f"   {i}/{len(e3)}")
e3[['match_id', 'match_date', 'match_status_360']].to_csv(f"{BASE}/salidas/ids_etapa3_corte_{CORTE}.csv", index=False)

# ---- 3) ¿StatsBomb cambió 2022/23 después de nuestra descarga (2026-10-02)? ----
m235 = partidos(235)
for col in ['last_updated', 'last_updated_360']:
    if col in m235.columns:
        n = int((m235[col].astype(str) > '2026-10-02').sum())
        print(f"\n3) 2022/23: partidos con {col} posterior al 2026-10-02: {n}")

# ---- Resumen ----
lg = pd.DataFrame(log, columns=['match_id', 'tipo', 'estado'])
print("\nRESUMEN:"); print(pd.crosstab(lg['tipo'], lg['estado']))
fallos = lg[~lg['estado'].isin(['ok', 'ya existía'])]
fallos.to_csv(f"{BASE}/salidas/fallos_descarga_{HOY}.csv", index=False)
print("Fallos guardados:", len(fallos))

Lista de IDs para conversor Etapa 3

In [ ]:
# ===== IDS. Lista de partidos para la conversión =====
import os, pandas as pd
fa = pd.read_csv(f"{BASE}/salidas/fallos_descarga_2026-10-06.csv")
ids_318 = sorted(int(x) for x in fa['match_id'].unique()
                 if os.path.exists(f"{BASE}/datos_crudos/liga/frames360/{int(x)}.json.gz"))
ids_351 = sorted(pd.read_csv(f"{BASE}/salidas/ids_etapa3_corte_2026-10-05.csv")['match_id'].astype(int))
IDS_NUEVOS = ids_318 + ids_351
print(f"2025/26 con 360 nuevo: {len(ids_318)} | 2026/27 hasta el corte: {len(ids_351)} | total: {len(IDS_NUEVOS)}")

Conversor etapa 3

In [ ]:
import gzip, json, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

TEMPORADAS = [318, 351]
MAX_PARTIDOS = None
TIEMPO_MAX_MIN = 45

RAW = BASE / 'datos_crudos' / 'liga'
OUT = BASE / 'datos_parquet' / 'liga'
for c in ['eventos', 'frames360', 'jugadores360', 'lineups']:
    (OUT / c).mkdir(parents=True, exist_ok=True)
LOG = BASE / 'salidas' / 'validacion_etapa1.jsonl'

archivo = sorted((BASE / 'datos_parquet').glob('matches_73_*.parquet'))[-1]
matches = pd.read_parquet(archivo)
obj = matches[matches['home_score'].notna() & matches['season_id'].isin(TEMPORADAS)]
obj = obj.sort_values(['season_id', 'match_date'])
partidos = obj.to_dict('records')


def leer(tipo, mid):
    with gzip.open(RAW / tipo / f'{mid}.json.gz', 'rt', encoding='utf-8') as f:
        return json.load(f)


def a_texto(v):
    if v is None or (isinstance(v, float) and v != v):
        return None
    return json.dumps(v, ensure_ascii=False)


def anidado_a_texto(df):
    for c in df.columns:
        if pd.api.types.is_object_dtype(df[c]):
            if df[c].apply(lambda v: isinstance(v, (list, dict))).any():
                df[c] = df[c].apply(a_texto)
    return df


def separar_coordenadas(df, col, pref):
    if col not in df.columns:
        return
    for i, eje in enumerate(['x', 'y', 'z']):
        vals = df[col].apply(lambda a: a[i] if isinstance(a, list) and len(a) > i else np.nan)
        if eje == 'z' and vals.isna().all():
            continue
        df[f'{pref}_{eje}'] = vals.astype('float32')
    df.drop(columns=[col], inplace=True)


def aplanar_eventos(raw, mid):
    df = pd.json_normalize(raw)
    df.insert(0, 'match_id', mid)
    for col, pref in [('location', 'loc'), ('pass.end_location', 'pass_end'),
                      ('carry.end_location', 'carry_end'), ('shot.end_location', 'shot_end')]:
        separar_coordenadas(df, col, pref)
    return anidado_a_texto(df)


def guardar(df, ruta):
    try:
        df.to_parquet(ruta, index=False, compression='zstd')
        return 'ok'
    except Exception:
        d = df.copy()
        for c in d.columns:
            if pd.api.types.is_object_dtype(d[c]):
                d[c] = d[c].apply(lambda v: None if v is None or (isinstance(v, float) and v != v) else str(v))
        d.to_parquet(ruta, index=False, compression='zstd')
        return 'respaldo_texto'


def convertir_360(raw, mapa, mid):
    fr, jg = [], []
    for f in raw:
        uuid = f.get('event_uuid')
        idx = mapa.get(uuid, -1)
        ff = f.get('freeze_frame') or []
        va = f.get('visible_area') or []
        mates = sum(1 for p in ff if p.get('teammate'))
        fr.append({
            'match_id': mid, 'event_index': idx, 'event_uuid': uuid,
            'line_breaking_pass': f.get('line_breaking_pass'),
            'num_defenders_on_goal_side_of_actor': f.get('num_defenders_on_goal_side_of_actor'),
            'distance_to_nearest_defender': f.get('distance_to_nearest_defender'),
            'ball_receipt_in_space': f.get('ball_receipt_in_space'),
            'ball_receipt_exceeds_distance': f.get('ball_receipt_exceeds_distance'),
            'n_visibles': len(ff), 'n_visibles_companeros': mates,
            'n_visibles_rivales': len(ff) - mates,
            'tiene_actor': any(p.get('actor') for p in ff),
            'area_visible_vacia': len(va) == 0,
            'visible_area': a_texto(va) if va else None,
            'visible_player_counts': a_texto(f.get('visible_player_counts')),
        })
        for p in ff:
            loc = p.get('location') or [np.nan, np.nan]
            jg.append((mid, idx, bool(p.get('teammate')), bool(p.get('actor')),
                       bool(p.get('keeper')), loc[0], loc[1]))
    frames = pd.DataFrame(fr)
    jug = pd.DataFrame(jg, columns=['match_id', 'event_index', 'teammate', 'actor', 'keeper', 'x', 'y'])
    jug = jug.astype({'match_id': 'int32', 'event_index': 'int32', 'x': 'float32', 'y': 'float32'})
    return frames, jug


def convertir_lineups(raw, mid):
    filas = []
    for eq in raw:
        for p in eq.get('lineup') or []:
            r = {'match_id': mid, 'team_id': eq.get('team_id'), 'team_name': eq.get('team_name'),
                 'equipo_formations': a_texto(eq.get('formations')),
                 'equipo_events': a_texto(eq.get('events'))}
            r.update(p)
            filas.append(r)
    return anidado_a_texto(pd.DataFrame(filas))


def goles_equipos(ev):
    e = ev[ev['period'] < 5]   # excluye la tanda de penales
    g = {}
    if 'shot.outcome.name' in e.columns:
        s = e[(e['type.name'] == 'Shot') & (e['shot.outcome.name'] == 'Goal')]
        for t, n in s['team.id'].value_counts().items():
            g[int(t)] = g.get(int(t), 0) + int(n)
    og = e[e['type.name'] == 'Own Goal For']
    for t, n in og['team.id'].value_counts().items():
        g[int(t)] = g.get(int(t), 0) + int(n)
    return g


# ---------- Qué falta por convertir ----------
hechos = set()
if LOG.exists():
    with open(LOG) as f:
        for linea in f:
            try:
                hechos.add(json.loads(linea)['match_id'])
            except Exception:
                pass
pend = [m for m in partidos if int(m['match_id']) not in hechos]
if MAX_PARTIDOS:
    paso = max(1, len(pend) // MAX_PARTIDOS)
    pend = pend[::paso][:MAX_PARTIDOS]
print(f"Partidos objetivo: {len(partidos)} | ya convertidos: {len(hechos)} | a convertir ahora: {len(pend)}")

# ---------- Conversión ----------
t0 = time.time()
errores, respaldo, n_proc = [], 0, 0
with open(LOG, 'a') as flog:
    for m in pend:
        if time.time() - t0 > TIEMPO_MAX_MIN * 60:
            print("Tiempo máximo alcanzado. Vuelve a correr la celda para continuar.")
            break
        mid = int(m['match_id'])
        try:
            ev = aplanar_eventos(leer('eventos', mid), mid)
            if guardar(ev, OUT / 'eventos' / f'{mid}.parquet') != 'ok':
                respaldo += 1
            mapa = dict(zip(ev['id'], ev['index']))

            lu = convertir_lineups(leer('lineups', mid), mid)
            guardar(lu, OUT / 'lineups' / f'{mid}.parquet')

            tiene360 = (RAW / 'frames360' / f'{mid}.json.gz').exists()
            n_frames = unm = sin_actor = 0
            prom_vis = None
            if tiene360:
                frames, jug = convertir_360(leer('frames360', mid), mapa, mid)
                guardar(frames, OUT / 'frames360' / f'{mid}.parquet')
                guardar(jug, OUT / 'jugadores360' / f'{mid}.parquet')
                n_frames = len(frames)
                unm = int((frames['event_index'] < 0).sum())
                sin_actor = int((~frames['tiene_actor']).sum())
                prom_vis = float(frames['n_visibles'].mean())

            g = goles_equipos(ev)
            gl = g.get(int(m['home_team.home_team_id']), 0)
            gv = g.get(int(m['away_team.away_team_id']), 0)
            fuera = -1
            if 'loc_x' in ev.columns:
                fuera = int(((ev['loc_x'] < 0) | (ev['loc_x'] > 120) |
                             (ev['loc_y'] < 0) | (ev['loc_y'] > 80)).sum())
            fila = {
                'match_id': mid, 'season_id': int(m['season_id']),
                'n_eventos': int(len(ev)), 'n_columnas': int(ev.shape[1]),
                'n_starting_xi': int((ev['type.name'] == 'Starting XI').sum()),
                'periodo_max': int(ev['period'].max()), 'minuto_max': int(ev['minute'].max()),
                'loc_fuera_campo': fuera,
                'n_lineup_min_equipo': int(lu.groupby('team_id').size().min()) if len(lu) else 0,
                'tiene_360': bool(tiene360), 'n_frames': int(n_frames),
                'frames_sin_evento': unm, 'frames_sin_actor': sin_actor, 'prom_visibles': prom_vis,
                'goles_ev_local': int(gl), 'goles_ev_visita': int(gv),
                'goles_marcador_local': int(m['home_score']),
                'goles_marcador_visita': int(m['away_score']),
            }
            fila['coincide_marcador'] = bool(gl == fila['goles_marcador_local']
                                             and gv == fila['goles_marcador_visita'])
            flog.write(json.dumps(fila) + '\n')
            flog.flush()
            n_proc += 1
            if n_proc % 25 == 0:
                print(f"  {n_proc} partidos | {(time.time() - t0) / 60:.1f} min")
        except Exception as e:
            errores.append((mid, type(e).__name__, str(e)[:150]))

print(f"\nTiempo promedio: {(time.time() - t0) / max(1, n_proc):.1f} s por partido ({n_proc} partidos)")

# ---------- Resumen para pegar ----------
print("\n=== RESUMEN ===")
val = pd.read_json(LOG, lines=True).drop_duplicates('match_id', keep='last')
print("Partidos convertidos en total:", len(val), "de", len(partidos))
print("Errores en esta corrida:", len(errores))
for e in errores[:10]:
    print("  ", e)
print("Eventos guardados con respaldo de texto:", respaldo)

print("\n1) Peso en Drive por carpeta")
for c in ['eventos', 'frames360', 'jugadores360', 'lineups']:
    arch = list((OUT / c).glob('*.parquet'))
    print(f"{c}: {len(arch)} archivos | {sum(a.stat().st_size for a in arch) / 1e9:.3f} GB")

print("\n2) Por temporada")
print(val.groupby('season_id').agg(partidos=('match_id', 'count'),
                                   eventos_mediana=('n_eventos', 'median'),
                                   con_360=('tiene_360', 'sum'),
                                   marcador_ok=('coincide_marcador', 'sum')).to_string())

print("\n3) Chequeos de sanidad")
mal = val[~val['coincide_marcador']]
print("Partidos con goles de eventos distintos al marcador:", len(mal))
if len(mal):
    print(mal[['match_id', 'goles_ev_local', 'goles_ev_visita',
               'goles_marcador_local', 'goles_marcador_visita']].head(10).to_string(index=False))
print("Starting XI distinto de 2:", int((val['n_starting_xi'] != 2).sum()))
print("Mínimo de jugadores por equipo en lineups:", int(val['n_lineup_min_equipo'].min()))
print("Eventos con ubicación fuera de 0-120 / 0-80:", int(val['loc_fuera_campo'].clip(lower=0).sum()))
print("Periodo máximo:", val['periodo_max'].value_counts().to_dict())
v360 = val[val['tiene_360']]
if len(v360):
    print("Frames 360 sin evento enlazado:", int(v360['frames_sin_evento'].sum()),
          "de", int(v360['n_frames'].sum()))
    print("Frames 360 sin actor:", int(v360['frames_sin_actor'].sum()))
    print("Jugadores visibles por frame (promedio):", round(v360['prom_visibles'].mean(), 1))

print("\n4) Esquema de eventos (muestra de hasta 100 archivos)")
arch = sorted((OUT / 'eventos').glob('*.parquet'))
if arch:
    muestra = arch[::max(1, len(arch) // 100)]
    tipos, cols_arch = {}, []
    for a in muestra:
        s = pq.read_schema(a)
        cols_arch.append(set(s.names))
        for n, t in zip(s.names, s.types):
            tipos.setdefault(n, set()).add(str(t))
    print("Columnas: unión", len(set().union(*cols_arch)),
          "| comunes a todos", len(set.intersection(*cols_arch)))
    conf = {n: sorted(t) for n, t in tipos.items() if len(t - {'null'}) > 1}
    print("Columnas con tipos distintos entre partidos:", len(conf))
    for n, t in list(conf.items())[:15]:
        print("  ", n, t)

    print("\n5) Presencia de columnas v11 por temporada (fracción de la muestra)")
    clave = ['control_degree', 'is_transition', 'is_controlled_possession',
             'possession_directness', 'location_bucket', 'obv_total_net',
             'defensive_responsibility.probability_sum']
    temp = dict(zip(val['match_id'], val['season_id']))
    filas = []
    for a, cols in zip(muestra, cols_arch):
        r = {'season_id': temp.get(int(a.stem))}
        for k in clave:
            r[k] = k in cols
        filas.append(r)
    print(pd.DataFrame(filas).groupby('season_id').mean().round(2).to_string())

print("\n6) Versiones de datos por temporada (tabla de partidos)")
sel = matches[matches['season_id'].isin(TEMPORADAS)]
for c in ['metadata.data_version', 'metadata.shot_fidelity_version', 'metadata.xy_fidelity_version']:
    if c in sel.columns:
        print(c)
        print(sel.groupby(['season_label', c]).size().to_string())

ELO-1

In [ ]:
# ===== ELO-1. Búsqueda ampliada de parámetros del Elo (solo temporadas de entrenamiento) =====
# Usa la tabla de partidos del 2026-10-02. No toca 2025/26 ni 2026/27.
import glob, itertools, math, warnings, numpy as np, pandas as pd
from statsmodels.miscmodels.ordinal_model import OrderedModel
warnings.filterwarnings('ignore')

m = pd.read_parquet(glob.glob(f"{BASE}/**/matches_73_2026-10-02.parquet", recursive=True)[0])
cs  = [c for c in m.columns if c.endswith('season_id')][0]
hid = [c for c in m.columns if c.endswith('home_team_id')][0]
aid = [c for c in m.columns if c.endswith('away_team_id')][0]
TRAIN = [108, 235, 281, 317]                    # 2021/22 a 2024/25
d = m[m[cs].isin(TRAIN) & m['home_score'].notna()].copy()
d['kick_off'] = d['kick_off'].fillna('')
d = d.sort_values(['match_date', 'kick_off', 'match_id']).reset_index(drop=True)
d['res'] = (np.sign(d['home_score'] - d['away_score']) + 1).astype(int)   # 0 gana visita, 1 empate, 2 gana local
d['gd'] = (d['home_score'] - d['away_score']).abs().astype(int)
calib = (d['match_date'] >= '2021-10-01') & d[cs].isin([108, 235, 281])  # ajusta las probabilidades
test  = d[cs] == 317                                                     # evalúa (2024/25)
print(f"Partidos: {len(d)} | calibración: {int(calib.sum())} | prueba 2024/25: {int(test.sum())}")

def mult_margen(gd, tipo):
    if tipo == 'ninguno' or gd <= 1: return 1.0
    return 1.5 if gd == 2 else (11 + gd) / 8          # escala de eloratings.net

def correr_elo(K, H, R, margen):
    r, temp, dif = {}, None, np.empty(len(d))
    for i, (s, h, a, res, gd) in enumerate(zip(d[cs], d[hid], d[aid], d['res'], d['gd'])):
        if temp is not None and s != temp:                       # regresión entre temporadas
            r = {t: v - R * (v - 1500) for t, v in r.items()}
        temp = s
        rh, ra = r.get(h, 1500.0), r.get(a, 1500.0)
        dif[i] = rh + H - ra                                     # diferencia ANTES del partido
        e = 1 / (1 + 10 ** (-dif[i] / 400))
        delta = K * mult_margen(gd, margen) * (res / 2 - e)
        r[h], r[a] = rh + delta, ra - delta
    return dif

def evaluar(dif):
    y = pd.Series(d['res'], dtype=pd.CategoricalDtype([0, 1, 2], ordered=True))
    X = pd.DataFrame({'dif': dif / 100})
    fit = OrderedModel(y[calib], X[calib], distr='logit').fit(method='bfgs', disp=False)
    p = np.asarray(fit.model.predict(fit.params, exog=X[test]))
    Y = np.eye(3)[d.loc[test, 'res']]
    ll = -np.mean(np.log(np.clip((p * Y).sum(1), 1e-12, 1)))
    brier = np.mean(((p - Y) ** 2).sum(1))
    return ll, brier, p

# Referencia: frecuencias de la liga (sin Elo)
f = np.bincount(d.loc[calib, 'res'], minlength=3) / calib.sum()
Yt = np.eye(3)[d.loc[test, 'res']]
print(f"Frecuencias de la liga: log-loss {-np.mean(np.log((f * Yt).sum(1))):.4f} | Brier {np.mean(((f - Yt) ** 2).sum(1)):.4f}")

GRID = dict(K=[4, 6, 8, 10, 15, 20, 30, 40], H=[0, 25, 50, 75, 100, 150, 200],
            R=[0, 0.25, 0.5], margen=['ninguno', 'eloratings'])
filas = []
for K, H, R, mg in itertools.product(*GRID.values()):
    ll, br, _ = evaluar(correr_elo(K, H, R, mg))
    filas.append((K, H, R, mg, ll, br))
res = pd.DataFrame(filas, columns=['K', 'H', 'R', 'margen', 'logloss', 'brier']).sort_values('logloss')
print(f"\nCombinaciones probadas: {len(res)}")
print("\nTOP 10:"); print(res.head(10).round(4).to_string(index=False))
print("\nProvisional de la sesión 2 (K=10, H=100, R=0.25):")
print(res[(res.K == 10) & (res.H == 100) & (res.R == 0.25)].round(4).to_string(index=False))
b = res.iloc[0]
for p in ['K', 'H', 'R']:
    borde = b[p] == max(GRID[p]) or (b[p] == min(GRID[p]) and min(GRID[p]) > 0)
    print(f"Mejor {p} = {b[p]} {'(EN EL BORDE: ampliar)' if borde else '(interior o límite natural 0)'}")
print("Diferencia log-loss entre el 1° y el 10°:", round(res['logloss'].iloc[9] - res['logloss'].iloc[0], 4))

# Calibración del mejor: probabilidad de victoria local predicha vs observada, por quintiles
_, _, p = evaluar(correr_elo(b.K, b.H, b.R, b.margen))
q = pd.qcut(p[:, 2], 5, labels=False)
cal = pd.DataFrame({'quintil': q, 'pred_local': p[:, 2], 'obs_local': (d.loc[test, 'res'].values == 2)})
print("\nCalibración (victoria local, 2024/25):")
print(cal.groupby('quintil').agg(pred=('pred_local', 'mean'), obs=('obs_local', 'mean'), n=('obs_local', 'size')).round(3).to_string())
print(f"Empates: predicho medio {p[:, 1].mean():.3f} | observado {(d.loc[test, 'res'] == 1).mean():.3f}")

Tasas de resultados por temporada

In [ ]:
t = d.groupby(cs)['res'].value_counts(normalize=True).unstack().round(3)
t.columns = ['gana_visita', 'empate', 'gana_local']
print(t)

Definir Balon Parado

In [ ]:
# ===== BP1. Evidencia para balón parado (temporadas de entrenamiento, muestra de 200 partidos) =====
import os, glob, random, numpy as np, pandas as pd, pyarrow.parquet as pq

ptp = pd.read_parquet(f"{BASE}/datos_parquet/partido_tecnico_periodo.parquet")
TRAIN = sorted(ptp['temporada'].dropna().unique())[:4]
ids = sorted(ptp.loc[ptp['temporada'].isin(TRAIN), 'match_id'].unique())
random.seed(2026); muestra = random.sample(ids, min(200, len(ids)))     # misma muestra que D2

COLS = ['index', 'period', 'minute', 'type.name', 'loc_x', 'loc_y', 'pass_end_x', 'pass_end_y',
        'pass.type.name', 'pass.length', 'pass.outcome.name', 'set_piece_phase', 'shot.statsbomb_xg']
dfs = []
for mid in muestra:
    p = f"{BASE}/datos_parquet/liga/eventos/{mid}.parquet"
    if not os.path.exists(p): continue
    cols = [c for c in COLS if c in pq.read_schema(p).names]
    d = pd.read_parquet(p, columns=cols); d['match_id'] = mid; dfs.append(d)
ev = pd.concat(dfs, ignore_index=True)
for c in COLS:
    if c not in ev.columns: ev[c] = np.nan
ev = ev[ev['period'].isin([1, 2])]
print(f"Partidos: {ev['match_id'].nunique()} | eventos: {len(ev):,}")

# [1] Reanudaciones: de dónde salen, a dónde llegan y cuáles marca StatsBomb como balón parado
r = ev[(ev['type.name'] == 'Pass') & ev['pass.type.name'].isin(['Corner', 'Free Kick', 'Throw-in'])].copy()
r['tercio'] = pd.cut(r['loc_x'], [-1, 40, 80, 121], labels=['1 propio', '2 medio', '3 último'])
r['al_area'] = (r['pass_end_x'] >= 102) & r['pass_end_y'].between(18, 62)
r['con_sp'] = r['set_piece_phase'].notna()
print("\n[1] REANUDACIONES por tipo y tercio de origen")
t = r.groupby(['pass.type.name', 'tercio'], observed=True).agg(
    n=('con_sp', 'size'), pct_con_sp=('con_sp', 'mean'), pct_al_area=('al_area', 'mean'),
    largo_mediana=('pass.length', 'median'))
t[['pct_con_sp', 'pct_al_area']] *= 100
print(t.round(1).to_string())

print("\n[2] ¿Qué separa a las reanudaciones CON marca de las SIN marca? (tiros libres y saques de banda)")
fs = r[r['pass.type.name'].isin(['Free Kick', 'Throw-in'])]
t2 = fs.groupby(['pass.type.name', 'con_sp']).agg(n=('al_area', 'size'), pct_al_area=('al_area', 'mean'),
        x_origen_mediana=('loc_x', 'median'), largo_mediana=('pass.length', 'median'))
t2['pct_al_area'] *= 100
print(t2.round(1).to_string())

print("\n[3] Saques de banda en el último tercio: ¿suben los envíos al área al final del partido?")
sb = r[(r['pass.type.name'] == 'Throw-in') & (r['tercio'] == '3 último')].copy()
sb['tramo'] = pd.cut(sb['minute'], [-1, 59, 74, 200], labels=['0-59', '60-74', '75+'])
t3 = sb.groupby('tramo', observed=True).agg(n=('al_area', 'size'), pct_al_area=('al_area', 'mean'),
                                            pct_con_sp=('con_sp', 'mean'))
t3[['pct_al_area', 'pct_con_sp']] *= 100
print(t3.round(1).to_string())

print("\n[4] xG de balón parado por fase (1 = primer envío, 2 = ¿segunda jugada?)")
sh = ev[ev['type.name'] == 'Shot']
print(sh.groupby(sh['set_piece_phase'].fillna(0))['shot.statsbomb_xg'].agg(['size', 'sum', 'mean']).round(3).to_string())

print("\n[5] Estructura de las tablas 360 (para la siguiente celda)")
for carpeta in ['frames360', 'jugadores360']:
    f = sorted(glob.glob(f"{BASE}/datos_parquet/liga/{carpeta}/*.parquet"))
    if f:
        x = pd.read_parquet(f[0])
        print(f"  {carpeta}: {len(f)} archivos | columnas: {list(x.columns)}")
        print(x.head(2).to_string())

Validación de Balon Parado

In [ ]:
# ===== BP2. Viabilidad del 360 en córners (todas las temporadas de entrenamiento) =====
import os, numpy as np, pandas as pd, pyarrow.parquet as pq

ptp = pd.read_parquet(f"{BASE}/datos_parquet/partido_tecnico_periodo.parquet")
TRAIN = sorted(ptp['temporada'].dropna().unique())[:4]
temp_de = ptp.drop_duplicates('match_id').set_index('match_id')['temporada']
ids = sorted(ptp.loc[ptp['temporada'].isin(TRAIN), 'match_id'].unique())
EC = ['index', 'period', 'type.name', 'team.id', 'loc_y', 'pass.type.name', 'pass_end_x', 'pass_end_y']

filas = []
for n, mid in enumerate(ids, 1):
    pe = f"{BASE}/datos_parquet/liga/eventos/{mid}.parquet"
    pj = f"{BASE}/datos_parquet/liga/jugadores360/{mid}.parquet"
    if not (os.path.exists(pe) and os.path.exists(pj)): continue
    e = pd.read_parquet(pe, columns=[c for c in EC if c in pq.read_schema(pe).names])
    equipos = list(e['team.id'].dropna().unique())
    c = e[(e['type.name'] == 'Pass') & (e.get('pass.type.name') == 'Corner') & e['period'].isin([1, 2])]
    if c.empty: continue
    j = pd.read_parquet(pj); j = j[j['event_index'].isin(c['index'])]
    for _, k in c.iterrows():
        f = j[j['event_index'] == k['index']]
        defiende = [t for t in equipos if t != k['team.id']]
        fila = dict(match_id=mid, temporada=temp_de.get(mid), ataca=k['team.id'],
                    defiende=defiende[0] if defiende else np.nan, con_frame=len(f) > 0,
                    al_area=(k['pass_end_x'] >= 102) and (18 <= k['pass_end_y'] <= 62))
        if len(f):
            area  = (f['x'] >= 102) & f['y'].between(18, 62)
            chica = (f['x'] >= 114) & f['y'].between(30, 50)
            dfn = f[~f['teammate'] & ~f['keeper'] & area]
            atk = f[f['teammate'] & ~f['actor'] & area]
            fila.update(n_vis=len(f), def_area=len(dfn), atk_area=len(atk),
                        def_chica=int((~f['teammate'] & ~f['keeper'] & chica).sum()))
            if len(dfn) and len(atk):
                d = np.hypot(dfn['x'].values[:, None] - atk['x'].values[None, :],
                             dfn['y'].values[:, None] - atk['y'].values[None, :])
                fila['emparejamiento'] = d.min(axis=1).mean()
        filas.append(fila)
    if n % 300 == 0: print(f"  {n}/{len(ids)} partidos")
co = pd.DataFrame(filas)

print(f"\n[1] Córners: {len(co):,} | con foto 360: {100*co['con_frame'].mean():.1f}% | directos al área: {100*co['al_area'].mean():.1f}%")
v = co[co['con_frame']]
print("Jugadores visibles por foto:", v['n_vis'].describe(percentiles=[.1, .5, .9]).round(1)[['10%', '50%', '90%']].to_dict())
print(f"% fotos con 10+ jugadores dentro del área (área bien visible): {100*((v['def_area'] + v['atk_area']) >= 10).mean():.1f}")
for c in ['def_area', 'atk_area', 'def_chica', 'emparejamiento']:
    print(f"  {c:15s} p10/p50/p90: {v[c].quantile([.1, .5, .9]).round(2).tolist()}")

def mitades(df, quien, var, min_n=20):
    pares = []
    for _, s in df.dropna(subset=[var]).groupby([quien, 'temporada']):
        if len(s) < min_n: continue
        s = s.sample(frac=1, random_state=2026)
        pares.append((s[var].iloc[::2].mean(), s[var].iloc[1::2].mean()))
    a = np.array(pares)
    r = np.corrcoef(a[:, 0], a[:, 1])[0, 1]
    return len(a), round(r, 2), round(2 * r / (1 + r), 2), round(a.mean(1).min(), 2), round(a.mean(1).max(), 2)

print("\n[2] ¿Es un rasgo del equipo? Confiabilidad entre mitades por equipo-temporada (20+ córners)")
print("  variable         quién      grupos  r_mitades  confiab_total  rango_equipos")
for var, quien in [('def_area', 'defiende'), ('def_chica', 'defiende'), ('emparejamiento', 'defiende'),
                   ('atk_area', 'ataca'), ('al_area', 'ataca')]:
    g, r, sb, lo, hi = mitades(v if var != 'al_area' else co.assign(al_area=co['al_area'].astype(float)), quien, var)
    print(f"  {var:15s}  {quien:9s}  {g:6d}  {r:9.2f}  {sb:13.2f}  {lo} a {hi}")

# Sesión 4

R00 - Diagnostico

In [ ]:
# =====================================================================
# R-00 · Inspección de esquema (SOLO LECTURA: no crea ni modifica archivos)
# Sesión 4 · 2026-10-08 · Pegar en una celda nueva después de las celdas 1 y 2
# =====================================================================
import os, glob, re
import pandas as pd
import pyarrow.parquet as pq

try:
    BASE
except NameError:
    BASE = "/content/drive/MyDrive/isac2026"

DP = os.path.join(BASE, "datos_parquet")
LIGA = os.path.join(DP, "liga")
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 150)


def match_id_de(ruta):
    return int(re.findall(r"\d+", os.path.basename(ruta))[0])


def titulo(t):
    print("\n" + "=" * 72 + f"\n{t}\n" + "=" * 72)


# ------------------------------------------------------------------ A
titulo("A. Archivos en datos_parquet")
for f in sorted(os.listdir(DP)):
    p = os.path.join(DP, f)
    if os.path.isdir(p):
        print(f"  [carpeta] {f}")
    else:
        print(f"  {f:55s} {os.path.getsize(p) / 1e6:8.2f} MB")
for sub in sorted(os.listdir(LIGA)):
    n = len(glob.glob(os.path.join(LIGA, sub, "*.parquet")))
    print(f"  liga/{sub:15s} {n:5d} archivos")

# ------------------------------------------------------------------ B
titulo("B. Cobertura por temporada")
ptp = pd.read_parquet(os.path.join(DP, "partido_tecnico_periodo.parquet"))
print("  columnas de partido_tecnico_periodo:", list(ptp.columns))
part = ptp.drop_duplicates("match_id")[["match_id", "temporada"]].copy()

arch_ev = sorted(glob.glob(os.path.join(LIGA, "eventos", "*.parquet")))
ruta_ev = {match_id_de(a): a for a in arch_ev}
ids_360 = {match_id_de(a) for a in glob.glob(os.path.join(LIGA, "frames360", "*.parquet"))}
part["con_eventos"] = part["match_id"].isin(set(ruta_ev))
part["con_360"] = part["match_id"].isin(ids_360)
print(part.groupby("temporada").agg(partidos=("match_id", "size"),
                                    con_eventos=("con_eventos", "sum"),
                                    con_360=("con_360", "sum")).to_string())
sobrantes = set(ruta_ev) - set(part["match_id"])
print(f"  archivos de eventos sin fila en partido_tecnico_periodo: {len(sobrantes)}")

# ------------------------------------------------------------------ C
titulo("C. Esquema de eventos en TODOS los partidos (solo lee encabezados; 1-4 min)")
temp_de = dict(zip(part["match_id"], part["temporada"]))
V11 = ["control_degree", "is_transition", "is_controlled_possession", "possession_directness"]
conteo, sin_v11 = {}, []
for i, (m, a) in enumerate(ruta_ev.items(), 1):
    cols = pq.read_schema(a).names
    for c in cols:
        conteo[c] = conteo.get(c, 0) + 1
    faltan = [c for c in V11 if c not in cols]
    if faltan:
        sin_v11.append((m, temp_de.get(m), ", ".join(faltan)))
    if i % 400 == 0:
        print(f"  ... {i} de {len(ruta_ev)}")
N = len(ruta_ev)
print(f"  partidos leídos: {N} | columnas distintas: {len(conteo)} | "
      f"presentes en todos: {sum(v == N for v in conteo.values())}")
print("  columnas que NO están en todos los partidos (% de partidos que la tienen):")
for c, v in sorted(conteo.items(), key=lambda t: -t[1]):
    if v < N:
        print(f"    {c:58s} {100 * v / N:6.1f}%")

fv = pd.DataFrame(sin_v11, columns=["match_id", "temporada", "faltan"])
print(f"\n  partidos sin algún campo v11: {len(fv)}")
if len(fv):
    print(fv.groupby(["temporada", "faltan"]).size().to_string())
    if len(fv) <= 25:
        print("  ids:", fv["match_id"].tolist())

# ------------------------------------------------------------------ D
titulo("D. Columnas que usará el recálculo")
NECESARIAS = [
    "id", "index", "period", "timestamp", "minute", "second", "duration",
    "type.name", "possession", "possession_team.name", "possession_team.id",
    "play_pattern.name", "team.name", "team.id", "player.id", "position.id", "position.name",
    "loc_x", "loc_y", "under_pressure", "counterpress", "set_piece_phase", "location_bucket",
    "control_degree", "is_controlled_possession", "possession_directness",
    "pass.length", "pass.angle", "pass.type.name", "pass.outcome.name", "pass.height.name",
    "pass.recipient.id", "pass.cross", "pass.switch", "pass_success_probability",
    "pass_end_x", "pass_end_y", "carry_end_x", "carry_end_y",
    "shot.statsbomb_xg", "shot.type.name", "shot.outcome.name",
    "duel.type.name", "duel.outcome.name", "interception.outcome.name",
    "foul_committed.offensive", "foul_committed.card.name", "bad_behaviour.card.name",
    "substitution.replacement.id", "substitution.outcome.name",
    "tactics.formation", "tactics.lineup", "related_events",
    "defensive_responsibility", "obv_total_net", "out", "off_camera",
]
for n in NECESARIAS:
    if n in conteo:
        print(f"  OK       {n:38s} en {100 * conteo[n] / N:5.1f}% de partidos")
    else:
        ult = n.split(".")[-1]
        cand = [c for c in conteo if ult in c][:4]
        print(f"  NO ESTÁ  {n:38s} candidatas: {cand if cand else 'ninguna'}")

# ------------------------------------------------------------------ E
titulo("E. Un partido de muestra (semilla 2026)")
m = part[part["con_eventos"] & part["con_360"]].sample(1, random_state=2026).iloc[0]
mid = int(m["match_id"])
ev = pd.read_parquet(ruta_ev[mid])
print(f"  match_id {mid} · temporada {m['temporada']} · filas {len(ev)} · columnas {ev.shape[1]}")
print("  tipo de 'timestamp':", ev["timestamp"].dtype, "| primeros:", ev["timestamp"].head(3).tolist())
print("  ¿'index' ordenado?:", ev["index"].is_monotonic_increasing)


def vc(col, n=40):
    if col in ev.columns:
        print(f"\n  {col}:")
        print(ev[col].value_counts(dropna=False).head(n).to_string())
    else:
        print(f"\n  {col}: (no está en este partido)")


for col in ["period", "type.name", "play_pattern.name", "pass.type.name", "shot.type.name",
            "set_piece_phase", "counterpress", "under_pressure", "location_bucket", "duel.type.name"]:
    vc(col)

if "tactics.formation" in ev.columns:
    print("\n  formaciones registradas:",
          ev.loc[ev["tactics.formation"].notna(), ["type.name", "tactics.formation"]].values.tolist())
if "defensive_responsibility" in ev.columns:
    x = ev["defensive_responsibility"].dropna()
    print(f"\n  defensive_responsibility: {len(x)} eventos con valor; tipo "
          f"{type(x.iloc[0]).__name__ if len(x) else '-'}")
    if len(x):
        print("  estructura (truncada a 300 caracteres):", str(x.iloc[0])[:300])

# Vista previa del tiempo de posesión con la opción B (O-12)
if "possession_team.name" in ev.columns:
    ts = pd.to_timedelta(ev["timestamp"]).dt.total_seconds()
    g = pd.DataFrame({"period": ev["period"], "pos": ev["possession"],
                      "equipo": ev["possession_team.name"],
                      "ini": ts, "fin": ts + ev["duration"].fillna(0)})
    g = g[g["period"] <= 2]
    pos = (g.groupby(["period", "pos", "equipo"])
             .agg(ini=("ini", "min"), fin=("fin", "max")).reset_index())
    pos["dur"] = pos["fin"] - pos["ini"]
    print("\n  Tiempo de posesión (opción B), minutos por equipo:")
    print((pos.groupby("equipo")["dur"].sum() / 60).round(1).to_string())
    print(f"  total {pos['dur'].sum() / 60:.1f} min · posesiones {len(pos)} · "
          f"mediana por posesión {pos['dur'].median():.1f} s · máxima {pos['dur'].max():.0f} s")

# ------------------------------------------------------------------ F
titulo("F. Tablas 360 y alineación del mismo partido (solo columnas y tipos)")
for sub in ["frames360", "jugadores360", "lineups"]:
    a = os.path.join(LIGA, sub, f"{mid}.parquet")
    if os.path.exists(a):
        t = pd.read_parquet(a)
        print(f"\n  {sub}: {t.shape[0]} filas")
        print("   ", dict(t.dtypes.astype(str)))
    else:
        print(f"\n  {sub}: no existe {a}")

# ------------------------------------------------------------------ G
titulo("G. Otras tablas (solo nombres de columnas)")
for f in ["equipo_partido_marcador.parquet", "equipo_partido_elo_v1.parquet", "etapas_tecnico.parquet",
          "viabilidad_etapas.parquet", "resumen_partido_etapa1.parquet", "resumen_partido_etapa2.parquet",
          "hitos_v1.parquet", "estilo_equipo_partido_v1.parquet", "competitions.parquet"]:
    p = os.path.join(DP, f)
    if os.path.exists(p):
        s = pq.read_schema(p).names
        print(f"\n  {f} ({len(s)} col): {s}")
    else:
        print(f"\n  {f}: no está en datos_parquet")

print("\n  Archivos de partidos (matches) en todo el proyecto:")
for p in sorted(glob.glob(os.path.join(BASE, "**", "matches*"), recursive=True)):
    print("   ", os.path.relpath(p, BASE))
    if p.endswith(".parquet"):
        print("      columnas:", pq.read_schema(p).names)

print("\nFIN R-00")

R0 - Configuración

In [ ]:
# =====================================================================
# R0 · Configuración del framework v1 (correr una vez por sesión, antes de R1, R2, ...)
# Sesión 4 · 2026-10-08 · No lee datos: solo fija rutas y parámetros decididos
# =====================================================================
import os, json
import numpy as np
import pandas as pd

try:
    BASE
except NameError:
    BASE = "/content/drive/MyDrive/isac2026"

DP = os.path.join(BASE, "datos_parquet")
LIGA = os.path.join(DP, "liga")
FW = os.path.join(DP, "fw1")              # salidas del framework v1
os.makedirs(FW, exist_ok=True)

SEMILLA = 2026                            # D-10
np.random.seed(SEMILLA)

# D-1 / D-2 · particiones
TEMPORADAS = {
    "entrenamiento": ["2021/2022", "2022/2023", "2023/2024", "2024/2025"],
    "validacion":    ["2025/2026"],
    "prueba_vivo":   ["2026/2027"],
}

# Parámetros decididos (el identificador del framework va en el comentario)
P = {
    "tercio_medio_x": 40, "tercio_ultimo_x": 80,          # G-2
    "zona_ppda_x": 48,                                     # G-3
    "area_x": 102, "area_y": [18, 62],                     # G-4
    "transicion_s": 10, "transicion_sens_s": 15,           # P-1 / O-7
    "contrapresion_s": 5, "falta_tactica_sens_s": 6,       # M6
    "presion_pausa_candidatas": [2, 3, 4, 5, 6],           # P-2 / O-9
    "presion_exito_s": 5,                                  # P-3
    "pase_largo_yd": 35,                                   # P-4
    "cambio_orientacion_dy": 40, "vertical_grados": 30,    # P-13 / M2a
    "progresivo_avance_yd": 10,                            # P-12 / M14
    "tope_intervalo_C_s": 10,                              # P-11 (solo sensibilidad)
    "ventana_min": 5,                                      # P-9
    "etapa_min": 15,                                       # P-5
    "conf_nucleo": 0.7, "conf_secundario": 0.5,            # P-6
    "repeticiones": 200,                                   # P-6
    "dominio_ft": 0.70, "dominio_ventana_min": 10, "dominio_min_acciones": 20,  # P-10
}

REANUD_PASE = ["Goal Kick", "Kick Off", "Free Kick", "Throw-in", "Corner"]   # O-2
REANUD_TIRO = ["Free Kick", "Corner", "Penalty", "Kick Off"]                # O-2 (S4-06d)
PATRON_RECUPERACION = ["Regular Play", "From Counter"]                      # O-6
TIPOS_CONTRAGOLPE = ["Ball Receipt*", "Carry", "Shot", "Dribble", "50/50"]  # O-8

GRUPO_POSICION = {                                                          # S4-19
    1: "POR", 3: "CEN", 4: "CEN", 5: "CEN",
    2: "LAT", 6: "LAT", 7: "LAT", 8: "LAT",
    9: "MCD", 10: "MCD", 11: "MCD",
    13: "INT", 14: "INT", 15: "INT",
    18: "MP", 19: "MP", 20: "MP",
    12: "EXT", 16: "EXT", 17: "EXT", 21: "EXT",
    22: "DEL", 23: "DEL", 24: "DEL", 25: "DEL",
}

config = {"version": "fw1", "semilla": SEMILLA, "temporadas": TEMPORADAS, "P": P,
          "reanud_pase": REANUD_PASE, "reanud_tiro": REANUD_TIRO,
          "patron_recuperacion": PATRON_RECUPERACION, "tipos_contragolpe": TIPOS_CONTRAGOLPE,
          "grupo_posicion": GRUPO_POSICION}
with open(os.path.join(FW, "parametros_fw1.json"), "w") as f:
    json.dump(config, f, indent=1, ensure_ascii=False)

print("R0 lista")
print("  carpeta de salidas :", FW)
print("  parámetros         :", len(P), "| grupos de posición:", len(set(GRUPO_POSICION.values())))
print("  copia guardada en  : parametros_fw1.json")

R1 - Objetos

In [ ]:
# =====================================================================
# R1 · Objetos base del framework v1 para TODOS los partidos
#      O-1 balón parado · O-2 reanudación · O-3/O-12 posesión y su tiempo
#      O-6 recuperación · O-7 transición · O-8 contragolpe · O-9 pausas de presión
#      O-11 fase · O-13 marcador previo · P-9 ventana de 5 minutos
# Requiere haber corrido R0. Tarda ~10-25 min la primera vez; se puede reanudar.
# Escribe en datos_parquet/fw1/: eventos_<temporada>.parquet y posesiones_<temporada>.parquet
# =====================================================================
import os, glob, re, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

REHACER = False   # True = recalcula aunque ya existan los archivos de una temporada

COLS = ["index", "period", "timestamp", "minute", "second", "duration", "type.name",
        "possession", "possession_team.id", "possession_team.name", "play_pattern.name",
        "team.id", "team.name", "player.id", "position.id", "loc_x", "loc_y",
        "under_pressure", "counterpress", "set_piece_phase", "location_bucket",
        "control_degree", "is_controlled_possession", "possession_directness",
        "pass.length", "pass.angle", "pass.type.name", "pass.outcome.name", "pass.height.name",
        "pass.recipient.id", "pass.cross", "pass.switch", "pass.pass_success_probability",
        "pass_end_x", "pass_end_y", "carry_end_x", "carry_end_y", "ball_receipt.outcome.name",
        "shot.statsbomb_xg", "shot.type.name", "shot.outcome.name",
        "duel.type.name", "duel.outcome.name", "interception.outcome.name",
        "ball_recovery.recovery_failure",
        "foul_committed.offensive", "foul_committed.card.name", "bad_behaviour.card.name",
        "substitution.replacement.id", "substitution.outcome.name", "tactics.formation",
        "out", "off_camera", "obv_total_net"]
BOOLS = ["under_pressure", "counterpress", "pass.cross", "pass.switch",
         "ball_recovery.recovery_failure", "foul_committed.offensive", "out", "off_camera"]


def match_id_de(ruta):
    return int(re.findall(r"\d+", os.path.basename(ruta))[0])


def titulo(t):
    print("\n" + "=" * 72 + f"\n{t}\n" + "=" * 72)


# --------------------------------------------------------------------------------------
# Bloque 1 · Procesar un partido
# --------------------------------------------------------------------------------------
def procesar_partido(mid, ruta):
    nombres = set(pq.read_schema(ruta).names)
    ev = pd.read_parquet(ruta, columns=[c for c in COLS if c in nombres])
    for c in COLS:
        if c not in ev.columns:
            ev[c] = np.nan
    ev = ev.sort_values("index").reset_index(drop=True)
    ev.insert(0, "match_id", mid)
    ev["v11"] = "control_degree" in nombres
    for c in BOOLS:
        ev[c] = ev[c].eq(True)
    ev["t"] = pd.to_timedelta(ev["timestamp"]).dt.total_seconds()   # segundos dentro del periodo
    ev = ev.drop(columns="timestamp")
    tipo = ev["type.name"]

    # O-2 · reanudaciones (pases y tiros)
    ev["es_reanudacion"] = ev["pass.type.name"].isin(REANUD_PASE) | ev["shot.type.name"].isin(REANUD_TIRO)

    # O-1 + S4-05 · el balón parado es del equipo que lo ejecuta y acaba cuando el rival controla
    marca = ev["set_piece_phase"].notna()
    dueno = ev["team.id"].where(ev["es_reanudacion"] & marca)
    ev["bp_equipo"] = dueno.groupby(ev["period"]).ffill()
    ev["bp_sin_dueno"] = marca & ev["bp_equipo"].isna()
    ev["es_bp"] = marca & ((ev["possession_team.id"] == ev["bp_equipo"]) | ev["bp_sin_dueno"])

    # O-3 / O-12 · tabla de posesiones y su tiempo (B decidido; C sensibilidad)
    cb = ev[ev["loc_x"].notna()]
    g = cb.groupby("possession")
    pos = pd.DataFrame({
        "period": g["period"].first(),
        "equipo_id": g["possession_team.id"].first(),
        "equipo": g["possession_team.name"].first(),
        "patron": g["play_pattern.name"].first(),
        "primer_tipo": g["type.name"].first(),
        "primer_reanudacion": g["es_reanudacion"].first().astype(bool),
        "inicia_bp": g["es_bp"].first().astype(bool),
        "t0": g["t"].min(),
        "t1": (cb["t"] + cb["duration"].fillna(0)).groupby(cb["possession"]).max(),
        "controlada": g["is_controlled_possession"].first(),
        "directness": g["possession_directness"].first(),
    })
    pos["dur_B"] = (pos["t1"] - pos["t0"]).clip(lower=0)
    t_sig = g["t"].shift(-1)
    dt = (t_sig - cb["t"]).clip(lower=0)
    ultimo = t_sig.isna()
    dt = dt.where(~ultimo, cb["duration"].fillna(0))
    dt = dt.where(ultimo | (dt <= P["tope_intervalo_C_s"]), 0)
    pos["dur_C"] = dt.groupby(cb["possession"]).sum()

    # O-6 · posesión que nace de una recuperación en juego abierto
    pos["nace_rec"] = (pos["patron"].isin(PATRON_RECUPERACION)
                       & ~pos["primer_reanudacion"] & (pos["period"] <= 4))

    # O-7 · transición (se guarda t_pos para poder probar 15 s sin recalcular)
    ev = ev.join(pos[["t0", "nace_rec"]], on="possession")
    ev["nace_rec"] = ev["nace_rec"].eq(True)
    ev["t_pos"] = ev["t"] - ev["t0"]
    ev["en_transicion"] = ev["nace_rec"] & ev["t_pos"].between(0, P["transicion_s"])

    # O-11 · jerarquía de fases: balón parado > transición > establecido
    ev["fase"] = np.select([ev["es_bp"], ev["en_transicion"]], ["bp", "transicion"], default="establecido")

    # O-8 · contragolpe
    propio = ev["team.id"] == ev["possession_team.id"]
    m_cg = (ev["en_transicion"] & propio & (ev["loc_x"] >= P["tercio_ultimo_x"])
            & tipo.isin(TIPOS_CONTRAGOLPE))
    pos["contragolpe"] = pos.index.isin(ev.loc[m_cg, "possession"].unique()) & pos["nace_rec"]

    # xG sin penales y tiros por posesión
    tiro = (tipo == "Shot") & (ev["shot.type.name"] != "Penalty")
    pos["xg_np"] = ev.loc[tiro].groupby("possession")["shot.statsbomb_xg"].sum().reindex(pos.index).fillna(0)
    pos["tiros"] = ev.loc[tiro].groupby("possession").size().reindex(pos.index).fillna(0).astype(int)

    # O-9 · pausa desde la presión anterior del mismo equipo en la misma posesión rival
    pr = ev.loc[tipo == "Pressure"]
    ev["presion_gap_prev"] = pr.groupby(["possession", "team.id"])["t"].diff()

    # O-13 · diferencia de goles ANTES del evento, vista desde el equipo del evento
    gol = (((tipo == "Shot") & (ev["shot.outcome.name"] == "Goal") & (ev["period"] <= 4))
           | (tipo == "Own Goal For"))
    equipos = list(ev["team.id"].dropna().unique())
    if len(equipos) == 2:
        a, b = equipos
        prev = {}
        for e in equipos:
            ge = (gol & (ev["team.id"] == e)).astype(int)
            prev[e] = ge.cumsum() - ge
        ev["dif_antes"] = np.where(ev["team.id"] == a, prev[a] - prev[b],
                                   np.where(ev["team.id"] == b, prev[b] - prev[a], np.nan))
    else:
        ev["dif_antes"] = np.nan

    # P-9 · ventana de 5 minutos (0-8 primer tiempo con añadido en la 8; 9-17 segundo; 18 = añadido 2T)
    m = ev["minute"]
    ev["ventana"] = np.select(
        [ev["period"] == 1, ev["period"] == 2],
        [(m // 5).clip(upper=8), np.where(m >= 90, 18, ((m - 45) // 5).clip(lower=0, upper=8) + 9)],
        default=-1)

    pos = pos.reset_index()
    pos.insert(0, "match_id", mid)
    ev = ev.drop(columns=["t0"])
    return ev, pos


# --------------------------------------------------------------------------------------
# Bloque 2 · Recorrer temporadas (reanudable)
# --------------------------------------------------------------------------------------
titulo("R1 · Procesando partidos")
ptp = pd.read_parquet(os.path.join(DP, "partido_tecnico_periodo.parquet"))
part = ptp.drop_duplicates("match_id")[["match_id", "temporada"]]
temp_de = dict(zip(part["match_id"], part["temporada"]))
ruta_ev = {match_id_de(a): a for a in glob.glob(os.path.join(LIGA, "eventos", "*.parquet"))}


def archivo(tipo, temp):
    return os.path.join(FW, f"{tipo}_{temp.replace('/', '-')}.parquet")


for temp in sorted(part["temporada"].unique()):
    if os.path.exists(archivo("eventos", temp)) and not REHACER:
        print(f"  {temp}: ya existe, se omite (REHACER=False)")
        continue
    ids = part.loc[part["temporada"] == temp, "match_id"].tolist()
    evs, poss, fallos = [], [], []
    t_ini = time.time()
    for i, mid in enumerate(ids, 1):
        try:
            e, p = procesar_partido(int(mid), ruta_ev[int(mid)])
            evs.append(e)
            poss.append(p)
        except Exception as err:
            fallos.append((mid, repr(err)[:160]))
        if i % 100 == 0:
            print(f"    {temp}: {i}/{len(ids)} · {time.time() - t_ini:.0f} s")
    E = pd.concat(evs, ignore_index=True)
    PZ = pd.concat(poss, ignore_index=True)
    E.to_parquet(archivo("eventos", temp), index=False)
    PZ.to_parquet(archivo("posesiones", temp), index=False)
    print(f"  {temp}: {len(evs)} partidos OK · {len(fallos)} fallos · "
          f"{len(E):,} eventos · {len(PZ):,} posesiones · {time.time() - t_ini:.0f} s")
    for f in fallos[:5]:
        print("    FALLO", f)
    del E, PZ, evs, poss

# --------------------------------------------------------------------------------------
# Bloque 3 · Chequeos de sanidad (lee lo guardado; una temporada a la vez)
# --------------------------------------------------------------------------------------
titulo("R1 · Chequeos de sanidad")
mar = pd.read_parquet(os.path.join(DP, "equipo_partido_marcador.parquet"))[["match_id", "team", "gf"]]
filas, malos_goles, vent = [], [], None
cols_res = ["match_id", "index", "period", "t", "type.name", "team.name", "loc_x", "fase",
            "shot.statsbomb_xg", "shot.type.name", "shot.outcome.name",
            "bp_sin_dueno", "presion_gap_prev", "dif_antes", "ventana"]
for temp in sorted(part["temporada"].unique()):
    E = pd.read_parquet(archivo("eventos", temp), columns=cols_res)
    PZ = pd.read_parquet(archivo("posesiones", temp))
    est = E[(E["period"] <= 2) & E["loc_x"].notna()]
    fases = est["fase"].value_counts(normalize=True) * 100
    tiros = E[(E["type.name"] == "Shot") & (E["period"] <= 2) & (E["shot.type.name"] != "Penalty")]
    xg = tiros.groupby("fase")["shot.statsbomb_xg"].sum()
    xg = xg / xg.sum() * 100
    pz = PZ[PZ["period"] <= 2]
    pm = pz.groupby("match_id")[["dur_B", "dur_C"]].sum() / 60
    gp = E["presion_gap_prev"].dropna()
    # goles reconstruidos vs equipo_partido_marcador
    gol = E[((E["type.name"] == "Shot") & (E["shot.outcome.name"] == "Goal") & (E["period"] <= 4))
            | (E["type.name"] == "Own Goal For")]
    ge = gol.groupby(["match_id", "team.name"]).size().rename("gf_ev").reset_index()
    cmp = mar[mar["match_id"].isin(E["match_id"].unique())].merge(
        ge, left_on=["match_id", "team"], right_on=["match_id", "team.name"], how="left")
    cmp["gf_ev"] = cmp["gf_ev"].fillna(0)
    malos = cmp[cmp["gf"] != cmp["gf_ev"]]
    malos_goles.append(malos.assign(temporada=temp)[["temporada", "match_id", "team", "gf", "gf_ev"]])
    # tiempo por estado de marcador (|diferencia|), periodos 1-2
    s = E[E["period"] <= 2].sort_values(["match_id", "period", "index"]).copy()
    s["dif"] = s.groupby("match_id")["dif_antes"].ffill().abs()
    s["dt"] = (s.groupby(["match_id", "period"])["t"].shift(-1) - s["t"]).clip(lower=0, upper=60)
    est_m = s.groupby(np.minimum(s["dif"], 2))["dt"].sum()
    est_m = est_m / est_m.sum() * 100
    filas.append({
        "temporada": temp,
        "partidos": E["match_id"].nunique(),
        "%ev_bp": fases.get("bp", 0), "%ev_trans": fases.get("transicion", 0),
        "%ev_estab": fases.get("establecido", 0),
        "%xG_bp": xg.get("bp", 0), "%xG_trans": xg.get("transicion", 0),
        "%xG_estab": xg.get("establecido", 0),
        "bp_sin_dueno": int(E["bp_sin_dueno"].sum()),
        "posB_min": pm["dur_B"].mean(), "posC_min": pm["dur_C"].mean(),
        "%pos_nace_rec": 100 * pz["nace_rec"].mean(),
        "%contragolpe_de_rec": 100 * pz.loc[pz["nace_rec"], "contragolpe"].mean(),
        "pres_p50": gp.quantile(.5), "pres_p75": gp.quantile(.75),
        "%t_empate": est_m.get(0, 0), "%t_dif1": est_m.get(1, 0), "%t_dif2+": est_m.get(2, 0),
        "goles_no_cuadran": len(malos),
    })
    if temp == "2024/2025":
        vent = (E.loc[E["period"] <= 2, "ventana"].value_counts(normalize=True).sort_index() * 100).round(1)
    del E, PZ, s

res = pd.DataFrame(filas).set_index("temporada").round(1)
print(res.T.to_string())

mg = pd.concat(malos_goles)
print(f"\n  equipo-partidos con goles que no cuadran: {len(mg)}")
if len(mg):
    print(mg.head(10).to_string(index=False))

print("\n  % de eventos por ventana de 5 min (2024/25; 8 = 40-45+añadido 1T; 18 = añadido 2T):")
print(vent.to_string())
print("\nFIN R1")

R2a - metricas x partido, estado, eqipo

In [ ]:
# =====================================================================
# R2a · Componentes de métricas por equipo-partido-estado de marcador
#       M1 posesión (tiempo B/C y tercios) · M2/M2a pase largo · M2p riesgo y ejecución
#       M3 saque de meta · M7 directness · M8 centros · M9 xG por fase · M12 PPDA
#       M13 field tilt · M14 progresión y embudo · contragolpes
# Requiere R0 y R1. Escribe datos_parquet/fw1/componentes_<temporada>.parquet
# Guarda NUMERADORES y DENOMINADORES por separado (no razones): así cualquier
# agregación posterior (etapa, estado, estandarización) se hace como suma/suma.
# =====================================================================
import os, glob, time
import numpy as np
import pandas as pd

REHACER = False


def archivo(tipo, temp):
    return os.path.join(FW, f"{tipo}_{temp.replace('/', '-')}.parquet")


def titulo(t):
    print("\n" + "=" * 72 + f"\n{t}\n" + "=" * 72)


COLS2 = ["match_id", "index", "period", "minute", "t", "duration", "type.name", "possession",
         "possession_team.id", "team.id", "team.name", "loc_x", "loc_y", "under_pressure",
         "es_bp", "es_reanudacion", "fase", "en_transicion", "dif_antes",
         "pass.length", "pass.angle", "pass.type.name", "pass.outcome.name", "pass.switch",
         "pass.cross", "pass.pass_success_probability", "pass_end_x", "pass_end_y",
         "carry_end_x", "carry_end_y", "ball_receipt.outcome.name",
         "shot.statsbomb_xg", "shot.type.name", "duel.type.name", "foul_committed.offensive"]

X_MED, X_ULT = P["tercio_medio_x"], P["tercio_ultimo_x"]
AX, (AY0, AY1) = P["area_x"], P["area_y"]


def agrupa(df, eq, d, cols):
    """Suma columnas por (partido, equipo, estado d de -2 a +2)."""
    out = pd.DataFrame({"match_id": df["match_id"].to_numpy(),
                        "equipo_id": np.asarray(eq, dtype=float),
                        "d": np.asarray(d, dtype=float)})
    for k, v in cols.items():
        out[k] = np.asarray(v, dtype=float)
    return out.groupby(["match_id", "equipo_id", "d"]).sum()


def estado(df, eq):
    """Diferencia de goles (-2..+2) vista desde el equipo eq."""
    eq = np.asarray(eq, dtype=float)
    return np.clip(np.where(eq == df["A"].to_numpy(), df["dif_A"].to_numpy(), -df["dif_A"].to_numpy()), -2, 2)


def en_area(x, y):
    return (x >= AX) & (y >= AY0) & (y <= AY1)


def progresivo(x0, y0, x1, y1):
    cruza = ((x0 < X_MED) & (x1 >= X_MED)) | ((x0 < X_ULT) & (x1 >= X_ULT))
    avanza = (x1 - x0) >= P["progresivo_avance_yd"]
    entra = en_area(x1, y1) & ~en_area(x0, y0)
    return (cruza & avanza) | entra


def componentes(temp):
    E = pd.read_parquet(archivo("eventos", temp), columns=COLS2)
    extras = {"anadido_1T": (E[E["period"] == 1].groupby("match_id")["minute"].max() - 45).clip(lower=0).median(),
              "anadido_2T": (E[E["period"] == 2].groupby("match_id")["minute"].max() - 90).clip(lower=0).median()}
    E = E[E["period"] <= 2].copy()
    eqs = E.groupby("match_id")["team.id"].agg(["min", "max"])
    E["A"] = E["match_id"].map(eqs["min"])
    E["S"] = E["match_id"].map(eqs["min"] + eqs["max"])        # rival = S - equipo
    E["dif_A"] = np.where(E["team.id"] == E["A"], E["dif_antes"], -E["dif_antes"])
    E["dif_A"] = E.groupby("match_id")["dif_A"].ffill().fillna(0)
    tipo = E["type.name"]
    abierto = (E["fase"] != "bp") & ~E["es_reanudacion"]
    partes = []

    # ---- M1 · tiempo de posesión propio y del rival (O-12)
    PZ = pd.read_parquet(archivo("posesiones", temp))
    PZ = PZ[PZ["period"] <= 2]
    pr = (E[E["loc_x"].notna()].groupby(["match_id", "possession"])[["dif_A", "A", "S"]]
          .first().reset_index())
    PZ = PZ.merge(pr, on=["match_id", "possession"], how="left")
    PZ = PZ[PZ["A"].notna()]
    dP = estado(PZ, PZ["equipo_id"])
    partes.append(agrupa(PZ, PZ["equipo_id"], dP, {"t_pos_B": PZ["dur_B"], "t_pos_C": PZ["dur_C"]}))
    partes.append(agrupa(PZ, PZ["S"] - PZ["equipo_id"], -dP, {"t_riv_B": PZ["dur_B"], "t_riv_C": PZ["dur_C"]}))

    # ---- M1-tercio · tiempo propio por tercio (cada intervalo va a la ubicación del evento que lo inicia)
    cb = E[E["loc_x"].notna()].copy()
    cb["t_sig"] = cb.groupby(["match_id", "possession"])["t"].shift(-1)
    dt = (cb["t_sig"] - cb["t"]).clip(lower=0)
    dt = dt.where(cb["t_sig"].notna(), cb["duration"].fillna(0)).to_numpy()
    xpos = np.where(cb["team.id"] == cb["possession_team.id"], cb["loc_x"], 120 - cb["loc_x"])
    partes.append(agrupa(cb, cb["possession_team.id"], estado(cb, cb["possession_team.id"]), {
        "t_ter_def": dt * (xpos < X_MED),
        "t_ter_med": dt * ((xpos >= X_MED) & (xpos < X_ULT)),
        "t_ter_ult": dt * (xpos >= X_ULT)}))

    # ---- M7, M8, M14-embudo, contragolpes · por posesión en juego abierto (no inicia en balón parado)
    own = E[(E["team.id"] == E["possession_team.id"]) & abierto]
    mx = own.groupby(["match_id", "possession"]).agg(max_x=("loc_x", "max"),
                                                     centro=("pass.cross", "any")).reset_index()
    op = PZ.merge(mx, on=["match_id", "possession"], how="left")
    op = op[~op["inicia_bp"].astype(bool)]
    alc_u = op["max_x"] >= X_ULT
    partes.append(agrupa(op, op["equipo_id"], estado(op, op["equipo_id"]), {
        "n_pos_op": 1,
        "n_alc_medio": op["max_x"] >= X_MED,
        "n_alc_u3": alc_u,
        "n_con_tiro": op["tiros"] > 0,
        "n_seq_centro": alc_u & op["centro"].eq(True),
        "sum_directness": pd.to_numeric(op["directness"], errors="coerce").fillna(0),
        "n_directness": pd.to_numeric(op["directness"], errors="coerce").notna(),
        "n_rec": op["nace_rec"].astype(bool),
        "n_contragolpe": op["contragolpe"].astype(bool)}))

    # ---- M2, M2a, M2p, M14 pases · pases de juego abierto sin reanudaciones
    pa = E[(tipo == "Pass") & abierto]
    comp = pa["pass.outcome.name"].isna()
    prob = pd.to_numeric(pa["pass.pass_success_probability"], errors="coerce")
    tienep = prob.notna()
    largo = pa["pass.length"] >= P["pase_largo_yd"]
    forz = largo & pa["under_pressure"]
    contra = largo & ~forz & pa["en_transicion"]
    dy = (pa["pass_end_y"] - pa["loc_y"]).abs()
    cambio = largo & (dy >= P["cambio_orientacion_dy"])
    vert = largo & ~cambio & (pa["pass.angle"].abs() <= np.radians(P["vertical_grados"]))
    partes.append(agrupa(pa, pa["team.id"], estado(pa, pa["team.id"]), {
        "n_pases": 1, "n_completos": comp,
        "n_prob": tienep, "sum_prob": prob.fillna(0), "n_comp_prob": comp & tienep,
        "n_largo": largo, "n_largo_forz": forz, "n_largo_contra": contra, "n_largo_plan": largo & ~forz & ~contra,
        "n_cambio": cambio, "n_vertical": vert, "n_otro_dir": largo & ~cambio & ~vert,
        "n_switch_sb": largo & pa["pass.switch"], "n_cambio_y_switch": cambio & pa["pass.switch"],
        "n_prog_pase": comp & progresivo(pa["loc_x"], pa["loc_y"], pa["pass_end_x"], pa["pass_end_y"])}))

    # ---- M3 · saque de meta corto (primer receptor en x < 40)
    gk = E[E["pass.type.name"] == "Goal Kick"]
    partes.append(agrupa(gk, gk["team.id"], estado(gk, gk["team.id"]), {
        "n_saque_meta": 1, "n_saque_meta_corto": gk["pass_end_x"] < X_MED}))

    # ---- M14 conducciones (S4-23 C: conteo + yardas + subtipo larga)
    co = E[(tipo == "Carry") & abierto]
    largo_c = np.hypot(co["carry_end_x"] - co["loc_x"], co["carry_end_y"] - co["loc_y"])
    prog_c = progresivo(co["loc_x"], co["loc_y"], co["carry_end_x"], co["carry_end_y"])
    partes.append(agrupa(co, co["team.id"], estado(co, co["team.id"]), {
        "n_cond": 1, "n_prog_cond": prog_c,
        "yd_prog_cond": (co["carry_end_x"] - co["loc_x"]).clip(lower=0) * prog_c,
        "n_prog_cond_larga": prog_c & (largo_c >= 30)}))
    extras["cond_largo"] = largo_c.to_numpy()
    extras["cond_prog_largo"] = largo_c[prog_c].to_numpy()

    # ---- M13 · field tilt (F2 acciones; F1 solo pases)
    ac = E[abierto & tipo.isin(["Pass", "Carry", "Ball Receipt*", "Shot"])]
    ac = ac[~((ac["type.name"] == "Ball Receipt*") & ac["ball_receipt.outcome.name"].notna())]
    u3 = ac["loc_x"] >= X_ULT
    partes.append(agrupa(ac, ac["team.id"], estado(ac, ac["team.id"]), {
        "ft_acc_u3": u3, "ft_pas_u3": u3 & (ac["type.name"] == "Pass")}))

    # ---- M12 · PPDA: pases del rival en su 60% propio / acciones del que presiona en x >= 48
    rp = pa[pa["loc_x"] <= 120 - P["zona_ppda_x"]]
    partes.append(agrupa(rp, rp["S"] - rp["team.id"], -estado(rp, rp["team.id"]), {"ppda_pases_rival": 1}))
    de = E[(E["team.id"] != E["possession_team.id"]) & abierto & (E["loc_x"] >= P["zona_ppda_x"])]
    acc = (((de["type.name"] == "Duel") & (de["duel.type.name"] == "Tackle"))
           | (de["type.name"] == "Interception")
           | ((de["type.name"] == "Foul Committed") & ~de["foul_committed.offensive"]))
    partes.append(agrupa(de, de["team.id"], estado(de, de["team.id"]), {
        "ppda_acciones": acc, "ppda_presiones": de["type.name"] == "Pressure"}))

    # ---- M9 · xG sin penales por fase
    sh = E[(tipo == "Shot") & (E["shot.type.name"] != "Penalty")]
    xg = sh["shot.statsbomb_xg"].fillna(0)
    partes.append(agrupa(sh, sh["team.id"], estado(sh, sh["team.id"]), {
        "n_tiros_np": 1, "xg_np": xg, "xg_bp": xg * (sh["fase"] == "bp"),
        "xg_trans": xg * (sh["fase"] == "transicion"), "xg_estab": xg * (sh["fase"] == "establecido")}))

    C = pd.concat(partes, axis=1).fillna(0).reset_index()
    C.insert(0, "temporada", temp)
    nombres = E.drop_duplicates("team.id").set_index("team.id")["team.name"]
    C.insert(3, "equipo", C["equipo_id"].map(nombres))
    return C, extras


# --------------------------------------------------------------------------------------
# Bloque 2 · Recorrer temporadas
# --------------------------------------------------------------------------------------
titulo("R2a · Calculando componentes")
temporadas = sorted(pd.read_parquet(os.path.join(DP, "partido_tecnico_periodo.parquet"))["temporada"].unique())
extras_todas = {}
for temp in temporadas:
    t0 = time.time()
    if os.path.exists(archivo("componentes", temp)) and not REHACER:
        print(f"  {temp}: ya existe, se omite")
        continue
    C, ex = componentes(temp)
    C.to_parquet(archivo("componentes", temp), index=False)
    extras_todas[temp] = ex
    print(f"  {temp}: {len(C):,} filas equipo-partido-estado · {C['match_id'].nunique()} partidos · {time.time() - t0:.0f} s")

# --------------------------------------------------------------------------------------
# Bloque 3 · Chequeos a nivel liga (suma/suma por temporada)
# --------------------------------------------------------------------------------------
titulo("R2a · Valores de liga por temporada")
C = pd.concat([pd.read_parquet(archivo("componentes", t)) for t in temporadas], ignore_index=True)
S = C.groupby("temporada").sum(numeric_only=True)
L = pd.DataFrame(index=S.index)
L["M1 tercio def %"] = 100 * S.t_ter_def / (S.t_ter_def + S.t_ter_med + S.t_ter_ult)
L["M1 tercio med %"] = 100 * S.t_ter_med / (S.t_ter_def + S.t_ter_med + S.t_ter_ult)
L["M1 tercio ult %"] = 100 * S.t_ter_ult / (S.t_ter_def + S.t_ter_med + S.t_ter_ult)
L["chequeo tercios/B"] = (S.t_ter_def + S.t_ter_med + S.t_ter_ult) / S.t_pos_B
L["precisión cruda %"] = 100 * S.n_completos / S.n_pases
L["M2p riesgo"] = 1 - S.sum_prob / S.n_prob
L["M2p ejecución /100"] = 100 * (S.n_comp_prob - S.sum_prob) / S.n_prob
L["M2 % largo"] = 100 * S.n_largo / S.n_pases
L["  plan %"] = 100 * S.n_largo_plan / S.n_largo
L["  contragolpe %"] = 100 * S.n_largo_contra / S.n_largo
L["  forzado %"] = 100 * S.n_largo_forz / S.n_largo
L["M2a cambio %"] = 100 * S.n_cambio / S.n_largo
L["M2a vertical %"] = 100 * S.n_vertical / S.n_largo
L["M2a otro %"] = 100 * S.n_otro_dir / S.n_largo
L["switch SB que es cambio %"] = 100 * S.n_cambio_y_switch / S.n_switch_sb
L["cambio que SB marca switch %"] = 100 * S.n_cambio_y_switch / S.n_cambio
L["M3 saque meta corto %"] = 100 * S.n_saque_meta_corto / S.n_saque_meta
L["M7 directness media"] = S.sum_directness / S.n_directness
L["M8 secuencias con centro %"] = 100 * S.n_seq_centro / S.n_alc_u3
L["embudo: llega medio %"] = 100 * S.n_alc_medio / S.n_pos_op
L["embudo: llega último %"] = 100 * S.n_alc_u3 / S.n_pos_op
L["embudo: tiro %"] = 100 * S.n_con_tiro / S.n_pos_op
L["contragolpe / recuperación %"] = 100 * S.n_contragolpe / S.n_rec
L["M14 pases prog / 10 min pos"] = S.n_prog_pase / (S.t_pos_B / 600)
L["M14 cond prog / 10 min pos"] = S.n_prog_cond / (S.t_pos_B / 600)
L["  cond prog larga %"] = 100 * S.n_prog_cond_larga / S.n_prog_cond
L["M12 PPDA"] = S.ppda_pases_rival / S.ppda_acciones
L["M12 intensidad /100"] = 100 * S.ppda_acciones / S.ppda_pases_rival
L["M12 PPDA con presiones"] = S.ppda_pases_rival / (S.ppda_acciones + S.ppda_presiones)
L["M9 xG trans %"] = 100 * S.xg_trans / S.xg_np
print(L.round(2).T.to_string())

# --------------------------------------------------------------------------------------
# Bloque 4 · Dispersión entre equipos-temporada (entrenamiento): ¿los rasgos separan equipos?
# --------------------------------------------------------------------------------------
titulo("R2a · Dispersión por equipo-temporada (2021/22 a 2024/25)")
ent = C[C["temporada"].isin(TEMPORADAS["entrenamiento"])]
T = ent.groupby(["temporada", "match_id", "equipo_id", "equipo"]).sum(numeric_only=True).reset_index()
riv = T[["match_id", "equipo_id", "ft_acc_u3", "ft_pas_u3"]].rename(
    columns={"equipo_id": "riv_id", "ft_acc_u3": "riv_ft_acc", "ft_pas_u3": "riv_ft_pas"})
T = T.merge(riv, on="match_id")
T = T[T["equipo_id"] != T["riv_id"]]
ES = T.groupby(["temporada", "equipo"]).sum(numeric_only=True)
D = pd.DataFrame(index=ES.index)
D["posesión B %"] = 100 * ES.t_pos_B / (ES.t_pos_B + ES.t_riv_B)
D["posesión C %"] = 100 * ES.t_pos_C / (ES.t_pos_C + ES.t_riv_C)
D["field tilt F2 %"] = 100 * ES.ft_acc_u3 / (ES.ft_acc_u3 + ES.riv_ft_acc)
D["field tilt F1 %"] = 100 * ES.ft_pas_u3 / (ES.ft_pas_u3 + ES.riv_ft_pas)
D["% largo"] = 100 * ES.n_largo / ES.n_pases
D["riesgo"] = 1 - ES.sum_prob / ES.n_prob
D["ejecución /100"] = 100 * (ES.n_comp_prob - ES.sum_prob) / ES.n_prob
D["saque meta corto %"] = 100 * ES.n_saque_meta_corto / ES.n_saque_meta
D["pases prog /10min"] = ES.n_prog_pase / (ES.t_pos_B / 600)
D["intensidad PPDA"] = 100 * ES.ppda_acciones / ES.ppda_pases_rival
D["directness"] = ES.sum_directness / ES.n_directness
print(f"  equipos-temporada: {len(D)}")
print(D.describe(percentiles=[.1, .5, .9]).T[["min", "10%", "50%", "90%", "max"]].round(2).to_string())
print("\n  correlación posesión B vs C:", round(D["posesión B %"].corr(D["posesión C %"]), 3),
      "| field tilt F1 vs F2:", round(D["field tilt F1 %"].corr(D["field tilt F2 %"]), 3),
      "| riesgo vs ejecución:", round(D["riesgo"].corr(D["ejecución /100"]), 3))

# --------------------------------------------------------------------------------------
# Bloque 5 · Evidencia para decisiones abiertas
# --------------------------------------------------------------------------------------
titulo("R2a · Evidencia para decisiones abiertas")
if extras_todas:
    print("  Mediana de minutos añadidos por partido:")
    for t, ex in extras_todas.items():
        print(f"    {t}: 1T {ex['anadido_1T']:.0f} min · 2T {ex['anadido_2T']:.0f} min")
    cl = np.concatenate([ex["cond_largo"] for t, ex in extras_todas.items() if t in TEMPORADAS["entrenamiento"]])
    cp = np.concatenate([ex["cond_prog_largo"] for t, ex in extras_todas.items() if t in TEMPORADAS["entrenamiento"]])
    q = [.25, .5, .75, .9, .95]
    print("\n  Largo de conducciones (yd), entrenamiento — todas:", np.round(np.nanquantile(cl, q), 1),
          "\n                                            progresivas:", np.round(np.nanquantile(cp, q), 1),
          "\n  (percentiles 25, 50, 75, 90, 95)")
else:
    print("  (las temporadas ya existían; pon REHACER=True si necesitas esta evidencia)")
print("\nFIN R2a")

R2b - Presión

In [ ]:
# =====================================================================
# R2b · Presión: fases de presión (O-9) con X = 2..6 s, M4 altura y M6 transición defensiva
#       + primera prueba de confiabilidad (solo temporadas de entrenamiento)
# Requiere R0 y R1. Escribe datos_parquet/fw1/presion_<temporada>.parquet
# =====================================================================
import os, time
import numpy as np
import pandas as pd

REHACER = False
REANUD_S = 5    # [PROVISIONAL-Claude, S4-29] presión "sobre reanudación" = en los primeros 5 s de una posesión que nace de reanudación
XS = P["presion_pausa_candidatas"]
N_TR = P["transicion_s"]
X_MED, X_ULT = P["tercio_medio_x"], P["tercio_ultimo_x"]
COLS3 = ["match_id", "index", "period", "t", "type.name", "possession", "possession_team.id",
         "team.id", "team.name", "player.id", "loc_x", "counterpress", "under_pressure", "fase",
         "nace_rec", "t_pos", "dif_antes", "foul_committed.offensive"]


def archivo(tipo, temp):
    return os.path.join(FW, f"{tipo}_{temp.replace('/', '-')}.parquet")


def titulo(t):
    print("\n" + "=" * 72 + f"\n{t}\n" + "=" * 72)


def agrupa(df, eq, d, cols):
    out = pd.DataFrame({"match_id": df["match_id"].to_numpy(),
                        "equipo_id": np.asarray(eq, dtype=float),
                        "d": np.asarray(d, dtype=float)})
    for k, v in cols.items():
        out[k] = np.asarray(v, dtype=float)
    return out.groupby(["match_id", "equipo_id", "d"]).sum()


def estado(df, eq):
    eq = np.asarray(eq, dtype=float)
    return np.clip(np.where(eq == df["A"].to_numpy(), df["dif_A"].to_numpy(), -df["dif_A"].to_numpy()), -2, 2)


def tercio(x):
    x = np.asarray(x, dtype=float)
    return np.where(x >= X_ULT, "alta", np.where(x >= X_MED, "media", "baja"))


# --------------------------------------------------------------------------------------
# Bloque 1 · Una temporada
# --------------------------------------------------------------------------------------
def presion_temp(temp):
    E = pd.read_parquet(archivo("eventos", temp), columns=COLS3)
    E = E[E["period"] <= 2].copy()
    eqs = E.groupby("match_id")["team.id"].agg(["min", "max"])
    E["A"] = E["match_id"].map(eqs["min"])
    E["S"] = E["match_id"].map(eqs["min"] + eqs["max"])
    E["dif_A"] = np.where(E["team.id"] == E["A"], E["dif_antes"], -E["dif_antes"])
    E["dif_A"] = E.groupby("match_id")["dif_A"].ffill().fillna(0)

    PZ = pd.read_parquet(archivo("posesiones", temp)).sort_values(["match_id", "possession"])
    for c, s in [("sig_eq", "equipo_id"), ("sig_t0", "t0"), ("sig_per", "period")]:
        PZ[c] = PZ.groupby("match_id")[s].shift(-1)
    PZ = PZ[PZ["period"] <= 2]
    partes = []

    # ---- O-9 · presiones del equipo sin balón
    pr = E[(E["type.name"] == "Pressure") & (E["team.id"] != E["possession_team.id"])]
    pr = pr.merge(PZ[["match_id", "possession", "t0", "primer_reanudacion", "sig_eq", "sig_t0", "sig_per"]],
                  on=["match_id", "possession"], how="left")
    ult = (E[E["team.id"] == E["possession_team.id"]].sort_values("index")
           .groupby(["match_id", "possession"]).agg(ult_bajo_presion=("under_pressure", "last")).reset_index())
    pr = pr.merge(ult, on=["match_id", "possession"], how="left")
    pr = pr.sort_values(["match_id", "possession", "team.id", "index"]).reset_index(drop=True)
    pr["gap"] = pr.groupby(["match_id", "possession", "team.id"])["t"].diff()
    pr["d"] = estado(pr, pr["team.id"])
    pr["sobre_reanud"] = pr["primer_reanudacion"].eq(True) & ((pr["t"] - pr["t0"]) <= REANUD_S)
    nucleo = (pr["fase"] != "bp") & ~pr["sobre_reanud"]

    # Sensibilidad M4: presiones individuales (eventos)
    pe = pr[nucleo]
    alt_e = tercio(pe["loc_x"])
    partes.append(agrupa(pe, pe["team.id"], pe["d"], {
        "ev_pres": 1, "ev_alta": alt_e == "alta", "ev_media": alt_e == "media", "ev_baja": alt_e == "baja"}))

    # Fases con cada X candidata
    for X in XS:
        pr["fid"] = (pr["gap"].isna() | (pr["gap"] > X)).cumsum()
        f = pr.groupby("fid").agg(
            match_id=("match_id", "first"), equipo_id=("team.id", "first"), d=("d", "first"),
            period=("period", "first"), x0=("loc_x", "first"), fase0=("fase", "first"),
            reanud=("sobre_reanud", "first"), n_jug=("player.id", "nunique"), n_pres=("t", "size"),
            t_last=("t", "last"), sig_eq=("sig_eq", "first"), sig_t0=("sig_t0", "first"),
            sig_per=("sig_per", "first"), ult_bp=("ult_bajo_presion", "first"))
        f = f[(f["fase0"] != "bp") & ~f["reanud"].astype(bool)]
        exito = ((f["sig_eq"] == f["equipo_id"]) & (f["sig_per"] == f["period"])
                 & ((f["sig_t0"] - f["t_last"]) <= P["presion_exito_s"]) & f["ult_bp"].eq(True))
        alt = tercio(f["x0"])
        partes.append(agrupa(f, f["equipo_id"], f["d"], {
            f"fases_x{X}": 1, f"grupal_x{X}": f["n_jug"] >= 2, f"exito_x{X}": exito,
            f"alta_x{X}": alt == "alta", f"media_x{X}": alt == "media", f"baja_x{X}": alt == "baja",
            f"pres_x{X}": f["n_pres"]}))

    # ---- M6 · transición defensiva (opción D: respuesta 5 s + resolución 10 s)
    pinfo = E[E["loc_x"].notna()].groupby(["match_id", "possession"])[["dif_A", "A", "S"]].first().reset_index()
    T = PZ[PZ["nace_rec"].astype(bool)].merge(pinfo, on=["match_id", "possession"], how="left")
    T = T[T["A"].notna()].copy()
    T["L"] = T["S"] - T["equipo_id"]                       # equipo que perdió el balón
    dL = -estado(T, T["equipo_id"])
    Ein = E[E["nace_rec"].astype(bool) & E["t_pos"].between(0, N_TR)]
    deL = Ein[Ein["team.id"] != Ein["possession_team.id"]]
    k = ["match_id", "possession"]
    cp = (deL[deL["counterpress"] & (deL["t_pos"] <= P["contrapresion_s"])].sort_values("index")
          .groupby(k).agg(cp_t=("t_pos", "first"), cp_jug=("player.id", "nunique"), cp_x=("loc_x", "first")))
    fo = (deL[(deL["type.name"] == "Foul Committed") & ~deL["foul_committed.offensive"]]
          .groupby(k).agg(falta_t=("t_pos", "min")))
    ps = deL[(deL["type.name"] == "Pressure") | deL["counterpress"]].groupby(k).agg(pres_t=("t_pos", "min"))
    cg = (Ein[(Ein["team.id"] == Ein["possession_team.id"]) & (Ein["loc_x"] >= X_ULT)
              & Ein["type.name"].isin(TIPOS_CONTRAGOLPE)].groupby(k).agg(cg_t=("t_pos", "min")))
    for tabla in (cp, fo, ps, cg):
        T = T.merge(tabla, left_on=k, right_index=True, how="left")
    T["rec_t"] = np.where((T["sig_eq"] == T["L"]) & (T["sig_per"] == T["period"]), T["sig_t0"] - T["t0"], np.nan)
    tiempos = pd.DataFrame({"falta": T["falta_t"], "recuperacion": T["rec_t"], "contragolpe": T["cg_t"]})
    tiempos = tiempos.where(tiempos <= N_TR).fillna(np.inf)
    res = tiempos.idxmin(axis=1).where(np.isfinite(tiempos.min(axis=1)), "asienta")
    hay_cp = T["cp_t"].notna()
    zona = tercio(T["cp_x"])
    temprana = T["falta_t"] <= P["falta_tactica_sens_s"]
    partes.append(agrupa(T, T["L"], dL, {
        "td_n": 1, "td_contrapresion": hay_cp, "td_cp_grupal": hay_cp & (T["cp_jug"] >= 2),
        "td_cp_alta": hay_cp & (zona == "alta"), "td_cp_media": hay_cp & (zona == "media"),
        "td_cp_baja": hay_cp & (zona == "baja"),
        "td_res_recuperacion": res == "recuperacion", "td_res_falta": res == "falta",
        "td_res_contragolpe_rival": res == "contragolpe", "td_res_asienta": res == "asienta",
        "td_res_falta_6s": (res == "falta") & temprana,
        "td_falta_temprana": temprana,
        "td_falta_temprana_sin_presion": temprana & (T["pres_t"].isna() | (T["pres_t"] > T["falta_t"]))}))

    C = pd.concat(partes, axis=1).fillna(0).reset_index()
    C.insert(0, "temporada", temp)
    nombres = E.drop_duplicates("team.id").set_index("team.id")["team.name"]
    C.insert(3, "equipo", C["equipo_id"].map(nombres))
    return C


titulo("R2b · Calculando presión y transición defensiva")
temporadas = sorted(pd.read_parquet(os.path.join(DP, "partido_tecnico_periodo.parquet"))["temporada"].unique())
for temp in temporadas:
    t0 = time.time()
    if os.path.exists(archivo("presion", temp)) and not REHACER:
        print(f"  {temp}: ya existe, se omite")
        continue
    C = presion_temp(temp)
    C.to_parquet(archivo("presion", temp), index=False)
    print(f"  {temp}: {len(C):,} filas · {C['match_id'].nunique()} partidos · {time.time() - t0:.0f} s")


# --------------------------------------------------------------------------------------
# Bloque 2 · Confiabilidad entre mitades (N-1) — función reutilizable
# --------------------------------------------------------------------------------------
def confiabilidad(M, num, den, reps=P["repeticiones"], min_p=P["etapa_min"]):
    """M: una fila por equipo-partido con columna etapa_id. Devuelve (confiabilidad media, n etapas)."""
    rng = np.random.default_rng(SEMILLA)
    grupos = [(g[num].to_numpy(float), g[den].to_numpy(float))
              for _, g in M.groupby("etapa_id") if len(g) >= min_p]
    if len(grupos) < 5:
        return np.nan, len(grupos)
    rs = []
    for _ in range(reps):
        a, b = [], []
        for nu, de in grupos:
            idx = rng.permutation(len(nu))
            h = len(nu) // 2
            i1, i2 = idx[:h], idx[h:2 * h]
            if de[i1].sum() == 0 or de[i2].sum() == 0:
                continue
            a.append(nu[i1].sum() / de[i1].sum())
            b.append(nu[i2].sum() / de[i2].sum())
        r = np.corrcoef(a, b)[0, 1]
        rs.append(2 * r / (1 + r))
    return float(np.mean(rs)), len(grupos)


titulo("R2b · Evidencia para elegir X (solo entrenamiento)")
C = pd.concat([pd.read_parquet(archivo("presion", t)) for t in TEMPORADAS["entrenamiento"]], ignore_index=True)
M = C.groupby(["temporada", "match_id", "equipo"]).sum(numeric_only=True).reset_index()
elo = pd.read_parquet(os.path.join(DP, "equipo_partido_elo_v1.parquet"))[["match_id", "team", "etapa_id"]]
M = M.merge(elo, left_on=["match_id", "equipo"], right_on=["match_id", "team"], how="left")
print(f"  equipo-partidos: {len(M)} · sin etapa asignada: {M['etapa_id'].isna().sum()} · "
      f"etapas con ≥{P['etapa_min']} partidos en entrenamiento: {(M.groupby('etapa_id').size() >= P['etapa_min']).sum()}")

filas = []
for X in XS:
    S = M[[c for c in M.columns if c.endswith(f"_x{X}")]].sum()
    fila = {"X (s)": X,
            "fases/eq-partido": S[f"fases_x{X}"] / len(M),
            "presiones/fase": S[f"pres_x{X}"] / S[f"fases_x{X}"],
            "% grupal": 100 * S[f"grupal_x{X}"] / S[f"fases_x{X}"],
            "% éxito": 100 * S[f"exito_x{X}"] / S[f"fases_x{X}"],
            "% alta": 100 * S[f"alta_x{X}"] / S[f"fases_x{X}"],
            "% media": 100 * S[f"media_x{X}"] / S[f"fases_x{X}"],
            "% baja": 100 * S[f"baja_x{X}"] / S[f"fases_x{X}"]}
    for nombre, num in [("conf % alta", f"alta_x{X}"), ("conf % grupal", f"grupal_x{X}"), ("conf % éxito", f"exito_x{X}")]:
        fila[nombre], n_et = confiabilidad(M, num, f"fases_x{X}")
    filas.append(fila)
print(pd.DataFrame(filas).set_index("X (s)").round(3).T.to_string())
r_ev, _ = confiabilidad(M, "ev_alta", "ev_pres")
print(f"\n  Sensibilidad M4 con eventos (no fases): % alta liga {100 * M['ev_alta'].sum() / M['ev_pres'].sum():.1f} · confiabilidad {r_ev:.3f}")

titulo("R2b · M6 transición defensiva (entrenamiento)")
S = M.sum(numeric_only=True)
n = S["td_n"]
print(f"  transiciones defensivas: {n:,.0f}  ({n / len(M):.1f} por equipo-partido)")
print(f"  Capa 1 · contrapresión {100 * S['td_contrapresion'] / n:.1f}% · repliegue {100 - 100 * S['td_contrapresion'] / n:.1f}%")
print(f"           de las contrapresiones: grupal {100 * S['td_cp_grupal'] / S['td_contrapresion']:.1f}% · "
      f"alta {100 * S['td_cp_alta'] / S['td_contrapresion']:.1f}% · media {100 * S['td_cp_media'] / S['td_contrapresion']:.1f}% · "
      f"baja {100 * S['td_cp_baja'] / S['td_contrapresion']:.1f}%")
print(f"  Capa 2 · recuperamos {100 * S['td_res_recuperacion'] / n:.1f}% · falta {100 * S['td_res_falta'] / n:.1f}% · "
      f"contragolpe rival {100 * S['td_res_contragolpe_rival'] / n:.1f}% · rival asienta {100 * S['td_res_asienta'] / n:.1f}%")
print(f"           falta táctica solo ≤{P['falta_tactica_sens_s']} s: {100 * S['td_res_falta_6s'] / n:.1f}%")
print(f"  Tu pregunta · faltas tempranas (≤{P['falta_tactica_sens_s']} s) sin presión previa: "
      f"{100 * S['td_falta_temprana_sin_presion'] / max(S['td_falta_temprana'], 1):.1f}% de {S['td_falta_temprana']:,.0f}")
for nombre, num in [("% contrapresión", "td_contrapresion"), ("% recuperamos", "td_res_recuperacion"),
                    ("% falta", "td_res_falta"), ("% contragolpe rival", "td_res_contragolpe_rival")]:
    r, n_et = confiabilidad(M, num, "td_n")
    print(f"  confiabilidad {nombre:22s} {r:.3f}   ({n_et} etapas)")
print("\nFIN R2b")

# Sesión 5

R3 - ELO-1.1 (se actualizo a ELO 2)

In [ ]:
# =====================================================================
# R3 · Elo y capa predictiva (framework v1.1, sección 7)
# Sesión 5 · 2026-10-09 · Hackathon ISAC 2026, Track 2
# Cuatro celdas: R3-0 (funciones), R3a (Elo base), R3b (grilla), R3c (selección)
# Antes: celda 1 (Drive) y R0.
# REGLA DE HOLDOUT: este código NUNCA imprime resultados ni frecuencias de
# 2025/26 ni de 2026/27. Solo calcula el Elo prepartido de esos partidos.
# =====================================================================

# [R3-0] Configuración y funciones ---------------------------------
import os, re, glob, json, gzip, time, warnings
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit

SEMILLA = 2026
BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026/')
DP = os.path.join(BASE, 'datos_parquet')
FW = os.path.join(DP, 'fw1')
DC = os.path.join(BASE, 'datos_crudos')
SAL = os.path.join(BASE, 'salidas')
os.makedirs(FW, exist_ok=True); os.makedirs(SAL, exist_ok=True)
REHACER = False

# E-2 (decidido)
E2 = dict(K=20, H=75, R_ver=0.25, R_inv=0.0)   # H = 75 decidido por Arturo (S5-08, sesión 5)
H_ELO1 = 150                                     # H de ELO-1: solo para la referencia de comparación
INICIO = 1500.0
CALENTAMIENTO = pd.Timestamp('2021-10-01')

# Grillas que cubren TODAS las opciones de S5-01 a S5-04
VIDAS_MEDIAS = [90, 180, 365, 730]          # S5-01 (días)
CALIBRADORES = ['A'] + [f'B{h}' for h in VIDAS_MEDIAS] + ['Ca', 'Cb']   # S5-02
R_VER_GRID = [0.15, 0.20, 0.25]             # S5-03
R_INV_GRID = [0.0, 0.05, 0.10, 0.15]        # S5-03
H_TIPOS = ['universal', 'estadio']          # S5-04
MODOS = ['congelado', 'wf']                 # S5-05

# Particiones (por orden cronológico de temporadas, ver R3a)
#   índice 0 = 2021/22, 1 = 2022/23, 2 = 2023/24, 3 = 2024/25, 4 = 2025/26 (sellada), 5 = 2026/27 (en vivo)
SPLITS = {'interno': dict(calib_hasta=1, evalua=2),   # elegir
          'externo': dict(calib_hasta=2, evalua=3)}   # estimar
TEMPORADAS_PROHIBIDAS = {4, 5}


# ---------- Carga de partidos ----------
def _leer_matches(ruta):
    if ruta.endswith('.parquet'):
        df = pd.read_parquet(ruta)
        obj = [c for c in df.columns if df[c].dtype == object]
        if any(df[c].map(lambda v: isinstance(v, dict)).any() for c in obj):
            df = pd.json_normalize(df.to_dict('records'))
    else:
        abrir = gzip.open if ruta.endswith('.gz') else open
        with abrir(ruta, 'rt', encoding='utf-8') as f:
            df = pd.json_normalize(json.load(f))
    m = re.search(r'(\d{4}-\d{2}-\d{2})', os.path.basename(ruta))
    df['_fecha_archivo'] = m.group(1) if m else '0000-00-00'
    df['_archivo'] = os.path.basename(ruta)
    return df

def _col(df, opciones):
    for o in opciones:
        if o in df.columns:
            return o
    return None

CANDIDATAS = {
    'match_id': ['match_id'],
    'fecha_m': ['match_date'],
    'hora': ['kick_off'],
    'local_id': ['home_team.home_team_id', 'home_team_id'],
    'local': ['home_team.home_team_name', 'home_team_name'],
    'visita_id': ['away_team.away_team_id', 'away_team_id'],
    'visita': ['away_team.away_team_name', 'away_team_name'],
    'gl': ['home_score'], 'gv': ['away_score'],
    'estadio': ['stadium.name', 'stadium_name', 'stadium'],
}

def cargar_partidos():
    rutas = sorted(glob.glob(os.path.join(DC, 'matches_73_*.json*')) +
                   glob.glob(os.path.join(DP, 'matches_73_*.parquet')))
    piezas = []
    for r in rutas:
        try:
            df = _leer_matches(r)
        except Exception as e:
            print(f'  aviso: no se pudo leer {os.path.basename(r)}: {e}')
            continue
        cols = {k: _col(df, v) for k, v in CANDIDATAS.items()}
        faltan = [k for k in ['match_id', 'local', 'visita', 'gl', 'gv'] if cols[k] is None]
        if faltan:
            print(f'  aviso: {os.path.basename(r)} sin columnas {faltan}; se omite')
            continue
        out = pd.DataFrame({k: (df[c] if c else np.nan) for k, c in cols.items()})
        if out['estadio'].map(lambda v: isinstance(v, dict)).any():
            out['estadio'] = out['estadio'].map(lambda v: v.get('name') if isinstance(v, dict) else v)
        out['_fecha_archivo'] = df['_fecha_archivo']; out['_archivo'] = df['_archivo']
        piezas.append(out)
    M = pd.concat(piezas, ignore_index=True)
    M = (M.sort_values('_fecha_archivo', ascending=False)
           .drop_duplicates('match_id', keep='first'))
    return M, rutas


# ---------- Elo ----------
def correr_elo(P, K, H, R_ver, R_inv, H_est=None):
    """Recorre los partidos en orden. Devuelve Elo pre y post de local y visita, y la H usada."""
    rating = {}
    temp_act = torneo_act = None
    pre_l = np.empty(len(P)); pre_v = np.empty(len(P))
    post_l = np.empty(len(P)); post_v = np.empty(len(P)); h_us = np.empty(len(P))
    for i, r in enumerate(P.itertuples(index=False)):
        if temp_act is not None and r.temporada != temp_act:          # reinicio de verano
            for t in rating: rating[t] = INICIO + (1 - R_ver) * (rating[t] - INICIO)
        elif torneo_act is not None and r.torneo != torneo_act:       # reinicio de invierno
            for t in rating: rating[t] = INICIO + (1 - R_inv) * (rating[t] - INICIO)
        temp_act, torneo_act = r.temporada, r.torneo
        rl = rating.get(r.local_id, INICIO); rv = rating.get(r.visita_id, INICIO)
        h = H + (H_est.get(r.estadio, 0.0) if H_est else 0.0)
        E = 1.0 / (1.0 + 10 ** (-(rl + h - rv) / 400))
        dif = r.gl - r.gv
        S = 1.0 if dif > 0 else (0.5 if dif == 0 else 0.0)
        n = abs(dif)
        G = 1.0 if n <= 1 else (1.5 if n == 2 else (11 + n) / 8)     # escala de eloratings
        delta = K * G * (S - E)
        rating[r.local_id] = rl + delta; rating[r.visita_id] = rv - delta
        pre_l[i], pre_v[i], post_l[i], post_v[i], h_us[i] = rl, rv, rl + delta, rv - delta, h
    return pre_l, pre_v, post_l, post_v, h_us


def estimar_H_estadio(P, pre_l, pre_v, H, mascara):
    """Ventaja local extra por estadio, con contracción (empirical Bayes). Solo usa filas de 'mascara'."""
    d = P.loc[mascara].copy()
    E = 1 / (1 + 10 ** (-(pre_l[mascara] + H - pre_v[mascara]) / 400))
    S = np.where(d.gl > d.gv, 1.0, np.where(d.gl == d.gv, 0.5, 0.0))
    d['r'] = S - E
    d['deriv'] = np.log(10) / 400 * E * (1 - E)
    var_r = d['r'].var()
    g = d.groupby('estadio').agg(n=('r', 'size'), rbar=('r', 'mean'), dbar=('deriv', 'mean'))
    g['rbar_c'] = g['rbar'] - d['r'].mean()      # desviación respecto al promedio de la liga
    g['se2'] = var_r / g['n']
    tau2 = max(0.0, g['rbar_c'].var() - g['se2'].mean())
    g['k'] = tau2 / (tau2 + g['se2']) if tau2 > 0 else 0.0
    g['dH'] = g['k'] * g['rbar_c'] / g['dbar']
    return g['dH'].to_dict(), g, np.sqrt(tau2) / g['dbar'].mean() if tau2 > 0 else 0.0


# ---------- Calibración ordenada ----------
NPAR = {'A': 3, 'B': 3, 'Ca': 4, 'Cb': 5}

def _tipo(cal):
    return 'B' if cal.startswith('B') else cal

def probs_cal(p, x, t, tipo):
    """Columnas: 0 = gana visita, 1 = empate, 2 = gana local."""
    b, c1, lg = p[0], p[1], p[2]
    c1t = c1 + (p[3] * t if tipo in ('Ca', 'Cb') else 0.0)
    lgt = lg + (p[4] * t if tipo == 'Cb' else 0.0)
    c2t = c1t + np.exp(lgt)
    z = b * x
    pv = expit(c1t - z); pve = expit(c2t - z)
    Pm = np.column_stack([pv, pve - pv, 1 - pve])
    Pm = np.clip(Pm, 1e-9, 1); return Pm / Pm.sum(1, keepdims=True)

def ajustar_cal(x, y, t, w, tipo, init=None):
    k = NPAR[tipo]
    p0 = np.array([2.0, -0.9, 0.17, 0.0, 0.0])[:k] if init is None or len(init) != k else init
    idx = np.arange(len(y))
    def nll(p):
        return -(w * np.log(probs_cal(p, x, t, tipo)[idx, y])).sum() / w.sum()
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        res = minimize(nll, p0, method='BFGS')
    return res.x, bool(res.success)

def pesos(cal, t_fit, t_ref):
    if cal.startswith('B'):
        hl = float(cal[1:])
        return 0.5 ** ((t_ref - t_fit) * 365.25 / hl)
    return np.ones_like(t_fit)


def evaluar(P, x, cal, modo, split):
    """Probabilidades para la temporada de evaluación del split. Solo usa información previa."""
    s = SPLITS[split]
    assert s['evalua'] not in TEMPORADAS_PROHIBIDAS, 'Prohibido evaluar en 2025/26 o 2026/27'
    y = P['y'].values; t = P['t'].values; sidx = P['sidx'].values; fecha = P['fecha'].values
    elegible = (P['fecha'] >= CALENTAMIENTO).values
    base_fit = elegible & (sidx <= s['calib_hasta'])
    ev = sidx == s['evalua']
    out = np.full((len(P), 3), np.nan); fallos = 0
    if cal == 'freq':
        f = np.bincount(y[base_fit], minlength=3) / base_fit.sum()
        out[ev] = f; return out[ev], y[ev], 0
    tipo = _tipo(cal)
    if modo == 'congelado':
        tref = t[base_fit].max()
        p, ok = ajustar_cal(x[base_fit], y[base_fit], t[base_fit], pesos(cal, t[base_fit], tref), tipo)
        fallos += (not ok)
        out[ev] = probs_cal(p, x[ev], t[ev], tipo)
    else:  # wf: recalibra antes de cada fecha con todo lo anterior
        p = None
        for f_ in np.unique(fecha[ev]):
            fit = base_fit | (ev & (fecha < f_))
            hoy = ev & (fecha == f_)
            tref = t[hoy].min()
            p, ok = ajustar_cal(x[fit], y[fit], t[fit], pesos(cal, t[fit], tref), tipo, init=p)
            fallos += (not ok)
            out[hoy] = probs_cal(p, x[hoy], t[hoy], tipo)
    return out[ev], y[ev], fallos


def metricas(Pm, y):
    ll = -np.mean(np.log(Pm[np.arange(len(y)), y]))
    O = np.eye(3)[y]
    br = np.mean(((Pm - O) ** 2).sum(1))
    return ll, br

print('R3-0 listo. BASE =', BASE)


# [R3a] Partidos, Elo base (E-2), chequeos y guardado ---------------
t0 = time.time()
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp.columns:
    ptp = ptp[ptp['jugado'].astype(bool)]
ptp['fecha'] = pd.to_datetime(ptp['fecha'])
uni = (ptp.groupby('match_id').agg(fecha=('fecha', 'first'), temporada=('temporada', 'first'),
                                   n_filas=('team', 'size')).reset_index())
loc_ptp = ptp[ptp['lado'] == 'home'][['match_id', 'team']].rename(columns={'team': 'local_ptp'})

M, rutas = cargar_partidos()
print(f'Archivos de partidos leídos: {len(rutas)}')
P = uni.merge(M, on='match_id', how='left').merge(loc_ptp, on='match_id', how='left')
P['hora'] = P['hora'].fillna('00:00:00.000')
P = P.sort_values(['fecha', 'hora', 'match_id']).reset_index(drop=True)
P['torneo'] = np.where(P['fecha'].dt.month >= 7, 'Apertura', 'Clausura')
orden = P.groupby('temporada')['fecha'].min().sort_values().index.tolist()
P['sidx'] = P['temporada'].map({s: i for i, s in enumerate(orden)})
P['t'] = (P['fecha'] - CALENTAMIENTO).dt.days / 365.25
P['y'] = np.where(P.gl > P.gv, 2, np.where(P.gl == P.gv, 1, 0))
for c in ['local_id', 'visita_id']:
    if P[c].isna().all():
        P[c] = P[c.replace('_id', '')]
P['estadio'] = P['estadio'].fillna('(sin estadio)')

print('\n--- Chequeos de carga ---')
print('Partidos:', len(P), '| sin marcador:', int(P[['gl', 'gv']].isna().any(axis=1).sum()),
      '| filas ptp distintas de 2:', int((P['n_filas'] != 2).sum()))
print('Local de Matches distinto al de ptp:', int((P['local'] != P['local_ptp']).sum()))
tab = P.groupby('sidx').agg(temporada=('temporada', 'first'), partidos=('match_id', 'size'),
                            desde=('fecha', 'min'), hasta=('fecha', 'max'))
tab['desde'] = tab['desde'].dt.date; tab['hasta'] = tab['hasta'].dt.date
print(tab.to_string())
print('Equipos distintos:', P['local_id'].nunique(), '| estadios distintos:', P['estadio'].nunique())

print('\n--- Resultados por temporada (SOLO entrenamiento, índices 0-3) ---')
ent = P[P['sidx'] <= 3]
fr = (ent.groupby('sidx')['y'].value_counts(normalize=True).unstack().reindex(columns=[2, 1, 0]) * 100)
fr.columns = ['% local', '% empate', '% visita']
fr['goles/partido'] = ent.assign(g=ent.gl + ent.gv).groupby('sidx')['g'].mean()
print(fr.round(1).to_string())

pre_l, pre_v, post_l, post_v, h_us = correr_elo(P, **E2)
P['elo_local_pre'], P['elo_visita_pre'] = pre_l, pre_v
P['elo_local_post'], P['elo_visita_post'] = post_l, post_v
P['H'] = h_us
P['d'] = P['elo_local_pre'] + P['H'] - P['elo_visita_pre']
P['E_local'] = 1 / (1 + 10 ** (-P['d'] / 400))

print('\n--- Elo base (E-2) ---')
fin3 = P[P['sidx'] <= 3]
ult = pd.concat([fin3[['fecha', 'local', 'elo_local_post']].rename(columns={'local': 'eq', 'elo_local_post': 'elo'}),
                 fin3[['fecha', 'visita', 'elo_visita_post']].rename(columns={'visita': 'eq', 'elo_visita_post': 'elo'})])
ult = ult.sort_values('fecha').groupby('eq').tail(1).sort_values('elo', ascending=False)
print(f'Al cierre de 2024/25: media {ult.elo.mean():.0f}, desv. est. {ult.elo.std():.0f}, '
      f'rango {ult.elo.min():.0f} a {ult.elo.max():.0f}')
print('Top 3:', ', '.join(f'{r.eq} {r.elo:.0f}' for r in ult.head(3).itertuples()))
print('Últimos 3:', ', '.join(f'{r.eq} {r.elo:.0f}' for r in ult.tail(3).itertuples()))
print('E medio del local por temporada (0-3):', P[P.sidx <= 3].groupby('sidx')['E_local'].mean().round(3).to_dict())

cols_out = ['match_id', 'fecha', 'temporada', 'sidx', 'torneo', 'local', 'visita', 'local_id', 'visita_id',
            'estadio', 'gl', 'gv', 'elo_local_pre', 'elo_visita_pre', 'elo_local_post', 'elo_visita_post', 'H', 'd']
P[cols_out].to_parquet(os.path.join(FW, 'elo_partido_e2.parquet'), index=False)
largo = pd.concat([
    P.assign(team=P.local, lado='home', elo_pre=P.elo_local_pre, elo_post=P.elo_local_post, elo_rival_pre=P.elo_visita_pre),
    P.assign(team=P.visita, lado='away', elo_pre=P.elo_visita_pre, elo_post=P.elo_visita_post, elo_rival_pre=P.elo_local_pre)
])[['match_id', 'fecha', 'temporada', 'sidx', 'team', 'lado', 'elo_pre', 'elo_post', 'elo_rival_pre']]
largo.to_parquet(os.path.join(FW, 'elo_equipo_partido_e2.parquet'), index=False)

print('\n--- América por etapa de técnico (solo hasta 2024/25) ---')
am = ptp[ptp['team'].str.contains('Am[eé]rica', case=False, regex=True)][['match_id', 'team', 'tec_p', 'etapa']]
am = am.merge(largo, on=['match_id', 'team'])
am = am[am['sidx'] <= 3].sort_values('fecha')
res_am = am.groupby('etapa').agg(tecnico=('tec_p', 'first'), partidos=('match_id', 'size'),
                                 desde=('fecha', 'min'), elo_inicio=('elo_pre', 'first'),
                                 elo_fin=('elo_post', 'last'), elo_medio=('elo_pre', 'mean')).sort_values('desde')
res_am['cambio'] = res_am['elo_fin'] - res_am['elo_inicio']
res_am['desde'] = res_am['desde'].dt.date
print(res_am.round(0).to_string())
print(f'\nR3a terminó en {time.time() - t0:.0f} s. Guardado: fw1/elo_partido_e2.parquet y fw1/elo_equipo_partido_e2.parquet')


# [R3b] Grilla completa: Elo (R de verano, R de invierno, H) x calibrador x modo x split
ruta_grid = os.path.join(SAL, f"r3_grilla_H{E2['H']}.csv")
if os.path.exists(ruta_grid) and not REHACER:
    G = pd.read_csv(ruta_grid); print('Grilla cargada de caché:', len(G), 'filas')
else:
    t0 = time.time(); filas = []
    for split in SPLITS:
        s = SPLITS[split]
        mask_fit_H = ((P['fecha'] >= CALENTAMIENTO) & (P['sidx'] <= s['calib_hasta'])).values
        for rv in R_VER_GRID:
            for ri in R_INV_GRID:
                base = correr_elo(P, E2['K'], E2['H'], rv, ri)
                for ht in H_TIPOS:
                    if ht == 'universal':
                        pl, pv_ = base[0], base[1]; hh = np.full(len(P), E2['H'])
                    else:
                        dH, _, _ = estimar_H_estadio(P, base[0], base[1], E2['H'], mask_fit_H)
                        r2 = correr_elo(P, E2['K'], E2['H'], rv, ri, H_est=dH)
                        pl, pv_, hh = r2[0], r2[1], r2[4]
                    x = (pl + hh - pv_) / 400
                    for cal in ['freq'] + CALIBRADORES:
                        for modo in (['congelado'] if cal == 'freq' else MODOS):
                            Pm, y, fallos = evaluar(P, x, cal, modo, split)
                            ll, br = metricas(Pm, y)
                            filas.append(dict(split=split, R_ver=rv, R_inv=ri, H=ht, cal=cal, modo=modo,
                                              n=len(y), logloss=ll, brier=br, fallos=fallos))
        print(f'  split {split} listo ({time.time() - t0:.0f} s)')
    G = pd.DataFrame(filas); G.to_csv(ruta_grid, index=False)
    print(f'R3b terminó en {time.time() - t0:.0f} s. Filas: {len(G)}. Fallos de ajuste totales: {int(G.fallos.sum())}')

# Vista 1: calibradores con el Elo de E-2 (R 0.25 / 0, H universal)
v1 = G[(G.R_ver == 0.25) & (G.R_inv == 0.0) & (G.H == 'universal')]
v1 = v1.pivot_table(index=['cal', 'modo'], columns='split', values=['logloss', 'brier'])
v1.columns = [f'{a}_{b}' for a, b in v1.columns]
print('\n--- Vista 1: calibradores con el Elo de E-2 ---')
print(v1[['logloss_interno', 'logloss_externo', 'brier_interno', 'brier_externo']].round(4)
      .sort_values('logloss_interno').to_string())
print(f"Elo de esta corrida: H = {E2['H']}. " + 'Referencias 2024/25 (ELO-1, H = 150): frecuencias 1.0526 / 0.6346 · Elo 0.9785 / 0.5803 · 1/3 1.0986 / 0.6667')


# [R3c] Selección con la regla preregistrada (S5-05 y S5-06) --------
MODO_PRINCIPAL = 'congelado'   # decidido por Arturo (S5-05, sesión 5)
UMBRAL = 0.003          # [PROVISIONAL-Claude hasta S5-06]
N_BOOT = 2000

def ll_int(f):
    q = G[(G.split == 'interno')]
    for k, v in f.items(): q = q[q[k] == v]
    return float(q.logloss.iloc[0])
def ll_ext(f):
    q = G[(G.split == 'externo')]
    for k, v in f.items(): q = q[q[k] == v]
    return float(q.logloss.iloc[0])

pasos = []
cfg = dict(R_ver=0.25, R_inv=0.0, H='universal', modo=MODO_PRINCIPAL)
# Paso 1: vida media de B (pieza decidida: se elige la mejor sin umbral)
cand = {c: ll_int({**cfg, 'cal': c}) for c in CALIBRADORES if c.startswith('B')}
mejor_B = min(cand, key=cand.get); cfg['cal'] = mejor_B
pasos.append(('1 vida media de B', mejor_B, cand[mejor_B], ll_ext(cfg), 'elegida (sin umbral)'))
# Paso 2: tendencia C contra B (entra solo con mejora >= UMBRAL)
for c in ['Ca', 'Cb']:
    li = ll_int({**cfg, 'cal': c}); entra = (ll_int(cfg) - li) >= UMBRAL
    pasos.append((f'2 tendencia {c}', c, li, ll_ext({**cfg, 'cal': c}), f'mejora {ll_int(cfg) - li:+.4f}: ' + ('ENTRA' if entra else 'no entra')))
    if entra and li < ll_int(cfg): cfg['cal'] = c
# Paso 3: reinicios (verano, invierno)
ref = ll_int(cfg); mejor = None
for rv in R_VER_GRID:
    for ri in R_INV_GRID:
        li = ll_int({**cfg, 'R_ver': rv, 'R_inv': ri})
        if mejor is None or li < mejor[2]: mejor = (rv, ri, li)
entra = (ref - mejor[2]) >= UMBRAL
pasos.append(('3 reinicios', f'R_ver {mejor[0]} / R_inv {mejor[1]}', mejor[2],
              ll_ext({**cfg, 'R_ver': mejor[0], 'R_inv': mejor[1]}), f'mejora {ref - mejor[2]:+.4f}: ' + ('ENTRA' if entra else 'no entra')))
if entra: cfg['R_ver'], cfg['R_inv'] = mejor[0], mejor[1]
# Paso 4: H por estadio
li = ll_int({**cfg, 'H': 'estadio'}); entra = (ll_int(cfg) - li) >= UMBRAL
pasos.append(('4 H por estadio', 'estadio', li, ll_ext({**cfg, 'H': 'estadio'}), f'mejora {ll_int(cfg) - li:+.4f}: ' + ('ENTRA' if entra else 'no entra')))
if entra: cfg['H'] = 'estadio'

print(f'--- Selección (modo {MODO_PRINCIPAL}, umbral {UMBRAL}) ---')
print(pd.DataFrame(pasos, columns=['paso', 'opción', 'logloss_interno', 'logloss_externo', 'resultado']).round(4).to_string(index=False))
print('Configuración elegida:', cfg)

# Vista 2: los 8 mejores Elo (R y H) con el calibrador elegido, para ver la meseta
v2 = G[(G.split == 'interno') & (G.cal == cfg['cal']) & (G.modo == cfg['modo'])].sort_values('logloss')
v2 = v2.merge(G[G.split == 'externo'][['R_ver', 'R_inv', 'H', 'cal', 'modo', 'logloss']],
              on=['R_ver', 'R_inv', 'H', 'cal', 'modo'], suffixes=('_int', '_ext'))
print('\n--- Vista 2: meseta de variantes de Elo (calibrador elegido) ---')
print(v2[['R_ver', 'R_inv', 'H', 'logloss_int', 'logloss_ext']].head(8).round(4).to_string(index=False))
print('Rango del log-loss interno entre las 24 variantes:', round(v2.logloss_int.max() - v2.logloss_int.min(), 4))

# Reproducir la configuración elegida y la referencia tipo ELO-1 (E-2 + A congelado) en 2024/25
def correr_cfg(c, split, H=None):
    H = E2['H'] if H is None else H
    s = SPLITS[split]
    base = correr_elo(P, E2['K'], H, c['R_ver'], c['R_inv'])
    if c['H'] == 'estadio':
        m = ((P['fecha'] >= CALENTAMIENTO) & (P['sidx'] <= s['calib_hasta'])).values
        dH, tabH, sdH = estimar_H_estadio(P, base[0], base[1], H, m)
        base = correr_elo(P, E2['K'], H, c['R_ver'], c['R_inv'], H_est=dH)
    x = (base[0] + base[4] - base[1]) / 400
    Pm, y, _ = evaluar(P, x, c['cal'], c['modo'], split)
    return Pm, y

Pm_sel, y_ext = correr_cfg(cfg, 'externo')
Pm_ref, _ = correr_cfg(dict(R_ver=0.25, R_inv=0.0, H='universal', cal='A', modo='congelado'), 'externo', H=H_ELO1)
Pm_frq, _ = evaluar(P, np.zeros(len(P)), 'freq', 'congelado', 'externo')[:2]

print('\n--- 2024/25: elegida contra referencias ---')
for nom, Pm in [('elegida', Pm_sel), ('ELO-1 (H 150 + A congelado)', Pm_ref), ('frecuencias', Pm_frq)]:
    ll, br = metricas(Pm, y_ext); print(f'{nom:32s} log-loss {ll:.4f} · Brier {br:.4f}')

rng = np.random.default_rng(SEMILLA)
li_sel = -np.log(Pm_sel[np.arange(len(y_ext)), y_ext]); li_ref = -np.log(Pm_ref[np.arange(len(y_ext)), y_ext])
li_frq = -np.log(Pm_frq[np.arange(len(y_ext)), y_ext])
def ic(a, b):
    dif = a - b; bs = [dif[rng.integers(0, len(dif), len(dif))].mean() for _ in range(N_BOOT)]
    return dif.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5)
for nom, b in [('elegida - tipo ELO-1', li_ref), ('elegida - frecuencias', li_frq)]:
    m, lo, hi = ic(li_sel, b); print(f'{nom:24s} diferencia {m:+.4f}  IC95% [{lo:+.4f}, {hi:+.4f}]  (negativo = elegida mejor)')

print('\n--- Calibración en 2024/25 (elegida): nivel por resultado ---')
obs = np.bincount(y_ext, minlength=3) / len(y_ext)
cal_tab = pd.DataFrame({'predicho %': Pm_sel.mean(0) * 100, 'observado %': obs * 100},
                       index=['visita', 'empate', 'local'])
cal_tab['tipo ELO-1 predicho %'] = Pm_ref.mean(0) * 100
print(cal_tab.round(1).to_string())

print('\n--- Calibración por quintiles de P(local) (elegida) ---')
q = pd.qcut(Pm_sel[:, 2], 5, labels=False, duplicates='drop')
ct = pd.DataFrame({'q': q, 'pred': Pm_sel[:, 2], 'obs': (y_ext == 2).astype(float)}).groupby('q').agg(
    n=('obs', 'size'), pred_local=('pred', 'mean'), obs_local=('obs', 'mean'))
print((ct.assign(pred_local=ct.pred_local * 100, obs_local=ct.obs_local * 100)).round(1).to_string())

if True:  # diagnóstico de H por estadio, siempre (E-5)
    m = ((P['fecha'] >= CALENTAMIENTO) & (P['sidx'] <= 2)).values
    b0 = correr_elo(P, E2['K'], E2['H'], cfg['R_ver'], cfg['R_inv'])
    dH, tabH, sdH = estimar_H_estadio(P, b0[0], b0[1], E2['H'], m)
    tabH = tabH.sort_values('dH')
    print(f'\n--- Diagnóstico H por estadio (calibración hasta 2023/24) · dispersión real estimada: {sdH:.0f} puntos ---')
    print('Más ventaja que el promedio (dH en puntos de Elo):'); print(tabH.tail(4)[['n', 'k', 'dH']].round(2).to_string())
    print('Menos ventaja que el promedio:'); print(tabH.head(4)[['n', 'k', 'dH']].round(2).to_string())
print('\nR3c listo. La configuración queda PROVISIONAL hasta que Arturo cierre S5-01 a S5-06.')


# [R3d] Diagnóstico del Elo (solo 2021/22 a 2024/25) ---------------- Requiere
# R3-0, R3a y R3b en memoria (la grilla se lee de caché). Tarda ~2 min.
# Nada de esta celda elige parámetros: mide por qué salió lo que salió.
from scipy.optimize import brentq
t0 = time.time()
CANDIDATAS['fase'] = ['competition_stage.name', 'competition_stage']
M2, _ = cargar_partidos()
P['fase_nombre'] = P['match_id'].map(M2.drop_duplicates('match_id').set_index('match_id')['fase'])
_fn = P['fase_nombre'].fillna('').astype(str).str.strip()
P['fase_tipo'] = np.select(
    [_fn.isin(['Apertura', 'Clausura', 'Regular Season']),
     _fn.str.contains('Reclasificaci|Play-In', case=False, regex=True)],
    ['regular', 'play-in'], default='liguilla')
ent = P[P['sidx'] <= 3].copy()

print('--- (a) Nombres de fase en entrenamiento: ¿"regular" quedó bien separado? ---')
print(ent.groupby(['fase_nombre', 'fase_tipo'], dropna=False).size().to_string())

print('\n--- (b) Resultados por temporada, torneo y fase ---')
tb = pd.crosstab([ent.sidx, ent.torneo, ent.fase_tipo], ent.y, normalize='index').reindex(columns=[2, 1, 0]) * 100
tb.columns = ['% local', '% empate', '% visita']
tb.insert(0, 'n', ent.groupby(['sidx', 'torneo', 'fase_tipo']).size().astype(int))
print(tb.round(1).to_string())

x0 = ((P['elo_local_pre'] + P['H'] - P['elo_visita_pre']) / 400).values
def _ll_vec(x, cal, modo, split):
    Pm, y, _ = evaluar(P, x, cal, modo, split)
    return -np.log(Pm[np.arange(len(y)), y])

print('\n--- (c) Recalibrar menos congelado, por tramo (en milésimas; POSITIVO = recalibrar fue PEOR) ---')
filas = []
for split in SPLITS:
    ev = P[P['sidx'] == SPLITS[split]['evalua']].copy()
    ev['tramo'] = ev['torneo'] + '-' + ev['fase_tipo']
    for cal in ['A', 'B365', 'B90', 'Ca']:
        ev['dif'] = _ll_vec(x0, cal, 'wf', split) - _ll_vec(x0, cal, 'congelado', split)
        g = (ev.groupby('tramo')['dif'].mean() * 1000).round(1).to_dict()
        filas.append({'split': split, 'cal': cal, **g, 'total': round(ev['dif'].mean() * 1000, 1)})
print(pd.DataFrame(filas).to_string(index=False))
for split in SPLITS:
    ev = P[P['sidx'] == SPLITS[split]['evalua']]
    print(f'  n por tramo ({split}):', (ev['torneo'] + '-' + ev['fase_tipo']).value_counts().to_dict())

print('\n--- (d) Ventaja local: la que usa el Elo frente a la real ---')
S_obs = np.where(P.y == 2, 1.0, np.where(P.y == 1, 0.5, 0.0))
filas = []
for s_ in range(4):
    m = (P['sidx'] == s_).values
    dl = (P['elo_local_pre'] - P['elo_visita_pre']).values[m]
    S_m = S_obs[m].mean()
    f = lambda h: (1 / (1 + 10 ** (-(dl + h) / 400))).mean() - S_m
    filas.append(dict(sidx=s_, puntos_local_obs=S_m, E_local_con_H150=(1 / (1 + 10 ** (-(dl + 150) / 400))).mean(),
                      H_justa=brentq(f, -300, 400)))
print(pd.DataFrame(filas).round(3).to_string(index=False))
print('  (puntos_local_obs = % victorias + mitad de % empates; H_justa = la H con la que el Elo esperaría eso)')

print('\n--- (e) Efecto de calendario en el Elo de E-2 ---')
dl_home = (P['elo_local_post'] - P['elo_local_pre'])
sesgo = dl_home.groupby(P['sidx']).mean()
print('Puntos que gana (o pierde) en promedio el LOCAL por partido:', sesgo[sesgo.index <= 3].round(2).to_dict())
dh = sesgo[sesgo.index <= 3].mean()
print('Sesgo en una ventana de 10 partidos según cuántos juega de local (puntos de Elo):')
print({f'{h} de local': round(float((2 * h - 10) * dh), 1) for h in [3, 4, 5, 6, 7]})

print('\n--- (f) Convergencia: dispersión del Elo entre equipos ---')
lg = largo.sort_values('fecha')
for s_ in range(4):
    d = lg[lg['sidx'] == s_]
    ini = d.groupby('team')['elo_pre'].first(); fin = d.groupby('team')['elo_post'].last()
    print(f'  temporada {s_}: desv. est. al inicio {ini.std():.0f} · al cierre {fin.std():.0f}')

print('\n--- (g) ¿Cuánto cuesta usar una H cercana a la real? (R 0.25 / 0; calibración congelada) ---')
filas = []
for h in [50, 75, 100, 150]:
    r = correr_elo(P, E2['K'], h, 0.25, 0.0)
    x = (r[0] + h - r[1]) / 400
    sh = pd.Series(r[2] - r[0]).groupby(P['sidx']).mean()
    fila = dict(H=h, sesgo_local_por_partido=round(sh[sh.index <= 3].mean(), 2))
    for split in SPLITS:
        for cal in ['A', 'B365']:
            Pm, y, _ = evaluar(P, x, cal, 'congelado', split)
            fila[f'{cal}_{split}'] = metricas(Pm, y)[0]
    filas.append(fila)
print(pd.DataFrame(filas).round(4).to_string(index=False))

print('\n--- (h) TECHO del problema de nivel (oráculo: NO se usa para elegir nada) ---')
print('    Pendiente fija (calibrada con el pasado); solo los cortes se ajustan con la misma temporada evaluada.')
def _ajusta_cortes(b, x, y):
    idx = np.arange(len(y))
    nll = lambda c: -np.log(probs_cal(np.array([b, c[0], c[1]]), x, np.zeros(len(y)), 'A')[idx, y]).mean()
    return minimize(nll, np.array([-0.9, 0.17]), method='BFGS').x
for split in SPLITS:
    s = SPLITS[split]
    bf = ((P['fecha'] >= CALENTAMIENTO) & (P['sidx'] <= s['calib_hasta'])).values
    ev = (P['sidx'] == s['evalua']).values
    p, _ = ajustar_cal(x0[bf], P['y'].values[bf], P['t'].values[bf], np.ones(bf.sum()), 'A')
    c = _ajusta_cortes(p[0], x0[ev], P['y'].values[ev])
    Pm = probs_cal(np.array([p[0], c[0], c[1]]), x0[ev], np.zeros(ev.sum()), 'A')
    ll_or = metricas(Pm, P['y'].values[ev])[0]
    ll_A = G[(G.split == split) & (G.R_ver == 0.25) & (G.R_inv == 0) & (G.H == 'universal') &
             (G.cal == 'A') & (G.modo == 'congelado')].logloss.iloc[0]
    print(f'  {split}: A congelado {ll_A:.4f} · techo con nivel perfecto {ll_or:.4f} · en juego {ll_A - ll_or:.4f}')

print('\n--- (i) Partido parejo (mismo Elo; solo pesa la localía): qué predice cada calibrador ---')
s = SPLITS['externo']
bf = ((P['fecha'] >= CALENTAMIENTO) & (P['sidx'] <= s['calib_hasta'])).values
y_, t_ = P['y'].values, P['t'].values
t_ini, t_fin = P.loc[P.sidx == 3, 't'].min(), P.loc[P.sidx == 3, 't'].max()
filas = []
for cal in ['A', 'B365', 'B90', 'Ca', 'Cb']:
    tipo = _tipo(cal)
    p, ok = ajustar_cal(x0[bf], y_[bf], t_[bf], pesos(cal, t_[bf], t_[bf].max()), tipo)
    for nom, tt in [('jul-2024', t_ini), ('may-2025', t_fin)]:
        pr = probs_cal(p, np.array([E2['H'] / 400]), np.array([tt]), tipo)[0]
        filas.append(dict(cal=cal, momento=nom, pendiente_b=round(p[0], 2),
                          local=pr[2] * 100, empate=pr[1] * 100, visita=pr[0] * 100))
print(pd.DataFrame(filas).round(1).to_string(index=False))
print(f'R3d terminó en {time.time() - t0:.0f} s.')

Elo R3d

In [ ]:
# [R3d] Diagnóstico del Elo (solo 2021/22 a 2024/25) ----------------
# Requiere R3-0, R3a y R3b en memoria (la grilla se lee de caché). Tarda ~2 min.
# Nada de esta celda elige parámetros: mide por qué salió lo que salió.
from scipy.optimize import brentq
t0 = time.time()
CANDIDATAS['fase'] = ['competition_stage.name', 'competition_stage']
M2, _ = cargar_partidos()
P['fase_nombre'] = P['match_id'].map(M2.drop_duplicates('match_id').set_index('match_id')['fase'])
P['fase_tipo'] = np.where(P['fase_nombre'].fillna('').astype(str).str.contains('Regular', case=False),
                          'regular', 'liguilla')
ent = P[P['sidx'] <= 3].copy()

print('--- (a) Nombres de fase en entrenamiento: ¿"regular" quedó bien separado? ---')
print(ent.groupby(['fase_nombre', 'fase_tipo'], dropna=False).size().to_string())

print('\n--- (b) Resultados por temporada, torneo y fase ---')
tb = pd.crosstab([ent.sidx, ent.torneo, ent.fase_tipo], ent.y, normalize='index').reindex(columns=[2, 1, 0]) * 100
tb.columns = ['% local', '% empate', '% visita']
tb.insert(0, 'n', ent.groupby(['sidx', 'torneo', 'fase_tipo']).size().astype(int))
print(tb.round(1).to_string())

x0 = ((P['elo_local_pre'] + P['H'] - P['elo_visita_pre']) / 400).values
def _ll_vec(x, cal, modo, split):
    Pm, y, _ = evaluar(P, x, cal, modo, split)
    return -np.log(Pm[np.arange(len(y)), y])

print('\n--- (c) Recalibrar menos congelado, por tramo (en milésimas; POSITIVO = recalibrar fue PEOR) ---')
filas = []
for split in SPLITS:
    ev = P[P['sidx'] == SPLITS[split]['evalua']].copy()
    ev['tramo'] = ev['torneo'] + '-' + ev['fase_tipo']
    for cal in ['A', 'B365', 'B90', 'Ca']:
        ev['dif'] = _ll_vec(x0, cal, 'wf', split) - _ll_vec(x0, cal, 'congelado', split)
        g = (ev.groupby('tramo')['dif'].mean() * 1000).round(1).to_dict()
        filas.append({'split': split, 'cal': cal, **g, 'total': round(ev['dif'].mean() * 1000, 1)})
print(pd.DataFrame(filas).to_string(index=False))
for split in SPLITS:
    ev = P[P['sidx'] == SPLITS[split]['evalua']]
    print(f'  n por tramo ({split}):', (ev['torneo'] + '-' + ev['fase_tipo']).value_counts().to_dict())

print('\n--- (d) Ventaja local: la que usa el Elo frente a la real ---')
S_obs = np.where(P.y == 2, 1.0, np.where(P.y == 1, 0.5, 0.0))
filas = []
for s_ in range(4):
    m = (P['sidx'] == s_).values
    dl = (P['elo_local_pre'] - P['elo_visita_pre']).values[m]
    S_m = S_obs[m].mean()
    f = lambda h: (1 / (1 + 10 ** (-(dl + h) / 400))).mean() - S_m
    filas.append(dict(sidx=s_, puntos_local_obs=S_m, E_local_con_H150=(1 / (1 + 10 ** (-(dl + 150) / 400))).mean(),
                      H_justa=brentq(f, -300, 400)))
print(pd.DataFrame(filas).round(3).to_string(index=False))
print('  (puntos_local_obs = % victorias + mitad de % empates; H_justa = la H con la que el Elo esperaría eso)')

print('\n--- (e) Efecto de calendario en el Elo de E-2 ---')
dl_home = (P['elo_local_post'] - P['elo_local_pre'])
sesgo = dl_home.groupby(P['sidx']).mean()
print('Puntos que gana (o pierde) en promedio el LOCAL por partido:', sesgo[sesgo.index <= 3].round(2).to_dict())
dh = sesgo[sesgo.index <= 3].mean()
print('Sesgo en una ventana de 10 partidos según cuántos juega de local (puntos de Elo):')
print({f'{h} de local': round(float((2 * h - 10) * dh), 1) for h in [3, 4, 5, 6, 7]})

print('\n--- (f) Convergencia: dispersión del Elo entre equipos ---')
lg = largo.sort_values('fecha')
for s_ in range(4):
    d = lg[lg['sidx'] == s_]
    ini = d.groupby('team')['elo_pre'].first(); fin = d.groupby('team')['elo_post'].last()
    print(f'  temporada {s_}: desv. est. al inicio {ini.std():.0f} · al cierre {fin.std():.0f}')

print('\n--- (g) ¿Cuánto cuesta usar una H cercana a la real? (R 0.25 / 0; calibración congelada) ---')
filas = []
for h in [50, 75, 100, 150]:
    r = correr_elo(P, E2['K'], h, 0.25, 0.0)
    x = (r[0] + h - r[1]) / 400
    sh = pd.Series(r[2] - r[0]).groupby(P['sidx']).mean()
    fila = dict(H=h, sesgo_local_por_partido=round(sh[sh.index <= 3].mean(), 2))
    for split in SPLITS:
        for cal in ['A', 'B365']:
            Pm, y, _ = evaluar(P, x, cal, 'congelado', split)
            fila[f'{cal}_{split}'] = metricas(Pm, y)[0]
    filas.append(fila)
print(pd.DataFrame(filas).round(4).to_string(index=False))

print('\n--- (h) TECHO del problema de nivel (oráculo: NO se usa para elegir nada) ---')
print('    Pendiente fija (calibrada con el pasado); solo los cortes se ajustan con la misma temporada evaluada.')
def _ajusta_cortes(b, x, y):
    idx = np.arange(len(y))
    nll = lambda c: -np.log(probs_cal(np.array([b, c[0], c[1]]), x, np.zeros(len(y)), 'A')[idx, y]).mean()
    return minimize(nll, np.array([-0.9, 0.17]), method='BFGS').x
for split in SPLITS:
    s = SPLITS[split]
    bf = ((P['fecha'] >= CALENTAMIENTO) & (P['sidx'] <= s['calib_hasta'])).values
    ev = (P['sidx'] == s['evalua']).values
    p, _ = ajustar_cal(x0[bf], P['y'].values[bf], P['t'].values[bf], np.ones(bf.sum()), 'A')
    c = _ajusta_cortes(p[0], x0[ev], P['y'].values[ev])
    Pm = probs_cal(np.array([p[0], c[0], c[1]]), x0[ev], np.zeros(ev.sum()), 'A')
    ll_or = metricas(Pm, P['y'].values[ev])[0]
    ll_A = G[(G.split == split) & (G.R_ver == 0.25) & (G.R_inv == 0) & (G.H == 'universal') &
             (G.cal == 'A') & (G.modo == 'congelado')].logloss.iloc[0]
    print(f'  {split}: A congelado {ll_A:.4f} · techo con nivel perfecto {ll_or:.4f} · en juego {ll_A - ll_or:.4f}')

print('\n--- (i) Partido parejo (mismo Elo; solo pesa la localía): qué predice cada calibrador ---')
s = SPLITS['externo']
bf = ((P['fecha'] >= CALENTAMIENTO) & (P['sidx'] <= s['calib_hasta'])).values
y_, t_ = P['y'].values, P['t'].values
t_ini, t_fin = P.loc[P.sidx == 3, 't'].min(), P.loc[P.sidx == 3, 't'].max()
filas = []
for cal in ['A', 'B365', 'B90', 'Ca', 'Cb']:
    tipo = _tipo(cal)
    p, ok = ajustar_cal(x0[bf], y_[bf], t_[bf], pesos(cal, t_[bf], t_[bf].max()), tipo)
    for nom, tt in [('jul-2024', t_ini), ('may-2025', t_fin)]:
        pr = probs_cal(p, np.array([E2['H'] / 400]), np.array([tt]), tipo)[0]
        filas.append(dict(cal=cal, momento=nom, pendiente_b=round(p[0], 2),
                          local=pr[2] * 100, empate=pr[1] * 100, visita=pr[0] * 100))
print(pd.DataFrame(filas).round(1).to_string(index=False))
print(f'R3d terminó en {time.time() - t0:.0f} s.')

Elo v3

In [ ]:
# =====================================================================
# R3 · Elo y capa predictiva (framework v1.1, sección 7)
# Sesión 5 · 2026-10-09 · Hackathon ISAC 2026, Track 2
# Cuatro celdas: R3-0 (funciones), R3a (Elo base), R3b (grilla), R3c (selección)
# Antes: celda 1 (Drive) y R0.
# REGLA DE HOLDOUT: este código NUNCA imprime resultados ni frecuencias de
# 2025/26 ni de 2026/27. Solo calcula el Elo prepartido de esos partidos.
# =====================================================================

# [R3-0] Configuración y funciones ---------------------------------
import os, re, glob, json, gzip, time, warnings
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit

SEMILLA = 2026
BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026/')
DP = os.path.join(BASE, 'datos_parquet')
FW = os.path.join(DP, 'fw1')
DC = os.path.join(BASE, 'datos_crudos')
SAL = os.path.join(BASE, 'salidas')
os.makedirs(FW, exist_ok=True); os.makedirs(SAL, exist_ok=True)
REHACER = False

# E-2 (decidido)
E2 = dict(K=20, H=75, R_ver=0.25, R_inv=0.0)   # H = 75 decidido por Arturo (S5-08, sesión 5)
H_ELO1 = 150                                     # H de ELO-1: solo para la referencia de comparación
INICIO = 1500.0
ELO_INICIAL = {}          # {team_id: Elo de arranque}; se llena en R3a con la tabla Guard1anes 2020-2021 (S5-09)
SD_OBJETIVO = 68          # dispersión inicial de las temporadas ya maduras (bloque f de R3d, H = 75)
RUTA_TABLA = os.path.join(BASE, 'datos_externos', 'tabla_guardianes_2020_2021.csv')
ALIAS = {}                # si un nombre no cruza: {'nombre en la tabla': 'nombre en StatsBomb'}
CALENTAMIENTO = pd.Timestamp('2021-10-01')

# Grillas que cubren TODAS las opciones de S5-01 a S5-04
VIDAS_MEDIAS = [90, 180, 365, 730]          # S5-01 (días)
CALIBRADORES = ['A'] + [f'B{h}' for h in VIDAS_MEDIAS] + ['Ca', 'Cb']   # S5-02
R_VER_GRID = [0.15, 0.20, 0.25]             # S5-03
R_INV_GRID = [0.0, 0.05, 0.10, 0.15]        # S5-03
H_TIPOS = ['universal', 'estadio']          # S5-04
MODOS = ['congelado', 'wf']                 # S5-05

# Particiones (por orden cronológico de temporadas, ver R3a)
#   índice 0 = 2021/22, 1 = 2022/23, 2 = 2023/24, 3 = 2024/25, 4 = 2025/26 (sellada), 5 = 2026/27 (en vivo)
SPLITS = {'interno': dict(calib_hasta=1, evalua=2),   # elegir
          'externo': dict(calib_hasta=2, evalua=3)}   # estimar
TEMPORADAS_PROHIBIDAS = {4, 5}


# ---------- Carga de partidos ----------
def _leer_matches(ruta):
    if ruta.endswith('.parquet'):
        df = pd.read_parquet(ruta)
        obj = [c for c in df.columns if df[c].dtype == object]
        if any(df[c].map(lambda v: isinstance(v, dict)).any() for c in obj):
            df = pd.json_normalize(df.to_dict('records'))
    else:
        abrir = gzip.open if ruta.endswith('.gz') else open
        with abrir(ruta, 'rt', encoding='utf-8') as f:
            df = pd.json_normalize(json.load(f))
    m = re.search(r'(\d{4}-\d{2}-\d{2})', os.path.basename(ruta))
    df['_fecha_archivo'] = m.group(1) if m else '0000-00-00'
    df['_archivo'] = os.path.basename(ruta)
    return df

def _col(df, opciones):
    for o in opciones:
        if o in df.columns:
            return o
    return None

CANDIDATAS = {
    'match_id': ['match_id'],
    'fecha_m': ['match_date'],
    'hora': ['kick_off'],
    'local_id': ['home_team.home_team_id', 'home_team_id'],
    'local': ['home_team.home_team_name', 'home_team_name'],
    'visita_id': ['away_team.away_team_id', 'away_team_id'],
    'visita': ['away_team.away_team_name', 'away_team_name'],
    'gl': ['home_score'], 'gv': ['away_score'],
    'estadio': ['stadium.name', 'stadium_name', 'stadium'],
}

def cargar_partidos():
    rutas = sorted(glob.glob(os.path.join(DC, 'matches_73_*.json*')) +
                   glob.glob(os.path.join(DP, 'matches_73_*.parquet')))
    piezas = []
    for r in rutas:
        try:
            df = _leer_matches(r)
        except Exception as e:
            print(f'  aviso: no se pudo leer {os.path.basename(r)}: {e}')
            continue
        cols = {k: _col(df, v) for k, v in CANDIDATAS.items()}
        faltan = [k for k in ['match_id', 'local', 'visita', 'gl', 'gv'] if cols[k] is None]
        if faltan:
            print(f'  aviso: {os.path.basename(r)} sin columnas {faltan}; se omite')
            continue
        out = pd.DataFrame({k: (df[c] if c else np.nan) for k, c in cols.items()})
        if out['estadio'].map(lambda v: isinstance(v, dict)).any():
            out['estadio'] = out['estadio'].map(lambda v: v.get('name') if isinstance(v, dict) else v)
        out['_fecha_archivo'] = df['_fecha_archivo']; out['_archivo'] = df['_archivo']
        piezas.append(out)
    M = pd.concat(piezas, ignore_index=True)
    M = (M.sort_values('_fecha_archivo', ascending=False)
           .drop_duplicates('match_id', keep='first'))
    return M, rutas


# ---------- Elo ----------
def correr_elo(P, K, H, R_ver, R_inv, H_est=None):
    """Recorre los partidos en orden. Devuelve Elo pre y post de local y visita, y la H usada."""
    rating = dict(ELO_INICIAL)
    temp_act = torneo_act = None
    pre_l = np.empty(len(P)); pre_v = np.empty(len(P))
    post_l = np.empty(len(P)); post_v = np.empty(len(P)); h_us = np.empty(len(P))
    for i, r in enumerate(P.itertuples(index=False)):
        if temp_act is not None and r.temporada != temp_act:          # reinicio de verano
            for t in rating: rating[t] = INICIO + (1 - R_ver) * (rating[t] - INICIO)
        elif torneo_act is not None and r.torneo != torneo_act:       # reinicio de invierno
            for t in rating: rating[t] = INICIO + (1 - R_inv) * (rating[t] - INICIO)
        temp_act, torneo_act = r.temporada, r.torneo
        rl = rating.get(r.local_id, INICIO); rv = rating.get(r.visita_id, INICIO)
        h = H + (H_est.get(r.estadio, 0.0) if H_est else 0.0)
        E = 1.0 / (1.0 + 10 ** (-(rl + h - rv) / 400))
        dif = r.gl - r.gv
        S = 1.0 if dif > 0 else (0.5 if dif == 0 else 0.0)
        n = abs(dif)
        G = 1.0 if n <= 1 else (1.5 if n == 2 else (11 + n) / 8)     # escala de eloratings
        delta = K * G * (S - E)
        rating[r.local_id] = rl + delta; rating[r.visita_id] = rv - delta
        pre_l[i], pre_v[i], post_l[i], post_v[i], h_us[i] = rl, rv, rl + delta, rv - delta, h
    return pre_l, pre_v, post_l, post_v, h_us


def estimar_H_estadio(P, pre_l, pre_v, H, mascara):
    """Ventaja local extra por estadio, con contracción (empirical Bayes). Solo usa filas de 'mascara'."""
    d = P.loc[mascara].copy()
    E = 1 / (1 + 10 ** (-(pre_l[mascara] + H - pre_v[mascara]) / 400))
    S = np.where(d.gl > d.gv, 1.0, np.where(d.gl == d.gv, 0.5, 0.0))
    d['r'] = S - E
    d['deriv'] = np.log(10) / 400 * E * (1 - E)
    var_r = d['r'].var()
    g = d.groupby('estadio').agg(n=('r', 'size'), rbar=('r', 'mean'), dbar=('deriv', 'mean'))
    g['rbar_c'] = g['rbar'] - d['r'].mean()      # desviación respecto al promedio de la liga
    g['se2'] = var_r / g['n']
    tau2 = max(0.0, g['rbar_c'].var() - g['se2'].mean())
    g['k'] = tau2 / (tau2 + g['se2']) if tau2 > 0 else 0.0
    g['dH'] = g['k'] * g['rbar_c'] / g['dbar']
    return g['dH'].to_dict(), g, np.sqrt(tau2) / g['dbar'].mean() if tau2 > 0 else 0.0


# ---------- Calibración ordenada ----------
NPAR = {'A': 3, 'B': 3, 'Ca': 4, 'Cb': 5}

def _tipo(cal):
    return 'B' if cal.startswith('B') else cal

def probs_cal(p, x, t, tipo):
    """Columnas: 0 = gana visita, 1 = empate, 2 = gana local."""
    b, c1, lg = p[0], p[1], p[2]
    c1t = c1 + (p[3] * t if tipo in ('Ca', 'Cb') else 0.0)
    lgt = lg + (p[4] * t if tipo == 'Cb' else 0.0)
    c2t = c1t + np.exp(lgt)
    z = b * x
    pv = expit(c1t - z); pve = expit(c2t - z)
    Pm = np.column_stack([pv, pve - pv, 1 - pve])
    Pm = np.clip(Pm, 1e-9, 1); return Pm / Pm.sum(1, keepdims=True)

def ajustar_cal(x, y, t, w, tipo, init=None):
    k = NPAR[tipo]
    p0 = np.array([2.0, -0.9, 0.17, 0.0, 0.0])[:k] if init is None or len(init) != k else init
    idx = np.arange(len(y))
    def nll(p):
        return -(w * np.log(probs_cal(p, x, t, tipo)[idx, y])).sum() / w.sum()
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        res = minimize(nll, p0, method='BFGS')
    return res.x, bool(res.success)

def pesos(cal, t_fit, t_ref):
    if cal.startswith('B'):
        hl = float(cal[1:])
        return 0.5 ** ((t_ref - t_fit) * 365.25 / hl)
    return np.ones_like(t_fit)


def evaluar(P, x, cal, modo, split):
    """Probabilidades para la temporada de evaluación del split. Solo usa información previa."""
    s = SPLITS[split]
    assert s['evalua'] not in TEMPORADAS_PROHIBIDAS, 'Prohibido evaluar en 2025/26 o 2026/27'
    y = P['y'].values; t = P['t'].values; sidx = P['sidx'].values; fecha = P['fecha'].values
    elegible = (P['fecha'] >= CALENTAMIENTO).values
    base_fit = elegible & (sidx <= s['calib_hasta'])
    ev = sidx == s['evalua']
    out = np.full((len(P), 3), np.nan); fallos = 0
    if cal == 'freq':
        f = np.bincount(y[base_fit], minlength=3) / base_fit.sum()
        out[ev] = f; return out[ev], y[ev], 0
    tipo = _tipo(cal)
    if modo == 'congelado':
        tref = t[base_fit].max()
        p, ok = ajustar_cal(x[base_fit], y[base_fit], t[base_fit], pesos(cal, t[base_fit], tref), tipo)
        fallos += (not ok)
        out[ev] = probs_cal(p, x[ev], t[ev], tipo)
    else:  # wf: recalibra antes de cada fecha con todo lo anterior
        p = None
        for f_ in np.unique(fecha[ev]):
            fit = base_fit | (ev & (fecha < f_))
            hoy = ev & (fecha == f_)
            tref = t[hoy].min()
            p, ok = ajustar_cal(x[fit], y[fit], t[fit], pesos(cal, t[fit], tref), tipo, init=p)
            fallos += (not ok)
            out[hoy] = probs_cal(p, x[hoy], t[hoy], tipo)
    return out[ev], y[ev], fallos


def metricas(Pm, y):
    ll = -np.mean(np.log(Pm[np.arange(len(y)), y]))
    O = np.eye(3)[y]
    br = np.mean(((Pm - O) ** 2).sum(1))
    return ll, br

print('R3-0 listo. BASE =', BASE)


# [R3a] Partidos, Elo base (E-2), chequeos y guardado ---------------
t0 = time.time()
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp.columns:
    ptp = ptp[ptp['jugado'].astype(bool)]
ptp['fecha'] = pd.to_datetime(ptp['fecha'])
uni = (ptp.groupby('match_id').agg(fecha=('fecha', 'first'), temporada=('temporada', 'first'),
                                   n_filas=('team', 'size')).reset_index())
loc_ptp = ptp[ptp['lado'] == 'home'][['match_id', 'team']].rename(columns={'team': 'local_ptp'})

M, rutas = cargar_partidos()
print(f'Archivos de partidos leídos: {len(rutas)}')
P = uni.merge(M, on='match_id', how='left').merge(loc_ptp, on='match_id', how='left')
P['hora'] = P['hora'].fillna('00:00:00.000')
P = P.sort_values(['fecha', 'hora', 'match_id']).reset_index(drop=True)
P['torneo'] = np.where(P['fecha'].dt.month >= 7, 'Apertura', 'Clausura')
orden = P.groupby('temporada')['fecha'].min().sort_values().index.tolist()
P['sidx'] = P['temporada'].map({s: i for i, s in enumerate(orden)})
P['t'] = (P['fecha'] - CALENTAMIENTO).dt.days / 365.25
P['y'] = np.where(P.gl > P.gv, 2, np.where(P.gl == P.gv, 1, 0))
for c in ['local_id', 'visita_id']:
    if P[c].isna().all():
        P[c] = P[c.replace('_id', '')]
P['estadio'] = P['estadio'].fillna('(sin estadio)')

print('\n--- Chequeos de carga ---')
print('Partidos:', len(P), '| sin marcador:', int(P[['gl', 'gv']].isna().any(axis=1).sum()),
      '| filas ptp distintas de 2:', int((P['n_filas'] != 2).sum()))
print('Local de Matches distinto al de ptp:', int((P['local'] != P['local_ptp']).sum()))
tab = P.groupby('sidx').agg(temporada=('temporada', 'first'), partidos=('match_id', 'size'),
                            desde=('fecha', 'min'), hasta=('fecha', 'max'))
tab['desde'] = tab['desde'].dt.date; tab['hasta'] = tab['hasta'].dt.date
print(tab.to_string())
print('Equipos distintos:', P['local_id'].nunique(), '| estadios distintos:', P['estadio'].nunique())

print('\n--- Resultados por temporada (SOLO entrenamiento, índices 0-3) ---')
ent = P[P['sidx'] <= 3]
fr = (ent.groupby('sidx')['y'].value_counts(normalize=True).unstack().reindex(columns=[2, 1, 0]) * 100)
fr.columns = ['% local', '% empate', '% visita']
fr['goles/partido'] = ent.assign(g=ent.gl + ent.gv).groupby('sidx')['g'].mean()
print(fr.round(1).to_string())

# --- Arranque por tabla 2020-2021 (S5-09, idea de Arturo) ---
import unicodedata, difflib
def _norm(x):
    x = unicodedata.normalize('NFKD', str(x)).encode('ascii', 'ignore').decode()
    return ' '.join(x.lower().replace('.', ' ').split())
try:
    TB = pd.read_csv(RUTA_TABLA, encoding='utf-8')
except UnicodeDecodeError:
    TB = pd.read_csv(RUTA_TABLA, encoding='latin-1')
TB['Equipo'] = TB['Equipo'].astype(str).str.strip().replace(ALIAS)
tb = TB.groupby('Equipo')[['Partidos Jugados', 'Puntos']].sum()
tb['ppp'] = tb['Puntos'] / tb['Partidos Jugados']
k_sem = SD_OBJETIVO / tb['ppp'].std(ddof=1)
tb['elo_inicial'] = INICIO + k_sem * (tb['ppp'] - tb['ppp'].mean())

prim = P[P['sidx'] == 0]
sb = pd.concat([prim[['local', 'local_id']].set_axis(['nombre', 'id'], axis=1),
                prim[['visita', 'visita_id']].set_axis(['nombre', 'id'], axis=1)]).drop_duplicates()
sb_norm = {_norm(n): (n, i) for n, i in sb.itertuples(index=False)}
cruce = []
for eq, r in tb.iterrows():
    n = _norm(eq)
    if n in sb_norm:
        hit, via = sb_norm[n], 'exacto'
    else:
        cand = difflib.get_close_matches(n, list(sb_norm), n=1, cutoff=0.5)
        hit, via = (sb_norm[cand[0]], 'aproximado') if cand else ((None, None), 'SIN CRUCE')
    cruce.append(dict(tabla=eq, statsbomb=hit[0], team_id=hit[1], via=via, ppp=r.ppp, elo_inicial=r.elo_inicial))
cruce = pd.DataFrame(cruce).sort_values('elo_inicial', ascending=False)
print('\n--- Arranque por tabla Guard1anes 2020-2021 ---')
print(f'k = {k_sem:.1f} puntos de Elo por punto por partido · dispersión objetivo {SD_OBJETIVO}')
print(cruce.round(2).to_string(index=False))
faltan_sb = sorted(set(sb['nombre']) - set(cruce['statsbomb'].dropna()))
dup = cruce['statsbomb'].dropna().duplicated().any()
print('Equipos de 2021/22 sin arranque (empiezan en 1,500):', faltan_sb if faltan_sb else 'ninguno')
assert not dup, 'Dos equipos de la tabla cruzaron con el mismo de StatsBomb: corregir con ALIAS'
assert (cruce['via'] != 'SIN CRUCE').all(), 'Hay equipos de la tabla sin cruce: corregir con ALIAS'
ELO_INICIAL.clear()
ELO_INICIAL.update({r.team_id: r.elo_inicial for r in cruce.itertuples()})

pre_l, pre_v, post_l, post_v, h_us = correr_elo(P, **E2)
P['elo_local_pre'], P['elo_visita_pre'] = pre_l, pre_v
P['elo_local_post'], P['elo_visita_post'] = post_l, post_v
P['H'] = h_us
P['d'] = P['elo_local_pre'] + P['H'] - P['elo_visita_pre']
P['E_local'] = 1 / (1 + 10 ** (-P['d'] / 400))

print('\n--- Elo base (E-2) ---')
fin3 = P[P['sidx'] <= 3]
ult = pd.concat([fin3[['fecha', 'local', 'elo_local_post']].rename(columns={'local': 'eq', 'elo_local_post': 'elo'}),
                 fin3[['fecha', 'visita', 'elo_visita_post']].rename(columns={'visita': 'eq', 'elo_visita_post': 'elo'})])
ult = ult.sort_values('fecha').groupby('eq').tail(1).sort_values('elo', ascending=False)
print(f'Al cierre de 2024/25: media {ult.elo.mean():.0f}, desv. est. {ult.elo.std():.0f}, '
      f'rango {ult.elo.min():.0f} a {ult.elo.max():.0f}')
print('Top 3:', ', '.join(f'{r.eq} {r.elo:.0f}' for r in ult.head(3).itertuples()))
print('Últimos 3:', ', '.join(f'{r.eq} {r.elo:.0f}' for r in ult.tail(3).itertuples()))
print('E medio del local por temporada (0-3):', P[P.sidx <= 3].groupby('sidx')['E_local'].mean().round(3).to_dict())

cols_out = ['match_id', 'fecha', 'temporada', 'sidx', 'torneo', 'local', 'visita', 'local_id', 'visita_id',
            'estadio', 'gl', 'gv', 'elo_local_pre', 'elo_visita_pre', 'elo_local_post', 'elo_visita_post', 'H', 'd']
P[cols_out].to_parquet(os.path.join(FW, 'elo_partido_e2.parquet'), index=False)
largo = pd.concat([
    P.assign(team=P.local, lado='home', elo_pre=P.elo_local_pre, elo_post=P.elo_local_post, elo_rival_pre=P.elo_visita_pre),
    P.assign(team=P.visita, lado='away', elo_pre=P.elo_visita_pre, elo_post=P.elo_visita_post, elo_rival_pre=P.elo_local_pre)
])[['match_id', 'fecha', 'temporada', 'sidx', 'team', 'lado', 'elo_pre', 'elo_post', 'elo_rival_pre']]
largo.to_parquet(os.path.join(FW, 'elo_equipo_partido_e2.parquet'), index=False)

print('\n--- América por etapa de técnico (solo hasta 2024/25) ---')
am = ptp[ptp['team'].str.contains('Am[eé]rica', case=False, regex=True)][['match_id', 'team', 'tec_p', 'etapa']]
am = am.merge(largo, on=['match_id', 'team'])
am = am[am['sidx'] <= 3].sort_values('fecha')
res_am = am.groupby('etapa').agg(tecnico=('tec_p', 'first'), partidos=('match_id', 'size'),
                                 desde=('fecha', 'min'), elo_inicio=('elo_pre', 'first'),
                                 elo_fin=('elo_post', 'last'), elo_medio=('elo_pre', 'mean')).sort_values('desde')
res_am['cambio'] = res_am['elo_fin'] - res_am['elo_inicio']
res_am['desde'] = res_am['desde'].dt.date
print(res_am.round(0).to_string())
print(f'\nR3a terminó en {time.time() - t0:.0f} s. Guardado: fw1/elo_partido_e2.parquet y fw1/elo_equipo_partido_e2.parquet')


# [R3b] Grilla completa: Elo (R de verano, R de invierno, H) x calibrador x modo x split
ruta_grid = os.path.join(SAL, f"r3_grilla_H{E2['H']}_semilla.csv")
if os.path.exists(ruta_grid) and not REHACER:
    G = pd.read_csv(ruta_grid); print('Grilla cargada de caché:', len(G), 'filas')
else:
    t0 = time.time(); filas = []
    for split in SPLITS:
        s = SPLITS[split]
        mask_fit_H = ((P['fecha'] >= CALENTAMIENTO) & (P['sidx'] <= s['calib_hasta'])).values
        for rv in R_VER_GRID:
            for ri in R_INV_GRID:
                base = correr_elo(P, E2['K'], E2['H'], rv, ri)
                for ht in H_TIPOS:
                    if ht == 'universal':
                        pl, pv_ = base[0], base[1]; hh = np.full(len(P), E2['H'])
                    else:
                        dH, _, _ = estimar_H_estadio(P, base[0], base[1], E2['H'], mask_fit_H)
                        r2 = correr_elo(P, E2['K'], E2['H'], rv, ri, H_est=dH)
                        pl, pv_, hh = r2[0], r2[1], r2[4]
                    x = (pl + hh - pv_) / 400
                    for cal in ['freq'] + CALIBRADORES:
                        for modo in (['congelado'] if cal == 'freq' else MODOS):
                            Pm, y, fallos = evaluar(P, x, cal, modo, split)
                            ll, br = metricas(Pm, y)
                            filas.append(dict(split=split, R_ver=rv, R_inv=ri, H=ht, cal=cal, modo=modo,
                                              n=len(y), logloss=ll, brier=br, fallos=fallos))
        print(f'  split {split} listo ({time.time() - t0:.0f} s)')
    G = pd.DataFrame(filas); G.to_csv(ruta_grid, index=False)
    print(f'R3b terminó en {time.time() - t0:.0f} s. Filas: {len(G)}. Fallos de ajuste totales: {int(G.fallos.sum())}')

# Vista 1: calibradores con el Elo de E-2 (R 0.25 / 0, H universal)
v1 = G[(G.R_ver == 0.25) & (G.R_inv == 0.0) & (G.H == 'universal')]
v1 = v1.pivot_table(index=['cal', 'modo'], columns='split', values=['logloss', 'brier'])
v1.columns = [f'{a}_{b}' for a, b in v1.columns]
print('\n--- Vista 1: calibradores con el Elo de E-2 ---')
print(v1[['logloss_interno', 'logloss_externo', 'brier_interno', 'brier_externo']].round(4)
      .sort_values('logloss_interno').to_string())
print(f"Elo de esta corrida: H = {E2['H']}. " + 'Referencias 2024/25 (ELO-1, H = 150): frecuencias 1.0526 / 0.6346 · Elo 0.9785 / 0.5803 · 1/3 1.0986 / 0.6667')


# [R3c] Selección con la regla preregistrada (S5-05 y S5-06) --------
MODO_PRINCIPAL = 'congelado'   # decidido por Arturo (S5-05, sesión 5)
UMBRAL = 0.003          # [PROVISIONAL-Claude hasta S5-06]
N_BOOT = 2000

def ll_int(f):
    q = G[(G.split == 'interno')]
    for k, v in f.items(): q = q[q[k] == v]
    return float(q.logloss.iloc[0])
def ll_ext(f):
    q = G[(G.split == 'externo')]
    for k, v in f.items(): q = q[q[k] == v]
    return float(q.logloss.iloc[0])

pasos = []
cfg = dict(R_ver=0.25, R_inv=0.0, H='universal', modo=MODO_PRINCIPAL)
# Paso 1: vida media de B (pieza decidida: se elige la mejor sin umbral)
cand = {c: ll_int({**cfg, 'cal': c}) for c in CALIBRADORES if c.startswith('B')}
mejor_B = min(cand, key=cand.get); cfg['cal'] = mejor_B
pasos.append(('1 vida media de B', mejor_B, cand[mejor_B], ll_ext(cfg), 'elegida (sin umbral)'))
# Paso 2: tendencia C contra B (entra solo con mejora >= UMBRAL)
for c in ['Ca', 'Cb']:
    li = ll_int({**cfg, 'cal': c}); entra = (ll_int(cfg) - li) >= UMBRAL
    pasos.append((f'2 tendencia {c}', c, li, ll_ext({**cfg, 'cal': c}), f'mejora {ll_int(cfg) - li:+.4f}: ' + ('ENTRA' if entra else 'no entra')))
    if entra and li < ll_int(cfg): cfg['cal'] = c
# Paso 3: reinicios (verano, invierno)
ref = ll_int(cfg); mejor = None
for rv in R_VER_GRID:
    for ri in R_INV_GRID:
        li = ll_int({**cfg, 'R_ver': rv, 'R_inv': ri})
        if mejor is None or li < mejor[2]: mejor = (rv, ri, li)
entra = (ref - mejor[2]) >= UMBRAL
pasos.append(('3 reinicios', f'R_ver {mejor[0]} / R_inv {mejor[1]}', mejor[2],
              ll_ext({**cfg, 'R_ver': mejor[0], 'R_inv': mejor[1]}), f'mejora {ref - mejor[2]:+.4f}: ' + ('ENTRA' if entra else 'no entra')))
if entra: cfg['R_ver'], cfg['R_inv'] = mejor[0], mejor[1]
# Paso 4: H por estadio
li = ll_int({**cfg, 'H': 'estadio'}); entra = (ll_int(cfg) - li) >= UMBRAL
pasos.append(('4 H por estadio', 'estadio', li, ll_ext({**cfg, 'H': 'estadio'}), f'mejora {ll_int(cfg) - li:+.4f}: ' + ('ENTRA' if entra else 'no entra')))
if entra: cfg['H'] = 'estadio'

print(f'--- Selección (modo {MODO_PRINCIPAL}, umbral {UMBRAL}) ---')
print(pd.DataFrame(pasos, columns=['paso', 'opción', 'logloss_interno', 'logloss_externo', 'resultado']).round(4).to_string(index=False))
print('Configuración elegida:', cfg)

# Vista 2: los 8 mejores Elo (R y H) con el calibrador elegido, para ver la meseta
v2 = G[(G.split == 'interno') & (G.cal == cfg['cal']) & (G.modo == cfg['modo'])].sort_values('logloss')
v2 = v2.merge(G[G.split == 'externo'][['R_ver', 'R_inv', 'H', 'cal', 'modo', 'logloss']],
              on=['R_ver', 'R_inv', 'H', 'cal', 'modo'], suffixes=('_int', '_ext'))
print('\n--- Vista 2: meseta de variantes de Elo (calibrador elegido) ---')
print(v2[['R_ver', 'R_inv', 'H', 'logloss_int', 'logloss_ext']].head(8).round(4).to_string(index=False))
print('Rango del log-loss interno entre las 24 variantes:', round(v2.logloss_int.max() - v2.logloss_int.min(), 4))

# Reproducir la configuración elegida y la referencia tipo ELO-1 (E-2 + A congelado) en 2024/25
def correr_cfg(c, split, H=None):
    H = E2['H'] if H is None else H
    s = SPLITS[split]
    base = correr_elo(P, E2['K'], H, c['R_ver'], c['R_inv'])
    if c['H'] == 'estadio':
        m = ((P['fecha'] >= CALENTAMIENTO) & (P['sidx'] <= s['calib_hasta'])).values
        dH, tabH, sdH = estimar_H_estadio(P, base[0], base[1], H, m)
        base = correr_elo(P, E2['K'], H, c['R_ver'], c['R_inv'], H_est=dH)
    x = (base[0] + base[4] - base[1]) / 400
    Pm, y, _ = evaluar(P, x, c['cal'], c['modo'], split)
    return Pm, y

Pm_sel, y_ext = correr_cfg(cfg, 'externo')
Pm_ref, _ = correr_cfg(dict(R_ver=0.25, R_inv=0.0, H='universal', cal='A', modo='congelado'), 'externo', H=H_ELO1)
Pm_frq, _ = evaluar(P, np.zeros(len(P)), 'freq', 'congelado', 'externo')[:2]

print('\n--- 2024/25: elegida contra referencias ---')
for nom, Pm in [('elegida', Pm_sel), ('ELO-1 (H 150 + A congelado)', Pm_ref), ('frecuencias', Pm_frq)]:
    ll, br = metricas(Pm, y_ext); print(f'{nom:32s} log-loss {ll:.4f} · Brier {br:.4f}')

rng = np.random.default_rng(SEMILLA)
li_sel = -np.log(Pm_sel[np.arange(len(y_ext)), y_ext]); li_ref = -np.log(Pm_ref[np.arange(len(y_ext)), y_ext])
li_frq = -np.log(Pm_frq[np.arange(len(y_ext)), y_ext])
def ic(a, b):
    dif = a - b; bs = [dif[rng.integers(0, len(dif), len(dif))].mean() for _ in range(N_BOOT)]
    return dif.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5)
for nom, b in [('elegida - tipo ELO-1', li_ref), ('elegida - frecuencias', li_frq)]:
    m, lo, hi = ic(li_sel, b); print(f'{nom:24s} diferencia {m:+.4f}  IC95% [{lo:+.4f}, {hi:+.4f}]  (negativo = elegida mejor)')

print('\n--- Calibración en 2024/25 (elegida): nivel por resultado ---')
obs = np.bincount(y_ext, minlength=3) / len(y_ext)
cal_tab = pd.DataFrame({'predicho %': Pm_sel.mean(0) * 100, 'observado %': obs * 100},
                       index=['visita', 'empate', 'local'])
cal_tab['tipo ELO-1 predicho %'] = Pm_ref.mean(0) * 100
print(cal_tab.round(1).to_string())

print('\n--- Calibración por quintiles de P(local) (elegida) ---')
q = pd.qcut(Pm_sel[:, 2], 5, labels=False, duplicates='drop')
ct = pd.DataFrame({'q': q, 'pred': Pm_sel[:, 2], 'obs': (y_ext == 2).astype(float)}).groupby('q').agg(
    n=('obs', 'size'), pred_local=('pred', 'mean'), obs_local=('obs', 'mean'))
print((ct.assign(pred_local=ct.pred_local * 100, obs_local=ct.obs_local * 100)).round(1).to_string())

if True:  # diagnóstico de H por estadio, siempre (E-5)
    m = ((P['fecha'] >= CALENTAMIENTO) & (P['sidx'] <= 2)).values
    b0 = correr_elo(P, E2['K'], E2['H'], cfg['R_ver'], cfg['R_inv'])
    dH, tabH, sdH = estimar_H_estadio(P, b0[0], b0[1], E2['H'], m)
    tabH = tabH.sort_values('dH')
    print(f'\n--- Diagnóstico H por estadio (calibración hasta 2023/24) · dispersión real estimada: {sdH:.0f} puntos ---')
    print('Más ventaja que el promedio (dH en puntos de Elo):'); print(tabH.tail(4)[['n', 'k', 'dH']].round(2).to_string())
    print('Menos ventaja que el promedio:'); print(tabH.head(4)[['n', 'k', 'dH']].round(2).to_string())
print('\nR3c listo. La configuración queda PROVISIONAL hasta que Arturo cierre S5-01 a S5-06.')


# [R3d] Diagnóstico del Elo (solo 2021/22 a 2024/25) ----------------
# Requiere R3-0, R3a y R3b en memoria (la grilla se lee de caché). Tarda ~2 min.
# Nada de esta celda elige parámetros: mide por qué salió lo que salió.
from scipy.optimize import brentq
t0 = time.time()
CANDIDATAS['fase'] = ['competition_stage.name', 'competition_stage']
M2, _ = cargar_partidos()
P['fase_nombre'] = P['match_id'].map(M2.drop_duplicates('match_id').set_index('match_id')['fase'])
_fn = P['fase_nombre'].fillna('').astype(str).str.strip()
P['fase_tipo'] = np.select(
    [_fn.isin(['Apertura', 'Clausura', 'Regular Season']),
     _fn.str.contains('Reclasificaci|Play-In', case=False, regex=True)],
    ['regular', 'play-in'], default='liguilla')
ent = P[P['sidx'] <= 3].copy()

print('--- (a) Nombres de fase en entrenamiento: ¿"regular" quedó bien separado? ---')
print(ent.groupby(['fase_nombre', 'fase_tipo'], dropna=False).size().to_string())

print('\n--- (b) Resultados por temporada, torneo y fase ---')
tb = pd.crosstab([ent.sidx, ent.torneo, ent.fase_tipo], ent.y, normalize='index').reindex(columns=[2, 1, 0]) * 100
tb.columns = ['% local', '% empate', '% visita']
tb.insert(0, 'n', ent.groupby(['sidx', 'torneo', 'fase_tipo']).size().astype(int))
print(tb.round(1).to_string())

x0 = ((P['elo_local_pre'] + P['H'] - P['elo_visita_pre']) / 400).values
def _ll_vec(x, cal, modo, split):
    Pm, y, _ = evaluar(P, x, cal, modo, split)
    return -np.log(Pm[np.arange(len(y)), y])

print('\n--- (c) Recalibrar menos congelado, por tramo (en milésimas; POSITIVO = recalibrar fue PEOR) ---')
filas = []
for split in SPLITS:
    ev = P[P['sidx'] == SPLITS[split]['evalua']].copy()
    ev['tramo'] = ev['torneo'] + '-' + ev['fase_tipo']
    for cal in ['A', 'B365', 'B90', 'Ca']:
        ev['dif'] = _ll_vec(x0, cal, 'wf', split) - _ll_vec(x0, cal, 'congelado', split)
        g = (ev.groupby('tramo')['dif'].mean() * 1000).round(1).to_dict()
        filas.append({'split': split, 'cal': cal, **g, 'total': round(ev['dif'].mean() * 1000, 1)})
print(pd.DataFrame(filas).to_string(index=False))
for split in SPLITS:
    ev = P[P['sidx'] == SPLITS[split]['evalua']]
    print(f'  n por tramo ({split}):', (ev['torneo'] + '-' + ev['fase_tipo']).value_counts().to_dict())

print('\n--- (d) Ventaja local: la que usa el Elo frente a la real ---')
S_obs = np.where(P.y == 2, 1.0, np.where(P.y == 1, 0.5, 0.0))
filas = []
for s_ in range(4):
    m = (P['sidx'] == s_).values
    dl = (P['elo_local_pre'] - P['elo_visita_pre']).values[m]
    S_m = S_obs[m].mean()
    f = lambda h: (1 / (1 + 10 ** (-(dl + h) / 400))).mean() - S_m
    filas.append(dict(sidx=s_, puntos_local_obs=S_m, E_local_con_H150=(1 / (1 + 10 ** (-(dl + 150) / 400))).mean(),
                      H_justa=brentq(f, -300, 400)))
print(pd.DataFrame(filas).round(3).to_string(index=False))
print('  (puntos_local_obs = % victorias + mitad de % empates; H_justa = la H con la que el Elo esperaría eso)')

print('\n--- (e) Efecto de calendario en el Elo de E-2 ---')
dl_home = (P['elo_local_post'] - P['elo_local_pre'])
sesgo = dl_home.groupby(P['sidx']).mean()
print('Puntos que gana (o pierde) en promedio el LOCAL por partido:', sesgo[sesgo.index <= 3].round(2).to_dict())
dh = sesgo[sesgo.index <= 3].mean()
print('Sesgo en una ventana de 10 partidos según cuántos juega de local (puntos de Elo):')
print({f'{h} de local': round(float((2 * h - 10) * dh), 1) for h in [3, 4, 5, 6, 7]})

print('\n--- (f) Convergencia: dispersión del Elo entre equipos ---')
lg = largo.sort_values('fecha')
for s_ in range(4):
    d = lg[lg['sidx'] == s_]
    ini = d.groupby('team')['elo_pre'].first(); fin = d.groupby('team')['elo_post'].last()
    print(f'  temporada {s_}: desv. est. al inicio {ini.std():.0f} · al cierre {fin.std():.0f}')

print('\n--- (g) ¿Cuánto cuesta usar una H cercana a la real? (R 0.25 / 0; calibración congelada) ---')
filas = []
for h in [50, 75, 100, 150]:
    r = correr_elo(P, E2['K'], h, 0.25, 0.0)
    x = (r[0] + h - r[1]) / 400
    sh = pd.Series(r[2] - r[0]).groupby(P['sidx']).mean()
    fila = dict(H=h, sesgo_local_por_partido=round(sh[sh.index <= 3].mean(), 2))
    for split in SPLITS:
        for cal in ['A', 'B365']:
            Pm, y, _ = evaluar(P, x, cal, 'congelado', split)
            fila[f'{cal}_{split}'] = metricas(Pm, y)[0]
    filas.append(fila)
print(pd.DataFrame(filas).round(4).to_string(index=False))

print('\n--- (h) TECHO del problema de nivel (oráculo: NO se usa para elegir nada) ---')
print('    Pendiente fija (calibrada con el pasado); solo los cortes se ajustan con la misma temporada evaluada.')
def _ajusta_cortes(b, x, y):
    idx = np.arange(len(y))
    nll = lambda c: -np.log(probs_cal(np.array([b, c[0], c[1]]), x, np.zeros(len(y)), 'A')[idx, y]).mean()
    return minimize(nll, np.array([-0.9, 0.17]), method='BFGS').x
for split in SPLITS:
    s = SPLITS[split]
    bf = ((P['fecha'] >= CALENTAMIENTO) & (P['sidx'] <= s['calib_hasta'])).values
    ev = (P['sidx'] == s['evalua']).values
    p, _ = ajustar_cal(x0[bf], P['y'].values[bf], P['t'].values[bf], np.ones(bf.sum()), 'A')
    c = _ajusta_cortes(p[0], x0[ev], P['y'].values[ev])
    Pm = probs_cal(np.array([p[0], c[0], c[1]]), x0[ev], np.zeros(ev.sum()), 'A')
    ll_or = metricas(Pm, P['y'].values[ev])[0]
    ll_A = G[(G.split == split) & (G.R_ver == 0.25) & (G.R_inv == 0) & (G.H == 'universal') &
             (G.cal == 'A') & (G.modo == 'congelado')].logloss.iloc[0]
    print(f'  {split}: A congelado {ll_A:.4f} · techo con nivel perfecto {ll_or:.4f} · en juego {ll_A - ll_or:.4f}')

print('\n--- (i) Partido parejo (mismo Elo; solo pesa la localía): qué predice cada calibrador ---')
s = SPLITS['externo']
bf = ((P['fecha'] >= CALENTAMIENTO) & (P['sidx'] <= s['calib_hasta'])).values
y_, t_ = P['y'].values, P['t'].values
t_ini, t_fin = P.loc[P.sidx == 3, 't'].min(), P.loc[P.sidx == 3, 't'].max()
filas = []
for cal in ['A', 'B365', 'B90', 'Ca', 'Cb']:
    tipo = _tipo(cal)
    p, ok = ajustar_cal(x0[bf], y_[bf], t_[bf], pesos(cal, t_[bf], t_[bf].max()), tipo)
    for nom, tt in [('jul-2024', t_ini), ('may-2025', t_fin)]:
        pr = probs_cal(p, np.array([E2['H'] / 400]), np.array([tt]), tipo)[0]
        filas.append(dict(cal=cal, momento=nom, pendiente_b=round(p[0], 2),
                          local=pr[2] * 100, empate=pr[1] * 100, visita=pr[0] * 100))
print(pd.DataFrame(filas).round(1).to_string(index=False))
print(f'R3d terminó en {time.time() - t0:.0f} s.')

R2c Calibración de pases

In [ ]:
# =====================================================================
# R2c · Calibración de pass.pass_success_probability por tramos de dificultad
# Prueba 18 del framework v1.1 · Sesión 5 · 2026-10-09
# Pregunta: el −0.53 entre riesgo y ejecución (M2p), ¿es del juego o del modelo de SB?
# Solo entrenamiento (las 4 primeras temporadas). Antes: celda 1 (Drive) y R0. Tarda ~5-10 min.
# =====================================================================

# [R2c] ----------------------------------------------------------------
import os, glob, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026/')
DP = os.path.join(BASE, 'datos_parquet'); FW = os.path.join(DP, 'fw1'); SAL = os.path.join(BASE, 'salidas')
DIR_EV = os.path.join(DP, 'liga', 'eventos')
REHACER = False
REANUDACIONES = {'Goal Kick', 'Kick Off', 'Free Kick', 'Throw-in', 'Corner'}
COLS = ['type.name', 'team.name', 'period', 'pass.pass_success_probability', 'pass.outcome.name',
        'pass.type.name', 'set_piece_phase', 'under_pressure']
TRAMOS = [0, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 1.0001]
ETQ = ['<30%', '30-50%', '50-70%', '70-80%', '80-90%', '90-95%', '95%+']
t0 = time.time()

# 1. Partidos de entrenamiento (las 4 temporadas más antiguas, por fecha)
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp.columns: ptp = ptp[ptp['jugado'].astype(bool)]
ptp['fecha'] = pd.to_datetime(ptp['fecha'])
orden = ptp.groupby('temporada')['fecha'].min().sort_values().index.tolist()
ENTRENAMIENTO = orden[:4]
mt = ptp[ptp['temporada'].isin(ENTRENAMIENTO)].drop_duplicates('match_id')[['match_id', 'temporada']]
print('Temporadas de entrenamiento:', ENTRENAMIENTO, '| partidos:', len(mt))

# 2. Pases de juego abierto (periodos 1 y 2, sin reanudaciones, sin marca de balón parado)
ruta_cache = os.path.join(FW, 'pases_prob_entrenamiento.parquet')
if os.path.exists(ruta_cache) and not REHACER:
    X = pd.read_parquet(ruta_cache); print('Pases cargados de caché:', len(X))
else:
    piezas, sin_archivo, sin_prob = [], 0, 0
    for mid, temp in mt.itertuples(index=False):
        ruta = os.path.join(DIR_EV, f'{mid}.parquet')
        if not os.path.exists(ruta): sin_archivo += 1; continue
        disp = set(pq.read_schema(ruta).names)
        d = pd.read_parquet(ruta, columns=[c for c in COLS if c in disp])
        for c in COLS:
            if c not in d.columns: d[c] = np.nan
        d = d[(d['type.name'] == 'Pass') & d['period'].isin([1, 2])]
        d = d[~d['pass.type.name'].isin(REANUDACIONES) & d['set_piece_phase'].isna()]
        if d['pass.pass_success_probability'].isna().all(): sin_prob += 1
        d = d[d['pass.pass_success_probability'].notna()]
        piezas.append(pd.DataFrame({
            'match_id': mid, 'temporada': temp, 'team': d['team.name'].values,
            'p': d['pass.pass_success_probability'].astype(float).values,
            'ok': d['pass.outcome.name'].isna().values.astype(float),
            'presion': d['under_pressure'].eq(True).values}))
    X = pd.concat(piezas, ignore_index=True); X.to_parquet(ruta_cache, index=False)
    print(f'Pases leídos: {len(X):,} | partidos sin archivo: {sin_archivo} | partidos sin probabilidad: {sin_prob}')

X['tramo'] = pd.cut(X['p'], TRAMOS, labels=ETQ, right=False)
print('\n--- Chequeos (deben parecerse a R2a) ---')
print(f"Precisión {X.ok.mean()*100:.1f}% · riesgo {1 - X.p.mean():.3f} · ejecución {100*(X.ok - X.p).mean():+.2f} por cada 100 · "
      f"p fuera de [0,1]: {int(((X.p < 0) | (X.p > 1)).sum())} · bajo presión {X.presion.mean()*100:.1f}%")
print('Por temporada (ejecución por cada 100):', (100 * (X.ok - X.p)).groupby(X.temporada).mean().round(2).to_dict())

# 3. Curva de calibración por tramo
def curva(d):
    g = d.groupby('tramo', observed=False).agg(n=('ok', 'size'), predicho=('p', 'mean'), observado=('ok', 'mean'))
    g['% de pases'] = 100 * g['n'] / g['n'].sum()
    g['dif_pp'] = 100 * (g['observado'] - g['predicho'])
    g['z'] = g['dif_pp'] / (100 * np.sqrt(g['predicho'] * (1 - g['predicho']) / g['n']))
    g['predicho'] *= 100; g['observado'] *= 100
    return g[['n', '% de pases', 'predicho', 'observado', 'dif_pp', 'z']]
print('\n--- (a) Calibración por tramo de dificultad (toda la muestra) ---')
print('    dif_pp = observado − predicho, en puntos porcentuales; |z| > 3 = no es azar')
print(curva(X).round(2).to_string())

print('\n--- (b) dif_pp por tramo y temporada (¿el modelo se comporta igual cada año?) ---')
X['res_c'] = X['ok'] - X['p']
tb = (100 * X.groupby(['temporada', 'tramo'], observed=False)['res_c'].mean()).unstack()
print(tb.round(2).to_string())

print('\n--- (c) dif_pp por tramo, con y sin presión ---')
tp = (100 * X.groupby(['presion', 'tramo'], observed=False)['res_c'].mean()).unstack()
print(tp.round(2).to_string())

# 4. Del juego o del modelo: tres versiones de la ejecución por equipo-temporada
#    cruda      = completados − Σp                      (M2p tal cual)
#    corregida  = completados − lo que la LIGA completa de verdad en pases de esa dificultad (20 tramos finos)
#    estandar.  = la corregida, pero con la mezcla de dificultad de la liga (estandarización directa, guía §6)
X['fino'] = pd.qcut(X['p'], 20, labels=False, duplicates='drop')
X['p_liga'] = X.groupby('fino')['ok'].transform('mean')
X['res_l'] = X['ok'] - X['p_liga']
peso_liga = X['tramo'].value_counts(normalize=True)
E = X.groupby(['temporada', 'team']).agg(pases=('ok', 'size'), riesgo=('p', lambda s: 1 - s.mean()),
                                         ej_cruda=('res_c', 'mean'), ej_corregida=('res_l', 'mean'))
E[['ej_cruda', 'ej_corregida']] *= 100
R_ = X.groupby(['temporada', 'team', 'tramo'], observed=True)['res_l'].mean().unstack()
W_ = R_.notna() * peso_liga.reindex(R_.columns).values
E['ej_estandarizada'] = 100 * (R_.fillna(0) * W_).sum(1) / W_.sum(1)

ruta_elo = os.path.join(FW, 'elo_equipo_partido_e2.parquet')
if os.path.exists(ruta_elo):
    elo = pd.read_parquet(ruta_elo)
    elo = elo[elo['temporada'].isin(ENTRENAMIENTO)].groupby(['temporada', 'team'])['elo_pre'].mean()
    E = E.join(elo.rename('elo_medio'))
    print(f'\nEquipos-temporada: {len(E)} | con Elo: {int(E.elo_medio.notna().sum())} (deben ser todos; si no, los nombres no cruzan)')

def r_ic(a, b):
    m = a.notna() & b.notna(); r = np.corrcoef(a[m], b[m])[0, 1]; n = int(m.sum())
    z = np.arctanh(r); se = 1 / np.sqrt(n - 3)
    return f'{r:+.2f}  [{np.tanh(z - 1.96*se):+.2f}, {np.tanh(z + 1.96*se):+.2f}]  n={n}'
print('\n--- (d) Correlación entre equipos-temporada (IC 95%) ---')
for c in ['ej_cruda', 'ej_corregida', 'ej_estandarizada']:
    print(f'  riesgo vs {c:17s} {r_ic(E.riesgo, E[c])}')
if 'elo_medio' in E:
    for c in ['riesgo', 'ej_cruda', 'ej_corregida']:
        print(f'  Elo medio vs {c:13s} {r_ic(E.elo_medio, E[c])}')
print('\n--- (e) Dispersión entre equipos-temporada (mín / p10 / mediana / p90 / máx) ---')
print(E[['riesgo', 'ej_cruda', 'ej_corregida', 'ej_estandarizada']].quantile([0, .1, .5, .9, 1]).round(3).to_string())
E.reset_index().to_parquet(os.path.join(FW, 'ejecucion_pase_equipo_temporada.parquet'), index=False)
print(f'\nR2c terminó en {time.time() - t0:.0f} s. Guardado: fw1/pases_prob_entrenamiento.parquet y '
      'fw1/ejecucion_pase_equipo_temporada.parquet')

R2c v2

In [ ]:
# =====================================================================
# R2c · Calibración de pass.pass_success_probability por tramos de dificultad
# Prueba 18 del framework v1.1 · Sesión 5 · 2026-10-09
# Pregunta: el −0.53 entre riesgo y ejecución (M2p), ¿es del juego o del modelo de SB?
# Solo entrenamiento (las 4 primeras temporadas). Antes: celda 1 (Drive) y R0. Tarda ~5-10 min.
# =====================================================================

# [R2c] ----------------------------------------------------------------
import os, glob, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026/')
DP = os.path.join(BASE, 'datos_parquet'); FW = os.path.join(DP, 'fw1'); SAL = os.path.join(BASE, 'salidas')
DIR_EV = os.path.join(DP, 'liga', 'eventos')
REHACER = False
REANUDACIONES = {'Goal Kick', 'Kick Off', 'Free Kick', 'Throw-in', 'Corner'}
COLS = ['type.name', 'team.name', 'period', 'pass.pass_success_probability', 'pass.outcome.name',
        'pass.type.name', 'set_piece_phase', 'under_pressure']
TRAMOS = [0, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95, 1.0001]
ETQ = ['<30%', '30-50%', '50-70%', '70-80%', '80-90%', '90-95%', '95%+']
t0 = time.time()

# 1. Partidos de entrenamiento (las 4 temporadas más antiguas, por fecha)
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp.columns: ptp = ptp[ptp['jugado'].astype(bool)]
ptp['fecha'] = pd.to_datetime(ptp['fecha'])
orden = ptp.groupby('temporada')['fecha'].min().sort_values().index.tolist()
ENTRENAMIENTO = orden[:4]
mt = ptp[ptp['temporada'].isin(ENTRENAMIENTO)].drop_duplicates('match_id')[['match_id', 'temporada']]
print('Temporadas de entrenamiento:', ENTRENAMIENTO, '| partidos:', len(mt))

# 2. Pases de juego abierto (periodos 1 y 2, sin reanudaciones, sin marca de balón parado)
ruta_cache = os.path.join(FW, 'pases_prob_entrenamiento.parquet')
if os.path.exists(ruta_cache) and not REHACER:
    X = pd.read_parquet(ruta_cache); print('Pases cargados de caché:', len(X))
else:
    piezas, sin_archivo, sin_prob = [], 0, 0
    for mid, temp in mt.itertuples(index=False):
        ruta = os.path.join(DIR_EV, f'{mid}.parquet')
        if not os.path.exists(ruta): sin_archivo += 1; continue
        disp = set(pq.read_schema(ruta).names)
        d = pd.read_parquet(ruta, columns=[c for c in COLS if c in disp])
        for c in COLS:
            if c not in d.columns: d[c] = np.nan
        d = d[(d['type.name'] == 'Pass') & d['period'].isin([1, 2])]
        d = d[~d['pass.type.name'].isin(REANUDACIONES) & d['set_piece_phase'].isna()]
        if d['pass.pass_success_probability'].isna().all(): sin_prob += 1
        d = d[d['pass.pass_success_probability'].notna()]
        piezas.append(pd.DataFrame({
            'match_id': mid, 'temporada': temp, 'team': d['team.name'].values,
            'p': d['pass.pass_success_probability'].astype(float).values,
            'ok': d['pass.outcome.name'].isna().values.astype(float),
            'presion': d['under_pressure'].eq(True).values}))
    X = pd.concat(piezas, ignore_index=True); X.to_parquet(ruta_cache, index=False)
    print(f'Pases leídos: {len(X):,} | partidos sin archivo: {sin_archivo} | partidos sin probabilidad: {sin_prob}')

X['tramo'] = pd.cut(X['p'], TRAMOS, labels=ETQ, right=False)
print('\n--- Chequeos (deben parecerse a R2a) ---')
print(f"Precisión {X.ok.mean()*100:.1f}% · riesgo {1 - X.p.mean():.3f} · ejecución {100*(X.ok - X.p).mean():+.2f} por cada 100 · "
      f"p fuera de [0,1]: {int(((X.p < 0) | (X.p > 1)).sum())} · bajo presión {X.presion.mean()*100:.1f}%")
print('Por temporada (ejecución por cada 100):', (100 * (X.ok - X.p)).groupby(X.temporada).mean().round(2).to_dict())

# 3. Curva de calibración por tramo
def curva(d):
    g = d.groupby('tramo', observed=False).agg(n=('ok', 'size'), predicho=('p', 'mean'), observado=('ok', 'mean'))
    g['% de pases'] = 100 * g['n'] / g['n'].sum()
    g['dif_pp'] = 100 * (g['observado'] - g['predicho'])
    g['z'] = g['dif_pp'] / (100 * np.sqrt(g['predicho'] * (1 - g['predicho']) / g['n']))
    g['predicho'] *= 100; g['observado'] *= 100
    return g[['n', '% de pases', 'predicho', 'observado', 'dif_pp', 'z']]
print('\n--- (a) Calibración por tramo de dificultad (toda la muestra) ---')
print('    dif_pp = observado − predicho, en puntos porcentuales; |z| > 3 = no es azar')
print(curva(X).round(2).to_string())

print('\n--- (b) dif_pp por tramo y temporada (¿el modelo se comporta igual cada año?) ---')
X['res_c'] = X['ok'] - X['p']
tb = (100 * X.groupby(['temporada', 'tramo'], observed=False)['res_c'].mean()).unstack()
print(tb.round(2).to_string())

print('\n--- (c) dif_pp por tramo, con y sin presión ---')
tp = (100 * X.groupby(['presion', 'tramo'], observed=False)['res_c'].mean()).unstack()
print(tp.round(2).to_string())

# 4. Del juego o del modelo: tres versiones de la ejecución por equipo-temporada
#    cruda      = completados − Σp                      (M2p tal cual)
#    corregida  = completados − lo que la LIGA completa de verdad en pases de esa dificultad (20 tramos finos)
#    estandar.  = la corregida, pero con la mezcla de dificultad de la liga (estandarización directa, guía §6)
X['fino'] = pd.qcut(X['p'], 20, labels=False, duplicates='drop')
X['p_liga'] = X.groupby('fino')['ok'].transform('mean')
X['res_l'] = X['ok'] - X['p_liga']
# S5-10 opción C (decidida): referencia de liga por dificultad Y presión
X['p_liga_pres'] = X.groupby(['fino', 'presion'])['ok'].transform('mean')
X['res_lp'] = X['ok'] - X['p_liga_pres']
peso_liga = X['tramo'].value_counts(normalize=True)
E = X.groupby(['temporada', 'team']).agg(pases=('ok', 'size'), riesgo=('p', lambda s: 1 - s.mean()),
                                         ej_cruda=('res_c', 'mean'), ej_corregida=('res_l', 'mean'),
                                         ej_corr_presion=('res_lp', 'mean'), pct_presion=('presion', 'mean'))
E[['ej_cruda', 'ej_corregida', 'ej_corr_presion', 'pct_presion']] *= 100
R_ = X.groupby(['temporada', 'team', 'tramo'], observed=True)['res_l'].mean().unstack()
W_ = R_.notna() * peso_liga.reindex(R_.columns).values
E['ej_estandarizada'] = 100 * (R_.fillna(0) * W_).sum(1) / W_.sum(1)

ruta_elo = os.path.join(FW, 'elo_equipo_partido_e2.parquet')
if os.path.exists(ruta_elo):
    elo = pd.read_parquet(ruta_elo)
    elo = elo[elo['temporada'].isin(ENTRENAMIENTO)].groupby(['temporada', 'team'])['elo_pre'].mean()
    E = E.join(elo.rename('elo_medio'))
    print(f'\nEquipos-temporada: {len(E)} | con Elo: {int(E.elo_medio.notna().sum())} (deben ser todos; si no, los nombres no cruzan)')

def r_ic(a, b):
    m = a.notna() & b.notna(); r = np.corrcoef(a[m], b[m])[0, 1]; n = int(m.sum())
    z = np.arctanh(r); se = 1 / np.sqrt(n - 3)
    return f'{r:+.2f}  [{np.tanh(z - 1.96*se):+.2f}, {np.tanh(z + 1.96*se):+.2f}]  n={n}'
print('\n--- (d) Correlación entre equipos-temporada (IC 95%) ---')
for c in ['ej_cruda', 'ej_corregida', 'ej_estandarizada', 'ej_corr_presion']:
    print(f'  riesgo vs {c:17s} {r_ic(E.riesgo, E[c])}')
if 'elo_medio' in E:
    for c in ['riesgo', 'ej_cruda', 'ej_corregida', 'ej_corr_presion']:
        print(f'  Elo medio vs {c:13s} {r_ic(E.elo_medio, E[c])}')
print('\n--- (d2) ¿La corrección por presión le quita castigo al que más presión recibe? ---')
print(f"  % de pases bajo presión vs ej_corregida     {r_ic(E.pct_presion, E.ej_corregida)}")
print(f"  % de pases bajo presión vs ej_corr_presion  {r_ic(E.pct_presion, E.ej_corr_presion)}")
print(f"  ej_corregida vs ej_corr_presion             {r_ic(E.ej_corregida, E.ej_corr_presion)}")
print('\n--- (e) Dispersión entre equipos-temporada (mín / p10 / mediana / p90 / máx) ---')
print(E[['riesgo', 'ej_cruda', 'ej_corregida', 'ej_estandarizada', 'ej_corr_presion']].quantile([0, .1, .5, .9, 1]).round(3).to_string())
E.reset_index().to_parquet(os.path.join(FW, 'ejecucion_pase_equipo_temporada.parquet'), index=False)
print(f'\nR2c terminó en {time.time() - t0:.0f} s. Guardado: fw1/pases_prob_entrenamiento.parquet y '
      'fw1/ejecucion_pase_equipo_temporada.parquet')

R3 Elo xg

In [ ]:
# [R3e] Elo de xG (S4-22, decidido en la sesión 5) ------------------
# Requiere R3-0 y R3a en memoria (R3b, R3c y R3d NO hacen falta). Lee los tiros de cada partido (~10-15 min la
# primera vez; luego sale de caché). Imprime solo 2021/22 a 2024/25.
# Reglas: puntos esperados exactos (B) · tiros agrupados por posesión · penales incluidos · mismos parámetros que E-2
#         · periodos 1 a 4 (sin tanda). Papel: emparejar por suerte en T-1; no es el resultado principal.
import pyarrow.parquet as pq
t0 = time.time()
DIR_EV = os.path.join(DP, 'liga', 'eventos')
ruta_tiros = os.path.join(FW, 'tiros_xg.parquet')
COLS_T = ['type.name', 'team.name', 'period', 'possession', 'shot.statsbomb_xg', 'shot.type.name']
if os.path.exists(ruta_tiros) and not REHACER:
    T = pd.read_parquet(ruta_tiros); print('Tiros cargados de caché:', len(T))
else:
    piezas, sin_archivo = [], 0
    for mid in P['match_id']:
        ruta = os.path.join(DIR_EV, f'{mid}.parquet')
        if not os.path.exists(ruta): sin_archivo += 1; continue
        disp = set(pq.read_schema(ruta).names)
        d = pd.read_parquet(ruta, columns=[c for c in COLS_T if c in disp])
        for c in COLS_T:
            if c not in d.columns: d[c] = np.nan
        d = d[(d['type.name'] == 'Shot') & d['period'].isin([1, 2, 3, 4])]
        piezas.append(pd.DataFrame({'match_id': mid, 'team': d['team.name'].values, 'possession': d['possession'].values,
                                    'xg': d['shot.statsbomb_xg'].astype(float).values,
                                    'penal': (d['shot.type.name'] == 'Penalty').values}))
    T = pd.concat(piezas, ignore_index=True); T.to_parquet(ruta_tiros, index=False)
    print(f'Tiros leídos: {len(T):,} | partidos sin archivo: {sin_archivo}')
print(f'Tiros sin xG: {int(T.xg.isna().sum())} | penales: {int(T.penal.sum())} | xG por partido: {T.xg.sum() / P.match_id.nunique():.2f}')

# Probabilidad de gol por posesión: 1 − Π(1 − xG)
Tp = (T.dropna(subset=['xg']).assign(q=lambda d: np.log1p(-d.xg.clip(0, 0.999)))
        .groupby(['match_id', 'team', 'possession'])['q'].sum().reset_index())
Tp['p_gol'] = 1 - np.exp(Tp['q'])

def _dist_goles(ps):
    d = np.array([1.0])
    for p in ps: d = np.convolve(d, [1 - p, p])
    return d
grupos = {k: g['p_gol'].values for k, g in Tp.groupby(['match_id', 'team'])}
xs = []
for r in P.itertuples(index=False):
    dl = _dist_goles(grupos.get((r.match_id, r.local), []))
    dv = _dist_goles(grupos.get((r.match_id, r.visita), []))
    M_ = np.outer(dl, dv)                      # M_[i, j] = P(local i goles, visita j goles)
    p_l = np.tril(M_, -1).sum(); p_e = np.trace(M_)
    xs.append((p_l, p_e, 1 - p_l - p_e,
               sum(grupos.get((r.match_id, r.local), [])), sum(grupos.get((r.match_id, r.visita), []))))
X3 = pd.DataFrame(xs, columns=['xp_local_gana', 'xp_empate', 'xp_visita_gana', 'xg_local', 'xg_visita'])
for c in X3: P[c] = X3[c].values
P['S_xg'] = P['xp_local_gana'] + 0.5 * P['xp_empate']
P.loc[~P['match_id'].isin(T['match_id'].unique()), 'S_xg'] = np.nan   # sin eventos: el Elo de xG no se mueve
P['S_real'] = np.where(P.gl > P.gv, 1.0, np.where(P.gl == P.gv, 0.5, 0.0))

def correr_elo_s(P, S, K, H, R_ver, R_inv):
    """Igual que correr_elo, pero con un resultado continuo S y sin multiplicador de margen."""
    rating = dict(ELO_INICIAL); temp_act = torneo_act = None
    pre_l = np.empty(len(P)); pre_v = np.empty(len(P)); post_l = np.empty(len(P)); post_v = np.empty(len(P))
    for i, r in enumerate(P.itertuples(index=False)):
        if temp_act is not None and r.temporada != temp_act:
            for t in rating: rating[t] = INICIO + (1 - R_ver) * (rating[t] - INICIO)
        elif torneo_act is not None and r.torneo != torneo_act:
            for t in rating: rating[t] = INICIO + (1 - R_inv) * (rating[t] - INICIO)
        temp_act, torneo_act = r.temporada, r.torneo
        rl = rating.get(r.local_id, INICIO); rv = rating.get(r.visita_id, INICIO)
        E = 1.0 / (1.0 + 10 ** (-(rl + H - rv) / 400))
        delta = 0.0 if np.isnan(S[i]) else K * (S[i] - E)
        rating[r.local_id] = rl + delta; rating[r.visita_id] = rv - delta
        pre_l[i], pre_v[i], post_l[i], post_v[i] = rl, rv, rl + delta, rv - delta
    return pre_l, pre_v, post_l, post_v
xl, xv, xlp, xvp = correr_elo_s(P, P['S_xg'].values, **E2)
P['eloxg_local_pre'], P['eloxg_visita_pre'], P['eloxg_local_post'], P['eloxg_visita_post'] = xl, xv, xlp, xvp

ent = P[P['sidx'] <= 3]
print('\n--- (a) Chequeos por temporada: puntos del local, reales frente a esperados por xG ---')
print(ent.groupby('sidx').agg(partidos=('match_id', 'size'), S_real=('S_real', 'mean'), S_xg=('S_xg', 'mean'),
                              xg_local=('xg_local', 'mean'), xg_visita=('xg_visita', 'mean'),
                              p_empate_xg=('xp_empate', 'mean')).round(3).to_string())
print('Partidos sin eventos (el Elo de xG no se mueve en ellos):', int(P['S_xg'].isna().sum()))
print('Partidos sin ningún tiro de alguno de los dos equipos:', int(((ent.xg_local == 0) | (ent.xg_visita == 0)).sum()))

# Tabla por equipo-temporada: puntos reales frente a esperados
lg2 = pd.concat([
    ent.assign(team=ent.local, pts=np.select([ent.gl > ent.gv, ent.gl == ent.gv], [3, 1], 0),
               xpts=3 * ent.xp_local_gana + ent.xp_empate, elo_post=ent.elo_local_post, eloxg_post=ent.eloxg_local_post),
    ent.assign(team=ent.visita, pts=np.select([ent.gv > ent.gl, ent.gl == ent.gv], [3, 1], 0),
               xpts=3 * ent.xp_visita_gana + ent.xp_empate, elo_post=ent.elo_visita_post, eloxg_post=ent.eloxg_visita_post)
])[['match_id', 'fecha', 'temporada', 'sidx', 'team', 'pts', 'xpts', 'elo_post', 'eloxg_post']]
ts = lg2.groupby(['sidx', 'team']).agg(pj=('pts', 'size'), pts=('pts', 'sum'), xpts=('xpts', 'sum'))
ts['suerte_pts'] = ts['pts'] - ts['xpts']
print('\n--- (b) Equipo-temporada: puntos reales frente a esperados ---')
print(f"Correlación puntos-xPts: {ts.pts.corr(ts.xpts):.2f} · suerte (pts − xPts): desv. est. {ts.suerte_pts.std():.1f}, "
      f"rango {ts.suerte_pts.min():.1f} a {ts.suerte_pts.max():.1f} puntos por temporada")
print('Más afortunados:'); print(ts.sort_values('suerte_pts').tail(3).round(1).to_string())
print('Menos afortunados:'); print(ts.sort_values('suerte_pts').head(3).round(1).to_string())

fin = lg2.sort_values('fecha').groupby('team').tail(1).set_index('team')
fin = fin[fin['sidx'] == 3]
print('\n--- (c) Al cierre de 2024/25: Elo real frente a Elo de xG ---')
print(f"Correlación: {fin.elo_post.corr(fin.eloxg_post):.2f} · desv. est. real {fin.elo_post.std():.0f}, xG {fin.eloxg_post.std():.0f}")
fin['brecha'] = fin['elo_post'] - fin['eloxg_post']
print(fin.sort_values('eloxg_post', ascending=False)[['elo_post', 'eloxg_post', 'brecha']].round(0).to_string())

print('\n--- (d) América por etapa: Elo real frente a Elo de xG ---')
am2 = ptp[ptp['team'].str.contains('Am[eé]rica', case=False, regex=True)][['match_id', 'team', 'tec_p', 'etapa']]
am2 = am2.merge(lg2, on=['match_id', 'team']).sort_values('fecha')
am2 = am2.merge(largo[['match_id', 'team', 'elo_pre']], on=['match_id', 'team'])
am2['eloxg_pre'] = am2.apply(lambda r: P.loc[P.match_id == r.match_id, 'eloxg_local_pre' if
                             P.loc[P.match_id == r.match_id, 'local'].iloc[0] == r.team else 'eloxg_visita_pre'].iloc[0], axis=1)
ra = am2.groupby('etapa').agg(tecnico=('tec_p', 'first'), pj=('pts', 'size'), pts=('pts', 'sum'), xpts=('xpts', 'sum'),
                              elo_ini=('elo_pre', 'first'), elo_fin=('elo_post', 'last'),
                              eloxg_ini=('eloxg_pre', 'first'), eloxg_fin=('eloxg_post', 'last'))
ra['cambio_real'] = ra.elo_fin - ra.elo_ini; ra['cambio_xg'] = ra.eloxg_fin - ra.eloxg_ini
ra['suerte_pts'] = ra.pts - ra.xpts
print(ra[['tecnico', 'pj', 'pts', 'xpts', 'suerte_pts', 'cambio_real', 'cambio_xg']].round(1).to_string())

cols_x = ['match_id', 'fecha', 'temporada', 'sidx', 'local', 'visita', 'xg_local', 'xg_visita', 'xp_local_gana',
          'xp_empate', 'xp_visita_gana', 'S_xg', 'eloxg_local_pre', 'eloxg_visita_pre', 'eloxg_local_post', 'eloxg_visita_post']
P[cols_x].to_parquet(os.path.join(FW, 'elo_xg_partido.parquet'), index=False)
print(f'\nR3e terminó en {time.time() - t0:.0f} s. Guardado: fw1/tiros_xg.parquet y fw1/elo_xg_partido.parquet')


#Sesión 6

R4 00 Inspección

In [ ]:
# =====================================================================
# R4-00 · Inspección para el ajuste por contexto (solo lectura)
# Sesión 6 · 2026-10-12 · Hackathon ISAC 2026, Track 2
# Antes: celda 1 (Drive) y R0. No escribe nada. Tarda ~1 min.
# Objetivo: ver las columnas reales de las tablas de R1, R2a y R2b y de los
# partidos (árbitro, fase), para construir R4 sobre las mismas definiciones.
# =====================================================================

#  [R4-00] -----------------------------------------------------------
import os, glob, json, re, gzip
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026/')
DP = os.path.join(BASE, 'datos_parquet'); FW = os.path.join(DP, 'fw1'); DC = os.path.join(BASE, 'datos_crudos')

def mb(r): return os.path.getsize(r) / 1e6

print('=== (a) Archivos en fw1 ===')
for r in sorted(glob.glob(os.path.join(FW, '*'))):
    print(f'  {os.path.basename(r):45s} {mb(r):8.1f} MB')

def esquema(ruta, muestra=True, max_cols_muestra=40):
    pf = pq.ParquetFile(ruta)
    nc = pf.metadata.num_columns
    print(f'\n### {os.path.relpath(ruta, BASE)} · filas {pf.metadata.num_rows:,} · columnas {nc}')
    print('  ' + ', '.join(f'{f.name}:{str(f.type)[:10]}' for f in pf.schema_arrow))
    df = pf.read_row_group(0).to_pandas()
    if muestra and nc <= max_cols_muestra:
        with pd.option_context('display.max_columns', 60, 'display.width', 250, 'display.max_colwidth', 22):
            print(df.head(3).to_string())
    # Columnas con pocos valores distintos: muestran cómo están codificados estados, fases, ventanas, etc.
    for c in df.columns:
        try:
            u = df[c].nunique(dropna=False)
        except TypeError:
            continue
        if 1 < u <= 25 and df[c].dtype != float:
            vc = df[c].value_counts(dropna=False).head(12)
            print(f'  · {c}: ' + ', '.join(f'{k}={v}' for k, v in vc.items()))

print('\n=== (b) Tablas por temporada de R1, R2a y R2b (se muestra la primera de cada tipo) ===')
for patron in ['eventos_*.parquet', 'posesiones_*.parquet', 'componentes_*.parquet', 'presion_*.parquet']:
    rutas = sorted(glob.glob(os.path.join(FW, patron)))
    print(f'\n--- {patron}: {len(rutas)} archivos: {[os.path.basename(r) for r in rutas]}')
    if rutas:
        esquema(rutas[0])

print('\n=== (c) Tablas de apoyo ===')
for nombre in ['partido_tecnico_periodo.parquet', 'equipo_partido_marcador.parquet']:
    r = os.path.join(DP, nombre)
    if os.path.exists(r): esquema(r)
    else: print('  no existe:', nombre)

r = os.path.join(FW, 'parametros_fw1.json')
if os.path.exists(r):
    with open(r) as f: prm = json.load(f)
    print('\n=== (d) parametros_fw1.json ===')
    print(json.dumps(prm, ensure_ascii=False, indent=1)[:2500])

print('\n=== (e) Partidos: árbitro y fase ===')
rutas = sorted(glob.glob(os.path.join(DC, 'matches_73_*.json*')) + glob.glob(os.path.join(DP, 'matches_73_*.parquet')))
piezas = []
for ruta in rutas:
    try:
        if ruta.endswith('.parquet'):
            d = pd.read_parquet(ruta)
            if any(d[c].map(lambda v: isinstance(v, dict)).any() for c in d.columns if d[c].dtype == object):
                d = pd.json_normalize(d.to_dict('records'))
        else:
            abrir = gzip.open if ruta.endswith('.gz') else open
            with abrir(ruta, 'rt', encoding='utf-8') as f: d = pd.json_normalize(json.load(f))
        m = re.search(r'(\d{4}-\d{2}-\d{2})', os.path.basename(ruta))
        d['_fecha_archivo'] = m.group(1) if m else '0000-00-00'
        piezas.append(d)
    except Exception as e:
        print('  aviso:', os.path.basename(ruta), e)
M = pd.concat(piezas, ignore_index=True).sort_values('_fecha_archivo', ascending=False).drop_duplicates('match_id')
print('Partidos únicos en archivos de matches:', len(M))
cols_ref = [c for c in M.columns if re.search('referee|stage|season', c, re.I)]
print('Columnas de árbitro / fase / temporada:', cols_ref)
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
ptp['fecha'] = pd.to_datetime(ptp['fecha'])
orden = ptp.groupby('temporada')['fecha'].min().sort_values().index.tolist()
ent_ids = set(ptp.loc[ptp['temporada'].isin(orden[:4]), 'match_id'])
Me = M[M['match_id'].isin(ent_ids)]
print('Partidos de entrenamiento en matches:', len(Me), 'de', len(ent_ids))
c_ref = next((c for c in ['referee.name', 'referee_name', 'referee'] if c in Me.columns), None)
if c_ref:
    ref = Me[c_ref].map(lambda v: v.get('name') if isinstance(v, dict) else v)
    n = ref.value_counts()
    print(f'Árbitro ({c_ref}): nulos {int(ref.isna().sum())} · árbitros distintos {n.size} · '
          f'partidos por árbitro: mín {n.min()}, mediana {n.median():.0f}, p90 {n.quantile(.9):.0f}, máx {n.max()}')
    print('  árbitros con menos de 10 partidos:', int((n < 10).sum()))
else:
    print('No hay columna de árbitro en los archivos de matches.')
c_st = next((c for c in ['competition_stage.name', 'competition_stage'] if c in Me.columns), None)
if c_st:
    st = Me[c_st].map(lambda v: v.get('name') if isinstance(v, dict) else v)
    print('Fases en entrenamiento:', st.value_counts().to_dict())
print('\nR4-00 terminó. Pega toda la salida.')

R4a

In [ ]:
# =====================================================================
# R4 · Ajuste por contexto (framework v1.1, sección C; decisiones S6-01 a S6-03, A1-A6, B1-B5)
# Sesión 6 · 2026-10-12 · Hackathon ISAC 2026, Track 2
# Dos celdas: R4a (tramos y componentes) y R4b (receta, residuos y perfiles).
# Antes: celda 1 (Drive) y R0. Datos: SOLO entrenamiento (2021/22 a 2024/25). B5.
# Se construye sobre las tablas de R1 (eventos_, posesiones_) para mantener las mismas definiciones,
# y compara sus sumas contra R2a (componentes_) y R2b (presion_) como chequeo de coherencia.
# =====================================================================



# [R4a] Tramos estado × cuarto y componentes por tramo ---------------
import os, glob, json, re, gzip, time, warnings
import numpy as np
import pandas as pd

SEMILLA = 2026
BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026/')
DP = os.path.join(BASE, 'datos_parquet'); FW = os.path.join(DP, 'fw1')
DC = os.path.join(BASE, 'datos_crudos'); SAL = os.path.join(BASE, 'salidas')
os.makedirs(SAL, exist_ok=True)
REHACER = False
t_ini = time.time()

with open(os.path.join(FW, 'parametros_fw1.json')) as f:
    PRM = json.load(f)
PP = PRM['P']
ENT = PRM['temporadas']['entrenamiento']                    # B5: solo entrenamiento
X_MED, X_U3, X_PPDA = PP['tercio_medio_x'], PP['tercio_ultimo_x'], PP['zona_ppda_x']
AREA_X, AREA_Y = PP['area_x'], PP['area_y']
LARGO, AVANCE, CP_S = PP['pase_largo_yd'], PP['progresivo_avance_yd'], PP['contrapresion_s']
PAUSA = 5             # P-2 (X = 5 s)
EXCL_REANUD = 5       # P-14 (S4-29)
CORTE_C1, CORTE_C3 = 25, 70   # A2: cuartos 0-25 | 25-45+añ | 45-70 | 70-90+añ (minuto del reloj de SB)
REANUD_PASE = set(PRM['reanud_pase'])

# ---------- 1. Partidos de entrenamiento: fase, árbitro, marcador ----------
def cargar_matches():
    rutas = sorted(glob.glob(os.path.join(DC, 'matches_73_*.json*')) + glob.glob(os.path.join(DP, 'matches_73_*.parquet')))
    piezas = []
    for ruta in rutas:
        try:
            if ruta.endswith('.parquet'):
                d = pd.read_parquet(ruta)
                if any(d[c].map(lambda v: isinstance(v, dict)).any() for c in d.columns if d[c].dtype == object):
                    d = pd.json_normalize(d.to_dict('records'))
            else:
                abrir = gzip.open if ruta.endswith('.gz') else open
                with abrir(ruta, 'rt', encoding='utf-8') as f:
                    d = pd.json_normalize(json.load(f))
        except Exception as e:
            print('  aviso:', os.path.basename(ruta), e); continue
        m = re.search(r'(\d{4}-\d{2}-\d{2})', os.path.basename(ruta))
        d['_fa'] = m.group(1) if m else '0000-00-00'
        if len(d): piezas.append(d.dropna(axis=1, how='all'))
    M = pd.concat(piezas, ignore_index=True).sort_values('_fa', ascending=False).drop_duplicates('match_id')
    def col(ops):
        return next((c for c in ops if c in M.columns), None)
    out = pd.DataFrame({
        'match_id': M['match_id'],
        'local': M[col(['home_team.home_team_name', 'home_team_name'])],
        'visita': M[col(['away_team.away_team_name', 'away_team_name'])],
        'gl': M['home_score'], 'gv': M['away_score'],
        'fase_nombre': M[col(['competition_stage.name'])],
        'arbitro': M[col(['referee.name'])] if col(['referee.name']) else np.nan})
    return out

ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
ptp = ptp[ptp['jugado'].astype(bool)].copy()
ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp_e = ptp[ptp['temporada'].isin(ENT)].copy()
mids = set(ptp_e['match_id'])
Mm = cargar_matches()
Mm = Mm[Mm['match_id'].isin(mids)].merge(
    ptp_e.drop_duplicates('match_id')[['match_id', 'fecha', 'temporada']], on='match_id')
Mm['torneo'] = Mm['temporada'] + np.where(Mm['fecha'].dt.month >= 7, '-A', '-C')    # D-6

def tipo_fase(n):
    n = '' if pd.isna(n) else str(n)
    if n in ('Apertura', 'Clausura', 'Regular Season'): return 'regular'
    if re.search('Reclasificaci|Play-In', n, re.I): return 'playin'
    if re.search('Quarter', n, re.I): return 'cuartos'
    if re.search('Semi', n, re.I): return 'semis'
    if re.search('Final', n, re.I): return 'final'
    return 'otro'
Mm['tipo'] = Mm['fase_nombre'].map(tipo_fase)
# Corrección de etiquetas de SB (R4 v2.1), por reglas de calendario y no por partido:
#  (i) un partido de eliminación ANTES del primer partido de cuartos de su torneo es play-in;
#  (ii) la final es la serie que contiene el último partido del torneo (sus dos partidos pasan a 'final').
Mm = Mm.sort_values('fecha').reset_index(drop=True)
Mm['tipo_sb'] = Mm['tipo']
ini_cuartos = Mm[Mm['tipo'] == 'cuartos'].groupby('torneo')['fecha'].min()
antes = Mm['tipo'].isin(['semis', 'final']) & (Mm['fecha'] < Mm['torneo'].map(ini_cuartos))
Mm.loc[antes, 'tipo'] = 'playin'
par_ = pd.Series([' | '.join(sorted([a_, b_])) for a_, b_ in zip(Mm['local'], Mm['visita'])], index=Mm.index)
ko_ = Mm['tipo'].isin(['cuartos', 'semis', 'final'])
ultimo = Mm[ko_].groupby('torneo').tail(1)
for t_, p_ in zip(ultimo['torneo'], par_[ultimo.index]):
    Mm.loc[ko_ & (Mm['torneo'] == t_) & (par_ == p_), 'tipo'] = 'final'
cambios = Mm[Mm['tipo'] != Mm['tipo_sb']]
print('Etiquetas corregidas:', len(cambios))
if len(cambios):
    print(cambios[['torneo', 'fecha', 'fase_nombre', 'tipo_sb', 'tipo', 'local', 'visita']]
          .assign(fecha=lambda d: d.fecha.dt.date).to_string(index=False))

# Series de ida y vuelta (O-13 cerrado en la sesión 6): mejor sembrado = local de la vuelta (S6-02).
Mm['par'] = [' | '.join(sorted([a, b])) for a, b in zip(Mm['local'], Mm['visita'])]
Mm = Mm.sort_values('fecha').reset_index(drop=True)
ko = Mm['tipo'].isin(['cuartos', 'semis', 'final'])
g = Mm[ko].groupby(['torneo', 'tipo', 'par'])
Mm.loc[ko, 'n_serie'] = g['match_id'].transform('size')
Mm.loc[ko, 'pierna'] = g.cumcount() + 1
vu = Mm[ko & (Mm['pierna'] == 2)].set_index(['torneo', 'tipo', 'par'])
ida = Mm[ko & (Mm['pierna'] == 1)].set_index(['torneo', 'tipo', 'par'])
clave = list(zip(Mm['torneo'], Mm['tipo'], Mm['par']))
Mm['sembrado'] = [vu['local'].get(k, np.nan) if k_ else np.nan for k, k_ in zip(clave, ko)]
# Diferencia de la ida vista desde el LOCAL del partido actual (solo se usa en la vuelta)
def dif_ida_local(r):
    k = (r.torneo, r.tipo, r.par)
    if not (r.tipo in ('cuartos', 'semis', 'final') and r.pierna == 2 and k in ida.index): return 0.0
    i = ida.loc[k]
    return float(i.gl - i.gv) if i.local == r.local else float(i.gv - i.gl)
Mm['dif_ida'] = [dif_ida_local(r) for r in Mm.itertuples(index=False)]
media = np.where(Mm['tipo'].isin(['cuartos', 'semis']), 0.5, 0.0)                  # medio gol de siembra (S6-01)
signo_sem = np.where(Mm['local'] == Mm['sembrado'], 1.0, -1.0)
Mm['offset'] = Mm['dif_ida'] + media * signo_sem                                     # regla principal (ida incluida, S6-03)
Mm['offset_b'] = np.where(Mm['pierna'] == 1, 0.0, Mm['offset'])                      # sensibilidad: ida como partido normal
mal = ko & (Mm['n_serie'] != 2)
Mm.loc[mal, ['offset', 'offset_b']] = 0.0

print('--- (a) Partidos de entrenamiento por tipo de fase ---')
print(Mm.groupby('tipo').size().to_dict(), '| total', len(Mm))
print('Series de ida y vuelta con un número de partidos distinto de 2 (quedan con marcador simple):',
      int(Mm.loc[mal].groupby(['torneo', 'tipo', 'par']).ngroups) if mal.any() else 0)
if mal.any():
    print(Mm.loc[mal, ['torneo', 'fecha', 'fase_nombre', 'tipo', 'local', 'visita', 'gl', 'gv']]
          .assign(fecha=lambda d: d.fecha.dt.date).sort_values(['torneo', 'fecha']).to_string(index=False))
# Chequeo de siembra: el local de la vuelta debería tener más puntos en la fase regular del torneo
reg = Mm[Mm['tipo'] == 'regular']
pts = pd.concat([reg.assign(eq=reg.local, p=np.select([reg.gl > reg.gv, reg.gl == reg.gv], [3, 1], 0)),
                 reg.assign(eq=reg.visita, p=np.select([reg.gv > reg.gl, reg.gl == reg.gv], [3, 1], 0))]
                ).groupby(['torneo', 'eq'])['p'].sum()
sv = Mm[ko & (Mm['pierna'] == 2) & ~mal].copy()
sv['pts_sem'] = [pts.get((t, s), np.nan) for t, s in zip(sv.torneo, sv.sembrado)]
sv['pts_otro'] = [pts.get((t, (b if a == s else a)), np.nan) for t, a, b, s in zip(sv.torneo, sv.local, sv.visita, sv.sembrado)]
print(f"Series: {len(sv)} · local de la vuelta con MÁS puntos: {int((sv.pts_sem > sv.pts_otro).sum())} · "
      f"empatado en puntos: {int((sv.pts_sem == sv.pts_otro).sum())} · con MENOS: {int((sv.pts_sem < sv.pts_otro).sum())}")
if (sv.pts_sem < sv.pts_otro).any():
    print(sv.loc[sv.pts_sem < sv.pts_otro, ['torneo', 'tipo', 'local', 'visita', 'pts_sem', 'pts_otro']].to_string(index=False))

# ---------- 2. Eventos de R1: marcador efectivo y cuarto por evento ----------
COLS = ['match_id', 'index', 'period', 'minute', 't', 'duration', 'type.name', 'possession', 'possession_team.name',
        'team.name', 'loc_x', 'loc_y', 'under_pressure', 'counterpress', 'es_reanudacion', 'es_bp', 'fase', 't_pos',
        'presion_gap_prev', 'pass.length', 'pass.type.name', 'pass.outcome.name', 'pass.pass_success_probability',
        'pass_end_x', 'pass_end_y', 'carry_end_x', 'carry_end_y', 'shot.outcome.name', 'duel.type.name',
        'foul_committed.offensive', 'ball_receipt.outcome.name', 'nace_rec']
TIPOS_CG = set(PRM['tipos_contragolpe'])
TRANS_S = PP['transicion_s']
ruta_tr = os.path.join(FW, 'r4_tramos_v21.parquet')   # v2: reglas de R2a/R2b replicadas
if os.path.exists(ruta_tr) and not REHACER:
    TR = pd.read_parquet(ruta_tr); MIN = pd.read_parquet(os.path.join(FW, 'r4_minutos_v21.parquet'))
    FAL = pd.read_parquet(os.path.join(FW, 'r4_faltas_v21.parquet')); CHK = pd.read_parquet(os.path.join(FW, 'r4_chequeo_v21.parquet'))
    print('\nTramos cargados de caché:', len(TR), 'filas')
else:
    def e5(m):
        m = np.asarray(m, dtype=float)
        return np.clip(np.sign(m) * np.ceil(np.abs(m) - 1e-9), -2, 2).astype(int)
    loc = Mm.set_index('match_id')
    piezas, minutos, faltas, chk, gol_chk = [], [], [], [], []

    def agrega(d, team, es_local, num, den, nombre):
        s = np.where(es_local, 1.0, -1.0)
        g = pd.DataFrame({'match_id': d['match_id'].values, 'team': np.asarray(team),
                          'e5': e5(s * d['m_home'].values), 'e5b': e5(s * d['m_home_b'].values),
                          'cuarto': d['cuarto'].values, 'num': np.asarray(num, dtype=float),
                          'den': np.asarray(den, dtype=float)})
        g = g.groupby(['match_id', 'team', 'e5', 'e5b', 'cuarto'], as_index=False)[['num', 'den']].sum()
        g['metrica'] = nombre
        return g

    for temp in ENT:
        t0 = time.time()
        E = pd.read_parquet(os.path.join(FW, f"eventos_{temp.replace('/', '-')}.parquet"))
        E = E[[c for c in COLS if c in E.columns]]
        for c in COLS:
            if c not in E.columns: E[c] = np.nan
        E = E[E['match_id'].isin(mids)].sort_values(['match_id', 'index']).reset_index(drop=True)
        E['local'] = E['match_id'].map(loc['local']); E['visita'] = E['match_id'].map(loc['visita'])
        E['ev_local'] = E['team.name'] == E['local']
        # Goles (periodos 1 a 4): tiro convertido + autogol a favor
        es_gol = E['period'].between(1, 4) & (((E['type.name'] == 'Shot') & (E['shot.outcome.name'] == 'Goal')) |
                                              (E['type.name'] == 'Own Goal For'))
        gh = (es_gol & E['ev_local']).astype(int); ga = (es_gol & ~E['ev_local'] & E['team.name'].notna()).astype(int)
        gol_chk.append(pd.DataFrame({'match_id': E['match_id'], 'gh': gh, 'ga': ga}).groupby('match_id').sum())
        E['dif_home'] = (gh.groupby(E['match_id']).cumsum() - gh) - (ga.groupby(E['match_id']).cumsum() - ga)
        E = E[E['period'].isin([1, 2])].copy()                                      # D-9
        E['m_home'] = E['dif_home'] + E['match_id'].map(loc['offset'])
        E['m_home_b'] = E['dif_home'] + E['match_id'].map(loc['offset_b'])
        E['cuarto'] = np.where(E['period'] == 1, np.where(E['minute'] < CORTE_C1, 1, 2),
                               np.where(E['minute'] < CORTE_C3, 3, 4))
        E['rival'] = np.where(E['ev_local'], E['visita'], E['local'])
        abierto = (E['fase'] != 'bp') & ~E['es_reanudacion'].fillna(False).astype(bool)
        tn = E['type.name']; x = E['loc_x']

        # Minutos por estado y cuarto (intervalo entre eventos consecutivos, asignado al evento que lo abre)
        dt = (E.groupby(['match_id', 'period'])['t'].shift(-1) - E['t']).fillna(0).clip(lower=0)
        for lado, s in [('local', 1.0), ('visita', -1.0)]:
            minutos.append(pd.DataFrame({'match_id': E['match_id'], 'team': E[lado], 'e5': e5(s * E['m_home']),
                                         'e5b': e5(s * E['m_home_b']), 'cuarto': E['cuarto'], 'minutos': dt / 60})
                           .groupby(['match_id', 'team', 'e5', 'e5b', 'cuarto'], as_index=False)['minutos'].sum())
        # Faltas por equipo y partido (para el árbitro, B3)
        fc = E[(tn == 'Foul Committed') & ~E['foul_committed.offensive'].eq(True)]
        faltas.append(fc.groupby(['match_id', 'team.name']).size().rename('faltas').reset_index())

        # M13 field tilt (F2): pases, conducciones, recepciones y tiros en x ≥ 80, juego abierto
        u3 = E[abierto & tn.isin(['Pass', 'Carry', 'Ball Receipt*', 'Shot']) & (x >= X_U3) & E['team.name'].notna()
               & ~((tn == 'Ball Receipt*') & E['ball_receipt.outcome.name'].notna())]          # R2a: sin recepciones fallidas
        piezas += [agrega(u3, u3['team.name'], u3['ev_local'], 100.0, 1.0, 'field_tilt'),
                   agrega(u3, u3['rival'], ~u3['ev_local'], 0.0, 1.0, 'field_tilt')]
        # Pases de juego abierto, sin reanudaciones
        pa = E[(tn == 'Pass') & abierto]
        piezas.append(agrega(pa, pa['team.name'], pa['ev_local'], 100.0 * (pa['pass.length'] >= LARGO), 1.0, 'pase_largo'))
        pr = pa[pa['pass.pass_success_probability'].notna()]
        piezas.append(agrega(pr, pr['team.name'], pr['ev_local'], 100.0 * (1 - pr['pass.pass_success_probability']), 1.0, 'riesgo_pase'))
        # Saque de meta corto (M3)
        sm = E[(tn == 'Pass') & (E['pass.type.name'] == 'Goal Kick')]
        piezas.append(agrega(sm, sm['team.name'], sm['ev_local'], 100.0 * (sm['pass_end_x'] < X_MED), 1.0, 'saque_meta_corto'))
        # M14 progresión: pase completo o conducción, juego abierto; cruza x = 40 u 80 avanzando ≥ 10 yd, o entra al área
        def progresivo(x0, y0, x1, y1):
            cruza = (((x0 < X_MED) & (x1 >= X_MED)) | ((x0 < X_U3) & (x1 >= X_U3))) & ((x1 - x0) >= AVANCE)
            en_area = lambda xx, yy: (xx >= AREA_X) & (yy >= AREA_Y[0]) & (yy <= AREA_Y[1])
            return cruza | (~en_area(x0, y0) & en_area(x1, y1))
        pc = pa[pa['pass.outcome.name'].isna()]
        prog_p = progresivo(pc['loc_x'], pc['loc_y'], pc['pass_end_x'], pc['pass_end_y'])
        ca = E[(tn == 'Carry') & (E['fase'] != 'bp')]
        prog_c = progresivo(ca['loc_x'], ca['loc_y'], ca['carry_end_x'], ca['carry_end_y'])
        piezas += [agrega(pc, pc['team.name'], pc['ev_local'], prog_p.astype(float), 0.0, 'progresion'),
                   agrega(ca, ca['team.name'], ca['ev_local'], prog_c.astype(float), 0.0, 'progresion')]
        # M12 intensidad de disputa: acciones (entrada, intercepción, falta) del que presiona en su x ≥ 48, en posesión rival
        acc = E[(((tn == 'Duel') & (E['duel.type.name'] == 'Tackle')) | (tn == 'Interception') |
                 ((tn == 'Foul Committed') & ~E['foul_committed.offensive'].eq(True))) &
                (x >= X_PPDA) & (E['possession_team.name'] != E['team.name']) & abierto]    # R2a: juego abierto
        pr_riv = pa[pa['loc_x'] <= 120 - X_PPDA]
        piezas += [agrega(acc, acc['team.name'], acc['ev_local'], 100.0, 0.0, 'intensidad_ppda'),
                   agrega(pr_riv, pr_riv['rival'], ~pr_riv['ev_local'], 0.0, 1.0, 'intensidad_ppda')]
        # M4 presión alta: réplica de R2b. Cadena de presiones del mismo equipo en la misma posesión rival con pausas
        # (inicio a inicio) de hasta 5 s; la fase se descarta si su PRIMERA presión es de balón parado o cae en los
        # primeros 5 s de una posesión que nace de reanudación (S4-29).
        Pz0 = pd.read_parquet(os.path.join(FW, f"posesiones_{temp.replace('/', '-')}.parquet")).sort_values(['match_id', 'possession'])
        for c_, s_ in [('sig_eq', 'equipo'), ('sig_t0', 't0'), ('sig_per', 'period')]:
            Pz0[c_] = Pz0.groupby('match_id')[s_].shift(-1)
        Pz = Pz0[Pz0['match_id'].isin(mids) & Pz0['period'].isin([1, 2])].copy()
        ps = E[(tn == 'Pressure') & (E['possession_team.name'] != E['team.name'])]
        ps = ps.merge(Pz[['match_id', 'possession', 't0', 'primer_reanudacion']], on=['match_id', 'possession'], how='left')
        ps = ps.sort_values(['match_id', 'possession', 'team.name', 'index']).reset_index(drop=True)
        ps['gap'] = ps.groupby(['match_id', 'possession', 'team.name'])['t'].diff()
        ps['sobre'] = ps['primer_reanudacion'].eq(True) & ((ps['t'] - ps['t0']) <= EXCL_REANUD)
        ps['fid'] = (ps['gap'].isna() | (ps['gap'] > PAUSA)).cumsum()
        ini = ps.drop_duplicates('fid', keep='first')
        ini = ini[(ini['fase'] != 'bp') & ~ini['sobre']]
        piezas.append(agrega(ini, ini['team.name'], ini['ev_local'], 100.0 * (ini['loc_x'] >= X_U3), 1.0, 'presion_alta'))
        # Primer evento de cada posesión: estado y cuarto de la posesión
        pe = E.groupby(['match_id', 'possession'], as_index=False).first()[['match_id', 'possession', 'm_home', 'm_home_b', 'cuarto']]
        Pz = Pz.merge(pe, on=['match_id', 'possession'], how='inner')
        Pz['local'] = Pz['match_id'].map(loc['local']); Pz['visita'] = Pz['match_id'].map(loc['visita'])
        Pz['pos_local'] = Pz['equipo'] == Pz['local']
        Pz['rival'] = np.where(Pz['pos_local'], Pz['visita'], Pz['local'])
        # M1 posesión por tiempo (O-12, B)
        piezas += [agrega(Pz, Pz['equipo'], Pz['pos_local'], 100.0 * Pz['dur_B'], Pz['dur_B'], 'posesion'),
                   agrega(Pz, Pz['rival'], ~Pz['pos_local'], 0.0, Pz['dur_B'], 'posesion')]
        # Denominador de la progresión: tiempo de posesión propia en unidades de 10 minutos
        piezas.append(agrega(Pz, Pz['equipo'], Pz['pos_local'], 0.0, Pz['dur_B'] / 600.0, 'progresion'))
        # M7 directness: posesiones propias, sin las que nacen de balón parado
        dz = Pz[Pz['directness'].notna() & ~Pz['inicia_bp'].fillna(False).astype(bool)]
        piezas.append(agrega(dz, dz['equipo'], dz['pos_local'], 100.0 * dz['directness'], 1.0, 'directness'))
        # M6 (réplica de R2b): posesiones rivales que nacen de recuperación. Capa 1 = contrapresión en los primeros 5 s.
        # Capa 2 = primer desenlace dentro de 10 s entre falta nuestra, recuperación nuestra y contragolpe rival; si no hay, asienta.
        tr = Pz[Pz['nace_rec'].fillna(False).astype(bool)].copy()
        k_ = ['match_id', 'possession']
        Ein = E[E['nace_rec'].fillna(False).astype(bool) & E['t_pos'].between(0, TRANS_S)]
        deL = Ein[Ein['team.name'] != Ein['possession_team.name']]
        cpt = deL[deL['counterpress'].eq(True) & (deL['t_pos'] <= CP_S)].groupby(k_)['t_pos'].min().rename('cp_t')
        fot = deL[(deL['type.name'] == 'Foul Committed') & ~deL['foul_committed.offensive'].eq(True)].groupby(k_)['t_pos'].min().rename('falta_t')
        cgt = Ein[(Ein['team.name'] == Ein['possession_team.name']) & (Ein['loc_x'] >= X_U3) &
                  Ein['type.name'].isin(TIPOS_CG)].groupby(k_)['t_pos'].min().rename('cg_t')
        tr = tr.join(cpt, on=k_).join(fot, on=k_).join(cgt, on=k_)
        tr['rec_t'] = np.where((tr['sig_eq'] == tr['rival']) & (tr['sig_per'] == tr['period']), tr['sig_t0'] - tr['t0'], np.nan)
        tiempos = pd.DataFrame({'falta': tr['falta_t'], 'recuperacion': tr['rec_t'], 'contragolpe': tr['cg_t']})
        tiempos = tiempos.where(tiempos <= TRANS_S).fillna(np.inf)
        tr['res'] = tiempos.idxmin(axis=1).where(np.isfinite(tiempos.min(axis=1)), 'asienta')
        con_cp = tr['cp_t'].notna().values
        piezas += [agrega(tr, tr['rival'], ~tr['pos_local'], 100.0 * con_cp, 1.0, 'contrapresion'),
                   agrega(tr, tr['rival'], ~tr['pos_local'], 100.0 * (tr['res'] == 'recuperacion'), 1.0, 'recuperacion_tras_perdida'),
                   agrega(tr, tr['rival'], ~tr['pos_local'], 100.0 * (tr['res'] == 'contragolpe'), 1.0, 'contragolpe_rival')]
        # Conteos sueltos a nivel equipo-partido para el chequeo de coherencia contra R2a y R2b
        def cuenta(df, team, val, nombre):
            return pd.DataFrame({'match_id': df['match_id'].values, 'team': np.asarray(team), nombre: np.asarray(val, float)}
                                ).groupby(['match_id', 'team'])[nombre].sum()
        chk.append(pd.concat([
            cuenta(u3, u3['team.name'], 1, 'ft_acc_u3'), cuenta(pa, pa['team.name'], 1, 'n_pases'),
            cuenta(pa, pa['team.name'], pa['pass.length'] >= LARGO, 'n_largo'),
            cuenta(pr, pr['team.name'], 1, 'n_prob'), cuenta(pr, pr['team.name'], pr['pass.pass_success_probability'], 'sum_prob'),
            cuenta(sm, sm['team.name'], 1, 'n_saque_meta'), cuenta(sm, sm['team.name'], sm['pass_end_x'] < X_MED, 'n_saque_meta_corto'),
            cuenta(pc, pc['team.name'], prog_p, 'n_prog_pase'), cuenta(ca, ca['team.name'], prog_c, 'n_prog_cond'),
            cuenta(acc, acc['team.name'], 1, 'ppda_acciones'), cuenta(pr_riv, pr_riv['rival'], 1, 'ppda_pases_rival'),
            cuenta(Pz, Pz['equipo'], Pz['dur_B'], 't_pos_B'),
            cuenta(dz, dz['equipo'], dz['directness'], 'sum_directness'), cuenta(dz, dz['equipo'], 1, 'n_directness'),
            cuenta(ini, ini['team.name'], 1, 'fases_x5'), cuenta(ini, ini['team.name'], ini['loc_x'] >= X_U3, 'alta_x5'),
            cuenta(tr, tr['rival'], 1, 'td_n'), cuenta(tr, tr['rival'], con_cp, 'td_contrapresion'),
            cuenta(tr, tr['rival'], tr['res'] == 'contragolpe', 'td_res_contragolpe_rival'),
            cuenta(tr, tr['rival'], tr['res'] == 'recuperacion', 'td_res_recuperacion')], axis=1))
        print(f'  {temp}: {E.match_id.nunique()} partidos, {len(E):,} eventos ({time.time() - t0:.0f} s)')

    TR = pd.concat(piezas, ignore_index=True).groupby(['metrica', 'match_id', 'team', 'e5', 'e5b', 'cuarto'], as_index=False)[['num', 'den']].sum()
    MIN = pd.concat(minutos, ignore_index=True).groupby(['match_id', 'team', 'e5', 'e5b', 'cuarto'], as_index=False)['minutos'].sum()
    FAL = pd.concat(faltas, ignore_index=True).rename(columns={'team.name': 'team'})
    CHK = pd.concat(chk).fillna(0).reset_index()
    G = pd.concat(gol_chk)
    TR.to_parquet(ruta_tr, index=False); MIN.to_parquet(os.path.join(FW, 'r4_minutos_v21.parquet'), index=False)
    FAL.to_parquet(os.path.join(FW, 'r4_faltas_v21.parquet'), index=False); CHK.to_parquet(os.path.join(FW, 'r4_chequeo_v21.parquet'), index=False)
    gg = Mm.set_index('match_id')[['gl', 'gv']].join(G)
    print(f"\nGoles reconstruidos (periodos 1-4) = marcador oficial: {int(((gg.gl == gg.gh) & (gg.gv == gg.ga)).sum())} de {len(gg)}")
Mm.to_parquet(os.path.join(FW, 'r4_partidos.parquet'), index=False)

print('\n--- (b) Tramos por métrica (filas equipo-partido-estado-cuarto con denominador > 0) ---')
print(TR[TR.den > 0].groupby('metrica').agg(filas=('den', 'size'), den_total=('den', 'sum')).round(0).to_string())

print('\n--- (c) Coherencia contra R2a / R2b (sumas por equipo-partido; 1.000 = idéntico) ---')
comp = pd.concat([pd.read_parquet(os.path.join(FW, f"componentes_{t.replace('/', '-')}.parquet")) for t in ENT])
pres = pd.concat([pd.read_parquet(os.path.join(FW, f"presion_{t.replace('/', '-')}.parquet")) for t in ENT])
ref = (comp.groupby(['match_id', 'equipo']).sum(numeric_only=True)
       .join(pres.groupby(['match_id', 'equipo']).sum(numeric_only=True), rsuffix='_p'))
ref.index.names = ['match_id', 'team']
mio = CHK.set_index(['match_id', 'team'])
filas = []
for c in mio.columns:
    if c not in ref.columns: continue
    j = mio[[c]].join(ref[[c]], rsuffix='_r2', how='inner').dropna()
    a, b = j[c], j[c + '_r2']
    filas.append(dict(componente=c, n=len(j), r4=a.sum(), r2=b.sum(), cociente=a.sum() / b.sum() if b.sum() else np.nan,
                      iguales_pct=100 * np.mean(np.isclose(a, b, rtol=0.005, atol=0.01)), corr=a.corr(b)))
print(pd.DataFrame(filas).round(3).to_string(index=False))

print('\n--- (d) Minutos por estado (marcador simple) contra equipo_partido_marcador, fase regular ---')
epm = pd.read_parquet(os.path.join(DP, 'equipo_partido_marcador.parquet')).set_index(['match_id', 'team'])
mreg = MIN[MIN.match_id.isin(Mm.loc[Mm.tipo == 'regular', 'match_id'])].groupby(['match_id', 'team', 'e5'])['minutos'].sum().unstack(fill_value=0)
mapa = {-2: 'min_pierde2+', -1: 'min_pierde1', 0: 'min_empate', 1: 'min_gana1', 2: 'min_gana2+'}
j = mreg.join(epm[list(mapa.values())], how='inner')
print('Partidos comparados:', len(j), '| diferencia media absoluta (min):',
      {k: round(float((j[k] - j[v]).abs().mean()), 2) for k, v in mapa.items() if k in j})

W = MIN.assign(e3=np.sign(MIN.e5)).groupby('e3')['minutos'].sum()
print('\n--- (e) Mezcla de estados de la liga (B1, minutos, regla principal) ---')
print((100 * W / W.sum()).round(1).rename({-1: 'abajo', 0: 'empate', 1: 'arriba'}).to_dict())
print(f'\nR4a terminó en {time.time() - t_ini:.0f} s. Guardado: fw1/r4_tramos_v21, r4_minutos_v21, r4_faltas_v21, r4_chequeo_v21, r4_partidos.')


R4b

In [ ]:
# [R4b] Receta por métrica, residuos, perfiles por etapa y sensibilidades
# Requiere R4a en memoria (o su caché). Tarda ~1-2 min.
t_ini = time.time()
Mm = pd.read_parquet(os.path.join(FW, 'r4_partidos.parquet'))
TR = pd.read_parquet(os.path.join(FW, 'r4_tramos_v21.parquet')); MIN = pd.read_parquet(os.path.join(FW, 'r4_minutos_v21.parquet'))
FAL = pd.read_parquet(os.path.join(FW, 'r4_faltas_v21.parquet'))
assert set(Mm['temporada']) <= set(ENT), 'R4 solo usa entrenamiento (B5)'

METRICAS = ['posesion', 'field_tilt', 'pase_largo', 'saque_meta_corto', 'riesgo_pase', 'progresion',
            'intensidad_ppda', 'presion_alta', 'contrapresion', 'recuperacion_tras_perdida', 'contragolpe_rival', 'directness']
CON_ARBITRO = {'intensidad_ppda'}                                     # B3: solo métricas que cuentan faltas
UNIDAD = {'progresion': 'por 10 min', 'intensidad_ppda': 'por 100 pases'}

# ---------- Contexto por equipo-partido ----------
lado = pd.concat([Mm.assign(team=Mm.local, rival=Mm.visita, local_=1), Mm.assign(team=Mm.visita, rival=Mm.local, local_=0)])
CTX = lado[['match_id', 'team', 'rival', 'local_', 'torneo', 'tipo', 'arbitro', 'temporada']].copy()
elo = pd.read_parquet(os.path.join(FW, 'elo_equipo_partido_e2.parquet'))[['match_id', 'team', 'elo_pre', 'elo_rival_pre']]
CTX = CTX.merge(elo, on=['match_id', 'team'], how='left')
ex = pd.read_parquet(os.path.join(FW, 'elo_xg_partido.parquet'))
exl = pd.concat([ex.assign(team=ex.local, eloxg_rival=ex.eloxg_visita_pre), ex.assign(team=ex.visita, eloxg_rival=ex.eloxg_local_pre)])
CTX = CTX.merge(exl[['match_id', 'team', 'eloxg_rival']], on=['match_id', 'team'], how='left')
print('Contexto: equipo-partidos', len(CTX), '| sin Elo rival:', int(CTX.elo_rival_pre.isna().sum()),
      '| sin Elo de xG rival:', int(CTX.eloxg_rival.isna().sum()))
CTX['elo_c'] = (CTX['elo_rival_pre'] - 1500) / 100
CTX['eloxg_c'] = (CTX['eloxg_rival'] - 1500) / 100
CTX['delo_c'] = (CTX['elo_rival_pre'] - CTX['elo_pre']) / 100      # sensibilidad 12 (ΔElo)

# Árbitro (B3): faltas por partido, sin contar este partido, con contracción según cuántos partidos dirigió
fm = FAL.groupby('match_id')['faltas'].sum().rename('f').to_frame().join(Mm.set_index('match_id')[['arbitro']])
fm = fm[fm['arbitro'].notna()]
s_r = fm.groupby('arbitro')['f'].agg(['sum', 'count'])
fm['loo'] = [(s_r.loc[a, 'sum'] - f) / (s_r.loc[a, 'count'] - 1) if s_r.loc[a, 'count'] > 1 else np.nan
             for a, f in zip(fm.arbitro, fm.f)]
mu, s2 = fm['f'].mean(), fm['f'].var()
med_ref = s_r['sum'] / s_r['count']
tau2_ref = max(0.0, med_ref.var() - (s2 / s_r['count']).mean())
n_loo = fm['arbitro'].map(s_r['count']) - 1
k_ref = tau2_ref / (tau2_ref + s2 / n_loo.clip(lower=1))
fm['arb_a'] = np.where(fm['loo'].notna(), k_ref * (fm['loo'] - mu), 0.0)
# Sensibilidad b: faltas del partido frente a las esperadas por los equipos (promedio de faltas de cada equipo)
FT = FAL.merge(Mm[['match_id', 'arbitro']], on='match_id')
prom_eq = FT.groupby('team')['faltas'].mean()
FT['esp'] = FT['team'].map(prom_eq)
dm = FT.groupby('match_id').agg(f=('faltas', 'sum'), esp=('esp', 'sum')).join(Mm.set_index('match_id')[['arbitro']])
dm['exc'] = dm['f'] - dm['esp']
s_b = dm.groupby('arbitro')['exc'].agg(['sum', 'count'])
dm['loo_b'] = [(s_b.loc[a, 'sum'] - e) / (s_b.loc[a, 'count'] - 1) if (pd.notna(a) and s_b.loc[a, 'count'] > 1) else np.nan
               for a, e in zip(dm.arbitro, dm.exc)]
tau2_b = max(0.0, (s_b['sum'] / s_b['count']).var() - (dm['exc'].var() / s_b['count']).mean())
nb = dm['arbitro'].map(s_b['count']).fillna(2) - 1
dm['arb_b'] = np.where(dm['loo_b'].notna(), tau2_b / (tau2_b + dm['exc'].var() / nb.clip(lower=1)) * dm['loo_b'], 0.0)
CTX['arb_a'] = CTX['match_id'].map(fm['arb_a']).fillna(0.0)
CTX['arb_b'] = CTX['match_id'].map(dm['arb_b']).fillna(0.0)
print(f"\n--- (a) Árbitro: {s_r.shape[0]} árbitros · faltas por partido {mu:.1f} · dispersión real entre árbitros "
      f"(desv. est.) {np.sqrt(tau2_ref):.2f} faltas · versión b (frente a lo esperado por los equipos) {np.sqrt(tau2_b):.2f}")
par = CTX[CTX['arbitro'].notna()].groupby(['arbitro', 'team']).size()
print(f"Repetición árbitro-equipo en 4 temporadas: mediana {par.median():.0f}, p90 {par.quantile(.9):.0f}, máx {par.max()} partidos")

# ---------- Estilo del rival: lo que concede en sus OTROS partidos del torneo (dejar uno fuera) ----------
TM = TR.groupby(['metrica', 'match_id', 'team'], as_index=False)[['num', 'den']].sum()
TM = TM.merge(CTX[['match_id', 'team', 'rival', 'torneo']], on=['match_id', 'team'])
# Lo que el rival R concede en el partido m = lo que hizo su oponente (team) en m
conc = TM.groupby(['metrica', 'torneo', 'rival'])[['num', 'den']].sum().rename(columns={'num': 'cn', 'den': 'cd'})
liga_t = TM.groupby(['metrica', 'torneo'])[['num', 'den']].sum()
TM = TM.join(conc, on=['metrica', 'torneo', 'rival']).join(liga_t.rename(columns={'num': 'ln', 'den': 'ld'}), on=['metrica', 'torneo'])
TM['estilo'] = (TM['cn'] - TM['num']) / (TM['cd'] - TM['den']) - TM['ln'] / TM['ld']
TM['estilo'] = TM['estilo'].replace([np.inf, -np.inf], np.nan).fillna(0.0)

# ---------- Etapas de técnico ----------
et = ptp_e[['match_id', 'team', 'tec_p', 'etapa']].copy()
et['etapa_id'] = et['team'] + ' · ' + et['etapa'].astype(str)
n_et = et.groupby('etapa_id')['match_id'].nunique()
ETAPAS_15 = set(n_et[n_et >= PP['etapa_min']].index)
W3 = MIN.assign(e3=np.sign(MIN.e5)).groupby('e3')['minutos'].sum(); W3 = W3 / W3.sum()

def ajustar(met, col_e='e5', elo='elo_c', sin_liguilla=False, arb='arb_a'):
    d = TR[(TR.metrica == met)].groupby(['match_id', 'team', col_e, 'cuarto'], as_index=False)[['num', 'den']].sum()
    d = d[d['den'] > 0].rename(columns={col_e: 'e'})
    d = d.merge(CTX, on=['match_id', 'team']).merge(TM[TM.metrica == met][['match_id', 'team', 'estilo']], on=['match_id', 'team'])
    if sin_liguilla: d = d[d['tipo'].isin(['regular', 'playin'])]
    d = d.dropna(subset=[elo]).reset_index(drop=True)
    y = (d['num'] / d['den']).values; w = d['den'].values
    cols = {'intercepto': np.ones(len(d))}
    for k in [-2, -1, 1, 2]: cols[f'e{k:+d}'] = (d['e'] == k).astype(float).values
    for c in [2, 3, 4]: cols[f'C{c}'] = (d['cuarto'] == c).astype(float).values
    cols['local'] = d['local_'].astype(float).values
    cols['elo_rival_100'] = d[elo].values
    cols['estilo_rival'] = d['estilo'].values
    if met in CON_ARBITRO: cols['arbitro'] = d[arb].values
    for t in sorted(d['torneo'].unique())[1:]: cols[f'T_{t}'] = (d['torneo'] == t).astype(float).values
    X = np.column_stack(list(cols.values())); sw = np.sqrt(w)
    beta = np.linalg.lstsq(X * sw[:, None], y * sw, rcond=None)[0]
    e = y - X @ beta
    s2 = (w * e ** 2).sum() / (len(y) - X.shape[1])
    # Celdas estado × cuarto: lo que agregan sobre los efectos separados, con contracción (A2)
    c = pd.DataFrame({'e': d['e'], 'q': d['cuarto'], 'we': w * e, 'w': w}).groupby(['e', 'q'])[['we', 'w']].sum()
    c['r'] = c['we'] / c['w']; c['se2'] = s2 / c['w']
    # DerSimonian-Laird: dispersión real entre celdas, descontando el ruido de cada una (ponderado por su precisión)
    wi = 1 / c['se2']; gl_ = max(1, len(c) - 8)
    tau2 = max(0.0, ((wi * c['r'] ** 2).sum() - gl_) / (wi.sum() - (wi ** 2).sum() / wi.sum()))
    c['k'] = tau2 / (tau2 + c['se2']) if tau2 > 0 else 0.0
    c['celda'] = c['k'] * c['r']
    adj = pd.Series(list(zip(d['e'], d['cuarto']))).map(c['celda']).values
    yhat = X @ beta + adj; e = y - yhat
    r2 = 1 - (w * e ** 2).sum() / (w * (y - np.average(y, weights=w)) ** 2).sum()
    d['y'], d['w'], d['res'], d['yhat'] = y, w, e, yhat
    coef = dict(zip(cols.keys(), beta))
    return d, coef, dict(r2=r2, s2=s2, tau_celdas=np.sqrt(tau2), media=np.average(y, weights=w)), c

def perfiles(d, s2):
    d = d.merge(et[['match_id', 'team', 'etapa_id', 'tec_p']], on=['match_id', 'team'])
    d['e3'] = np.sign(d['e']).astype(int)
    a = d.assign(we=d.w * d.res).groupby(['etapa_id', 'e3'])[['we', 'w']].sum()
    a['r'] = a['we'] / a['w']
    tot = d.assign(we=d.w * d.res).groupby('etapa_id')[['we', 'w']].sum()
    a['rbar'] = (tot['we'] / tot['w']).reindex(a.index.get_level_values(0)).values
    a['se2'] = s2 / a['w']
    dd = a['r'] - a['rbar']
    wi = 1 / a['se2']; gl_ = max(1, len(a) - a.index.get_level_values(0).nunique())
    tau2 = max(0.0, ((wi * dd ** 2).sum() - gl_) / (wi.sum() - (wi ** 2).sum() / wi.sum()))   # DerSimonian-Laird
    a['k'] = tau2 / (tau2 + a['se2']) if tau2 > 0 else 0.0
    a['r_star'] = a['rbar'] + a['k'] * dd                                       # B2: hacia el propio técnico
    piv = a['r_star'].unstack()
    rbar = tot['we'] / tot['w']
    for s in [-1, 0, 1]:
        if s not in piv: piv[s] = np.nan
        piv[s] = piv[s].fillna(rbar)
    perfil = sum(W3.get(s, 0) * piv[s] for s in [-1, 0, 1])                    # B1: mezcla de la liga
    sg = d.groupby('etapa_id')[['num', 'den']].sum(); crudo = sg['num'] / sg['den']
    out = pd.DataFrame({'perfil': perfil, 'crudo': crudo, 'abajo': piv[-1], 'empate': piv[0], 'arriba': piv[1],
                        'tau_estado': np.sqrt(tau2)})
    out['tecnico'] = d.groupby('etapa_id')['tec_p'].first(); out['partidos'] = n_et.reindex(out.index)
    return out

# ---------- Corrida principal ----------
res, coefs, perf, celdas = {}, [], {}, {}
for met in METRICAS:
    d, coef, info, c = ajustar(met)
    res[met], celdas[met] = d, c
    perf[met] = perfiles(d, info['s2'])
    coefs.append({'metrica': met, 'filas': len(d), 'media_liga': info['media'], 'R2': info['r2'], **coef,
                  'tau_celdas': info['tau_celdas'], 'tau_estado_etapa': perf[met]['tau_estado'].iloc[0]})
CO = pd.DataFrame(coefs)
CO.to_csv(os.path.join(SAL, 'r4_coeficientes.csv'), index=False)

print('\n--- (b) Receta por métrica: efectos genéricos (misma unidad que la métrica) ---')
ver = ['metrica', 'filas', 'media_liga', 'R2', 'e-2', 'e-1', 'e+1', 'e+2', 'C2', 'C3', 'C4', 'local', 'elo_rival_100',
       'estilo_rival', 'arbitro', 'tau_celdas', 'tau_estado_etapa']
print(CO.reindex(columns=ver).round(3).to_string(index=False))
print('  e±k = diferencia frente al empate · C2-C4 frente al arranque del 1T · elo_rival_100 = por cada 100 puntos del rival ·'
      ' estilo_rival = por cada unidad que concede el rival sobre la media · tau_* = dispersión real tras la contracción')

print('\n--- (c) Field tilt y posesión esperados por estado y cuarto (efecto total frente a empate en C1) ---')
for met in ['field_tilt', 'posesion']:
    cf = CO.set_index('metrica').loc[met]
    tab = pd.DataFrame({q: {k: (cf.get(f'e{k:+d}', 0) if k else 0) + (cf.get(f'C{q}', 0) if q > 1 else 0) +
                            celdas[met]['celda'].get((k, q), 0) for k in [-2, -1, 0, 1, 2]} for q in [1, 2, 3, 4]})
    tab.columns = ['C1 0-25', 'C2 25-45+', 'C3 45-70', 'C4 70-90+']; tab.index = ['-2', '-1', '0', '+1', '+2']
    print(f'{met}:'); print(tab.round(1).to_string())

# ---------- Sensibilidades ----------
SENS = {'Elo de xG del rival': dict(elo='eloxg_c'), 'ΔElo (prueba 12)': dict(elo='delo_c'),
        'sin liguilla (prueba 14)': dict(sin_liguilla=True), 'ida como partido normal': dict(col_e='e5b')}
filas = []
for met in METRICAS:
    base = perf[met].loc[perf[met].index.isin(ETAPAS_15), 'perfil']
    fila = {'metrica': met, 'crudo_vs_ajustado': base.corr(perf[met].loc[base.index, 'crudo'], method='spearman')}
    for nom, kw in SENS.items():
        d2, _, i2, _ = ajustar(met, **kw)
        p2 = perfiles(d2, i2['s2'])['perfil']
        fila[nom] = base.corr(p2.reindex(base.index), method='spearman')
    if met in CON_ARBITRO:
        d2, _, i2, _ = ajustar(met, arb='arb_b')
        fila['árbitro b'] = base.corr(perfiles(d2, i2['s2'])['perfil'].reindex(base.index), method='spearman')
    filas.append(fila)
print(f'\n--- (d) Orden de las {len(ETAPAS_15)} etapas de 15+ partidos: correlación de rangos con la corrida principal ---')
print('  crudo_vs_ajustado = cuánto reordena el ajuste; el resto = qué tanto cambia el orden con cada sensibilidad')
print(pd.DataFrame(filas).round(2).to_string(index=False))

# ---------- América ----------
am = sorted({e for e in perf['posesion'].index if e.startswith('América')},
            key=lambda e: et.loc[et.etapa_id == e, 'match_id'].min())
print('\n--- (e) América por etapa: crudo, perfil ajustado (desviación frente a la liga) y valor ajustado ---')
filas = []
for met in METRICAS:
    p = perf[met]; liga = CO.set_index('metrica').loc[met, 'media_liga']
    for e in am:
        filas.append(dict(metrica=met, etapa=e, tecnico=p.loc[e, 'tecnico'].split()[0] + ' ' + p.loc[e, 'tecnico'].split()[-1]
                          if isinstance(p.loc[e, 'tecnico'], str) else '', partidos=p.loc[e, 'partidos'],
                          liga=liga, crudo=p.loc[e, 'crudo'], perfil=p.loc[e, 'perfil'], ajustado=liga + p.loc[e, 'perfil']))
A = pd.DataFrame(filas)
print(A.pivot_table(index='metrica', columns='tecnico', values=['crudo', 'ajustado'], sort=False).round(1).to_string())
print('\nPor estado (residuo contraído hacia el propio técnico, B2), métricas seleccionadas:')
for met in ['field_tilt', 'posesion', 'saque_meta_corto', 'intensidad_ppda', 'presion_alta', 'pase_largo']:
    t = perf[met].loc[am, ['tecnico', 'abajo', 'empate', 'arriba', 'perfil']].copy()
    t['tecnico'] = t['tecnico'].map(lambda s: s.split()[0] + ' ' + s.split()[-1] if isinstance(s, str) else s)
    print(f'{met}:'); print(t.round(1).to_string())

# ---------- Guardado ----------
pd.concat([d.assign(metrica=m) for m, d in res.items()])[
    ['metrica', 'match_id', 'team', 'e', 'cuarto', 'num', 'den', 'y', 'yhat', 'res']].to_parquet(os.path.join(FW, 'r4_residuos.parquet'), index=False)
pd.concat([p.assign(metrica=m) for m, p in perf.items()]).reset_index().rename(columns={'index': 'etapa_id'}).to_parquet(
    os.path.join(FW, 'r4_perfiles_etapa.parquet'), index=False)
print(f'\nR4b terminó en {time.time() - t_ini:.0f} s. Guardado: fw1/r4_residuos, fw1/r4_perfiles_etapa, salidas/r4_coeficientes.csv')


# Sesión 7

Confiabilidad y relojes

In [ ]:
# =====================================================================
# R5 · Confiabilidad, reloj y contracción de los perfiles ajustados
# Framework v1.1: N-1, N-2, P-5, P-6 · Decisiones de la sesión 7: R5-1 a R5-8
# Sesión 7 · 2026-10-13 · Hackathon ISAC 2026, Track 2
# Antes: celda 1 (Drive) y R0. Requiere que R4b haya corrido al menos una vez (lee su caché).
# Datos: SOLO entrenamiento (2021/22 a 2024/25). Tarda ~3-6 min.
# =====================================================================

# [R5] ---------------------------------------------------------------
import os, json, time
import numpy as np
import pandas as pd
from scipy.stats import rankdata

SEMILLA = 2026
rng = np.random.default_rng(SEMILLA)
BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026/')
DP = os.path.join(BASE, 'datos_parquet'); FW = os.path.join(DP, 'fw1'); SAL = os.path.join(BASE, 'salidas')
t_ini = time.time()

with open(os.path.join(FW, 'parametros_fw1.json')) as f:
    PRM = json.load(f)
ENT = PRM['temporadas']['entrenamiento']
ETAPA_MIN = PRM['P']['etapa_min']                 # P-5: 15 partidos
REPS, BOOT, R_SUB = 200, 500, 50                  # P-6: 200 repartos; bootstrap del reloj (500 remuestreos de etapas)
NUCLEO, SECUND = 0.7, 0.5                         # P-6
OBJ = NUCLEO / (1 - NUCLEO)                       # 2.33: factor del reloj de medición
KS, PLACEBOS = [3, 5, 8], 200                     # R5-6: primeros k partidos frente a k al azar
P_REG = 19                                        # parámetros aprox. de la receta de R4 (solo afecta s2 en ~0.01%)
CORTA = {'posesion': 'pos', 'field_tilt': 'ft', 'pase_largo': 'pl', 'saque_meta_corto': 'smc', 'riesgo_pase': 'rie',
         'progresion': 'prog', 'intensidad_ppda': 'int', 'presion_alta': 'pa', 'contrapresion': 'cp',
         'recuperacion_tras_perdida': 'rtp', 'contragolpe_rival': 'cgr', 'directness': 'dir'}

# ---------- Insumos ----------
RS = pd.read_parquet(os.path.join(FW, 'r4_residuos.parquet'))
PE = pd.read_parquet(os.path.join(FW, 'r4_perfiles_etapa.parquet'))
CO = pd.read_csv(os.path.join(SAL, 'r4_coeficientes.csv')).set_index('metrica')
MIN = pd.read_parquet(os.path.join(FW, 'r4_minutos_v21.parquet'))
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
ptp = ptp[ptp['jugado'].astype(bool)].copy()
ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp_e = ptp[ptp['temporada'].isin(ENT)].copy()
et = ptp_e[['match_id', 'team', 'tec_p', 'etapa', 'fecha']].copy()       # misma tabla de etapas que R4b
et['etapa_id'] = et['team'] + ' · ' + et['etapa'].astype(str)
assert set(RS['match_id']) <= set(ptp_e['match_id']), 'R5 solo usa entrenamiento'

METRICAS = list(CO.index)
ETAPAS = sorted(et['etapa_id'].unique()); NS = len(ETAPAS); COD = {e: i for i, e in enumerate(ETAPAS)}
n_et = et.groupby('etapa_id')['match_id'].nunique().reindex(ETAPAS).values
tec = et.groupby('etapa_id')['tec_p'].first().reindex(ETAPAS)
corto = lambda s: (s.split()[0] + ' ' + s.split()[-1]) if isinstance(s, str) else ''
W3 = MIN.assign(e3=np.sign(MIN.e5)).groupby('e3')['minutos'].sum()
W3 = (W3 / W3.sum()).reindex([-1, 0, 1]).values                          # B1: mezcla de estados de la liga

# Etapas cuyo inicio vemos: no son la primera etapa de su equipo dentro de los datos
ini = et.groupby(['team', 'etapa_id'])['fecha'].min().reset_index().sort_values(['team', 'fecha'])
ini['orden'] = ini.groupby('team').cumcount()
OBS = np.isin(ETAPAS, ini.loc[ini['orden'] > 0, 'etapa_id'])
AME = np.array([e.startswith('América') for e in ETAPAS])

print('--- (a) Insumos ---')
print(f"Residuos de R4: {len(RS):,} filas · {RS.metrica.nunique()} métricas · temporadas {sorted(ptp_e.temporada.unique())}")
print(f"Etapas: {NS} · con {ETAPA_MIN}+ partidos: {(n_et >= ETAPA_MIN).sum()} · "
      f"con {ETAPA_MIN}+ e inicio observado: {((n_et >= ETAPA_MIN) & OBS).sum()}")
print('Filas equipo-partido repetidas en la tabla de etapas (debe ser 0):', int(et.duplicated(['match_id', 'team']).sum()))
print('Mezcla de estados de la liga, % (abajo, empate, arriba):', np.round(100 * W3, 1))
print('América:', [(e, int(n_et[COD[e]]), 'inicio visto' if OBS[COD[e]] else 'inicio no visto') for e in np.array(ETAPAS)[AME]])

# ---------- Herramientas ----------
COLS = ['we_m', 'we_0', 'we_p', 'w_m', 'w_0', 'w_p', 'num', 'den']

def tabla_partidos(met):
    """Una fila por etapa-partido con lo necesario para rehacer el perfil de R4 con cualquier subconjunto de partidos."""
    d = RS[RS.metrica == met]
    s2 = float((d['den'] * d['res'] ** 2).sum() / (len(d) - P_REG))   # varianza residual de la receta (ponderada)
    d = d.merge(et[['match_id', 'team', 'etapa_id', 'fecha']], on=['match_id', 'team'])
    d = d.assign(e3=np.sign(d['e']).astype(int), we=d['den'] * d['res'])
    k = ['etapa_id', 'match_id', 'team', 'fecha']
    g = d.groupby(k + ['e3'])[['we', 'den']].sum().unstack('e3', fill_value=0)
    for c in ['we', 'den']:
        for s in [-1, 0, 1]:
            if (c, s) not in g.columns: g[(c, s)] = 0.0
    tot = d.groupby(k)[['num', 'den']].sum()
    M = pd.DataFrame({'we_m': g[('we', -1)], 'we_0': g[('we', 0)], 'we_p': g[('we', 1)],
                      'w_m': g[('den', -1)], 'w_0': g[('den', 0)], 'w_p': g[('den', 1)],
                      'num': tot['num'], 'den': tot['den']}).reset_index()
    M['cod'] = M['etapa_id'].map(COD)
    return M.sort_values(['cod', 'fecha']).reset_index(drop=True), s2

def perfil_sub(A, cod, mask, tau2, s2):
    """Mismo perfil que R4b (residuo por estado, contracción B2 hacia el propio técnico, mezcla B1), con los partidos de mask."""
    c = cod[mask]
    S = {k: np.bincount(c, weights=A[k][mask], minlength=NS) for k in COLS}
    we = np.column_stack([S['we_m'], S['we_0'], S['we_p']]); w = np.column_stack([S['w_m'], S['w_0'], S['w_p']])
    with np.errstate(divide='ignore', invalid='ignore'):
        rbar = we.sum(1) / w.sum(1)
        k = tau2 / (tau2 + s2 / w) if tau2 > 0 else np.zeros_like(w)
        rstar = np.where(w > 0, rbar[:, None] + k * (we / w - rbar[:, None]), rbar[:, None])
        return rstar @ W3, S['num'] / S['den']

def corr(a, b, rango=False):
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 5: return np.nan
    a, b = a[ok], b[ok]
    if rango: a, b = rankdata(a), rankdata(b)
    return np.corrcoef(a, b)[0, 1]

sb = lambda r: 2 * r / (1 + r)                                            # Spearman-Brown: de media etapa a etapa completa

def pos_en_etapa(orden, cod, start):
    """Posición de cada fila dentro de su etapa según 'orden' (que ya viene ordenado por etapa)."""
    p = np.empty(len(cod), int); p[orden] = np.arange(len(cod)) - start[cod[orden]]
    return p

def senal_ruido(a, b, inv_n):
    """τ² = covarianza entre mitades (lo común es la señal); σ² = ruido de un partido, con E[(A−B)²] = σ²(1/nA + 1/nB)."""
    ac = a - np.nanmean(a, 1, keepdims=True); bc = b - np.nanmean(b, 1, keepdims=True)
    n_ok = np.isfinite(ac * bc).sum(1)
    tau2 = float(np.nanmean(np.nansum(ac * bc, 1) / (n_ok - 1)))
    sig2 = float(np.nanmean((a - b) ** 2 / inv_n))
    return tau2, sig2

# ---------- Corrida por métrica ----------
filas, contr, implant, ame_imp, perf_full = [], [], [], [], {}
for met in METRICAS:
    t0 = time.time()
    M, s2 = tabla_partidos(met)
    tau2_e = float(PE.loc[PE.metrica == met, 'tau_estado'].iloc[0]) ** 2   # B2 congelado de R4b
    A = {k: M[k].values.astype(float) for k in COLS}; cod = M['cod'].values; N = len(cod)
    cnt = np.bincount(cod, minlength=NS); start = np.concatenate([[0], np.cumsum(cnt)[:-1]])
    pos = np.arange(N) - start[cod]                                       # orden cronológico dentro de la etapa
    P = lambda m: perfil_sub(A, cod, m, tau2_e, s2)
    inc = (n_et >= ETAPA_MIN) & (cnt >= 4)

    # (1) Perfil completo y chequeo de reproducción contra R4b
    pf, cf = P(np.ones(N, bool)); perf_full[met] = pf
    pe = PE[PE.metrica == met].set_index('etapa_id').reindex(ETAPAS)
    dif = np.abs(pf - pe['perfil'].values)[inc]; difc = np.abs(cf - pe['crudo'].values)[inc]

    # (2) 200 repartos al azar en mitades (alternando dentro de cada etapa: tamaños iguales o ±1)
    AA = np.full((REPS, NS), np.nan); BB = AA.copy(); r_cru = []
    for r_ in range(REPS):
        mA = pos_en_etapa(np.lexsort((rng.random(N), cod)), cod, start) % 2 == 0
        AA[r_], ca = P(mA); BB[r_], cb = P(~mA)
        r_cru.append(corr(ca[inc], cb[inc]))
    r_p = np.array([corr(AA[i, inc], BB[i, inc]) for i in range(REPS)])
    r_s = np.array([corr(AA[i, inc], BB[i, inc], rango=True) for i in range(REPS)])
    sens = {}
    for thr in [10, 20]:
        m_ = (n_et >= thr) & (cnt >= 4)
        sens[thr] = np.median(sb(np.array([corr(AA[i, m_], BB[i, m_]) for i in range(REPS)])))

    # (3) Señal y ruido, reloj de medición y su intervalo (remuestreo de etapas)
    nA, nB = (cnt + 1) // 2, cnt // 2
    inv_n = np.where(nB > 0, 1 / np.maximum(nA, 1) + 1 / np.maximum(nB, 1), np.nan)
    tau2, sig2 = senal_ruido(AA[:, inc], BB[:, inc], inv_n[inc])
    n_med = float(np.median(cnt[inc]))
    nst = []
    a5, b5, iv = AA[:R_SUB][:, inc], BB[:R_SUB][:, inc], inv_n[inc]; S_ = inc.sum()
    for _ in range(BOOT):
        ix = rng.integers(0, S_, S_)
        t2, sg = senal_ruido(a5[:, ix], b5[:, ix], iv[ix])
        nst.append(OBJ * sg / t2 if t2 > 0 else np.inf)
    nst = np.array(nst)
    conf = np.median(sb(r_p))
    filas.append(dict(metrica=met, etapas=int(S_), r_mitades=np.median(r_p), conf=conf,
                      conf_p5=np.percentile(sb(r_p), 5), conf_p95=np.percentile(sb(r_p), 95),
                      conf_crudo=np.median(sb(np.array(r_cru))), conf_spearman=np.median(sb(r_s)),
                      conf_modelo=tau2 / (tau2 + sig2 / n_med) if tau2 > 0 else 0.0,
                      conf_min10=sens[10], conf_min20=sens[20],
                      franja='núcleo' if conf >= NUCLEO else ('secundario' if conf >= SECUND else 'fuera'),
                      tau=np.sqrt(max(tau2, 0)), sigma_partido=np.sqrt(sig2), n_med=n_med,
                      reloj=OBJ * sig2 / tau2 if tau2 > 0 else np.inf,
                      reloj_p5=np.quantile(nst, .05, method='nearest'), reloj_p95=np.quantile(nst, .95, method='nearest'),
                      repro_corr=corr(pf[inc], pe['perfil'].values[inc]), repro_maxdif=np.nanmax(dif), crudo_maxdif=np.nanmax(difc)))

    # (4) Contracción P-5 por etapa (R5-7): k = τ² / (τ² + σ²/n)
    with np.errstate(divide='ignore', invalid='ignore'):
        k_et = np.where(cnt > 0, tau2 / (tau2 + sig2 / cnt), np.nan) if tau2 > 0 else np.zeros(NS)
    contr.append(pd.DataFrame({'metrica': met, 'etapa_id': ETAPAS, 'tecnico': tec.values, 'partidos': n_et, 'partidos_con_dato': cnt,
                               'perfil_r4': pf, 'k_p5': k_et, 'perfil_mostrado': k_et * pf,
                               'liga': CO.loc[met, 'media_liga'], 'crudo': cf}))

    # (5) Reloj de implantación (R5-6 B): primeros k partidos frente a k al azar de la misma etapa
    for k in KS:
        el = inc & OBS & (cnt >= k + 10); el_row = el[cod]
        f = (pos < k) & el_row
        p1, _ = P(f); p2, _ = P(el_row & ~f)
        d1 = np.abs(p1 - p2)[el]
        m_az, d_az = np.empty(PLACEBOS), np.zeros(el.sum())
        for p in range(PLACEBOS):
            clave = np.where(pos >= k, rng.random(N), 2.0)          # solo partidos fuera del arranque pueden ser placebo
            ch = (pos_en_etapa(np.lexsort((clave, cod)), cod, start) < k) & el_row
            q1, _ = P(ch); q2, _ = P(el_row & ~ch)
            dd = np.abs(q1 - q2)[el]; m_az[p] = np.nanmean(dd); d_az += dd
        d_az /= PLACEBOS
        implant.append(dict(metrica=met, k=k, etapas=int(el.sum()), dist_inicio=np.nanmean(d1), dist_azar=m_az.mean(),
                            cociente=np.nanmean(d1) / m_az.mean(), exceso=np.nanmean(d1) - m_az.mean(),
                            p_valor=float((m_az >= np.nanmean(d1)).mean())))
        if k == 5:
            for e_, a_, b_ in zip(np.array(ETAPAS)[el], d1, d_az):
                if e_.startswith('América'):
                    ame_imp.append(dict(metrica=met, etapa=e_, tecnico=corto(tec[e_]), dist_inicio=a_, dist_azar=b_))
    print(f'  {met}: {time.time() - t0:.0f} s')

R = pd.DataFrame(filas)
pd.set_option('display.width', 250)

print('\n--- (b) Chequeo: el perfil completo de R5 reproduce el de R4b (corr ≈ 1.000, diferencias ≈ 0) ---')
print(R[['metrica', 'repro_corr', 'repro_maxdif', 'crudo_maxdif']].round(4).to_string(index=False))

print(f'\n--- (c) Confiabilidad de la etapa completa (Spearman-Brown, mediana de {REPS} repartos), etapas de {ETAPA_MIN}+ ---')
print('  conf = ajustado (decide el núcleo) · p5-p95 = rango de los repartos · crudo = diagnóstico (R5-1) ·'
      ' spearman = por rangos · modelo = τ²/(τ²+σ²/n) con el n mediano · min10/min20 = etapas de 10+ y 20+')
print(R[['metrica', 'etapas', 'r_mitades', 'conf', 'conf_p5', 'conf_p95', 'conf_crudo', 'conf_spearman', 'conf_modelo',
         'conf_min10', 'conf_min20', 'franja']].round(2).sort_values('conf', ascending=False).to_string(index=False))

print('\n--- (d) Señal, ruido y reloj de medición (partidos para llegar a 0.7) ---')
print('  tau = diferencia real entre etapas (desv. est.) · sigma_partido = ruido de un partido · misma unidad que la métrica')
print(R[['metrica', 'tau', 'sigma_partido', 'n_med', 'reloj', 'reloj_p5', 'reloj_p95']].round(2)
      .sort_values('reloj').to_string(index=False))
NN = [5, 10, 15, 20, 30, 40, 60, 90]
cur = pd.DataFrame({n: [(t ** 2) / (t ** 2 + s ** 2 / n) if t > 0 else 0 for t, s in zip(R.tau, R.sigma_partido)] for n in NN},
                   index=R.metrica)
print('\nCurva de confiabilidad según el número de partidos de la etapa:')
print(cur.round(2).to_string())

print(f'\n--- (e) Correlación entre los perfiles de las métricas (etapas de {ETAPA_MIN}+; insumo de la no redundancia, sin agrupar) ---')
inc15 = n_et >= ETAPA_MIN
CM = pd.DataFrame({CORTA.get(m, m): perf_full[m][inc15] for m in METRICAS}).corr()
print(CM.round(2).to_string())

I = pd.DataFrame(implant)
print('\n--- (f) Reloj de implantación: distancia de los primeros k partidos al resto de la etapa, frente a k partidos al azar ---')
print('  cociente > 1 = el arranque se aparta más de lo que explica el azar · p_valor = % de placebos con distancia ≥ la del arranque')
print(I.round(3).pivot_table(index='metrica', columns='k', values=['cociente', 'p_valor'], sort=False).round(2).to_string())
print('Etapas usadas por k:', I.groupby('k')['etapas'].max().to_dict())

C = pd.concat(contr, ignore_index=True)
print('\n--- (g) América: perfil de R4b (desviación frente a la liga) → perfil mostrado tras la contracción P-5 ---')
ca = C[C.etapa_id.str.startswith('América') & (C.partidos > 0)].copy()
ca['tec'] = ca['tecnico'].map(corto)
ca['txt'] = ca.apply(lambda r: f"{r.perfil_r4:+.1f} → {r.perfil_mostrado:+.1f} (k {r.k_p5:.2f})", axis=1)
print(ca.pivot_table(index='metrica', columns='tec', values='txt', aggfunc='first', sort=False).to_string())
if ame_imp:
    print('\nAmérica, arranque de 5 partidos (distancia al resto de su etapa; azar = promedio de los placebos):')
    print(pd.DataFrame(ame_imp).pivot_table(index='metrica', columns='tecnico', values=['dist_inicio', 'dist_azar'],
                                            sort=False).round(1).to_string())

# ---------- Guardado ----------
R.to_parquet(os.path.join(FW, 'r5_confiabilidad.parquet'), index=False); R.to_csv(os.path.join(SAL, 'r5_confiabilidad.csv'), index=False)
C.to_parquet(os.path.join(FW, 'r5_perfiles_contraidos.parquet'), index=False)
I.to_csv(os.path.join(SAL, 'r5_implantacion.csv'), index=False)
print(f'\nR5 terminó en {time.time() - t_ini:.0f} s. Guardado: fw1/r5_confiabilidad, fw1/r5_perfiles_contraidos, '
      'salidas/r5_confiabilidad.csv, salidas/r5_implantacion.csv')

R5b Arranque

In [ ]:
# =====================================================================
# R5b · Arranque de los técnicos de un equipo: dirección, rareza y herencia
# Framework v1.1: N-2 (reloj) · Sesión 7: R5-6 (implantación) y R5-9 (regla C) · adelanto del plan C (T-3, herencia)
# Sesión 7 · 2026-10-13 · Hackathon ISAC 2026, Track 2
# Antes: celda 1 (Drive), R0 y la celda R5 en ESTA sesión de Colab (usa sus funciones y tablas en memoria).
# Datos: SOLO entrenamiento. Tarda ~1 min.
# =====================================================================

# [R5b] ---------------------------------------------------------------
assert all(v in globals() for v in ['perfil_sub', 'tabla_partidos', 'perf_full', 'R', 'ini']), 'Corre primero la celda R5 en esta sesión'
t_ini = time.time()
EQUIPO = 'América'                                  # el código acepta cualquier equipo
KS_B, PLAC_B = [5, 10], 500                         # arranques de 5 y 10 partidos; 500 placebos por etapa
BLOQUE, N_BLOQ = 5, 6                               # trayectoria: bloques de 5 partidos, los primeros 30
CLAVE = ['posesion', 'field_tilt', 'saque_meta_corto', 'progresion', 'presion_alta']   # 4 con implantación + posesión (control)
SIG = R.set_index('metrica')['sigma_partido']       # ruido de un partido (R5, bloque d)

# Etapa anterior del mismo equipo (para la herencia)
prev_de = {}
for _, g in ini.groupby('team'):
    ids = list(g.sort_values('fecha')['etapa_id'])
    prev_de.update(dict(zip(ids[1:], ids[:-1])))
OBJ_ET = [e for e in ETAPAS if e.startswith(EQUIPO) and OBS[COD[e]] and e in prev_de]
print('--- (a) Etapas analizadas (inicio visto y con técnico anterior en los datos) ---')
for e in OBJ_ET:
    print(f"  {e}: {corto(tec[e])}, {n_et[COD[e]]} partidos · técnico anterior: {corto(tec[prev_de[e]])} ({prev_de[e]})")

filas_b, tray = [], []
for met in METRICAS:
    M, s2 = tabla_partidos(met)
    tau2_e = float(PE.loc[PE.metrica == met, 'tau_estado'].iloc[0]) ** 2
    A = {k: M[k].values.astype(float) for k in COLS}; cod = M['cod'].values; N = len(cod)
    cnt = np.bincount(cod, minlength=NS); start = np.concatenate([[0], np.cumsum(cnt)[:-1]])
    pos = np.arange(N) - start[cod]
    P = lambda m: perfil_sub(A, cod, m, tau2_e, s2)[0]
    for e in OBJ_ET:
        i = COD[e]; fe = cod == i
        prev = perf_full[met][COD[prev_de[e]]]                       # perfil completo del técnico anterior
        for k in KS_B:
            if cnt[i] < k + 10: continue
            f = fe & (pos < k)
            p1, p2 = P(f)[i], P(fe & ~f)[i]                          # arranque y resto de la etapa
            cand = np.where(fe & (pos >= k))[0]
            dist = np.empty(PLAC_B)
            for j in range(PLAC_B):
                ch = np.zeros(N, bool); ch[rng.choice(cand, k, replace=False)] = True
                dist[j] = abs(P(ch)[i] - P(fe & ~ch)[i])
            gap = p2 - prev                                          # cuánto se separa el técnico de su antecesor
            ruido_k = SIG[met] / np.sqrt(k)                          # ruido aproximado de un arranque de k partidos
            filas_b.append(dict(metrica=met, tecnico=corto(tec[e]), k=k, inicio=p1, resto=p2, dif=p1 - p2,
                                anterior=prev, brecha=gap,
                                herencia=(p1 - prev) / gap if abs(gap) > ruido_k else np.nan,
                                p_etapa=float((dist >= abs(p1 - p2)).mean())))
        if met in CLAVE:
            fila = dict(metrica=met, tecnico=corto(tec[e]))
            for b in range(N_BLOQ):
                fila[f'p{b * BLOQUE + 1}-{(b + 1) * BLOQUE}'] = P(fe & (pos >= b * BLOQUE) & (pos < (b + 1) * BLOQUE))[i]
            fila['resto_31+'] = P(fe & (pos >= N_BLOQ * BLOQUE))[i]
            fila['anterior'] = prev
            fila['banda±'] = SIG[met] / np.sqrt(BLOQUE)
            tray.append(fila)

B5 = pd.DataFrame(filas_b)
orden = {m: j for j, m in enumerate(CLAVE + [m for m in METRICAS if m not in CLAVE])}
B5 = B5.sort_values(['metrica', 'tecnico', 'k'], key=lambda s: s.map(orden) if s.name == 'metrica' else s)
print('\n--- (b) Arranque frente al resto de la etapa (perfil = desviación frente a la liga, unidad de la métrica) ---')
print('  dif = inicio − resto (signo: arrancó por arriba o por abajo) · anterior = perfil completo del técnico previo ·'
      ' brecha = resto − anterior')
print('  herencia: 0 = arrancó igual que su antecesor · 1 = arrancó ya como él mismo · vacío = brecha menor que el ruido')
print('  p_etapa = % de 500 arranques al azar de ESA etapa con distancia ≥ la real (rareza del arranque)')
print(B5.round(2).to_string(index=False))

T = pd.DataFrame(tray)
print(f'\n--- (c) Trayectoria en bloques de {BLOQUE} partidos (perfil de cada bloque; banda± = ruido de un bloque, 1 desv. est.) ---')
print(T.round(1).to_string(index=False))

B5.to_csv(os.path.join(SAL, 'r5b_arranque.csv'), index=False); T.to_csv(os.path.join(SAL, 'r5b_trayectoria.csv'), index=False)
print(f'\nR5b terminó en {time.time() - t_ini:.0f} s. Guardado: salidas/r5b_arranque.csv, salidas/r5b_trayectoria.csv')

Jardine y Ortiz comp pt.2

In [ ]:
for nombre in ['Jardine', 'Ortiz']:
    for e in ETAPAS:
        if isinstance(tec[e], str) and nombre in tec[e]:
            i = COD[e]
            print(e, '|', tec[e], '|', int(n_et[i]), 'partidos |',
                  {m: round(float(perf_full[m][i]), 1) for m in
                   ['saque_meta_corto', 'riesgo_pase', 'posesion', 'field_tilt', 'presion_alta']})

# Sesión 8

E9a  - Validación v1.1

In [ ]:
# =====================================================================
# E9a v1.1 · Capa predictiva de comportamiento: validación en entrenamiento
# v1.1 (2026-10-14): perfil ponderado por eventos (corrección A, E9-8) y diagnóstico del sesgo de la v1.0
# Framework v1.2, E-9 · Decisiones de la sesión 8: E9-1 a E9-7
# Sesión 8 · 2026-10-14 · Hackathon ISAC 2026, Track 2
# Antes: celda 1 (Drive) y R0. Lee las cachés de R4 (tramos, partidos) y R5 (solo para comparar).
# Datos: SOLO entrenamiento. Elige en 2023/24 (validación) y estima en 2024/25 (D-2, E-11).
# Ninguna línea lee eventos de 2025/26 ni 2026/27; el bloque 0 solo cuenta técnicos en la tabla de técnicos.
# Tarda ~3-8 min.
# =====================================================================

# [E9a v1.1] ---------------------------------------------------------------
import os, json, time
import numpy as np
import pandas as pd

SEMILLA = 2026
BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026/')
DP = os.path.join(BASE, 'datos_parquet'); FW = os.path.join(DP, 'fw1'); SAL = os.path.join(BASE, 'salidas')
os.makedirs(SAL, exist_ok=True)
t_ini = time.time()
pd.set_option('display.width', 250)

with open(os.path.join(FW, 'parametros_fw1.json')) as f:
    PRM = json.load(f)
ENT = sorted(PRM['temporadas']['entrenamiento'])
VAL, EST = ENT[2], ENT[3]                       # D-2: se elige en 2023/24; 2024/25 solo estima

# Decisiones E9-1 a E9-7 (sesión 8)
CONDUCTAS = ['saque_meta_corto', 'presion_alta', 'pase_largo', 'recuperacion_tras_perdida']   # E9-1: la primera es la principal
PRINC = CONDUCTAS[0]
N0S, N0_DEF = [3, 5, 8], 5                      # mezcla del estilo del rival y del nivel del torneo: w = n / (n + n0)
ESCALA_DEF = 'A'                                # E9-6: A = sumar en puntos; B = sumar en log-momios
UMBRAL = 0.003                                  # regla de entrada (E-11): mejora mínima de log-loss por evento
CLIP = (0.01, 0.99)                             # E9-6 A: recorte de probabilidades
B_REPS = 2000                                   # E9-4: bootstrap pareado por partido
JORNADA = 18                                    # equipo-partidos por jornada (18 equipos)
MIN_ETAPA_VAR = 10                              # etapas con 10+ partidos para estimar τ² y σ²
MODELOS = ['M0', 'M1', 'M2', 'M3a', 'M3b', 'M3c']
PONDERADO = True                                # E9-8 (corrección A): el perfil promedia residuos ponderando por eventos, como la receta

# ---------- 0. Técnicos nuevos en las temporadas de prueba (solo la tabla de técnicos; sin eventos) ----------
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)].copy()
ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ini_et = ptp.groupby('etapa_id')['fecha'].min()
PRUEBA = sorted(set(ptp['temporada']) - set(ENT))
print('--- (0) Técnicos nuevos en las temporadas de prueba (tabla de técnicos, sin eventos) ---')
for temp in PRUEBA:
    d = ptp[ptp['temporada'] == temp]
    f0 = d['fecha'].min()
    filas = []
    for e in sorted(d['etapa_id'].unique(), key=lambda e: ini_et[e]):
        if ini_et[e] < f0: continue                                 # la etapa viene de antes: técnico que continúa
        r = d[d['etapa_id'] == e].iloc[0]
        previo = ptp[(ptp['tec_p'] == r['tec_p']) & (ptp['team'] != r['team']) & ptp['temporada'].isin(ENT)]
        antes = ptp[(ptp['team'] == r['team']) & (ptp['fecha'] < ini_et[e])].sort_values('fecha')
        filas.append(dict(equipo=r['team'], tecnico=r['tec_p'], inicio=ini_et[e].date(),
                          partidos_en_tabla=int((d['etapa_id'] == e).sum()),
                          tecnico_anterior=antes['tec_p'].iloc[-1] if len(antes) else '',
                          etapa_previa_en_otro_club=int(previo['match_id'].nunique())))
    print(f"{temp}: equipos {d['team'].nunique()} · etapas nuevas {len(filas)} · "
          f"con etapa previa en otro club dentro del entrenamiento: {sum(f['etapa_previa_en_otro_club'] > 0 for f in filas)}")
    if filas: print(pd.DataFrame(filas).to_string(index=False))

# ---------- 1. Insumos de entrenamiento ----------
TR = pd.read_parquet(os.path.join(FW, 'r4_tramos_v21.parquet'))
Mm = pd.read_parquet(os.path.join(FW, 'r4_partidos.parquet'))
Mm['fecha'] = pd.to_datetime(Mm['fecha'])
assert set(Mm['temporada']) <= set(ENT), 'E9a solo usa entrenamiento'
elo = pd.read_parquet(os.path.join(FW, 'elo_equipo_partido_e2.parquet'))[['match_id', 'team', 'elo_rival_pre']]
R5 = pd.read_parquet(os.path.join(FW, 'r5_confiabilidad.parquet')).set_index('metrica')
ptp_e = ptp[ptp['temporada'].isin(ENT)]
assert not ptp_e.duplicated(['match_id', 'team']).any(), 'tabla de técnicos con filas repetidas'

TORNEOS = list(Mm.groupby('torneo')['fecha'].min().sort_values().index)
PREV_T = {t: (TORNEOS[i - 1] if i else None) for i, t in enumerate(TORNEOS)}
lado = pd.concat([Mm.assign(team=Mm.local, rival=Mm.visita, local_=1), Mm.assign(team=Mm.visita, rival=Mm.local, local_=0)])
CTX = (lado[['match_id', 'team', 'rival', 'local_', 'torneo', 'temporada', 'fecha']]
       .merge(elo, on=['match_id', 'team'], how='left')
       .merge(ptp_e[['match_id', 'team', 'etapa_id', 'tec_p']], on=['match_id', 'team'], how='left'))
CTX['elo_c'] = (CTX['elo_rival_pre'] - 1500) / 100
print(f"\n--- (1) Insumos --- partidos {Mm.match_id.nunique()} · torneos {len(TORNEOS)} · validación {VAL} · estimación {EST}")
print(f"Equipo-partidos sin Elo del rival: {int(CTX.elo_c.isna().sum())} · sin etapa de técnico: {int(CTX.etapa_id.isna().sum())}")

TRM, TMM = {}, {}
for met in CONDUCTAS:
    d = TR[TR.metrica == met].groupby(['match_id', 'team', 'e5', 'cuarto'], as_index=False)[['num', 'den']].sum()
    d = d[d['den'] > 0].merge(CTX, on=['match_id', 'team'])
    d = d.dropna(subset=['elo_c', 'etapa_id']).reset_index(drop=True)
    d['y'] = d['num'] / d['den']                                   # puntos (0-100)
    d['k'] = d['num'] / 100                                        # eventos con la conducta
    TRM[met] = d
    tm = d.groupby(['match_id', 'team'], as_index=False)[['num', 'den']].sum().merge(CTX, on=['match_id', 'team'])
    TMM[met] = tm
    exceso = int((d['k'] > d['den'] + 1e-6).sum())
    print(f"  {met}: tramos {len(d):,} · eventos {d['den'].sum():,.0f} · tasa liga {100 * d['k'].sum() / d['den'].sum():.1f}% · "
          f"eventos por equipo-partido (mediana) {tm['den'].median():.0f} · tramos con más éxitos que eventos (debe ser 0): {exceso}")

# ---------- 2. Herramientas ----------
def estilo_pred(tm, n0):
    """Estilo del rival con información previa (E-8): mezcla de su torneo anterior y su torneo actual hasta la fecha.
    w = n / (n + n0), n = partidos previos del rival en el torneo actual. Unidad: puntos sobre la liga."""
    c = tm[['match_id', 'fecha', 'torneo', 'rival', 'num', 'den']].sort_values(['fecha', 'match_id']).copy()
    dia = c.groupby(['torneo', 'fecha'])[['num', 'den']].sum()
    acu = dia.groupby(level=0).cumsum() - dia                        # liga antes de esa fecha
    c = c.join(acu.rename(columns={'num': 'ln', 'den': 'ld'}), on=['torneo', 'fecha'])
    g = c.groupby(['torneo', 'rival'])
    c['cn'] = g['num'].cumsum() - c['num']; c['cd'] = g['den'].cumsum() - c['den']; c['n'] = g.cumcount()
    with np.errstate(divide='ignore', invalid='ignore'):
        actual = np.where((c['cd'] > 0) & (c['ld'] > 0), c['cn'] / c['cd'] - c['ln'] / c['ld'], 0.0)
    full = tm.groupby(['torneo', 'rival'])[['num', 'den']].sum()
    liga = tm.groupby('torneo')[['num', 'den']].sum()
    dev = (full['num'] / full['den']) - (liga['num'] / liga['den']).reindex(full.index.get_level_values(0)).values
    previo = np.array([dev.get((PREV_T[t], r), 0.0) if PREV_T[t] else 0.0 for t, r in zip(c['torneo'], c['rival'])])
    w = c['n'] / (c['n'] + n0)
    c['estilo'] = np.nan_to_num(w * actual + (1 - w) * previo)
    return c.set_index(['match_id', 'rival'])['estilo']

BASE_COLS = ['e-2', 'e-1', 'e+1', 'e+2', 'C2', 'C3', 'C4', 'local', 'elo_rival_100', 'estilo_rival']
def x_base(d):
    X = [(d['e5'] == k).astype(float).values for k in [-2, -1, 1, 2]]
    X += [(d['cuarto'] == c).astype(float).values for c in [2, 3, 4]]
    X += [d['local_'].astype(float).values, d['elo_c'].values, d['estilo'].values]
    return np.column_stack(X)

def ajustar(d):
    """Receta de R4 (sección C) sin árbitro, con un nivel por torneo en lugar de intercepto, y celdas estado × cuarto
    con contracción DerSimonian-Laird. Solo con las temporadas anteriores a la evaluada."""
    tors = [t for t in TORNEOS if t in set(d['torneo'])]
    X = np.column_stack([x_base(d)] + [(d['torneo'] == t).astype(float).values for t in tors])
    y, w = d['y'].values, d['den'].values; sw = np.sqrt(w)
    beta = np.linalg.lstsq(X * sw[:, None], y * sw, rcond=None)[0]
    e = y - X @ beta
    s2 = (w * e ** 2).sum() / (len(y) - X.shape[1])
    c = pd.DataFrame({'e': d['e5'].values, 'q': d['cuarto'].values, 'we': w * e, 'w': w}).groupby(['e', 'q'])[['we', 'w']].sum()
    c['r'] = c['we'] / c['w']; c['se2'] = s2 / c['w']
    wi = 1 / c['se2']; gl_ = max(1, len(c) - 8)
    tau2 = max(0.0, ((wi * c['r'] ** 2).sum() - gl_) / (wi.sum() - (wi ** 2).sum() / wi.sum()))
    c['celda'] = (tau2 / (tau2 + c['se2']) if tau2 > 0 else 0.0) * c['r']
    nb = len(BASE_COLS)
    return dict(beta=beta[:nb], L=dict(zip(tors, beta[nb:])), celda=c['celda'],
                pbar=np.average(y, weights=w) / 100, ultimo=tors[-1])

def base_pred(d, mod):
    cel = pd.Series(list(zip(d['e5'], d['cuarto']))).map(mod['celda']).fillna(0.0).values
    return x_base(d) @ mod['beta'] + cel

def niveles(dE, mod, n0):
    """Nivel del torneo en la temporada evaluada, en línea: nivel del torneo anterior + w × lo que la liga se ha
    desviado en las jornadas previas. w = J / (J + n0), J = jornadas equivalentes jugadas."""
    lev = np.full(len(dE), np.nan); L_prev = mod['L'][mod['ultimo']]; Lfin = {}
    for t in [t for t in TORNEOS if t in set(dE['torneo'])]:
        m = (dE['torneo'] == t).values; s = dE[m]
        rl = s['den'] * (s['y'] - s['b'] - L_prev)
        dia = pd.DataFrame({'fecha': s['fecha'], 'rl': rl, 'den': s['den']}).groupby('fecha').sum()
        ntm = s.drop_duplicates(['match_id', 'team']).groupby('fecha').size().reindex(dia.index, fill_value=0)
        crl, cden, ctm = dia['rl'].cumsum() - dia['rl'], dia['den'].cumsum() - dia['den'], ntm.cumsum() - ntm
        J = ctm / JORNADA
        nivel = L_prev + np.where(cden > 0, (J / (J + n0)) * crl / cden.where(cden > 0, 1), 0.0)
        lev[m] = s['fecha'].map(pd.Series(nivel, index=dia.index)).values
        L_prev = L_prev + dia['rl'].sum() / dia['den'].sum(); Lfin[t] = L_prev
    return lev, Lfin

def k_(n, tau2, sig2):
    n = np.asarray(n, float)
    if tau2 <= 0: return np.zeros_like(n)
    return np.where(n > 0, tau2 / (tau2 + sig2 / np.maximum(n, 1)), 0.0)

def prob(yhat, perfil, escala, pbar):
    if escala == 'A':
        p = (yhat + perfil) / 100
    else:
        p0 = np.clip(yhat / 100, *CLIP)
        p = 1 / (1 + np.exp(-(np.log(p0 / (1 - p0)) + perfil / (100 * pbar * (1 - pbar)))))
    return np.clip(p, *CLIP)

def correr(met, temp, n0, escala, ponderado=PONDERADO):
    d = TRM[met].copy()
    d['estilo'] = pd.Series(list(zip(d['match_id'], d['rival']))).map(estilo_pred(TMM[met], n0)).fillna(0.0).values
    fit = [t for t in ENT if t < temp]
    dF = d[d['temporada'].isin(fit)].copy(); dE = d[d['temporada'] == temp].copy()
    mod = ajustar(dF)
    dF['yhat'] = base_pred(dF, mod) + dF['torneo'].map(mod['L']).values
    dE['b'] = base_pred(dE, mod)
    lev, _ = niveles(dE, mod, n0)
    dE['yhat'] = dE['b'] + lev
    # Residuo por equipo-partido (puntos): en las temporadas de ajuste, dentro de muestra; en la evaluada, fuera de muestra
    H = pd.concat([dF, dE])
    H = (H.assign(we=H['den'] * (H['y'] - H['yhat'])).groupby(['match_id', 'team'], as_index=False)[['we', 'den']].sum()
         .merge(CTX[['match_id', 'team', 'fecha', 'temporada', 'etapa_id', 'tec_p']], on=['match_id', 'team']))
    H['r'] = H['we'] / H['den']
    H = H.sort_values(['fecha', 'match_id']).reset_index(drop=True)
    # τ² (diferencia real entre etapas) y σ² (ruido de un partido), solo con las temporadas de ajuste
    h = H[H['temporada'].isin(fit)].groupby('etapa_id')['r']
    n_s, m_s, v_s = h.size(), h.mean(), h.var(ddof=1)
    ok = n_s >= MIN_ETAPA_VAR
    sig2 = float(np.average(v_s[ok], weights=n_s[ok] - 1))
    tau2 = float(max(0.0, m_s[ok].var(ddof=1) - np.mean(sig2 / n_s[ok])))
    # Perfiles en línea (solo partidos previos de cada fila)
    g = H.groupby('etapa_id')
    H['ns'] = g.cumcount(); H['ss'] = g['r'].cumsum() - H['r']
    H['sw'] = g['we'].cumsum() - H['we']; H['sd'] = g['den'].cumsum() - H['den']
    E = H[H['temporada'] == temp].copy()
    f0 = E['fecha'].min()
    def mu(x):
        if not len(x): return 0.0
        m = x['we'].sum() / x['den'].sum() if ponderado else x['r'].mean()
        return float(k_(len(x), tau2, sig2) * m)
    mub, muc, tienec = {}, {}, {}
    for s_ in E['etapa_id'].unique():
        r0 = E[E['etapa_id'] == s_].iloc[0]; st = ini_et[s_]
        mub[s_] = mu(H.loc[(H['team'] == r0['team']) & (H['fecha'] < st)])
        prev = H.loc[(H['tec_p'] == r0['tec_p']) & (H['team'] != r0['team']) & (H['fecha'] < st)]
        muc[s_], tienec[s_] = mu(prev), len(prev) > 0
    mu2 = {t_: mu(H.loc[(H['team'] == t_) & (H['fecha'] < f0)]) for t_ in E['team'].unique()}
    kE = k_(E['ns'], tau2, sig2)
    with np.errstate(divide='ignore', invalid='ignore'):
        rbar = np.where(E['ns'] > 0, (E['sw'] / E['sd'].where(E['sd'] > 0, 1)) if ponderado else E['ss'] / E['ns'].clip(lower=1), 0.0)
    for v, m0 in [('M3a', 0.0), ('M3b', E['etapa_id'].map(mub).values), ('M3c', E['etapa_id'].map(muc).values)]:
        E['p_' + v] = m0 + kE * (rbar - m0)
    E['p_M2'] = E['team'].map(mu2).values
    E['nueva'] = E['etapa_id'].map(ini_et).values >= f0
    E['npart'] = E['ns'] + 1
    E['con_previa'] = E['etapa_id'].map(tienec).values
    dE = dE.merge(E[['match_id', 'team', 'p_M2', 'p_M3a', 'p_M3b', 'p_M3c', 'nueva', 'npart', 'con_previa']],
                  on=['match_id', 'team'], how='left')
    p0 = dF.loc[dF['temporada'] == fit[-1], 'num'].sum() / dF.loc[dF['temporada'] == fit[-1], 'den'].sum() / 100
    P = {'M0': np.full(len(dE), np.clip(p0, *CLIP)), 'M1': prob(dE['yhat'].values, 0.0, escala, mod['pbar'])}
    for v in ['M2', 'M3a', 'M3b', 'M3c']:
        P[v] = prob(dE['yhat'].values, dE['p_' + v].values, escala, mod['pbar'])
    kk, nn = dE['k'].values, dE['den'].values
    for v, p in P.items():
        dE['q_' + v] = p
        dE['ll_' + v] = -(kk * np.log(p) + (nn - kk) * np.log(1 - p))
        dE['br_' + v] = kk * (1 - p) ** 2 + (nn - kk) * p ** 2
    hf = H[H['temporada'].isin(fit)]
    info = dict(tau=np.sqrt(tau2), sigma=np.sqrt(sig2), etapas_var=int(ok.sum()), L=mod['L'], p0=p0,
                r_igual=hf['r'].mean(), r_pond=hf['we'].sum() / hf['den'].sum(), corr_den_r=hf['den'].corr(hf['r']))
    return dE, info

def boot(dE, a, b, mask=None):
    """Diferencia de log-loss por evento (a − b; negativo = a mejor), bootstrap pareado por partido."""
    x = dE if mask is None else dE[mask]
    g = x.groupby('match_id')[['ll_' + a, 'll_' + b, 'den']].sum().values
    M = len(g)
    if M < 5: return dict(dif=np.nan, lo=np.nan, hi=np.nan, skill=np.nan, partidos=M)
    idx = np.random.default_rng(SEMILLA).integers(0, M, (B_REPS, M))
    dif = (g[idx, 0].sum(1) - g[idx, 1].sum(1)) / g[idx, 2].sum(1)
    return dict(dif=(g[:, 0].sum() - g[:, 1].sum()) / g[:, 2].sum(), lo=np.quantile(dif, .025), hi=np.quantile(dif, .975),
                skill=1 - g[:, 0].sum() / g[:, 1].sum(), partidos=M)

def r2(dE, v):
    x = dE.assign(pn=dE['q_' + v] * dE['den'], vb=dE['den'] * dE['q_' + v] * (1 - dE['q_' + v]))
    g = x.groupby(['match_id', 'team'])[['k', 'den', 'pn', 'vb']].sum()
    y, yh, w = g['k'] / g['den'], g['pn'] / g['den'], g['den']
    sst = (w * (y - np.average(y, weights=w)) ** 2).sum()
    return 1 - (w * (y - yh) ** 2).sum() / sst, 1 - (g['vb'] / g['den']).sum() / sst

def calibracion(dE, v):
    x = dE.sort_values('q_' + v)
    cum = x['den'].cumsum() / x['den'].sum()
    x['bin'] = np.clip(np.ceil(cum * 10) - 1, 0, 9).astype(int)
    g = x.assign(pn=x['q_' + v] * x['den']).groupby('bin')[['pn', 'k', 'den']].sum()
    return pd.DataFrame({'eventos': g['den'], 'pronostico': 100 * g['pn'] / g['den'], 'observado': 100 * g['k'] / g['den']})

por_evento = lambda dE, col: dE[col].sum() / dE['den'].sum()

# ---------- 3. Diagnóstico del sesgo de la v1.0 (validación, conducta principal) ----------
print(f'\n--- (1b) Diagnóstico de la v1.0 en {VAL} ({PRINC}): residuo medio de los partidos de ajuste, con peso igual y ponderado ---')
print('  Si la hipótesis es cierta: residuo con peso igual > 0, ponderado ≈ 0 y correlación eventos-residuo < 0')
dg = []
for pond in [False, True]:
    dE_, inf_ = correr(PRINC, VAL, N0_DEF, ESCALA_DEF, ponderado=pond)
    dg.append(dict(perfil='ponderado (v1.1)' if pond else 'peso igual (v1.0)', residuo_peso_igual=inf_['r_igual'],
                   residuo_ponderado=inf_['r_pond'], corr_eventos_residuo=inf_['corr_den_r'], tasa=100 * dE_['k'].sum() / dE_['den'].sum(),
                   media_M1=100 * (dE_['q_M1'] * dE_['den']).sum() / dE_['den'].sum(),
                   media_M3a=100 * (dE_['q_M3a'] * dE_['den']).sum() / dE_['den'].sum(), ll_M3a=por_evento(dE_, 'll_M3a')))
print(pd.DataFrame(dg).round(4).to_string(index=False))

# ---------- 3b. Elección en validación (solo la conducta principal) ----------
print(f'\n--- (2) Elección en validación {VAL}, conducta principal ({PRINC}) · regla: se cambia el valor por defecto '
      f'solo si mejora el log-loss por evento en ≥ {UMBRAL} ---')
sel = []
for n0 in N0S:
    dE, _ = correr(PRINC, VAL, n0, ESCALA_DEF)
    sel.append(dict(n0=n0, ll_M1=por_evento(dE, 'll_M1'), ll_M3a=por_evento(dE, 'll_M3a')))
S = pd.DataFrame(sel).set_index('n0')
mejor = S['ll_M1'].idxmin()
N0 = mejor if S.loc[N0_DEF, 'll_M1'] - S.loc[mejor, 'll_M1'] >= UMBRAL else N0_DEF
print(S.round(4).to_string()); print(f'n0 elegido: {N0} (por defecto {N0_DEF}; mejor {mejor})')
llA = por_evento(correr(PRINC, VAL, N0, 'A')[0], 'll_M3a'); llB = por_evento(correr(PRINC, VAL, N0, 'B')[0], 'll_M3a')
ESCALA = 'B' if llA - llB >= UMBRAL else 'A'
for esc_ in ['A', 'B']:
    c_ = calibracion(correr(PRINC, VAL, N0, esc_)[0], 'M3a')
    print(f'  Escala {esc_}, calibración por deciles (pronosticado → observado): ' +
          ' · '.join(f'{p:.0f}→{o:.0f}' for p, o in zip(c_['pronostico'], c_['observado'])))
print(f'Escala del perfil (M3a): A (puntos) {llA:.4f} · B (log-momios) {llB:.4f} → elegida {ESCALA}')

# ---------- 4. Resultados con la configuración elegida ----------
res, cal, cur, guard = [], [], [], {}
for temp in [VAL, EST]:
    for met in CONDUCTAS:
        dE, info = correr(met, temp, N0, ESCALA)
        guard[(temp, met)] = dE
        fila = dict(temporada=temp, conducta=met, partidos=dE['match_id'].nunique(), eventos=int(dE['den'].sum()),
                    tasa=100 * dE['k'].sum() / dE['den'].sum(), tau=info['tau'], sigma=info['sigma'],
                    tau_R5=R5.loc[met, 'tau'] if met in R5.index else np.nan,
                    sigma_R5=R5.loc[met, 'sigma_partido'] if met in R5.index else np.nan)
        for v in MODELOS:
            fila['ll_' + v] = por_evento(dE, 'll_' + v); fila['br_' + v] = por_evento(dE, 'br_' + v)
            fila['media_' + v] = 100 * (dE['q_' + v] * dE['den']).sum() / dE['den'].sum()
        nuevas5 = dE['nueva'] & (dE['npart'] <= 5) & dE['con_previa']
        for nom, a, b, m in [('M1_vs_M0', 'M1', 'M0', None), ('M3a_vs_M1', 'M3a', 'M1', None), ('M2_vs_M1', 'M2', 'M1', None),
                             ('M3b_vs_M2', 'M3b', 'M2', None), ('M3a_vs_M1_nuevas', 'M3a', 'M1', dE['nueva']),
                             ('M3c_vs_M3b_nuevas5', 'M3c', 'M3b', nuevas5)]:
            for k_b, v_b in boot(dE, a, b, m).items(): fila[f'{nom}_{k_b}'] = v_b
        fila['R2_M1'], fila['techo'] = r2(dE, 'M1'); fila['R2_M3a'], _ = r2(dE, 'M3a')
        fila['etapas_nuevas'] = dE.loc[dE['nueva'], 'etapa_id'].nunique()
        fila['nuevas_con_previa'] = dE.loc[dE['nueva'] & dE['con_previa'], 'etapa_id'].nunique()
        res.append(fila)
        c = calibracion(dE, 'M3a'); c['temporada'], c['conducta'] = temp, met; cal.append(c.reset_index())
        x = dE[dE['nueva']].copy()
        if len(x):
            x['bloque'] = pd.cut(x['npart'], [0, 5, 10, 17, 999], labels=['1-5', '6-10', '11-17', '18+'])
            gb = x.groupby('bloque', observed=True)[['ll_M1', 'll_M3a', 'll_M3b', 'll_M3c', 'den']].sum()
            for v in ['M3a', 'M3b', 'M3c']: gb['mejora_' + v] = 1 - gb['ll_' + v] / gb['ll_M1']
            gb['temporada'], gb['conducta'] = temp, met; cur.append(gb.reset_index())
        print(f'  {temp} · {met}: listo ({time.time() - t_ini:.0f} s)')
R = pd.DataFrame(res)

print('\n--- (3) Chequeos: τ y σ estimados con las temporadas de ajuste frente a R5 (misma unidad; deben parecerse) ---')
print(R[['temporada', 'conducta', 'partidos', 'eventos', 'tasa', 'tau', 'tau_R5', 'sigma', 'sigma_R5']].round(2).to_string(index=False))
print('Calibración global (% pronosticado medio frente al observado; deben estar cerca):')
print(R[['temporada', 'conducta', 'tasa'] + ['media_' + v for v in MODELOS]].round(1).to_string(index=False))

print('\n--- (4) Log-loss y Brier por evento (menor = mejor) ---')
print(R[['temporada', 'conducta'] + ['ll_' + v for v in MODELOS]].round(4).to_string(index=False))
print(R[['temporada', 'conducta'] + ['br_' + v for v in MODELOS]].round(4).to_string(index=False))

print('\n--- (5) Comparaciones (diferencia de log-loss por evento, IC95% bootstrap pareado por partido; negativo = el primero mejora) ---')
print('  skill = % de mejora del primero sobre el segundo · M3a_vs_M1 es el contraste OFICIAL (E9-5)')
for nom in ['M3a_vs_M1', 'M1_vs_M0', 'M2_vs_M1', 'M3b_vs_M2', 'M3a_vs_M1_nuevas', 'M3c_vs_M3b_nuevas5']:
    t_ = R[['temporada', 'conducta'] + [f'{nom}_{c}' for c in ['dif', 'lo', 'hi', 'skill', 'partidos']]].copy()
    t_.columns = ['temporada', 'conducta', 'dif', 'lo', 'hi', 'skill', 'partidos']
    t_['skill'] = 100 * t_['skill']
    print(f'{nom}:'); print(t_.round(4).to_string(index=False))
print('Etapas nuevas por temporada evaluada:', R.groupby('temporada')[['etapas_nuevas', 'nuevas_con_previa']].max().to_dict('index'))

print('\n--- (6) R² por equipo-partido (% de la conducta en cada partido) y su techo por el azar de pocos eventos ---')
print(R[['temporada', 'conducta', 'R2_M1', 'R2_M3a', 'techo']].round(3).to_string(index=False))

C_ = pd.concat(cal, ignore_index=True)
print('\n--- (7) Calibración de M3a por deciles (% pronosticado frente a % observado), conducta principal ---')
print(C_[C_.conducta == PRINC].pivot_table(index='bin', columns='temporada', values=['pronostico', 'observado', 'eventos'])
      .round(1).to_string())
if cur:
    K_ = pd.concat(cur, ignore_index=True)
    print('\n--- (8) Etapas nuevas: mejora sobre M1 por bloque de partidos de la etapa (apuesta 4) ---')
    print(K_[['temporada', 'conducta', 'bloque', 'den', 'mejora_M3a', 'mejora_M3b', 'mejora_M3c']]
          .assign(**{c: lambda d, c=c: 100 * d[c] for c in ['mejora_M3a', 'mejora_M3b', 'mejora_M3c']})
          .round(1).to_string(index=False))
    K_.to_csv(os.path.join(SAL, 'e9a_curva.csv'), index=False)

# ---------- 5. Guardado ----------
R.to_csv(os.path.join(SAL, 'e9a_resumen.csv'), index=False)
C_.to_csv(os.path.join(SAL, 'e9a_calibracion.csv'), index=False)
with open(os.path.join(FW, 'e9a_config.json'), 'w') as f:
    json.dump(dict(estado='candidata v1.1: se congela al anotarla en la bitácora v9', perfil_ponderado=PONDERADO, conductas=CONDUCTAS, principal=PRINC,
                   n0=int(N0), escala=ESCALA, clip=CLIP, umbral=UMBRAL, bootstrap=B_REPS, semilla=SEMILLA,
                   contraste_oficial='M3a_vs_M1', metrica_principal='log-loss por evento',
                   eleccion_validacion=S.reset_index().to_dict('records'), escala_A=llA, escala_B=llB), f, indent=1, default=str)
print(f'\nE9a v1.1 terminó en {time.time() - t_ini:.0f} s. Guardado: salidas/e9a_resumen.csv, e9a_calibracion.csv, e9a_curva.csv, '
      'fw1/e9a_config.json')

V-T Validación

In [ ]:
# =====================================================================
# V-T · Viabilidad del bloque causal (framework v1.2, sección T)
# Sesión 8 · 2026-10-14 · Hackathon ISAC 2026, Track 2 · v1.0
# Decisiones: T-0 (plan C principal: ¿el técnico causa el cambio de estilo?; plan A secundario: ¿mejoran los
#             resultados?), VT-1 A (eventos del entrenamiento; 2025/26 solo se cuenta por fechas),
#             VT-2 (controles a 0.5 desviaciones en nivel, caída y suerte; 0.25 y 0.75 como sensibilidad),
#             VT-3 (plan A vive con ≥ 20 eventos con control y EMD ≤ 35 puntos de Elo ≈ 2.5 puntos por torneo).
# Antes: celda 1 (Drive) y R0. Lee las cachés de R3 (Elo y Elo de xG) y de R4/R5 (residuos y ruido).
# REGLA: esta celda NO calcula ni imprime el efecto real de los relevos. Solo cuenta eventos y mide, con
# placebos (equipos sin cambio), qué tan grande tendría que ser un efecto para poder verlo.
# Datos: SOLO entrenamiento. Tarda ~1-3 min.
# =====================================================================

# [V-T v1.0] ----------------------------------------------------------
import os, json, time
import numpy as np
import pandas as pd

SEMILLA = 2026
rng = np.random.default_rng(SEMILLA)
BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026/')
DP = os.path.join(BASE, 'datos_parquet'); FW = os.path.join(DP, 'fw1'); SAL = os.path.join(BASE, 'salidas')
os.makedirs(SAL, exist_ok=True)
t_ini = time.time()
pd.set_option('display.width', 250)

with open(os.path.join(FW, 'parametros_fw1.json')) as f:
    PRM = json.load(f)
ENT = sorted(PRM['temporadas']['entrenamiento'])
VENT = 10                              # T-1: 10 partidos antes y 10 después
KS_C = [5, 10]                         # plan C: bloques de 5 y 10 partidos a cada lado
JORN = 2                               # controles: jornada ±2 dentro del mismo torneo
CALIPERS = [0.25, 0.5, 0.75]           # VT-2: tolerancia en desviaciones estándar (0.5 principal)
CAL0 = 0.5
N_CTRL = 3                             # hasta 3 controles por evento
INTERINO = 3                           # D-5: una etapa de ≤ 3 partidos es interinato
REPS = 300                             # repeticiones de placebo
MIN_EVENTOS, EMD_MAX = 20, 35.0        # VT-3
CALENT = pd.Timestamp('2021-10-01')
METRICAS_C = None                      # None = las 12 de R4

# ---------- 1. Partidos de cada equipo en orden, con técnico, Elo y Elo de xG ----------
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)].copy()
ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ptp['torneo'] = ptp['temporada'] + np.where(ptp['fecha'].dt.month >= 7, '-A', '-C')
largo_et = ptp.groupby('etapa_id')['match_id'].nunique()

elo = pd.read_parquet(os.path.join(FW, 'elo_equipo_partido_e2.parquet'))[['match_id', 'team', 'elo_pre', 'elo_post']]
ex = pd.read_parquet(os.path.join(FW, 'elo_xg_partido.parquet'))
exl = pd.concat([ex.assign(team=ex.local, xg_pre=ex.eloxg_local_pre, xg_post=ex.eloxg_local_post),
                 ex.assign(team=ex.visita, xg_pre=ex.eloxg_visita_pre, xg_post=ex.eloxg_visita_post)])[
    ['match_id', 'team', 'xg_pre', 'xg_post']]

S = (ptp[ptp['temporada'].isin(ENT)][['match_id', 'team', 'fecha', 'temporada', 'torneo', 'tec_p', 'etapa_id']]
     .merge(elo, on=['match_id', 'team'], how='left').merge(exl, on=['match_id', 'team'], how='left')
     .sort_values(['team', 'fecha', 'match_id']).reset_index(drop=True))
assert not S.duplicated(['match_id', 'team']).any(), 'filas repetidas'
S['i'] = S.groupby('team').cumcount()
S['jornada'] = S.groupby(['team', 'torneo']).cumcount() + 1
print('--- (1) Insumos (solo entrenamiento) ---')
print(f"Equipo-partidos {len(S):,} · equipos {S.team.nunique()} · temporadas {sorted(S.temporada.unique())} · "
      f"sin Elo {int(S.elo_pre.isna().sum())} · sin Elo de xG {int(S.xg_pre.isna().sum())}")

# ---------- 2. Ventanas (equipo, partido j): covariables en t = 0 y desenlace a t + 10 ----------
def ventanas(S):
    filas = []
    for team, g in S.groupby('team', sort=False):
        g = g.reset_index(drop=True); n = len(g)
        et, tp = g['etapa_id'].values, g['temporada'].values
        for j in range(VENT, n - VENT + 1):
            a, b = j - VENT, j + VENT - 1
            if tp[a] != tp[b]: continue                                         # sin cruzar el verano
            pre_ok = (et[a:j] == et[j - 1]).all()
            cambio = et[j] != et[j - 1]
            if cambio:
                if largo_et[et[j - 1]] <= INTERINO or not pre_ok: continue      # sale un interino o pre mixto
                tipo = 'evento'
            else:
                if not (et[a:b + 1] == et[j]).all(): continue
                tipo = 'sin_cambio'
            post_et = list(dict.fromkeys(et[j:b + 1]))
            filas.append(dict(team=team, j=j, tipo=tipo, match_id=g.at[j, 'match_id'], fecha=g.at[j, 'fecha'],
                              torneo=g.at[j, 'torneo'], jornada=g.at[j, 'jornada'], temporada=tp[j],
                              sale=g.at[j - 1, 'tec_p'], entra=g.at[j, 'tec_p'],
                              interino=cambio and largo_et[et[j]] <= INTERINO, terceros=len(post_et) > 2,
                              calent=g.at[a, 'fecha'] < CALENT,
                              nivel=g.at[j, 'elo_pre'], caida=g.at[j, 'elo_pre'] - g.at[a, 'elo_pre'],
                              suerte=(g.at[j, 'elo_pre'] - g.at[a, 'elo_pre']) - (g.at[j, 'xg_pre'] - g.at[a, 'xg_pre']),
                              out=g.at[b, 'elo_post'] - g.at[j, 'elo_pre'], out_xg=g.at[b, 'xg_post'] - g.at[j, 'xg_pre']))
    return pd.DataFrame(filas)

V = ventanas(S).dropna(subset=['nivel', 'caida', 'suerte', 'out', 'out_xg']).reset_index(drop=True)
EV = V[V.tipo == 'evento'].reset_index(drop=True)
W = V[V.tipo == 'sin_cambio'].reset_index(drop=True)
COV = ['nivel', 'caida', 'suerte']
SD = W[COV].std()

# Todos los relevos de titular del entrenamiento (sin exigir ventana), para el conteo
rel = []
for team, g in S.groupby('team', sort=False):
    g = g.reset_index(drop=True)
    for j in range(1, len(g)):
        if g.at[j, 'etapa_id'] != g.at[j - 1, 'etapa_id'] and largo_et[g.at[j - 1, 'etapa_id']] > INTERINO:
            rel.append(dict(team=team, j=j, fecha=g.at[j, 'fecha'], temporada=g.at[j, 'temporada'],
                            torneo_ant=g.at[j - 1, 'torneo'], torneo=g.at[j, 'torneo'],
                            antes_misma_etapa=int((g.loc[:j - 1, 'etapa_id'] == g.at[j - 1, 'etapa_id']).sum()),
                            despues=len(g) - j))
REL = pd.DataFrame(rel)

print('\n--- (2) Cómo se distribuyen nivel, caída y suerte en t = 0 (ventanas sin cambio de técnico; puntos de Elo) ---')
print('  caída = cambio de Elo en los 10 partidos previos · suerte = caída del Elo real − caída del Elo de xG (negativa = mala suerte)')
print(W[COV].describe(percentiles=[.1, .5, .9]).T[['count', 'mean', 'std', '10%', '50%', '90%']].round(1).to_string())
print('Tolerancia de 0.5 desviaciones en puntos de Elo:', (CAL0 * SD).round(1).to_dict())
print('Relevos (equipos con relevo) frente al resto, medias:')
print(pd.DataFrame({'relevos': EV[COV].mean(), 'sin cambio': W[COV].mean()}).round(1).T.to_string())

# ---------- 3. Conteo de eventos ----------
print('\n--- (3) Eventos: relevos de técnico titular ---')
print(f"Relevos de titular en el entrenamiento: {len(REL)} · entre torneos (cruzan mercado): "
      f"{int((REL.torneo_ant != REL.torneo).sum())} · a media temporada: {int((REL.torneo_ant == REL.torneo).sum())}")
print(f"Plan A (10 antes con el técnico saliente y 10 después, misma temporada): {len(EV)} · "
      f"con interino en el arranque: {int(EV.interino.sum())} · con un tercer técnico dentro de la ventana: {int(EV.terceros.sum())} · "
      f"ventana previa antes del {CALENT.date()}: {int(EV.calent.sum())} · sin 2021/22: {int((EV.temporada != ENT[0]).sum())}")

# 2025/26: solo por fechas (tabla de técnicos), sin resultados
P26 = ptp[ptp['temporada'] == sorted(set(ptp.temporada) - set(ENT))[0]].sort_values(['team', 'fecha'])
n26 = 0
for team, g in P26.groupby('team'):
    et = g['etapa_id'].values
    for j in range(VENT, len(g) - VENT + 1):
        if et[j] != et[j - 1] and largo_et[et[j - 1]] > INTERINO and (et[j - VENT:j] == et[j - 1]).all():
            n26 += 1
print(f"2025/26 (solo fechas de la tabla de técnicos, sin resultados): {n26} eventos adicionales con ventana completa")

# ---------- 4. Emparejamiento ----------
Wv = W[COV].values / SD.values
W_tor, W_jor, W_team = W['torneo'].values, W['jornada'].values, W['team'].values

def controles(x_std, torneo, jornada, team, cal, excluir_idx=None):
    m = (W_tor == torneo) & (np.abs(W_jor - jornada) <= JORN) & (W_team != team)
    if excluir_idx is not None: m[excluir_idx] = False
    m &= (np.abs(Wv - x_std) <= cal).all(1)
    idx = np.where(m)[0]
    if not len(idx): return idx
    d = np.sqrt(((Wv[idx] - x_std) ** 2).sum(1))
    o = pd.DataFrame({'idx': idx, 'd': d, 'team': W_team[idx]}).sort_values('d').drop_duplicates('team')
    return o['idx'].values[:N_CTRL]

def emparejar(E, cal):
    res = []
    for r in E.itertuples(index=False):
        x = np.array([getattr(r, c) for c in COV]) / SD.values
        res.append(controles(x, r.torneo, r.jornada, r.team, cal))
    return res

print('\n--- (4) Eventos con controles según la tolerancia (mismo torneo, jornada ±2, sin cambio de técnico en su ventana) ---')
filas, EMP = [], {}
for cal in CALIPERS:
    c = emparejar(EV, cal); EMP[cal] = c
    n_c = np.array([len(x) for x in c]); ok = n_c > 0
    bal = {}
    if ok.any():
        ctrl_mean = np.array([W.loc[x, COV].mean().values for x in np.array(c, dtype=object)[ok]])
        bal = dict(zip([f'desbalance_{k}' for k in COV], ((EV.loc[ok, COV].values - ctrl_mean).mean(0) / SD.values).round(2)))
    filas.append(dict(tolerancia=cal, eventos=len(EV), con_1_control=int(ok.sum()), con_3=int((n_c >= 3).sum()),
                      controles_medios=round(n_c[ok].mean(), 2) if ok.any() else 0,
                      sin_2021_22=int((ok & (EV.temporada != ENT[0]).values).sum()), **bal))
print(pd.DataFrame(filas).to_string(index=False))
print('  desbalance = diferencia media evento − controles, en desviaciones estándar (|x| < 0.1 = bien emparejado)')

# ---------- 5. Placebos: sesgo y efecto mínimo detectable (planes A y B) ----------
def placebo(cal, col):
    """Cada repetición: para cada evento con control, se toma una ventana SIN cambio parecida al evento (pseudo-tratada)
    y se empareja con otras ventanas sin cambio. Como nadie cambió de técnico, el 'efecto' verdadero es 0."""
    ok_ev = [i for i, x in enumerate(EMP[cal]) if len(x)]
    Xev = EV.loc[ok_ev, COV].values / SD.values
    out = W[col].values
    medias, ingenuo = [], []
    for _ in range(REPS):
        efs, ing = [], []
        for x in Xev:
            cand = np.where((np.abs(Wv - x) <= cal).all(1))[0]
            if not len(cand): continue
            w = rng.choice(cand)
            c = controles(Wv[w], W_tor[w], W_jor[w], W_team[w], cal)
            if not len(c): continue
            efs.append(out[w] - out[c].mean()); ing.append(out[w])
        if efs: medias.append(np.mean(efs)); ingenuo.append(np.mean(ing))
    medias = np.array(medias)
    return dict(eventos_usados=len(ok_ev), sesgo_emparejado=medias.mean(), sesgo_sin_controles=np.mean(ingenuo),
                error_estandar=medias.std(ddof=1), EMD=2.8 * medias.std(ddof=1))

print(f'\n--- (5) Placebos ({REPS} repeticiones): qué pasa sin cambio de técnico, y efecto mínimo detectable (EMD) ---')
print('  sesgo_sin_controles = lo que "recupera" en 10 partidos un equipo parecido a los despedidos SIN cambiar de técnico (regresión a la media)')
print('  sesgo_emparejado = lo que queda tras restar a los controles (debe ser ≈ 0) · EMD = 2.8 × error estándar (80% de potencia, 5%)')
filas = []
for col, plan in [('out', 'A · Elo real'), ('out_xg', 'B · Elo de xG')]:
    for cal in CALIPERS:
        filas.append(dict(plan=plan, tolerancia=cal, **placebo(cal, col)))
PL = pd.DataFrame(filas)
PL['puntos_por_torneo_EMD'] = PL['EMD'] * np.log(10) / 400 * 0.25 * 3 * 17
print(PL.round(2).to_string(index=False))
print('  puntos_por_torneo_EMD = el EMD traducido a puntos en 17 partidos (≈ 0.0043 puntos por partido por cada punto de Elo)')

# ---------- 6. Plan C: ruptura de estilo en el relevo frente a fechas placebo ----------
RS = pd.read_parquet(os.path.join(FW, 'r4_residuos.parquet'))
R5 = pd.read_parquet(os.path.join(FW, 'r5_confiabilidad.parquet')).set_index('metrica')
METS = METRICAS_C or list(R5.index)
RS = RS[RS.metrica.isin(METS)]
G = (RS.assign(we=RS['den'] * RS['res']).groupby(['metrica', 'match_id', 'team'], as_index=False)[['we', 'den']].sum())
orden = S[['match_id', 'team', 'i', 'etapa_id', 'torneo']]
G = G.merge(orden, on=['match_id', 'team'])

def rupturas(k):
    """d = media ponderada del residuo en los k partidos después − en los k antes. Eventos: relevo de titular con k
    partidos previos del técnico saliente. Placebos: posiciones dentro de una misma etapa."""
    ev, pl = [], []
    for (met, team), g in G.groupby(['metrica', 'team'], sort=False):
        g = g.sort_values('i').reset_index(drop=True); n = len(g)
        we, den, et, tor = g['we'].values, g['den'].values, g['etapa_id'].values, g['torneo'].values
        cw, cd = np.concatenate([[0], np.cumsum(we)]), np.concatenate([[0], np.cumsum(den)])
        for p in range(k, n - k + 1):
            pre = (cw[p] - cw[p - k]) / (cd[p] - cd[p - k]) if cd[p] > cd[p - k] else np.nan
            post = (cw[p + k] - cw[p]) / (cd[p + k] - cd[p]) if cd[p + k] > cd[p] else np.nan
            mercado = tor[p] != tor[p - 1]
            if et[p] != et[p - 1]:
                if largo_et[et[p - 1]] > INTERINO and (et[p - k:p] == et[p - 1]).all():
                    ev.append((met, team, p, mercado, post - pre))
            elif (et[p - k:p + k] == et[p]).all():
                pl.append((met, mercado, post - pre))
    return (pd.DataFrame(ev, columns=['metrica', 'team', 'p', 'mercado', 'd']).dropna(),
            pd.DataFrame(pl, columns=['metrica', 'mercado', 'd']).dropna())

def emd_c(dn_m, dn_n, n_m, n_n, sims=2000):
    """EMD de un cambio de estilo: el menor desplazamiento s (con signo al azar) que la prueba 'la ruptura media en los
    relevos supera a la de los placebos' detecta con 80% de potencia al 5%. Cada evento usa los placebos de su tipo
    (con o sin cambio de torneo / mercado)."""
    if n_m + n_n == 0: return np.nan
    def stat(s):
        a = np.abs(rng.choice(dn_m, (sims, n_m)) + s * rng.choice([-1, 1], (sims, n_m))) if n_m else np.zeros((sims, 0))
        b = np.abs(rng.choice(dn_n, (sims, n_n)) + s * rng.choice([-1, 1], (sims, n_n))) if n_n else np.zeros((sims, 0))
        return np.concatenate([a, b], 1).mean(1)
    q = np.quantile(stat(0.0), 0.95)
    paso = np.std(np.concatenate([dn_m, dn_n])) / 10
    for s in np.arange(0, 60) * paso:
        if (stat(s) > q).mean() >= 0.8: return s
    return np.inf

print('\n--- (6) Plan C (principal): ¿el relevo rompe el estilo más que una fecha cualquiera sin relevo? ---')
filas = []
for k in KS_C:
    E_, P_ = rupturas(k)
    for met in METS:
        e, p = E_[E_.metrica == met], P_[P_.metrica == met]
        dn_m, dn_n = p.loc[p.mercado, 'd'].values, p.loc[~p.mercado, 'd'].values
        if not len(dn_m): dn_m = p['d'].values
        n_m, n_n = int(e.mercado.sum()), int((~e.mercado).sum())
        s = emd_c(dn_m, dn_n, n_m, n_n)
        banda = R5.loc[met, 'sigma_partido'] / np.sqrt(5) if met in R5.index else np.nan
        filas.append(dict(k=k, metrica=met, eventos=n_m + n_n, entre_torneos=n_m, media_temporada=n_n,
                          placebos=len(p), placebos_entre_torneos=len(dn_m),
                          ruido_placebo=p['d'].std(), EMD=s, banda_5=banda, EMD_en_bandas=s / banda if banda else np.nan))
PC = pd.DataFrame(filas)
print(PC.round(2).to_string(index=False))
print('  ruido_placebo = desviación de la ruptura en fechas sin relevo (misma unidad que la métrica) · banda_5 = σ/√5 de R5 ·'
      ' EMD_en_bandas ≤ 1 = detectamos cambios del tamaño del ruido de un bloque de 5 partidos')

# ---------- 7. Lista de eventos (sin desenlaces) y regla de decisión ----------
print('\n--- (7) Eventos del plan A con su número de controles (tolerancia 0.5; sin desenlaces) ---')
L = EV[['team', 'fecha', 'sale', 'entra', 'interino', 'nivel', 'caida', 'suerte']].copy()
L['controles'] = [len(x) for x in EMP[CAL0]]
L['fecha'] = L['fecha'].dt.date
print(L.round(0).sort_values('fecha').to_string(index=False))

pa = PL[(PL.plan.str.startswith('A')) & (PL.tolerancia == CAL0)].iloc[0]
pb = PL[(PL.plan.str.startswith('B')) & (PL.tolerancia == CAL0)].iloc[0]
n_ok = int(pa.eventos_usados)
print('\n--- (8) Regla VT-3 (escrita antes de correr) ---')
print(f"Plan C (principal): {int(PC[PC.k == 10].eventos.max())} relevos con 10 partidos a cada lado · "
      f"métricas con EMD ≤ 1 banda (k = 10): {int((PC[PC.k == 10].EMD_en_bandas <= 1).sum())} de {len(METS)}")
print(f"Plan A (secundario): eventos con control {n_ok} (mínimo {MIN_EVENTOS}) · EMD {pa.EMD:.1f} (máximo {EMD_MAX}) → "
      + ('VIVE' if n_ok >= MIN_EVENTOS and pa.EMD <= EMD_MAX else
         ('pasa a B (Elo de xG): EMD ' + f'{pb.EMD:.1f}' + (' → VIVE B' if n_ok >= MIN_EVENTOS and pb.EMD <= EMD_MAX else ' → tampoco; A y B solo descriptivos'))))

PL.to_csv(os.path.join(SAL, 'vt_placebos.csv'), index=False); PC.to_csv(os.path.join(SAL, 'vt_plan_c.csv'), index=False)
L.to_csv(os.path.join(SAL, 'vt_eventos.csv'), index=False)
print(f'\nV-T terminó en {time.time() - t_ini:.0f} s. Guardado: salidas/vt_placebos.csv, vt_plan_c.csv, vt_eventos.csv')

R6a v1.1

In [ ]:
# =====================================================================
# R6a v1.2 · Embudo de ataque: progresión, ocasiones, zonas de remate y centros
# v1.1: pases hasta el último tercio (R6-5), banda izquierda/derecha verificada (R6-6), pase extra (R6-7), centro preparado (R6-8)
# v1.2: reconstruye la caché si viene de la v1.0 (sin volver a leer los crudos); la verificación de bandas usa una muestra de 40 partidos
# Framework v1.2: O-5, M8, M14 (embudo) y M10 (generación de ocasiones) · Decisiones de la sesión 8: R6-1 a R6-8
# Sesión 8 · 2026-10-14 · Hackathon ISAC 2026, Track 2
# Antes: celda 1 (Drive) y R0. Lee las tablas de R1 (eventos_, posesiones_) y r4_partidos (marcador efectivo).
# La primera vez puede leer columnas extra de los eventos crudos (altura del pase, pase atrás, pase clave,
# xG de ejecución) y las guarda en caché (~10-15 min); después tarda ~2-4 min.
# Datos: SOLO entrenamiento. Guarda los tramos en el mismo formato que R4 para pasarlos por la receta y R5 (celda R6r).
# =====================================================================

# [R6a v1.2] -----------------------------------------------------------
import os, glob, json, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)    # divisiones entre 0 en rasgos sin variación (se ven como NaN)

SEMILLA = 2026
rng = np.random.default_rng(SEMILLA)
BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026/')
DP = os.path.join(BASE, 'datos_parquet'); FW = os.path.join(DP, 'fw1'); SAL = os.path.join(BASE, 'salidas')
DIR_EV = os.path.join(DP, 'liga', 'eventos')
os.makedirs(SAL, exist_ok=True)
REHACER = False
t_ini = time.time()
pd.set_option('display.width', 250)

with open(os.path.join(FW, 'parametros_fw1.json')) as f:
    PRM = json.load(f)
PP = PRM.get('P', {})
ENT = sorted(PRM['temporadas']['entrenamiento'])
X_MED, X_U3 = PP.get('tercio_medio_x', 40), PP.get('tercio_ultimo_x', 80)
AREA_X, AREA_Y = PP.get('area_x', 102), tuple(PP.get('area_y', (18, 62)))
CHICA_X, CHICA_Y = 114, (30, 50)                 # G-4
CARRIL = (80 / 3, 160 / 3)                       # carriles por y: banda, centro, banda
CORTE_C1, CORTE_C3 = 25, 70                      # P-18
ETAPA_MIN = PP.get('etapa_min', 15)
EXTRA = {'id': ['id'], 'altura': ['pass.height.name'], 'pase_atras': ['pass.cut_back', 'pass.cut-back'],
         'pase_clave': ['shot.key_pass_id'], 'xg_ejec': ['shot.shot_execution_xg', 'shot.execution_xg']}
N_MUESTRA_BANDAS = 40
ATRAS_YD = 3                                     # R6-8: 'hacia atrás' = termina ≥ 3 yd más lejos del arco (sensibilidad: 0)
ZONA_TIRO_X = 90                                 # R6-7 C: solo pases desde donde tirar era razonable

def en_area(x, y): return (x >= AREA_X) & (y >= AREA_Y[0]) & (y <= AREA_Y[1])
def e5(m):
    m = np.asarray(m, dtype=float)
    return np.clip(np.sign(m) * np.ceil(np.abs(m) - 1e-9), -2, 2).astype(int)

# ---------- 1. Partidos (marcador efectivo de R4: offset por liguilla) ----------
Mm = pd.read_parquet(os.path.join(FW, 'r4_partidos.parquet'))
assert set(Mm['temporada']) <= set(ENT), 'R6a solo usa entrenamiento'
loc = Mm.set_index('match_id')
mids = set(Mm['match_id'])

# ---------- 2. Columnas extra (de R1 si existen; si no, de los eventos crudos, con caché) ----------
def leer_extra(temp, cols_r1):
    falt = {k: v for k, v in EXTRA.items() if not any(c in cols_r1 for c in v)}
    if not falt: return None, []
    ruta = os.path.join(FW, f"r6_extra_{temp.replace('/', '-')}.parquet")
    if os.path.exists(ruta) and not REHACER: return pd.read_parquet(ruta), list(falt)
    piezas, sin = [], 0
    m_temp = Mm.loc[Mm.temporada == temp, 'match_id']
    for mid in m_temp:
        r = os.path.join(DIR_EV, f'{mid}.parquet')
        if not os.path.exists(r): sin += 1; continue
        disp = set(pq.read_schema(r).names)
        usar = {'index': 'index'}
        for k, ops in falt.items():
            c = next((o for o in ops if o in disp), None)
            if c: usar[c] = k
        d = pd.read_parquet(r, columns=list(usar)).rename(columns=usar)
        for k in falt:
            if k not in d: d[k] = np.nan
        d['match_id'] = mid
        piezas.append(d)
    X = pd.concat(piezas, ignore_index=True); X.to_parquet(ruta, index=False)
    print(f'  {temp}: columnas extra leídas de {len(piezas)} partidos crudos (sin archivo: {sin})')
    return X, list(falt)

# ---------- 3. Recorrido por temporada ----------
COLS = ['match_id', 'index', 'period', 'minute', 't', 'type.name', 'possession', 'possession_team.name', 'team.name',
        'loc_x', 'loc_y', 'es_reanudacion', 'fase', 'pass.cross', 'pass.outcome.name', 'pass_end_x', 'pass_end_y',
        'carry_end_x', 'carry_end_y', 'ball_receipt.outcome.name', 'shot.statsbomb_xg', 'shot.type.name',
        'shot.outcome.name'] + [c for v in EXTRA.values() for c in v]
ruta_tr = os.path.join(FW, 'r6_tramos.parquet'); ruta_ti = os.path.join(FW, 'r6_tiros.parquet')
ruta_sq = os.path.join(FW, 'r6_secuencias.parquet'); ruta_ck = os.path.join(FW, 'r6_chequeo.parquet')
ruta_lat, ruta_prep = os.path.join(FW, 'r6_laterales.parquet'), os.path.join(FW, 'r6_prep.parquet')
cache_ok = all(os.path.exists(r) for r in [ruta_tr, ruta_ti, ruta_sq, ruta_ck, ruta_lat, ruta_prep])
if cache_ok and 'pases_hasta_ultimo' not in set(pd.read_parquet(ruta_tr, columns=['metrica'])['metrica']):
    cache_ok = False
if not cache_ok and os.path.exists(ruta_tr): print('Caché de una versión anterior: se reconstruyen los tramos (los crudos ya están en caché)')
if cache_ok and not REHACER:
    TR, TI, SQ, CK = (pd.read_parquet(r) for r in [ruta_tr, ruta_ti, ruta_sq, ruta_ck])
    LAT, PREP = pd.read_parquet(ruta_lat), pd.read_parquet(ruta_prep)
    print('Cargado de caché:', len(TR), 'tramos ·', len(TI), 'tiros ·', len(SQ), 'posesiones')
else:
    piezas, tiros, secs, chk, faltaron, lat, pre_c = [], [], [], [], set(), [], []
    for temp in ENT:
        t0 = time.time()
        ruta_e = os.path.join(FW, f"eventos_{temp.replace('/', '-')}.parquet")
        cols_r1 = set(pq.read_schema(ruta_e).names)
        E = pd.read_parquet(ruta_e, columns=[c for c in COLS if c in cols_r1])
        E = E[E['match_id'].isin(mids)].sort_values(['match_id', 'index']).reset_index(drop=True)
        for k, ops in EXTRA.items():
            c = next((o for o in ops if o in E.columns), None)
            E[k] = E[c] if c else np.nan
        X, falt = leer_extra(temp, cols_r1); faltaron |= set(falt)
        if X is not None:
            E = E.drop(columns=falt).merge(X[['match_id', 'index'] + falt], on=['match_id', 'index'], how='left')
        for c in COLS:
            if c not in E.columns: E[c] = np.nan
        E['local'] = E['match_id'].map(loc['local']); E['visita'] = E['match_id'].map(loc['visita'])
        E['ev_local'] = E['team.name'] == E['local']
        # Marcador (igual que R4a): goles en periodos 1-4, tiro convertido o autogol a favor
        es_gol = E['period'].between(1, 4) & (((E['type.name'] == 'Shot') & (E['shot.outcome.name'] == 'Goal')) |
                                              (E['type.name'] == 'Own Goal For'))
        gh = (es_gol & E['ev_local']).astype(int); ga = (es_gol & ~E['ev_local'] & E['team.name'].notna()).astype(int)
        E['dif_home'] = (gh.groupby(E['match_id']).cumsum() - gh) - (ga.groupby(E['match_id']).cumsum() - ga)
        E = E[E['period'].isin([1, 2])].copy()                                           # D-9
        E['m_home'] = E['dif_home'] + E['match_id'].map(loc['offset'])
        E['m_home_b'] = E['dif_home'] + E['match_id'].map(loc['offset_b'])
        E['cuarto'] = np.where(E['period'] == 1, np.where(E['minute'] < CORTE_C1, 1, 2),
                               np.where(E['minute'] < CORTE_C3, 3, 4))
        abierto = (E['fase'] != 'bp') & ~E['es_reanudacion'].fillna(False).astype(bool)
        tn = E['type.name']

        # --- Posesiones propias de juego abierto (no nacen de balón parado), con estado y cuarto de su primer evento
        Pz = pd.read_parquet(os.path.join(FW, f"posesiones_{temp.replace('/', '-')}.parquet"))
        Pz = Pz[Pz['match_id'].isin(mids) & Pz['period'].isin([1, 2])]
        Pz = Pz[~Pz['inicia_bp'].fillna(False).astype(bool)][['match_id', 'possession', 'equipo']]
        pe = E.groupby(['match_id', 'possession'], as_index=False).first()[['match_id', 'possession', 'm_home', 'm_home_b', 'cuarto']]
        Pz = Pz.merge(pe, on=['match_id', 'possession'], how='inner')
        own = E[(E['team.name'] == E['possession_team.name']) & abierto & E['loc_x'].notna()]
        g = own.groupby(['match_id', 'possession'])
        f = pd.DataFrame({'x_ini': g['loc_x'].first(), 'max_x': g['loc_x'].max()})
        toque = own[tn.loc[own.index].isin(['Pass', 'Carry', 'Shot', 'Dribble', 'Ball Receipt*']) &
                    ~((own['type.name'] == 'Ball Receipt*') & own['ball_receipt.outcome.name'].notna()) &
                    en_area(own['loc_x'], own['loc_y'])]
        f['area'] = toque.groupby(['match_id', 'possession']).size().reindex(f.index).fillna(0) > 0
        sh_op = own[(own['type.name'] == 'Shot') & (own['shot.type.name'] == 'Open Play')]
        f['tiro'] = sh_op.groupby(['match_id', 'possession']).size().reindex(f.index).fillna(0) > 0
        cr = own[(own['type.name'] == 'Pass') & own['pass.cross'].eq(True)]
        f['centro'] = cr.groupby(['match_id', 'possession']).size().reindex(f.index).fillna(0) > 0
        ent3 = own[own['loc_x'] >= X_U3].groupby(['match_id', 'possession'])['loc_y'].first()
        f['carril'] = pd.cut(ent3, [-1, CARRIL[0], CARRIL[1], 81], labels=['banda_1', 'centro', 'banda_2']).reindex(f.index)
        # R6-5: pases propios antes de la primera acción en el último tercio, y segundos desde el inicio de la posesión
        o2 = own.assign(u3=own['loc_x'] >= X_U3)
        o2['u3_antes'] = o2.groupby(['match_id', 'possession'])['u3'].cumsum() - o2['u3']
        pre = o2[o2['u3_antes'] == 0]
        f['pases_hasta'] = pre[(pre['type.name'] == 'Pass') & ~pre['u3']].groupby(['match_id', 'possession']).size().reindex(f.index).fillna(0)
        t_u3 = o2[o2['u3']].groupby(['match_id', 'possession'])['t'].first()
        f['seg_hasta'] = (t_u3 - g['t'].first()).reindex(f.index)
        # Banda de cada evento (para la verificación de izquierda/derecha)
        if temp == ENT[-1]:                                   # muestra de partidos para verificar las bandas (posición del jugador)
            for mid in rng.choice(sorted(own['match_id'].unique()), N_MUESTRA_BANDAS, replace=False):
                r = os.path.join(DIR_EV, f'{mid}.parquet')
                if not os.path.exists(r) or 'position.name' not in pq.read_schema(r).names: continue
                ps = pd.read_parquet(r, columns=['index', 'position.name']).rename(columns={'position.name': 'posicion'})
                o_ = own[own['match_id'] == mid][['index', 'loc_y']].merge(ps, on='index')
                lat.append(o_[o_['posicion'].isin(['Left Back', 'Right Back', 'Left Wing', 'Right Wing'])][['posicion', 'loc_y']])
        Pz = Pz.join(f, on=['match_id', 'possession'], how='inner')
        Pz['pos_local'] = Pz['equipo'] == Pz['match_id'].map(loc['local'])
        Pz['seq'] = Pz['max_x'] >= X_U3                                                  # O-5
        secs.append(Pz.assign(temporada=temp))

        def agrega(d, val, den, nombre):
            s = np.where(d['pos_local'] if 'pos_local' in d else d['ev_local'], 1.0, -1.0)
            team = d['equipo'] if 'equipo' in d else d['team.name']
            gg = pd.DataFrame({'match_id': d['match_id'].values, 'team': team.values, 'e5': e5(s * d['m_home'].values),
                               'e5b': e5(s * d['m_home_b'].values), 'cuarto': d['cuarto'].values,
                               'num': np.asarray(val, float), 'den': np.asarray(den, float)})
            gg = gg.groupby(['match_id', 'team', 'e5', 'e5b', 'cuarto'], as_index=False)[['num', 'den']].sum()
            gg['metrica'] = nombre
            return gg

        a = Pz[Pz['x_ini'] < X_MED]; piezas.append(agrega(a, 100.0 * (a['max_x'] >= X_MED), 1.0, 'paso_medio'))
        b = Pz[Pz['max_x'] >= X_MED]; piezas.append(agrega(b, 100.0 * (b['max_x'] >= X_U3), 1.0, 'paso_ultimo'))
        s_ = Pz[Pz['seq']]
        piezas += [agrega(s_, 100.0 * s_['area'], 1.0, 'toque_area'), agrega(s_, 100.0 * s_['tiro'], 1.0, 'tiro_por_secuencia'),
                   agrega(s_, 100.0 * s_['centro'], 1.0, 'centros'),
                   agrega(s_[s_['carril'].notna()], 100.0 * (s_.loc[s_['carril'].notna(), 'carril'] == 'centro'), 1.0, 'entrada_central'),
                   agrega(s_, s_['pases_hasta'], 1.0, 'pases_hasta_ultimo'),
                   agrega(s_[s_['seg_hasta'].notna()], s_['seg_hasta'].dropna(), 1.0, 'seg_hasta_ultimo')]
        bb = s_[s_['carril'].isin(['banda_1', 'banda_2'])]
        piezas.append(agrega(bb, 100.0 * (bb['carril'] == 'banda_1'), 1.0, 'banda_1_raw'))      # se etiqueta tras verificar
        # --- Tiros de juego abierto (sin balón parado ni penales): xG por tiro y % en el área
        sh = E[(tn == 'Shot') & (E['shot.type.name'] == 'Open Play') & (E['fase'] != 'bp') & E['shot.statsbomb_xg'].notna()].copy()
        piezas += [agrega(sh, 100.0 * sh['shot.statsbomb_xg'], 1.0, 'xg_por_tiro'),
                   agrega(sh, 100.0 * en_area(sh['loc_x'], sh['loc_y']), 1.0, 'tiro_area')]
        # --- Centros de juego abierto: % por arriba (altura del pase)
        cc = E[(tn == 'Pass') & abierto & E['pass.cross'].eq(True) & E['altura'].notna()]
        piezas.append(agrega(cc, 100.0 * (cc['altura'] == 'High Pass'), 1.0, 'centro_alto'))
        # R6-8: centro preparado = el pase propio anterior (misma posesión) fue hacia atrás, en el último tercio y en la misma banda
        pp_ = E[(tn == 'Pass') & abierto & (E['team.name'] == E['possession_team.name'])].copy()
        gp = pp_.groupby(['match_id', 'possession'])
        for c_ in ['loc_x', 'loc_y', 'pass_end_x']: pp_['ant_' + c_] = gp[c_].shift(1)
        carr = lambda y: np.where(y < CARRIL[0], 0, np.where(y > CARRIL[1], 2, 1))
        cz = pp_[pp_['pass.cross'].eq(True)].copy()
        base_prep = cz['ant_loc_x'].ge(X_U3) & (carr(cz['ant_loc_y']) == carr(cz['loc_y'])) & (carr(cz['loc_y']) != 1)
        cz['prep'] = base_prep & (cz['ant_pass_end_x'] <= cz['ant_loc_x'] - ATRAS_YD)
        cz['prep0'] = base_prep & (cz['ant_pass_end_x'] < cz['ant_loc_x'])
        cz['ev_local'] = cz['team.name'] == cz['local']
        piezas.append(agrega(cz, 100.0 * cz['prep'], 1.0, 'centro_preparado'))
        pre_c.append(cz[['prep', 'prep0']])
        # --- Tabla de tiros para lo descriptivo (zona, origen del pase clave, conversión, ejecución)
        pases = E.loc[tn == 'Pass', ['id', 'pass.cross', 'altura', 'pase_atras', 'loc_x', 'loc_y']].dropna(subset=['id']).set_index('id')
        kp = sh['pase_clave']
        sh['kp_x'], sh['kp_y'] = kp.map(pases['loc_x']), kp.map(pases['loc_y'])
        con_kp = sh['kp_x'].notna()
        piezas.append(agrega(sh[con_kp], 100.0 * en_area(sh.loc[con_kp, 'kp_x'], sh.loc[con_kp, 'kp_y']), 1.0, 'pase_extra'))
        sh['origen'] = np.select(
            [kp.isna(), kp.map(pases['pass.cross']).eq(True) & kp.map(pases['altura']).eq('High Pass'),
             kp.map(pases['pass.cross']).eq(True), kp.map(pases['pase_atras']).eq(True)],
            ['sin pase clave', 'centro alto', 'centro raso o bajo', 'pase atrás'], 'otro pase')
        x_, y_ = sh['loc_x'], sh['loc_y']
        sh['zona'] = np.select(
            [(x_ >= CHICA_X) & y_.between(*CHICA_Y), en_area(x_, y_) & y_.between(*CHICA_Y), en_area(x_, y_),
             y_.between(*AREA_Y)], ['1 área chica', '2 área central', '3 área lateral', '4 frontal'], '5 fuera, costado')
        sh['gol'] = sh['shot.outcome.name'] == 'Goal'
        tiros.append(sh[['match_id', 'team.name', 'm_home', 'ev_local', 'zona', 'origen', 'shot.statsbomb_xg', 'xg_ejec', 'gol',
                         'loc_x', 'loc_y', 'kp_x', 'kp_y']]
                     .rename(columns={'team.name': 'team', 'shot.statsbomb_xg': 'xg'}).assign(temporada=temp))
        # --- Conteos para el chequeo de coherencia contra R2a (mismas definiciones de posesión y secuencia)
        chk.append(Pz.assign(medio=Pz['max_x'] >= X_MED, seq_centro=Pz['seq'] & Pz['centro'])
                   .groupby(['match_id', 'equipo']).agg(n_pos_op=('seq', 'size'), n_alc_medio=('medio', 'sum'),
                                                         n_alc_u3=('seq', 'sum'), n_seq_centro=('seq_centro', 'sum')).reset_index())
        print(f'  {temp}: {E.match_id.nunique()} partidos · {len(Pz):,} posesiones de juego abierto · {len(sh):,} tiros ({time.time() - t0:.0f} s)')
    TR = pd.concat(piezas, ignore_index=True).groupby(['metrica', 'match_id', 'team', 'e5', 'e5b', 'cuarto'], as_index=False)[['num', 'den']].sum()
    TI = pd.concat(tiros, ignore_index=True); SQ = pd.concat(secs, ignore_index=True); CK = pd.concat(chk, ignore_index=True)
    LAT = pd.concat(lat, ignore_index=True) if lat else pd.DataFrame(columns=['posicion', 'loc_y'])
    PREP = pd.concat(pre_c, ignore_index=True)
    LAT.to_parquet(ruta_lat, index=False); PREP.to_parquet(ruta_prep, index=False)
    SQ['carril'] = SQ['carril'].astype(str)
    for d_, r_ in [(TR, ruta_tr), (TI, ruta_ti), (SQ, ruta_sq), (CK, ruta_ck)]: d_.to_parquet(r_, index=False)
    if faltaron: print('Columnas que no estaban en R1 y se leyeron de los crudos:', sorted(faltaron))

# ---------- R6-6: ¿la banda de y baja es la izquierda del atacante? Se verifica con laterales y extremos ----------
print('\n--- (0) Verificación de bandas: y media de las acciones por posición (campo de 0 a 80; ataque hacia x = 120) ---')
if len(LAT):
    vy = LAT.groupby('posicion')['loc_y'].agg(['size', 'mean']).round(1)
    print(vy.to_string())
    izq = LAT.loc[LAT.posicion.str.startswith('Left'), 'loc_y'].mean(); der = LAT.loc[LAT.posicion.str.startswith('Right'), 'loc_y'].mean()
    BANDA1 = 'izquierda' if izq < der else 'derecha'
    print(f'→ la banda de y baja es la {BANDA1.upper()} del atacante (izquierdos {izq:.1f} frente a derechos {der:.1f})')
else:
    BANDA1 = None; print('Sin posiciones en los eventos: el rasgo de banda queda como "banda de y baja" sin etiqueta')
nombre_banda = f'banda_{BANDA1}' if BANDA1 == 'izquierda' else ('banda_izquierda' if BANDA1 else 'banda_y_baja')
if BANDA1 == 'derecha':                                       # se invierte para que el rasgo sea siempre "% por la izquierda"
    m_ = TR.metrica == 'banda_1_raw'; TR.loc[m_, 'num'] = 100.0 * TR.loc[m_, 'den'] - TR.loc[m_, 'num']
TR.loc[TR.metrica == 'banda_1_raw', 'metrica'] = nombre_banda
TR.to_parquet(ruta_tr, index=False)

METRICAS = ['paso_medio', 'paso_ultimo', 'toque_area', 'tiro_por_secuencia', 'pases_hasta_ultimo', 'xg_por_tiro', 'tiro_area',
            'pase_extra', 'centros', 'centro_alto', 'centro_preparado', 'entrada_central', nombre_banda]
SENS = ['seg_hasta_ultimo']
UNIDAD = {'xg_por_tiro': '% de gol esperado por tiro', 'pases_hasta_ultimo': 'pases por secuencia', 'seg_hasta_ultimo': 'segundos'}

# ---------- 4. Chequeos ----------
print('\n--- (a) Faltantes en columnas extra (si un campo falta en todos los tiros o centros, su métrica no se puede usar) ---')
print({'tiros sin xG de ejecución %': round(100 * TI['xg_ejec'].isna().mean(), 1),
       'tiros con origen "sin pase clave" %': round(100 * (TI['origen'] == 'sin pase clave').mean(), 1),
       'tramos de centro_alto': int((TR.metrica == 'centro_alto').sum())})

print('\n--- (b) Coherencia contra R2a (sumas por equipo-partido; cociente 1.000 = idéntico) ---')
comp = pd.concat([pd.read_parquet(os.path.join(FW, f"componentes_{t.replace('/', '-')}.parquet")) for t in ENT])
ref = comp.groupby(['match_id', 'equipo'])[['n_pos_op', 'n_alc_medio', 'n_alc_u3', 'n_seq_centro']].sum()
mio = CK.set_index(['match_id', 'equipo'])
filas = []
for c in ref.columns:
    j = mio[[c]].join(ref[[c]], rsuffix='_r2a', how='inner')
    filas.append(dict(componente=c, filas=len(j), r6a=j[c].sum(), r2a=j[c + '_r2a'].sum(), cociente=j[c].sum() / j[c + '_r2a'].sum(),
                      iguales_pct=100 * np.isclose(j[c], j[c + '_r2a']).mean()))
print(pd.DataFrame(filas).round(3).to_string(index=False))
print('  Nota: n_pos_op difiere a propósito: R6a deja fuera las posesiones sin ninguna acción propia de juego abierto con'
      ' ubicación (no pueden avanzar). Los otros tres componentes deben coincidir al 100%.')

print('\n--- (c) Valores de liga por temporada (suma entre suma) ---')
L = TR.merge(Mm[['match_id', 'temporada']], on='match_id').groupby(['temporada', 'metrica'])[['num', 'den']].sum()
print((L['num'] / L['den']).unstack('metrica')[METRICAS + SENS].round(2).T.to_string())
print(f"Centro preparado con 'hacia atrás' ≥ {ATRAS_YD} yd: {100 * PREP['prep'].mean():.1f}% · con 0 yd (sensibilidad): {100 * PREP['prep0'].mean():.1f}%")
print('Eventos por equipo-partido (mediana):',
      TR.groupby(['metrica', 'match_id', 'team'])['den'].sum().groupby('metrica').median().reindex(METRICAS).round(1).to_dict())

# ---------- 5. Confiabilidad CRUDA preliminar (diagnóstico; la oficial sale de R6r, ajustada como en R5) ----------
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
et = ptp[ptp['temporada'].isin(ENT)][['match_id', 'team', 'tec_p', 'etapa']].copy()
et['etapa_id'] = et['team'] + ' · ' + et['etapa'].astype(str)
n_et = et.groupby('etapa_id')['match_id'].nunique()
E15 = set(n_et[n_et >= ETAPA_MIN].index)
print(f'\n--- (d) Confiabilidad CRUDA entre mitades (etapas de {ETAPA_MIN}+ partidos, 200 repartos; solo diagnóstico) ---')
filas = []
for met in METRICAS + SENS:
    d = TR[TR.metrica == met].groupby(['match_id', 'team'], as_index=False)[['num', 'den']].sum().merge(et, on=['match_id', 'team'])
    d = d[d.etapa_id.isin(E15)]
    rs = []
    for _ in range(200):
        h = d.assign(r=rng.random(len(d))).sort_values(['etapa_id', 'r'])
        h['mitad'] = h.groupby('etapa_id').cumcount() % 2
        p = h.groupby(['etapa_id', 'mitad'])[['num', 'den']].sum()
        p = (p['num'] / p['den']).unstack()
        r = p[0].corr(p[1]); rs.append(2 * r / (1 + r))
    pm = d.groupby('etapa_id')[['num', 'den']].sum(); pm = pm['num'] / pm['den']
    filas.append(dict(metrica=met, etapas=len(pm), confiabilidad_cruda=np.median(rs), p5=np.percentile(rs, 5), p95=np.percentile(rs, 95),
                      dispersion_entre_etapas=pm.std(), eventos_por_partido=d['den'].sum() / len(d)))
print(pd.DataFrame(filas).round(2).to_string(index=False))

# ---------- 6. Descriptivos: América por etapa y liga ----------
TI = TI.merge(et[['match_id', 'team', 'etapa_id', 'tec_p']], on=['match_id', 'team'], how='left')
am = sorted({e for e in et.etapa_id.unique() if e.startswith('América')}, key=lambda e: et.loc[et.etapa_id == e, 'match_id'].min())
etiq = lambda e: ('Liga' if e == 'Liga' else et.loc[et.etapa_id == e, 'tec_p'].iloc[0].split()[0] + ' ' +
                  et.loc[et.etapa_id == e, 'tec_p'].iloc[0].split()[-1 if len(et.loc[et.etapa_id == e, 'tec_p'].iloc[0].split()) < 3 else -2])
def tabla_pct(col):
    t = pd.crosstab(TI['etapa_id'], TI[col], normalize='index') * 100
    lg = (TI[col].value_counts(normalize=True) * 100).rename('Liga')
    t = pd.concat([t.loc[[e for e in am if e in t.index]], lg.to_frame().T])
    t.index = [etiq(e) for e in t.index]; return t.round(1)
print('\n--- (e) Mapa de remate: % de tiros de juego abierto por zona ---')
print(tabla_pct('zona').to_string())
print('\n--- (f) Origen del tiro (pase clave): % de tiros ---')
print(tabla_pct('origen').to_string())
print('\n--- (g) Zona 2 (área central) según el origen: ¿por arriba o por abajo? (% de los tiros de la zona 2) ---')
z2 = TI[TI['zona'] == '2 área central']
t = pd.crosstab(z2['etapa_id'], z2['origen'], normalize='index') * 100
t = pd.concat([t.loc[[e for e in am if e in t.index]], (z2['origen'].value_counts(normalize=True) * 100).rename('Liga').to_frame().T])
t.index = [etiq(e) for e in t.index]; print(t.round(1).to_string())

ag = TI.groupby('etapa_id').agg(partidos=('match_id', 'nunique'), tiros=('xg', 'size'), xg=('xg', 'sum'), goles=('gol', 'sum'),
                                xg_ejec=('xg_ejec', 'sum'), n_ejec=('xg_ejec', 'count'))
ag['xg_por_partido'] = ag['xg'] / ag['partidos']
ag['goles_menos_xg'] = ag['goles'] - ag['xg']
ag['ejecucion_por_100_tiros'] = 100 * (ag['xg_ejec'] - TI[TI.xg_ejec.notna()].groupby('etapa_id')['xg'].sum()) / ag['n_ejec']
riv = TI.merge(Mm[['match_id', 'local', 'visita']], on='match_id')
riv['rival'] = np.where(riv['team'] == riv['local'], riv['visita'], riv['local'])
contra = riv.merge(et.rename(columns={'team': 'rival', 'etapa_id': 'etapa_rival'})[['match_id', 'rival', 'etapa_rival']], on=['match_id', 'rival'])
ag['xg_contra_por_partido'] = contra.groupby('etapa_rival')['xg'].sum().reindex(ag.index) / ag['partidos']
print('\n--- (h) Contexto (no son rasgos): xG de juego abierto a favor y en contra por partido, definición y ejecución ---')
print('  goles_menos_xg = definición (suerte + calidad del tirador) · ejecucion_por_100_tiros = xG de ejecución − xG de situación'
      ' por cada 100 tiros (colocación y potencia; mide al tirador, no al técnico, salvo por quién juega)')
v = ag.loc[[e for e in am if e in ag.index], ['partidos', 'tiros', 'xg_por_partido', 'xg_contra_por_partido', 'goles_menos_xg', 'ejecucion_por_100_tiros']]
v.index = [etiq(e) for e in v.index]
print(v.round(2).to_string())
print('Liga (etapas de 15+), mediana:', ag.loc[ag.index.isin(E15), ['xg_por_partido', 'goles_menos_xg', 'ejecucion_por_100_tiros']].median().round(2).to_dict())

print('\n--- (i) Embudo por carril de entrada al último tercio (% de secuencias que acaban en tiro o tocan el área) ---')
s_ = SQ[SQ['seq'] & SQ['carril'].isin(['banda_1', 'centro', 'banda_2'])].copy()
lab = {'centro': 'centro', 'banda_1': f'banda {BANDA1 or "y baja"}', 'banda_2': f'banda {("derecha" if BANDA1 == "izquierda" else "izquierda") if BANDA1 else "y alta"}'}
s_['carril2'] = s_['carril'].map(lab)
print(s_.groupby('carril2').agg(secuencias=('seq', 'size'), pct_secuencias=('seq', lambda v: 100 * len(v) / len(s_)),
                                toque_area=('area', lambda v: 100 * v.mean()), tiro=('tiro', lambda v: 100 * v.mean())).round(1).to_string())

# ---------- R6-7 C: ganancia del pase extra (descriptivo; modelo propio de xG por ubicación) ----------
print(f'\n--- (j) Ganancia del pase extra: xG del tiro − xG de haber tirado desde donde salió el pase (pases desde x ≥ {ZONA_TIRO_X}) ---')
def dist_ang(x, y):
    dx, dy = 120 - x, 40 - y
    d = np.hypot(dx, dy)
    a = np.abs(np.arctan2(8 * dx, dx ** 2 + dy ** 2 - 16))         # ángulo de apertura del arco (postes en y = 36 y 44)
    return d, a
d_, a_ = dist_ang(TI['loc_x'], TI['loc_y'])
TI['bd'] = np.minimum(d_ // 2, 20); TI['ba'] = np.minimum(np.degrees(a_) // 5, 18)
grid = TI.groupby(['bd', 'ba'])['xg'].agg(['mean', 'size'])
grid = grid[grid['size'] >= 20]['mean']
print(f'  Modelo de xG por ubicación: {len(grid)} celdas (2 yd × 5°) con 20+ tiros de juego abierto de la liga')
k_ = TI[TI['kp_x'].notna() & (TI['kp_x'] >= ZONA_TIRO_X)].copy()
dk, ak = dist_ang(k_['kp_x'], k_['kp_y'])
k_['xg_desde_pase'] = pd.Series(list(zip(np.minimum(dk // 2, 20), np.minimum(np.degrees(ak) // 5, 18))), index=k_.index).map(grid)
k_ = k_[k_['xg_desde_pase'].notna()]
k_['ganancia'] = k_['xg'] - k_['xg_desde_pase']
gx = k_.groupby('etapa_id').agg(tiros=('ganancia', 'size'), ganancia_media=('ganancia', 'mean'), pct_mejora=('ganancia', lambda v: 100 * (v > 0).mean()))
vv = gx.loc[[e for e in am if e in gx.index]].copy(); vv.index = [etiq(e) for e in vv.index]
vv.loc['Liga'] = [len(k_), k_['ganancia'].mean(), 100 * (k_['ganancia'] > 0).mean()]
print(vv.round(3).to_string())
print('  ganancia_media en xG por tiro asistido (+0.05 = el pase sumó 5 puntos de probabilidad de gol) · pct_mejora = % de pases que dejaron un tiro mejor')
print('  Hipótesis de Arturo (registro): el pase extra es más común en equipos que centran por abajo; se cruza con centro_alto en R6r')

print(f'\nR6a terminó en {time.time() - t_ini:.0f} s. Guardado: fw1/r6_tramos, r6_tiros, r6_secuencias, r6_chequeo (+ r6_extra_* si hizo falta)')

R6r 1 - Correr con el prefijo correspondiente (a, b o c)

In [ ]:
PREFIJO = 'r6c'

In [ ]:
# =====================================================================
# R6r v1.1 · Receta de contexto (R4) y confiabilidad con relojes (R5) para los rasgos de R6a y R6b
# v1.1: PREFIJO = 'r6' (embudo, R6a), 'r6b' (transición, conducciones y encare) o 'r6c' (OBV); bloque (g) por estado de marcador
# Framework v1.2: C (método de R4 v2.1) y N-1, N-2, N-5, P-5, P-21 (método de R5) · Sesión 8 · 2026-10-14
# Antes: celda 1 (Drive), R0 y R6a o R6b (lee fw1/{PREFIJO}_tramos.parquet). Mismo código que R4b y R5, aplicado a otras métricas:
# no recalcula ni sobrescribe nada de R4 ni de R5. Datos: SOLO entrenamiento. Tarda ~5-10 min.
# =====================================================================

# [R6r-1 v1.1] Receta, residuos y perfiles (copia de R4b sin árbitro: ninguna métrica del embudo cuenta faltas)
import os, json, time, warnings
import numpy as np
import pandas as pd
from scipy.stats import rankdata
warnings.filterwarnings('ignore', category=RuntimeWarning)

PREFIJO = globals().get('PREFIJO', 'r6')             # 'r6' = R6a · 'r6b' = R6b
SEMILLA = 2026
rng = np.random.default_rng(SEMILLA)
BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026/')
DP = os.path.join(BASE, 'datos_parquet'); FW = os.path.join(DP, 'fw1'); SAL = os.path.join(BASE, 'salidas')
os.makedirs(SAL, exist_ok=True)
t_ini = time.time()
pd.set_option('display.width', 250)
with open(os.path.join(FW, 'parametros_fw1.json')) as f:
    PRM = json.load(f)
PP = PRM.get('P', {})
ENT = PRM['temporadas']['entrenamiento']
ETAPA_MIN = PP.get('etapa_min', 15)

Mm = pd.read_parquet(os.path.join(FW, 'r4_partidos.parquet'))
TR = pd.read_parquet(os.path.join(FW, f'{PREFIJO}_tramos.parquet'))
MIN = pd.read_parquet(os.path.join(FW, 'r4_minutos_v21.parquet'))
assert set(Mm['temporada']) <= set(ENT), 'R6r solo usa entrenamiento'
ORDEN = ['paso_medio', 'paso_ultimo', 'toque_area', 'tiro_por_secuencia', 'pases_hasta_ultimo', 'xg_por_tiro', 'tiro_area', 'pase_extra',
         'centros', 'centro_alto', 'centro_preparado', 'entrada_central', 'banda_izquierda', 'seg_hasta_ultimo',
         'contragolpe_propio', 'jugadores_contra', 'contra_a_area', 'contra_a_tiro', 'mezcla_transicion', 'xg_transicion',
         'mezcla_progresion', 'conduccion_larga', 'libertad_encarar', 'exito_regate', 'centro_tras_regate', 'pase_pico',
         'reparto_conduccion', 'reparto_regate', 'obv_ultimo_tercio', 'obv_transicion_pct', 'tasa_pase_area']
METRICAS = [m for m in ORDEN if m in set(TR.metrica)] + sorted(set(TR.metrica) - set(ORDEN))
print(f'Métricas de {PREFIJO}_tramos encontradas:', len(METRICAS), METRICAS)

lado = pd.concat([Mm.assign(team=Mm.local, rival=Mm.visita, local_=1), Mm.assign(team=Mm.visita, rival=Mm.local, local_=0)])
CTX = lado[['match_id', 'team', 'rival', 'local_', 'torneo', 'temporada']].copy()
elo = pd.read_parquet(os.path.join(FW, 'elo_equipo_partido_e2.parquet'))[['match_id', 'team', 'elo_pre', 'elo_rival_pre']]
CTX = CTX.merge(elo, on=['match_id', 'team'], how='left')
CTX['elo_c'] = (CTX['elo_rival_pre'] - 1500) / 100

TM = TR.groupby(['metrica', 'match_id', 'team'], as_index=False)[['num', 'den']].sum()
TM = TM.merge(CTX[['match_id', 'team', 'rival', 'torneo']], on=['match_id', 'team'])
conc = TM.groupby(['metrica', 'torneo', 'rival'])[['num', 'den']].sum().rename(columns={'num': 'cn', 'den': 'cd'})
liga_t = TM.groupby(['metrica', 'torneo'])[['num', 'den']].sum()
TM = TM.join(conc, on=['metrica', 'torneo', 'rival']).join(liga_t.rename(columns={'num': 'ln', 'den': 'ld'}), on=['metrica', 'torneo'])
TM['estilo'] = ((TM['cn'] - TM['num']) / (TM['cd'] - TM['den']) - TM['ln'] / TM['ld']).replace([np.inf, -np.inf], np.nan).fillna(0.0)

ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp_e = ptp[ptp['temporada'].isin(ENT)].copy()
et = ptp_e[['match_id', 'team', 'tec_p', 'etapa', 'fecha']].copy()
et['etapa_id'] = et['team'] + ' · ' + et['etapa'].astype(str)
n_et = et.groupby('etapa_id')['match_id'].nunique()
W3 = MIN.assign(e3=np.sign(MIN.e5)).groupby('e3')['minutos'].sum(); W3 = (W3 / W3.sum()).reindex([-1, 0, 1]).fillna(0)

def ajustar(met):
    d = TR[TR.metrica == met].groupby(['match_id', 'team', 'e5', 'cuarto'], as_index=False)[['num', 'den']].sum()
    d = d[d['den'] > 0].rename(columns={'e5': 'e'})
    d = d.merge(CTX, on=['match_id', 'team']).merge(TM[TM.metrica == met][['match_id', 'team', 'estilo']], on=['match_id', 'team'])
    d = d.dropna(subset=['elo_c']).reset_index(drop=True)
    y = (d['num'] / d['den']).values; w = d['den'].values
    cols = {'intercepto': np.ones(len(d))}
    for k in [-2, -1, 1, 2]: cols[f'e{k:+d}'] = (d['e'] == k).astype(float).values
    for c in [2, 3, 4]: cols[f'C{c}'] = (d['cuarto'] == c).astype(float).values
    cols['local'] = d['local_'].astype(float).values
    cols['elo_rival_100'] = d['elo_c'].values
    cols['estilo_rival'] = d['estilo'].values
    for t in sorted(d['torneo'].unique())[1:]: cols[f'T_{t}'] = (d['torneo'] == t).astype(float).values
    X = np.column_stack(list(cols.values())); sw = np.sqrt(w)
    beta = np.linalg.lstsq(X * sw[:, None], y * sw, rcond=None)[0]
    e = y - X @ beta
    s2 = (w * e ** 2).sum() / (len(y) - X.shape[1])
    c = pd.DataFrame({'e': d['e'], 'q': d['cuarto'], 'we': w * e, 'w': w}).groupby(['e', 'q'])[['we', 'w']].sum()
    c['r'] = c['we'] / c['w']; c['se2'] = s2 / c['w']
    wi = 1 / c['se2']; gl_ = max(1, len(c) - 8)
    tau2 = max(0.0, ((wi * c['r'] ** 2).sum() - gl_) / (wi.sum() - (wi ** 2).sum() / wi.sum()))
    c['celda'] = (tau2 / (tau2 + c['se2']) if tau2 > 0 else 0.0) * c['r']
    adj = pd.Series(list(zip(d['e'], d['cuarto']))).map(c['celda']).values
    yhat = X @ beta + adj; e = y - yhat
    r2 = 1 - (w * e ** 2).sum() / (w * (y - np.average(y, weights=w)) ** 2).sum()
    d['y'], d['w'], d['res'], d['yhat'] = y, w, e, yhat
    return d, dict(zip(cols.keys(), beta)), dict(r2=r2, s2=s2, media=np.average(y, weights=w))

def perfiles(d, s2):
    d = d.merge(et[['match_id', 'team', 'etapa_id', 'tec_p']], on=['match_id', 'team'])
    d['e3'] = np.sign(d['e']).astype(int)
    a = d.assign(we=d.w * d.res).groupby(['etapa_id', 'e3'])[['we', 'w']].sum()
    a['r'] = a['we'] / a['w']
    tot = d.assign(we=d.w * d.res).groupby('etapa_id')[['we', 'w']].sum()
    a['rbar'] = (tot['we'] / tot['w']).reindex(a.index.get_level_values(0)).values
    a['se2'] = s2 / a['w']
    dd = a['r'] - a['rbar']
    wi = 1 / a['se2']; gl_ = max(1, len(a) - a.index.get_level_values(0).nunique())
    tau2 = max(0.0, ((wi * dd ** 2).sum() - gl_) / (wi.sum() - (wi ** 2).sum() / wi.sum()))
    a['k'] = tau2 / (tau2 + a['se2']) if tau2 > 0 else 0.0
    a['r_star'] = a['rbar'] + a['k'] * dd
    piv = a['r_star'].unstack()
    rbar = tot['we'] / tot['w']
    for s in [-1, 0, 1]:
        if s not in piv: piv[s] = np.nan
        piv[s] = piv[s].fillna(rbar)
    perfil = sum(W3.get(s, 0) * piv[s] for s in [-1, 0, 1])
    sg = d.groupby('etapa_id')[['num', 'den']].sum(); crudo = sg['num'] / sg['den']
    out = pd.DataFrame({'perfil': perfil, 'crudo': crudo, 'abajo': piv[-1], 'empate': piv[0], 'arriba': piv[1], 'tau_estado': np.sqrt(tau2)})
    out['tecnico'] = d.groupby('etapa_id')['tec_p'].first(); out['partidos'] = n_et.reindex(out.index)
    return out

res, coefs, perf = {}, [], {}
for met in METRICAS:
    d, coef, info = ajustar(met)
    res[met] = d; perf[met] = perfiles(d, info['s2'])
    coefs.append({'metrica': met, 'filas': len(d), 'media_liga': info['media'], 'R2': info['r2'], **coef,
                  'tau_estado_etapa': perf[met]['tau_estado'].iloc[0]})
CO = pd.DataFrame(coefs)
print('\n--- (a) Receta por métrica: efectos genéricos (misma unidad que la métrica; e±k frente al empate) ---')
print(CO.reindex(columns=['metrica', 'filas', 'media_liga', 'R2', 'e-2', 'e-1', 'e+1', 'e+2', 'C4', 'local', 'elo_rival_100',
                          'estilo_rival']).round(2).to_string(index=False))
CO.to_csv(os.path.join(SAL, f'{PREFIJO}_coeficientes.csv'), index=False)
pd.concat([d.assign(metrica=m) for m, d in res.items()])[['metrica', 'match_id', 'team', 'e', 'cuarto', 'num', 'den', 'y', 'yhat', 'res']] \
    .to_parquet(os.path.join(FW, f'{PREFIJO}_residuos.parquet'), index=False)
pd.concat([p.assign(metrica=m) for m, p in perf.items()]).reset_index().rename(columns={'index': 'etapa_id'}) \
    .to_parquet(os.path.join(FW, f'{PREFIJO}_perfiles_etapa.parquet'), index=False)
print(f'R6r-1 terminó en {time.time() - t_ini:.0f} s.')


R6r - 2

In [ ]:
# [R6r-2 v1.1] Confiabilidad ajustada, relojes, implantación y redundancia (copia de R5)
t_ini = time.time()
REPS, BOOT, R_SUB = 200, 500, 50
NUCLEO, SECUND = 0.7, 0.5
OBJ = NUCLEO / (1 - NUCLEO)
KS, PLACEBOS = [3, 5, 8], 200
P_REG = 19
RS = pd.read_parquet(os.path.join(FW, f'{PREFIJO}_residuos.parquet'))
PE = pd.read_parquet(os.path.join(FW, f'{PREFIJO}_perfiles_etapa.parquet'))
ETAPAS = sorted(et['etapa_id'].unique()); NS = len(ETAPAS); COD = {e: i for i, e in enumerate(ETAPAS)}
n_arr = et.groupby('etapa_id')['match_id'].nunique().reindex(ETAPAS).values
tec = et.groupby('etapa_id')['tec_p'].first().reindex(ETAPAS)
W3v = W3.reindex([-1, 0, 1]).fillna(0).values
ini = et.groupby(['team', 'etapa_id'])['fecha'].min().reset_index().sort_values(['team', 'fecha'])
ini['orden'] = ini.groupby('team').cumcount()
OBS = np.isin(ETAPAS, ini.loc[ini['orden'] > 0, 'etapa_id'])
COLS = ['we_m', 'we_0', 'we_p', 'w_m', 'w_0', 'w_p', 'num', 'den']

def tabla_partidos(met):
    d = RS[RS.metrica == met]
    s2 = float((d['den'] * d['res'] ** 2).sum() / (len(d) - P_REG))
    d = d.merge(et[['match_id', 'team', 'etapa_id', 'fecha']], on=['match_id', 'team'])
    d = d.assign(e3=np.sign(d['e']).astype(int), we=d['den'] * d['res'])
    k = ['etapa_id', 'match_id', 'team', 'fecha']
    g = d.groupby(k + ['e3'])[['we', 'den']].sum().unstack('e3', fill_value=0)
    for c in ['we', 'den']:
        for s in [-1, 0, 1]:
            if (c, s) not in g.columns: g[(c, s)] = 0.0
    tot = d.groupby(k)[['num', 'den']].sum()
    M = pd.DataFrame({'we_m': g[('we', -1)], 'we_0': g[('we', 0)], 'we_p': g[('we', 1)],
                      'w_m': g[('den', -1)], 'w_0': g[('den', 0)], 'w_p': g[('den', 1)],
                      'num': tot['num'], 'den': tot['den']}).reset_index()
    M['cod'] = M['etapa_id'].map(COD)
    return M.sort_values(['cod', 'fecha']).reset_index(drop=True), s2

def perfil_sub(A, cod, mask, tau2, s2):
    c = cod[mask]
    S = {k: np.bincount(c, weights=A[k][mask], minlength=NS) for k in COLS}
    we = np.column_stack([S['we_m'], S['we_0'], S['we_p']]); w = np.column_stack([S['w_m'], S['w_0'], S['w_p']])
    with np.errstate(divide='ignore', invalid='ignore'):
        rbar = we.sum(1) / w.sum(1)
        k = tau2 / (tau2 + s2 / w) if tau2 > 0 else np.zeros_like(w)
        rstar = np.where(w > 0, rbar[:, None] + k * (we / w - rbar[:, None]), rbar[:, None])
        return rstar @ W3v, S['num'] / S['den']

def corr(a, b, rango=False):
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 5: return np.nan
    a, b = a[ok], b[ok]
    if rango: a, b = rankdata(a), rankdata(b)
    return np.corrcoef(a, b)[0, 1]
sb = lambda r: 2 * r / (1 + r)
def pos_en_etapa(orden, cod, start):
    p = np.empty(len(cod), int); p[orden] = np.arange(len(cod)) - start[cod[orden]]; return p
def senal_ruido(a, b, inv_n):
    ac = a - np.nanmean(a, 1, keepdims=True); bc = b - np.nanmean(b, 1, keepdims=True)
    n_ok = np.isfinite(ac * bc).sum(1)
    return float(np.nanmean(np.nansum(ac * bc, 1) / (n_ok - 1))), float(np.nanmean((a - b) ** 2 / inv_n))

filas, implant, perf_full, contr = [], [], {}, []
for met in METRICAS:
    t0 = time.time()
    M, s2 = tabla_partidos(met)
    tau2_e = float(PE.loc[PE.metrica == met, 'tau_estado'].iloc[0]) ** 2
    A = {k: M[k].values.astype(float) for k in COLS}; cod = M['cod'].values; N = len(cod)
    cnt = np.bincount(cod, minlength=NS); start = np.concatenate([[0], np.cumsum(cnt)[:-1]])
    pos = np.arange(N) - start[cod]
    P = lambda m: perfil_sub(A, cod, m, tau2_e, s2)
    inc = (n_arr >= ETAPA_MIN) & (cnt >= 4)
    pf, cf = P(np.ones(N, bool)); perf_full[met] = pf
    pe = PE[PE.metrica == met].set_index('etapa_id').reindex(ETAPAS)
    AA = np.full((REPS, NS), np.nan); BB = AA.copy(); r_cru = []
    for r_ in range(REPS):
        mA = pos_en_etapa(np.lexsort((rng.random(N), cod)), cod, start) % 2 == 0
        AA[r_], ca = P(mA); BB[r_], cb = P(~mA); r_cru.append(corr(ca[inc], cb[inc]))
    r_p = np.array([corr(AA[i, inc], BB[i, inc]) for i in range(REPS)])
    nA, nB = (cnt + 1) // 2, cnt // 2
    inv_n = np.where(nB > 0, 1 / np.maximum(nA, 1) + 1 / np.maximum(nB, 1), np.nan)
    tau2, sig2 = senal_ruido(AA[:, inc], BB[:, inc], inv_n[inc])
    nst = []; a5, b5, iv = AA[:R_SUB][:, inc], BB[:R_SUB][:, inc], inv_n[inc]; S_ = inc.sum()
    for _ in range(BOOT):
        ix = rng.integers(0, S_, S_); t2, sg = senal_ruido(a5[:, ix], b5[:, ix], iv[ix])
        nst.append(OBJ * sg / t2 if t2 > 0 else np.inf)
    conf = np.median(sb(r_p))
    filas.append(dict(metrica=met, etapas=int(S_), conf=conf, conf_p5=np.percentile(sb(r_p), 5), conf_p95=np.percentile(sb(r_p), 95),
                      conf_crudo=np.median(sb(np.array(r_cru))),
                      franja='núcleo' if conf >= NUCLEO else ('secundario' if conf >= SECUND else 'fuera'),
                      tau=np.sqrt(max(tau2, 0)), sigma_partido=np.sqrt(sig2),
                      reloj=OBJ * sig2 / tau2 if tau2 > 0 else np.inf,
                      reloj_p5=np.quantile(nst, .05, method='nearest'), reloj_p95=np.quantile(nst, .95, method='nearest'),
                      repro_corr=corr(pf[inc], pe['perfil'].values[inc])))
    with np.errstate(divide='ignore', invalid='ignore'):
        k_et = np.where(cnt > 0, tau2 / (tau2 + sig2 / cnt), np.nan) if tau2 > 0 else np.zeros(NS)
    contr.append(pd.DataFrame({'metrica': met, 'etapa_id': ETAPAS, 'tecnico': tec.values, 'partidos': n_arr, 'perfil_r4': pf,
                               'k_p5': k_et, 'perfil_mostrado': k_et * pf, 'liga': CO.set_index('metrica').loc[met, 'media_liga'], 'crudo': cf}))
    for k in KS:
        el = inc & OBS & (cnt >= k + 10); el_row = el[cod]
        f = (pos < k) & el_row
        p1, _ = P(f); p2, _ = P(el_row & ~f)
        d1 = np.abs(p1 - p2)[el]
        m_az = np.empty(PLACEBOS)
        for p in range(PLACEBOS):
            clave = np.where(pos >= k, rng.random(N), 2.0)
            ch = (pos_en_etapa(np.lexsort((clave, cod)), cod, start) < k) & el_row
            q1, _ = P(ch); q2, _ = P(el_row & ~ch)
            m_az[p] = np.nanmean(np.abs(q1 - q2)[el])
        implant.append(dict(metrica=met, k=k, etapas=int(el.sum()), cociente=np.nanmean(d1) / m_az.mean(),
                            p_valor=float((m_az >= np.nanmean(d1)).mean())))
    print(f'  {met}: {time.time() - t0:.0f} s')

R = pd.DataFrame(filas)
print('\n--- (b) Chequeo: el perfil de R6r-2 reproduce el de R6r-1 (corr ≈ 1.000) ---')
print(R[['metrica', 'repro_corr']].round(4).to_string(index=False))
print(f'\n--- (c) Confiabilidad AJUSTADA de la etapa completa (etapas de {ETAPA_MIN}+; P-6: núcleo ≥ 0.7, secundario 0.5-0.7) y reloj ---')
print(R[['metrica', 'etapas', 'conf', 'conf_p5', 'conf_p95', 'conf_crudo', 'franja', 'tau', 'sigma_partido', 'reloj', 'reloj_p5', 'reloj_p95']]
      .round(2).sort_values('conf', ascending=False).to_string(index=False))

I = pd.DataFrame(implant)
print('\n--- (d) Reloj de implantación (primeros k partidos frente a k al azar) y regla C (P-21) ---')
if len(I) and I['etapas'].max() > 0:
    piv = I.pivot_table(index='metrica', columns='k', values=['cociente', 'p_valor'], sort=False)
    reg_c = {m: ((piv.loc[m, 'p_valor'] < 0.05).sum() >= 2) and (piv.loc[m, 'cociente'] > 1).all() for m in piv.index}
    print(piv.round(2).assign(regla_C=pd.Series(reg_c)).to_string())
else:
    print('Sin etapas con inicio visto: no se puede medir la implantación')

inc15 = n_arr >= ETAPA_MIN
P4 = pd.read_parquet(os.path.join(FW, 'r4_perfiles_etapa.parquet'))
for pre_ in ['r6', 'r6b']:                                            # se compara también con los bloques previos de R6
    rp_ = os.path.join(FW, f'{pre_}_perfiles_etapa.parquet')
    if pre_ != PREFIJO and os.path.exists(rp_):
        P4 = pd.concat([P4, pd.read_parquet(rp_)[['etapa_id', 'metrica', 'perfil']]])
p4 = P4.pivot_table(index='etapa_id', columns='metrica', values='perfil').reindex(np.array(ETAPAS)[inc15])
p6 = pd.DataFrame({m: perf_full[m][inc15] for m in METRICAS}, index=np.array(ETAPAS)[inc15])
CM = pd.concat([p6, p4], axis=1).corr()
print('\n--- (e) Redundancia: correlación entre perfiles ajustados (etapas de 15+) ---')
print('Dentro de este bloque:'); print(CM.loc[METRICAS, METRICAS].round(2).to_string())
print('\nFrente a los rasgos ya medidos (R4' + (' y R6a' if PREFIJO != 'r6' else '') + '; |r| ≥ 0.7 sugiere familia):')
print(CM.loc[METRICAS, list(p4.columns)].round(2).to_string())
pares = [(a, b, CM.loc[a, b]) for i, a in enumerate(METRICAS) for b in list(METRICAS[i + 1:]) + list(p4.columns) if abs(CM.loc[a, b]) >= 0.7]
print('Pares con |r| ≥ 0.7:', [(a, b, round(r, 2)) for a, b, r in pares] or 'ninguno')

C = pd.concat(contr, ignore_index=True)
cdf = C[C.etapa_id.str.startswith('América') & (C.partidos > 0)].copy()
cdf['tec'] = cdf['tecnico'].map(lambda s: s.split()[0] + ' ' + (s.split()[-2] if len(s.split()) >= 3 else s.split()[-1]) if isinstance(s, str) else '')
cdf['txt'] = cdf.apply(lambda r: f"{r.liga + r.perfil_mostrado:.1f} ({r.perfil_mostrado:+.1f})", axis=1)
print('\n--- (f) América: valor ajustado y mostrado tras la contracción P-5 (entre paréntesis, desviación frente a la liga) ---')
print(cdf.pivot_table(index='metrica', columns='tec', values='txt', aggfunc='first', sort=False).reindex(METRICAS).to_string())

print('\n--- (g) América por estado de marcador (residuo frente a la receta, contraído hacia el propio técnico; antes de P-5) ---')
pa_ = PE[PE.etapa_id.str.startswith('América')].copy()
pa_['tec'] = pa_['tecnico'].map(lambda s: s.split()[0] + ' ' + (s.split()[-2] if len(s.split()) >= 3 else s.split()[-1]) if isinstance(s, str) else '')
print(pa_.pivot_table(index='metrica', columns='tec', values=['abajo', 'empate', 'arriba'], sort=False).reindex(METRICAS)
      .swaplevel(axis=1).sort_index(axis=1, level=0).round(1).to_string())
R.to_csv(os.path.join(SAL, f'{PREFIJO}_confiabilidad.csv'), index=False); I.to_csv(os.path.join(SAL, f'{PREFIJO}_implantacion.csv'), index=False)
C.to_parquet(os.path.join(FW, f'{PREFIJO}_perfiles_contraidos.parquet'), index=False)
print(f'\nR6r-2 terminó en {time.time() - t_ini:.0f} s. Guardado: salidas/{PREFIJO}_confiabilidad.csv, {PREFIJO}_implantacion.csv, fw1/{PREFIJO}_perfiles_contraidos')

R6b - Encare

In [ ]:
# =====================================================================
# R6b v1.0 · Transición ofensiva, conducciones y encare
# Framework v1.2: O-6, O-7, O-8, M9, M14 (conducciones, S4-23 y S4-28) · Decisiones de la sesión 8: R6-11 a R6-13
# Sesión 8 · 2026-10-14 · Hackathon ISAC 2026, Track 2
# Antes: celda 1 (Drive) y R0. Lee R1 (eventos_, posesiones_), r4_partidos y la caché de crudos de R6a (r6_extra_*).
# La primera vez lee de los crudos el jugador y el resultado del regate (~15-25 min) y los guarda en caché.
# Después: pasar los tramos por la receta y R5 con la celda R6r (poner PREFIJO = 'r6b' antes de correrla).
# Datos: SOLO entrenamiento.
# =====================================================================

#[R6b v1.0] -----------------------------------------------------------
import os, json, time, warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
warnings.filterwarnings('ignore', category=RuntimeWarning)

SEMILLA = 2026
rng = np.random.default_rng(SEMILLA)
BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026/')
DP = os.path.join(BASE, 'datos_parquet'); FW = os.path.join(DP, 'fw1'); SAL = os.path.join(BASE, 'salidas')
DIR_EV = os.path.join(DP, 'liga', 'eventos')
REHACER = False
t_ini = time.time()
pd.set_option('display.width', 250)

with open(os.path.join(FW, 'parametros_fw1.json')) as f:
    PRM = json.load(f)
PP = PRM.get('P', {})
ENT = sorted(PRM['temporadas']['entrenamiento'])
X_MED, X_U3 = PP.get('tercio_medio_x', 40), PP.get('tercio_ultimo_x', 80)
AREA_X, AREA_Y = PP.get('area_x', 102), tuple(PP.get('area_y', (18, 62)))
AVANCE = PP.get('progresivo_avance_yd', 10)
TRANS_S = PP.get('transicion_s', 10)                                  # O-7
TIPOS_CG = set(PRM.get('tipos_contragolpe', ['Ball Receipt*', 'Carry', 'Duel', 'Shot']))   # O-8
CARRIL = (80 / 3, 160 / 3)
CORTE_C1, CORTE_C3 = 25, 70
X_MITAD = 60                                                          # R6-12: jugadores que se suman = acción propia en campo rival
ATRAS_YD = 3                                                          # pase hacia atrás (igual que R6-8)
CONDUCE_YD = 5                                                        # conducción antes de decidir
LARGA_YD = PP.get('conduccion_larga_yd', 30)                          # P-15
REGATE_CENTRO_S = 5                                                   # centro tras regate: ≤ 5 s
PICO = dict(x=(96, 108), dy=8)                                        # pase al pico del área: llega a ≤ 8 yd de la esquina cercana
EXTRA = {'jugador': ['player.id'], 'regate': ['dribble.outcome.name'], 'pase_atras': ['pass.cut_back', 'pass.cut-back']}

def en_area(x, y): return (x >= AREA_X) & (y >= AREA_Y[0]) & (y <= AREA_Y[1])
def e5(m):
    m = np.asarray(m, dtype=float)
    return np.clip(np.sign(m) * np.ceil(np.abs(m) - 1e-9), -2, 2).astype(int)
def progresivo(x0, y0, x1, y1):
    cruza = (((x0 < X_MED) & (x1 >= X_MED)) | ((x0 < X_U3) & (x1 >= X_U3))) & ((x1 - x0) >= AVANCE)
    return cruza | (~en_area(x0, y0) & en_area(x1, y1))

Mm = pd.read_parquet(os.path.join(FW, 'r4_partidos.parquet'))
assert set(Mm['temporada']) <= set(ENT), 'R6b solo usa entrenamiento'
loc = Mm.set_index('match_id'); mids = set(Mm['match_id'])

def leer_extra(temp, cols_r1):
    """Columnas que R1 no guardó: primero de la caché de R6a, luego de la caché propia, al final de los crudos."""
    falt = {k: v for k, v in EXTRA.items() if not any(c in cols_r1 for c in v)}
    if not falt: return None, []
    X = None
    r6a = os.path.join(FW, f"r6_extra_{temp.replace('/', '-')}.parquet")
    if 'pase_atras' in falt and os.path.exists(r6a):
        X = pd.read_parquet(r6a, columns=['match_id', 'index', 'pase_atras'])
    resto = {k: v for k, v in falt.items() if X is None or k not in X.columns}
    if resto:
        ruta = os.path.join(FW, f"r6b_extra_{temp.replace('/', '-')}.parquet")
        if os.path.exists(ruta) and not REHACER:
            Y = pd.read_parquet(ruta)
        else:
            piezas = []
            for mid in Mm.loc[Mm.temporada == temp, 'match_id']:
                r = os.path.join(DIR_EV, f'{mid}.parquet')
                if not os.path.exists(r): continue
                disp = set(pq.read_schema(r).names); usar = {'index': 'index'}
                for k, ops in resto.items():
                    c = next((o for o in ops if o in disp), None)
                    if c: usar[c] = k
                d = pd.read_parquet(r, columns=list(usar)).rename(columns=usar)
                for k in resto:
                    if k not in d: d[k] = np.nan
                d['match_id'] = mid; piezas.append(d)
            Y = pd.concat(piezas, ignore_index=True); Y.to_parquet(ruta, index=False)
            print(f'  {temp}: columnas {sorted(resto)} leídas de {len(piezas)} partidos crudos')
        X = Y if X is None else X.merge(Y, on=['match_id', 'index'], how='outer')
    return X, list(falt)

COLS = ['match_id', 'index', 'period', 'minute', 't', 'type.name', 'possession', 'possession_team.name', 'team.name',
        'loc_x', 'loc_y', 'es_reanudacion', 'fase', 't_pos', 'pass.cross', 'pass.height.name', 'pass.outcome.name',
        'pass_end_x', 'pass_end_y', 'carry_end_x', 'carry_end_y', 'ball_receipt.outcome.name', 'shot.statsbomb_xg',
        'shot.type.name', 'shot.outcome.name'] + [c for v in EXTRA.values() for c in v]
RUTAS = {k: os.path.join(FW, f'r6b_{k}.parquet') for k in ['tramos', 'decision', 'cadena', 'chequeo']}
if all(os.path.exists(r) for r in RUTAS.values()) and not REHACER:
    TR, DEC, CAD, CK = (pd.read_parquet(RUTAS[k]) for k in ['tramos', 'decision', 'cadena', 'chequeo'])
    print('Cargado de caché:', len(TR), 'tramos ·', len(DEC), 'recepciones en banda')
else:
    piezas, decs, cads, chk = [], [], [], []
    for temp in ENT:
        t0 = time.time()
        ruta_e = os.path.join(FW, f"eventos_{temp.replace('/', '-')}.parquet")
        cols_r1 = set(pq.read_schema(ruta_e).names)
        E = pd.read_parquet(ruta_e, columns=[c for c in COLS if c in cols_r1])
        E = E[E['match_id'].isin(mids)].sort_values(['match_id', 'index']).reset_index(drop=True)
        for k, ops in EXTRA.items():
            c = next((o for o in ops if o in E.columns), None)
            E[k] = E[c] if c else np.nan
        X, falt = leer_extra(temp, cols_r1)
        if X is not None:
            E = E.drop(columns=falt).merge(X[['match_id', 'index'] + falt], on=['match_id', 'index'], how='left')
        for c in COLS:
            if c not in E.columns: E[c] = np.nan
        E['local'] = E['match_id'].map(loc['local'])
        E['ev_local'] = E['team.name'] == E['local']
        es_gol = E['period'].between(1, 4) & (((E['type.name'] == 'Shot') & (E['shot.outcome.name'] == 'Goal')) |
                                              (E['type.name'] == 'Own Goal For'))
        gh = (es_gol & E['ev_local']).astype(int); ga = (es_gol & ~E['ev_local'] & E['team.name'].notna()).astype(int)
        E['dif_home'] = (gh.groupby(E['match_id']).cumsum() - gh) - (ga.groupby(E['match_id']).cumsum() - ga)
        E = E[E['period'].isin([1, 2])].copy()
        E['m_home'] = E['dif_home'] + E['match_id'].map(loc['offset'])
        E['m_home_b'] = E['dif_home'] + E['match_id'].map(loc['offset_b'])
        E['cuarto'] = np.where(E['period'] == 1, np.where(E['minute'] < CORTE_C1, 1, 2), np.where(E['minute'] < CORTE_C3, 3, 4))
        abierto = (E['fase'] != 'bp') & ~E['es_reanudacion'].fillna(False).astype(bool)
        tn = E['type.name']
        propio = E['team.name'] == E['possession_team.name']

        def agrega(d, val, den, nombre, team_col='team.name', local_col='ev_local'):
            s = np.where(d[local_col], 1.0, -1.0)
            gg = pd.DataFrame({'match_id': d['match_id'].values, 'team': d[team_col].values, 'e5': e5(s * d['m_home'].values),
                               'e5b': e5(s * d['m_home_b'].values), 'cuarto': d['cuarto'].values,
                               'num': np.asarray(val, float), 'den': np.asarray(den, float)})
            gg = gg.groupby(['match_id', 'team', 'e5', 'e5b', 'cuarto'], as_index=False)[['num', 'den']].sum()
            gg['metrica'] = nombre; return gg

        # ---------- Transición ofensiva: posesiones que nacen de recuperación en juego abierto (O-6, O-7, O-8) ----------
        Pz = pd.read_parquet(os.path.join(FW, f"posesiones_{temp.replace('/', '-')}.parquet"))
        Pz = Pz[Pz['match_id'].isin(mids) & Pz['period'].isin([1, 2]) & Pz['nace_rec'].fillna(False).astype(bool)]
        pe = E.groupby(['match_id', 'possession'], as_index=False).first()[['match_id', 'possession', 'm_home', 'm_home_b', 'cuarto']]
        Pz = Pz.merge(pe, on=['match_id', 'possession'], how='inner')
        Pz['pos_local'] = Pz['equipo'] == Pz['match_id'].map(loc['local'])
        own = E[propio & E['loc_x'].notna()]
        win = own[own['t_pos'].between(0, TRANS_S)]
        k_ = ['match_id', 'possession']
        cg = win[(win['loc_x'] >= X_U3) & win['type.name'].isin(TIPOS_CG)].groupby(k_).size().rename('cg') > 0
        pc = win[(win['type.name'] == 'Pass') & win['pass.outcome.name'].isna()]
        dx_p = (pc['pass_end_x'] - pc['loc_x']).clip(lower=0).groupby([pc['match_id'], pc['possession']]).sum().rename('dx_pase')
        ca = win[win['type.name'] == 'Carry']
        dx_c = (ca['carry_end_x'] - ca['loc_x']).clip(lower=0).groupby([ca['match_id'], ca['possession']]).sum().rename('dx_cond')
        jug = win[win['loc_x'] >= X_MITAD].groupby(k_)['jugador'].nunique().rename('jugadores')
        area = own[own['type.name'].isin(['Pass', 'Carry', 'Shot', 'Dribble', 'Ball Receipt*']) & en_area(own['loc_x'], own['loc_y'])
                   ].groupby(k_).size().rename('area') > 0
        tiro = own[(own['type.name'] == 'Shot') & (own['shot.type.name'] == 'Open Play')].groupby(k_).size().rename('tiro') > 0
        Pz = Pz.join(cg, on=k_).join(dx_p, on=k_).join(dx_c, on=k_).join(jug, on=k_).join(area, on=k_).join(tiro, on=k_)
        for c in ['cg', 'area', 'tiro']: Pz[c] = Pz[c].fillna(False).astype(bool)
        for c in ['dx_pase', 'dx_cond', 'jugadores']: Pz[c] = Pz[c].fillna(0)
        A = dict(team_col='equipo', local_col='pos_local')
        piezas.append(agrega(Pz, 100.0 * Pz['cg'], 1.0, 'contragolpe_propio', **A))
        dx = Pz['dx_pase'] + Pz['dx_cond']; m_ = dx > 0
        piezas.append(agrega(Pz[m_], 100.0 * Pz.loc[m_, 'dx_cond'], dx[m_], 'mezcla_transicion', **A))
        C_ = Pz[Pz['cg']]
        piezas += [agrega(C_, C_['jugadores'], 1.0, 'jugadores_contra', **A),
                   agrega(C_, 100.0 * C_['area'], 1.0, 'contra_a_area', **A),
                   agrega(C_, 100.0 * C_['tiro'], 1.0, 'contra_a_tiro', **A)]
        if 'contragolpe' in Pz: chk.append(pd.DataFrame({'temporada': [temp], 'recuperaciones': [len(Pz)], 'cg_r6b': [int(Pz['cg'].sum())],
                                                         'cg_r1': [int(Pz['contragolpe'].fillna(False).astype(bool).sum())],
                                                         'coinciden_pct': [100 * (Pz['cg'] == Pz['contragolpe'].fillna(False).astype(bool)).mean()]}))
        # M9: % del xG de juego abierto (sin penales) que nace en transición (fase de R1)
        sh = E[(tn == 'Shot') & (E['shot.type.name'] == 'Open Play') & (E['fase'] != 'bp') & E['shot.statsbomb_xg'].notna()]
        sh = sh[sh['shot.statsbomb_xg'] > 0]
        piezas.append(agrega(sh, 100.0 * sh['shot.statsbomb_xg'] * (sh['fase'] == 'transicion'), sh['shot.statsbomb_xg'], 'xg_transicion'))

        # ---------- Conducciones (M14: mezcla y conducción larga) ----------
        pa = E[(tn == 'Pass') & abierto & E['pass.outcome.name'].isna()]
        pp = pa[progresivo(pa['loc_x'], pa['loc_y'], pa['pass_end_x'], pa['pass_end_y'])]
        co = E[(tn == 'Carry') & abierto]
        cp = co[progresivo(co['loc_x'], co['loc_y'], co['carry_end_x'], co['carry_end_y'])]
        prog = pd.concat([pp.assign(es_c=0.0), cp.assign(es_c=1.0)])
        largo = np.hypot(cp['carry_end_x'] - cp['loc_x'], cp['carry_end_y'] - cp['loc_y'])
        piezas += [agrega(prog, 100.0 * prog['es_c'], 1.0, 'mezcla_progresion'),
                   agrega(cp, 100.0 * (largo >= LARGA_YD), 1.0, 'conduccion_larga')]

        # ---------- Encare: decisión tras recibir en banda en el último tercio ----------
        o = E[propio & abierto & E['loc_x'].notna()].copy()
        o['seq'] = np.arange(len(o))
        acc = o[o['type.name'].isin(['Pass', 'Dribble', 'Shot', 'Miscontrol', 'Dispossessed'])][
            ['match_id', 'possession', 'seq', 'index', 't', 'type.name', 'jugador', 'loc_x', 'loc_y', 'pass.cross', 'pass.height.name',
             'pass_end_x', 'pass_end_y', 'pase_atras', 'regate']]
        rec = o[(o['type.name'] == 'Ball Receipt*') & o['ball_receipt.outcome.name'].isna() & (o['loc_x'] >= X_U3) &
                ((o['loc_y'] < CARRIL[0]) | (o['loc_y'] > CARRIL[1]))][['match_id', 'possession', 'seq', 'jugador', 'loc_x', 'loc_y',
                                                                          'm_home', 'm_home_b', 'cuarto', 'team.name', 'ev_local']]
        sig = pd.merge_asof(rec.sort_values('seq'), acc.sort_values('seq').rename(columns=lambda c: 'a_' + c if c not in ['match_id', 'possession', 'seq'] else c)
                            .assign(seq_a=lambda d: d['seq']), on='seq', by=['match_id', 'possession'], direction='forward', allow_exact_matches=False)
        nxt = pd.merge_asof(rec.sort_values('seq')[['match_id', 'possession', 'seq']],
                            o[['match_id', 'possession', 'seq', 'type.name', 'jugador', 'loc_x', 'loc_y', 'carry_end_x', 'carry_end_y']]
                            .sort_values('seq').rename(columns={'type.name': 'n_tipo', 'jugador': 'n_jug', 'loc_x': 'n_x', 'loc_y': 'n_y'}),
                            on='seq', by=['match_id', 'possession'], direction='forward', allow_exact_matches=False)
        sig = sig.merge(nxt, on=['match_id', 'possession', 'seq'], how='left')
        mismo = sig['a_jugador'].eq(sig['jugador'])
        cerca_y = np.where(sig['loc_y'] < 40, AREA_Y[0], AREA_Y[1])
        pico = sig['a_pass_end_x'].between(*PICO['x']) & ((sig['a_pass_end_y'] - cerca_y).abs() <= PICO['dy'])
        es_p = sig['a_type.name'] == 'Pass'; cruz = sig['a_pass.cross'].eq(True)
        sig['accion'] = np.select(
            [~mismo | sig['a_type.name'].isna(), sig['a_type.name'] == 'Dribble', sig['a_type.name'] == 'Shot',
             sig['a_type.name'].isin(['Miscontrol', 'Dispossessed']),
             es_p & cruz & sig['a_pass.height.name'].eq('High Pass'), es_p & cruz,
             es_p & pico, es_p & (sig['a_pass_end_x'] <= sig['a_loc_x'] - ATRAS_YD)],
            ['otro', 'encara', 'tira', 'pierde', 'centro alto', 'centro raso', 'pase al pico', 'pase atrás'], 'pase adelante o lateral')
        sig['conduce'] = (sig['n_tipo'] == 'Carry') & sig['n_jug'].eq(sig['jugador']) & \
                         (np.hypot(sig['carry_end_x'] - sig['n_x'], sig['carry_end_y'] - sig['n_y']) >= CONDUCE_YD)
        sig['regate_ok'] = (sig['a_type.name'] == 'Dribble') & sig['a_regate'].eq('Complete')
        sig['lado'] = np.where(sig['loc_y'] < 40, 'izquierda', 'derecha')                 # verificado en R6a (y baja = izquierda)
        piezas += [agrega(sig, 100.0 * (sig['accion'] == 'encara'), 1.0, 'libertad_encarar'),
                   agrega(sig, 100.0 * (sig['accion'] == 'pase al pico'), 1.0, 'pase_pico')]
        decs.append(sig[['match_id', 'team.name', 'lado', 'accion', 'conduce', 'regate_ok']].rename(columns={'team.name': 'team'}).assign(temporada=temp))
        # Éxito del regate (todos los regates propios de juego abierto)
        dr = E[(tn == 'Dribble') & abierto & propio & E['regate'].notna()]
        piezas.append(agrega(dr, 100.0 * (dr['regate'] == 'Complete'), 1.0, 'exito_regate'))
        # Centro tras regate ganado: la acción anterior del equipo (misma posesión) fue un regate completo del mismo jugador en ≤ 5 s
        ac2 = acc.sort_values('seq').copy()
        g2 = ac2.groupby(['match_id', 'possession'])
        for c in ['type.name', 'jugador', 'regate', 't']: ac2['ant_' + c] = g2[c].shift(1)
        cz = ac2[(ac2['type.name'] == 'Pass') & ac2['pass.cross'].eq(True)]
        cz = cz.merge(o[['seq', 'm_home', 'm_home_b', 'cuarto', 'team.name', 'ev_local']], on='seq')
        tras = (cz['ant_type.name'] == 'Dribble') & cz['ant_regate'].eq('Complete') & cz['ant_jugador'].eq(cz['jugador']) & \
               ((cz['t'] - cz['ant_t']) <= REGATE_CENTRO_S)
        piezas.append(agrega(cz, 100.0 * tras, 1.0, 'centro_tras_regate'))
        # Cadena (descriptivo): secuencias ofensivas con y sin regate ganado en el último tercio → tiro, distancia y xG
        rg = o[(o['type.name'] == 'Dribble') & o['regate'].eq('Complete') & (o['loc_x'] >= X_U3)].groupby(k_).size().rename('reg') > 0
        sq = o.groupby(k_).agg(max_x=('loc_x', 'max'), team=('team.name', 'first')).join(rg)
        sq = sq[sq['max_x'] >= X_U3]; sq['reg'] = sq['reg'].fillna(False).astype(bool)
        st = sh.assign(dist=np.hypot(120 - sh['loc_x'], 40 - sh['loc_y'])).groupby(k_).agg(xg=('shot.statsbomb_xg', 'sum'), dist=('dist', 'min'),
                                                                                          n=('shot.statsbomb_xg', 'size'))
        cads.append(sq.join(st).reset_index().assign(temporada=temp))
        print(f'  {temp}: {len(Pz):,} recuperaciones · {len(rec):,} recepciones en banda · {len(prog):,} acciones progresivas ({time.time() - t0:.0f} s)')
    TR = pd.concat(piezas, ignore_index=True).groupby(['metrica', 'match_id', 'team', 'e5', 'e5b', 'cuarto'], as_index=False)[['num', 'den']].sum()
    DEC, CAD = pd.concat(decs, ignore_index=True), pd.concat(cads, ignore_index=True)
    CK = pd.concat(chk, ignore_index=True) if chk else pd.DataFrame()
    for k, d_ in [('tramos', TR), ('decision', DEC), ('cadena', CAD), ('chequeo', CK)]: d_.to_parquet(RUTAS[k], index=False)

METRICAS = ['contragolpe_propio', 'jugadores_contra', 'contra_a_area', 'contra_a_tiro', 'mezcla_transicion', 'xg_transicion',
            'mezcla_progresion', 'conduccion_larga', 'libertad_encarar', 'exito_regate', 'centro_tras_regate', 'pase_pico']

# ---------- Chequeos ----------
print('\n--- (a) Faltantes y coherencia ---')
print({'recepciones en banda sin jugador %': round(100 * DEC['accion'].eq('otro').mean(), 1),
       'tramos por métrica': TR.groupby('metrica').size().reindex(METRICAS).to_dict()})
if len(CK): print('Contragolpe recalculado (O-8) frente a la marca de R1, por temporada:'); print(CK.round(1).to_string(index=False))

print('\n--- (b) Valores de liga por temporada (suma entre suma) ---')
L = TR.merge(Mm[['match_id', 'temporada']], on='match_id').groupby(['temporada', 'metrica'])[['num', 'den']].sum()
print((L['num'] / L['den']).unstack('metrica')[METRICAS].round(2).T.to_string())
print('Unidades: jugadores_contra = jugadores propios con acción en campo rival en los 10 s · el resto en %'
      ' (en mezcla_transicion el denominador son yardas avanzadas, no eventos)')
print('Eventos por equipo-partido (mediana):', TR.groupby(['metrica', 'match_id', 'team'])['den'].sum().groupby('metrica').median().reindex(METRICAS).round(1).to_dict())

ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
et = ptp[ptp['temporada'].isin(ENT)][['match_id', 'team', 'tec_p', 'etapa']].copy()
et['etapa_id'] = et['team'] + ' · ' + et['etapa'].astype(str)
n_et = et.groupby('etapa_id')['match_id'].nunique(); E15 = set(n_et[n_et >= PP.get('etapa_min', 15)].index)
print('\n--- (c) Confiabilidad CRUDA entre mitades (etapas de 15+, 200 repartos; la oficial sale de R6r) ---')
filas = []
for met in METRICAS:
    d = TR[TR.metrica == met].groupby(['match_id', 'team'], as_index=False)[['num', 'den']].sum().merge(et, on=['match_id', 'team'])
    d = d[d.etapa_id.isin(E15) & (d.den > 0)]
    rs = []
    for _ in range(200):
        h = d.assign(r=rng.random(len(d))).sort_values(['etapa_id', 'r']); h['mitad'] = h.groupby('etapa_id').cumcount() % 2
        p = h.groupby(['etapa_id', 'mitad'])[['num', 'den']].sum(); p = (p['num'] / p['den']).unstack()
        r = p[0].corr(p[1]); rs.append(2 * r / (1 + r))
    filas.append(dict(metrica=met, confiabilidad_cruda=np.median(rs), p5=np.percentile(rs, 5), p95=np.percentile(rs, 95),
                      eventos_por_partido=d['den'].sum() / len(d)))
print(pd.DataFrame(filas).round(2).to_string(index=False))

# ---------- Descriptivos ----------
am = sorted({e for e in et.etapa_id.unique() if e.startswith('América')}, key=lambda e: et.loc[et.etapa_id == e, 'match_id'].min())
def etiq(e):
    if e == 'Liga': return e
    p = et.loc[et.etapa_id == e, 'tec_p'].iloc[0].split(); return p[0] + ' ' + (p[-2] if len(p) >= 3 else p[-1])
DEC = DEC.merge(et[['match_id', 'team', 'etapa_id']], on=['match_id', 'team'], how='left')
D_ = DEC[DEC['accion'] != 'otro']
def tabla(df, col):
    t = pd.crosstab(df['etapa_id'], df[col], normalize='index') * 100
    t = pd.concat([t.loc[[e for e in am if e in t.index]], (df[col].value_counts(normalize=True) * 100).rename('Liga').to_frame().T])
    t.index = [etiq(e) for e in t.index]; return t
print('\n--- (d) Decisión del extremo: qué hace tras recibir en banda en el último tercio (% de recepciones) ---')
T = tabla(D_, 'accion')
cond = pd.concat([D_[D_.etapa_id.isin(am)].groupby('etapa_id')['conduce'].mean().rename(index=etiq), pd.Series({'Liga': D_['conduce'].mean()})])
T['conduce antes %'] = 100 * cond.reindex(T.index)
fin = [c for c in ['centro alto', 'centro raso', 'pase al pico', 'pase atrás', 'pase adelante o lateral', 'encara', 'tira'] if c in T]
T['variedad'] = 1 - ((T[fin].div(T[fin].sum(axis=1), axis=0)) ** 2).sum(axis=1)
print(T.round(1).to_string())
print('  variedad = probabilidad de que dos decisiones al azar sean distintas (0 = siempre lo mismo; ~0.85 = muy repartido)')
print('\n--- (e) Por banda: encara y éxito del regate en izquierda frente a derecha ---')
def por_banda(df, nombre):
    x = df.assign(enc=(df['accion'] == 'encara').astype(float), ok=df['regate_ok'].astype(float))
    t = x.groupby('lado').agg(recepciones=('enc', 'size'), encares=('enc', 'sum'), regates_ok=('ok', 'sum'))
    t['encara_%'] = 100 * t['encares'] / t['recepciones']; t['exito_%'] = 100 * t['regates_ok'] / t['encares'].where(t['encares'] > 0)
    t.index = [f'{nombre} · {i}' for i in t.index]; return t[['recepciones', 'encara_%', 'exito_%']]
print(pd.concat([por_banda(D_[D_.etapa_id == e], etiq(e)) for e in am] + [por_banda(D_, 'Liga')]).round(1).to_string())
print('\n--- (f) Cadena: secuencias ofensivas con regate ganado en el último tercio frente a sin él ---')
CAD = CAD.merge(et[['match_id', 'team', 'etapa_id']], on=['match_id', 'team'], how='left')
def cad(df):
    return df.groupby('reg').agg(secuencias=('max_x', 'size'), tiro_pct=('n', lambda v: 100 * v.notna().mean()),
                                 distancia_media_tiro=('dist', 'mean'), xg_por_secuencia_con_tiro=('xg', 'mean'))
out = []
for e in am + ['Liga']:
    c = cad(CAD if e == 'Liga' else CAD[CAD.etapa_id == e]); c.index = [f'{etiq(e)} · {"con" if i else "sin"} regate' for i in c.index]; out.append(c)
print(pd.concat(out).round(2).to_string())
print('  Hipótesis de Arturo: con regate ganado, el tiro sale más cerca (pase más corto tras superar al defensor)')
print(f'\nR6b terminó en {time.time() - t_ini:.0f} s. Guardado: fw1/r6b_tramos, r6b_decision, r6b_cadena, r6b_chequeo.'
      " Siguiente: R6r con PREFIJO = 'r6b'.")

R6c - OBV

In [ ]:
# =====================================================================
# R6c v1.0 · OBV: por qué vía y dónde genera valor el equipo, pase extra corregido y valor de la transición
# Framework v1.2: M10 (OBV) · Decisiones de la sesión 8: R6c-1 a R6c-5 (OBV defensivo fuera: va a R9)
# Sesión 8 · 2026-10-14 · Hackathon ISAC 2026, Track 2
# Antes: celda 1 (Drive) y R0. Lee R1 (eventos_), r4_partidos y, si existen, los perfiles de R6a (r6_perfiles_etapa).
# La primera vez lee el OBV de los eventos crudos (~15-25 min) y lo guarda en caché.
# Después: R6r con PREFIJO = 'r6c'. Datos: SOLO entrenamiento.
# =====================================================================

# [R6c v1.0] -----------------------------------------------------------
import os, json, time, warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

SEMILLA = 2026
rng = np.random.default_rng(SEMILLA)
BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026/')
DP = os.path.join(BASE, 'datos_parquet'); FW = os.path.join(DP, 'fw1'); SAL = os.path.join(BASE, 'salidas')
DIR_EV = os.path.join(DP, 'liga', 'eventos')
REHACER = False
t_ini = time.time()
pd.set_option('display.width', 250)

with open(os.path.join(FW, 'parametros_fw1.json')) as f:
    PRM = json.load(f)
PP = PRM.get('P', {})
ENT = sorted(PRM['temporadas']['entrenamiento'])
X_MED, X_U3 = PP.get('tercio_medio_x', 40), PP.get('tercio_ultimo_x', 80)
AREA_X, AREA_Y = PP.get('area_x', 102), tuple(PP.get('area_y', (18, 62)))
CHICA_Y = (30, 50)
TRANS_S = PP.get('transicion_s', 10)
CORTE_C1, CORTE_C3 = 25, 70
VIAS = ['Pass', 'Carry', 'Dribble']                       # acciones de creación (los tiros se miden aparte)
EXTRA = {'obv': ['obv_total_net']}

def en_area(x, y): return (x >= AREA_X) & (y >= AREA_Y[0]) & (y <= AREA_Y[1])
def e5(m):
    m = np.asarray(m, dtype=float)
    return np.clip(np.sign(m) * np.ceil(np.abs(m) - 1e-9), -2, 2).astype(int)

Mm = pd.read_parquet(os.path.join(FW, 'r4_partidos.parquet'))
assert set(Mm['temporada']) <= set(ENT), 'R6c solo usa entrenamiento'
loc = Mm.set_index('match_id'); mids = set(Mm['match_id'])

def leer_extra(temp, cols_r1):
    falt = {k: v for k, v in EXTRA.items() if not any(c in cols_r1 for c in v)}
    if not falt: return None, []
    ruta = os.path.join(FW, f"r6c_extra_{temp.replace('/', '-')}.parquet")
    if os.path.exists(ruta) and not REHACER: return pd.read_parquet(ruta), list(falt)
    piezas = []
    for mid in Mm.loc[Mm.temporada == temp, 'match_id']:
        r = os.path.join(DIR_EV, f'{mid}.parquet')
        if not os.path.exists(r): continue
        disp = set(pq.read_schema(r).names); usar = {'index': 'index'}
        for k, ops in falt.items():
            c = next((o for o in ops if o in disp), None)
            if c: usar[c] = k
        d = pd.read_parquet(r, columns=list(usar)).rename(columns=usar)
        for k in falt:
            if k not in d: d[k] = np.nan
        d['match_id'] = mid; piezas.append(d)
    X = pd.concat(piezas, ignore_index=True); X.to_parquet(ruta, index=False)
    print(f'  {temp}: OBV leído de {len(piezas)} partidos crudos')
    return X, list(falt)

COLS = ['match_id', 'index', 'period', 'minute', 'type.name', 'possession', 'possession_team.name', 'team.name', 'loc_x', 'loc_y',
        'es_reanudacion', 'fase', 't_pos', 'nace_rec', 'pass.cross', 'shot.type.name', 'shot.outcome.name'] + [c for v in EXTRA.values() for c in v]
RUTAS = {k: os.path.join(FW, f'r6c_{k}.parquet') for k in ['tramos', 'acciones', 'partido']}
if all(os.path.exists(r) for r in RUTAS.values()) and not REHACER:
    TR, AC, PA = (pd.read_parquet(RUTAS[k]) for k in ['tramos', 'acciones', 'partido'])
    print('Cargado de caché:', len(TR), 'tramos')
else:
    piezas, acs, pas = [], [], []
    for temp in ENT:
        t0 = time.time()
        ruta_e = os.path.join(FW, f"eventos_{temp.replace('/', '-')}.parquet")
        cols_r1 = set(pq.read_schema(ruta_e).names)
        E = pd.read_parquet(ruta_e, columns=[c for c in COLS if c in cols_r1])
        E = E[E['match_id'].isin(mids)].sort_values(['match_id', 'index']).reset_index(drop=True)
        for k, ops in EXTRA.items():
            c = next((o for o in ops if o in E.columns), None)
            E[k] = E[c] if c else np.nan
        X, falt = leer_extra(temp, cols_r1)
        if X is not None:
            E = E.drop(columns=falt).merge(X[['match_id', 'index'] + falt], on=['match_id', 'index'], how='left')
        for c in COLS:
            if c not in E.columns: E[c] = np.nan
        E['obv'] = pd.to_numeric(E['obv'], errors='coerce')
        E['local'] = E['match_id'].map(loc['local'])
        E['ev_local'] = E['team.name'] == E['local']
        es_gol = E['period'].between(1, 4) & (((E['type.name'] == 'Shot') & (E['shot.outcome.name'] == 'Goal')) |
                                              (E['type.name'] == 'Own Goal For'))
        gh = (es_gol & E['ev_local']).astype(int); ga = (es_gol & ~E['ev_local'] & E['team.name'].notna()).astype(int)
        E['dif_home'] = (gh.groupby(E['match_id']).cumsum() - gh) - (ga.groupby(E['match_id']).cumsum() - ga)
        E = E[E['period'].isin([1, 2])].copy()
        E['m_home'] = E['dif_home'] + E['match_id'].map(loc['offset'])
        E['m_home_b'] = E['dif_home'] + E['match_id'].map(loc['offset_b'])
        E['cuarto'] = np.where(E['period'] == 1, np.where(E['minute'] < CORTE_C1, 1, 2), np.where(E['minute'] < CORTE_C3, 3, 4))
        abierto = (E['fase'] != 'bp') & ~E['es_reanudacion'].fillna(False).astype(bool)
        propio = E['team.name'] == E['possession_team.name']

        def agrega(d, val, den, nombre):
            s = np.where(d['ev_local'], 1.0, -1.0)
            gg = pd.DataFrame({'match_id': d['match_id'].values, 'team': d['team.name'].values, 'e5': e5(s * d['m_home'].values),
                               'e5b': e5(s * d['m_home_b'].values), 'cuarto': d['cuarto'].values,
                               'num': np.asarray(val, float), 'den': np.asarray(den, float)})
            gg = gg.groupby(['match_id', 'team', 'e5', 'e5b', 'cuarto'], as_index=False)[['num', 'den']].sum()
            gg['metrica'] = nombre; return gg

        # Contexto por partido: OBV neto de todas las acciones de cada equipo (a favor; el en contra es el del rival)
        pas.append(E[E['obv'].notna()].groupby(['match_id', 'team.name'])['obv'].agg(['sum', 'size']).reset_index()
                   .rename(columns={'team.name': 'team', 'sum': 'obv_neto', 'size': 'acciones_con_obv'}).assign(temporada=temp))

        A = E[propio & abierto & E['type.name'].isin(VIAS) & E['obv'].notna() & E['loc_x'].notna()].copy()
        A['pos'] = A['obv'].clip(lower=0)
        A['trans'] = A['nace_rec'].fillna(False).astype(bool) & A['t_pos'].between(0, TRANS_S)
        # R6c-1: reparto del valor positivo por vía
        piezas += [agrega(A, 100.0 * A['pos'] * (A['type.name'] == 'Carry'), A['pos'], 'reparto_conduccion'),
                   agrega(A, 100.0 * A['pos'] * (A['type.name'] == 'Dribble'), A['pos'], 'reparto_regate')]
        # R6c-2: dónde (último tercio frente al medio)
        B = A[A['loc_x'] >= X_MED]
        piezas.append(agrega(B, 100.0 * B['pos'] * (B['loc_x'] >= X_U3), B['pos'], 'obv_ultimo_tercio'))
        # R6c-4: % del valor positivo generado en transición (10 s tras recuperar)
        piezas.append(agrega(A, 100.0 * A['pos'] * A['trans'], A['pos'], 'obv_transicion_pct'))
        # R6c-3: en el área, ¿pasa o tira? (pases sin centros y tiros de juego abierto)
        F = E[propio & abierto & en_area(E['loc_x'], E['loc_y']) &
              (((E['type.name'] == 'Pass') & ~E['pass.cross'].eq(True)) | ((E['type.name'] == 'Shot') & (E['shot.type.name'] == 'Open Play')))].copy()
        piezas.append(agrega(F, 100.0 * (F['type.name'] == 'Pass'), 1.0, 'tasa_pase_area'))
        F['zona'] = np.where(F['loc_y'].between(*CHICA_Y), 'área central', 'área lateral')
        acs.append(pd.concat([
            A[['match_id', 'team.name', 'type.name', 'obv', 'trans']].assign(grupo='via'),
            F[['match_id', 'team.name', 'type.name', 'obv', 'zona']].assign(grupo='area')]).rename(columns={'team.name': 'team'}).assign(temporada=temp))
        print(f'  {temp}: {len(A):,} acciones de creación con OBV · {len(F):,} decisiones en el área ({time.time() - t0:.0f} s)')
    TR = pd.concat(piezas, ignore_index=True).groupby(['metrica', 'match_id', 'team', 'e5', 'e5b', 'cuarto'], as_index=False)[['num', 'den']].sum()
    AC, PA = pd.concat(acs, ignore_index=True), pd.concat(pas, ignore_index=True)
    for k, d_ in [('tramos', TR), ('acciones', AC), ('partido', PA)]: d_.to_parquet(RUTAS[k], index=False)

METRICAS = ['reparto_conduccion', 'reparto_regate', 'obv_ultimo_tercio', 'obv_transicion_pct', 'tasa_pase_area']

# ---------- Chequeos ----------
print('\n--- (a) Cobertura del OBV y valores de liga ---')
v = AC[AC.grupo == 'via']
print({'acciones de creación con OBV': len(v), 'OBV medio por acción (milésimas de gol)': round(1000 * v['obv'].mean(), 2),
       '% de acciones con OBV positivo': round(100 * (v['obv'] > 0).mean(), 1)})
print('Nota: si "acciones de creación con OBV" es 0, el campo no existe en estos datos y R6c no se puede usar.')
L = TR.merge(Mm[['match_id', 'temporada']], on='match_id').groupby(['temporada', 'metrica'])[['num', 'den']].sum()
print((L['num'] / L['den']).unstack('metrica')[METRICAS].round(2).T.to_string())
print('Unidades: % del OBV positivo (los repartos y el de transición) · % de decisiones (tasa_pase_area)')

ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
et = ptp[ptp['temporada'].isin(ENT)][['match_id', 'team', 'tec_p', 'etapa']].copy()
et['etapa_id'] = et['team'] + ' · ' + et['etapa'].astype(str)
n_et = et.groupby('etapa_id')['match_id'].nunique(); E15 = set(n_et[n_et >= PP.get('etapa_min', 15)].index)
print('\n--- (b) Confiabilidad CRUDA entre mitades (etapas de 15+, 200 repartos; la oficial sale de R6r) ---')
filas, crudo = [], {}
for met in METRICAS:
    d = TR[TR.metrica == met].groupby(['match_id', 'team'], as_index=False)[['num', 'den']].sum().merge(et, on=['match_id', 'team'])
    d = d[d.etapa_id.isin(E15) & (d.den > 0)]
    rs = []
    for _ in range(200):
        h = d.assign(r=rng.random(len(d))).sort_values(['etapa_id', 'r']); h['mitad'] = h.groupby('etapa_id').cumcount() % 2
        p = h.groupby(['etapa_id', 'mitad'])[['num', 'den']].sum(); p = (p['num'] / p['den']).unstack()
        r = p[0].corr(p[1]); rs.append(2 * r / (1 + r))
    pm = d.groupby('etapa_id')[['num', 'den']].sum(); crudo[met] = pm['num'] / pm['den']
    filas.append(dict(metrica=met, confiabilidad_cruda=np.median(rs), p5=np.percentile(rs, 5), p95=np.percentile(rs, 95)))
print(pd.DataFrame(filas).round(2).to_string(index=False))

# ---------- Descriptivos ----------
am = sorted({e for e in et.etapa_id.unique() if e.startswith('América')}, key=lambda e: et.loc[et.etapa_id == e, 'match_id'].min())
def etiq(e):
    if e == 'Liga': return e
    p = et.loc[et.etapa_id == e, 'tec_p'].iloc[0].split(); return p[0] + ' ' + (p[-2] if len(p) >= 3 else p[-1])
AC = AC.merge(et[['match_id', 'team', 'etapa_id']], on=['match_id', 'team'], how='left')
v = AC[AC.grupo == 'via']
print('\n--- (c) Eficiencia por vía (contexto, mide sobre todo a los jugadores): OBV neto por cada 100 acciones, en goles ---')
def efi(df):
    return (100 * df.groupby('type.name')['obv'].mean()).reindex(VIAS)
T = pd.DataFrame({etiq(e): efi(v[v.etapa_id == e]) for e in am}); T['Liga'] = efi(v)
print(T.round(3).T.to_string())
print('  Ejemplo de lectura: 0.50 = cada 100 pases suman medio gol de diferencia esperada')

print('\n--- (d) Valor del pase extra en el área: OBV medio de los pases (incluidos los fallidos) frente al de los tiros, por zona ---')
a = AC[AC.grupo == 'area']
def ext(df):
    g = df.groupby(['zona', 'type.name'])['obv'].agg(['mean', 'size']).unstack('type.name')
    out = pd.DataFrame({'pases': g[('size', 'Pass')], 'tiros': g[('size', 'Shot')], 'tasa_pase_%': 100 * g[('size', 'Pass')] / (g[('size', 'Pass')] + g[('size', 'Shot')]),
                        'obv_pase': g[('mean', 'Pass')], 'obv_tiro': g[('mean', 'Shot')]})
    out['ventaja_pasar'] = out['obv_pase'] - out['obv_tiro']; return out
out = [ext(a).assign(quien='Liga')] + [ext(a[a.etapa_id == e]).assign(quien=etiq(e)) for e in am]
print(pd.concat(out).reset_index().set_index(['quien', 'zona']).round(4).to_string())
print('  ventaja_pasar > 0: en promedio, dar el pase vale más que tirar desde esa zona (sin sesgo: cuenta los pases fallidos)')

print('\n--- (e) Valor de la transición: OBV de los 10 s tras recuperar, por recuperación, y % del OBV positivo en transición ---')
tv = v[v['trans'].eq(True)].groupby('etapa_id')['obv'].sum()
tot_pos = v.assign(p=v['obv'].clip(lower=0)).groupby('etapa_id')['p'].sum()
tr_pos = v[v['trans'].eq(True)].assign(p=lambda d: d['obv'].clip(lower=0)).groupby('etapa_id')['p'].sum()
PAe = PA.merge(et[['match_id', 'team', 'etapa_id']], on=['match_id', 'team'])
riv = PA.merge(Mm[['match_id', 'local', 'visita']], on='match_id')
riv['rival'] = np.where(riv['team'] == riv['local'], riv['visita'], riv['local'])
riv = riv[['match_id', 'rival', 'obv_neto']].rename(columns={'rival': 'team', 'obv_neto': 'obv_contra'})
PAe = PAe.merge(riv, on=['match_id', 'team'], how='left')
ctx = PAe.groupby('etapa_id').agg(partidos=('match_id', 'nunique'), obv_favor=('obv_neto', 'mean'), obv_contra=('obv_contra', 'mean'))
ctx['transicion_%_del_valor'] = 100 * tr_pos.reindex(ctx.index) / tot_pos.reindex(ctx.index)
ctx['obv_transicion_por_partido'] = tv.reindex(ctx.index) / ctx['partidos']
vv = ctx.loc[[e for e in am if e in ctx.index]].copy(); vv.index = [etiq(e) for e in vv.index]
vv.loc['Liga (mediana de etapas 15+)'] = ctx.loc[ctx.index.isin(E15)].median()
print(vv.round(3).to_string())
print('  obv_favor / obv_contra = OBV neto por partido (diferencia esperada de goles) · contexto, no rasgo (R6c-5)')

print('\n--- (f) Hipótesis de Arturo: ¿el pase extra es más común en equipos que centran por abajo? ---')
r6 = os.path.join(FW, 'r6_perfiles_etapa.parquet')
if os.path.exists(r6):
    P6r = pd.read_parquet(r6)
    P6 = P6r.pivot_table(index='etapa_id', columns='metrica', values='crudo' if 'crudo' in P6r else 'perfil')
    if 'centro_alto' not in P6:
        print('r6_perfiles_etapa no tiene centro_alto.')
    else:
        x = pd.DataFrame({'tasa_pase_area': crudo['tasa_pase_area']}).join(P6[['centro_alto']], how='inner')
        print(f"Etapas de 15+: {len(x)} · correlación (cruda) tasa de pase en el área con % de centro alto: {x.corr().iloc[0, 1]:.2f}"
              ' (negativa = la hipótesis se sostiene)')
else:
    print('Sin r6_perfiles_etapa (correr antes R6a y R6r con PREFIJO = "r6").')
print(f'\nR6c terminó en {time.time() - t_ini:.0f} s. Guardado: fw1/r6c_tramos, r6c_acciones, r6c_partido. Siguiente: R6r con PREFIJO = "r6c".')

R6d- Descriptivo

In [ ]:
# =====================================================================
# R6d v1.0 · Descriptivos sin rasgo: empatar frente a que te empaten (S6-06) y periodos de dominio (P-10)
# Framework v1.2: C (S6-06: solo descripción) y P-10 (dominio: field tilt F2 ≥ 70% en ventana móvil de 10 min, mínimo de acciones)
# Sesión 8 · 2026-10-14 · Hackathon ISAC 2026, Track 2
# Antes: celda 1 (Drive) y R0. Lee R1 (eventos_, posesiones_) y r4_partidos. Datos: SOLO entrenamiento. Tarda ~3-6 min.
# Nada de esta celda entra al perfil del técnico: describe cómo reacciona a un gol y qué hace cuando domina o lo dominan.
# =====================================================================

# [R6d v1.0] -----------------------------------------------------------
import os, json, time, warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
warnings.filterwarnings('ignore')

BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026/')
DP = os.path.join(BASE, 'datos_parquet'); FW = os.path.join(DP, 'fw1'); SAL = os.path.join(BASE, 'salidas')
t_ini = time.time()
pd.set_option('display.width', 250)
with open(os.path.join(FW, 'parametros_fw1.json')) as f:
    PRM = json.load(f)
PP = PRM.get('P', {})
ENT = sorted(PRM['temporadas']['entrenamiento'])
X_U3 = PP.get('tercio_ultimo_x', 80)
VENT_GOL = 10                    # S6-06: minutos antes y después del gol
DOM_UMBRAL = 70                  # P-10: field tilt ≥ 70% en la ventana
DOM_VENT = 10                    # P-10: ventana móvil de 10 minutos
DOM_MIN = 20                     # P-10: mínimo de acciones en la ventana (inicial; se revisa con la distribución del bloque c)

Mm = pd.read_parquet(os.path.join(FW, 'r4_partidos.parquet'))
assert set(Mm['temporada']) <= set(ENT), 'R6d solo usa entrenamiento'
loc = Mm.set_index('match_id'); mids = set(Mm['match_id'])
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
et = ptp[ptp['temporada'].isin(ENT)][['match_id', 'team', 'tec_p', 'etapa']].copy()
et['etapa_id'] = et['team'] + ' · ' + et['etapa'].astype(str)
am = sorted({e for e in et.etapa_id.unique() if e.startswith('América')}, key=lambda e: et.loc[et.etapa_id == e, 'match_id'].min())
def etiq(e):
    p = et.loc[et.etapa_id == e, 'tec_p'].iloc[0].split(); return p[0] + ' ' + (p[-2] if len(p) >= 3 else p[-1])

COLS = ['match_id', 'index', 'period', 'minute', 't', 'type.name', 'possession', 'possession_team.name', 'team.name', 'loc_x',
        'es_reanudacion', 'fase', 'ball_receipt.outcome.name', 'shot.statsbomb_xg', 'shot.type.name', 'shot.outcome.name', 'nace_rec', 't_pos']
MIN_, GOL, REC = [], [], []
for temp in ENT:
    t0 = time.time()
    ruta = os.path.join(FW, f"eventos_{temp.replace('/', '-')}.parquet")
    cols = [c for c in COLS if c in set(pq.read_schema(ruta).names)]
    E = pd.read_parquet(ruta, columns=cols)
    for c in COLS:
        if c not in E.columns: E[c] = np.nan
    E = E[E['match_id'].isin(mids) & E['period'].isin([1, 2])].sort_values(['match_id', 'index']).reset_index(drop=True)
    E['local'] = E['match_id'].map(loc['local']); E['visita'] = E['match_id'].map(loc['visita'])
    E['rival'] = np.where(E['team.name'] == E['local'], E['visita'], E['local'])
    E['m'] = E['minute'].astype(int)                                            # minuto del reloj (el 2T empieza en 45)
    abierto = (E['fase'] != 'bp') & ~E['es_reanudacion'].fillna(False).astype(bool)
    tn = E['type.name']
    # Acciones en último tercio por minuto y equipo (las de F2: pases, conducciones, recepciones completas y tiros)
    ac = E[abierto & tn.isin(['Pass', 'Carry', 'Ball Receipt*', 'Shot']) & (E['loc_x'] >= X_U3) &
           ~((tn == 'Ball Receipt*') & E['ball_receipt.outcome.name'].notna()) & E['team.name'].notna()]
    u3 = ac.groupby(['match_id', 'period', 'm', 'team.name']).size().rename('u3')
    pr = E[(tn == 'Pressure') & abierto]
    pres = pr.groupby(['match_id', 'period', 'm', 'team.name']).agg(pres=('loc_x', 'size'), pres_alta=('loc_x', lambda v: (v >= X_U3).sum()))
    sh = E[(tn == 'Shot') & (E['shot.type.name'] == 'Open Play') & (E['fase'] != 'bp')]
    xg = sh.groupby(['match_id', 'period', 'm', 'team.name'])['shot.statsbomb_xg'].sum().rename('xg')
    # Posesión por minuto: segundos de posesión (del primer al último evento de cada posesión, repartidos al minuto de inicio)
    ps = E.groupby(['match_id', 'possession']).agg(period=('period', 'first'), m=('m', 'first'), team=('possession_team.name', 'first'),
                                                  t_ini=('t', 'first'), t_fin=('t', 'last'))
    ps['dur'] = (ps['t_fin'] - ps['t_ini']).clip(lower=0)
    pos = ps.groupby(['match_id', 'period', 'm', 'team'])['dur'].sum().rename('pos_s')
    pos.index = pos.index.set_names(['match_id', 'period', 'm', 'team.name'])
    rec = E[E['nace_rec'].fillna(False).astype(bool) & (E['team.name'] == E['possession_team.name'])].groupby(['match_id', 'possession']).agg(
        period=('period', 'first'), m=('m', 'first'), team=('team.name', 'first'),
        cg=('loc_x', lambda v: False))                                          # se rellena abajo
    w = E[E['nace_rec'].fillna(False).astype(bool) & (E['team.name'] == E['possession_team.name']) & E['t_pos'].between(0, 10) & (E['loc_x'] >= X_U3)]
    rec['cg'] = rec.index.isin(w.groupby(['match_id', 'possession']).size().index)
    REC.append(rec.reset_index().assign(temporada=temp))
    M = pd.concat([u3, pres, xg, pos], axis=1).fillna(0).reset_index().rename(columns={'team.name': 'team'})
    MIN_.append(M.assign(temporada=temp))
    # Goles (marcador simple del partido, periodos 1 y 2)
    g = E[((tn == 'Shot') & (E['shot.outcome.name'] == 'Goal')) | (tn == 'Own Goal For')][['match_id', 'period', 'm', 'team.name', 'local']]
    GOL.append(g.assign(temporada=temp))
    print(f'  {temp}: {E.match_id.nunique()} partidos ({time.time() - t0:.0f} s)')
M = pd.concat(MIN_, ignore_index=True); G = pd.concat(GOL, ignore_index=True); RC = pd.concat(REC, ignore_index=True)

# ---------- Rejilla completa: cada equipo, cada minuto de cada periodo ----------
lados = pd.concat([Mm[['match_id', 'local']].rename(columns={'local': 'team'}), Mm[['match_id', 'visita']].rename(columns={'visita': 'team'})])
lados['rival'] = lados['match_id'].map(loc['local']).where(lados['team'] != lados['match_id'].map(loc['local']), lados['match_id'].map(loc['visita']))
rng_m = M.groupby(['match_id', 'period'])['m'].agg(['min', 'max']).reset_index()
grid = [(a, b, m) for a, b, lo, hi in rng_m[['match_id', 'period', 'min', 'max']].itertuples(index=False) for m in range(int(lo), int(hi) + 1)]
grid = pd.DataFrame(grid, columns=['match_id', 'period', 'm']).merge(lados, on='match_id')
Gm = grid.merge(M, on=['match_id', 'period', 'm', 'team'], how='left')
Gr = M.rename(columns={'team': 'rival', 'u3': 'u3_r', 'pres': 'pres_r', 'pres_alta': 'pres_alta_r', 'xg': 'xg_r', 'pos_s': 'pos_s_r'}).drop(columns='temporada')
Gm = Gm.merge(Gr, on=['match_id', 'period', 'm', 'rival'], how='left').fillna(0)
Gm = Gm.sort_values(['match_id', 'team', 'period', 'm']).reset_index(drop=True)
# Marcador simple antes de cada minuto, desde el equipo
G['gl'] = (G['team.name'] == G['local']).astype(int)
G = G.sort_values(['match_id', 'period', 'm'])
ev = G.groupby(['match_id', 'period', 'm']).agg(gl=('gl', 'sum'), n=('gl', 'size')).reset_index()
ev['gv'] = ev['n'] - ev['gl']
Gm = Gm.merge(ev[['match_id', 'period', 'm', 'gl', 'gv']], on=['match_id', 'period', 'm'], how='left').fillna({'gl': 0, 'gv': 0})
Gm['es_local'] = Gm['team'] == Gm['match_id'].map(loc['local'])
Gm['gf'] = np.where(Gm['es_local'], Gm['gl'], Gm['gv']); Gm['gc'] = np.where(Gm['es_local'], Gm['gv'], Gm['gl'])
Gm['ord'] = (Gm['period'] - 1) * 1000 + Gm['m']
Gm = Gm.sort_values(['match_id', 'team', 'ord'])
g_ = Gm.groupby(['match_id', 'team'])
Gm['dif_antes'] = (g_['gf'].cumsum() - Gm['gf']) - (g_['gc'].cumsum() - Gm['gc'])
Gm = Gm.merge(et[['match_id', 'team', 'etapa_id']], on=['match_id', 'team'], how='left')

# ---------- (a) S6-06: empatar frente a que te empaten ----------
print('\n--- (a) S6-06: cambio en los 10 minutos posteriores al gol frente a los 10 anteriores (mismo periodo) ---')
print('  field tilt F2 y posesión en % · presión alta = % de presiones en el último tercio rival · xG por 10 min (juego abierto)')
TIPO = {( -1, 0): 'empata', (1, 0): 'le empatan', (0, 1): 'se adelanta', (0, -1): 'le marcan primero'}
filas = []
golpes = Gm[(Gm['gf'] + Gm['gc']) == 1].copy()
golpes['dif_despues'] = golpes['dif_antes'] + golpes['gf'] - golpes['gc']
golpes['tipo'] = [TIPO.get((a, b)) for a, b in zip(golpes['dif_antes'], golpes['dif_despues'])]
golpes = golpes[golpes['tipo'].notna()]
idx = Gm.set_index(['match_id', 'team', 'period', 'm'])[['u3', 'u3_r', 'pos_s', 'pos_s_r', 'pres', 'pres_alta', 'xg', 'xg_r']]
def ventana(r, signo):
    ms = range(r.m - VENT_GOL, r.m) if signo < 0 else range(r.m + 1, r.m + 1 + VENT_GOL)
    k = [(r.match_id, r.team, r.period, m) for m in ms]
    s = idx.reindex(k).dropna(how='all')
    if len(s) < VENT_GOL * 0.7: return None
    t = s.sum()
    return dict(ft=100 * t['u3'] / max(t['u3'] + t['u3_r'], 1), pos=100 * t['pos_s'] / max(t['pos_s'] + t['pos_s_r'], 1),
                pa=100 * t['pres_alta'] / max(t['pres'], 1), xg=10 * t['xg'] / len(s), xg_c=10 * t['xg_r'] / len(s))
for r in golpes.itertuples(index=False):
    a, d = ventana(r, -1), ventana(r, 1)
    if a and d: filas.append(dict(tipo=r.tipo, etapa_id=r.etapa_id, **{f'Δ{k}': d[k] - a[k] for k in a}))
S = pd.DataFrame(filas, columns=['tipo', 'etapa_id', 'Δft', 'Δpos', 'Δpa', 'Δxg', 'Δxg_c'])
tab = S.groupby('tipo').agg(goles=('Δft', 'size'), **{c: (c, 'mean') for c in ['Δft', 'Δpos', 'Δpa', 'Δxg', 'Δxg_c']})
print('Liga:'); print(tab.round(2).to_string())
for e in am:
    t = S[S.etapa_id == e].groupby('tipo').agg(goles=('Δft', 'size'), **{c: (c, 'mean') for c in ['Δft', 'Δpos', 'Δpa', 'Δxg', 'Δxg_c']})
    print(f'{etiq(e)}:'); print(t.round(2).to_string())
print('  Lectura: en "empata", ¿el equipo sigue empujando (Δft > 0) o se conforma? En "le empatan", ¿reacciona o se cae?'
      ' Con pocos goles por etapa (columna goles) es anécdota, no rasgo.')

# ---------- (b) y (c) P-10: periodos de dominio ----------
Gm = Gm.sort_values(['match_id', 'team', 'period', 'm'])
r_ = Gm.groupby(['match_id', 'team', 'period'])
for c in ['u3', 'u3_r']: Gm[c + '_10'] = r_[c].transform(lambda v: v.rolling(DOM_VENT, min_periods=DOM_VENT).sum())
Gm['tot10'] = Gm['u3_10'] + Gm['u3_r_10']
Gm['ft10'] = 100 * Gm['u3_10'] / Gm['tot10'].replace(0, np.nan)
print(f'\n--- (b) Distribución de acciones en el último tercio (de ambos equipos) por ventana de {DOM_VENT} min, para fijar el mínimo de P-10 ---')
print(Gm['tot10'].describe(percentiles=[.05, .1, .25, .5]).round(1).to_dict())
for mn in [10, 15, 20, 25, 30]:
    v = Gm[Gm['tot10'] >= mn]
    print(f'  mínimo {mn}: {100 * len(v) / Gm["tot10"].notna().sum():.1f}% de las ventanas válidas · {100 * (v["ft10"] >= DOM_UMBRAL).mean():.1f}% de ellas con dominio')
Gm['domina'] = (Gm['tot10'] >= DOM_MIN) & (Gm['ft10'] >= DOM_UMBRAL)
Gm['dominado'] = (Gm['tot10'] >= DOM_MIN) & (Gm['ft10'] <= 100 - DOM_UMBRAL)

print(f'\n--- (c) P-10 (field tilt ≥ {DOM_UMBRAL}% en {DOM_VENT} min con ≥ {DOM_MIN} acciones): minutos de dominio y qué rinde ---')
def dom(df):
    n = df['tot10'].notna().sum()
    d, o = df[df['domina']], df[~df['domina'] & ~df['dominado']]
    return pd.Series({'min_domina_%': 100 * df['domina'].sum() / n, 'min_dominado_%': 100 * df['dominado'].sum() / n,
                      'xG/90 dominando': 90 * d['xg'].mean(), 'xG/90 sin dominio': 90 * o['xg'].mean(),
                      'xG contra/90 dominando': 90 * d['xg_r'].mean(), 'xG/90 siendo dominado': 90 * df[df['dominado']]['xg'].mean()})
T = pd.DataFrame({etiq(e): dom(Gm[Gm.etapa_id == e]) for e in am}); T['Liga'] = dom(Gm)
print(T.T.round(2).to_string())
print('  Lectura: ¿el dominio de un técnico rinde ocasiones, o es dominio estéril? ¿Cuánto genera cuando lo dominan (contra)?')

# ---------- (d) La contra cuando te dominan (nota de Arturo) ----------
print('\n--- (d) Contragolpe propio (% de recuperaciones que llegan al último tercio en 10 s) según el momento: dominado o no, y marcador ---')
RC = RC.merge(Gm[['match_id', 'team', 'period', 'm', 'dominado', 'dif_antes', 'etapa_id']], on=['match_id', 'team', 'period', 'm'], how='left')
RC['momento'] = np.where(RC['dominado'].eq(True), 'siendo dominado', 'sin ser dominado')
RC['marcador'] = np.select([RC['dif_antes'] < 0, RC['dif_antes'] > 0], ['abajo', 'arriba'], 'empate')
def cgt(df):
    return df.groupby(['momento', 'marcador'])['cg'].agg(lambda v: 100 * v.mean()).unstack('marcador').reindex(columns=['abajo', 'empate', 'arriba'])
print('Liga:'); print(cgt(RC).round(1).to_string())
for e in am:
    print(f'{etiq(e)}:'); print(cgt(RC[RC.etapa_id == e]).round(1).to_string())
print('  Lectura de Arturo: la contra perdiendo y dominado no es la misma que la contra ganando. Aquí se ve si cambia la tasa.')

Gm[['match_id', 'team', 'period', 'm', 'etapa_id', 'dif_antes', 'tot10', 'ft10', 'domina', 'dominado', 'xg', 'xg_r']].to_parquet(
    os.path.join(FW, 'r6d_minutos.parquet'), index=False)
S.to_csv(os.path.join(SAL, 'r6d_goles.csv'), index=False)
print(f'\nR6d terminó en {time.time() - t_ini:.0f} s. Guardado: fw1/r6d_minutos, salidas/r6d_goles.csv')

# Sesión 9

D5V - Verificación D5 (interinos y suspendidos)

In [ ]:
# =====================================================================
# D5V v1.0 · Verificación de interinatos y suspensiones (D-5) en 2025/26 y 2026/27
# Framework v1.2, D-5 · Sesión 9 · 2026-10-14 · Hackathon ISAC 2026, Track 2
# Antes: celda 1 (Drive) y R0.
# Datos: SOLO la tabla de técnicos (partido_tecnico_periodo). No lee eventos, marcadores ni ninguna conducta (D-2).
# Tarda < 10 s.
# =====================================================================

# [D5V v1.0] ---------------------------------------------------------------
import os, json
import numpy as np
import pandas as pd

BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026/')
DP = os.path.join(BASE, 'datos_parquet'); FW = os.path.join(DP, 'fw1'); SAL = os.path.join(BASE, 'salidas')
os.makedirs(SAL, exist_ok=True)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)

with open(os.path.join(FW, 'parametros_fw1.json')) as f:
    PRM = json.load(f)
ENT = sorted(PRM['temporadas']['entrenamiento'])
MAX_SUSP, MAX_INT = 5, 3                         # D-5: suspensión ≤ 5 partidos (A-B-A); interino ≤ 3 (A-B-C), nunca la última racha

ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)].copy()
ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp = ptp.sort_values(['team', 'fecha', 'match_id']).reset_index(drop=True)
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
PRUEBA = sorted(set(ptp['temporada']) - set(ENT))

# ---------- 0. Chequeos de sanidad de la tabla ----------
print('--- (0) Tabla de técnicos: chequeos ---')
print(f"Filas (equipo-partido jugados): {len(ptp):,} · repetidas (match_id, team): {int(ptp.duplicated(['match_id', 'team']).sum())} (debe ser 0)")
print('Nulos:', {c: int(ptp[c].isna().sum()) for c in ['tec', 'tec_p', 'etapa', 'rol']})
res_t = ptp.groupby('temporada').agg(equipo_partidos=('match_id', 'size'), partidos=('match_id', 'nunique'),
                                     equipos=('team', 'nunique'), primera=('fecha', 'min'), ultima=('fecha', 'max'))
res_t['papel'] = ['entrenamiento' if t in ENT else 'PRUEBA' for t in res_t.index]
print(res_t.to_string())

def rachas(df, col):
    """Rachas consecutivas del mismo valor de `col` dentro de cada equipo (orden por fecha)."""
    d = df.copy()
    d['r'] = (d[col] != d.groupby('team')[col].shift()).astype(int).groupby(d['team']).cumsum()
    R = (d.groupby(['team', 'r'])
           .agg(tec=(col, 'first'), n=('match_id', 'size'), ini=('fecha', 'min'), fin=('fecha', 'max'),
                temp_ini=('temporada', 'first'), temp_fin=('temporada', 'last'),
                rol_tabla=('rol', lambda s: '/'.join(sorted(set(s.astype(str))))),
                etapas=('etapa_id', 'nunique'))
           .reset_index())
    R['prev'] = R.groupby('team')['tec'].shift(1)
    R['sig'] = R.groupby('team')['tec'].shift(-1)
    R['primera'] = R['prev'].isna(); R['ultima'] = R['sig'].isna()
    return d, R

# ---------- 1. Recalcular D-5 desde cero y comparar con la columna `rol` ----------
d_raw, R_raw = rachas(ptp, 'tec')                 # técnico registrado (antes de reasignar suspensiones)
R_raw['susp'] = (R_raw['n'] <= MAX_SUSP) & ~R_raw['primera'] & ~R_raw['ultima'] & (R_raw['prev'] == R_raw['sig'])
d_p, R_p = rachas(ptp, 'tec_p')                   # técnico del periodo (después de reasignar)
R_p['interino'] = (R_p['n'] <= MAX_INT) & ~R_p['primera'] & ~R_p['ultima']

exp = pd.Series('titular', index=ptp.index)
k_int = set(map(tuple, R_p.loc[R_p['interino'], ['team', 'r']].values))
k_sus = set(map(tuple, R_raw.loc[R_raw['susp'], ['team', 'r']].values))
exp[[ (t, r) in k_int for t, r in zip(d_p['team'], d_p['r'])]] = 'interino'
exp[[ (t, r) in k_sus for t, r in zip(d_raw['team'], d_raw['r'])]] = 'sustituto_suspension'
ptp['rol_recalc'] = exp.values

print('\n--- (1) Partidos por rol en la tabla, por temporada ---')
print(pd.crosstab(ptp['temporada'], ptp['rol'], margins=True).to_string())
print(f"\nCoherencia: rachas de tec_p {len(R_p)} · etapas en la tabla {ptp['etapa_id'].nunique()} · "
      f"rachas que abarcan más de una etapa: {int((R_p['etapas'] > 1).sum())} (debe ser 0)")
dis = ptp[ptp['rol'].astype(str) != ptp['rol_recalc']]
print(f"Partidos donde el rol de la tabla NO coincide con D-5 recalculado: {len(dis)} "
      f"(entrenamiento {int(dis['temporada'].isin(ENT).sum())} · prueba {int((~dis['temporada'].isin(ENT)).sum())})")
if len(dis):
    print(dis[['temporada', 'team', 'fecha', 'tec', 'tec_p', 'rol', 'rol_recalc']].head(25).to_string(index=False))

# ---------- 2. Rachas cortas que tocan las temporadas de prueba ----------
def nota(r):
    if r['ultima']: return 'ÚLTIMA racha: aún no clasificable (puede ser técnico nuevo o interino)'
    if r['primera']: return 'primera racha de la tabla'
    if r['n'] <= MAX_INT: return 'interino por D-5'
    return f'{MAX_INT + 1}-{MAX_SUSP} partidos: titular corto (D-5 no lo marca)'

toca = lambda R: R['temp_ini'].isin(PRUEBA) | R['temp_fin'].isin(PRUEBA)
cortas = R_p[toca(R_p) & (R_p['n'] <= MAX_SUSP)].copy()
cortas['nota'] = cortas.apply(nota, axis=1)
print(f'\n--- (2) Rachas de ≤ {MAX_SUSP} partidos (técnico del periodo) que tocan {PRUEBA} ---')
cols = ['team', 'tec', 'n', 'ini', 'fin', 'temp_ini', 'prev', 'sig', 'rol_tabla', 'nota']
print(cortas[cols].assign(ini=cortas['ini'].dt.date, fin=cortas['fin'].dt.date).to_string(index=False) if len(cortas) else '(ninguna)')
susp = R_raw[toca(R_raw) & R_raw['susp']]
print(f'\nSuspensiones (A-B-A, técnico registrado) en prueba: {len(susp)}')
if len(susp):
    print(susp[['team', 'tec', 'n', 'ini', 'prev', 'rol_tabla']].assign(ini=susp['ini'].dt.date).to_string(index=False))

# ---------- 3. Etapas nuevas en prueba (misma definición que E9a) y su etapa previa en otro club ----------
ini_et = ptp.groupby('etapa_id')['fecha'].min()
rol_et = ptp.groupby('etapa_id')['rol'].agg(lambda s: '/'.join(sorted(set(s.astype(str)))))
filas = []
for temp in PRUEBA:
    d = ptp[ptp['temporada'] == temp]; f0 = d['fecha'].min()
    for e in sorted(d['etapa_id'].unique(), key=lambda e: ini_et[e]):
        if ini_et[e] < f0: continue
        r = d[d['etapa_id'] == e].iloc[0]
        prev = ptp[(ptp['tec_p'] == r['tec_p']) & (ptp['team'] != r['team']) & ptp['temporada'].isin(ENT)]
        prev_tit = prev[prev['rol'].astype(str) != 'interino']
        larga = prev.groupby('etapa_id').size().max() if len(prev) else 0
        filas.append(dict(temporada=temp, equipo=r['team'], tecnico=r['tec_p'], inicio=ini_et[e].date(),
                          partidos=int((ptp['etapa_id'] == e).sum()), rol=rol_et[e],
                          ultima=bool(e == ptp[ptp['team'] == r['team']]['etapa_id'].iloc[-1]),
                          previa_partidos=int(len(prev)), previa_sin_interinos=int(len(prev_tit)),
                          previa_etapa_mas_larga=int(larga)))
N = pd.DataFrame(filas)
print('\n--- (3) Etapas nuevas en prueba (definición de E9a) y etapa previa en otro club dentro del entrenamiento ---')
print('  previa_partidos = lo que usó E9a para M3c · previa_sin_interinos = quitando partidos como interino')
print(N.to_string(index=False) if len(N) else '(ninguna)')
if len(N):
    for temp in PRUEBA:
        x = N[N['temporada'] == temp]
        es_int = x['rol'].str.contains('interino')
        print(f"{temp}: etapas nuevas {len(x)} · marcadas interino {int(es_int.sum())} · "
              f"con previa (como E9a) {int((x['previa_partidos'] > 0).sum())} · "
              f"con previa sin interinos y sin ser interino {int(((x['previa_sin_interinos'] > 0) & ~es_int).sum())} · "
              f"previa solo como interino {int(((x['previa_partidos'] > 0) & (x['previa_sin_interinos'] == 0)).sum())}")

# ---------- 4. Guardado ----------
cortas.to_csv(os.path.join(SAL, 'd5v_rachas_cortas.csv'), index=False)
N.to_csv(os.path.join(SAL, 'd5v_etapas_nuevas.csv'), index=False)
print('\nD5V v1.0 terminó. Guardado: salidas/d5v_rachas_cortas.csv, salidas/d5v_etapas_nuevas.csv')

E9b-0

In [ ]:
# =====================================================================
# E9b v1.0 · Capa predictiva de comportamiento (E-9): evaluación en el holdout 2025/26
# Preregistro: claude/preregistro_E9.md (congelado 2026-10-14, 17:27). Código base: E9a v1.1 (congelado).
# Sesión 9 · 2026-10-14 · Hackathon ISAC 2026, Track 2
# Dos celdas:
#   [E9b-0] Preparación: tramos de 2025/26 con el código de R4a (solo las 4 conductas de E-9).
#           NO imprime ninguna tasa ni ningún resultado de 2025/26: solo conteos de partidos y eventos.
#           Incluye una prueba de reproducción: el mismo código, corrido en 2024/25, debe dar los tramos de R4a.
#   [E9b-1] APERTURA ÚNICA: corre E-9 congelada en 2025/26, la recalibración secundaria (E-15) y las 5 apuestas.
# Antes: celda 1 (Drive) y R0. Requiere las tablas de R1 de 2025/26 (eventos_2025-2026 y posesiones_2025-2026).
# =====================================================================

# [E9b-0 v1.0] Preparación: tramos de 2025/26 (sin mirar resultados) ---------------------------
import os, glob, json, re, gzip, time
import numpy as np
import pandas as pd

BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026/')
DP = os.path.join(BASE, 'datos_parquet'); FW = os.path.join(DP, 'fw1')
DC = os.path.join(BASE, 'datos_crudos'); SAL = os.path.join(BASE, 'salidas')
t_ini = time.time()
pd.set_option('display.width', 250)

with open(os.path.join(FW, 'parametros_fw1.json')) as f:
    PRM = json.load(f)
PP = PRM['P']
ENT = sorted(PRM['temporadas']['entrenamiento'])
TEST = '2025/2026'
X_MED, X_U3 = PP['tercio_medio_x'], PP['tercio_ultimo_x']
LARGO, CP_S, TRANS_S = PP['pase_largo_yd'], PP['contrapresion_s'], PP['transicion_s']
PAUSA, EXCL_REANUD = 5, 5                       # P-2 y P-14, como R4a
CORTE_C1, CORTE_C3 = 25, 70                     # A2, como R4a
TIPOS_CG = set(PRM['tipos_contragolpe'])
CONDUCTAS = ['saque_meta_corto', 'presion_alta', 'pase_largo', 'recuperacion_tras_perdida']
archivo = lambda tipo, t: os.path.join(FW, f"{tipo}_{t.replace('/', '-')}.parquet")

faltan = [archivo(t_, TEST) for t_ in ['eventos', 'posesiones'] if not os.path.exists(archivo(t_, TEST))]
assert not faltan, f'Faltan las tablas de R1 de {TEST}: {faltan}. Hay que correr R1 para {TEST} antes de E9b-0.'

ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
ptp = ptp[ptp['jugado'].astype(bool)].copy()
ptp['fecha'] = pd.to_datetime(ptp['fecha'])

# ---------- 1. Partidos: fase, series y marcador efectivo (copia de R4a, sección 1, para cualquier temporada) ----------
def cargar_matches():
    rutas = sorted(glob.glob(os.path.join(DC, 'matches_73_*.json*')) + glob.glob(os.path.join(DP, 'matches_73_*.parquet')))
    piezas = []
    for ruta in rutas:
        try:
            if ruta.endswith('.parquet'):
                d = pd.read_parquet(ruta)
                if any(d[c].map(lambda v: isinstance(v, dict)).any() for c in d.columns if d[c].dtype == object):
                    d = pd.json_normalize(d.to_dict('records'))
            else:
                abrir = gzip.open if ruta.endswith('.gz') else open
                with abrir(ruta, 'rt', encoding='utf-8') as f:
                    d = pd.json_normalize(json.load(f))
        except Exception as e:
            print('  aviso:', os.path.basename(ruta), e); continue
        m = re.search(r'(\d{4}-\d{2}-\d{2})', os.path.basename(ruta))
        d['_fa'] = m.group(1) if m else '0000-00-00'
        if len(d): piezas.append(d.dropna(axis=1, how='all'))
    M = pd.concat(piezas, ignore_index=True).sort_values('_fa', ascending=False).drop_duplicates('match_id')
    col = lambda ops: next((c for c in ops if c in M.columns), None)
    return pd.DataFrame({'match_id': M['match_id'],
                         'local': M[col(['home_team.home_team_name', 'home_team_name'])],
                         'visita': M[col(['away_team.away_team_name', 'away_team_name'])],
                         'gl': M['home_score'], 'gv': M['away_score'],
                         'fase_nombre': M[col(['competition_stage.name'])]})

def tipo_fase(n):
    n = '' if pd.isna(n) else str(n)
    if n in ('Apertura', 'Clausura', 'Regular Season'): return 'regular'
    if re.search('Reclasificaci|Play-In', n, re.I): return 'playin'
    if re.search('Quarter', n, re.I): return 'cuartos'
    if re.search('Semi', n, re.I): return 'semis'
    if re.search('Final', n, re.I): return 'final'
    return 'otro'

def partidos(temporadas, MM):
    p_ = ptp[ptp['temporada'].isin(temporadas)]
    Mm = MM[MM['match_id'].isin(set(p_['match_id']))].merge(
        p_.drop_duplicates('match_id')[['match_id', 'fecha', 'temporada']], on='match_id')
    Mm['torneo'] = Mm['temporada'] + np.where(Mm['fecha'].dt.month >= 7, '-A', '-C')      # D-6
    Mm['tipo'] = Mm['fase_nombre'].map(tipo_fase)
    Mm = Mm.sort_values('fecha').reset_index(drop=True)
    ini_c = Mm[Mm['tipo'] == 'cuartos'].groupby('torneo')['fecha'].min()
    Mm.loc[Mm['tipo'].isin(['semis', 'final']) & (Mm['fecha'] < pd.to_datetime(Mm['torneo'].map(ini_c.to_dict()))), 'tipo'] = 'playin'
    par_ = pd.Series([' | '.join(sorted([a, b])) for a, b in zip(Mm['local'], Mm['visita'])], index=Mm.index)
    ko_ = Mm['tipo'].isin(['cuartos', 'semis', 'final'])
    ult = Mm[ko_].groupby('torneo').tail(1)
    for t_, pp_ in zip(ult['torneo'], par_[ult.index]):
        Mm.loc[ko_ & (Mm['torneo'] == t_) & (par_ == pp_), 'tipo'] = 'final'
    Mm['par'] = par_.values
    ko = Mm['tipo'].isin(['cuartos', 'semis', 'final'])
    g = Mm[ko].groupby(['torneo', 'tipo', 'par'])
    Mm.loc[ko, 'n_serie'] = g['match_id'].transform('size')
    Mm.loc[ko, 'pierna'] = g.cumcount() + 1
    vu = Mm[ko & (Mm['pierna'] == 2)].set_index(['torneo', 'tipo', 'par'])
    ida = Mm[ko & (Mm['pierna'] == 1)].set_index(['torneo', 'tipo', 'par'])
    clave = list(zip(Mm['torneo'], Mm['tipo'], Mm['par']))
    Mm['sembrado'] = [vu['local'].get(k, np.nan) if k_ else np.nan for k, k_ in zip(clave, ko)]
    def dif_ida_local(r):
        k = (r.torneo, r.tipo, r.par)
        if not (r.tipo in ('cuartos', 'semis', 'final') and r.pierna == 2 and k in ida.index): return 0.0
        i = ida.loc[k]
        return float(i.gl - i.gv) if i.local == r.local else float(i.gv - i.gl)
    Mm['dif_ida'] = [dif_ida_local(r) for r in Mm.itertuples(index=False)]
    media = np.where(Mm['tipo'].isin(['cuartos', 'semis']), 0.5, 0.0)
    Mm['offset'] = Mm['dif_ida'] + media * np.where(Mm['local'] == Mm['sembrado'], 1.0, -1.0)
    Mm['offset_b'] = np.where(Mm['pierna'] == 1, 0.0, Mm['offset'])
    mal = ko & (Mm['n_serie'] != 2)
    Mm.loc[mal, ['offset', 'offset_b']] = 0.0
    return Mm

# ---------- 2. Tramos de las 4 conductas (copia literal de R4a, sección 2, para una temporada) ----------
COLS = ['match_id', 'index', 'period', 'minute', 't', 'type.name', 'possession', 'possession_team.name', 'team.name',
        'loc_x', 'counterpress', 'es_reanudacion', 'fase', 't_pos', 'pass.length', 'pass.type.name', 'pass_end_x',
        'shot.outcome.name', 'foul_committed.offensive', 'nace_rec']

def e5(m):
    m = np.asarray(m, dtype=float)
    return np.clip(np.sign(m) * np.ceil(np.abs(m) - 1e-9), -2, 2).astype(int)

def agrega(d, team, es_local, num, den, nombre):
    s = np.where(es_local, 1.0, -1.0)
    g = pd.DataFrame({'match_id': d['match_id'].values, 'team': np.asarray(team),
                      'e5': e5(s * d['m_home'].values), 'e5b': e5(s * d['m_home_b'].values),
                      'cuarto': d['cuarto'].values, 'num': np.asarray(num, dtype=float), 'den': np.asarray(den, dtype=float)})
    g = g.groupby(['match_id', 'team', 'e5', 'e5b', 'cuarto'], as_index=False)[['num', 'den']].sum()
    g['metrica'] = nombre
    return g

def tramos_temporada(temp, Mm):
    loc = Mm.set_index('match_id'); mids = set(Mm['match_id'])
    E = pd.read_parquet(archivo('eventos', temp))
    E = E[[c for c in COLS if c in E.columns]]
    for c in COLS:
        if c not in E.columns: E[c] = np.nan
    E = E[E['match_id'].isin(mids)].sort_values(['match_id', 'index']).reset_index(drop=True)
    E['local'] = E['match_id'].map(loc['local']); E['visita'] = E['match_id'].map(loc['visita'])
    E['ev_local'] = E['team.name'] == E['local']
    es_gol = E['period'].between(1, 4) & (((E['type.name'] == 'Shot') & (E['shot.outcome.name'] == 'Goal')) |
                                          (E['type.name'] == 'Own Goal For'))
    gh = (es_gol & E['ev_local']).astype(int); ga = (es_gol & ~E['ev_local'] & E['team.name'].notna()).astype(int)
    G = pd.DataFrame({'match_id': E['match_id'], 'gh': gh, 'ga': ga}).groupby('match_id').sum()
    E['dif_home'] = (gh.groupby(E['match_id']).cumsum() - gh) - (ga.groupby(E['match_id']).cumsum() - ga)
    E = E[E['period'].isin([1, 2])].copy()                                                # D-9
    E['m_home'] = E['dif_home'] + E['match_id'].map(loc['offset'])
    E['m_home_b'] = E['dif_home'] + E['match_id'].map(loc['offset_b'])
    E['cuarto'] = np.where(E['period'] == 1, np.where(E['minute'] < CORTE_C1, 1, 2), np.where(E['minute'] < CORTE_C3, 3, 4))
    E['rival'] = np.where(E['ev_local'], E['visita'], E['local'])
    abierto = (E['fase'] != 'bp') & ~E['es_reanudacion'].fillna(False).astype(bool)
    tn = E['type.name']
    piezas = []
    pa = E[(tn == 'Pass') & abierto]
    piezas.append(agrega(pa, pa['team.name'], pa['ev_local'], 100.0 * (pa['pass.length'] >= LARGO), 1.0, 'pase_largo'))
    sm = E[(tn == 'Pass') & (E['pass.type.name'] == 'Goal Kick')]
    piezas.append(agrega(sm, sm['team.name'], sm['ev_local'], 100.0 * (sm['pass_end_x'] < X_MED), 1.0, 'saque_meta_corto'))
    Pz0 = pd.read_parquet(archivo('posesiones', temp)).sort_values(['match_id', 'possession'])
    for c_, s_ in [('sig_eq', 'equipo'), ('sig_t0', 't0'), ('sig_per', 'period')]:
        Pz0[c_] = Pz0.groupby('match_id')[s_].shift(-1)
    Pz = Pz0[Pz0['match_id'].isin(mids) & Pz0['period'].isin([1, 2])].copy()
    ps = E[(tn == 'Pressure') & (E['possession_team.name'] != E['team.name'])]
    ps = ps.merge(Pz[['match_id', 'possession', 't0', 'primer_reanudacion']], on=['match_id', 'possession'], how='left')
    ps = ps.sort_values(['match_id', 'possession', 'team.name', 'index']).reset_index(drop=True)
    ps['gap'] = ps.groupby(['match_id', 'possession', 'team.name'])['t'].diff()
    ps['sobre'] = ps['primer_reanudacion'].eq(True) & ((ps['t'] - ps['t0']) <= EXCL_REANUD)
    ps['fid'] = (ps['gap'].isna() | (ps['gap'] > PAUSA)).cumsum()
    ini = ps.drop_duplicates('fid', keep='first')
    ini = ini[(ini['fase'] != 'bp') & ~ini['sobre']]
    piezas.append(agrega(ini, ini['team.name'], ini['ev_local'], 100.0 * (ini['loc_x'] >= X_U3), 1.0, 'presion_alta'))
    pe = E.groupby(['match_id', 'possession'], as_index=False).first()[['match_id', 'possession', 'm_home', 'm_home_b', 'cuarto']]
    Pz = Pz.merge(pe, on=['match_id', 'possession'], how='inner')
    Pz['local'] = Pz['match_id'].map(loc['local']); Pz['visita'] = Pz['match_id'].map(loc['visita'])
    Pz['pos_local'] = Pz['equipo'] == Pz['local']
    Pz['rival'] = np.where(Pz['pos_local'], Pz['visita'], Pz['local'])
    tr = Pz[Pz['nace_rec'].fillna(False).astype(bool)].copy()
    k_ = ['match_id', 'possession']
    Ein = E[E['nace_rec'].fillna(False).astype(bool) & E['t_pos'].between(0, TRANS_S)]
    deL = Ein[Ein['team.name'] != Ein['possession_team.name']]
    fot = deL[(deL['type.name'] == 'Foul Committed') & ~deL['foul_committed.offensive'].eq(True)].groupby(k_)['t_pos'].min().rename('falta_t')
    cgt = Ein[(Ein['team.name'] == Ein['possession_team.name']) & (Ein['loc_x'] >= X_U3) &
              Ein['type.name'].isin(TIPOS_CG)].groupby(k_)['t_pos'].min().rename('cg_t')
    tr = tr.join(fot, on=k_).join(cgt, on=k_)
    tr['rec_t'] = np.where((tr['sig_eq'] == tr['rival']) & (tr['sig_per'] == tr['period']), tr['sig_t0'] - tr['t0'], np.nan)
    tiempos = pd.DataFrame({'falta': tr['falta_t'], 'recuperacion': tr['rec_t'], 'contragolpe': tr['cg_t']})
    tiempos = tiempos.where(tiempos <= TRANS_S).fillna(np.inf)
    tr['res'] = tiempos.idxmin(axis=1).where(np.isfinite(tiempos.min(axis=1)), 'asienta')
    piezas.append(agrega(tr, tr['rival'], ~tr['pos_local'], 100.0 * (tr['res'] == 'recuperacion'), 1.0, 'recuperacion_tras_perdida'))
    TRt = pd.concat(piezas, ignore_index=True).groupby(['metrica', 'match_id', 'team', 'e5', 'e5b', 'cuarto'],
                                                       as_index=False)[['num', 'den']].sum()
    return TRt, G, E['match_id'].nunique(), len(E)

# ---------- 3. Prueba de reproducción en entrenamiento (2024/25): el código copiado debe dar exactamente R4a ----------
MM = cargar_matches()
REF = ENT[-1]
Mm_ref = partidos([REF], MM)
r4p = pd.read_parquet(os.path.join(FW, 'r4_partidos.parquet'))
o = Mm_ref.merge(r4p[['match_id', 'tipo', 'offset']], on='match_id', suffixes=('', '_r4'))
print(f'--- (a) Reproducción en {REF} (entrenamiento; aquí sí se pueden ver números) ---')
print(f"Partidos: {len(Mm_ref)} · cruzan con r4_partidos: {len(o)} · tipo de fase igual: {(o['tipo'] == o['tipo_r4']).mean():.3f} · "
      f"offset igual: {np.isclose(o['offset'], o['offset_r4']).mean():.3f} (ambos deben ser 1.000)")
TRr, _, _, _ = tramos_temporada(REF, Mm_ref)
r4t = pd.read_parquet(os.path.join(FW, 'r4_tramos_v21.parquet'))
r4t = r4t[r4t['metrica'].isin(CONDUCTAS) & r4t['match_id'].isin(set(Mm_ref['match_id']))]
k = ['metrica', 'match_id', 'team', 'e5', 'e5b', 'cuarto']
j = TRr.merge(r4t, on=k, how='outer', suffixes=('', '_r4'), indicator=True)
rep = (j.assign(num_ok=np.isclose(j['num'].fillna(-1), j['num_r4'].fillna(-1)), den_ok=np.isclose(j['den'].fillna(-1), j['den_r4'].fillna(-1)))
        .groupby('metrica').agg(tramos=('_merge', 'size'), solo_uno=('_merge', lambda s: int((s != 'both').sum())),
                                num_igual=('num_ok', 'mean'), den_igual=('den_ok', 'mean')))
print(rep.round(4).to_string())
REPRO_OK = bool((rep['solo_uno'] == 0).all() and (rep[['num_igual', 'den_igual']] > 0.9999).all().all())
print('Reproducción exacta:', 'SÍ' if REPRO_OK else 'NO (no seguir: avisar a Claude)')

# ---------- 4. Tramos de 2025/26 (solo conteos; NINGUNA tasa) ----------
assert REPRO_OK, 'La reproducción falló: E9b-0 se detiene antes de tocar 2025/26.'
Mm_t = partidos([TEST], MM)
TRt, Gt, n_par, n_ev = tramos_temporada(TEST, Mm_t)
gg = Mm_t.set_index('match_id')[['gl', 'gv']].join(Gt)
elo = pd.read_parquet(os.path.join(FW, 'elo_equipo_partido_e2.parquet'))
lado = pd.concat([Mm_t.assign(team=Mm_t.local), Mm_t.assign(team=Mm_t.visita)])[['match_id', 'team']]
sin_elo = lado.merge(elo[['match_id', 'team', 'elo_rival_pre']], on=['match_id', 'team'], how='left')['elo_rival_pre'].isna().sum()
sin_tec = lado.merge(ptp[['match_id', 'team']].assign(ok=1), on=['match_id', 'team'], how='left')['ok'].isna().sum()
print(f'\n--- (b) {TEST}: conteos (sin tasas ni resultados) ---')
print(f"Partidos en la tabla de técnicos: {ptp.loc[ptp.temporada == TEST, 'match_id'].nunique()} · con partido (matches): {len(Mm_t)} · "
      f"con eventos de R1: {n_par} · eventos de periodos 1-2: {n_ev:,}")
print(f"Tipos de fase: {Mm_t['tipo'].value_counts().to_dict()}")
print(f"Goles reconstruidos = marcador oficial: {int(((gg.gl == gg.gh) & (gg.gv == gg.ga)).sum())} de {len(gg)} (chequeo de datos)")
print(f"Equipo-partidos sin Elo del rival: {int(sin_elo)} · sin técnico: {int(sin_tec)} (ambos deben ser 0)")
print('Oportunidades por conducta (eventos en los que se puede dar la conducta; no dice cuántas veces se dio):')
cnt = TRt.groupby('metrica').agg(tramos=('den', 'size'), oportunidades=('den', 'sum'))
cnt['equipo_partidos'] = TRt.groupby(['metrica', 'match_id', 'team']).size().groupby('metrica').size()
print(cnt.round(0).astype(int).to_string())

TRt.to_parquet(os.path.join(FW, 'e9b_tramos_2025-2026.parquet'), index=False)
Mm_t.to_parquet(os.path.join(FW, 'e9b_partidos_2025-2026.parquet'), index=False)
print(f'\nE9b-0 v1.0 terminó en {time.time() - t_ini:.0f} s. Guardado: fw1/e9b_tramos_2025-2026, fw1/e9b_partidos_2025-2026. '
      'El holdout sigue cerrado: no se imprimió ninguna tasa.')


E11b - Elo Holdout

In [ ]:
# =====================================================================
# E11b v1.0 · APERTURA ÚNICA del holdout 2025/26 para el Elo (E-11)
# Preregistro: bitácora v8, sección 3 (congelado 2026-10-13). Código base: R3 (sesión 5).
# Sesión 9 · 2026-10-15 · Hackathon ISAC 2026, Track 2
# Antes: celda 1 (Drive) y R0. NO correr R3a de nuevo (no hace falta y reescribe archivos).
# Lee fw1/elo_partido_e2.parquet (Elo prepartido ya calculado por R3a con la configuración congelada).
# Orden interno: (0) chequeos sin resultados · (1) reproducción del Elo y de 2023/24-2024/25 · (2) apertura de 2025/26.
# Nada de 2026/27 se imprime. Tarda ~1-2 min.
# =====================================================================

# [E11b v1.0] ---------------------------------------------------------
import os, json, time, warnings
from datetime import datetime, timezone, timedelta
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit

REABRIR_POR_ERROR = False      # solo por un ERROR DE CÓDIGO; se anota con fecha y motivo
MOTIVO = ''

SEMILLA, N_BOOT = 2026, 2000
BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026/')
DP = os.path.join(BASE, 'datos_parquet'); FW = os.path.join(DP, 'fw1'); SAL = os.path.join(BASE, 'salidas')
MX = timezone(timedelta(hours=-6))
t0 = time.time()
pd.set_option('display.width', 250)

# Configuración congelada (bitácora v8, sección 3; framework E-2, E-10, E-12, E-13, E-14)
E2 = dict(K=20, H=75, R_ver=0.25, R_inv=0.0)
H_ELO1, INICIO = 150, 1500.0
CALENTAMIENTO = pd.Timestamp('2021-10-01')
PRINCIPAL, SECUNDARIA = 'B365', 'Ca'
SIDX_TEST, SIDX_VIVO = 4, 5                     # 4 = 2025/26 (se abre) · 5 = 2026/27 (no se toca)

marca = os.path.join(SAL, 'e11b_apertura.json')
YA_ABIERTO = os.path.exists(marca)
if YA_ABIERTO:
    with open(marca) as f: prev_ap = json.load(f)
    print(f"AVISO: el holdout del Elo ya se abrió el {prev_ap['fecha_hora']}.")
    assert REABRIR_POR_ERROR and MOTIVO, 'Reabrir solo por un error de código: REABRIR_POR_ERROR = True y MOTIVO escrito.'

# ---------- Funciones (copia literal de R3-0) ----------
def correr_elo(P, K, H, R_ver, R_inv, ini):
    rating = dict(ini); temp_act = torneo_act = None
    pre_l = np.empty(len(P)); pre_v = np.empty(len(P))
    for i, r in enumerate(P.itertuples(index=False)):
        if temp_act is not None and r.temporada != temp_act:
            for t in rating: rating[t] = INICIO + (1 - R_ver) * (rating[t] - INICIO)
        elif torneo_act is not None and r.torneo != torneo_act:
            for t in rating: rating[t] = INICIO + (1 - R_inv) * (rating[t] - INICIO)
        temp_act, torneo_act = r.temporada, r.torneo
        rl = rating.get(r.local_id, INICIO); rv = rating.get(r.visita_id, INICIO)
        E = 1.0 / (1.0 + 10 ** (-(rl + H - rv) / 400))
        dif = r.gl - r.gv
        S = 1.0 if dif > 0 else (0.5 if dif == 0 else 0.0)
        n = abs(dif)
        G = 1.0 if n <= 1 else (1.5 if n == 2 else (11 + n) / 8)
        delta = K * G * (S - E)
        rating[r.local_id] = rl + delta; rating[r.visita_id] = rv - delta
        pre_l[i], pre_v[i] = rl, rv
    return pre_l, pre_v

NPAR = {'A': 3, 'B': 3, 'Ca': 4, 'Cb': 5}
_tipo = lambda cal: 'B' if cal.startswith('B') else cal

def probs_cal(p, x, t, tipo):
    b, c1, lg = p[0], p[1], p[2]
    c1t = c1 + (p[3] * t if tipo in ('Ca', 'Cb') else 0.0)
    lgt = lg + (p[4] * t if tipo == 'Cb' else 0.0)
    c2t = c1t + np.exp(lgt)
    z = b * x
    pv = expit(c1t - z); pve = expit(c2t - z)
    Pm = np.column_stack([pv, pve - pv, 1 - pve])
    Pm = np.clip(Pm, 1e-9, 1); return Pm / Pm.sum(1, keepdims=True)

def ajustar_cal(x, y, t, w, tipo):
    p0 = np.array([2.0, -0.9, 0.17, 0.0, 0.0])[:NPAR[tipo]]
    idx = np.arange(len(y))
    nll = lambda p: -(w * np.log(probs_cal(p, x, t, tipo)[idx, y])).sum() / w.sum()
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        res = minimize(nll, p0, method='BFGS')
    return res.x, bool(res.success)

def pesos(cal, t_fit, t_ref):
    if cal.startswith('B'):
        return 0.5 ** ((t_ref - t_fit) * 365.25 / float(cal[1:]))
    return np.ones_like(t_fit)

def evaluar(P, x, cal, calib_hasta, evalua):
    """Modo congelado (E-10): calibra con todo lo elegible hasta 'calib_hasta' y aplica sin cambios a 'evalua'."""
    y = P['y'].values; t = P['t'].values; sidx = P['sidx'].values
    base_fit = (P['fecha'] >= CALENTAMIENTO).values & (sidx <= calib_hasta)
    ev = sidx == evalua
    if cal == 'freq':
        f = np.bincount(y[base_fit], minlength=3) / base_fit.sum()
        return np.tile(f, (ev.sum(), 1)), y[ev], True, f
    tipo = _tipo(cal)
    p, ok = ajustar_cal(x[base_fit], y[base_fit], t[base_fit], pesos(cal, t[base_fit], t[base_fit].max()), tipo)
    return probs_cal(p, x[ev], t[ev], tipo), y[ev], ok, p

def metricas(Pm, y):
    return -np.mean(np.log(Pm[np.arange(len(y)), y])), np.mean(((Pm - np.eye(3)[y]) ** 2).sum(1))

# ---------- 0. Datos y chequeos sin resultados ----------
P = pd.read_parquet(os.path.join(FW, 'elo_partido_e2.parquet'))       # orden de R3a (fecha, hora, match_id): no se reordena
P['fecha'] = pd.to_datetime(P['fecha'])
P['t'] = (P['fecha'] - CALENTAMIENTO).dt.days / 365.25
P['y'] = np.where(P.gl > P.gv, 2, np.where(P.gl == P.gv, 1, 0))
tab = P.groupby('sidx').agg(temporada=('temporada', 'first'), partidos=('match_id', 'size'))
print('--- (0) Partidos por temporada en el archivo del Elo (sin resultados) ---')
print(tab.to_string())
print(f"Sin marcador: {int(P[['gl', 'gv']].isna().any(axis=1).sum())} · H usada: {sorted(P['H'].unique())} (debe ser [75.0]) · "
      f"partidos repetidos: {int(P['match_id'].duplicated().sum())}")
assert tab.loc[SIDX_TEST, 'temporada'] == '2025/2026' and sorted(P['H'].unique()) == [75.0]
assert not P.loc[P.sidx <= SIDX_TEST, ['gl', 'gv']].isna().any().any(), 'Hay partidos sin marcador hasta 2025/26'

# ---------- 1. Reproducción: el Elo guardado es el de la configuración congelada ----------
# Arranque por tabla (E-14): el Elo con el que cada equipo de 2021/22 jugó su primer partido es su Elo inicial.
lado = pd.concat([P[['sidx', 'local_id', 'elo_local_pre']].set_axis(['sidx', 'id', 'pre'], axis=1),
                  P[['sidx', 'visita_id', 'elo_visita_pre']].set_axis(['sidx', 'id', 'pre'], axis=1)])
lado = lado.reset_index().sort_values(['index'])
prim = lado.groupby('id').first()
INI = prim.loc[prim['sidx'] == 0, 'pre'].to_dict()
pl, pv = correr_elo(P, **E2, ini=INI)
dif_elo = np.maximum(np.abs(pl - P['elo_local_pre'].values), np.abs(pv - P['elo_visita_pre'].values))
print(f'\n--- (1) Reproducción del Elo congelado (K 20 · H 75 · R 0.25/0 · arranque por tabla) ---')
print(f"Equipos con arranque por tabla: {len(INI)} · dispersión inicial {np.std(list(INI.values()), ddof=1):.0f} (congelado: 68)")
print(f"Diferencia máxima entre el Elo recalculado y el guardado, hasta 2025/26: {dif_elo[P.sidx.values <= SIDX_TEST].max():.6f} (debe ser ≈ 0)")
assert dif_elo[P.sidx.values <= SIDX_TEST].max() < 1e-6, 'El Elo guardado no es el de la configuración congelada: NO abrir. Avisar a Claude.'

x = (P['elo_local_pre'] + P['H'] - P['elo_visita_pre']).values / 400
pl1, pv1 = correr_elo(P, E2['K'], H_ELO1, E2['R_ver'], E2['R_inv'], ini={})    # ELO-1: todos en 1,500, H = 150
x1 = (pl1 + H_ELO1 - pv1) / 400
REF = {'B365 2023/24': (1.0240, 0.6145), 'B365 2024/25': (0.9783, 0.5802), 'ELO-1 2024/25': (0.9785, 0.5803),
       'freq 2024/25': (1.0526, 0.6346)}
rep = []
for nom, xx, cal, ch, ev in [('B365 2023/24', x, 'B365', 1, 2), ('B365 2024/25', x, 'B365', 2, 3),
                             ('ELO-1 2024/25', x1, 'A', 2, 3), ('freq 2024/25', x, 'freq', 2, 3)]:
    Pm, y, ok, _ = evaluar(P, xx, cal, ch, ev)
    ll, br = metricas(Pm, y)
    rep.append(dict(prueba=nom, logloss=ll, logloss_bitacora=REF[nom][0], brier=br, brier_bitacora=REF[nom][1], ajuste_ok=ok))
REP = pd.DataFrame(rep)
REP['dif_max'] = np.maximum((REP.logloss - REP.logloss_bitacora).abs(), (REP.brier - REP.brier_bitacora).abs())
print('Reproducción de la validación (bitácora v6, sección 3.1; dif_max debe ser ≤ 0.00006, por el redondeo a 4 decimales):')
print(REP.round(5).to_string(index=False))
assert (REP['dif_max'] <= 6e-5).all() and REP['ajuste_ok'].all(), 'No reproduce la validación: NO abrir. Avisar a Claude.'

# ---------- 2. APERTURA de 2025/26 ----------
MOD = {}
for nom, xx, cal in [(PRINCIPAL + ' (OFICIAL)', x, PRINCIPAL), (SECUNDARIA + ' (secundaria declarada)', x, SECUNDARIA),
                     ('ELO-1 (H 150 + A)', x1, 'A'), ('frecuencias', x, 'freq')]:
    Pm, y, ok, prm = evaluar(P, xx, cal, SIDX_TEST - 1, SIDX_TEST)
    MOD[nom] = Pm
    assert ok, f'El ajuste de {nom} no convergió'
Y = y
ahora = datetime.now(MX).strftime('%Y-%m-%d %H:%M')
with open(marca, 'w') as f:
    json.dump(dict(fecha_hora=ahora, reapertura=YA_ABIERTO, motivo=MOTIVO, partidos=int(len(Y))), f)
print(f'\n========== APERTURA DEL HOLDOUT 2025/26 (Elo) · {ahora} (hora de Ciudad de México) · {len(Y)} partidos ==========')

print('\n--- (2) Log-loss y Brier por partido (menor = mejor) ---')
fil = [dict(modelo=k, logloss=metricas(v, Y)[0], brier=metricas(v, Y)[1]) for k, v in MOD.items()]
print(pd.DataFrame(fil).round(4).to_string(index=False))

rng = np.random.default_rng(SEMILLA)
li = {k: -np.log(v[np.arange(len(Y)), Y]) for k, v in MOD.items()}
kP, kS, kE, kF = list(MOD)
def ic(a, b):
    d = a - b; bs = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(N_BOOT)]
    return d.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5)
print('\n--- (3) Diferencias de log-loss por partido (IC95% bootstrap, 2,000, semilla 2026; negativo = el primero mejor) ---')
comp = []
for nom, a, b in [('B365 − frecuencias', kP, kF), ('B365 − ELO-1', kP, kE), ('Ca − B365', kS, kP)]:
    m, lo, hi = ic(li[a], li[b]); comp.append(dict(comparacion=nom, dif=m, ic_lo=lo, ic_hi=hi))
COMP = pd.DataFrame(comp)
print(COMP.round(4).to_string(index=False))

print('\n--- (4) Calibración por resultado (% pronosticado medio frente a % observado) ---')
obs = np.bincount(Y, minlength=3) / len(Y) * 100
ct = pd.DataFrame({k: v.mean(0) * 100 for k, v in MOD.items() if k != kF}, index=['visita', 'empate', 'local'])
ct.insert(0, 'observado', obs)
print(ct.round(1).to_string())

print('\n--- (5) Calibración por quintiles de P(local), modelo oficial ---')
q = pd.qcut(MOD[kP][:, 2], 5, labels=False, duplicates='drop')
cq = pd.DataFrame({'q': q, 'pred': MOD[kP][:, 2] * 100, 'obs': (Y == 2) * 100.0}).groupby('q').agg(
    partidos=('obs', 'size'), pred_local=('pred', 'mean'), obs_local=('obs', 'mean'))
print(cq.round(1).to_string())

# ---------- Predicciones escritas (bitácora v8, sección 3; framework E-12) ----------
a_ca = COMP.loc[COMP.comparacion == 'Ca − B365', 'dif'].iloc[0]
print('\n--- (6) Predicción escrita del preregistro ---')
print(f"E-12: 'si la deriva de la ventaja local era la recuperación tras los aforos de 2021, se frena y Ca pierde' → "
      f"Ca − B365 = {a_ca:+.4f} → {'Ca PIERDE (se cumple)' if a_ca > 0 else 'Ca GANA (no se cumple)'}")

pd.DataFrame(fil).to_csv(os.path.join(SAL, 'e11b_resumen.csv'), index=False)
COMP.to_csv(os.path.join(SAL, 'e11b_comparaciones.csv'), index=False)
ct.to_csv(os.path.join(SAL, 'e11b_calibracion.csv'))
REP.to_csv(os.path.join(SAL, 'e11b_reproduccion.csv'), index=False)
print(f'\nE11b v1.0 terminó en {time.time() - t0:.0f} s. Guardado: salidas/e11b_*.csv y salidas/e11b_apertura.json ({ahora}).')

Apertura Unica

In [ ]:
# [E9b-1 v1.1] APERTURA ÚNICA del holdout 2025/26 para E-9 ---------------------------------------
# Requiere E9b-0 (sus dos archivos) y las cachés de entrenamiento de R4 y E9a. Tarda ~3-8 min.
# Configuración: la congelada en fw1/e9a_config.json (preregistro, sección 3). Funciones: copia literal de E9a v1.1.
import os, json, time, hashlib
from datetime import datetime, timezone, timedelta
import numpy as np
import pandas as pd

REABRIR_POR_ERROR = False      # solo si hubo un ERROR DE CÓDIGO (no de diseño); se anota en el preregistro con fecha y motivo
MOTIVO = ''
APUESTA5 = 'A'                # v1.1: criterio A acordado con Arturo el 2026-10-15 00:14, antes de abrir (preregistro, sección 10)

BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026/')
DP = os.path.join(BASE, 'datos_parquet'); FW = os.path.join(DP, 'fw1'); SAL = os.path.join(BASE, 'salidas')
t_ini = time.time()
pd.set_option('display.width', 250)
MX = timezone(timedelta(hours=-6))
TEST = '2025/2026'
assert APUESTA5 in ('A', 'B'), 'Falta fijar el criterio de la apuesta 5 (A o B) antes de abrir'

marca = os.path.join(SAL, 'e9b_apertura.json')
YA_ABIERTO = os.path.exists(marca)
if YA_ABIERTO:
    with open(marca) as f: prev_ap = json.load(f)
    print(f"AVISO: el holdout de E-9 ya se abrió el {prev_ap['fecha_hora']}.")
    assert REABRIR_POR_ERROR and MOTIVO, ('Reabrir solo por un error de código: REABRIR_POR_ERROR = True y MOTIVO escrito; '
                                          'anotarlo en el preregistro antes de mirar el resultado nuevo.')

with open(os.path.join(FW, 'parametros_fw1.json')) as f:
    PRM = json.load(f)
ENT = sorted(PRM['temporadas']['entrenamiento'])
VAL, EST = ENT[2], ENT[3]
with open(os.path.join(FW, 'e9a_config.json')) as f:
    CFG = json.load(f)

# ---------- 0. Configuración congelada (preregistro, sección 3): se lee y se verifica, no se elige ----------
CONDUCTAS = CFG['conductas']; PRINC = CFG['principal']
N0, ESCALA, PONDERADO = int(CFG['n0']), CFG['escala'], bool(CFG['perfil_ponderado'])
CLIP, B_REPS, SEMILLA = tuple(CFG['clip']), int(CFG['bootstrap']), int(CFG['semilla'])
assert CONDUCTAS == ['saque_meta_corto', 'presion_alta', 'pase_largo', 'recuperacion_tras_perdida'] and PRINC == 'saque_meta_corto'
assert (N0, ESCALA, PONDERADO, CLIP, B_REPS, SEMILLA) == (5, 'A', True, (0.01, 0.99), 2000, 2026), 'La configuración no es la del preregistro'
assert CFG['contraste_oficial'] == 'M3a_vs_M1'
JORNADA, MIN_ETAPA_VAR = 18, 10
MODELOS = ['M0', 'M1', 'M2', 'M3a', 'M3b', 'M3c']
huella = hashlib.sha256(json.dumps({k: CFG[k] for k in ['conductas', 'n0', 'escala', 'perfil_ponderado', 'clip', 'bootstrap', 'semilla']},
                                   sort_keys=True).encode()).hexdigest()[:12]
print(f'--- (0) Configuración congelada verificada: n0={N0} · escala {ESCALA} · perfil ponderado · recorte {CLIP} · '
      f'bootstrap {B_REPS} · semilla {SEMILLA} · huella {huella}')

# ---------- 1. Insumos: entrenamiento + 2025/26 ----------
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
ptp = ptp[ptp['jugado'].astype(bool)].copy()
ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ini_et = ptp.groupby('etapa_id')['fecha'].min()
ROL_ET = ptp.groupby('etapa_id')['rol'].agg(lambda s: 'interino' if (s.astype(str) == 'interino').all() else 'titular')
ptp_e = ptp[ptp['temporada'].isin(ENT + [TEST])]
assert not ptp_e.duplicated(['match_id', 'team']).any()

cols_m = ['match_id', 'local', 'visita', 'torneo', 'temporada', 'fecha']
Mm = pd.concat([pd.read_parquet(os.path.join(FW, 'r4_partidos.parquet'))[cols_m],
                pd.read_parquet(os.path.join(FW, 'e9b_partidos_2025-2026.parquet'))[cols_m]], ignore_index=True)
Mm['fecha'] = pd.to_datetime(Mm['fecha'])
assert not Mm['match_id'].duplicated().any() and set(Mm['temporada']) == set(ENT + [TEST])
TR = pd.concat([pd.read_parquet(os.path.join(FW, 'r4_tramos_v21.parquet')),
                pd.read_parquet(os.path.join(FW, 'e9b_tramos_2025-2026.parquet'))], ignore_index=True)
elo = pd.read_parquet(os.path.join(FW, 'elo_equipo_partido_e2.parquet'))[['match_id', 'team', 'elo_rival_pre']]

TORNEOS = list(Mm.groupby('torneo')['fecha'].min().sort_values().index)
PREV_T = {t: (TORNEOS[i - 1] if i else None) for i, t in enumerate(TORNEOS)}
lado = pd.concat([Mm.assign(team=Mm.local, rival=Mm.visita, local_=1), Mm.assign(team=Mm.visita, rival=Mm.local, local_=0)])
CTX = (lado[['match_id', 'team', 'rival', 'local_', 'torneo', 'temporada', 'fecha']]
       .merge(elo, on=['match_id', 'team'], how='left')
       .merge(ptp_e[['match_id', 'team', 'etapa_id', 'tec_p']], on=['match_id', 'team'], how='left'))
CTX['elo_c'] = (CTX['elo_rival_pre'] - 1500) / 100
ct = CTX[CTX.temporada == TEST]
print(f"--- (1) Insumos --- torneos {len(TORNEOS)} · partidos de {TEST}: {ct.match_id.nunique()} · "
      f"sin Elo del rival: {int(ct.elo_c.isna().sum())} · sin etapa: {int(ct.etapa_id.isna().sum())}")

TRM, TMM = {}, {}
for met in CONDUCTAS:
    d = TR[TR.metrica == met].groupby(['match_id', 'team', 'e5', 'cuarto'], as_index=False)[['num', 'den']].sum()
    d = d[d['den'] > 0].merge(CTX, on=['match_id', 'team'])
    d = d.dropna(subset=['elo_c', 'etapa_id']).reset_index(drop=True)
    d['y'] = d['num'] / d['den']; d['k'] = d['num'] / 100
    TRM[met] = d
    TMM[met] = d.groupby(['match_id', 'team'], as_index=False)[['num', 'den']].sum().merge(CTX, on=['match_id', 'team'])

# ---------- 2. Funciones: copia literal de E9a v1.1 ----------
def estilo_pred(tm, n0):
    c = tm[['match_id', 'fecha', 'torneo', 'rival', 'num', 'den']].sort_values(['fecha', 'match_id']).copy()
    dia = c.groupby(['torneo', 'fecha'])[['num', 'den']].sum()
    acu = dia.groupby(level=0).cumsum() - dia
    c = c.join(acu.rename(columns={'num': 'ln', 'den': 'ld'}), on=['torneo', 'fecha'])
    g = c.groupby(['torneo', 'rival'])
    c['cn'] = g['num'].cumsum() - c['num']; c['cd'] = g['den'].cumsum() - c['den']; c['n'] = g.cumcount()
    with np.errstate(divide='ignore', invalid='ignore'):
        actual = np.where((c['cd'] > 0) & (c['ld'] > 0), c['cn'] / c['cd'] - c['ln'] / c['ld'], 0.0)
    full = tm.groupby(['torneo', 'rival'])[['num', 'den']].sum()
    liga = tm.groupby('torneo')[['num', 'den']].sum()
    dev = (full['num'] / full['den']) - (liga['num'] / liga['den']).reindex(full.index.get_level_values(0)).values
    previo = np.array([dev.get((PREV_T[t], r), 0.0) if PREV_T[t] else 0.0 for t, r in zip(c['torneo'], c['rival'])])
    w = c['n'] / (c['n'] + n0)
    c['estilo'] = np.nan_to_num(w * actual + (1 - w) * previo)
    return c.set_index(['match_id', 'rival'])['estilo']

def x_base(d):
    X = [(d['e5'] == k).astype(float).values for k in [-2, -1, 1, 2]]
    X += [(d['cuarto'] == c).astype(float).values for c in [2, 3, 4]]
    X += [d['local_'].astype(float).values, d['elo_c'].values, d['estilo'].values]
    return np.column_stack(X)
NB = 10

def ajustar(d):
    tors = [t for t in TORNEOS if t in set(d['torneo'])]
    X = np.column_stack([x_base(d)] + [(d['torneo'] == t).astype(float).values for t in tors])
    y, w = d['y'].values, d['den'].values; sw = np.sqrt(w)
    beta = np.linalg.lstsq(X * sw[:, None], y * sw, rcond=None)[0]
    e = y - X @ beta
    s2 = (w * e ** 2).sum() / (len(y) - X.shape[1])
    c = pd.DataFrame({'e': d['e5'].values, 'q': d['cuarto'].values, 'we': w * e, 'w': w}).groupby(['e', 'q'])[['we', 'w']].sum()
    c['r'] = c['we'] / c['w']; c['se2'] = s2 / c['w']
    wi = 1 / c['se2']; gl_ = max(1, len(c) - 8)
    tau2 = max(0.0, ((wi * c['r'] ** 2).sum() - gl_) / (wi.sum() - (wi ** 2).sum() / wi.sum()))
    c['celda'] = (tau2 / (tau2 + c['se2']) if tau2 > 0 else 0.0) * c['r']
    return dict(beta=beta[:NB], L=dict(zip(tors, beta[NB:])), celda=c['celda'], pbar=np.average(y, weights=w) / 100, ultimo=tors[-1])

def base_pred(d, mod):
    cel = pd.Series(list(zip(d['e5'], d['cuarto']))).map(mod['celda']).fillna(0.0).values
    return x_base(d) @ mod['beta'] + cel

def niveles(dE, mod, n0):
    lev = np.full(len(dE), np.nan); L_prev = mod['L'][mod['ultimo']]
    for t in [t for t in TORNEOS if t in set(dE['torneo'])]:
        m = (dE['torneo'] == t).values; s = dE[m]
        rl = s['den'] * (s['y'] - s['b'] - L_prev)
        dia = pd.DataFrame({'fecha': s['fecha'], 'rl': rl, 'den': s['den']}).groupby('fecha').sum()
        ntm = s.drop_duplicates(['match_id', 'team']).groupby('fecha').size().reindex(dia.index, fill_value=0)
        crl, cden, ctm = dia['rl'].cumsum() - dia['rl'], dia['den'].cumsum() - dia['den'], ntm.cumsum() - ntm
        J = ctm / JORNADA
        nivel = L_prev + np.where(cden > 0, (J / (J + n0)) * crl / cden.where(cden > 0, 1), 0.0)
        lev[m] = s['fecha'].map(pd.Series(nivel, index=dia.index)).values
        L_prev = L_prev + dia['rl'].sum() / dia['den'].sum()
    return lev

def k_(n, tau2, sig2):
    n = np.asarray(n, float)
    if tau2 <= 0: return np.zeros_like(n)
    return np.where(n > 0, tau2 / (tau2 + sig2 / np.maximum(n, 1)), 0.0)

def prob(yhat, perfil, escala, pbar):
    if escala == 'A':
        p = (yhat + perfil) / 100
    else:
        p0 = np.clip(yhat / 100, *CLIP)
        p = 1 / (1 + np.exp(-(np.log(p0 / (1 - p0)) + perfil / (100 * pbar * (1 - pbar)))))
    return np.clip(p, *CLIP)

def correr(met, temp, n0, escala, ponderado=PONDERADO):
    d = TRM[met].copy()
    d['estilo'] = pd.Series(list(zip(d['match_id'], d['rival']))).map(estilo_pred(TMM[met], n0)).fillna(0.0).values
    fit = [t for t in ENT if t < temp]
    dF = d[d['temporada'].isin(fit)].copy(); dE = d[d['temporada'] == temp].copy()
    mod = ajustar(dF)
    dF['yhat'] = base_pred(dF, mod) + dF['torneo'].map(mod['L']).values
    dE['b'] = base_pred(dE, mod)
    dE['yhat'] = dE['b'] + niveles(dE, mod, n0)
    H = pd.concat([dF, dE])
    H = (H.assign(we=H['den'] * (H['y'] - H['yhat'])).groupby(['match_id', 'team'], as_index=False)[['we', 'den']].sum()
         .merge(CTX[['match_id', 'team', 'fecha', 'temporada', 'etapa_id', 'tec_p']], on=['match_id', 'team']))
    H['r'] = H['we'] / H['den']
    H = H.sort_values(['fecha', 'match_id']).reset_index(drop=True)
    h = H[H['temporada'].isin(fit)].groupby('etapa_id')['r']
    n_s, m_s, v_s = h.size(), h.mean(), h.var(ddof=1)
    ok = n_s >= MIN_ETAPA_VAR
    sig2 = float(np.average(v_s[ok], weights=n_s[ok] - 1))
    tau2 = float(max(0.0, m_s[ok].var(ddof=1) - np.mean(sig2 / n_s[ok])))
    g = H.groupby('etapa_id')
    H['ns'] = g.cumcount(); H['ss'] = g['r'].cumsum() - H['r']
    H['sw'] = g['we'].cumsum() - H['we']; H['sd'] = g['den'].cumsum() - H['den']
    E = H[H['temporada'] == temp].copy()
    f0 = E['fecha'].min()
    def mu(x):
        if not len(x): return 0.0
        m = x['we'].sum() / x['den'].sum() if ponderado else x['r'].mean()
        return float(k_(len(x), tau2, sig2) * m)
    mub, muc, tienec = {}, {}, {}
    for s_ in E['etapa_id'].unique():
        r0 = E[E['etapa_id'] == s_].iloc[0]; st = ini_et[s_]
        mub[s_] = mu(H.loc[(H['team'] == r0['team']) & (H['fecha'] < st)])
        prev = H.loc[(H['tec_p'] == r0['tec_p']) & (H['team'] != r0['team']) & (H['fecha'] < st)]
        muc[s_], tienec[s_] = mu(prev), len(prev) > 0
    mu2 = {t_: mu(H.loc[(H['team'] == t_) & (H['fecha'] < f0)]) for t_ in E['team'].unique()}
    kE = k_(E['ns'], tau2, sig2)
    with np.errstate(divide='ignore', invalid='ignore'):
        rbar = np.where(E['ns'] > 0, (E['sw'] / E['sd'].where(E['sd'] > 0, 1)) if ponderado else E['ss'] / E['ns'].clip(lower=1), 0.0)
    for v, m0 in [('M3a', 0.0), ('M3b', E['etapa_id'].map(mub).values), ('M3c', E['etapa_id'].map(muc).values)]:
        E['p_' + v] = m0 + kE * (rbar - m0)
    E['p_M2'] = E['team'].map(mu2).values
    E['nueva'] = E['etapa_id'].map(ini_et).values >= f0
    E['npart'] = E['ns'] + 1
    E['con_previa'] = E['etapa_id'].map(tienec).values
    dE = dE.merge(E[['match_id', 'team', 'p_M2', 'p_M3a', 'p_M3b', 'p_M3c', 'nueva', 'npart', 'con_previa']],
                  on=['match_id', 'team'], how='left')
    p0 = dF.loc[dF['temporada'] == fit[-1], 'num'].sum() / dF.loc[dF['temporada'] == fit[-1], 'den'].sum() / 100
    P = {'M0': np.full(len(dE), np.clip(p0, *CLIP)), 'M1': prob(dE['yhat'].values, 0.0, escala, mod['pbar'])}
    for v in ['M2', 'M3a', 'M3b', 'M3c']:
        P[v] = prob(dE['yhat'].values, dE['p_' + v].values, escala, mod['pbar'])
    kk, nn = dE['k'].values, dE['den'].values
    for v, p in P.items():
        dE['q_' + v] = p
        dE['ll_' + v] = -(kk * np.log(p) + (nn - kk) * np.log(1 - p))
        dE['br_' + v] = kk * (1 - p) ** 2 + (nn - kk) * p ** 2
    return dE, dict(tau=np.sqrt(tau2), sigma=np.sqrt(sig2))

def boot(dE, a, b, mask=None):
    x = dE if mask is None else dE[mask]
    g = x.groupby('match_id')[['ll_' + a, 'll_' + b, 'den']].sum().values
    M = len(g)
    if M < 5: return dict(dif=np.nan, lo=np.nan, hi=np.nan, skill=np.nan, partidos=M)
    idx = np.random.default_rng(SEMILLA).integers(0, M, (B_REPS, M))
    dif = (g[idx, 0].sum(1) - g[idx, 1].sum(1)) / g[idx, 2].sum(1)
    return dict(dif=(g[:, 0].sum() - g[:, 1].sum()) / g[:, 2].sum(), lo=np.quantile(dif, .025), hi=np.quantile(dif, .975),
                skill=1 - g[:, 0].sum() / g[:, 1].sum(), partidos=M)

def r2(dE, v):
    x = dE.assign(pn=dE['q_' + v] * dE['den'], vb=dE['den'] * dE['q_' + v] * (1 - dE['q_' + v]))
    g = x.groupby(['match_id', 'team'])[['k', 'den', 'pn', 'vb']].sum()
    y, yh, w = g['k'] / g['den'], g['pn'] / g['den'], g['den']
    sst = (w * (y - np.average(y, weights=w)) ** 2).sum()
    return 1 - (w * (y - yh) ** 2).sum() / sst, 1 - (g['vb'] / g['den']).sum() / sst

def calibracion(dE, v):
    x = dE.sort_values('q_' + v)
    cum = x['den'].cumsum() / x['den'].sum()
    x['bin'] = np.clip(np.ceil(cum * 10) - 1, 0, 9).astype(int)
    g = x.assign(pn=x['q_' + v] * x['den']).groupby('bin')[['pn', 'k', 'den']].sum()
    return pd.DataFrame({'eventos': g['den'], 'pronostico': 100 * g['pn'] / g['den'], 'observado': 100 * g['k'] / g['den']})

por_evento = lambda dE, col: dE[col].sum() / dE['den'].sum()
logit = lambda p: np.log(p / (1 - p))

def ajustar_recal(k, n, q):
    """E-15: regresión logística binomial de lo observado sobre el log-momio pronosticado (a + b·logit q), por Newton."""
    z = logit(q); X = np.column_stack([np.ones_like(z), z]); beta = np.array([0.0, 1.0])
    for _ in range(50):
        p = 1 / (1 + np.exp(-(X @ beta)))
        g = X.T @ (k - n * p); Hs = (X * (n * p * (1 - p))[:, None]).T @ X
        paso = np.linalg.solve(Hs, g); beta = beta + paso
        if np.abs(paso).max() < 1e-10: break
    return beta

# ---------- 3. Reproducción de la validación (antes de mirar 2025/26): el código debe dar lo mismo que E9a v1.1 ----------
ref = pd.read_csv(os.path.join(SAL, 'e9a_resumen.csv')).set_index(['temporada', 'conducta'])
OOS, rep = {}, []
for temp in [VAL, EST]:
    for met in CONDUCTAS:
        dV, _ = correr(met, temp, N0, ESCALA)
        OOS[(temp, met)] = dV
        rep.append(dict(temporada=temp, conducta=met, ll_M3a=por_evento(dV, 'll_M3a'), ll_M3a_E9a=ref.loc[(temp, met), 'll_M3a'],
                        ll_M1=por_evento(dV, 'll_M1'), ll_M1_E9a=ref.loc[(temp, met), 'll_M1']))
REP = pd.DataFrame(rep)
REP['dif_max'] = np.maximum((REP.ll_M3a - REP.ll_M3a_E9a).abs(), (REP.ll_M1 - REP.ll_M1_E9a).abs())
print('\n--- (2) Reproducción de E9a v1.1 en validación (log-loss por evento; dif_max debe ser < 0.00001) ---')
print(REP.round(5).to_string(index=False))
assert (REP['dif_max'] < 1e-5).all(), 'E9b no reproduce E9a: NO abrir. Avisar a Claude.'

# Recalibración E-15: ajustada SOLO con los pronósticos fuera de muestra de 2023/24 y 2024/25
RECAL = {}
for met in CONDUCTAS:
    x = pd.concat([OOS[(VAL, met)], OOS[(EST, met)]])
    RECAL[met] = ajustar_recal(x['k'].values, x['den'].values, x['q_M3a'].values)
print('Recalibración E-15 (ajustada en validación; a = desplazamiento, b = pendiente; b < 1 = pronósticos demasiado extremos):')
print(pd.DataFrame(RECAL, index=['a', 'b']).T.round(3).to_string())

# ---------- 4. APERTURA: E-9 en 2025/26 ----------
print(f'\n--- Calculando {TEST} (no se imprime nada de 2025/26 hasta terminar el cálculo) ---')
res, cal, cur, cur_fina, preds = [], [], [], [], []
for met in CONDUCTAS:
    dE, info = correr(met, TEST, N0, ESCALA)
    a_, b_ = RECAL[met]
    dE['q_REC'] = np.clip(1 / (1 + np.exp(-(a_ + b_ * logit(dE['q_M3a'].values)))), *CLIP)
    kk, nn = dE['k'].values, dE['den'].values
    dE['ll_REC'] = -(kk * np.log(dE['q_REC']) + (nn - kk) * np.log(1 - dE['q_REC']))
    dE['br_REC'] = kk * (1 - dE['q_REC']) ** 2 + (nn - kk) * dE['q_REC'] ** 2
    et_ = dE['etapa_id']
    dE['interino'] = et_.map(ROL_ET).eq('interino').values
    fila = dict(conducta=met, partidos=dE['match_id'].nunique(), eventos=int(dE['den'].sum()),
                tasa=100 * dE['k'].sum() / dE['den'].sum(), tau=info['tau'], sigma=info['sigma'])
    for v in MODELOS + ['REC']:
        fila['ll_' + v] = por_evento(dE, 'll_' + v); fila['br_' + v] = por_evento(dE, 'br_' + v)
        if v != 'REC': fila['media_' + v] = 100 * (dE['q_' + v] * dE['den']).sum() / dE['den'].sum()
    nuevas5 = dE['nueva'] & (dE['npart'] <= 5) & dE['con_previa']
    for nom, a, b, m in [('M3a_vs_M1', 'M3a', 'M1', None), ('M1_vs_M0', 'M1', 'M0', None), ('M2_vs_M1', 'M2', 'M1', None),
                         ('M3b_vs_M2', 'M3b', 'M2', None), ('M3a_vs_M1_continuan', 'M3a', 'M1', ~dE['nueva']),
                         ('M3a_vs_M1_nuevas', 'M3a', 'M1', dE['nueva']),
                         ('M3a_vs_M1_nuevas_sin_interinos', 'M3a', 'M1', dE['nueva'] & ~dE['interino']),
                         ('M3c_vs_M3b_nuevas5', 'M3c', 'M3b', nuevas5), ('REC_vs_M3a', 'REC', 'M3a', None), ('REC_vs_M1', 'REC', 'M1', None)]:
        for k_b, v_b in boot(dE, a, b, m).items(): fila[f'{nom}_{k_b}'] = v_b
    fila['R2_M1'], fila['techo'] = r2(dE, 'M1'); fila['R2_M3a'], _ = r2(dE, 'M3a')
    fila['etapas_nuevas'] = int(et_[dE['nueva'].values].nunique())
    fila['nuevas_con_previa_5'] = int(et_[nuevas5.values].nunique())
    res.append(fila)
    for v in ['M3a', 'REC']:
        c = calibracion(dE, v); c['conducta'], c['modelo'] = met, v; cal.append(c.reset_index())
    x = dE[dE['nueva']].copy()
    if len(x):
        for cortes, etq, dest in [([0, 5, 10, 17, 999], ['1-5', '6-10', '11-17', '18+'], cur),
                                  ([0, 7, 10, 999], ['1-7', '8-10', '11+'], cur_fina)]:
            x['bloque'] = pd.cut(x['npart'], cortes, labels=etq)
            gb = x.groupby('bloque', observed=True)[['ll_M1', 'll_M3a', 'll_M3b', 'll_M3c', 'den']].sum()
            for v in ['M3a', 'M3b', 'M3c']: gb['mejora_' + v] = 100 * (1 - gb['ll_' + v] / gb['ll_M1'])
            gb['conducta'] = met; dest.append(gb.reset_index())
    preds.append(dE[['match_id', 'team', 'e5', 'cuarto', 'k', 'den', 'nueva', 'npart', 'interino'] +
                    [f'q_{v}' for v in MODELOS + ['REC']]].assign(conducta=met))
    print(f'  {met}: listo ({time.time() - t_ini:.0f} s)')
R = pd.DataFrame(res)
ahora = datetime.now(MX).strftime('%Y-%m-%d %H:%M')
with open(marca, 'w') as f:
    json.dump(dict(fecha_hora=ahora, huella_config=huella, reapertura=YA_ABIERTO, motivo=MOTIVO, criterio_apuesta5=APUESTA5), f)
print(f'\n========== APERTURA DEL HOLDOUT {TEST} · {ahora} (hora de Ciudad de México) ==========')

def tabla(nom):
    t_ = R[['conducta'] + [f'{nom}_{c}' for c in ['dif', 'lo', 'hi', 'skill', 'partidos']]].copy()
    t_.columns = ['conducta', 'dif', 'lo', 'hi', 'skill_%', 'partidos']; t_['skill_%'] *= 100
    return t_

print('\n--- (3) Chequeos: tasa observada frente a pronóstico medio de cada modelo (puntos %; deben estar cerca) ---')
print(R[['conducta', 'partidos', 'eventos', 'tasa', 'tau', 'sigma'] + ['media_' + v for v in MODELOS]].round(2).to_string(index=False))
print('\n--- (4) Log-loss y Brier por evento (menor = mejor) ---')
print(R[['conducta'] + ['ll_' + v for v in MODELOS + ['REC']]].round(4).to_string(index=False))
print(R[['conducta'] + ['br_' + v for v in MODELOS + ['REC']]].round(4).to_string(index=False))
print('\n--- (5) Comparaciones (diferencia de log-loss por evento; IC95% bootstrap pareado por partido; negativo = el primero mejora) ---')
for nom in ['M3a_vs_M1', 'M1_vs_M0', 'M2_vs_M1', 'M3b_vs_M2', 'M3a_vs_M1_continuan', 'M3a_vs_M1_nuevas',
            'M3a_vs_M1_nuevas_sin_interinos', 'M3c_vs_M3b_nuevas5', 'REC_vs_M3a', 'REC_vs_M1']:
    print(f"{nom}{'  ← CONTRASTE OFICIAL (saque_meta_corto)' if nom == 'M3a_vs_M1' else ''}:")
    print(tabla(nom).round(4).to_string(index=False))
print('Etapas nuevas · con previa (primeros 5 partidos):', R.set_index('conducta')[['etapas_nuevas', 'nuevas_con_previa_5']].to_dict('index'))
print('\n--- (6) R² por equipo-partido y su techo ---')
print(R[['conducta', 'R2_M1', 'R2_M3a', 'techo']].round(3).to_string(index=False))
C_ = pd.concat(cal, ignore_index=True)
print('\n--- (7) Calibración por deciles, conducta principal (pronóstico → observado, %) ---')
print(C_[C_.conducta == PRINC].pivot_table(index='bin', columns='modelo', values=['pronostico', 'observado']).round(1).to_string())
K_ = pd.concat(cur, ignore_index=True); KF = pd.concat(cur_fina, ignore_index=True)
print('\n--- (8) Técnicos nuevos: mejora sobre M1 por bloque de partidos de la etapa (%) ---')
print(K_[['conducta', 'bloque', 'den', 'mejora_M3a', 'mejora_M3b', 'mejora_M3c']].round(1).to_string(index=False))
print('Bloques de la apuesta 5 (1-7, 8-10, 11+):')
print(KF[['conducta', 'bloque', 'den', 'mejora_M3a']].round(1).to_string(index=False))

# ---------- 5. Apuestas del preregistro (sección 7) ----------
r0 = R.set_index('conducta'); s = r0.loc[PRINC]
f5 = KF[KF.conducta == PRINC].set_index('bloque')['mejora_M3a']
salto1 = f5.get('8-10', np.nan) - f5.get('1-7', np.nan); salto2 = f5.get('11+', np.nan) - f5.get('8-10', np.nan)
if APUESTA5 == 'A':
    a5 = bool(salto1 > 0 and abs(salto2) < salto1)
else:
    a5 = bool(salto1 > 0 and salto2 < 0.5 * salto1)
AP = pd.DataFrame([
    dict(apuesta='1. Skill M3a vs M1 (saque) entre 3% y 8% e IC excluye 0', valor=f"{100 * s['M3a_vs_M1_skill']:.2f}% · IC [{s['M3a_vs_M1_lo']:+.4f}, {s['M3a_vs_M1_hi']:+.4f}]",
         cumple=bool(3 <= 100 * s['M3a_vs_M1_skill'] <= 8 and s['M3a_vs_M1_hi'] < 0)),
    dict(apuesta='2. Las 4 conductas mejoran (IC excluye 0); recuperación < 0.5%',
         valor=' · '.join(f"{c[:8]} {100 * r0.loc[c, 'M3a_vs_M1_skill']:.2f}%" for c in CONDUCTAS),
         cumple=bool((r0['M3a_vs_M1_hi'] < 0).all() and 100 * r0.loc['recuperacion_tras_perdida', 'M3a_vs_M1_skill'] < 0.5)),
    dict(apuesta='3. M3b − M2 > M2 − M1 (saque; mejora en log-loss)', valor=f"{s['ll_M2'] - s['ll_M3b']:+.4f} frente a {s['ll_M1'] - s['ll_M2']:+.4f}",
         cumple=bool((s['ll_M2'] - s['ll_M3b']) > (s['ll_M1'] - s['ll_M2']))),
    dict(apuesta='4. M3b le gana a M3c en arranques con previa (saque, partidos 1-5)',
         valor=f"dif M3c − M3b {s['M3c_vs_M3b_nuevas5_dif']:+.4f} · IC [{s['M3c_vs_M3b_nuevas5_lo']:+.4f}, {s['M3c_vs_M3b_nuevas5_hi']:+.4f}]",
         cumple=bool(s['M3c_vs_M3b_nuevas5_dif'] > 0)),
    dict(apuesta=f'5. La mejora se aplana entre los partidos 8 y 10 (criterio {APUESTA5})',
         valor=f"1-7 {f5.get('1-7', np.nan):.1f}% · 8-10 {f5.get('8-10', np.nan):.1f}% · 11+ {f5.get('11+', np.nan):.1f}%", cumple=a5),
    dict(apuesta='E-15. Recalibración: mejora el decil alto y gana < 0.003', valor=f"REC − M3a {s['REC_vs_M3a_dif']:+.4f}",
         cumple=bool(-0.003 < s['REC_vs_M3a_dif'] <= 0)),
])
print('\n--- (9) APUESTAS DEL PREREGISTRO ---')
print(AP.to_string(index=False))
cd = C_[(C_.conducta == PRINC) & (C_.bin == 9)].set_index('modelo')
print(f"Decil alto (saque): M3a {cd.loc['M3a', 'pronostico']:.0f} → {cd.loc['M3a', 'observado']:.0f} · "
      f"REC {cd.loc['REC', 'pronostico']:.0f} → {cd.loc['REC', 'observado']:.0f}")

# ---------- 6. Guardado ----------
R.to_csv(os.path.join(SAL, 'e9b_resumen.csv'), index=False)
C_.to_csv(os.path.join(SAL, 'e9b_calibracion.csv'), index=False)
pd.concat([K_.assign(bloques='e9a'), KF.assign(bloques='apuesta5')]).to_csv(os.path.join(SAL, 'e9b_curva.csv'), index=False)
AP.to_csv(os.path.join(SAL, 'e9b_apuestas.csv'), index=False)
pd.DataFrame(RECAL, index=['a', 'b']).T.to_csv(os.path.join(SAL, 'e9b_recalibracion.csv'))
pd.concat(preds, ignore_index=True).to_parquet(os.path.join(FW, 'e9b_predicciones.parquet'), index=False)
print(f'\nE9b-1 v1.1 terminó en {time.time() - t_ini:.0f} s. Guardado: salidas/e9b_*.csv, fw1/e9b_predicciones.parquet, '
      f'salidas/e9b_apertura.json ({ahora}).')


LH- Lectura Holdout

In [ ]:
# =====================================================================
# LH v1.0 · Lectura de resultados ya guardados del holdout (NO calcula, NO reabre)
# Sesión 9 · 2026-10-15 · Solo se usa si Colab se desconectó DESPUÉS del letrero de APERTURA
# y se perdió la salida en pantalla. Lee los CSV que E11b y E9b-1 ya guardaron en salidas/.
# =====================================================================

# [LH v1.0] ---------------------------------------------------------
import os, json
import pandas as pd
BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026/')
SAL = os.path.join(BASE, 'salidas')
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 80)
for marca in ['e11b_apertura.json', 'e9b_apertura.json']:
    r = os.path.join(SAL, marca)
    print(marca, '→', json.load(open(r)) if os.path.exists(r) else 'no existe (esa apertura no llegó a mostrar resultados)')
for nombre in ['e11b_reproduccion', 'e11b_resumen', 'e11b_comparaciones', 'e11b_calibracion',
               'e9b_resumen', 'e9b_apuestas', 'e9b_recalibracion', 'e9b_curva', 'e9b_calibracion']:
    r = os.path.join(SAL, nombre + '.csv')
    if os.path.exists(r):
        print(f'\n--- {nombre} ---'); print(pd.read_csv(r).round(4).to_string(index=False))

Extensión de Pipelines (Celdas X)

In [ ]:
# =====================================================================
# X5 v1.0 · Extensión del pipeline a 2025/26 (D-1: reajuste con las cinco temporadas, mismas reglas)
# Sesión 9 · 2026-10-15 · Hackathon ISAC 2026, Track 2 · Preregistro del plan C, sección 2
# Idea: NO se modifica ninguna celda ya validada (R4a, R4b, R6a, R6b, R6c, R6r-1). Esta celda arma en el disco de Colab
# una carpeta "espejo" (/content/isac_fw5/) cuyas entradas apuntan (enlaces) a los archivos de Drive y cuyo
# parametros_fw1.json incluye 2025/26. Las celdas originales, corridas después, escriben ahí sin tocar fw1.
# X5-1 copia las salidas a Drive (datos_parquet/fw5/ y salidas/fw5/) y verifica que, en las temporadas de entrenamiento,
# los tramos salgan IDÉNTICOS a los de fw1 (mismo código, mismos datos). Si no, el plan C no corre.
# =====================================================================

# [X5-0 v1.0] Preparación de la carpeta espejo (correr DESPUÉS de la celda 1 y R0; NO volver a correr R0 después) ----
import os, json, glob, shutil
import pandas as pd

DRIVE = '/content/drive/MyDrive/isac2026/'
LOCAL = '/content/isac_fw5/'
TEST = '2025/2026'
D_DP, D_FW = os.path.join(DRIVE, 'datos_parquet'), os.path.join(DRIVE, 'datos_parquet', 'fw1')
L_DP, L_FW = os.path.join(LOCAL, 'datos_parquet'), os.path.join(LOCAL, 'datos_parquet', 'fw1')
os.makedirs(L_FW, exist_ok=True); os.makedirs(os.path.join(LOCAL, 'salidas'), exist_ok=True)

def enlazar(src, dst):
    if os.path.islink(dst): os.remove(dst)
    if not os.path.exists(dst): os.symlink(src, dst)

# 1. Todo lo que está fuera de fw1 (tabla de técnicos, partidos, eventos crudos por partido, datos externos)
for raiz_d, raiz_l, excluir in [(DRIVE, LOCAL, {'datos_parquet', 'salidas'}), (D_DP, L_DP, {'fw1', 'fw5'})]:
    for e in os.listdir(raiz_d):
        if e not in excluir: enlazar(os.path.join(raiz_d, e), os.path.join(raiz_l, e))

# 2. Dentro de fw1: SOLO entradas (R1, R2, cachés de crudos, Elo). Nada de r4_*, r6*_tramos, residuos, etc.
PATRONES = ['eventos_*.parquet', 'posesiones_*.parquet', 'componentes_*.parquet', 'presion_*.parquet',
            'r6_extra_*.parquet', 'r6b_extra_*.parquet', 'r6c_extra_*.parquet',
            'elo_equipo_partido_e2.parquet', 'elo_partido_e2.parquet', 'elo_xg_partido.parquet', 'tiros_xg.parquet']
enl = sorted({os.path.basename(r) for p in PATRONES for r in glob.glob(os.path.join(D_FW, p))})
for f in enl: enlazar(os.path.join(D_FW, f), os.path.join(L_FW, f))

# 3. Parámetros: los mismos, con 2025/26 agregada al conjunto de ajuste (D-1)
with open(os.path.join(D_FW, 'parametros_fw1.json')) as f:
    PRM_ = json.load(f)
ENT0 = list(PRM_['temporadas']['entrenamiento'])
PRM_['temporadas']['entrenamiento'] = sorted(set(ENT0) | {TEST})
PRM_['_fw5'] = 'copia de fw1 con 2025/26 agregada (sesión 9); no usar para el holdout'
with open(os.path.join(L_FW, 'parametros_fw1.json'), 'w') as f:
    json.dump(PRM_, f, indent=1)

# 4. R2a y R2b solo existen para el entrenamiento: archivos vacíos para 2025/26 (solo los usan los chequeos de coherencia)
t_ = TEST.replace('/', '-')
for tipo in ['componentes', 'presion']:
    if not os.path.exists(os.path.join(L_FW, f'{tipo}_{t_}.parquet')):
        pd.DataFrame({'match_id': pd.Series(dtype='int64'), 'equipo': pd.Series(dtype='object')}).to_parquet(
            os.path.join(L_FW, f'{tipo}_{t_}.parquet'), index=False)

# 5. Chequeos
falta = [f for f in [f'eventos_{t_}.parquet', f'posesiones_{t_}.parquet', 'elo_equipo_partido_e2.parquet', 'elo_xg_partido.parquet']
         if not os.path.exists(os.path.join(L_FW, f))]
ptp_ = pd.read_parquet(os.path.join(L_DP, 'partido_tecnico_periodo.parquet'))
ex_ = pd.read_parquet(os.path.join(L_FW, 'elo_xg_partido.parquet'))
m26 = set(ptp_.loc[ptp_.temporada == TEST, 'match_id'])
print('--- (X5-0) Carpeta espejo lista en', LOCAL)
print(f'Archivos de fw1 enlazados (solo lectura): {len(enl)} · temporadas de ajuste: {PRM_["temporadas"]["entrenamiento"]}')
print(f"Partidos de {TEST} en la tabla de técnicos: {len(m26)} · con Elo de xG: {ex_['match_id'].isin(m26).sum()} (debe ser {len(m26)})")
print('Faltantes:', falta or 'ninguno')
assert not falta, 'Faltan entradas: avisar a Claude antes de seguir'
BASE = LOCAL                                       # las celdas siguientes leen globals()['BASE']
print('BASE =', BASE, '→ ahora corre, en este orden y sin cambiarles nada: R4a, R4b, R6a, R6b, R6c, R6r-1 (×3), X5-1')


X5-1

In [ ]:
# [X5-1 v1.0] Guardar en Drive y validar (correr al final de la extensión) -----------------------------------------
import os, json, shutil
import numpy as np
import pandas as pd

DRIVE = '/content/drive/MyDrive/isac2026/'
LOCAL = '/content/isac_fw5/'
TEST = '2025/2026'
D_FW, L_FW = os.path.join(DRIVE, 'datos_parquet', 'fw1'), os.path.join(LOCAL, 'datos_parquet', 'fw1')
D_FW5, D_SAL5 = os.path.join(DRIVE, 'datos_parquet', 'fw5'), os.path.join(DRIVE, 'salidas', 'fw5')
os.makedirs(D_FW5, exist_ok=True); os.makedirs(D_SAL5, exist_ok=True)

# 1. Copia a Drive (solo archivos generados aquí; los enlaces no se copian)
n = 0
for src_dir, dst_dir in [(L_FW, D_FW5), (os.path.join(LOCAL, 'salidas'), D_SAL5)]:
    for f in os.listdir(src_dir):
        s = os.path.join(src_dir, f)
        if os.path.isfile(s) and not os.path.islink(s):
            shutil.copy2(s, os.path.join(dst_dir, f)); n += 1
print(f'--- (X5-1) Copiados a Drive: {n} archivos → datos_parquet/fw5/ y salidas/fw5/')

# 2. Reproducción: en las temporadas de entrenamiento, los tramos de fw5 deben ser idénticos a los de fw1
mid_ent = set(pd.read_parquet(os.path.join(D_FW, 'r4_partidos.parquet'))['match_id'])
filas = []
for arch in ['r4_tramos_v21', 'r6_tramos', 'r6b_tramos', 'r6c_tramos']:
    a = pd.read_parquet(os.path.join(D_FW, f'{arch}.parquet'))
    b = pd.read_parquet(os.path.join(D_FW5, f'{arch}.parquet'))
    k = [c for c in ['metrica', 'match_id', 'team', 'e5', 'e5b', 'cuarto'] if c in a.columns and c in b.columns]
    j = a.merge(b[b['match_id'].isin(mid_ent)], on=k, how='outer', suffixes=('_fw1', '_fw5'), indicator=True)
    igual = np.isclose(j['num_fw1'].fillna(-1), j['num_fw5'].fillna(-1)) & np.isclose(j['den_fw1'].fillna(-1), j['den_fw5'].fillna(-1))
    nuevos = b.loc[~b['match_id'].isin(mid_ent), 'match_id'].nunique()
    filas.append(dict(archivo=arch, tramos_fw1=len(a), solo_en_uno=int((j['_merge'] != 'both').sum()), iguales_pct=100 * igual.mean(),
                      partidos_nuevos_2025_26=nuevos))
V = pd.DataFrame(filas)
print('Reproducción del entrenamiento (debe ser solo_en_uno = 0 e iguales_pct = 100.0 en los cuatro):')
print(V.round(3).to_string(index=False))

# 3. Residuos con las cinco temporadas (insumo del plan C)
res = []
for pre in ['r4', 'r6', 'r6b', 'r6c']:
    r = pd.read_parquet(os.path.join(D_FW5, f'{pre}_residuos.parquet'), columns=['metrica', 'match_id'])
    res.append(dict(residuos=pre, metricas=r['metrica'].nunique(), partidos=r['match_id'].nunique(),
                    partidos_2025_26=r.loc[~r['match_id'].isin(mid_ent), 'match_id'].nunique()))
print('Residuos de la receta reajustada (5 temporadas):'); print(pd.DataFrame(res).to_string(index=False))

OK = bool((V['solo_en_uno'] == 0).all() and (V['iguales_pct'] > 99.999).all() and (V['partidos_nuevos_2025_26'] > 300).all())
with open(os.path.join(D_FW5, 'validacion_ok.json'), 'w') as f:
    json.dump(dict(ok=OK, tabla=V.to_dict('records')), f, indent=1, default=str)
print('VALIDACIÓN:', 'OK → se puede correr el plan C' if OK else 'FALLÓ → NO correr el plan C; mandar esta salida a Claude')


PC - Plan C (Principal)

In [ ]:
# =====================================================================
# PC v1.1 · Plan C: ¿el cambio de técnico causa el cambio de estilo? (corrida ÚNICA)
# Preregistro: claude/preregistro_planC.md (congelado 2026-10-15, 13:03) · Framework v1.3, sección T
# Sesión 9 · 2026-10-15 · Hackathon ISAC 2026, Track 2
# Antes: celda 1 (Drive) y la extensión X5 completa con "VALIDACIÓN: OK". Lee datos_parquet/fw5/ (residuos de la receta
# reajustada con 2021/22-2025/26) y de fw1 la tabla de técnicos, el Elo, el Elo de xG y R5 (ruido de un partido).
# 2026/27 no entra. Tarda ~5-15 min.
# v1.1 (antes de correr con datos reales): incertidumbre por bloques equipo-temporada (relevos y placebos juntos);
#      la v1.0 trataba a los placebos como fijos y daba falsos positivos por rasgo en la prueba sin efecto.
# =====================================================================

#[PC v1.1] -------------------------------------------------------------
import os, json, time, math
from datetime import datetime, timezone, timedelta
import numpy as np
import pandas as pd

REABRIR_POR_ERROR = False      # solo por un ERROR DE CÓDIGO; se anota en el preregistro con fecha y motivo
MOTIVO = ''

DRIVE = '/content/drive/MyDrive/isac2026/'
DP = os.path.join(DRIVE, 'datos_parquet'); FW1 = os.path.join(DP, 'fw1'); FW5 = os.path.join(DP, 'fw5')
SAL = os.path.join(DRIVE, 'salidas', 'fw5'); os.makedirs(SAL, exist_ok=True)
MX = timezone(timedelta(hours=-6)); t_ini = time.time()
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)

# ---------- Configuración congelada (preregistro, secciones 2 a 6) ----------
TEMPS = ['2021/2022', '2022/2023', '2023/2024', '2024/2025', '2025/2026']
ENT, TEST = TEMPS[:4], TEMPS[4]
K, K_SENS = 10, 5                       # P-32
LOOK = 10                               # crisis: caída y suerte en los 10 partidos previos
CAL, CAL_SENS = 0.75, 0.5               # P-30
N_CTRL, INTERINO = 3, 3                 # hasta 3 placebos por relevo · D-5
B, SEMILLA, ALFA = 2000, 2026, 0.05
NUCLEO = ['posesion', 'riesgo_pase', 'field_tilt', 'pase_largo', 'saque_meta_corto', 'presion_alta', 'directness',
          'intensidad_ppda', 'recuperacion_tras_perdida', 'progresion', 'contragolpe_rival',
          'pases_hasta_ultimo', 'paso_ultimo', 'toque_area', 'mezcla_progresion', 'entrada_central', 'centros',
          'contragolpe_propio', 'obv_transicion_pct', 'banda_izquierda', 'mezcla_transicion', 'tiro_por_secuencia', 'obv_ultimo_tercio']
SECUND = ['contrapresion', 'centro_alto', 'libertad_encarar', 'reparto_conduccion', 'paso_medio', 'contra_a_area',
          'conduccion_larga', 'tiro_area', 'centro_tras_regate', 'jugadores_contra', 'pase_pico', 'tasa_pase_area']
ESFUERZO = ['contrapresion', 'intensidad_ppda', 'recuperacion_tras_perdida']
PRINCIPAL_SAQUE = 'saque_meta_corto'

# ---------- Seguridad ----------
with open(os.path.join(FW5, 'validacion_ok.json')) as f:
    assert json.load(f)['ok'], 'La extensión X5 no pasó la validación: NO correr el plan C.'
marca = os.path.join(SAL, 'pc_apertura.json')
YA = os.path.exists(marca)
if YA:
    print('AVISO: el plan C ya se corrió el', json.load(open(marca))['fecha_hora'])
    assert REABRIR_POR_ERROR and MOTIVO, 'Reabrir solo por un error de código: REABRIR_POR_ERROR = True y MOTIVO escrito.'

# ---------- 1. Insumos ----------
RS = pd.concat([pd.read_parquet(os.path.join(FW5, f'{p}_residuos.parquet'), columns=['metrica', 'match_id', 'team', 'den', 'res'])
                for p in ['r4', 'r6', 'r6b', 'r6c']], ignore_index=True)
TODOS = NUCLEO + SECUND
faltan = [m for m in TODOS if m not in set(RS['metrica'])]
assert not faltan, f'Faltan rasgos en los residuos: {faltan}'
RS = RS[RS['metrica'].isin(TODOS)]
G = RS.assign(we=RS['den'] * RS['res']).groupby(['metrica', 'match_id', 'team'], as_index=False)[['we', 'den']].sum()

ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)].copy()
ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy()
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ptp['torneo'] = ptp['temporada'] + np.where(ptp['fecha'].dt.month >= 7, '-A', '-C')
largo = ptp.groupby('etapa_id')['match_id'].nunique()
elo = pd.read_parquet(os.path.join(FW1, 'elo_equipo_partido_e2.parquet'))[['match_id', 'team', 'elo_pre']]
ex = pd.read_parquet(os.path.join(FW1, 'elo_xg_partido.parquet'))
exl = pd.concat([ex.assign(team=ex.local, xg_pre=ex.eloxg_local_pre), ex.assign(team=ex.visita, xg_pre=ex.eloxg_visita_pre)])[['match_id', 'team', 'xg_pre']]
S = (ptp[['match_id', 'team', 'fecha', 'temporada', 'torneo', 'tec_p', 'etapa_id']].merge(elo, on=['match_id', 'team'], how='left')
     .merge(exl, on=['match_id', 'team'], how='left').sort_values(['team', 'fecha', 'match_id']).reset_index(drop=True))
assert not S.duplicated(['match_id', 'team']).any()
R5 = pd.read_parquet(os.path.join(FW1, 'r5_confiabilidad.parquet')).set_index('metrica')
print('--- (1) Insumos ---')
print(f"Equipo-partidos {len(S):,} (2021/22-2025/26) · sin Elo {int(S.elo_pre.isna().sum())} · sin Elo de xG {int(S.xg_pre.isna().sum())} · "
      f"rasgos: núcleo {len(NUCLEO)}, secundarios {len(SECUND)}")
cob = G.merge(S[['match_id', 'team', 'temporada']], on=['match_id', 'team']).groupby(['temporada'])['match_id'].nunique()
print('Partidos con residuos por temporada:', cob.to_dict())

# ---------- 2. Series por equipo y rasgo (sumas acumuladas para medias ponderadas por ventana) ----------
SER = {}
for team, g in S.groupby('team', sort=False):
    g = g.reset_index(drop=True)
    gg = G[G['team'] == team].pivot_table(index='match_id', columns='metrica', values=['we', 'den'], aggfunc='sum')
    we = gg['we'].reindex(index=g['match_id'], columns=TODOS).fillna(0.0).values
    dn = gg['den'].reindex(index=g['match_id'], columns=TODOS).fillna(0.0).values
    SER[team] = dict(g=g, cw=np.vstack([np.zeros(len(TODOS)), np.cumsum(we, 0)]), cd=np.vstack([np.zeros(len(TODOS)), np.cumsum(dn, 0)]))

def media(s, a, b):
    """Media ponderada del residuo de cada rasgo en las posiciones [a, b)."""
    w = s['cd'][b] - s['cd'][a]
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(w > 0, (s['cw'][b] - s['cw'][a]) / w, np.nan)

def puntos(k, deriva=False):
    """Relevos y placebos con su ruptura por rasgo y sus covariables de crisis."""
    filas = []
    for team, s in SER.items():
        g = s['g']; n = len(g); et = g['etapa_id'].values; tor = g['torneo'].values
        for p in range(max(k, LOOK), n - (2 * k if deriva else k) + 1):
            mercado = tor[p] != tor[p - 1]
            if et[p] != et[p - 1]:
                if largo[et[p - 1]] <= INTERINO or not (et[p - k:p] == et[p - 1]).all(): continue
                if deriva and not (et[p:p + 2 * k] == et[p]).all(): continue
                tipo = 'relevo'
            else:
                fin = p + (2 * k if deriva else k)
                if not (et[p - k:fin] == et[p]).all(): continue
                tipo = 'placebo'
            pre, A = media(s, p - k, p), media(s, p, p + k)
            fila = dict(team=team, p=p, tipo=tipo, mercado=bool(mercado), temporada=g.at[p, 'temporada'], fecha=g.at[p, 'fecha'],
                        sale=g.at[p - 1, 'tec_p'], entra=g.at[p, 'tec_p'], interino=bool(largo[et[p]] <= INTERINO),
                        nivel=g.at[p, 'elo_pre'], caida=g.at[p, 'elo_pre'] - g.at[p - LOOK, 'elo_pre'],
                        xg_caida=g.at[p, 'xg_pre'] - g.at[p - LOOK, 'xg_pre'])
            fila['suerte'] = fila['caida'] - fila['xg_caida']
            if deriva:
                Bm = media(s, p + k, p + 2 * k)
                fila.update(pre=pre, A=A, Bm=Bm, d=Bm - A)
            else:
                fila['d'] = A - pre
            filas.append(fila)
    return pd.DataFrame(filas)

# ---------- 3. Emparejamiento por crisis ----------
def emparejar(D, cal, covs=('caida', 'suerte')):
    pl = D[D.tipo == 'placebo'].dropna(subset=list(covs)).reset_index()
    sd = pl[list(covs)].std().values
    X = pl[list(covs)].values / sd
    out = {}
    for i, r in D[D.tipo == 'relevo'].iterrows():
        x = np.array([r[c] for c in covs], float) / sd
        if not np.isfinite(x).all(): out[i] = []; continue
        m = (pl['mercado'].values == r['mercado']) & (pl['team'].values != r['team']) & (np.abs(X - x) <= cal).all(1)
        c = pl[m].assign(dist=np.sqrt(((X[m] - x) ** 2).sum(1))).sort_values('dist').drop_duplicates('team')
        out[i] = list(c['index'].values[:N_CTRL])
    return out

def zmat(D, rasgos, absoluto=True):
    """d estandarizado por la desviación de d en TODOS los placebos del mismo tipo (con o sin mercado)."""
    idx = [TODOS.index(m) for m in rasgos]
    d = np.vstack(D['d'].values)[:, idx]
    Z = np.full_like(d, np.nan)
    for merc in [True, False]:
        mp = (D['tipo'].values == 'placebo') & (D['mercado'].values == merc)
        sd = np.nanstd(d[mp], 0, ddof=1)
        mm = D['mercado'].values == merc
        Z[mm] = d[mm] / sd
    return np.abs(Z) if absoluto else Z

def comparar(D, emp, Z, sub=None):
    """Diferencia por relevo: valor del relevo − media de sus placebos emparejados (o de todos los placebos del tipo)."""
    filas = []
    for i, c in emp.items():
        if sub is not None and not sub(D.loc[i]): continue
        if c == 'todos':
            c = list(np.where((D['tipo'].values == 'placebo') & (D['mercado'].values == D.at[i, 'mercado']))[0])
        if not len(c): continue
        filas.append((i, Z[i] - np.nanmean(Z[c], 0)))
    return filas

# ---------- Inferencia: remuestreo por bloques equipo-temporada (relevos Y placebos juntos) ----------
# Los placebos son ventanas que se traslapan dentro de pocas etapas: tratarlos como fijos subestima la incertidumbre
# (lo detectó la prueba con datos inventados antes de correr). Cada repetición sortea bloques equipo-temporada con
# reemplazo; cada relevo y cada placebo pesa lo que su bloque salió sorteado; los conjuntos emparejados se mantienen.
rng = np.random.default_rng(SEMILLA)
def preparar(D, emp):
    D = D.copy(); D['bloque'] = D['team'] + ' | ' + D['temporada']
    cod = pd.factorize(D['bloque'])[0]
    ev = np.array(sorted(emp)); ctrl = -np.ones((len(ev), N_CTRL), int)
    todos = [isinstance(emp[i], str) for i in ev]
    for r, i in enumerate(ev):
        if not todos[r]: c = emp[i][:N_CTRL]; ctrl[r, :len(c)] = c
    W = rng.multinomial(cod.max() + 1, np.full(cod.max() + 1, 1 / (cod.max() + 1)), size=B)[:, cod].astype(float)
    return dict(D=D, ev=ev, ctrl=ctrl, todos=np.array(todos), W=np.vstack([np.ones(len(D)), W]),
                pl=(D['tipo'].values == 'placebo'), merc=D['mercado'].values.astype(bool))

def _z(Pp, rasgos, absoluto, w):
    d = np.vstack(Pp['D']['d'].values)[:, [TODOS.index(m) for m in rasgos]]
    Z = np.full_like(d, np.nan)
    for merc in [True, False]:
        mp = Pp['pl'] & (Pp['merc'] == merc) & (w > 0)
        mu = np.nansum(w[mp, None] * d[mp], 0) / np.sum(w[mp, None] * np.isfinite(d[mp]), 0)
        sd = np.sqrt(np.nansum(w[mp, None] * (d[mp] - mu) ** 2, 0) / (np.sum(w[mp, None] * np.isfinite(d[mp]), 0) - 1))
        Z[Pp['merc'] == merc] = d[Pp['merc'] == merc] / sd
    return np.abs(Z) if absoluto else Z

def _dif(Pp, Z, w, sub, modo):
    """Diferencia media (ponderada por el sorteo) relevo − sus placebos, por columna de Z."""
    ev, ctrl = Pp['ev'], Pp['ctrl']
    ok = sub.copy()
    if modo == 'todos' or modo == 'crisis_vs_todos':
        ref = {}
        for merc in [True, False]:
            mp = Pp['pl'] & (Pp['merc'] == merc)
            ref[merc] = np.nansum(w[mp, None] * Z[mp], 0) / np.sum(w[mp, None] * np.isfinite(Z[mp]), 0)
        R_ = np.vstack([ref[m] for m in Pp['merc'][ev]])
    if modo == 'todos':
        val = Z[ev] - R_
    else:
        wc = np.where(ctrl >= 0, w[np.maximum(ctrl, 0)], 0.0)
        Zc = Z[np.maximum(ctrl, 0)]                                    # (eventos, 3, rasgos)
        num = np.nansum(wc[:, :, None] * Zc, 1); den = np.sum(wc[:, :, None] * np.isfinite(Zc), 1)
        with np.errstate(invalid='ignore', divide='ignore'):
            mc = num / den
        ok &= wc.sum(1) > 0
        val = (mc - R_) if modo == 'crisis_vs_todos' else (Z[ev] - mc)
    we = w[ev] * ok
    return np.nansum(we[:, None] * val, 0) / np.sum(we[:, None] * np.isfinite(val), 0), int(ok.sum())

def inferir(Pp, rasgos, absoluto=True, sub=None, modo='emparejado', indice=True):
    """Devuelve el valor observado, IC95%, p (aprox. normal con el error del remuestreo) y relevos usados.
    indice=True: una sola columna = media de |z| sobre los rasgos (respuesta global); False: una columna por rasgo."""
    sub = np.ones(len(Pp['ev']), bool) if sub is None else np.array([sub(Pp['D'].loc[i]) for i in Pp['ev']])
    sub &= ~Pp['todos'] if modo != 'todos' else sub
    out = []
    for b in range(B + 1):
        w = Pp['W'][b]; Z = _z(Pp, rasgos, absoluto, w)
        if indice: Z = np.nanmean(Z, 1, keepdims=True)
        v, n = _dif(Pp, Z, w, sub, modo); out.append(v)
        if b == 0: n0 = n
    out = np.array(out); obs, bs = out[0], out[1:]
    se = np.nanstd(bs, 0, ddof=1)
    p = np.array([math.erfc(abs(o / s) / math.sqrt(2)) if s > 0 else np.nan for o, s in zip(obs, se)])
    return dict(obs=obs, lo=np.nanquantile(bs, .025, 0), hi=np.nanquantile(bs, .975, 0), p=p, n=n0)

def indice(Zfila): return np.nanmean(Zfila)

# ---------- 4. Corrida principal y conteos ----------
D10 = puntos(K)
EMP = emparejar(D10, CAL)
Zn = zmat(D10, NUCLEO)
rel = D10[D10.tipo == 'relevo']
con = [i for i, c in EMP.items() if len(c)]
print(f'\n--- (2) Relevos con {K} partidos a cada lado (2021/22-2025/26) ---')
print(f"Relevos: {len(rel)} · entre torneos (mercado): {int(rel.mercado.sum())} · a media temporada: {int((~rel.mercado).sum())} · "
      f"entrante interino: {int(rel.interino.sum())} · de 2025/26: {int((rel.temporada == TEST).sum())}")
print(f"Placebos: {int((D10.tipo == 'placebo').sum()):,} · SOPORTE COMÚN: relevos con ≥ 1 placebo en crisis parecida: {len(con)} de {len(rel)} "
      f"(con 3: {sum(len(c) >= 3 for c in EMP.values())})")
bal = pd.DataFrame({'relevos': rel.loc[con, ['caida', 'suerte', 'nivel']].mean(),
                    'sus placebos': pd.concat([D10.loc[EMP[i], ['caida', 'suerte', 'nivel']] for i in con]).mean(),
                    'todos los placebos': D10[D10.tipo == 'placebo'][['caida', 'suerte', 'nivel']].mean()})
print('Balance de la crisis (puntos de Elo): debe parecerse relevos ≈ sus placebos:'); print(bal.T.round(1).to_string())

# ---------- 5. Respuesta global (sí / no) ----------
In = np.array([indice(z) for z in Zn])
def global_(D, emp, rasgos, sub=None, Pp=None, modo='emparejado'):
    Pp = Pp or preparar(D, emp)
    r = inferir(Pp, rasgos, sub=sub, modo=modo)
    Zr = np.nanmean(_z(Pp, rasgos, True, np.ones(len(D))), 1)
    ev = [i for i in Pp['ev'] if (sub is None or sub(D.loc[i])) and (modo == 'todos' or len(emp[i]))]
    base = np.nanmean(Zr[ev]) - r['obs'][0]
    return dict(relevos=r['n'], dif=r['obs'][0], ic_lo=r['lo'][0], ic_hi=r['hi'][0], p=r['p'][0],
                mas_que_placebos_pct=100 * r['obs'][0] / base if base else np.nan)

P10 = preparar(D10, EMP)
G0 = global_(D10, EMP, NUCLEO, Pp=P10)
print(f"\n========== (3) RESPUESTA GLOBAL: ¿el relevo mueve el estilo más que una crisis parecida sin relevo? ==========")
print(f"Índice de ruptura (media de |d| estandarizado sobre {len(NUCLEO)} rasgos): relevos − placebos = {G0['dif']:+.3f} "
      f"IC95% [{G0['ic_lo']:+.3f}, {G0['ic_hi']:+.3f}] · {G0['relevos']} relevos · los relevos se mueven {G0['mas_que_placebos_pct']:+.0f}% más")
print('RESPUESTA:', 'SÍ, el técnico mueve el estilo (el IC excluye 0)' if G0['ic_lo'] > 0 else
      ('NO se distingue de una crisis sin relevo (el IC incluye 0)' if G0['ic_hi'] >= 0 else 'los relevos se mueven MENOS'))

D5 = puntos(K_SENS); EMP5 = emparejar(D5, CAL)
filas = [dict(version='PRINCIPAL (k 10 · crisis 0.75 · núcleo)', **G0),
         dict(version='placebos sin emparejar', **global_(D10, EMP, NUCLEO, Pp=P10, modo='todos')),
         dict(version='tolerancia 0.5', **global_(D10, emparejar(D10, CAL_SENS), NUCLEO)),
         dict(version='crisis + nivel de Elo', **global_(D10, emparejar(D10, CAL, ('caida', 'suerte', 'nivel')), NUCLEO)),
         dict(version='k = 5', **global_(D5, EMP5, NUCLEO)),
         dict(version='núcleo + secundarios', **global_(D10, EMP, NUCLEO + SECUND, Pp=P10)),
         dict(version='sin entrante interino', **global_(D10, EMP, NUCLEO, sub=lambda r: not r['interino'], Pp=P10)),
         dict(version='solo entre torneos (mercado)', **global_(D10, EMP, NUCLEO, sub=lambda r: r['mercado'], Pp=P10)),
         dict(version='solo a media temporada', **global_(D10, EMP, NUCLEO, sub=lambda r: not r['mercado'], Pp=P10)),
         dict(version='RÉPLICA: relevos del entrenamiento', **global_(D10, EMP, NUCLEO, sub=lambda r: r['temporada'] in ENT, Pp=P10)),
         dict(version='RÉPLICA: relevos de 2025/26', **global_(D10, EMP, NUCLEO, sub=lambda r: r['temporada'] == TEST, Pp=P10))]
SENS = pd.DataFrame(filas)
print('\nSensibilidades y réplica (misma lectura: IC que excluye 0 = el relevo mueve más):')
print(SENS.drop(columns='p').round(3).to_string(index=False))

# ---------- 6. Crisis frente a calma (apuesta 3 de Claude frente a la hipótesis de Arturo) ----------
rc = inferir(P10, NUCLEO, modo='crisis_vs_todos')
cm = (rc['obs'][0], rc['lo'][0], rc['hi'][0])
print('\n--- (4) ¿Una crisis mueve el estilo sin cambiar de técnico? Placebos en crisis − placebos de cualquier equipo ---')
print(f"Diferencia del índice: {cm[0]:+.3f} IC95% [{cm[1]:+.3f}, {cm[2]:+.3f}] · Arturo: ≈ 0 · Claude: > 0")

# ---------- 7. Detalle por rasgo (Holm principal, Benjamini-Hochberg sensibilidad) ----------
rr = inferir(P10, NUCLEO, indice=False)
rs = inferir(P10, NUCLEO, absoluto=False, indice=False)
dr = np.vstack(D10['d'].values)[:, [TODOS.index(m) for m in NUCLEO]]
evs = [i for i in P10['ev'] if len(EMP[i])]
det = []
for j, m in enumerate(NUCLEO):
    sig = np.array([dr[i, j] - np.nanmean(dr[EMP[i], j]) for i in evs])
    det.append(dict(rasgo=m, relevos=rr['n'], dif_z=rr['obs'][j], ic_lo=rr['lo'][j], ic_hi=rr['hi'][j], p=rr['p'][j],
                    cambio_con_signo=np.nanmean(sig), relevos_que_suben_pct=100 * np.nanmean(sig > 0)))
DT = pd.DataFrame(det).sort_values('p').reset_index(drop=True)
mT = len(DT)
holm, seguir = [], True
for i, p in enumerate(DT['p']):
    ok = seguir and p <= ALFA / (mT - i); holm.append(ok); seguir = ok
DT['holm'] = holm
kmax = max([i + 1 for i, p in enumerate(DT['p']) if p <= ALFA * (i + 1) / mT], default=0)
DT['bh'] = [i < kmax for i in range(mT)]
DT['lectura'] = np.where(DT['holm'], 'SE AFIRMA', np.where(DT['bh'], 'señal probable', '—'))
print('\n--- (5) Detalle por rasgo (k = 10, crisis 0.75): ¿cuánto más se mueve en un relevo que en una crisis sin relevo? ---')
print('  dif_z en desviaciones del ruido de placebo · cambio_con_signo en la unidad del rasgo (dirección media del relevo frente a sus placebos)')
print(DT.round(4).to_string(index=False))

# ---------- 8. Deriva tras el partido 10 (E-9, sección 11) ----------
DD = puntos(K, deriva=True); EMPD = emparejar(DD, CAL); PD_ = preparar(DD, EMPD)
r_ = inferir(PD_, NUCLEO); gd = (r_['obs'][0], r_['lo'][0], r_['hi'][0])
rdv = inferir(PD_, NUCLEO, indice=False)
print(f'\n--- (6) Deriva: partidos 11-20 frente a 1-10 del técnico nuevo, contra placebos en crisis parecida ---')
print(f"Relevos con 20 partidos del mismo técnico: {int((DD.tipo == 'relevo').sum())} · índice de deriva relevos − placebos: "
      f"{gd[0]:+.3f} IC95% [{gd[1]:+.3f}, {gd[2]:+.3f}]")
def direccion(D, j):
    pre, A, Bm = (np.vstack(D[c].values)[:, j] for c in ['pre', 'A', 'Bm'])
    liga = np.abs(Bm) < np.abs(A); previo = np.abs(Bm - pre) < np.abs(A - pre)
    cat = np.select([liga & previo, liga, previo], ['hacia ambos', 'hacia la liga (moderación)', 'hacia el técnico anterior (herencia)'],
                    'se aleja de ambos')
    return pd.Series(cat[np.isfinite(A) & np.isfinite(Bm)]).value_counts(normalize=True) * 100
filas = []
for m in [PRINCIPAL_SAQUE] + [x for x in NUCLEO if x != PRINCIPAL_SAQUE]:
    j = TODOS.index(m); jn = NUCLEO.index(m)
    mm, lo, hi, n = rdv['obs'][jn], rdv['lo'][jn], rdv['hi'][jn], rdv['n']
    dr_ = direccion(DD[DD.tipo == 'relevo'], j); dp_ = direccion(DD[DD.tipo == 'placebo'], j)
    filas.append(dict(rasgo=m, relevos=n, deriva_dif_z=mm, ic_lo=lo, ic_hi=hi,
                      moderacion_relevos=dr_.get('hacia la liga (moderación)', 0) + dr_.get('hacia ambos', 0),
                      moderacion_placebos=dp_.get('hacia la liga (moderación)', 0) + dp_.get('hacia ambos', 0),
                      herencia_relevos=dr_.get('hacia el técnico anterior (herencia)', 0) + dr_.get('hacia ambos', 0),
                      se_aleja_relevos=dr_.get('se aleja de ambos', 0)))
DV = pd.DataFrame(filas)
print('Por rasgo (moderación = el perfil de 11-20 queda más cerca de la liga que el de 1-10; los placebos dan la base por puro ruido):')
print(DV.round(2).to_string(index=False))

# ---------- 9. Efecto "técnico nuevo" con signo: esfuerzo (hipótesis de Arturo) ----------
re_ = inferir(P10, ESFUERZO, absoluto=False, indice=False)
de = np.vstack(D10['d'].values)[:, [TODOS.index(m) for m in ESFUERZO]]
filas = []
for j, m in enumerate(ESFUERZO):
    unidades = np.nanmean([de[i, j] - np.nanmean(de[EMP[i], j]) for i in evs])
    banda = R5.loc[m, 'sigma_partido'] / np.sqrt(5) if m in R5.index else np.nan
    filas.append(dict(rasgo=m, relevos=re_['n'], cambio_z=re_['obs'][j], ic_lo=re_['lo'][j], ic_hi=re_['hi'][j], cambio_unidades=unidades,
                      banda_5=banda, relevos_que_suben_pct=100 * np.nanmean([de[i, j] > 0 for i in evs]),
                      placebos_que_suben_pct=100 * np.nanmean(de[(D10.tipo == 'placebo').values, j] > 0)))
DE = pd.DataFrame(filas)
print('\n--- (7) Efecto "técnico nuevo" con signo en rasgos de esfuerzo (positivo = más esfuerzo tras el relevo) ---')
print('  esfuerzo = sube en casi todos los relevos · instrucción = depende del técnico · banda_5 = ruido de un bloque de 5 partidos (R5)')
print(DE.round(3).to_string(index=False))

# ---------- 10. América ----------
am = [i for i in rel.index if 'Am' in rel.at[i, 'team'] and 'rica' in rel.at[i, 'team']]
Zs_am = _z(P10, NUCLEO, False, np.ones(len(D10)))
if am:
    print('\n--- (8) Relevos de América (índice de ruptura y los 3 rasgos que más se movieron; z con signo) ---')
    for i in am:
        z = Zs_am[i]; top = np.argsort(-np.abs(np.nan_to_num(z)))[:3]
        cs = np.nanmean(In[EMP[i]]) if len(EMP[i]) else np.nan
        print(f"  {rel.at[i, 'fecha'].date()} · {rel.at[i, 'sale']} → {rel.at[i, 'entra']} · índice {In[i]:.2f} (sus placebos {cs:.2f}; "
              f"{len(EMP[i])} placebos) · " + ', '.join(f'{NUCLEO[t]} {z[t]:+.1f}' for t in top))

# ---------- 11. Apuestas del preregistro ----------
ahora = datetime.now(MX).strftime('%Y-%m-%d %H:%M')
with open(marca, 'w') as f_:
    json.dump(dict(fecha_hora=ahora, reapertura=YA, motivo=MOTIVO, relevos=int(len(rel)), soporte=len(con)), f_)
sa = DT.set_index('rasgo'); sv = DV.set_index('rasgo').loc[PRINCIPAL_SAQUE]
cat_sv = max([('moderacion', sv['moderacion_relevos']), ('herencia', sv['herencia_relevos']), ('se aleja', sv['se_aleja_relevos'])], key=lambda t: t[1])[0]
r_ent = SENS.set_index('version').loc['RÉPLICA: relevos de 2025/26']
AP = pd.DataFrame([
    dict(apuesta='C1. Global: IC excluye 0 y los relevos se mueven 10-50% más', valor=f"{G0['dif']:+.3f} [{G0['ic_lo']:+.3f}, {G0['ic_hi']:+.3f}] · {G0['mas_que_placebos_pct']:+.0f}%",
         cumple=bool(G0['ic_lo'] > 0 and 10 <= G0['mas_que_placebos_pct'] <= 50)),
    dict(apuesta='C2. Saque corto pasa Holm; posesión no', valor=f"saque {sa.loc['saque_meta_corto', 'lectura']} · posesión {sa.loc['posesion', 'lectura']}",
         cumple=bool(sa.loc['saque_meta_corto', 'holm'] and not sa.loc['posesion', 'holm'])),
    dict(apuesta='C3. Crisis mueve algo sin relevo (crisis − calma > 0) y menos que el relevo', valor=f"{cm[0]:+.3f} [{cm[1]:+.3f}, {cm[2]:+.3f}]",
         cumple=bool(cm[1] > 0 and G0['dif'] > 0)),
    dict(apuesta='C4. Deriva del saque: relevos > placebos y dirección dominante = moderación',
         valor=f"{sv['deriva_dif_z']:+.2f} [{sv['ic_lo']:+.2f}, {sv['ic_hi']:+.2f}] · dominante: {cat_sv}",
         cumple=bool(sv['ic_lo'] > 0 and cat_sv == 'moderacion')),
    dict(apuesta='C5. Esfuerzo: positivo pero < 1 banda; contrapresión con IC que incluye 0',
         valor=' · '.join(f"{r.rasgo[:12]} {r.cambio_unidades:+.2f} (banda {r.banda_5:.2f})" for r in DE.itertuples()),
         cumple=bool((DE['cambio_unidades'] > 0).all() and (DE['cambio_unidades'] < DE['banda_5']).all()
                     and DE.set_index('rasgo').loc['contrapresion', 'ic_lo'] <= 0)),
    dict(apuesta='C6. Réplica 2025/26: mismo signo global', valor=f"{r_ent['dif']:+.3f} [{r_ent['ic_lo']:+.3f}, {r_ent['ic_hi']:+.3f}] ({int(r_ent['relevos'])} relevos)",
         cumple=bool(np.sign(r_ent['dif']) == np.sign(G0['dif']))),
    dict(apuesta='ARTURO. Sin relevo, una crisis no cambia el estilo (crisis ≈ calma)', valor=f"{cm[0]:+.3f} [{cm[1]:+.3f}, {cm[2]:+.3f}]",
         cumple=bool(cm[1] <= 0 <= cm[2]))])
print(f'\n========== (9) APUESTAS DEL PREREGISTRO · corrida del {ahora} ==========')
print(AP.to_string(index=False))

# ---------- Guardado ----------
SENS.to_csv(os.path.join(SAL, 'pc_global.csv'), index=False); DT.to_csv(os.path.join(SAL, 'pc_rasgos.csv'), index=False)
DV.to_csv(os.path.join(SAL, 'pc_deriva.csv'), index=False); DE.to_csv(os.path.join(SAL, 'pc_esfuerzo.csv'), index=False)
AP.to_csv(os.path.join(SAL, 'pc_apuestas.csv'), index=False)
ev = rel.drop(columns=['d']).assign(placebos=[len(EMP[i]) for i in rel.index], indice=In[rel.index])
ev.to_csv(os.path.join(SAL, 'pc_relevos.csv'), index=False)
print(f'\nPC v1.1 terminó en {time.time() - t_ini:.0f} s. Guardado: salidas/fw5/pc_*.csv y pc_apertura.json ({ahora}).')

Plan A (resultados con cambio de tecnico)

In [ ]:
# =====================================================================
# PA v1.1 · Plan A (secundario): ¿cambiar de técnico mejora los resultados? (corrida ÚNICA)
# Diseño: framework v1.3, T (plan A) · V-T (P-30, P-31) · VT-1 (2025/26 se suma tras abrir el holdout)
# Agregados de la sesión 9: inferencia por bloques equipo-temporada (como PC v1.1) y puente con el plan C (correlación, no causa).
# Sesión 9 · 2026-10-15 · Hackathon ISAC 2026, Track 2
# Antes: celda 1 (Drive). Lee de fw1 la tabla de técnicos, el Elo y el Elo de xG; de salidas/fw5 los relevos del plan C.
# 2026/27 no entra. Tarda ~1-3 min.
# v1.1 (antes de correr con datos reales): sensibilidad nueva con controles sin relevo en los 20 partidos previos. La prueba con
#      datos inventados mostró que los controles con relevo reciente (aún en implantación) le restan efecto al relevo: la
#      versión principal (P-30, sin cambios) es conservadora si el técnico tiene efecto.
# =====================================================================

# [PA v1.1] -------------------------------------------------------------
import os, json, time, math
from datetime import datetime, timezone, timedelta
import numpy as np
import pandas as pd

REABRIR_POR_ERROR = False
MOTIVO = ''
APUESTA_ARTURO ='A-1: el relevo sube el Elo más que la regresión a la media solo si cambia el estilo: en el puente, r > 0 y la mitad que más cambió el estilo tiene más efecto que la que menos cambió (caso Solari→Ortiz) · A-2: entre torneos y a media temporada no se distinguen (sus IC se traslapan)'            # escribir aquí tu apuesta ANTES de correr (una línea)

DRIVE = '/content/drive/MyDrive/isac2026/'
DP = os.path.join(DRIVE, 'datos_parquet'); FW1 = os.path.join(DP, 'fw1')
SAL = os.path.join(DRIVE, 'salidas', 'fw5'); os.makedirs(SAL, exist_ok=True)
MX = timezone(timedelta(hours=-6)); t_ini = time.time()
pd.set_option('display.width', 250)

TEMPS = ['2021/2022', '2022/2023', '2023/2024', '2024/2025', '2025/2026']
ENT, TEST = TEMPS[:4], TEMPS[4]
VENT, JORN, INTERINO = 10, 2, 3          # T-1 · controles: mismo torneo, jornada ±2 · D-5
CAL, CAL_SENS, N_CTRL = 0.75, 0.5, 3     # P-30
EDAD_CONTROL = 20                        # v1.1, sensibilidad: controles cuyo técnico lleva ≥ 20 partidos en t = 0
MIN_EVENTOS, EMD_MAX = 20, 35.0          # P-31
CALENT = pd.Timestamp('2021-10-01')
B, SEMILLA = 2000, 2026
COV = ['nivel', 'caida', 'suerte']
A_PUNTOS = math.log(10) / 400 * 0.25 * 3 * 17     # puntos por torneo (17 partidos) por cada punto de Elo, como V-T

assert APUESTA_ARTURO.strip(), 'Escribe tu apuesta en APUESTA_ARTURO antes de correr.'
marca = os.path.join(SAL, 'pa_apertura.json'); YA = os.path.exists(marca)
if YA:
    print('AVISO: el plan A ya se corrió el', json.load(open(marca))['fecha_hora'])
    assert REABRIR_POR_ERROR and MOTIVO, 'Reabrir solo por un error de código.'

# ---------- 1. Partidos de cada equipo con técnico, Elo y Elo de xG ----------
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)].copy()
ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy()
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ptp['torneo'] = ptp['temporada'] + np.where(ptp['fecha'].dt.month >= 7, '-A', '-C')
largo = ptp.groupby('etapa_id')['match_id'].nunique()
elo = pd.read_parquet(os.path.join(FW1, 'elo_equipo_partido_e2.parquet'))[['match_id', 'team', 'elo_pre', 'elo_post']]
ex = pd.read_parquet(os.path.join(FW1, 'elo_xg_partido.parquet'))
exl = pd.concat([ex.assign(team=ex.local, xg_pre=ex.eloxg_local_pre, xg_post=ex.eloxg_local_post),
                 ex.assign(team=ex.visita, xg_pre=ex.eloxg_visita_pre, xg_post=ex.eloxg_visita_post)])[['match_id', 'team', 'xg_pre', 'xg_post']]
S = (ptp[['match_id', 'team', 'fecha', 'temporada', 'torneo', 'tec_p', 'etapa_id']].merge(elo, on=['match_id', 'team'], how='left')
     .merge(exl, on=['match_id', 'team'], how='left').sort_values(['team', 'fecha', 'match_id']).reset_index(drop=True))
assert not S.duplicated(['match_id', 'team']).any()
S['jornada'] = S.groupby(['team', 'torneo']).cumcount() + 1
print('--- (1) Insumos ---')
print(f"Equipo-partidos {len(S):,} · sin Elo {int(S.elo_post.isna().sum())} · sin Elo de xG {int(S.xg_post.isna().sum())}")

# ---------- 2. Ventanas: covariables en t = 0 y desenlace a t + 10 (copia de V-T) ----------
filas = []
for team, g in S.groupby('team', sort=False):
    g = g.reset_index(drop=True); n = len(g); et, tp = g['etapa_id'].values, g['temporada'].values
    for j in range(VENT, n - VENT + 1):
        a, b = j - VENT, j + VENT - 1
        if tp[a] != tp[b]: continue
        if et[j] != et[j - 1]:
            if largo[et[j - 1]] <= INTERINO or not (et[a:j] == et[j - 1]).all(): continue
            tipo = 'relevo'
        else:
            if not (et[a:b + 1] == et[j]).all(): continue
            tipo = 'control'
        filas.append(dict(team=team, tipo=tipo, fecha=g.at[j, 'fecha'], torneo=g.at[j, 'torneo'], jornada=g.at[j, 'jornada'],
                          temporada=tp[j], sale=g.at[j - 1, 'tec_p'], entra=g.at[j, 'tec_p'], mercado=g.at[j, 'torneo'] != g.at[j - 1, 'torneo'],
                          interino=bool(largo[et[j]] <= INTERINO), calent=g.at[a, 'fecha'] < CALENT,
                          viejo=bool(j >= EDAD_CONTROL and (et[j - EDAD_CONTROL:j] == et[j]).all()),
                          nivel=g.at[j, 'elo_pre'], caida=g.at[j, 'elo_pre'] - g.at[a, 'elo_pre'],
                          suerte=(g.at[j, 'elo_pre'] - g.at[a, 'elo_pre']) - (g.at[j, 'xg_pre'] - g.at[a, 'xg_pre']),
                          out=g.at[b, 'elo_post'] - g.at[j, 'elo_pre'], out_xg=g.at[b, 'xg_post'] - g.at[j, 'xg_pre']))
V = pd.DataFrame(filas).dropna(subset=COV + ['out', 'out_xg']).reset_index(drop=True)
V['bloque'] = V['team'] + ' | ' + V['temporada']
W = V[V.tipo == 'control']; SD = W[COV].std()
Wv = W[COV].values / SD.values; W_idx = W.index.values

def emparejar(cal, solo_viejos=False):
    out = {}
    for i, r in V[V.tipo == 'relevo'].iterrows():
        x = r[COV].values.astype(float) / SD.values
        m = (W['torneo'].values == r['torneo']) & (np.abs(W['jornada'].values - r['jornada']) <= JORN) & \
            (W['team'].values != r['team']) & (np.abs(Wv - x) <= cal).all(1) & (W['viejo'].values if solo_viejos else True)
        c = W[m].assign(d=np.sqrt(((Wv[m] - x) ** 2).sum(1))).sort_values('d').drop_duplicates('team')
        out[i] = list(c.index[:N_CTRL])
    return out
EMP = emparejar(CAL)
REL = V[V.tipo == 'relevo'].copy(); REL['controles'] = [len(EMP[i]) for i in REL.index]
print(f"\n--- (2) Relevos con 10 partidos antes y 10 después en la misma temporada: {len(REL)} · con control: {int((REL.controles > 0).sum())} "
      f"(con 3: {int((REL.controles >= 3).sum())}) · de 2025/26: {int((REL.temporada == TEST).sum())} · entre torneos: {int(REL.mercado.sum())}")
ok = REL.index[REL.controles > 0]
bal = pd.DataFrame({'relevos con control': REL.loc[ok, COV].mean(),
                    'sus controles': pd.concat([V.loc[EMP[i], COV] for i in ok]).mean(), 'todos los controles': W[COV].mean()})
print('Balance (puntos de Elo; relevos ≈ sus controles):'); print(bal.T.round(1).to_string())

# ---------- 3. Inferencia por bloques equipo-temporada ----------
rng = np.random.default_rng(SEMILLA)
cod = pd.factorize(V['bloque'])[0]
WB = np.vstack([np.ones(len(V)), rng.multinomial(cod.max() + 1, np.full(cod.max() + 1, 1 / (cod.max() + 1)), size=B)[:, cod]])

def efecto(col, emp, sub=None, naive=False):
    ev = [i for i in emp if (len(emp[i]) or naive) and (sub is None or sub(V.loc[i]))]
    y = V[col].values
    C = -np.ones((len(ev), N_CTRL), int)
    for r, i in enumerate(ev): C[r, :len(emp[i])] = emp[i][:N_CTRL]
    vals = []
    for w in WB:
        we = w[ev]
        if naive:
            v = y[ev]
        else:
            wc = np.where(C >= 0, w[np.maximum(C, 0)], 0.0)
            mc = (wc * y[np.maximum(C, 0)]).sum(1) / np.where(wc.sum(1) > 0, wc.sum(1), np.nan)
            v = y[ev] - mc; we = we * (wc.sum(1) > 0)
        vals.append(np.nansum(we * v) / np.sum(we * np.isfinite(v)))
    vals = np.array(vals); se = np.nanstd(vals[1:], ddof=1)
    return dict(relevos=len(ev), efecto=vals[0], ic_lo=np.nanquantile(vals[1:], .025), ic_hi=np.nanquantile(vals[1:], .975),
                EMD=2.8 * se, puntos_por_torneo=vals[0] * A_PUNTOS)

print('\n========== (3) ¿Cambiar de técnico mejora los resultados? Cambio de Elo en los 10 partidos siguientes ==========')
fil = [dict(version='SIN controles (lo que se ve a simple vista)', **efecto('out', EMP, naive=True)),
       dict(version='PLAN A: relevo − controles (tolerancia 0.75)', **efecto('out', EMP)),
       dict(version='PLAN B: Elo de xG, relevo − controles', **efecto('out_xg', EMP))]
G = pd.DataFrame(fil)
print(G.round(2).to_string(index=False))
pa = G.iloc[1]; n_ok = int(pa['relevos'])
print(f"Regla P-31 (≥ {MIN_EVENTOS} relevos con control y EMD ≤ {EMD_MAX}): {n_ok} relevos · EMD {pa['EMD']:.1f} → "
      + ('VIVE' if n_ok >= MIN_EVENTOS and pa['EMD'] <= EMD_MAX else 'NO VIVE: el plan A solo es descriptivo'))
print('  efecto en puntos de Elo; puntos_por_torneo = traducido a puntos de liga en 17 partidos (0.0043 por punto de Elo por partido)')

E5 = emparejar(CAL_SENS)
fil = [dict(version='tolerancia 0.5', **efecto('out', E5)),
       dict(version='controles sin relevo en los 20 previos', **efecto('out', emparejar(CAL, solo_viejos=True))),
       dict(version='sin ventana en el calentamiento del Elo', **efecto('out', EMP, sub=lambda r: not r['calent'])),
       dict(version='sin entrante interino', **efecto('out', EMP, sub=lambda r: not r['interino'])),
       dict(version='entre torneos (mercado)', **efecto('out', EMP, sub=lambda r: r['mercado'])),
       dict(version='a media temporada', **efecto('out', EMP, sub=lambda r: not r['mercado'])),
       dict(version='RÉPLICA: entrenamiento', **efecto('out', EMP, sub=lambda r: r['temporada'] in ENT)),
       dict(version='RÉPLICA: 2025/26', **efecto('out', EMP, sub=lambda r: r['temporada'] == TEST)),
       dict(version='Elo de xG, entre torneos', **efecto('out_xg', EMP, sub=lambda r: r['mercado'])),
       dict(version='Elo de xG, a media temporada', **efecto('out_xg', EMP, sub=lambda r: not r['mercado']))]
SENS = pd.DataFrame(fil)
print('\nSensibilidades:'); print(SENS.round(2).to_string(index=False))

# ---------- 4. Puente con el plan C: ¿los que más cambiaron el estilo mejoraron más? (correlación entre relevos, no causa) ----------
print('\n--- (4) Puente con el plan C: ruptura de estilo frente a efecto en resultados (por relevo) ---')
rp = os.path.join(SAL, 'pc_relevos.csv')
PUENTE = pd.DataFrame()
if os.path.exists(rp):
    PCr = pd.read_csv(rp, parse_dates=['fecha'])[['team', 'fecha', 'indice']]
    y, yx = V['out'].values, V['out_xg'].values
    REL['efecto'] = [y[i] - np.mean(y[EMP[i]]) if len(EMP[i]) else np.nan for i in REL.index]
    REL['efecto_xg'] = [yx[i] - np.mean(yx[EMP[i]]) if len(EMP[i]) else np.nan for i in REL.index]
    PUENTE = REL.merge(PCr, on=['team', 'fecha'], how='inner').dropna(subset=['efecto', 'indice'])
    if len(PUENTE) >= 10:
        rs = []
        for _ in range(B):
            s = PUENTE.sample(len(PUENTE), replace=True, random_state=int(rng.integers(1e9)))
            rs.append(s['indice'].corr(s['efecto']))
        r0 = PUENTE['indice'].corr(PUENTE['efecto']); rx = PUENTE['indice'].corr(PUENTE['efecto_xg'])
        print(f"Relevos en ambos planes: {len(PUENTE)} · correlación ruptura de estilo – efecto en Elo: {r0:+.2f} "
              f"IC95% [{np.nanquantile(rs, .025):+.2f}, {np.nanquantile(rs, .975):+.2f}] · con el Elo de xG: {rx:+.2f}")
        PUENTE['ruptura'] = np.where(PUENTE['indice'] >= PUENTE['indice'].median(), 'mitad que más cambió', 'mitad que menos cambió')
        print(PUENTE.groupby('ruptura')[['efecto', 'efecto_xg']].agg(['mean', 'size']).round(1).to_string())
    else:
        r0 = np.nan; print('Muy pocos relevos en común con el plan C:', len(PUENTE))
else:
    r0 = np.nan; print('No se encontró salidas/fw5/pc_relevos.csv (correr antes el plan C).')

# ---------- 5. América ----------
am = REL[REL['team'].str.contains('Am[eé]rica', regex=True)]
if len(am):
    print('\n--- (5) Relevos de América (Elo en los 10 partidos siguientes; un relevo aislado es anécdota) ---')
    y = V['out'].values
    for i, r in am.iterrows():
        c = EMP[i]; mc = np.mean(y[c]) if len(c) else np.nan
        print(f"  {r['fecha'].date()} · {r['sale']} → {r['entra']} · relevo {r['out']:+.0f} · sus {len(c)} controles {mc:+.0f} · "
              f"efecto {r['out'] - mc:+.0f} Elo (≈ {(r['out'] - mc) * A_PUNTOS:+.1f} puntos por torneo) · caída previa {r['caida']:+.0f}, suerte {r['suerte']:+.0f}")

# ---------- 6. Apuestas ----------
ahora = datetime.now(MX).strftime('%Y-%m-%d %H:%M')
with open(marca, 'w') as f_:
    json.dump(dict(fecha_hora=ahora, reapertura=YA, motivo=MOTIVO, apuesta_arturo=APUESTA_ARTURO), f_)
s0, sx = G.iloc[0], G.iloc[2]; sm = SENS.set_index('version')
AP = pd.DataFrame([
    dict(apuesta='A1. Con controles: efecto 0 a +15 Elo e IC con 0', valor=f"{pa['efecto']:+.1f} [{pa['ic_lo']:+.1f}, {pa['ic_hi']:+.1f}]",
         cumple=bool(0 <= pa['efecto'] <= 15 and pa['ic_lo'] <= 0 <= pa['ic_hi'])),
    dict(apuesta='A2. Sin controles: +10 a +25 Elo (regresión a la media)', valor=f"{s0['efecto']:+.1f}", cumple=bool(10 <= s0['efecto'] <= 25)),
    dict(apuesta='A3. Elo de xG: efecto ≈ 0 (IC con 0)', valor=f"{sx['efecto']:+.1f} [{sx['ic_lo']:+.1f}, {sx['ic_hi']:+.1f}]",
         cumple=bool(sx['ic_lo'] <= 0 <= sx['ic_hi'])),
    dict(apuesta='A4. Ruptura de estilo y resultados: |r| < 0.2', valor=f"{r0:+.2f}", cumple=bool(abs(r0) < 0.2) if np.isfinite(r0) else False),
    dict(apuesta='A5. Mercado y media temporada no se distinguen (IC se traslapan)',
         valor=f"{sm.loc['entre torneos (mercado)', 'efecto']:+.1f} frente a {sm.loc['a media temporada', 'efecto']:+.1f}",
         cumple=bool(sm.loc['entre torneos (mercado)', 'ic_hi'] >= sm.loc['a media temporada', 'ic_lo'] and
                     sm.loc['a media temporada', 'ic_hi'] >= sm.loc['entre torneos (mercado)', 'ic_lo'])),
    dict(apuesta='ARTURO: ' + APUESTA_ARTURO, valor='se lee con la tabla (3)', cumple=None)])
print(f'\n========== (6) APUESTAS · corrida del {ahora} =========='); print(AP.to_string(index=False))

G.to_csv(os.path.join(SAL, 'pa_global.csv'), index=False); SENS.to_csv(os.path.join(SAL, 'pa_sensibilidades.csv'), index=False)
AP.to_csv(os.path.join(SAL, 'pa_apuestas.csv'), index=False); REL.to_csv(os.path.join(SAL, 'pa_relevos.csv'), index=False)
if len(PUENTE): PUENTE.to_csv(os.path.join(SAL, 'pa_puente.csv'), index=False)
print(f'\nPA v1.1 terminó en {time.time() - t_ini:.0f} s. Guardado: salidas/fw5/pa_*.csv y pa_apertura.json ({ahora}).')

In [ ]:
# =====================================================================
# CR v1.0 · Exploratorio (NO preregistrado): tipos de crisis, quién despide, el caso Solari y viabilidad de la compatibilidad
# Sesión 9 · 2026-10-15 · Hackathon ISAC 2026, Track 2 · Base del "protocolo de relevo"
# Antes: celda 1 (Drive) y el plan A (lee salidas/fw5/pa_relevos.csv y pc_relevos.csv). Solo 2021/22-2025/26. Tarda < 1 min.
# Todo lo de esta celda es DESCRIPTIVO: se reporta como exploración, con su límite.
# =====================================================================

# [CR v1.0] -------------------------------------------------------------
import os, math
import numpy as np
import pandas as pd

DRIVE = '/content/drive/MyDrive/isac2026/'
DP = os.path.join(DRIVE, 'datos_parquet'); FW1 = os.path.join(DP, 'fw1'); SAL = os.path.join(DRIVE, 'salidas', 'fw5')
pd.set_option('display.width', 250)
TEMPS = ['2021/2022', '2022/2023', '2023/2024', '2024/2025', '2025/2026']
VENT, INTERINO, B, SEMILLA = 10, 3, 1000, 2026
rng = np.random.default_rng(SEMILLA)

ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)].copy()
ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy()
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
largo = ptp.groupby('etapa_id')['match_id'].nunique()
elo = pd.read_parquet(os.path.join(FW1, 'elo_equipo_partido_e2.parquet'))[['match_id', 'team', 'elo_pre', 'elo_post']]
ex = pd.read_parquet(os.path.join(FW1, 'elo_xg_partido.parquet'))
exl = pd.concat([ex.assign(team=ex.local, xg_pre=ex.eloxg_local_pre, xg_post=ex.eloxg_local_post),
                 ex.assign(team=ex.visita, xg_pre=ex.eloxg_visita_pre, xg_post=ex.eloxg_visita_post)])[['match_id', 'team', 'xg_pre', 'xg_post']]
S = (ptp[['match_id', 'team', 'fecha', 'temporada', 'tec_p', 'etapa_id']].merge(elo, on=['match_id', 'team'], how='left')
     .merge(exl, on=['match_id', 'team'], how='left').sort_values(['team', 'fecha', 'match_id']).reset_index(drop=True))

# Todas las fechas j con 10 partidos previos del mismo técnico (saliente no interino): ¿hubo relevo en j? y qué pasó después
filas = []
for team, g in S.groupby('team', sort=False):
    g = g.reset_index(drop=True); n = len(g); et, tp = g['etapa_id'].values, g['temporada'].values
    for j in range(VENT, n):
        if not (et[j - VENT:j] == et[j - 1]).all() or largo[et[j - 1]] <= INTERINO: continue
        relevo = et[j] != et[j - 1]
        b = j + VENT - 1
        sin_cambio_despues = b < n and (et[j:b + 1] == et[j]).all() and tp[b] == tp[j]
        caida = g.at[j, 'elo_pre'] - g.at[j - VENT, 'elo_pre']
        juego = g.at[j, 'xg_pre'] - g.at[j - VENT, 'xg_pre']
        filas.append(dict(team=team, fecha=g.at[j, 'fecha'], temporada=tp[j], bloque=team + ' | ' + tp[j], relevo=relevo,
                          nivel=g.at[j, 'elo_pre'], caida=caida, juego=juego, suerte=caida - juego,
                          out=(g.at[b, 'elo_post'] - g.at[j, 'elo_pre']) if (not relevo and sin_cambio_despues) else np.nan,
                          out_xg=(g.at[b, 'xg_post'] - g.at[j, 'xg_pre']) if (not relevo and sin_cambio_despues) else np.nan,
                          sale=g.at[j - 1, 'tec_p'], entra=g.at[j, 'tec_p']))
H = pd.DataFrame(filas).dropna(subset=['nivel', 'caida', 'juego'])
cod = pd.factorize(H['bloque'])[0]

def ols(df, y, xs):
    X = np.column_stack([np.ones(len(df))] + [df[x].values for x in xs]); return np.linalg.lstsq(X, df[y].values, rcond=None)[0]
def logit(df, y, xs):
    X = np.column_stack([np.ones(len(df))] + [df[x].values for x in xs]); yy = df[y].values.astype(float); b = np.zeros(X.shape[1])
    for _ in range(50):
        p = 1 / (1 + np.exp(-X @ b)); g = X.T @ (yy - p); Hs = (X * (p * (1 - p))[:, None]).T @ X
        paso = np.linalg.solve(Hs + 1e-9 * np.eye(len(b)), g); b += paso
        if np.abs(paso).max() < 1e-9: break
    return b
def boot_bloques(df, f):
    c = pd.factorize(df['bloque'])[0]; k = c.max() + 1; out = []
    for _ in range(B):
        w = rng.multinomial(k, np.full(k, 1 / k))[c]; d = df.loc[np.repeat(df.index.values, w)]
        try: out.append(f(d))
        except Exception: pass
    return np.array(out)

# ---------- (a) Viabilidad de la compatibilidad: técnicos entrantes con etapa previa en otro club ----------
print('--- (a) ¿Cuántos relevos tienen un técnico entrante con etapa previa en OTRO club (dentro de los datos, antes del relevo)? ---')
previa = lambda r: ((S['tec_p'] == r['entra']) & (S['team'] != r['team']) & (S['fecha'] < r['fecha'])).sum()
for nombre, arch in [('plan A (con resultados)', 'pa_relevos.csv'), ('plan C (con estilo)', 'pc_relevos.csv')]:
    ruta = os.path.join(SAL, arch)
    if not os.path.exists(ruta): print(f'  {nombre}: no se encontró {arch}'); continue
    R = pd.read_csv(ruta, parse_dates=['fecha'])
    if 'entra' not in R: print(f'  {nombre}: el archivo no trae el técnico entrante'); continue
    R['previa'] = R.apply(previa, axis=1)
    con_ctrl = R['controles'] > 0 if 'controles' in R else (R['placebos'] > 0 if 'placebos' in R else pd.Series(True, index=R.index))
    print(f"  {nombre}: relevos {len(R)} · con control {int(con_ctrl.sum())} · con etapa previa del entrante: {int((R.previa > 0).sum())} "
          f"(con control: {int(((R.previa > 0) & con_ctrl).sum())}; con 10+ partidos previos: {int(((R.previa >= 10) & con_ctrl).sum())})")

# ---------- (b) ¿Las crisis de suerte se recuperan más que las de juego? (equipos SIN relevo en los 10 siguientes) ----------
Ctl = H[~H['relevo'] & H['out'].notna()].copy()
for c in ['caida', 'juego', 'suerte']: Ctl[c + '_10'] = Ctl[c] / 10
bR = ols(Ctl, 'out', ['juego_10', 'suerte_10']); bs = boot_bloques(Ctl, lambda d: ols(d, 'out', ['juego_10', 'suerte_10']))
print('\n--- (b) Equipos sin relevo: cambio de Elo en los 10 partidos siguientes según de dónde vino la caída de los 10 previos ---')
print(f"  Ventanas: {len(Ctl):,}. Por cada 10 puntos de caída POR JUEGO (Elo de xG): {bR[1]:+.2f} Elo después "
      f"[{np.quantile(bs[:, 1], .025):+.2f}, {np.quantile(bs[:, 1], .975):+.2f}] · por cada 10 de caída POR SUERTE: {bR[2]:+.2f} "
      f"[{np.quantile(bs[:, 2], .025):+.2f}, {np.quantile(bs[:, 2], .975):+.2f}]")
print('  Negativo = rebota (una caída de −10 se recupera). Si la suerte rebota más que el juego, la crisis de suerte se corrige sola.')
q = Ctl['caida'].quantile(0.2)
cr = Ctl[Ctl['caida'] <= q].copy()
cr['tipo'] = np.where(cr['suerte'] < cr['juego'], 'más suerte que juego', 'más juego que suerte')
print(f"  Crisis = el 20% de ventanas con mayor caída (≤ {q:.0f} Elo en 10 partidos): {len(cr)} ventanas")
print(cr.groupby('tipo').agg(ventanas=('out', 'size'), caida_media=('caida', 'mean'), suerte_media=('suerte', 'mean'),
                             juego_medio=('juego', 'mean'), rebote_elo=('out', 'mean'), rebote_elo_xg=('out_xg', 'mean')).round(1).to_string())

# ---------- (c) El caso Solari: rebote esperado sin cambio de técnico ----------
print('\n--- (c) Solari → Ortiz: ¿cuánto se esperaba que rebotara América sin cambiar de técnico? (extrapolación) ---')
am = H[H['relevo'] & H['team'].str.contains('Am[eé]rica', regex=True)].sort_values('fecha')
pa = pd.read_csv(os.path.join(SAL, 'pa_relevos.csv'), parse_dates=['fecha']) if os.path.exists(os.path.join(SAL, 'pa_relevos.csv')) else None
for _, r in am.iterrows():
    esp = bR[0] + bR[1] * r['juego'] / 10 + bR[2] * r['suerte'] / 10
    esp_b = bs[:, 0] + bs[:, 1] * r['juego'] / 10 + bs[:, 2] * r['suerte'] / 10
    real = pa.loc[(pa['team'] == r['team']) & (pa['fecha'] == r['fecha']), 'out'] if pa is not None else pd.Series(dtype=float)
    soporte = int(((Ctl['suerte'] <= r['suerte'] + 15) & (Ctl['juego'] <= r['juego'] + 15) & (Ctl['caida'] <= r['caida'] * 0.7)).sum())
    print(f"  {r['fecha'].date()} · {r['sale']} → {r['entra']}: caída {r['caida']:+.0f} (suerte {r['suerte']:+.0f}, juego {r['juego']:+.0f}) · "
          f"rebote esperado sin relevo {esp:+.0f} Elo [{np.quantile(esp_b, .025):+.0f}, {np.quantile(esp_b, .975):+.0f}] · "
          f"real con el técnico nuevo: {('%+.0f' % real.iloc[0]) if len(real) else 'sin ventana completa'} · "
          f"ventanas sin relevo con una crisis comparable: {soporte}")

# ---------- (d) ¿Quién despide? Probabilidad de relevo según la caída por juego, por suerte y el nivel del club ----------
H['juego_10'], H['suerte_10'], H['nivel_100'] = H['juego'] / 10, H['suerte'] / 10, (H['nivel'] - 1500) / 100
bL = logit(H, 'relevo', ['juego_10', 'suerte_10', 'nivel_100']); bl = boot_bloques(H, lambda d: logit(d, 'relevo', ['juego_10', 'suerte_10', 'nivel_100']))
print('\n--- (d) ¿Quién despide? (todas las fechas con 10 partidos del técnico; relevo = el siguiente partido ya es de otro técnico) ---')
print(f"  Fechas: {len(H):,} · relevos: {int(H.relevo.sum())} · tasa base por partido: {100 * H.relevo.mean():.2f}%")
for nom, j in [('cada 10 Elo de caída POR JUEGO', 1), ('cada 10 Elo de caída POR SUERTE', 2), ('cada 100 Elo más de NIVEL del club', 3)]:
    print(f"  {nom}: el riesgo de despido se multiplica por {math.exp(-bL[j] if j < 3 else bL[j]):.2f} "
          f"[{np.exp(-np.quantile(bl[:, j], .975) if j < 3 else np.quantile(bl[:, j], .025)):.2f}, "
          f"{np.exp(-np.quantile(bl[:, j], .025) if j < 3 else np.quantile(bl[:, j], .975)):.2f}]")
print('  Lectura: si la suerte multiplica el riesgo tanto como el juego, las directivas despiden por resultados, no por cómo se juega.')
bI = logit(H.assign(inter=H['suerte_10'] * H['nivel_100']), 'relevo', ['juego_10', 'suerte_10', 'nivel_100', 'inter'])
print(f"  ¿Los clubes de más nivel reaccionan más a la misma caída por suerte? interacción suerte × nivel = {bI[4]:+.3f} "
      '(negativo = sí: a más nivel, menos paciencia con la mala suerte)')
print('\nCR v1.0 terminó. Todo es exploratorio y descriptivo.')

# Sesión 10




R7-00

In [ ]:
# R7a v1.1 · Uso de jugadores, base: minutos (U1), sistema (U2), rotación y continuidad (U3), tipología de cambios
# v1.1 (2026-10-15): expulsiones desde los tramos (no eventos: excluye rojas a la banca y duplicadas); el esperado de la
#   validación descuenta salidas temporales (Player Off → On); tipología S10-11 C: movimiento real = cambio de grupo o
#   cambio de lado izquierda↔derecha (Arturo); posición exacta como sensibilidad; edad vectorizada.
# =====================================================================
# Sesión 10 · 2026-10-15 · Hackathon ISAC 2026, Track 2 · Framework v1.2 §11 · Decisiones S10-02 a S10-10
# Antes: celda 1 (Drive) y R0. Cinco temporadas (fw5, D-1). Lee: tabla de técnicos, alineaciones y eventos crudos
# por partido, Elo (fw1, solo lectura). Escribe SOLO en datos_parquet/fw5/ (r7_*). Tarda ~10-20 min la primera vez.
# Fuentes (S10-06 B): minutos y posiciones = tramos `positions` de las alineaciones; formaciones = `equipo_formations`;
# sustituciones (motivo), expulsiones y fin de cada tiempo = eventos. Validación cruzada obligatoria (bloque 3).
# =====================================================================

# [R7a v1.1] ------------------------------------------------------------
import os, json, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026/')
DP = os.path.join(BASE, 'datos_parquet'); FW1 = os.path.join(DP, 'fw1'); FW5 = os.path.join(DP, 'fw5')
DIR_EV, DIR_LU = os.path.join(DP, 'liga', 'eventos'), os.path.join(DP, 'liga', 'lineups')
REHACER = False
pd.set_option('display.width', 250)
t0 = time.time()

# ---------- Parámetros (decisiones de la sesión 10) ----------
COBERTURA, COB_SENS = 0.80, [0.70, 0.90]   # S10-07 B: jugadores para cubrir el 80% de los minutos (rotación efectiva)
NUCLEO = 0.60                              # S10-07 A (descriptivo): jugador con ≥ 60% de los minutos disponibles
DESCANSO_CORTO = 4                         # S10-05: ≤ 4 días desde el partido de liga anterior
FAVORITO, FAV_SENS = 0.65, [0.60, 0.70]    # S10-05 (matiz de Arturo): puntos esperados del Elo ≥ 0.65
H_ELO = 75                                 # P-16
VENTANA_ROJA = 10                          # S10-09: cambio de formación ≤ 10 min tras una roja = "por expulsión"
SIMULTANEO = 1.0                           # S10-10: movimientos a ±1 min de un cambio cuentan como parte de ese cambio
EDAD_JOVEN = 21                            # menores de 21 el día del partido (se discute en el paso 3)
INICIO = {1: 0.0, 2: 45.0, 3: 90.0, 4: 105.0}   # reloj de inicio de cada periodo (el periodo 5 = penales, fuera)
FAM = {'4': 'linea_4', '3': 'linea_3_5', '5': 'linea_3_5'}
GRUPO = {1: 'POR', 3: 'CEN', 4: 'CEN', 5: 'CEN', 2: 'LAT', 6: 'LAT', 7: 'LAT', 8: 'LAT', 9: 'MCD', 10: 'MCD', 11: 'MCD',
         13: 'INT', 14: 'INT', 15: 'INT', 18: 'MP', 19: 'MP', 20: 'MP', 12: 'EXT', 16: 'EXT', 17: 'EXT', 21: 'EXT',
         22: 'DEL', 23: 'DEL', 24: 'DEL', 25: 'DEL'}   # framework §11 (8 grupos)
LADO = {**{p: 'D' for p in [2, 3, 7, 9, 12, 13, 17, 18, 22]}, **{p: 'I' for p in [5, 6, 8, 11, 15, 16, 20, 21, 24]},
        **{p: 'C' for p in [1, 4, 10, 14, 19, 23, 25]}}   # lado del puesto en StatsBomb (D derecha, I izquierda, C centro)

def mueve(a, b):
    """S10-11 C: movimiento real = cambia de grupo, o pasa de izquierda a derecha (o al revés). Centro↔lado no cuenta
    (suele ser recodificación del dibujo, p. ej. MC → MCI al pasar de 4-4-2 a 4-3-3)."""
    if pd.isna(a) or pd.isna(b): return True
    a, b = int(a), int(b)
    if GRUPO.get(a) != GRUPO.get(b): return True
    return {LADO.get(a), LADO.get(b)} == {'I', 'D'}

def reloj(ts):
    """'01:27:25.790' -> 87.43 minutos de reloj del partido."""
    if not isinstance(ts, str) or not ts: return np.nan
    h, m, s = ts.split(':'); return int(h) * 60 + int(m) + float(s) / 60

def lista(s):
    if isinstance(s, str): return json.loads(s) if s else []
    if s is None or (isinstance(s, float) and np.isnan(s)): return []
    return list(s)

def familia(f):
    return FAM.get(str(int(f))[0], 'otra') if pd.notna(f) else None

# ---------- 1. Partidos (cinco temporadas) ----------
with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
    TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy()
ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ptp['torneo'] = ptp['temporada'] + np.where(ptp['fecha'].dt.month >= 7, '-A', '-C')
assert not ptp.duplicated(['match_id', 'team']).any()
MIDS = sorted(ptp['match_id'].unique())
print(f'--- (1) Partidos: {len(MIDS)} · equipo-partidos {len(ptp)} · temporadas {TEMPS}')

# ---------- 2. Lectura por partido (con caché) ----------
RUTAS = {k: os.path.join(FW5, f'r7_{k}.parquet') for k in ['tramos', 'convocados', 'formaciones', 'subs', 'rojas', 'periodos']}
if all(os.path.exists(r) for r in RUTAS.values()) and not REHACER:
    TRA, CON, FOR, SUB, ROJ, PER = (pd.read_parquet(RUTAS[k]) for k in RUTAS)
    print('Cargado de caché (r7_*):', len(TRA), 'tramos')
else:
    tra, con, form, subs, rojas, per, skills_k = [], [], [], [], [], [], {}
    COLS_E = {'type.name': 'type', 'team.name': 'team', 'period': 'period', 'minute': 'minute', 'second': 'second',
              'player.id': 'player_id', 'substitution.replacement.id': 'repl_id', 'substitution.outcome.name': 'motivo',
              'foul_committed.card.name': 'card_fc', 'bad_behaviour.card.name': 'card_bb'}
    for i, mid in enumerate(MIDS):
        # eventos: fin de periodos, sustituciones, rojas
        r = os.path.join(DIR_EV, f'{mid}.parquet'); disp = set(pq.read_schema(r).names)
        e = pd.read_parquet(r, columns=[c for c in COLS_E if c in disp]).rename(columns=COLS_E)
        for c in COLS_E.values():
            if c not in e: e[c] = np.nan
        e['clk'] = e['minute'] + e['second'] / 60
        he = e[(e['type'] == 'Half End') & e['period'].between(1, 4)].groupby('period')['clk'].max()
        for p, fin in he.items(): per.append((mid, int(p), INICIO[int(p)], float(fin)))
        s = e[e['type'] == 'Substitution']
        subs.append(pd.DataFrame({'match_id': mid, 'team': s['team'].values, 'period': s['period'].values, 'clk': s['clk'].values,
                                  'out_id': s['player_id'].values, 'in_id': s['repl_id'].values, 'motivo': s['motivo'].values}))
        rj = e[e['card_fc'].isin(['Red Card', 'Second Yellow']) | e['card_bb'].isin(['Red Card', 'Second Yellow'])]
        rojas.append(pd.DataFrame({'match_id': mid, 'team': rj['team'].values, 'player_id': rj['player_id'].values,
                                   'period': rj['period'].values, 'clk': rj['clk'].values}))
        # alineaciones: tramos de posición, convocados, formaciones
        lu = pd.read_parquet(os.path.join(DIR_LU, f'{mid}.parquet'))
        for eq, g in lu.groupby('team_name'):
            for fo in lista(g['equipo_formations'].iloc[0]):
                form.append((mid, eq, fo.get('period'), reloj(fo.get('timestamp')), fo.get('reason'), fo.get('formation')))
        for row in lu[['team_name', 'player_id', 'birth_date', 'positions', 'skills']].itertuples(index=False):
            ps = lista(row.positions)
            con.append((mid, row.team_name, row.player_id, row.birth_date, len(ps) > 0))
            for k in (lista(row.skills) if isinstance(row.skills, list) else (json.loads(row.skills) if isinstance(row.skills, str) and row.skills else {})):
                skills_k[k] = skills_k.get(k, 0) + 1
            for p in ps:
                tra.append((mid, row.team_name, row.player_id, p.get('position_id'), p.get('position'),
                            reloj(p.get('from')), p.get('from_period'), reloj(p.get('to')), p.get('to_period'),
                            p.get('start_reason'), p.get('end_reason')))
        if (i + 1) % 300 == 0: print(f'  {i + 1}/{len(MIDS)} partidos · {time.time() - t0:.0f} s')
    TRA = pd.DataFrame(tra, columns=['match_id', 'team', 'player_id', 'position_id', 'position', 'from_clk', 'from_period',
                                     'to_clk', 'to_period', 'start_reason', 'end_reason'])
    CON = pd.DataFrame(con, columns=['match_id', 'team', 'player_id', 'birth_date', 'jugo'])
    FOR = pd.DataFrame(form, columns=['match_id', 'team', 'period', 'clk', 'reason', 'formation'])
    SUB = pd.concat(subs, ignore_index=True); ROJ = pd.concat(rojas, ignore_index=True)
    PER = pd.DataFrame(per, columns=['match_id', 'period', 'ini', 'fin'])
    for k, d in zip(RUTAS, [TRA, CON, FOR, SUB, ROJ, PER]): d.to_parquet(RUTAS[k], index=False)
    print('Claves distintas en `skills` (jugador-partido con esa clave):', skills_k)
print(f'Lectura lista · {time.time() - t0:.0f} s')
print('Motivos de inicio de tramo:', TRA['start_reason'].value_counts().to_dict())
print('Motivos de fin de tramo:', TRA['end_reason'].value_counts().head(12).to_dict())

# ---------- 3. Minutos reales por tramo y validación cruzada (S10-03 B, S10-06 B) ----------
PER = PER[PER['period'].between(1, 4)]
fin_p = PER.set_index(['match_id', 'period'])['fin'].to_dict()
ultimo = PER.groupby('match_id')['period'].max().to_dict()
dur = PER.assign(d=PER['fin'] - PER['ini']).groupby('match_id')['d'].sum()

def minutos(r):
    p0 = int(r.from_period); p1 = int(r.to_period) if pd.notna(r.to_period) else ultimo.get(r.match_id, 2)
    tot = 0.0
    for p in range(p0, min(p1, 4) + 1):
        if (r.match_id, p) not in fin_p: continue
        a = r.from_clk if p == p0 else INICIO[p]
        b = r.to_clk if (p == p1 and pd.notna(r.to_clk)) else fin_p[(r.match_id, p)]
        tot += max(0.0, min(b, fin_p[(r.match_id, p)]) - max(a, INICIO[p]))
    return tot
TRA = TRA[TRA['from_period'].fillna(5) <= 4].copy()
TRA['min'] = [minutos(r) for r in TRA.itertuples(index=False)]
TRA['grupo'] = TRA['position_id'].map(GRUPO)

def min_rango(m, a, b):
    """Minutos reales de juego entre los relojes a y b (b nulo = hasta el final), saltando el descanso."""
    tot = 0.0
    for p in range(1, ultimo.get(m, 2) + 1):
        if (m, p) not in fin_p: continue
        hi = fin_p[(m, p)] if pd.isna(b) else min(b, fin_p[(m, p)])
        tot += max(0.0, hi - max(a, INICIO[p]))
    return tot
# v1.1: expulsiones = tramos que terminan en roja (jugador que estaba en el campo); se comparan con los eventos
RJT = TRA[TRA['end_reason'].isin(['Red Card', 'Second Yellow Card', 'Sent Off'])][['match_id', 'team', 'player_id', 'to_clk', 'to_period']]
RJT = RJT.rename(columns={'to_clk': 'clk', 'to_period': 'period'}).drop_duplicates(['match_id', 'player_id'])
RJT['perdidos'] = [min_rango(m, c, np.nan) for m, c in zip(RJT.match_id, RJT.clk)]
print(f'Expulsiones: eventos {len(ROJ)} · tramos (en el campo) {len(RJT)} · diferencia = rojas a la banca o registradas dos veces')
# v1.1: salidas temporales (Player Off → vuelve con Player On, o lo sustituyen, o no vuelve)
po = TRA[TRA['end_reason'].astype(str).str.startswith('Player Off')][['match_id', 'team', 'player_id', 'to_clk']]
sig = TRA[['match_id', 'player_id', 'from_clk']]
vuelta = po.merge(sig, on=['match_id', 'player_id'])
vuelta = vuelta[vuelta.from_clk >= vuelta.to_clk].groupby(['match_id', 'player_id', 'to_clk'])['from_clk'].min()
SUBn = SUB.assign(out_id=pd.to_numeric(SUB['out_id'], errors='coerce'))
reemp = po.merge(SUBn[['match_id', 'out_id', 'clk']], left_on=['match_id', 'player_id'], right_on=['match_id', 'out_id'])
reemp = reemp[reemp.clk >= reemp.to_clk - 0.1].groupby(['match_id', 'player_id', 'to_clk'])['clk'].min()
po['regreso'] = [np.nanmin([vuelta.get((m, p, t), np.nan), reemp.get((m, p, t), np.nan)])
                 if not (np.isnan(vuelta.get((m, p, t), np.nan)) and np.isnan(reemp.get((m, p, t), np.nan))) else np.nan
                 for m, p, t in zip(po.match_id, po.player_id, po.to_clk)]
po['fuera'] = [min_rango(m, a, b) for m, a, b in zip(po.match_id, po.to_clk, po.regreso)]
print(f"Salidas temporales: {len(po):,} · regresan o son sustituidos {100 * po.regreso.notna().mean():.1f}% · "
      f"minutos fuera: mediana {po.fuera.median():.1f} · p90 {po.fuera.quantile(.9):.1f}")
V = (TRA.groupby(['match_id', 'team'])['min'].sum().rename('min_equipo').reset_index()
     .merge(RJT.groupby(['match_id', 'team'])['perdidos'].sum().reset_index(), how='left')
     .merge(po.groupby(['match_id', 'team'])['fuera'].sum().reset_index(), how='left').fillna({'perdidos': 0, 'fuera': 0}))
V['esperado'] = 11 * V['match_id'].map(dur) - V['perdidos'] - V['fuera']
V['dif'] = V['min_equipo'] - V['esperado']
print('\n--- (3) Validación (v1.1): minutos de jugadores − [11 × duración real − minutos tras roja − salidas temporales] ---')
print(f"|dif| ≤ 1 min: {100 * (V.dif.abs() <= 1).mean():.2f}% · ≤ 3 min: {100 * (V.dif.abs() <= 3).mean():.2f}% · "
      f"mediana {V.dif.median():+.2f} · p1 {V.dif.quantile(.01):+.1f} · p99 {V.dif.quantile(.99):+.1f}")
print('Peores 5:'); print(V.reindex(V.dif.abs().sort_values(ascending=False).index).head(5).round(1).to_string(index=False))
# Entrante: su primer tramo empieza en el minuto de la sustitución (eventos)
SUB['in_id'] = pd.to_numeric(SUB['in_id'], errors='coerce'); SUB['out_id'] = pd.to_numeric(SUB['out_id'], errors='coerce')
prim = TRA.sort_values('from_clk').groupby(['match_id', 'player_id']).first().reset_index()
chk = SUB.merge(prim[['match_id', 'player_id', 'from_clk', 'start_reason']], left_on=['match_id', 'in_id'],
                right_on=['match_id', 'player_id'], how='left')
print(f"Sustituciones cuyo entrante tiene tramo: {100 * chk.from_clk.notna().mean():.2f}% · "
      f"empieza a ±1 min del evento: {100 * ((chk.from_clk - chk.clk).abs() <= 1).mean():.2f}%")
ok_eq = set(map(tuple, ptp[['match_id', 'team']].values)); en_lu = set(map(tuple, V[['match_id', 'team']].values))
print(f'Equipo-partidos de la tabla de técnicos con minutos: {len(ok_eq & en_lu)} de {len(ok_eq)} (nombres de equipo coinciden)')

# ---------- 4. Jugador-partido: minutos, titularidad, edad ----------
JP = TRA.groupby(['match_id', 'team', 'player_id']).agg(minutos=('min', 'sum'), puestos=('position_id', 'nunique'),
                                                        titular=('start_reason', lambda s: (s == 'Starting XI').any())).reset_index()
JP = JP.merge(ptp[['match_id', 'team', 'fecha', 'temporada', 'torneo', 'etapa_id', 'tec_p']], on=['match_id', 'team'], how='inner')
nac = CON.drop_duplicates(['player_id'])[['player_id', 'birth_date']]
JP = JP.merge(nac, on='player_id', how='left')
JP['edad'] = (JP['fecha'] - pd.to_datetime(JP['birth_date'], errors='coerce')).dt.days / 365.25
JP['joven'] = JP['edad'] < EDAD_JOVEN
nconv = CON.groupby(['match_id', 'team']).size().rename('listados')
print(f"\n--- (4) Jugador-partido: {len(JP):,} · sin edad {int(JP.edad.isna().sum())} · edad media ponderada por minutos "
      f"{np.average(JP.edad.dropna(), weights=JP.loc[JP.edad.notna(), 'minutos']):.1f} · minutos de menores de {EDAD_JOVEN}: "
      f"{100 * JP.loc[JP.joven, 'minutos'].sum() / JP.minutos.sum():.1f}%")

# ---------- 5. Formaciones y cambios de sistema (U2; S10-04 C y S10-09) ----------
FOR = FOR[FOR['period'].fillna(5) <= 4].sort_values(['match_id', 'team', 'clk']).copy()
FOR['familia'] = FOR['formation'].map(familia)
FOR['form_prev'] = FOR.groupby(['match_id', 'team'])['formation'].shift(); FOR['fam_prev'] = FOR.groupby(['match_id', 'team'])['familia'].shift()
FOR['cambia_form'] = FOR['form_prev'].notna() & (FOR['formation'] != FOR['form_prev'])
FOR['cambia_fam'] = FOR['fam_prev'].notna() & (FOR['familia'] != FOR['fam_prev'])
roj_clk = RJT.groupby('match_id')['clk'].apply(list).to_dict()      # v1.1: rojas en el campo, de cualquiera de los dos equipos
FOR['tras_roja'] = [any(0 <= c - x <= VENTANA_ROJA for x in roj_clk.get(m, [])) for m, c in zip(FOR.match_id, FOR.clk)]
# línea de tiempo con minutos reales por formación (insumo de la celda de OBV por sistema)
FOR['clk_fin'] = FOR.groupby(['match_id', 'team'])['clk'].shift(-1)
FOR['minutos'] = [min_rango(m, a, b) for m, a, b in zip(FOR.match_id, FOR.clk, FOR.clk_fin)]
cs = FOR[FOR.cambia_form]
print(f"\n--- (5) Cambios de formación: {len(cs):,} · de familia {int(cs.cambia_fam.sum()):,} · "
      f"dentro de {VENTANA_ROJA} min tras una roja {int(cs.tras_roja.sum()):,} ({100 * cs.tras_roja.mean():.1f}%)")

# ---------- 6. Tipología de cada cambio (S10-10; S10-11 C: grupo + lado izq↔der principal, posición exacta sensibilidad) ----------
pos_out = TRA.sort_values('to_clk').groupby(['match_id', 'player_id']).last()['position_id']
pos_in = prim.set_index(['match_id', 'player_id'])['position_id']
TRA = TRA.sort_values(['match_id', 'player_id', 'from_clk'])
TRA['pos_prev'] = TRA.groupby(['match_id', 'player_id'])['position_id'].shift()
tact = TRA[TRA['start_reason'].astype(str).str.contains('Tactical', case=False)][['match_id', 'team', 'player_id', 'from_clk', 'pos_prev', 'position_id']].copy()
tact['real'] = [mueve(a, b) for a, b in zip(tact.pos_prev, tact.position_id)]
SUB = SUB[SUB['period'].between(1, 4)].copy()
filas = []
for r in SUB.itertuples(index=False):
    po_, pi_ = pos_out.get((r.match_id, r.out_id)), pos_in.get((r.match_id, r.in_id))
    t_ = tact[(tact.match_id == r.match_id) & (tact.team == r.team) & ((tact.from_clk - r.clk).abs() <= SIMULTANEO)]
    t_ = t_[~t_.player_id.isin([r.in_id, r.out_id])]
    otros_x, otros_r = t_.player_id.nunique(), t_.loc[t_.real, 'player_id'].nunique()
    f_ = FOR[(FOR.match_id == r.match_id) & (FOR.team == r.team) & FOR.cambia_form & ((FOR.clk - r.clk).abs() <= SIMULTANEO)]
    io_real = mueve(po_, pi_)
    # principal: dibujo solo si cambia la formación Y hay movimiento real (cambio de familia o alguien cambia de grupo/lado)
    if len(f_) and (f_.cambia_fam.any() or otros_r > 0 or io_real): tipo = 'dibujo'
    elif io_real or otros_r > 0: tipo = 'reacomodo'
    else: tipo = 'espejo'
    tipo_x = 'dibujo' if len(f_) else ('espejo' if (po_ == pi_ and otros_x == 0) else 'reacomodo')   # v1.0 = sensibilidad
    filas.append((po_, pi_, otros_r, otros_x, tipo, tipo_x))
SUB[['pos_out', 'pos_in', 'otros_reales', 'otros_exacto', 'tipo', 'tipo_exacto']] = pd.DataFrame(filas, index=SUB.index)
SUB['lesion'] = SUB['motivo'].astype(str).str.contains('Injury', case=False)
# reacomodos sin sustitución: momentos con cambios de posición lejos de cualquier cambio (principal: solo movimientos reales)
sub_clk = SUB.groupby(['match_id', 'team'])['clk'].apply(list).to_dict()
tact['cerca'] = [any(abs(c - x) <= SIMULTANEO for x in sub_clk.get((m, t), [])) for m, t, c in zip(tact.match_id, tact.team, tact.from_clk)]
def momentos(d):
    d = d.sort_values(['match_id', 'team', 'from_clk'])
    d = d.assign(mom=(d.groupby(['match_id', 'team'])['from_clk'].diff().fillna(99) > SIMULTANEO).cumsum())
    return d.groupby(['match_id', 'team'])['mom'].nunique()
REAC = momentos(tact[~tact.cerca & tact.real]).rename('reacomodos_sin_cambio')
REACX = momentos(tact[~tact.cerca]).rename('reacomodos_sin_cambio_exacto')
print('\n--- (6) Tipología de sustituciones tácticas (sin lesión): principal (grupo + lado) frente a sensibilidad (posición exacta) ---')
tac = SUB[~SUB.lesion]
print(pd.concat([tac['tipo'].value_counts(normalize=True).rename('principal'),
                 tac['tipo_exacto'].value_counts(normalize=True).rename('exacta')], axis=1).mul(100).round(1).to_string())
print('Cruce principal × exacta (% de cambios tácticos):')
print((pd.crosstab(tac['tipo'], tac['tipo_exacto'], normalize='all') * 100).round(1).to_string())
idx_ep = V.set_index(['match_id', 'team']).index
print(f"Reacomodos sin sustitución por equipo-partido: principal {REAC.reindex(idx_ep).fillna(0).mean():.2f} · "
      f"exacta {REACX.reindex(idx_ep).fillna(0).mean():.2f}")
print('Principal por temporada (cambios tácticos por equipo-partido):')
print(tac.merge(ptp[['match_id', 'team', 'temporada']], on=['match_id', 'team']).pivot_table(
      index='temporada', columns='tipo', values='clk', aggfunc='size').div(ptp.groupby('temporada').size(), axis=0).round(2).to_string())

# ---------- 7. Equipo-partido: rasgos ----------
EP = ptp[['match_id', 'team', 'lado', 'fecha', 'temporada', 'torneo', 'etapa_id', 'tec_p', 'rol']].copy()
elo = pd.read_parquet(os.path.join(FW1, 'elo_equipo_partido_e2.parquet'))[['match_id', 'team', 'elo_pre', 'elo_rival_pre']]
EP = EP.merge(elo, on=['match_id', 'team'], how='left')
sgn = np.where(EP['lado'] == 'home', 1, -1)
EP['p_esp'] = 1 / (1 + 10 ** (-(EP['elo_pre'] + sgn * H_ELO - EP['elo_rival_pre']) / 400))
EP['favorito'] = EP['p_esp'] >= FAVORITO
EP = EP.sort_values(['team', 'fecha', 'match_id']).reset_index(drop=True)
EP['dias'] = EP.groupby('team')['fecha'].diff().dt.days
EP['descanso_corto'] = EP['dias'] <= DESCANSO_CORTO
tit = JP[JP.titular].groupby(['match_id', 'team'])['player_id'].apply(frozenset).rename('xi')
EP = EP.merge(tit, on=['match_id', 'team'], how='left')
EP['xi_prev'] = EP.groupby('team')['xi'].shift(); EP['etapa_prev'] = EP.groupby('team')['etapa_id'].shift()
EP['rotados'] = [len(a - b) if isinstance(a, frozenset) and isinstance(b, frozenset) and e == ep else np.nan
                 for a, b, e, ep in zip(EP.xi, EP.xi_prev, EP.etapa_id, EP.etapa_prev)]
# Titulares fuera del XI base (los 11 con más titularidades de la etapa en ese torneo): no cuenta dos veces el regreso
xi_base = (JP[JP.titular].groupby(['etapa_id', 'torneo', 'player_id']).size().rename('n').reset_index()
           .sort_values(['etapa_id', 'torneo', 'n', 'player_id'], ascending=[True, True, False, True])
           .groupby(['etapa_id', 'torneo']).head(11).groupby(['etapa_id', 'torneo'])['player_id'].apply(frozenset).to_dict())
EP['fuera_base'] = [len(a - xi_base.get((e, t), frozenset())) if isinstance(a, frozenset) else np.nan
                    for a, e, t in zip(EP.xi, EP.etapa_id, EP.torneo)]
k = ['match_id', 'team']
EP = (EP.merge(FOR[FOR.cambia_form & ~FOR.tras_roja].groupby(k).size().rename('cambios_form_libres').reset_index(), on=k, how='left')
        .merge(FOR[FOR.cambia_fam & ~FOR.tras_roja].groupby(k).size().rename('cambios_fam_libres').reset_index(), on=k, how='left')
        .merge(FOR[FOR.cambia_form & FOR.tras_roja].groupby(k).size().rename('cambios_por_roja').reset_index(), on=k, how='left')
        .merge(FOR.groupby(k)['familia'].first().rename('familia_inicial').reset_index(), on=k, how='left'))
fam_min = FOR.groupby(k + ['familia'])['minutos'].sum().unstack(fill_value=0)
EP = EP.merge(fam_min.div(fam_min.sum(axis=1), axis=0).add_prefix('pct_min_').reset_index(), on=k, how='left')
st = SUB[~SUB.lesion].groupby(k)['tipo'].value_counts().unstack(fill_value=0).add_prefix('subs_').reset_index()
stx = SUB[~SUB.lesion].groupby(k)['tipo_exacto'].value_counts().unstack(fill_value=0).add_prefix('subsx_').reset_index()
EP = (EP.merge(st, on=k, how='left').merge(stx, on=k, how='left')
        .merge(SUB.groupby(k)['lesion'].sum().rename('subs_lesion').reset_index(), on=k, how='left')
        .merge(REAC.reset_index(), on=k, how='left').merge(REACX.reset_index(), on=k, how='left')
        .merge(SUB[~SUB.lesion].groupby(k)['clk'].min().rename('min_primer_cambio').reset_index(), on=k, how='left'))
for c in ['cambios_form_libres', 'cambios_fam_libres', 'cambios_por_roja', 'subs_espejo', 'subs_reacomodo', 'subs_dibujo',
          'subs_lesion', 'reacomodos_sin_cambio', 'subsx_espejo', 'subsx_reacomodo', 'subsx_dibujo', 'reacomodos_sin_cambio_exacto']:
    if c not in EP: EP[c] = 0
    EP[c] = EP[c].fillna(0)
J_ = JP[JP.edad.notna()].assign(em=lambda d: d.edad * d.minutos, mj=lambda d: d.minutos * d.joven)
ed = J_.groupby(k)[['em', 'mj', 'minutos']].sum()
ed = pd.DataFrame({'edad_media': ed.em / ed.minutos, 'pct_min_jovenes': ed.mj / ed.minutos})
EP = EP.merge(ed.reset_index(), on=k, how='left').merge(nconv.reset_index(), on=k, how='left')
EP['banca_completa'] = EP['listados'] >= 21
EP = EP.drop(columns=['xi', 'xi_prev', 'etapa_prev'])
print(f"\n--- (7) Equipo-partido: {len(EP):,} · sin Elo {int(EP.p_esp.isna().sum())} · rotados calculables {int(EP.rotados.notna().sum()):,}")
print('Norma de la liga (media por equipo-partido):')
cols_n = ['rotados', 'fuera_base', 'cambios_form_libres', 'cambios_fam_libres', 'cambios_por_roja', 'subs_espejo', 'subs_reacomodo', 'subs_dibujo',
          'reacomodos_sin_cambio', 'min_primer_cambio', 'edad_media', 'pct_min_jovenes']
print(EP.groupby('temporada')[cols_n].mean().round(2).to_string())
print('\nTitulares rotados según calendario y favoritismo (media · n):')
for v_ in ['rotados', 'fuera_base']:
    print(f'-- {v_}'); print(EP.pivot_table(index='descanso_corto', columns='favorito', values=v_, aggfunc=['mean', 'count']).round(2).to_string())

# ---------- 8. Etapa × torneo: rotación efectiva, núcleo, continuidad (U1, U3) ----------
def cubre(m, q):
    m = np.sort(m)[::-1]; return int(np.searchsorted(np.cumsum(m) / m.sum(), q) + 1)
disp_et = EP.merge(V[['match_id', 'team', 'min_equipo']], on=k, how='left').groupby(['etapa_id', 'torneo'])
jt = JP.groupby(['etapa_id', 'tec_p', 'team', 'torneo', 'player_id'])['minutos'].sum().reset_index()
partidos_et = EP.groupby(['etapa_id', 'torneo'])['match_id'].nunique()
dur_et = EP.assign(d=EP.match_id.map(dur)).groupby(['etapa_id', 'torneo'])['d'].sum()
ET = jt.groupby(['etapa_id', 'tec_p', 'team', 'torneo']).agg(usados=('player_id', 'nunique')).reset_index()
for q in [COBERTURA] + COB_SENS:
    ET[f'rot_ef_{int(q * 100)}'] = [cubre(g.minutos.values, q) for _, g in jt.groupby(['etapa_id', 'tec_p', 'team', 'torneo'])]
ET['partidos'] = ET.set_index(['etapa_id', 'torneo']).index.map(partidos_et)
ET['nucleo'] = [int((g.minutos / dur_et[(e, t)] >= NUCLEO).sum()) for (e, _, _, t), g in jt.groupby(['etapa_id', 'tec_p', 'team', 'torneo'])]
TORNEOS = list(ptp.groupby('torneo')['fecha'].min().sort_values().index); PREV = {t: TORNEOS[i - 1] for i, t in enumerate(TORNEOS) if i}
club_t = JP.groupby(['team', 'torneo'])['player_id'].apply(set).to_dict()
ET['continuidad'] = [g.loc[g.player_id.isin(club_t.get((tm, PREV.get(t)), set())), 'minutos'].sum() / g.minutos.sum()
                     if (tm, PREV.get(t)) in club_t else np.nan
                     for (e, _, tm, t), g in jt.groupby(['etapa_id', 'tec_p', 'team', 'torneo'])]
print(f"\n--- (8) Etapa × torneo: {len(ET)} filas · con ≥ 10 partidos {int((ET.partidos >= 10).sum())}")
print(ET[ET.partidos >= 10][['usados', 'rot_ef_70', 'rot_ef_80', 'rot_ef_90', 'nucleo', 'continuidad']].describe(percentiles=[.1, .5, .9]).round(2).to_string())
print('Correlación entre rotación efectiva 70/80/90:',
      ET[ET.partidos >= 10][['rot_ef_70', 'rot_ef_80', 'rot_ef_90']].corr(method='spearman').round(2).values[[0, 1], [1, 2]].tolist())

# ---------- 9. Guardado (solo fw5) ----------
for nombre, d in [('jugador_partido', JP), ('equipo_partido', EP), ('etapa_torneo', ET), ('validacion', V)]:
    d.to_parquet(os.path.join(FW5, f'r7_{nombre}.parquet'), index=False)
FOR.to_parquet(RUTAS['formaciones'], index=False); SUB.to_parquet(RUTAS['subs'], index=False); TRA.to_parquet(RUTAS['tramos'], index=False)
print(f'\nR7a v1.1 terminó en {time.time() - t0:.0f} s. Guardado en fw5: r7_jugador_partido, r7_equipo_partido, r7_etapa_torneo, '
      'r7_validacion, r7_formaciones, r7_subs, r7_tramos (+ cachés). Pega toda la salida.')

R7a

In [ ]:
# R7a v1.0 · Uso de jugadores, base: minutos (U1), sistema (U2), rotación y continuidad (U3), tipología de cambios
# =====================================================================
# Sesión 10 · 2026-10-15 · Hackathon ISAC 2026, Track 2 · Framework v1.2 §11 · Decisiones S10-02 a S10-10
# Antes: celda 1 (Drive) y R0. Cinco temporadas (fw5, D-1). Lee: tabla de técnicos, alineaciones y eventos crudos
# por partido, Elo (fw1, solo lectura). Escribe SOLO en datos_parquet/fw5/ (r7_*). Tarda ~10-20 min la primera vez.
# Fuentes (S10-06 B): minutos y posiciones = tramos `positions` de las alineaciones; formaciones = `equipo_formations`;
# sustituciones (motivo), expulsiones y fin de cada tiempo = eventos. Validación cruzada obligatoria (bloque 3).
# =====================================================================

# [R7a v1.0] ------------------------------------------------------------
import os, json, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

BASE = globals().get('BASE', '/content/drive/MyDrive/isac2026/')
DP = os.path.join(BASE, 'datos_parquet'); FW1 = os.path.join(DP, 'fw1'); FW5 = os.path.join(DP, 'fw5')
DIR_EV, DIR_LU = os.path.join(DP, 'liga', 'eventos'), os.path.join(DP, 'liga', 'lineups')
REHACER = False
pd.set_option('display.width', 250)
t0 = time.time()

# ---------- Parámetros (decisiones de la sesión 10) ----------
COBERTURA, COB_SENS = 0.80, [0.70, 0.90]   # S10-07 B: jugadores para cubrir el 80% de los minutos (rotación efectiva)
NUCLEO = 0.60                              # S10-07 A (descriptivo): jugador con ≥ 60% de los minutos disponibles
DESCANSO_CORTO = 4                         # S10-05: ≤ 4 días desde el partido de liga anterior
FAVORITO, FAV_SENS = 0.65, [0.60, 0.70]    # S10-05 (matiz de Arturo): puntos esperados del Elo ≥ 0.65
H_ELO = 75                                 # P-16
VENTANA_ROJA = 10                          # S10-09: cambio de formación ≤ 10 min tras una roja = "por expulsión"
SIMULTANEO = 1.0                           # S10-10: movimientos a ±1 min de un cambio cuentan como parte de ese cambio
EDAD_JOVEN = 21                            # menores de 21 el día del partido (se discute en el paso 3)
INICIO = {1: 0.0, 2: 45.0, 3: 90.0, 4: 105.0}   # reloj de inicio de cada periodo (el periodo 5 = penales, fuera)
FAM = {'4': 'linea_4', '3': 'linea_3_5', '5': 'linea_3_5'}
GRUPO = {1: 'POR', 3: 'CEN', 4: 'CEN', 5: 'CEN', 2: 'LAT', 6: 'LAT', 7: 'LAT', 8: 'LAT', 9: 'MCD', 10: 'MCD', 11: 'MCD',
         13: 'INT', 14: 'INT', 15: 'INT', 18: 'MP', 19: 'MP', 20: 'MP', 12: 'EXT', 16: 'EXT', 17: 'EXT', 21: 'EXT',
         22: 'DEL', 23: 'DEL', 24: 'DEL', 25: 'DEL'}   # framework §11 (8 grupos)

def reloj(ts):
    """'01:27:25.790' -> 87.43 minutos de reloj del partido."""
    if not isinstance(ts, str) or not ts: return np.nan
    h, m, s = ts.split(':'); return int(h) * 60 + int(m) + float(s) / 60

def lista(s):
    if isinstance(s, str): return json.loads(s) if s else []
    if s is None or (isinstance(s, float) and np.isnan(s)): return []
    return list(s)

def familia(f):
    return FAM.get(str(int(f))[0], 'otra') if pd.notna(f) else None

# ---------- 1. Partidos (cinco temporadas) ----------
with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
    TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy()
ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ptp['torneo'] = ptp['temporada'] + np.where(ptp['fecha'].dt.month >= 7, '-A', '-C')
assert not ptp.duplicated(['match_id', 'team']).any()
MIDS = sorted(ptp['match_id'].unique())
print(f'--- (1) Partidos: {len(MIDS)} · equipo-partidos {len(ptp)} · temporadas {TEMPS}')

# ---------- 2. Lectura por partido (con caché) ----------
RUTAS = {k: os.path.join(FW5, f'r7_{k}.parquet') for k in ['tramos', 'convocados', 'formaciones', 'subs', 'rojas', 'periodos']}
if all(os.path.exists(r) for r in RUTAS.values()) and not REHACER:
    TRA, CON, FOR, SUB, ROJ, PER = (pd.read_parquet(RUTAS[k]) for k in RUTAS)
    print('Cargado de caché (r7_*):', len(TRA), 'tramos')
else:
    tra, con, form, subs, rojas, per, skills_k = [], [], [], [], [], [], {}
    COLS_E = {'type.name': 'type', 'team.name': 'team', 'period': 'period', 'minute': 'minute', 'second': 'second',
              'player.id': 'player_id', 'substitution.replacement.id': 'repl_id', 'substitution.outcome.name': 'motivo',
              'foul_committed.card.name': 'card_fc', 'bad_behaviour.card.name': 'card_bb'}
    for i, mid in enumerate(MIDS):
        # eventos: fin de periodos, sustituciones, rojas
        r = os.path.join(DIR_EV, f'{mid}.parquet'); disp = set(pq.read_schema(r).names)
        e = pd.read_parquet(r, columns=[c for c in COLS_E if c in disp]).rename(columns=COLS_E)
        for c in COLS_E.values():
            if c not in e: e[c] = np.nan
        e['clk'] = e['minute'] + e['second'] / 60
        he = e[(e['type'] == 'Half End') & e['period'].between(1, 4)].groupby('period')['clk'].max()
        for p, fin in he.items(): per.append((mid, int(p), INICIO[int(p)], float(fin)))
        s = e[e['type'] == 'Substitution']
        subs.append(pd.DataFrame({'match_id': mid, 'team': s['team'].values, 'period': s['period'].values, 'clk': s['clk'].values,
                                  'out_id': s['player_id'].values, 'in_id': s['repl_id'].values, 'motivo': s['motivo'].values}))
        rj = e[e['card_fc'].isin(['Red Card', 'Second Yellow']) | e['card_bb'].isin(['Red Card', 'Second Yellow'])]
        rojas.append(pd.DataFrame({'match_id': mid, 'team': rj['team'].values, 'player_id': rj['player_id'].values,
                                   'period': rj['period'].values, 'clk': rj['clk'].values}))
        # alineaciones: tramos de posición, convocados, formaciones
        lu = pd.read_parquet(os.path.join(DIR_LU, f'{mid}.parquet'))
        for eq, g in lu.groupby('team_name'):
            for fo in lista(g['equipo_formations'].iloc[0]):
                form.append((mid, eq, fo.get('period'), reloj(fo.get('timestamp')), fo.get('reason'), fo.get('formation')))
        for row in lu[['team_name', 'player_id', 'birth_date', 'positions', 'skills']].itertuples(index=False):
            ps = lista(row.positions)
            con.append((mid, row.team_name, row.player_id, row.birth_date, len(ps) > 0))
            for k in (lista(row.skills) if isinstance(row.skills, list) else (json.loads(row.skills) if isinstance(row.skills, str) and row.skills else {})):
                skills_k[k] = skills_k.get(k, 0) + 1
            for p in ps:
                tra.append((mid, row.team_name, row.player_id, p.get('position_id'), p.get('position'),
                            reloj(p.get('from')), p.get('from_period'), reloj(p.get('to')), p.get('to_period'),
                            p.get('start_reason'), p.get('end_reason')))
        if (i + 1) % 300 == 0: print(f'  {i + 1}/{len(MIDS)} partidos · {time.time() - t0:.0f} s')
    TRA = pd.DataFrame(tra, columns=['match_id', 'team', 'player_id', 'position_id', 'position', 'from_clk', 'from_period',
                                     'to_clk', 'to_period', 'start_reason', 'end_reason'])
    CON = pd.DataFrame(con, columns=['match_id', 'team', 'player_id', 'birth_date', 'jugo'])
    FOR = pd.DataFrame(form, columns=['match_id', 'team', 'period', 'clk', 'reason', 'formation'])
    SUB = pd.concat(subs, ignore_index=True); ROJ = pd.concat(rojas, ignore_index=True)
    PER = pd.DataFrame(per, columns=['match_id', 'period', 'ini', 'fin'])
    for k, d in zip(RUTAS, [TRA, CON, FOR, SUB, ROJ, PER]): d.to_parquet(RUTAS[k], index=False)
    print('Claves distintas en `skills` (jugador-partido con esa clave):', skills_k)
print(f'Lectura lista · {time.time() - t0:.0f} s')
print('Motivos de inicio de tramo:', TRA['start_reason'].value_counts().to_dict())
print('Motivos de fin de tramo:', TRA['end_reason'].value_counts().head(12).to_dict())

# ---------- 3. Minutos reales por tramo y validación cruzada (S10-03 B, S10-06 B) ----------
PER = PER[PER['period'].between(1, 4)]
fin_p = PER.set_index(['match_id', 'period'])['fin'].to_dict()
ultimo = PER.groupby('match_id')['period'].max().to_dict()
dur = PER.assign(d=PER['fin'] - PER['ini']).groupby('match_id')['d'].sum()

def minutos(r):
    p0 = int(r.from_period); p1 = int(r.to_period) if pd.notna(r.to_period) else ultimo.get(r.match_id, 2)
    tot = 0.0
    for p in range(p0, min(p1, 4) + 1):
        if (r.match_id, p) not in fin_p: continue
        a = r.from_clk if p == p0 else INICIO[p]
        b = r.to_clk if (p == p1 and pd.notna(r.to_clk)) else fin_p[(r.match_id, p)]
        tot += max(0.0, min(b, fin_p[(r.match_id, p)]) - max(a, INICIO[p]))
    return tot
TRA = TRA[TRA['from_period'].fillna(5) <= 4].copy()
TRA['min'] = [minutos(r) for r in TRA.itertuples(index=False)]
TRA['grupo'] = TRA['position_id'].map(GRUPO)

# Minutos perdidos por expulsión (desde la roja hasta el final)
def perdidos(r):
    p = int(r.period); tot = fin_p.get((r.match_id, p), r.clk) - r.clk
    for q in range(p + 1, ultimo.get(r.match_id, 2) + 1):
        if (r.match_id, q) in fin_p: tot += fin_p[(r.match_id, q)] - INICIO[q]
    return max(0.0, tot)
ROJ = ROJ[ROJ['period'].between(1, 4)].copy()
ROJ['perdidos'] = [perdidos(r) for r in ROJ.itertuples(index=False)]
V = (TRA.groupby(['match_id', 'team'])['min'].sum().rename('min_equipo').reset_index()
     .merge(ROJ.groupby(['match_id', 'team'])['perdidos'].sum().reset_index(), how='left').fillna({'perdidos': 0}))
V['esperado'] = 11 * V['match_id'].map(dur) - V['perdidos']
V['dif'] = V['min_equipo'] - V['esperado']
print('\n--- (3) Validación de minutos por equipo-partido (minutos de jugadores − [11 × duración real − minutos tras roja]) ---')
print(f"|dif| ≤ 1 min: {100 * (V.dif.abs() <= 1).mean():.2f}% · ≤ 3 min: {100 * (V.dif.abs() <= 3).mean():.2f}% · "
      f"mediana {V.dif.median():+.2f} · p1 {V.dif.quantile(.01):+.1f} · p99 {V.dif.quantile(.99):+.1f}")
print('Peores 5:'); print(V.reindex(V.dif.abs().sort_values(ascending=False).index).head(5).round(1).to_string(index=False))
# Entrante: su primer tramo empieza en el minuto de la sustitución (eventos)
SUB['in_id'] = pd.to_numeric(SUB['in_id'], errors='coerce'); SUB['out_id'] = pd.to_numeric(SUB['out_id'], errors='coerce')
prim = TRA.sort_values('from_clk').groupby(['match_id', 'player_id']).first().reset_index()
chk = SUB.merge(prim[['match_id', 'player_id', 'from_clk', 'start_reason']], left_on=['match_id', 'in_id'],
                right_on=['match_id', 'player_id'], how='left')
print(f"Sustituciones cuyo entrante tiene tramo: {100 * chk.from_clk.notna().mean():.2f}% · "
      f"empieza a ±1 min del evento: {100 * ((chk.from_clk - chk.clk).abs() <= 1).mean():.2f}%")
ok_eq = set(map(tuple, ptp[['match_id', 'team']].values)); en_lu = set(map(tuple, V[['match_id', 'team']].values))
print(f'Equipo-partidos de la tabla de técnicos con minutos: {len(ok_eq & en_lu)} de {len(ok_eq)} (nombres de equipo coinciden)')

# ---------- 4. Jugador-partido: minutos, titularidad, edad ----------
JP = TRA.groupby(['match_id', 'team', 'player_id']).agg(minutos=('min', 'sum'), puestos=('position_id', 'nunique'),
                                                        titular=('start_reason', lambda s: (s == 'Starting XI').any())).reset_index()
JP = JP.merge(ptp[['match_id', 'team', 'fecha', 'temporada', 'torneo', 'etapa_id', 'tec_p']], on=['match_id', 'team'], how='inner')
nac = CON.drop_duplicates(['player_id'])[['player_id', 'birth_date']]
JP = JP.merge(nac, on='player_id', how='left')
JP['edad'] = (JP['fecha'] - pd.to_datetime(JP['birth_date'], errors='coerce')).dt.days / 365.25
JP['joven'] = JP['edad'] < EDAD_JOVEN
nconv = CON.groupby(['match_id', 'team']).size().rename('listados')
print(f"\n--- (4) Jugador-partido: {len(JP):,} · sin edad {int(JP.edad.isna().sum())} · edad media ponderada por minutos "
      f"{np.average(JP.edad.dropna(), weights=JP.loc[JP.edad.notna(), 'minutos']):.1f} · minutos de menores de {EDAD_JOVEN}: "
      f"{100 * JP.loc[JP.joven, 'minutos'].sum() / JP.minutos.sum():.1f}%")

# ---------- 5. Formaciones y cambios de sistema (U2; S10-04 C y S10-09) ----------
FOR = FOR[FOR['period'].fillna(5) <= 4].sort_values(['match_id', 'team', 'clk']).copy()
FOR['familia'] = FOR['formation'].map(familia)
FOR['form_prev'] = FOR.groupby(['match_id', 'team'])['formation'].shift(); FOR['fam_prev'] = FOR.groupby(['match_id', 'team'])['familia'].shift()
FOR['cambia_form'] = FOR['form_prev'].notna() & (FOR['formation'] != FOR['form_prev'])
FOR['cambia_fam'] = FOR['fam_prev'].notna() & (FOR['familia'] != FOR['fam_prev'])
roj_clk = ROJ.groupby('match_id')['clk'].apply(list).to_dict()      # rojas de cualquiera de los dos equipos
FOR['tras_roja'] = [any(0 <= c - x <= VENTANA_ROJA for x in roj_clk.get(m, [])) for m, c in zip(FOR.match_id, FOR.clk)]
# línea de tiempo con minutos reales por formación (insumo de la celda de OBV por sistema)
FOR['clk_fin'] = FOR.groupby(['match_id', 'team'])['clk'].shift(-1)
def min_rango(m, a, b):
    tot = 0.0
    for p in range(1, ultimo.get(m, 2) + 1):
        if (m, p) not in fin_p: continue
        hi = fin_p[(m, p)] if pd.isna(b) else min(b, fin_p[(m, p)])
        tot += max(0.0, hi - max(a, INICIO[p]))
    return tot
FOR['minutos'] = [min_rango(m, a, b) for m, a, b in zip(FOR.match_id, FOR.clk, FOR.clk_fin)]
cs = FOR[FOR.cambia_form]
print(f"\n--- (5) Cambios de formación: {len(cs):,} · de familia {int(cs.cambia_fam.sum()):,} · "
      f"dentro de {VENTANA_ROJA} min tras una roja {int(cs.tras_roja.sum()):,} ({100 * cs.tras_roja.mean():.1f}%)")

# ---------- 6. Tipología de cada cambio (S10-10, idea de Arturo: hombre por hombre frente a reacomodo frente a dibujo) ----------
pos_out = TRA.sort_values('to_clk').groupby(['match_id', 'player_id']).last()['position_id']
pos_in = prim.set_index(['match_id', 'player_id'])['position_id']
tact = TRA[TRA['start_reason'].astype(str).str.contains('Tactical', case=False)][['match_id', 'team', 'player_id', 'from_clk']]
SUB = SUB[SUB['period'].between(1, 4)].copy()
filas = []
for r in SUB.itertuples(index=False):
    po = pos_out.get((r.match_id, r.out_id)); pi = pos_in.get((r.match_id, r.in_id))
    t_ = tact[(tact.match_id == r.match_id) & (tact.team == r.team) & ((tact.from_clk - r.clk).abs() <= SIMULTANEO)]
    otros = t_.loc[~t_.player_id.isin([r.in_id, r.out_id]), 'player_id'].nunique()
    f_ = FOR[(FOR.match_id == r.match_id) & (FOR.team == r.team) & FOR.cambia_form & ((FOR.clk - r.clk).abs() <= SIMULTANEO)]
    tipo = 'dibujo' if len(f_) else ('espejo' if (po == pi and otros == 0) else 'reacomodo')
    filas.append((po, pi, otros, tipo))
SUB[['pos_out', 'pos_in', 'otros_movidos', 'tipo']] = pd.DataFrame(filas, index=SUB.index)
SUB['lesion'] = SUB['motivo'].astype(str).str.contains('Injury', case=False)
# reacomodos sin sustitución: momentos con cambios de posición (Tactical) lejos de cualquier cambio
sub_clk = SUB.groupby(['match_id', 'team'])['clk'].apply(list).to_dict()
tact = tact.assign(cerca=[any(abs(c - x) <= SIMULTANEO for x in sub_clk.get((m, t), [])) for m, t, c in zip(tact.match_id, tact.team, tact.from_clk)])
mov = tact[~tact.cerca].sort_values(['match_id', 'team', 'from_clk'])
mov['momento'] = (mov.groupby(['match_id', 'team'])['from_clk'].diff().fillna(99) > SIMULTANEO).cumsum()
REAC = mov.groupby(['match_id', 'team'])['momento'].nunique().rename('reacomodos_sin_cambio')
print('\n--- (6) Tipología de sustituciones (todas / solo tácticas) ---')
print(pd.concat([SUB['tipo'].value_counts(normalize=True).rename('todas'),
                 SUB.loc[~SUB.lesion, 'tipo'].value_counts(normalize=True).rename('tacticas')], axis=1).mul(100).round(1).to_string())
print(f"Reacomodos sin sustitución por equipo-partido: media {REAC.reindex(V.set_index(['match_id', 'team']).index).fillna(0).mean():.2f}")

# ---------- 7. Equipo-partido: rasgos ----------
EP = ptp[['match_id', 'team', 'lado', 'fecha', 'temporada', 'torneo', 'etapa_id', 'tec_p', 'rol']].copy()
elo = pd.read_parquet(os.path.join(FW1, 'elo_equipo_partido_e2.parquet'))[['match_id', 'team', 'elo_pre', 'elo_rival_pre']]
EP = EP.merge(elo, on=['match_id', 'team'], how='left')
sgn = np.where(EP['lado'] == 'home', 1, -1)
EP['p_esp'] = 1 / (1 + 10 ** (-(EP['elo_pre'] + sgn * H_ELO - EP['elo_rival_pre']) / 400))
EP['favorito'] = EP['p_esp'] >= FAVORITO
EP = EP.sort_values(['team', 'fecha', 'match_id']).reset_index(drop=True)
EP['dias'] = EP.groupby('team')['fecha'].diff().dt.days
EP['descanso_corto'] = EP['dias'] <= DESCANSO_CORTO
tit = JP[JP.titular].groupby(['match_id', 'team'])['player_id'].apply(frozenset).rename('xi')
EP = EP.merge(tit, on=['match_id', 'team'], how='left')
EP['xi_prev'] = EP.groupby('team')['xi'].shift(); EP['etapa_prev'] = EP.groupby('team')['etapa_id'].shift()
EP['rotados'] = [len(a - b) if isinstance(a, frozenset) and isinstance(b, frozenset) and e == ep else np.nan
                 for a, b, e, ep in zip(EP.xi, EP.xi_prev, EP.etapa_id, EP.etapa_prev)]
# Titulares fuera del XI base (los 11 con más titularidades de la etapa en ese torneo): no cuenta dos veces el regreso
xi_base = (JP[JP.titular].groupby(['etapa_id', 'torneo', 'player_id']).size().rename('n').reset_index()
           .sort_values(['etapa_id', 'torneo', 'n', 'player_id'], ascending=[True, True, False, True])
           .groupby(['etapa_id', 'torneo']).head(11).groupby(['etapa_id', 'torneo'])['player_id'].apply(frozenset).to_dict())
EP['fuera_base'] = [len(a - xi_base.get((e, t), frozenset())) if isinstance(a, frozenset) else np.nan
                    for a, e, t in zip(EP.xi, EP.etapa_id, EP.torneo)]
k = ['match_id', 'team']
EP = (EP.merge(FOR[FOR.cambia_form & ~FOR.tras_roja].groupby(k).size().rename('cambios_form_libres').reset_index(), on=k, how='left')
        .merge(FOR[FOR.cambia_fam & ~FOR.tras_roja].groupby(k).size().rename('cambios_fam_libres').reset_index(), on=k, how='left')
        .merge(FOR[FOR.cambia_form & FOR.tras_roja].groupby(k).size().rename('cambios_por_roja').reset_index(), on=k, how='left')
        .merge(FOR.groupby(k)['familia'].first().rename('familia_inicial').reset_index(), on=k, how='left'))
fam_min = FOR.groupby(k + ['familia'])['minutos'].sum().unstack(fill_value=0)
EP = EP.merge(fam_min.div(fam_min.sum(axis=1), axis=0).add_prefix('pct_min_').reset_index(), on=k, how='left')
st = SUB[~SUB.lesion].groupby(k)['tipo'].value_counts().unstack(fill_value=0).add_prefix('subs_').reset_index()
EP = (EP.merge(st, on=k, how='left').merge(SUB.groupby(k)['lesion'].sum().rename('subs_lesion').reset_index(), on=k, how='left')
        .merge(REAC.reset_index(), on=k, how='left')
        .merge(SUB[~SUB.lesion].groupby(k)['clk'].min().rename('min_primer_cambio').reset_index(), on=k, how='left'))
for c in ['cambios_form_libres', 'cambios_fam_libres', 'cambios_por_roja', 'subs_espejo', 'subs_reacomodo', 'subs_dibujo',
          'subs_lesion', 'reacomodos_sin_cambio']:
    if c not in EP: EP[c] = 0
    EP[c] = EP[c].fillna(0)
ed = JP[JP.edad.notna()].groupby(k).apply(lambda g: pd.Series({'edad_media': np.average(g.edad, weights=g.minutos),
                                                             'pct_min_jovenes': g.loc[g.joven, 'minutos'].sum() / g.minutos.sum()}))
EP = EP.merge(ed.reset_index(), on=k, how='left').merge(nconv.reset_index(), on=k, how='left')
EP['banca_completa'] = EP['listados'] >= 21
EP = EP.drop(columns=['xi', 'xi_prev', 'etapa_prev'])
print(f"\n--- (7) Equipo-partido: {len(EP):,} · sin Elo {int(EP.p_esp.isna().sum())} · rotados calculables {int(EP.rotados.notna().sum()):,}")
print('Norma de la liga (media por equipo-partido):')
cols_n = ['rotados', 'fuera_base', 'cambios_form_libres', 'cambios_fam_libres', 'cambios_por_roja', 'subs_espejo', 'subs_reacomodo', 'subs_dibujo',
          'reacomodos_sin_cambio', 'min_primer_cambio', 'edad_media', 'pct_min_jovenes']
print(EP.groupby('temporada')[cols_n].mean().round(2).to_string())
print('\nTitulares rotados según calendario y favoritismo (media · n):')
for v_ in ['rotados', 'fuera_base']:
    print(f'-- {v_}'); print(EP.pivot_table(index='descanso_corto', columns='favorito', values=v_, aggfunc=['mean', 'count']).round(2).to_string())

# ---------- 8. Etapa × torneo: rotación efectiva, núcleo, continuidad (U1, U3) ----------
def cubre(m, q):
    m = np.sort(m)[::-1]; return int(np.searchsorted(np.cumsum(m) / m.sum(), q) + 1)
disp_et = EP.merge(V[['match_id', 'team', 'min_equipo']], on=k, how='left').groupby(['etapa_id', 'torneo'])
jt = JP.groupby(['etapa_id', 'tec_p', 'team', 'torneo', 'player_id'])['minutos'].sum().reset_index()
partidos_et = EP.groupby(['etapa_id', 'torneo'])['match_id'].nunique()
dur_et = EP.assign(d=EP.match_id.map(dur)).groupby(['etapa_id', 'torneo'])['d'].sum()
ET = jt.groupby(['etapa_id', 'tec_p', 'team', 'torneo']).agg(usados=('player_id', 'nunique')).reset_index()
for q in [COBERTURA] + COB_SENS:
    ET[f'rot_ef_{int(q * 100)}'] = [cubre(g.minutos.values, q) for _, g in jt.groupby(['etapa_id', 'tec_p', 'team', 'torneo'])]
ET['partidos'] = ET.set_index(['etapa_id', 'torneo']).index.map(partidos_et)
ET['nucleo'] = [int((g.minutos / dur_et[(e, t)] >= NUCLEO).sum()) for (e, _, _, t), g in jt.groupby(['etapa_id', 'tec_p', 'team', 'torneo'])]
TORNEOS = list(ptp.groupby('torneo')['fecha'].min().sort_values().index); PREV = {t: TORNEOS[i - 1] for i, t in enumerate(TORNEOS) if i}
club_t = JP.groupby(['team', 'torneo'])['player_id'].apply(set).to_dict()
ET['continuidad'] = [g.loc[g.player_id.isin(club_t.get((tm, PREV.get(t)), set())), 'minutos'].sum() / g.minutos.sum()
                     if (tm, PREV.get(t)) in club_t else np.nan
                     for (e, _, tm, t), g in jt.groupby(['etapa_id', 'tec_p', 'team', 'torneo'])]
print(f"\n--- (8) Etapa × torneo: {len(ET)} filas · con ≥ 10 partidos {int((ET.partidos >= 10).sum())}")
print(ET[ET.partidos >= 10][['usados', 'rot_ef_70', 'rot_ef_80', 'rot_ef_90', 'nucleo', 'continuidad']].describe(percentiles=[.1, .5, .9]).round(2).to_string())
print('Correlación entre rotación efectiva 70/80/90:',
      ET[ET.partidos >= 10][['rot_ef_70', 'rot_ef_80', 'rot_ef_90']].corr(method='spearman').round(2).values[[0, 1], [1, 2]].tolist())

# ---------- 9. Guardado (solo fw5) ----------
for nombre, d in [('jugador_partido', JP), ('equipo_partido', EP), ('etapa_torneo', ET), ('validacion', V)]:
    d.to_parquet(os.path.join(FW5, f'r7_{nombre}.parquet'), index=False)
FOR.to_parquet(RUTAS['formaciones'], index=False); SUB.to_parquet(RUTAS['subs'], index=False); TRA.to_parquet(RUTAS['tramos'], index=False)
print(f'\nR7a terminó en {time.time() - t0:.0f} s. Guardado en fw5: r7_jugador_partido, r7_equipo_partido, r7_etapa_torneo, '
      'r7_validacion, r7_formaciones, r7_subs, r7_tramos (+ cachés). Pega toda la salida.')

R7u0

In [ ]:
# R7u v1.1 · Rasgos de uso de jugadores en formato R4 y paso por la receta (R4) y la confiabilidad (R5) con el código de R6r
# v1.1 (2026-10-16): R7u-0 usa ruta fija (RAIZ) y freno de Drive montado; la v1.0 intentaba adivinar BASE y falló en Colab
#   (Arturo lo corrigió en su cuaderno; esta versión deja el archivo igual a lo que corrió). R7r-0 y R7r-9 sin cambios.
# =====================================================================
# Sesión 10 · 2026-10-15 · Hackathon ISAC 2026, Track 2 · Decisión S10-12 (7 rasgos) · Cinco temporadas (fw5)
# Orden de corrida: celda 1 y R0 → R7a (o su caché) → R7u-0 → R7r-0 → R6r-1 y R6r-2 SIN CAMBIOS (archivo de la sesión 8)
#                   → R7r-9. No modifica ninguna celda validada. Escribe solo en fw5 y salidas/fw5 (prefijo r7u).
# Rasgos (S10-12):
#   Antes del partido (fila única por equipo-partido; contexto = rival, localía, Elo, estilo del rival, torneo):
#     rot_fuera_base (titulares fuera del XI base) · inicio_linea_3_5 (% de partidos) · min_menores_23 (% de minutos)
#   Durante el partido (asignados al marcador y cuarto del momento, igual que R4; periodos 1-2, D-9):
#     cambio_familia_90 (sin los de tras una roja) · reacomodo_sin_cambio_90 · sub_espejo_pct · sub_dibujo_pct
# =====================================================================

# [R7u-0 v1.1] Tramos de los 7 rasgos (mismo formato que r4_tramos: metrica, match_id, team, e5, e5b, cuarto, num, den)
import os, json, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5 = os.path.join(DP, 'fw5'); DIR_EV = os.path.join(DP, 'liga', 'eventos')
REHACER = False
EDAD_JOVEN = 23                             # S10-12 (Arturo): menores de 23
SIMULTANEO = 1.0                            # S10-10
CORTE_C1, CORTE_C3 = 25, 70                 # cuartos de R4 (A2)
t0 = time.time()
GRUPO = {1: 'POR', 3: 'CEN', 4: 'CEN', 5: 'CEN', 2: 'LAT', 6: 'LAT', 7: 'LAT', 8: 'LAT', 9: 'MCD', 10: 'MCD', 11: 'MCD',
         13: 'INT', 14: 'INT', 15: 'INT', 18: 'MP', 19: 'MP', 20: 'MP', 12: 'EXT', 16: 'EXT', 17: 'EXT', 21: 'EXT',
         22: 'DEL', 23: 'DEL', 24: 'DEL', 25: 'DEL'}
LADO = {**{p: 'D' for p in [2, 3, 7, 9, 12, 13, 17, 18, 22]}, **{p: 'I' for p in [5, 6, 8, 11, 15, 16, 20, 21, 24]},
        **{p: 'C' for p in [1, 4, 10, 14, 19, 23, 25]}}
def mueve(a, b):                            # copia exacta de R7a v1.1 (S10-11 C)
    if pd.isna(a) or pd.isna(b): return True
    a, b = int(a), int(b)
    if GRUPO.get(a) != GRUPO.get(b): return True
    return {LADO.get(a), LADO.get(b)} == {'I', 'D'}
def e5(m):                                  # copia de R4a
    m = np.asarray(m, dtype=float)
    return np.clip(np.sign(m) * np.ceil(np.abs(m) - 1e-9), -2, 2).astype(int)

Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet'))
MIN = pd.read_parquet(os.path.join(FW5, 'r4_minutos_v21.parquet'))
EP = pd.read_parquet(os.path.join(FW5, 'r7_equipo_partido.parquet'))
JP = pd.read_parquet(os.path.join(FW5, 'r7_jugador_partido.parquet'))
SUB = pd.read_parquet(os.path.join(FW5, 'r7_subs.parquet'))
FOR = pd.read_parquet(os.path.join(FW5, 'r7_formaciones.parquet'))
TRA = pd.read_parquet(os.path.join(FW5, 'r7_tramos.parquet'))
loc = Mm.set_index('match_id')
print(f'--- (0) Insumos: partidos R4 {len(Mm)} · equipo-partidos R7 {len(EP)} · sustituciones {len(SUB):,} · formaciones {len(FOR):,}')

# ---------- 1. Goles con reloj (periodos 1-2, misma regla que R4a: tiro convertido + autogol a favor) ----------
ruta_g = os.path.join(FW5, 'r7u_goles.parquet')
if os.path.exists(ruta_g) and not REHACER:
    G = pd.read_parquet(ruta_g)
else:
    COLS = {'type.name': 'type', 'team.name': 'team', 'period': 'period', 'minute': 'minute', 'second': 'second', 'shot.outcome.name': 'out'}
    piezas = []
    for mid in Mm['match_id']:
        r = os.path.join(DIR_EV, f'{mid}.parquet'); disp = set(pq.read_schema(r).names)
        e = pd.read_parquet(r, columns=[c for c in COLS if c in disp]).rename(columns=COLS)
        if 'out' not in e: e['out'] = np.nan
        e = e[e['period'].between(1, 4) & (((e['type'] == 'Shot') & (e['out'] == 'Goal')) | (e['type'] == 'Own Goal For'))]
        piezas.append(pd.DataFrame({'match_id': mid, 'team': e['team'].values, 'period': e['period'].values,
                                    'clk': (e['minute'] + e['second'] / 60).values}))
    G = pd.concat(piezas, ignore_index=True); G.to_parquet(ruta_g, index=False)
gg = G.assign(h=(G.team == G.match_id.map(loc['local'])).astype(int)).groupby('match_id')['h'].agg(['sum', 'size'])
ok = (gg['sum'].reindex(Mm.match_id).fillna(0).values == Mm['gl'].values) & ((gg['size'] - gg['sum']).reindex(Mm.match_id).fillna(0).values == Mm['gv'].values)
print(f'--- (1) Goles reconstruidos = marcador oficial: {ok.sum()} de {len(Mm)} ({time.time() - t0:.0f} s)')
GL = {m: g for m, g in G.groupby('match_id')}

def estado(m, team, period, clk):
    """e5, e5b y cuarto del equipo en el reloj (period, clk), con los goles ANTERIORES y el offset de liguilla (R4)."""
    g = GL.get(m)
    if g is None: dif = 0
    else:
        antes = g[(g.period < period) | ((g.period == period) & (g.clk < clk))]
        dif = int((antes.team == loc.at[m, 'local']).sum() - (antes.team != loc.at[m, 'local']).sum())
    s = 1.0 if team == loc.at[m, 'local'] else -1.0
    cu = (1 if clk < CORTE_C1 else 2) if period == 1 else (3 if clk < CORTE_C3 else 4)
    return int(e5(s * (dif + loc.at[m, 'offset']))), int(e5(s * (dif + loc.at[m, 'offset_b']))), cu

# ---------- 2. Rasgos antes del partido: una fila por equipo-partido (e5 = 0, cuarto = 1) ----------
mid_r4 = set(Mm['match_id'])
pre = EP[EP.match_id.isin(mid_r4)][['match_id', 'team', 'fuera_base', 'familia_inicial']].copy()
jov = JP.assign(mj=JP.minutos * (JP.edad < EDAD_JOVEN)).groupby(['match_id', 'team'])[['mj', 'minutos']].sum()
pre = pre.merge((100 * jov.mj / jov.minutos).rename('pct_jov').reset_index(), on=['match_id', 'team'], how='left')
filas = []
for met, num in [('rot_fuera_base', pre['fuera_base']), ('inicio_linea_3_5', 100.0 * (pre['familia_inicial'] == 'linea_3_5')),
                 ('min_menores_23', pre['pct_jov'])]:
    d = pd.DataFrame({'metrica': met, 'match_id': pre.match_id, 'team': pre.team, 'e5': 0, 'e5b': 0, 'cuarto': 1,
                      'num': num.astype(float), 'den': 1.0}).dropna(subset=['num'])
    filas.append(d)

# ---------- 3. Rasgos durante el partido: eventos asignados a marcador y cuarto; minutos de R4 como denominador ----------
def con_estado(d, clk_col='clk', per_col='period'):
    d = d[d[per_col].isin([1, 2]) & d.match_id.isin(mid_r4)].copy()
    est = [estado(m, t, int(p), c) for m, t, p, c in zip(d.match_id, d.team, d[per_col], d[clk_col])]
    d[['e5', 'e5b', 'cuarto']] = pd.DataFrame(est, index=d.index)
    return d
base_min = MIN[MIN.match_id.isin(mid_r4)][['match_id', 'team', 'e5', 'e5b', 'cuarto', 'minutos']]
def por90(eventos, nombre):
    c = eventos.groupby(['match_id', 'team', 'e5', 'e5b', 'cuarto']).size().rename('n').reset_index()
    d = base_min.merge(c, on=['match_id', 'team', 'e5', 'e5b', 'cuarto'], how='outer').fillna({'n': 0, 'minutos': 0})
    sin_min = int(((d.minutos == 0) & (d.n > 0)).sum())
    d = d[d.minutos > 0]
    return pd.DataFrame({'metrica': nombre, 'match_id': d.match_id, 'team': d.team, 'e5': d.e5, 'e5b': d.e5b, 'cuarto': d.cuarto,
                         'num': 90.0 * d.n, 'den': d.minutos}), sin_min
cf = con_estado(FOR[FOR.cambia_fam.astype(bool) & ~FOR.tras_roja.astype(bool)])
t1, s1 = por90(cf, 'cambio_familia_90')
TRA = TRA.sort_values(['match_id', 'player_id', 'from_clk'])
if 'pos_prev' not in TRA: TRA['pos_prev'] = TRA.groupby(['match_id', 'player_id'])['position_id'].shift()
tact = TRA[TRA['start_reason'].astype(str).str.contains('Tactical', case=False)][['match_id', 'team', 'player_id', 'from_clk', 'from_period', 'pos_prev', 'position_id']].copy()
tact['real'] = [mueve(a, b) for a, b in zip(tact.pos_prev, tact.position_id)]
sub_clk = SUB.groupby(['match_id', 'team'])['clk'].apply(list).to_dict()
tact['cerca'] = [any(abs(c - x) <= SIMULTANEO for x in sub_clk.get((m, t), [])) for m, t, c in zip(tact.match_id, tact.team, tact.from_clk)]
mv = tact[~tact.cerca & tact.real].sort_values(['match_id', 'team', 'from_clk'])
mv['nuevo'] = mv.groupby(['match_id', 'team'])['from_clk'].diff().fillna(99) > SIMULTANEO
mom = con_estado(mv[mv.nuevo].rename(columns={'from_clk': 'clk', 'from_period': 'period'}))
t2, s2_ = por90(mom, 'reacomodo_sin_cambio_90')
st = con_estado(SUB[~SUB['lesion'].astype(bool)])
t3 = pd.DataFrame({'metrica': 'sub_espejo_pct', 'match_id': st.match_id, 'team': st.team, 'e5': st.e5, 'e5b': st.e5b, 'cuarto': st.cuarto,
                   'num': 100.0 * (st.tipo == 'espejo'), 'den': 1.0})
t4 = t3.assign(metrica='sub_dibujo_pct', num=100.0 * (st.tipo == 'dibujo').values)
TRU = pd.concat(filas + [t1, t2, t3, t4], ignore_index=True)
TRU = TRU.groupby(['metrica', 'match_id', 'team', 'e5', 'e5b', 'cuarto'], as_index=False)[['num', 'den']].sum()
TRU.to_parquet(os.path.join(FW5, 'r7u_tramos.parquet'), index=False)
print(f'--- (2-3) Eventos sin minutos de R4 en su estado (se pierden; deben ser ~0): familia {s1} · reacomodo {s2_}')
res = TRU.groupby('metrica').apply(lambda d: pd.Series({'filas': len(d), 'equipo_partidos': d[['match_id', 'team']].drop_duplicates().shape[0],
                                                       'media_liga': d.num.sum() / d.den.sum()}), include_groups=False)
print('Rasgos (media de la liga en su unidad: titulares, % o por 90 min):'); print(res.round(3).to_string())
e3 = np.sign(st.e5)
print('Cambios tácticos por estado (abajo/empate/arriba): espejo %', [round(100 * (st.tipo[e3 == s] == 'espejo').mean(), 1) for s in [-1, 0, 1]],
      '· dibujo %', [round(100 * (st.tipo[e3 == s] == 'dibujo').mean(), 1) for s in [-1, 0, 1]])
print(f'R7u-0 terminó en {time.time() - t0:.0f} s. Guardado: fw5/r7u_tramos.parquet (y caché r7u_goles).')




R7r0

In [ ]:

# [R7r-0 v1.0] Carpeta espejo para correr R6r-1 y R6r-2 sin cambios sobre fw5 (patrón de X5-0)
import os, json, shutil
DRIVE = globals().get('DRIVE_R7U', '/content/drive/MyDrive/isac2026/')
LOCAL = globals().get('LOCAL_R7U', '/content/isac_r7u/')
D_DP, D_FW1, D_FW5 = os.path.join(DRIVE, 'datos_parquet'), os.path.join(DRIVE, 'datos_parquet', 'fw1'), os.path.join(DRIVE, 'datos_parquet', 'fw5')
L_DP, L_FW = os.path.join(LOCAL, 'datos_parquet'), os.path.join(LOCAL, 'datos_parquet', 'fw1')
shutil.rmtree(LOCAL, ignore_errors=True)
os.makedirs(L_FW, exist_ok=True); os.makedirs(os.path.join(LOCAL, 'salidas'), exist_ok=True)
def enlazar(src, dst):
    if os.path.exists(src): os.symlink(src, dst); return True
    return False
enl = {'partido_tecnico_periodo.parquet': enlazar(os.path.join(D_DP, 'partido_tecnico_periodo.parquet'), os.path.join(L_DP, 'partido_tecnico_periodo.parquet'))}
for f in ['r4_partidos.parquet', 'r4_minutos_v21.parquet', 'parametros_fw1.json', 'r4_perfiles_etapa.parquet',
          'r6_perfiles_etapa.parquet', 'r6b_perfiles_etapa.parquet', 'r6c_perfiles_etapa.parquet', 'r7u_tramos.parquet']:
    enl[f] = enlazar(os.path.join(D_FW5, f), os.path.join(L_FW, f))
enl['elo_equipo_partido_e2.parquet (fw1, solo lectura)'] = enlazar(os.path.join(D_FW1, 'elo_equipo_partido_e2.parquet'), os.path.join(L_FW, 'elo_equipo_partido_e2.parquet'))
print('--- (R7r-0) Carpeta espejo', LOCAL); print({k: ('ok' if v else 'FALTA') for k, v in enl.items()})
oblig = ['partido_tecnico_periodo.parquet', 'r4_partidos.parquet', 'r4_minutos_v21.parquet', 'parametros_fw1.json', 'r4_perfiles_etapa.parquet',
         'r7u_tramos.parquet', 'elo_equipo_partido_e2.parquet (fw1, solo lectura)']
assert all(enl[k] for k in oblig), 'Falta un insumo obligatorio: avisar a Claude'
with open(os.path.join(L_FW, 'parametros_fw1.json')) as f:
    print('Temporadas de ajuste:', json.load(f)['temporadas']['entrenamiento'])
BASE = LOCAL; PREFIJO = 'r7u'
print("BASE y PREFIJO listos → corre R6r-1 v1.1 y R6r-2 v1.1 (claude/codigo/sesion8/R6r_receta_confiabilidad.py) sin cambiarles nada; luego R7r-9.")


R7r9

In [ ]:
 #[R7r-9 v1.0] Copiar resultados a Drive (fw5 y salidas/fw5) y regresar BASE a Drive
import os, shutil
DRIVE = globals().get('DRIVE_R7U', '/content/drive/MyDrive/isac2026/')
LOCAL = globals().get('LOCAL_R7U', '/content/isac_r7u/')
L_FW = os.path.join(LOCAL, 'datos_parquet', 'fw1')
D_FW5, D_SAL5 = os.path.join(DRIVE, 'datos_parquet', 'fw5'), os.path.join(DRIVE, 'salidas', 'fw5')
os.makedirs(D_SAL5, exist_ok=True); n = []
for src_dir, dst_dir in [(L_FW, D_FW5), (os.path.join(LOCAL, 'salidas'), D_SAL5)]:
    for f in os.listdir(src_dir):
        s = os.path.join(src_dir, f)
        if f.startswith('r7u_') and os.path.isfile(s) and not os.path.islink(s):
            shutil.copy2(s, os.path.join(dst_dir, f)); n.append(f)
print('--- (R7r-9) Copiados a Drive:', sorted(n))
BASE = DRIVE
print('BASE =', BASE)


H7

In [ ]:
# H7 v1.0 · Hipótesis de Arturo sobre uso de jugadores (registro del plan maestro v4, sección 4)
# =====================================================================
# Sesión 10 · 2026-10-15 · Hackathon ISAC 2026, Track 2 · Cinco temporadas (fw5) · Solo lectura de r7_* y r4_partidos
# H1 Ortiz y la Sub-20 · H2 paciencia con los titulares (rotación tras perder frente a tras ganar)
# H3 plantilla de Jardine (continuidad al llegar; concentración de minutos por grupo de posición, portero y centrales)
# H4 rotación por calendario club por club ("menos de 6 clubes")
# Antes: celda 1 (Drive). NO necesita R0 ni BASE: la ruta va fija (RAIZ). Tarda ~1-2 min. No escribe en fw1.
# Intervalos: remuestreo por bloques de 5 partidos consecutivos del mismo técnico (los partidos seguidos se parecen),
# 2,000 repeticiones, semilla 2026. Percentil = lugar del técnico entre las etapas de la liga con 15+ partidos.
# =====================================================================

# [H7 v1.0] ------------------------------------------------------------
import os, time
import numpy as np
import pandas as pd

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
FW5 = os.path.join(RAIZ, 'datos_parquet', 'fw5'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5'); os.makedirs(SAL5, exist_ok=True)
SEMILLA, B, BLOQUE, ETAPA_MIN = 2026, 2000, 5, 15
EDADES = [21, 23]                 # menores de 21 y de 23 el día del partido
EDAD_DEBUT = 21                   # debut joven: primera aparición en los datos con menos de 21 años
COB = 0.80                        # rotación efectiva por grupo (S10-07 B)
AMERICA = {'Solari': 'Solari', 'Ortiz': 'Ortiz', 'Jardine': 'Jardine'}
rng = np.random.default_rng(SEMILLA)
pd.set_option('display.width', 250)
t0 = time.time()

EP = pd.read_parquet(os.path.join(FW5, 'r7_equipo_partido.parquet'))
JP = pd.read_parquet(os.path.join(FW5, 'r7_jugador_partido.parquet'))
TRA = pd.read_parquet(os.path.join(FW5, 'r7_tramos.parquet'), columns=['match_id', 'team', 'player_id', 'position_id', 'min'])
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet'))[['match_id', 'local', 'visita', 'gl', 'gv']]
for d in (EP, JP): d['fecha'] = pd.to_datetime(d['fecha'])
GRUPO = {1: 'POR', 3: 'CEN', 4: 'CEN', 5: 'CEN', 2: 'LAT', 6: 'LAT', 7: 'LAT', 8: 'LAT', 9: 'MED', 10: 'MED', 11: 'MED',
         13: 'MED', 14: 'MED', 15: 'MED', 18: 'MED', 19: 'MED', 20: 'MED', 12: 'EXT', 16: 'EXT', 17: 'EXT', 21: 'EXT',
         22: 'DEL', 23: 'DEL', 24: 'DEL', 25: 'DEL'}   # 6 grupos para H3 (MCD, INT y MP juntos como medio)
n_et = EP.groupby('etapa_id')['match_id'].nunique()
ET_OK = set(n_et[n_et >= ETAPA_MIN].index)
def quien(tec, team):
    if not isinstance(team, str) or not team.startswith('Am') or not isinstance(tec, str): return None
    return next((k for k, v in AMERICA.items() if v in tec), None)
EP['am'] = [quien(t, e) for t, e in zip(EP.tec_p, EP.team)]
JP['am'] = [quien(t, e) for t, e in zip(JP.tec_p, JP.team)]
print('--- (0) Partidos de América por técnico:', EP.groupby('am')['match_id'].nunique().to_dict(),
      '· etapas de la liga con 15+ partidos:', len(ET_OK))

def bloques(df, col_orden='fecha'):
    """Índices de bloques de BLOQUE partidos consecutivos (para remuestrear)."""
    df = df.sort_values(col_orden).reset_index(drop=True)
    return df, [np.arange(i, min(i + BLOQUE, len(df))) for i in range(0, len(df), BLOQUE)]
def boot_media(df, f, B=B):
    """Media de f(df) y su IC 95% remuestreando bloques. f recibe un DataFrame y devuelve un número."""
    df, bl = bloques(df)
    est = f(df); res = np.empty(B)
    for b in range(B):
        idx = np.concatenate([bl[i] for i in rng.integers(0, len(bl), len(bl))])
        res[b] = f(df.iloc[idx])
    return est, np.nanpercentile(res, 2.5), np.nanpercentile(res, 97.5), res
def boot_dif(a, b, f):
    ea, _, _, ra = boot_media(a, f); eb, _, _, rb = boot_media(b, f)
    d = ra - rb
    return ea - eb, np.nanpercentile(d, 2.5), np.nanpercentile(d, 97.5)
def pct_liga(valores_liga, x):
    return 100 * np.mean(np.asarray(valores_liga) <= x)

# ======================= H1 · Ortiz y la Sub-20 =======================
print('\n=== H1 · Minutos a jóvenes y debuts (América frente a la liga) ===')
for E_ in EDADES: JP[f'm{E_}'] = JP['minutos'] * (JP['edad'] < E_)
pm = JP.groupby(['match_id', 'team', 'etapa_id', 'am', 'fecha'], dropna=False)[['minutos', 'm21', 'm23']].sum().reset_index()
f21 = lambda d: 100 * d.m21.sum() / d.minutos.sum(); f23 = lambda d: 100 * d.m23.sum() / d.minutos.sum()
liga = pm[pm.etapa_id.isin(ET_OK)].groupby('etapa_id').apply(lambda d: pd.Series({'p21': f21(d), 'p23': f23(d)}), include_groups=False)
# Debuts: primera aparición del jugador en los datos (cualquier club), menor de EDAD_DEBUT, fuera del primer torneo de los datos
JP['torneo_'] = JP['torneo']
primer_torneo = JP.sort_values('fecha')['torneo'].iloc[0]
deb = JP.sort_values('fecha').drop_duplicates('player_id')
deb = deb[(deb.edad < EDAD_DEBUT) & (deb.torneo != primer_torneo)][['player_id', 'match_id', 'team', 'etapa_id', 'am', 'edad']]
min_deb = JP.merge(deb[['player_id', 'team', 'etapa_id']], on=['player_id', 'team', 'etapa_id']).groupby('etapa_id')['minutos'].sum()
part_et = pm.groupby('etapa_id')['match_id'].nunique()
part_valid = pm[pm.match_id.isin(JP.loc[JP.torneo != primer_torneo, 'match_id'])].groupby('etapa_id')['match_id'].nunique()
DB = pd.DataFrame({'debuts': deb.groupby('etapa_id').size(), 'min_debutantes': min_deb, 'partidos_validos': part_valid}).fillna(0)
DB['debuts_x17'] = 17 * DB.debuts / DB.partidos_validos.replace(0, np.nan)
DB['min_deb_x17'] = 17 * DB.min_debutantes / DB.partidos_validos.replace(0, np.nan)
DBl = DB[DB.index.isin(ET_OK) & (DB.partidos_validos >= ETAPA_MIN)]
filas = []
for k in AMERICA:
    d = pm[pm.am == k]
    e21, l21, h21, _ = boot_media(d, f21); e23, l23, h23, _ = boot_media(d, f23)
    ets = d.etapa_id.unique()
    dbk = DB.loc[DB.index.isin(ets)].sum()
    dx = 17 * dbk.debuts / dbk.partidos_validos if dbk.partidos_validos else np.nan
    filas.append(dict(tecnico=k, partidos=d.match_id.nunique(), pct_menores_21=f'{e21:.1f} [{l21:.1f}, {h21:.1f}]', pctl_21=pct_liga(liga.p21, e21),
                      pct_menores_23=f'{e23:.1f} [{l23:.1f}, {h23:.1f}]', pctl_23=pct_liga(liga.p23, e23),
                      debuts_por_torneo=dx, pctl_debuts=pct_liga(DBl.debuts_x17, dx) if pd.notna(dx) else np.nan,
                      partidos_validos_debut=int(dbk.partidos_validos)))
print(pd.DataFrame(filas).round(1).to_string(index=False))
print(f'Liga (etapas 15+): menores de 21 mediana {liga.p21.median():.1f}% (p10 {liga.p21.quantile(.1):.1f}, p90 {liga.p21.quantile(.9):.1f}) · '
      f'menores de 23 mediana {liga.p23.median():.1f}% · debuts por torneo mediana {DBl.debuts_x17.median():.2f} (p90 {DBl.debuts_x17.quantile(.9):.2f})')
for a, b in [('Ortiz', 'Solari'), ('Ortiz', 'Jardine')]:
    for nom, f in [('menores de 21', f21), ('menores de 23', f23)]:
        dd, lo, hi = boot_dif(pm[pm.am == a], pm[pm.am == b], f)
        print(f'  {a} − {b}, % minutos {nom}: {dd:+.1f} puntos [{lo:+.1f}, {hi:+.1f}]')
print('Debutantes de América (edad, técnico):', deb[deb.am.notna()].groupby('am')['edad'].apply(lambda s: sorted(np.round(s, 1).tolist())).to_dict())

# ======================= H2 · Paciencia con los titulares =======================
print('\n=== H2 · Rotación tras perder frente a tras ganar (titulares que cambian respecto al partido anterior) ===')
res = pd.concat([Mm.assign(team=Mm.local, gf=Mm.gl, gc=Mm.gv), Mm.assign(team=Mm.visita, gf=Mm.gv, gc=Mm.gl)])
res['r'] = np.sign(res.gf - res.gc).map({1: 'G', 0: 'E', -1: 'P'})
R2 = EP.sort_values(['team', 'fecha', 'match_id']).merge(res[['match_id', 'team', 'r']], on=['match_id', 'team'], how='left')
R2['r_prev'] = R2.groupby('team')['r'].shift(); R2['et_prev'] = R2.groupby('team')['etapa_id'].shift()
R2 = R2[(R2.et_prev == R2.etapa_id) & R2.rotados.notna() & R2.r_prev.notna()].copy()
celda = R2.groupby(['descanso_corto', 'favorito'])['rotados'].transform('mean')
R2['res_rot'] = R2['rotados'] - celda                         # quita calendario y favoritismo (S10-05)
reac = lambda d: d.loc[d.r_prev == 'P', 'res_rot'].mean() - d.loc[d.r_prev == 'G', 'res_rot'].mean()
print(f"Liga: tras perder {R2.loc[R2.r_prev == 'P', 'rotados'].mean():.2f} · tras empatar {R2.loc[R2.r_prev == 'E', 'rotados'].mean():.2f} · "
      f"tras ganar {R2.loc[R2.r_prev == 'G', 'rotados'].mean():.2f} titulares cambiados")
el, ll, hl, _ = boot_media(R2.assign(fecha=R2.fecha), reac)
print(f'Reacción de la liga (perder − ganar, sin calendario ni favoritismo): {el:+.2f} titulares [{ll:+.2f}, {hl:+.2f}]')
rl = R2[R2.etapa_id.isin(ET_OK)].groupby('etapa_id').filter(lambda d: (d.r_prev == 'P').sum() >= 5 and (d.r_prev == 'G').sum() >= 5)
rl = rl.groupby('etapa_id').apply(reac, include_groups=False)
filas = []
for k in AMERICA:
    d = R2[R2.am == k]
    e, lo, hi, _ = boot_media(d, reac)
    filas.append(dict(tecnico=k, tras_perder=(d.r_prev == 'P').sum(), tras_ganar=(d.r_prev == 'G').sum(),
                      reaccion=f'{e:+.2f} [{lo:+.2f}, {hi:+.2f}]', percentil_liga=pct_liga(rl, e)))
print(pd.DataFrame(filas).round(0).to_string(index=False))
print(f'Etapas de la liga con 5+ derrotas y 5+ victorias: {len(rl)} · reacción mediana {rl.median():+.2f} (p10 {rl.quantile(.1):+.2f}, p90 {rl.quantile(.9):+.2f})')

# ======================= H3 · Plantilla de Jardine =======================
print('\n=== H3 · Continuidad al llegar y concentración de minutos por grupo de posición ===')
JPs = JP.sort_values('fecha')
ini_et = EP.groupby('etapa_id').agg(team=('team', 'first'), inicio=('fecha', 'min'), tec=('tec_p', 'first'), am=('am', 'first'))
filas = []
for e, r in ini_et.iterrows():
    prev = JPs[(JPs.team == r.team) & (JPs.fecha < r.inicio)]
    if prev.empty: continue
    previos = set(prev[prev.fecha >= r.inicio - pd.Timedelta(days=365)].player_id)     # plantel del año anterior
    d = JPs[(JPs.etapa_id == e)]
    d = d[d.match_id.isin(d.drop_duplicates('match_id').head(17).match_id)]          # sus primeros 17 partidos
    if d.match_id.nunique() < 10: continue
    filas.append(dict(etapa_id=e, am=r.am, tec=r.tec, continuidad_llegada=100 * d.loc[d.player_id.isin(previos), 'minutos'].sum() / d.minutos.sum()))
CL = pd.DataFrame(filas)
print(f"Continuidad en los primeros 17 partidos (% de minutos a jugadores del plantel del año anterior) · liga: mediana {CL.continuidad_llegada.median():.0f}% "
      f"(p10 {CL.continuidad_llegada.quantile(.1):.0f}, p90 {CL.continuidad_llegada.quantile(.9):.0f}; {len(CL)} llegadas)")
print(CL[CL.am.notna()][['am', 'continuidad_llegada']].assign(percentil=lambda d: [pct_liga(CL.continuidad_llegada, x) for x in d.continuidad_llegada]).round(0).to_string(index=False))
# Grupo principal de cada jugador en cada partido
TRA['g'] = TRA.position_id.map(GRUPO)
gp = TRA.groupby(['match_id', 'team', 'player_id', 'g'])['min'].sum().reset_index().sort_values('min').drop_duplicates(['match_id', 'team', 'player_id'], keep='last')
gp = gp.merge(EP[['match_id', 'team', 'etapa_id', 'am', 'torneo']], on=['match_id', 'team'])
def cubre(m, q=COB):
    m = np.sort(np.asarray(m))[::-1]
    return int(np.searchsorted(np.cumsum(m) / m.sum(), q) + 1) if m.sum() > 0 else np.nan
conc = (gp.groupby(['etapa_id', 'torneo', 'g', 'player_id'])['min'].sum().reset_index()
          .groupby(['etapa_id', 'torneo', 'g'])['min'].apply(cubre).rename('n80').reset_index())
conc = conc.merge(gp.groupby(['etapa_id', 'torneo'])['match_id'].nunique().rename('pj').reset_index(), on=['etapa_id', 'torneo'])
conc = conc[conc.pj >= 10]
tabla = conc.groupby(['etapa_id', 'g'])['n80'].mean().unstack()
tabla['am'] = tabla.index.map(ini_et['am'])
orden = ['POR', 'CEN', 'LAT', 'MED', 'EXT', 'DEL']
print('Jugadores que cubren el 80% de los minutos de cada grupo, promedio por torneo (torneos con 10+ partidos):')
fila_liga = tabla[tabla.index.isin(ET_OK)][orden].median().rename('liga (mediana)')
print(pd.concat([tabla[tabla.am.notna()].set_index('am')[orden], fila_liga.to_frame().T]).round(2).to_string())
por = gp[gp.g == 'POR'].groupby(['etapa_id', 'player_id'])['min'].sum()
top_por = (por.groupby(level=0).max() / por.groupby(level=0).sum() * 100)
print('Portero: % de los minutos de portero para el portero principal ·',
      {ini_et.loc[e, 'am']: round(v, 1) for e, v in top_por.items() if pd.notna(ini_et.loc[e, 'am'])},
      f'· liga mediana {top_por[top_por.index.isin(ET_OK)].median():.1f}%')

# ======================= H4 · Rotación por calendario, club por club =======================
print('\n=== H4 · Titulares fuera del XI base con descanso corto Y siendo favorito, frente al resto, por club ===')
filas = []
for club, d in EP[EP.fuera_base.notna()].groupby('team'):
    c = d.descanso_corto.fillna(False).astype(bool) & d.favorito.astype(bool)
    if c.sum() < 8: filas.append(dict(club=club, n_cond=int(c.sum()), dif=np.nan, lo=np.nan, hi=np.nan)); continue
    a, b = d[c], d[~c]
    dd = a.fuera_base.mean() - b.fuera_base.mean()
    se = np.sqrt(a.fuera_base.var() / len(a) + b.fuera_base.var() / len(b))
    filas.append(dict(club=club, n_cond=int(c.sum()), dif=dd, lo=dd - 1.96 * se, hi=dd + 1.96 * se))
H4 = pd.DataFrame(filas).sort_values('dif', ascending=False)
print(H4.round(2).to_string(index=False))
sig = H4[(H4.lo > 0)]
print(f'Clubes con rotación por calendario clara (IC sobre 0): {len(sig)} de {H4.dif.notna().sum()} con 8+ partidos en la condición → {sig.club.tolist()}')
print('Nota: sin corrección por comparaciones múltiples; con ~18 clubes se esperaría ~0.5 "claro" por azar (una cola, 2.5%).')
print(f'\nH7 terminó en {time.time() - t0:.0f} s. Pega toda la salida.')

R7b-0

In [ ]:
# R7b v1.0 · Duelo de banquillos (U5 completo) con TODAS las decisiones tácticas, y efecto asociado por tipo de decisión
# =====================================================================
# Sesión 10 · 2026-10-15 · Hackathon ISAC 2026, Track 2 · Framework v1.2 §11 U5 · Decisiones S10-13 a S10-17
# S10-13 A: cada decisión es una fila con marcador y cuarto → pasa por la receta R4 (= línea base de la liga) y por R5.
# S10-17 (Arturo): "decisión táctica" = cualquier movimiento del técnico, no solo sustituciones:
#   sustitución espejo · sustitución con reacomodo · sustitución con dibujo · cambio de dibujo sin sustitución ·
#   reacomodo sin sustitución. Se agrupan en "momentos" (±1 min). Fuera: cambios por lesión, recodificaciones sin
#   movimiento real, y TODO lo que ocurre ≤ 10 min después de una roja de cualquiera de los dos equipos (forzado).
# Rasgos (S10-14): iniciativa · minuto de la primera decisión · respuesta al rival (5') · reacción al gol en contra (10')
#                  · reacción al dominio rival (10'; P-10). Expulsión fuera como gatillo (Arturo, S10-15).
# Efecto asociado (S10-16, nivel liga, asociación NO causa): xG (a favor − en contra) y field tilt 15' antes y 15' después,
#   por tipo de decisión, frente a momentos SIN decisión en el mismo marcador y minuto (control de regresión a la media).
# Orden: celda 1 → R7b-0 → R7b-r0 → R6r-1 y R6r-2 SIN CAMBIOS → R7b-r9. Escribe solo en fw5 y salidas/fw5 (prefijo r7b).
# Ruta fija (sin adivinar): RAIZ. Tarda ~5-10 min la primera vez (lee los eventos de R1 de las cinco temporadas).
# =====================================================================

#[R7b-0 v1.1] Momentos de decisión, gatillos, tramos r7b y efecto asociado ---------------------------------------
import os, json, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5')
DIR_EV = os.path.join(DP, 'liga', 'eventos'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5'); os.makedirs(SAL5, exist_ok=True)
REHACER = False
SEMILLA, B = 2026, 2000
rng = np.random.default_rng(SEMILLA)
SIMULTANEO, VENTANA_ROJA = 1.0, 10           # S10-10, S10-09
V_RESP, V_GOL, V_DOM = 5, 10, 10             # S10-14 (sensibilidad 3 y 15 al final)
LIMITE = 85                                  # gatillos después del 85' no cuentan (no queda tiempo para responder)
DOM_PCT, DOM_MIN_ACC, DOM_VENT = 0.70, 20, 10  # P-10
EF_VENT, EF_INI, EF_FIN = 15, 60, 78         # efecto asociado: solo 2T, decisiones entre el 60' y el 78', ventanas de 15'
CORTE_C1, CORTE_C3 = 25, 70
GRUPO = {1: 'POR', 3: 'CEN', 4: 'CEN', 5: 'CEN', 2: 'LAT', 6: 'LAT', 7: 'LAT', 8: 'LAT', 9: 'MCD', 10: 'MCD', 11: 'MCD',
         13: 'INT', 14: 'INT', 15: 'INT', 18: 'MP', 19: 'MP', 20: 'MP', 12: 'EXT', 16: 'EXT', 17: 'EXT', 21: 'EXT',
         22: 'DEL', 23: 'DEL', 24: 'DEL', 25: 'DEL'}
LADO = {**{p: 'D' for p in [2, 3, 7, 9, 12, 13, 17, 18, 22]}, **{p: 'I' for p in [5, 6, 8, 11, 15, 16, 20, 21, 24]},
        **{p: 'C' for p in [1, 4, 10, 14, 19, 23, 25]}}
def mueve(a, b):                              # copia exacta de R7a v1.1 (S10-11 C)
    if pd.isna(a) or pd.isna(b): return True
    a, b = int(a), int(b)
    if GRUPO.get(a) != GRUPO.get(b): return True
    return {LADO.get(a), LADO.get(b)} == {'I', 'D'}
def e5(m):
    m = np.asarray(m, dtype=float); return np.clip(np.sign(m) * np.ceil(np.abs(m) - 1e-9), -2, 2).astype(int)
t0 = time.time(); pd.set_option('display.width', 250)
with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
    PRM = json.load(f)
TEMPS = sorted(PRM['temporadas']['entrenamiento']); X_U3 = PRM['P']['tercio_ultimo_x']

Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id'); MIDS = set(Mm.match_id)
SUB = pd.read_parquet(os.path.join(FW5, 'r7_subs.parquet')); FOR = pd.read_parquet(os.path.join(FW5, 'r7_formaciones.parquet'))
TRA = pd.read_parquet(os.path.join(FW5, 'r7_tramos.parquet')); G = pd.read_parquet(os.path.join(FW5, 'r7u_goles.parquet'))
GL = {m: g for m, g in G.groupby('match_id')}
def estado(m, team, period, clk):            # copia de R7u-0: goles ANTERIORES + offset de liguilla (R4)
    g = GL.get(m); dif = 0
    if g is not None:
        a = g[(g.period < period) | ((g.period == period) & (g.clk < clk))]
        dif = int((a.team == loc.at[m, 'local']).sum() - (a.team != loc.at[m, 'local']).sum())
    s = 1.0 if team == loc.at[m, 'local'] else -1.0
    cu = (1 if clk < CORTE_C1 else 2) if period == 1 else (3 if clk < CORTE_C3 else 4)
    return int(e5(s * (dif + loc.at[m, 'offset']))), int(e5(s * (dif + loc.at[m, 'offset_b']))), cu
equipos = pd.concat([Mm[['match_id', 'local']].rename(columns={'local': 'team'}), Mm[['match_id', 'visita']].rename(columns={'visita': 'team'})])
rival = {(m, t): (loc.at[m, 'visita'] if t == loc.at[m, 'local'] else loc.at[m, 'local']) for m, t in zip(equipos.match_id, equipos.team)}

# ---------- 1. Momentos de decisión (S10-17) ----------
rojas = TRA[TRA['end_reason'].isin(['Red Card', 'Second Yellow Card', 'Sent Off'])].groupby('match_id')['to_clk'].apply(list).to_dict()
tras_roja = lambda m, c: any(0 <= c - x <= VENTANA_ROJA for x in rojas.get(m, []))
TRA = TRA.sort_values(['match_id', 'player_id', 'from_clk'])
if 'pos_prev' not in TRA: TRA['pos_prev'] = TRA.groupby(['match_id', 'player_id'])['position_id'].shift()
tact = TRA[TRA['start_reason'].astype(str).str.contains('Tactical', case=False)]
tact = tact.assign(real=[mueve(a, b) for a, b in zip(tact.pos_prev, tact.position_id)])
acc = []                                     # acciones sueltas: (match, team, period, clk, clase)
s_ = SUB[~SUB['lesion'].astype(bool) & SUB['period'].isin([1, 2])]
acc += [(m, t, int(p), c, 'sub_' + str(ti)) for m, t, p, c, ti in zip(s_.match_id, s_.team, s_.period, s_.clk, s_.tipo)]
f_ = FOR[FOR['cambia_form'].astype(bool) & FOR['period'].isin([1, 2])]
acc += [(m, t, int(p), c, 'form_fam' if fam else 'form') for m, t, p, c, fam in zip(f_.match_id, f_.team, f_.period, f_.clk, f_.cambia_fam.astype(bool))]
t_ = tact[tact.real & tact.from_period.isin([1, 2])]
# v1.1: fuera los movimientos del que entra o del que sale en una sustitución a ±1 min (misma regla que R7a: 'otros' excluye in/out)
io = {}
for m, t, c, a, b in zip(SUB.match_id, SUB.team, SUB.clk, SUB.get('in_id', pd.Series(np.nan, index=SUB.index)), SUB.get('out_id', pd.Series(np.nan, index=SUB.index))):
    io.setdefault((m, t), []).append((c, a, b))
propio = [any(abs(c - c2) <= SIMULTANEO and p in (a, b) for c2, a, b in io.get((m, t), [])) for m, t, c, p in zip(t_.match_id, t_.team, t_.from_clk, t_.player_id)]
t_ = t_[~np.array(propio, dtype=bool)]
acc += [(m, t, int(p), c, 'mov') for m, t, p, c in zip(t_.match_id, t_.team, t_.from_period, t_.from_clk)]
AC = pd.DataFrame(acc, columns=['match_id', 'team', 'period', 'clk', 'clase'])
AC = AC[AC.match_id.isin(MIDS)].sort_values(['match_id', 'team', 'clk']).reset_index(drop=True)
AC['mom'] = (AC.groupby(['match_id', 'team'])['clk'].diff().fillna(99) > SIMULTANEO).cumsum()
def tipo_momento(cl):
    cl = set(cl)
    if 'sub_dibujo' in cl: return 'sub_dibujo'
    if 'sub_reacomodo' in cl: return 'sub_reacomodo'
    if 'sub_espejo' in cl: return 'sub_reacomodo' if 'mov' in cl else 'sub_espejo'
    if 'form_fam' in cl or ('form' in cl and 'mov' in cl): return 'dibujo_sin_cambio'
    if 'mov' in cl: return 'reacomodo_sin_cambio'
    return 'recodificacion'                  # cambio de código de formación sin movimiento real: no es decisión
MO = AC.groupby('mom').agg(match_id=('match_id', 'first'), team=('team', 'first'), period=('period', 'first'),
                           clk=('clk', 'min'), tipo=('clase', tipo_momento)).reset_index(drop=True)
MO['forzado'] = [tras_roja(m, c) for m, c in zip(MO.match_id, MO.clk)]
n_sub = AC[AC.clase.str.startswith('sub_')].groupby('mom').size()
print(f'Momentos con sustitución: {len(n_sub):,} · con 2 o más sustituciones juntas: {100 * (n_sub >= 2).mean():.1f}% '
      f'(si una es reacomodo o dibujo, el momento toma ese tipo)')
print('--- (1) Momentos de decisión (periodos 1-2, sin lesiones) por tipo · excluidos: recodificación y ≤10 min tras roja ---')
print(pd.crosstab(MO.tipo, MO.forzado, margins=True).rename(columns={False: 'válidos', True: 'tras roja'}).to_string())
DT = MO[(MO.tipo != 'recodificacion') & ~MO.forzado].copy()
print(f'Decisiones válidas por equipo-partido: {len(DT) / len(equipos):.2f} · equipo-partidos sin ninguna: '
      f'{100 * (1 - DT.groupby(["match_id", "team"]).ngroups / len(equipos)):.1f}%')
dt_t = {k: np.sort(g.clk.values) for k, g in DT.groupby(['match_id', 'team'])}
def hay(m, t, a, b):                         # ¿decisión propia en (a, b]?
    x = dt_t.get((m, t)); return bool(x is not None and ((x > a) & (x <= b)).any())

# ---------- 2. Series por minuto de R1: acciones en el último tercio (field tilt, M13) y xG ----------
ruta_s = os.path.join(FW5, 'r7b_series.parquet')
if os.path.exists(ruta_s) and not REHACER:
    SER = pd.read_parquet(ruta_s)
else:
    piezas = []
    for temp in TEMPS:
        r = os.path.join(FW1, f"eventos_{temp.replace('/', '-')}.parquet")
        disp = set(pq.read_schema(r).names)
        cols = [c for c in ['match_id', 'period', 'minute', 'type.name', 'team.name', 'loc_x', 'fase', 'es_reanudacion',
                            'ball_receipt.outcome.name', 'shot.statsbomb_xg'] if c in disp]
        E = pd.read_parquet(r, columns=cols); E = E[E.match_id.isin(MIDS) & E.period.isin([1, 2])]
        for c in ['ball_receipt.outcome.name', 'shot.statsbomb_xg']:
            if c not in E: E[c] = np.nan
        ab = (E['fase'] != 'bp') & ~E['es_reanudacion'].fillna(False).astype(bool)
        tn = E['type.name']
        u3 = ab & tn.isin(['Pass', 'Carry', 'Ball Receipt*', 'Shot']) & (E['loc_x'] >= X_U3) & E['team.name'].notna() \
             & ~((tn == 'Ball Receipt*') & E['ball_receipt.outcome.name'].notna())          # misma regla que R4a (M13)
        E['u3'] = u3.astype(float); E['xg'] = np.where(tn == 'Shot', E['shot.statsbomb_xg'].astype(float), 0.0)
        piezas.append(E.groupby(['match_id', 'period', 'minute', 'team.name'])[['u3', 'xg']].sum().reset_index())
        print(f'  {temp}: {E.match_id.nunique()} partidos · xG en R1: {"sí" if "shot.statsbomb_xg" in disp else "NO (efecto en xG vacío)"} '
              f'({time.time() - t0:.0f} s)')
    SER = pd.concat(piezas, ignore_index=True).rename(columns={'team.name': 'team'}); SER.to_parquet(ruta_s, index=False)
ARR = {}
for (m, p, t), g in SER.groupby(['match_id', 'period', 'team']):
    u = np.zeros(140); x = np.zeros(140); mi = g.minute.clip(0, 139).astype(int).values
    np.add.at(u, mi, g.u3.values); np.add.at(x, mi, g.xg.values); ARR[(m, p, t)] = (u, x)
vacio = (np.zeros(140), np.zeros(140))
def suma(m, p, t, a, b, k):                  # suma en minutos [a, b) del periodo p; k = 0 acciones u3, 1 xG
    a, b = max(int(np.floor(a)), 0), min(int(np.floor(b)), 140); return ARR.get((m, p, t), vacio)[k][a:b].sum()

# ---------- 3. Gatillos: goles en contra, decisiones del rival, dominio rival (P-10) ----------
GC = G[G.period.isin([1, 2]) & G.match_id.isin(MIDS)].copy()
GC['concede'] = [rival[(m, t)] for m, t in zip(GC.match_id, GC.team)]
dom = []
for (m, p, t), (u, _) in ARR.items():
    r_ = rival.get((m, t))
    if r_ is None: continue
    ur = ARR.get((m, p, r_), vacio)[0]
    ini = 0 if p == 1 else 45; ultimo_ep = -99; antes = False
    for k in range(ini + DOM_VENT, 140):
        a_ = u[k - DOM_VENT:k].sum(); b_ = ur[k - DOM_VENT:k].sum(); tot = a_ + b_
        dominado = tot >= DOM_MIN_ACC and b_ / tot >= DOM_PCT
        if dominado and not antes and k - ultimo_ep >= DOM_VENT: dom.append((m, t, p, float(k))); ultimo_ep = k
        antes = dominado
DOM = pd.DataFrame(dom, columns=['match_id', 'team', 'period', 'clk'])
print(f'\n--- (3) Gatillos: goles en contra {len(GC):,} · decisiones del rival {len(DT):,} · episodios de dominio rival {len(DOM):,} '
      f'({len(DOM) / len(equipos):.2f} por equipo-partido)')

# ---------- 4. Tramos r7b (mismo formato que r4_tramos) ----------
filas = []
def fila(met, m, t, p, c, num, cu_fijo=None):
    e, eb, cu = estado(m, t, p, c); filas.append((met, m, t, e, eb, cu if cu_fijo is None else cu_fijo, float(num), 1.0))
PRIM = DT.sort_values('clk').groupby(['match_id', 'team'])[['clk', 'period']].first()
for (m, t), r in PRIM.iterrows():                              # minuto de la primera decisión (cuarto fijo: no mecánico)
    fila('minuto_primera_decision', m, t, int(r.period), r.clk, r.clk, 1)
for m in MIDS:                                                 # iniciativa
    a, b = loc.at[m, 'local'], loc.at[m, 'visita']
    if (m, a) not in PRIM.index or (m, b) not in PRIM.index: continue
    ra, rb = PRIM.loc[(m, a)], PRIM.loc[(m, b)]
    c, p = (ra.clk, int(ra.period)) if ra.clk <= rb.clk else (rb.clk, int(rb.period))
    va = 50.0 if abs(ra.clk - rb.clk) <= SIMULTANEO else 100.0 * (ra.clk < rb.clk)
    fila('iniciativa', m, a, p, c, va); fila('iniciativa', m, b, p, c, 100.0 - va)
for r in DT.itertuples(index=False):                           # respuesta al rival (decisión propia en 5')
    if r.clk > LIMITE: continue
    yo = rival[(r.match_id, r.team)]
    fila('respuesta_rival', r.match_id, yo, r.period, r.clk, 100.0 * hay(r.match_id, yo, r.clk, r.clk + V_RESP))
for r in GC.itertuples(index=False):                           # reacción al gol en contra (10')
    if r.clk > LIMITE or tras_roja(r.match_id, r.clk): continue
    fila('reaccion_gol_contra', r.match_id, r.concede, r.period, r.clk + 0.01, 100.0 * hay(r.match_id, r.concede, r.clk, r.clk + V_GOL))
for r in DOM.itertuples(index=False):                          # reacción al dominio rival (10')
    if r.clk > LIMITE or tras_roja(r.match_id, r.clk): continue
    fila('reaccion_dominio', r.match_id, r.team, r.period, r.clk, 100.0 * hay(r.match_id, r.team, r.clk, r.clk + V_DOM))
TRB = pd.DataFrame(filas, columns=['metrica', 'match_id', 'team', 'e5', 'e5b', 'cuarto', 'num', 'den'])
TRB = TRB.groupby(['metrica', 'match_id', 'team', 'e5', 'e5b', 'cuarto'], as_index=False)[['num', 'den']].sum()
TRB.to_parquet(os.path.join(FW5, 'r7b_tramos.parquet'), index=False)
print('\n--- (4) Rasgos (media de la liga: % o minuto) ---')
print(TRB.groupby('metrica').apply(lambda d: pd.Series({'filas': len(d), 'casos': d.den.sum(), 'media_liga': d.num.sum() / d.den.sum()}),
                                   include_groups=False).round(2).to_string())
e3 = lambda d: np.sign(d.e5)
for met in ['respuesta_rival', 'reaccion_gol_contra', 'reaccion_dominio']:
    d = TRB[TRB.metrica == met]
    print(f'  {met} por marcador (abajo/empate/arriba, %):', [round(d[e3(d) == s].num.sum() / max(d[e3(d) == s].den.sum(), 1), 1) for s in [-1, 0, 1]])
sens = {}
for w in [3, V_RESP, 15]:
    v = [hay(r.match_id, rival[(r.match_id, r.team)], r.clk, r.clk + w) for r in DT[DT.clk <= LIMITE].itertuples(index=False)]
    sens[f'respuesta {w}\''] = round(100 * np.mean(v), 1)
for w in [5, V_GOL, 15]:
    v = [hay(r.match_id, r.concede, r.clk, r.clk + w) for r in GC[GC.clk <= LIMITE].itertuples(index=False)]
    sens[f'gol {w}\''] = round(100 * np.mean(v), 1)
print('Sensibilidad de ventanas (liga, %):', sens)
# v1.1: placebo de cada gatillo. Misma hora (y periodo) en OTRO equipo-partido elegido al azar que no tuvo ese gatillo en ±ventana.
# Si la reacción real no supera al placebo, no hay "reacción": solo la frecuencia normal de decisiones a esa hora.
tg = {'respuesta_rival': [(rival[(r.match_id, r.team)], r.match_id, r.clk) for r in DT[DT.clk <= LIMITE].itertuples(index=False)],
      'reaccion_gol_contra': [(r.concede, r.match_id, r.clk) for r in GC[GC.clk <= LIMITE].itertuples(index=False)],
      'reaccion_dominio': [(r.team, r.match_id, r.clk) for r in DOM[DOM.clk <= LIMITE].itertuples(index=False)]}
VENT = {'respuesta_rival': V_RESP, 'reaccion_gol_contra': V_GOL, 'reaccion_dominio': V_DOM}
pares = list(rival.keys())
def tiene_gatillo(tipo, m, t, c, w):
    if tipo == 'respuesta_rival': return hay(m, rival[(m, t)], c - w, c + w)
    if tipo == 'reaccion_gol_contra': x = GC[(GC.match_id == m) & (GC.concede == t)].clk.values
    else: x = DOM[(DOM.match_id == m) & (DOM.team == t)].clk.values
    return bool(((x >= c - w) & (x <= c + w)).any())
print('Reacción real frente a placebo (% con decisión propia dentro de la ventana; diferencia = reacción verdadera):')
for tipo, lst in tg.items():
    w = VENT[tipo]; real = np.mean([hay(m, t, c, c + w) for t, m, c in lst]); pl = []
    for t, m, c in lst[:3000]:
        for _ in range(20):
            m2, t2 = pares[rng.integers(len(pares))]
            if m2 != m and not tiene_gatillo(tipo, m2, t2, c, w): pl.append(hay(m2, t2, c, c + w)); break
    pl = np.array(pl, float); se = np.sqrt(real * (1 - real) / len(lst) + pl.var() / len(pl))
    print(f'  {tipo} ({w}\'): real {100 * real:.1f}% · placebo {100 * pl.mean():.1f}% · diferencia {100 * (real - pl.mean()):+.1f} '
          f'[{100 * (real - pl.mean() - 1.96 * se):+.1f}, {100 * (real - pl.mean() + 1.96 * se):+.1f}] puntos')

# ---------- 5. Efecto asociado por tipo de decisión (nivel liga; asociación, no causa) ----------
def delta(m, t, c):
    r_ = rival[(m, t)]; a0, a1, b1 = c - EF_VENT, c, c + EF_VENT
    xg = (suma(m, 2, t, a1, b1, 1) - suma(m, 2, r_, a1, b1, 1)) - (suma(m, 2, t, a0, a1, 1) - suma(m, 2, r_, a0, a1, 1))
    def ft(a, b):
        o, q = suma(m, 2, t, a, b, 0), suma(m, 2, r_, a, b, 0); return 100 * o / (o + q) if o + q > 0 else np.nan
    return xg, ft(a1, b1) - ft(a0, a1)
D = DT[(DT.period == 2) & DT.clk.between(EF_INI, EF_FIN)].copy()
# un solo momento por equipo en la ventana: si hubo otra decisión propia en los 15' previos o siguientes, se descarta (efectos mezclados)
D['aislado'] = [not (hay(m, t, c - EF_VENT, c - 1e-6) or hay(m, t, c + 1e-6, c + EF_VENT)) for m, t, c in zip(D.match_id, D.team, D.clk)]
D = D[D.aislado]
ctrl = []                                                      # control: minutos SIN decisión propia en ±15', mismo rango de minutos
for (m, t) in rival:
    if m not in MIDS: continue
    for c in rng.choice(np.arange(EF_INI, EF_FIN + 1), size=4, replace=False).astype(float):
        if not (hay(m, t, c - EF_VENT, c + EF_VENT)): ctrl.append((m, t, 2, c, 'sin_decision'))
D = pd.concat([D[['match_id', 'team', 'period', 'clk', 'tipo']], pd.DataFrame(ctrl, columns=['match_id', 'team', 'period', 'clk', 'tipo'])], ignore_index=True)
D[['d_xg', 'd_ft']] = pd.DataFrame([delta(m, t, c) for m, t, c in zip(D.match_id, D.team, D.clk)], index=D.index)
D['e3'] = [int(np.sign(estado(m, t, 2, c)[0])) for m, t, c in zip(D.match_id, D.team, D.clk)]
D['tramo'] = pd.cut(D.clk, [EF_INI - 1, 65, 70, EF_FIN + 1], labels=['60-65', '65-70', '70-78'])
def efecto(d, y):
    """Diferencia de cada tipo frente a 'sin_decision' en el mismo marcador y tramo de minutos (estratos), ponderando por casos."""
    out = {}
    for ti in [x for x in d.tipo.unique() if x != 'sin_decision']:
        dif, w = [], []
        for (_, _), g in d.groupby(['e3', 'tramo'], observed=True):
            a, b = g.loc[g.tipo == ti, y].dropna(), g.loc[g.tipo == 'sin_decision', y].dropna()
            if len(a) and len(b): dif.append(a.mean() - b.mean()); w.append(len(a))
        out[ti] = np.average(dif, weights=w) if w else np.nan
    return out
def ic_efecto(d, y):
    est = efecto(d, y); ms = d.match_id.unique(); bs = []
    pos = d.groupby('match_id').indices
    for _ in range(300):
        idx = np.concatenate([pos[m] for m in rng.choice(ms, len(ms))]); bs.append(efecto(d.iloc[idx], y))
    return {k: (v, np.nanpercentile([b_.get(k, np.nan) for b_ in bs], 2.5), np.nanpercentile([b_.get(k, np.nan) for b_ in bs], 97.5)) for k, v in est.items()}
D = D.reset_index(drop=True)
print(f'\n--- (5) Efecto asociado (2T, decisiones aisladas entre el {EF_INI}\' y el {EF_FIN}\'; 15\' después − 15\' antes) ---')
print('Casos por tipo y marcador:'); print(pd.crosstab(D.tipo, D.e3.map({-1: 'abajo', 0: 'empate', 1: 'arriba'})).to_string())
for nom, d in [('todos los marcadores', D), ('solo yendo ABAJO', D[D.e3 == -1])]:
    for y, u in [('d_xg', 'xG a favor − en contra'), ('d_ft', 'field tilt, puntos')]:
        r = ic_efecto(d, y)
        print(f'{nom} · {u} · diferencia frente a "sin decisión" en el mismo marcador y minuto [IC 95%, remuestreo por partido]:')
        print('   ' + ' · '.join(f'{k} {v[0]:+.3f} [{v[1]:+.3f}, {v[2]:+.3f}]' for k, v in sorted(r.items())))
print('Referencia: cambio medio "sin decisión" (regresión a la media) · xG', round(D.loc[D.tipo == 'sin_decision', 'd_xg'].mean(), 3),
      '· field tilt', round(D.loc[D.tipo == 'sin_decision', 'd_ft'].mean(), 2))
D.to_parquet(os.path.join(FW5, 'r7b_efecto.parquet'), index=False); DT.to_parquet(os.path.join(FW5, 'r7b_decisiones.parquet'), index=False)
print(f'\nR7b-0 terminó en {time.time() - t0:.0f} s. Guardado: fw5/r7b_tramos, r7b_decisiones, r7b_efecto, r7b_series.')


R7br0

In [ ]:
#[R7b-r0 v1.0] Carpeta espejo para R6r-1 y R6r-2 con PREFIJO = 'r7b' (copia de R7r-0 con otro prefijo)
import os, json, shutil
DRIVE = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
LOCAL = globals().get('LOCAL_R7B', '/content/isac_r7b/')
D_DP, D_FW1, D_FW5 = os.path.join(DRIVE, 'datos_parquet'), os.path.join(DRIVE, 'datos_parquet', 'fw1'), os.path.join(DRIVE, 'datos_parquet', 'fw5')
L_DP, L_FW = os.path.join(LOCAL, 'datos_parquet'), os.path.join(LOCAL, 'datos_parquet', 'fw1')
shutil.rmtree(LOCAL, ignore_errors=True); os.makedirs(L_FW, exist_ok=True); os.makedirs(os.path.join(LOCAL, 'salidas'), exist_ok=True)
def enlazar(src, dst):
    if os.path.exists(src): os.symlink(src, dst); return True
    return False
enl = {'partido_tecnico_periodo.parquet': enlazar(os.path.join(D_DP, 'partido_tecnico_periodo.parquet'), os.path.join(L_DP, 'partido_tecnico_periodo.parquet'))}
for f in ['r4_partidos.parquet', 'r4_minutos_v21.parquet', 'parametros_fw1.json', 'r4_perfiles_etapa.parquet', 'r6_perfiles_etapa.parquet',
          'r6b_perfiles_etapa.parquet', 'r6c_perfiles_etapa.parquet', 'r7b_tramos.parquet']:
    enl[f] = enlazar(os.path.join(D_FW5, f), os.path.join(L_FW, f))
enl['elo (fw1)'] = enlazar(os.path.join(D_FW1, 'elo_equipo_partido_e2.parquet'), os.path.join(L_FW, 'elo_equipo_partido_e2.parquet'))
print('--- (R7b-r0) Carpeta espejo', LOCAL); print({k: ('ok' if v else 'FALTA') for k, v in enl.items()})
assert all(enl[k] for k in ['partido_tecnico_periodo.parquet', 'r4_partidos.parquet', 'r4_minutos_v21.parquet', 'parametros_fw1.json',
                            'r4_perfiles_etapa.parquet', 'r7b_tramos.parquet', 'elo (fw1)']), 'Falta un insumo obligatorio: avisar a Claude'
BASE = LOCAL; PREFIJO = 'r7b'
print("Listo → corre R6r-1 v1.1 y R6r-2 v1.1 sin cambios; luego R7b-r9.")



R7b-r9

In [ ]:

# [R7b-r9 v1.0] Copiar resultados r7b_* a Drive y regresar BASE a Drive
import os, shutil
DRIVE = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
LOCAL = globals().get('LOCAL_R7B', '/content/isac_r7b/')
L_FW = os.path.join(LOCAL, 'datos_parquet', 'fw1')
D_FW5, D_SAL5 = os.path.join(DRIVE, 'datos_parquet', 'fw5'), os.path.join(DRIVE, 'salidas', 'fw5'); os.makedirs(D_SAL5, exist_ok=True)
n = []
for src_dir, dst_dir in [(L_FW, D_FW5), (os.path.join(LOCAL, 'salidas'), D_SAL5)]:
    for f in os.listdir(src_dir):
        s = os.path.join(src_dir, f)
        if f.startswith('r7b_') and os.path.isfile(s) and not os.path.islink(s):
            shutil.copy2(s, os.path.join(dst_dir, f)); n.append(f)
print('--- (R7b-r9) Copiados a Drive:', sorted(n)); BASE = DRIVE; print('BASE =', BASE)


R7o

In [ ]:
# R7o v1.1 · OBV por sistema: ¿la línea de 3/5 (y cada estructura) se asocia con más o menos valor que la de 4?
# v1.1 (2026-10-16): el OBV se busca en orden: columna de R1 → caché r6c_extra (fw5/fw1) → eventos crudos (v1.0 solo buscaba la caché).
# =====================================================================
# Sesión 10 · 2026-10-15 · Hackathon ISAC 2026, Track 2 · Idea de Arturo · Decisiones S10-18 a S10-21
# S10-18 A: OBV neto por 90 min (valor de mis acciones − valor de las del rival; obv_total_net, caché de R6c).
# S10-19: (1) dentro del mismo técnico · (2) ajustado por Elo del rival y localía · (3) por marcador · (4) arranque frente a
#         cambio en el partido. Mínimo 180 min en cada grupo (sensibilidad 90 y 360). IC: remuestreo de partidos, 2,000.
# S10-20: moda frente a rendimiento por temporada. S10-20b: ¿los semifinalistas anticipan la moda del torneo siguiente?
# S10-21: familia (resumen) y familia × puntas (subdivisión: 1, 2 o 3 delanteros = último dígito del código de formación).
# Tramos ("piezas"): intervalos con la misma formación activa y el mismo marcador, en periodos 1-2; desde la primera roja
# del partido en adelante no se usa (11 contra 10 distorsiona el valor). Asociación, no causa.
# Antes: celda 1. Ruta fija. Tarda ~5-10 min la primera vez (une el OBV de R6c con el reloj de los eventos de R1).
# =====================================================================

#[R7o v1.1] -----------------------------------------------------------
import os, json, glob, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5')
REHACER = False
SEMILLA, B = 2026, 2000
MIN_MIN, MIN_SENS = 180, [90, 360]
rng = np.random.default_rng(SEMILLA)
pd.set_option('display.width', 250)
t0 = time.time()
with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
    TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id'); MIDS = set(Mm.match_id)
FOR = pd.read_parquet(os.path.join(FW5, 'r7_formaciones.parquet'))
PER = pd.read_parquet(os.path.join(FW5, 'r7_periodos.parquet')); PER = PER[PER.period.isin([1, 2])]
G = pd.read_parquet(os.path.join(FW5, 'r7u_goles.parquet')); G = G[G.period.isin([1, 2])]
TRA = pd.read_parquet(os.path.join(FW5, 'r7_tramos.parquet'), columns=['match_id', 'to_clk', 'to_period', 'end_reason'])
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
elo = pd.read_parquet(os.path.join(FW1, 'elo_equipo_partido_e2.parquet'))[['match_id', 'team', 'elo_rival_pre']]
K = lambda p, c: p * 1000.0 + c                      # orden (periodo, reloj): evita el traslape 45-48 entre 1T y 2T
fin = {(m, p): f for m, p, f in zip(PER.match_id, PER.period, PER.fin)}
ini = {1: 0.0, 2: 45.0}
roja = (TRA[TRA.end_reason.isin(['Red Card', 'Second Yellow Card', 'Sent Off'])].assign(k=lambda d: K(d['to_period'], d['to_clk']))
        .groupby('match_id')['k'].min().to_dict())
def puntas(f):
    try: return int(str(int(f))[-1])
    except Exception: return np.nan
FAMILIA = lambda f: {'4': 'linea_4', '3': 'linea_3_5', '5': 'linea_3_5'}.get(str(int(f))[0], 'otra') if pd.notna(f) else None

# ---------- 1. Piezas: formación activa × marcador, con minutos reales ----------
ruta_p = os.path.join(FW5, 'r7o_piezas.parquet')
filas = []
FORk = {k: g.assign(k=K(g.period, g.clk)).sort_values('k') for k, g in FOR[FOR.period.isin([1, 2])].groupby(['match_id', 'team'])}
GLk = {m: g.assign(k=K(g.period, g.clk)) for m, g in G.groupby('match_id')}
for (m, t), f in FORk.items():
    if m not in MIDS: continue
    g = GLk.get(m, pd.DataFrame(columns=['team', 'k'])); s = 1 if t == loc.at[m, 'local'] else -1
    cortes = set(f.k) | set(g.k)
    for p in (1, 2):
        if (m, p) not in fin: continue
        a0, a1 = K(p, ini[p]), K(p, fin[(m, p)])
        bps = sorted({a0, a1} | {c for c in cortes if a0 < c < a1} | ({roja[m]} if m in roja and a0 < roja[m] < a1 else set()))
        for a, b in zip(bps[:-1], bps[1:]):
            if m in roja and a >= roja[m]: continue
            act = f[f.k <= a + 1e-9]
            if act.empty: continue
            r = act.iloc[-1]
            d = int((g.k < a + 1e-9)[g.team == t].sum() - (g.k < a + 1e-9)[g.team != t].sum()) if len(g) else 0
            off = loc.at[m, 'offset']; off = 0.0 if pd.isna(off) else float(off)
            e = int(np.sign(d + s * off))                      # marcador efectivo del equipo (offset de liguilla de R4)
            filas.append((m, t, p, a, b, (b - a), r.formation, str(r.reason), e))
PZ = pd.DataFrame(filas, columns=['match_id', 'team', 'period', 'k0', 'k1', 'minutos', 'formation', 'reason', 'e3'])
PZ['familia'] = PZ.formation.map(FAMILIA); PZ['puntas'] = PZ.formation.map(puntas)
PZ['estructura'] = PZ.familia + '_' + PZ.puntas.astype('Int64').astype(str)
PZ['origen'] = np.where(PZ.reason.str.contains('Starting', case=False), 'arranque', 'cambio')
PZ = PZ[(PZ.minutos > 0) & PZ.familia.isin(['linea_4', 'linea_3_5'])].reset_index(drop=True)
print(f'--- (1) Piezas: {len(PZ):,} · minutos {PZ.minutos.sum():,.0f} · % minutos con línea de 3/5: '
      f'{100 * PZ.loc[PZ.familia == "linea_3_5", "minutos"].sum() / PZ.minutos.sum():.1f} · partidos sin usar desde la roja: {len(roja)}')

# ---------- 2. OBV por pieza (eventos de R1 + obv_total_net de la caché de R6c) ----------
ruta_o = os.path.join(FW5, 'r7o_obv_eventos.parquet')
if os.path.exists(ruta_o) and not REHACER:
    OB = pd.read_parquet(ruta_o)
else:
    piezas = []
    DIR_EV = os.path.join(DP, 'liga', 'eventos')
    for temp in TEMPS:
        t_ = temp.replace('/', '-')
        r1 = os.path.join(FW1, f'eventos_{t_}.parquet'); disp = set(pq.read_schema(r1).names)
        col_obv = next((c for c in ['obv_total_net', 'obv'] if c in disp), None)          # v1.1: 1) el OBV ya viene en R1
        cols = [c for c in ['match_id', 'index', 'period', 'minute', 'second', 'team.name'] + ([col_obv] if col_obv else []) if c in disp]
        E = pd.read_parquet(r1, columns=cols); E = E[E.match_id.isin(MIDS) & E.period.isin([1, 2])]
        if 'second' not in E: E['second'] = 30
        if col_obv:
            E = E.rename(columns={col_obv: 'obv'}); fuente = f'R1 ({col_obv})'
        else:
            rx = [r for r in [os.path.join(FW5, f'r6c_extra_{t_}.parquet'), os.path.join(FW1, f'r6c_extra_{t_}.parquet')] if os.path.exists(r)]
            if rx:                                                                           # 2) caché de R6c
                X = pd.read_parquet(rx[0]); c_ = next(c for c in ['obv', 'obv_total_net'] if c in X.columns)
                E = E.merge(X[['match_id', 'index', c_]].rename(columns={c_: 'obv'}), on=['match_id', 'index'], how='inner'); fuente = os.path.basename(rx[0])
            else:                                                                            # 3) eventos crudos por partido (lento)
                xs = []
                for m in E.match_id.unique():
                    r = os.path.join(DIR_EV, f'{m}.parquet')
                    if not os.path.exists(r) or 'obv_total_net' not in pq.read_schema(r).names: continue
                    xs.append(pd.read_parquet(r, columns=['index', 'obv_total_net']).assign(match_id=m).rename(columns={'obv_total_net': 'obv'}))
                if not xs: print(f'  {temp}: NO se encontró OBV en ninguna fuente → temporada fuera'); continue
                E = E.merge(pd.concat(xs), on=['match_id', 'index'], how='inner'); fuente = 'eventos crudos'
        E = E[E.obv.notna()]
        piezas.append(pd.DataFrame({'match_id': E.match_id, 'team': E['team.name'], 'k': K(E.period, E.minute + E.second / 60), 'obv': E.obv}))
        print(f'  {temp}: {E.match_id.nunique()} partidos con OBV · {len(E):,} acciones · fuente: {fuente} ({time.time() - t0:.0f} s)')
    assert piezas, 'No se encontró OBV en ninguna temporada: mandar a Claude la lista de columnas de fw1/eventos_2024-2025.parquet'
    OB = pd.concat(piezas, ignore_index=True); OB.to_parquet(ruta_o, index=False)
PZ['pid'] = np.arange(len(PZ)); PZ['rival'] = [loc.at[m, 'visita'] if t == loc.at[m, 'local'] else loc.at[m, 'local'] for m, t in zip(PZ.match_id, PZ.team)]
OBm = {m: g for m, g in OB.groupby('match_id')}
of, oc = np.zeros(len(PZ)), np.zeros(len(PZ))
for (m, t), g in PZ.groupby(['match_id', 'team']):
    o = OBm.get(m)
    if o is None: continue
    k0, k1, ids = g.k0.values, g.k1.values, g.pid.values
    for propio, dest in [(True, of), (False, oc)]:
        x = o[(o.team == t) == propio]
        idx = np.searchsorted(k0, x.k.values, side='right') - 1
        ok = (idx >= 0) & (x.k.values < k1[np.clip(idx, 0, None)])
        np.add.at(dest, ids[idx[ok]], x.obv.values[ok])
PZ['obv_for'], PZ['obv_contra'] = of, oc
PZ['obv90'] = 90 * (PZ.obv_for - PZ.obv_contra) / PZ.minutos
print(f'--- (2) OBV asignado: a favor {PZ.obv_for.sum():,.1f} · en contra {PZ.obv_contra.sum():,.1f} · '
      f'OBV neto por 90 de la liga {90 * (PZ.obv_for - PZ.obv_contra).sum() / PZ.minutos.sum():+.3f} (debe ser ~0)')

# ---------- 3. Ajuste por rival y localía (S10-19, control 2): residuo frente a lo esperado por la liga ----------
PZ = PZ.merge(elo, on=['match_id', 'team'], how='left').merge(ptp[['match_id', 'team', 'etapa_id', 'tec_p', 'temporada']], on=['match_id', 'team'], how='left')
PZ['local'] = (PZ.team == PZ.match_id.map(loc['local'])).astype(float)
PZ['elo_c'] = (PZ.elo_rival_pre - 1500) / 100
# Pendientes estimadas DENTRO de cada etapa (efectos fijos por técnico): en una regresión de toda la liga, la fuerza propia
# se mezcla con la del rival (los equipos fuertes enfrentan rivales más débiles en promedio) y el ajuste sale sesgado.
PZ = PZ[PZ.etapa_id.notna()].reset_index(drop=True)
X = np.column_stack([PZ.elo_c.fillna(PZ.elo_c.mean()), PZ.local] + [(PZ.e3 == s).astype(float) for s in (-1, 1)])
w = PZ.minutos.values; y = PZ.obv90.values
def desmedia(v):
    v = pd.DataFrame(v); sw = pd.Series(w).groupby(PZ.etapa_id.values).transform('sum').values
    return v.values - (v.mul(w, axis=0).groupby(PZ.etapa_id.values).transform('sum').values / sw[:, None])
Xd, yd = desmedia(X), desmedia(y.reshape(-1, 1)).ravel()
beta = np.linalg.lstsq(Xd * np.sqrt(w)[:, None], yd * np.sqrt(w), rcond=None)[0]
PZ['res'] = y - X @ beta
print(f'--- (3) Esperado dentro del técnico: por 100 de Elo del rival {beta[0]:+.3f} · local {beta[1]:+.3f} · '
      f'abajo {beta[2]:+.3f} · arriba {beta[3]:+.3f} (OBV neto por 90; efectos fijos por etapa)')
naive = PZ.groupby('familia').apply(lambda d: np.average(d.obv90, weights=d.minutos), include_groups=False)
print(f'    Comparación ingenua (sin controles, toda la liga): línea de 3/5 − línea de 4 = {naive["linea_3_5"] - naive["linea_4"]:+.3f}')

# ---------- 4. Comparación dentro del técnico (controles 1, 3 y 4) con remuestreo de partidos ----------
mids_l = np.array(sorted(PZ.match_id.unique())); pos_m = {m: i for i, m in enumerate(mids_l)}
PZ['mi'] = PZ.match_id.map(pos_m)
BOOT = np.stack([np.bincount(rng.integers(0, len(mids_l), len(mids_l)), minlength=len(mids_l)) for _ in range(B)]).astype(float)
def comparar(d, col, a, b, minimo=MIN_MIN):
    """Diferencia a − b en OBV ajustado, dentro de cada etapa con ≥ minimo min en ambos; promedio ponderado por la media armónica de minutos."""
    d = d[d[col].isin([a, b])]
    if d.empty: return dict(etapas=0)
    mins = d.groupby(['etapa_id', col]).minutos.sum().unstack()
    ok = mins.index[(mins.get(a, 0) >= minimo) & (mins.get(b, 0) >= minimo)] if a in mins and b in mins else []
    d = d[d.etapa_id.isin(ok)]
    if d.empty: return dict(etapas=0)
    et_c = d.etapa_id.astype('category'); ei = et_c.cat.codes.values; ne = len(et_c.cat.categories)
    ga = (d[col] == a).values; W = d.minutos.values; R = d.res.values; MI = d.mi.values
    def estima(cnt):
        wv = cnt[MI] * W
        swa = np.bincount(ei[ga], weights=wv[ga], minlength=ne); swb = np.bincount(ei[~ga], weights=wv[~ga], minlength=ne)
        sra = np.bincount(ei[ga], weights=(wv * R)[ga], minlength=ne); srb = np.bincount(ei[~ga], weights=(wv * R)[~ga], minlength=ne)
        ok_ = (swa > 0) & (swb > 0)
        if not ok_.any(): return np.nan
        dif = sra[ok_] / swa[ok_] - srb[ok_] / swb[ok_]; hw = 2 / (1 / swa[ok_] + 1 / swb[ok_])
        return np.average(dif, weights=hw)
    est = estima(np.ones(len(mids_l))); bs = np.array([estima(c) for c in BOOT[:500]])     # 500 para cada estrato
    return dict(etapas=len(ok), dif=est, lo=np.nanpercentile(bs, 2.5), hi=np.nanpercentile(bs, 97.5))
def fila(nombre, r): return {'comparacion': nombre, **r}
res = []
res.append(fila('3/5 − 4 · todo', comparar(PZ, 'familia', 'linea_3_5', 'linea_4')))
for s, n in [(-1, 'abajo'), (0, 'empate'), (1, 'arriba')]:
    res.append(fila(f'3/5 − 4 · {n}', comparar(PZ[PZ.e3 == s], 'familia', 'linea_3_5', 'linea_4')))
for o in ['arranque', 'cambio']:
    res.append(fila(f'3/5 − 4 · {o}', comparar(PZ[PZ.origen == o], 'familia', 'linea_3_5', 'linea_4')))
for mm in MIN_SENS:
    res.append(fila(f'3/5 − 4 · todo (mínimo {mm} min)', comparar(PZ, 'familia', 'linea_3_5', 'linea_4', mm)))
print('\n--- (4) Dentro del técnico, OBV neto por 90 ajustado por rival y localía (IC 95%) ---')
print(pd.DataFrame(res).round(3).to_string(index=False))

# ---------- 5. Subdivisión familia × puntas (S10-21) ----------
print('\n--- (5) Estructuras: % de minutos de la liga y reparto por marcador ---')
est = PZ.groupby('estructura').minutos.sum().sort_values(ascending=False)
por_m = PZ.pivot_table(index='estructura', columns='e3', values='minutos', aggfunc='sum', fill_value=0)
por_m = 100 * por_m.div(por_m.sum(1), axis=0); por_m.columns = [{-1: '%abajo', 0: '%empate', 1: '%arriba'}[c] for c in por_m.columns]
print(pd.concat([(100 * est / est.sum()).rename('%_minutos'), por_m, PZ.groupby('estructura').apply(
      lambda d: 100 * d.loc[d.origen == 'cambio', 'minutos'].sum() / d.minutos.sum(), include_groups=False).rename('%_por_cambio')], axis=1)
      .round(1).loc[est.index].to_string())
res = []
for a, b in [('linea_4_1', 'linea_4_2'), ('linea_4_3', 'linea_4_1'), ('linea_3_5_1', 'linea_3_5_2'), ('linea_3_5_3', 'linea_3_5_2'),
             ('linea_3_5_1', 'linea_4_1'), ('linea_3_5_2', 'linea_4_2')]:
    res.append(fila(f'{a} − {b}', comparar(PZ, 'estructura', a, b)))
    res.append(fila(f'   … solo arriba', comparar(PZ[PZ.e3 == 1], 'estructura', a, b)))
print('Dentro del técnico (OBV neto por 90 ajustado; "etapas" = técnicos con ≥ 180 min en ambas):')
print(pd.DataFrame(res).round(3).to_string(index=False))

# ---------- 6. Moda frente a rendimiento (S10-20) y quién la impone (S10-20b) ----------
print('\n--- (6) Por temporada: % de minutos con línea de 3/5 y diferencia de OBV dentro del técnico ---')
filas = []
for temp, d in PZ.groupby('temporada'):
    r = comparar(d, 'familia', 'linea_3_5', 'linea_4')
    filas.append(dict(temporada=temp, pct_3_5=100 * d.loc[d.familia == 'linea_3_5', 'minutos'].sum() / d.minutos.sum(), **r))
print(pd.DataFrame(filas).round(3).to_string(index=False))
semis = Mm[Mm.tipo.isin(['semis', 'final'])]
sf = {t: set(g.local) | set(g.visita) for t, g in semis.groupby('torneo')}
PZ['torneo'] = PZ.match_id.map(loc['torneo'])
TOR = list(Mm.groupby('torneo').fecha.min().sort_values().index) if 'fecha' in Mm else sorted(PZ.torneo.dropna().unique())
filas = []
for i, t in enumerate(TOR):
    d = PZ[PZ.torneo == t]; s_ = sf.get(t, set())
    p = lambda x: 100 * x.loc[x.familia == 'linea_3_5', 'minutos'].sum() / max(x.minutos.sum(), 1e-9)
    filas.append(dict(torneo=t, semifinalistas=p(d[d.team.isin(s_)]), resto=p(d[~d.team.isin(s_)])))
S = pd.DataFrame(filas); S['resto_siguiente'] = S.resto.shift(-1); S['cambio_resto'] = S.resto_siguiente - S.resto
S['ventaja_semis'] = S.semifinalistas - S.resto
print('Por torneo: % de minutos con línea de 3/5 (semifinalistas y finalistas frente al resto) y el resto en el torneo siguiente')
print(S.round(1).to_string(index=False))
ok = S.dropna()
print(f'Correlación entre "los semifinalistas usan más 3/5 que el resto" y "el resto sube en el torneo siguiente": '
      f'{ok.ventaja_semis.corr(ok.cambio_resto):+.2f} ({len(ok)} pasos; descriptivo)')

# ---------- 7. América ----------
am = PZ[PZ.team.astype(str).str.startswith('Am')]
print('\n--- (7) América: minutos por familia y estructura por etapa ---')
print(am.pivot_table(index='etapa_id', columns='estructura', values='minutos', aggfunc='sum', fill_value=0).round(0).to_string())
PZ.drop(columns=['mi']).to_parquet(ruta_p, index=False)
print(f'\nR7o terminó en {time.time() - t0:.0f} s. Guardado: fw5/r7o_piezas, r7o_obv_eventos.')


R7c-0

In [ ]:
#[R7c-0 v1.0] Acciones con puesto, rasgos r7c y verificación espacial ------------------------------------------
import os, json, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5'); DIR_EV = os.path.join(DP, 'liga', 'eventos')
REHACER = False
MIN_ACC = 10                                   # un jugador cuenta para libertad y carga estructural con ≥ 10 acciones en el partido
CARRIL = (80 / 3, 160 / 3)                     # carriles izquierdo (y < 26.7), central y derecho (y > 53.3); y baja = izquierda
X_CARGA = 40                                   # carga de un costado: acciones de medio campo hacia adelante
VENT_V = 10                                    # verificación: 10 min antes y 10 min después del cambio
pd.set_option('display.width', 250)
t0 = time.time()
GRUPO = {1: 'POR', 3: 'CEN', 4: 'CEN', 5: 'CEN', 2: 'LAT', 6: 'LAT', 7: 'LAT', 8: 'LAT', 9: 'MCD', 10: 'MCD', 11: 'MCD',
         13: 'INT', 14: 'INT', 15: 'INT', 18: 'MP', 19: 'MP', 20: 'MP', 12: 'EXT', 16: 'EXT', 17: 'EXT', 21: 'EXT',
         22: 'DEL', 23: 'DEL', 24: 'DEL', 25: 'DEL'}
LADO = {**{p: 'D' for p in [2, 3, 7, 9, 12, 13, 17, 18, 22]}, **{p: 'I' for p in [5, 6, 8, 11, 15, 16, 20, 21, 24]},
        **{p: 'C' for p in [1, 4, 10, 14, 19, 23, 25]}}
def mueve(a, b):                                # copia exacta de R7a v1.1 (S10-11 C)
    if pd.isna(a) or pd.isna(b): return True
    a, b = int(a), int(b)
    if GRUPO.get(a) != GRUPO.get(b): return True
    return {LADO.get(a), LADO.get(b)} == {'I', 'D'}
K = lambda p, c: p * 1000.0 + c
with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
    TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); MIDS = set(Mm.match_id); TEMP_M = Mm.set_index('match_id')['temporada']

# ---------- 1. Acciones con balón (juego abierto) con jugador y ubicación ----------
ruta_a = os.path.join(FW5, 'r7c_acciones.parquet')
if os.path.exists(ruta_a) and not REHACER:
    AC = pd.read_parquet(ruta_a)
else:
    piezas = []
    for temp in TEMPS:
        t_ = temp.replace('/', '-'); r1 = os.path.join(FW1, f'eventos_{t_}.parquet'); disp = set(pq.read_schema(r1).names)
        cols = [c for c in ['match_id', 'index', 'period', 'minute', 'second', 'type.name', 'team.name', 'loc_x', 'loc_y', 'fase',
                            'es_reanudacion', 'ball_receipt.outcome.name', 'player.id'] if c in disp]
        E = pd.read_parquet(r1, columns=cols); E = E[E.match_id.isin(MIDS) & E.period.isin([1, 2])]
        for c in ['second', 'ball_receipt.outcome.name']:
            if c not in E: E[c] = 30 if c == 'second' else np.nan
        tn = E['type.name']
        E = E[(E['fase'] != 'bp') & ~E['es_reanudacion'].fillna(False).astype(bool) & E.loc_x.notna() & E.loc_y.notna()
              & tn.isin(['Pass', 'Carry', 'Ball Receipt*', 'Dribble', 'Shot']) & ~((tn == 'Ball Receipt*') & E['ball_receipt.outcome.name'].notna())]
        fuente = 'R1'
        if 'player.id' not in E:                                              # caché de R6b (jugador) o eventos crudos
            rx = [r for r in [os.path.join(FW5, f'r6b_extra_{t_}.parquet'), os.path.join(FW1, f'r6b_extra_{t_}.parquet')] if os.path.exists(r)]
            if rx and 'jugador' in pq.read_schema(rx[0]).names:
                E = E.merge(pd.read_parquet(rx[0], columns=['match_id', 'index', 'jugador']).rename(columns={'jugador': 'player.id'}),
                            on=['match_id', 'index'], how='inner'); fuente = os.path.basename(rx[0])
            else:
                xs = [pd.read_parquet(os.path.join(DIR_EV, f'{m}.parquet'), columns=['index', 'player.id']).assign(match_id=m)
                      for m in E.match_id.unique() if os.path.exists(os.path.join(DIR_EV, f'{m}.parquet'))]
                E = E.merge(pd.concat(xs), on=['match_id', 'index'], how='inner'); fuente = 'eventos crudos'
        piezas.append(pd.DataFrame({'match_id': E.match_id.values, 'team': E['team.name'].values, 'player_id': E['player.id'].values,
                                    'k': K(E.period, E.minute + E.second / 60).values, 'x': E.loc_x.values, 'y': E.loc_y.values}))
        print(f'  {temp}: {E.match_id.nunique()} partidos · {len(E):,} acciones con balón · jugador desde {fuente} ({time.time() - t0:.0f} s)')
    AC = pd.concat(piezas, ignore_index=True).dropna(subset=['player_id']); AC['player_id'] = AC.player_id.astype('int64')
    # Puesto activo de cada acción (tramos de R7a)
    TRA = pd.read_parquet(os.path.join(FW5, 'r7_tramos.parquet'), columns=['match_id', 'player_id', 'position_id', 'from_clk', 'from_period', 'to_clk', 'to_period'])
    TRA = TRA[TRA.from_period.isin([1, 2])].copy()
    TRA['k0'] = K(TRA['from_period'], TRA['from_clk']); TRA['k1'] = np.where(TRA['to_clk'].notna(), K(TRA['to_period'].fillna(2), TRA['to_clk']), np.inf)
    AC = AC.merge(TRA[['match_id', 'player_id', 'position_id', 'k0', 'k1']], on=['match_id', 'player_id'], how='inner')
    AC = AC[(AC.k >= AC.k0 - 0.02) & (AC.k < AC.k1 + 0.02)].drop_duplicates(['match_id', 'player_id', 'k', 'x', 'y']).drop(columns=['k0', 'k1'])
    AC.to_parquet(ruta_a, index=False)
AC['grupo'] = AC.position_id.map(GRUPO); AC['lado'] = AC.position_id.map(LADO); AC['puesto'] = AC.grupo + '_' + AC.lado
AC['temporada'] = AC.match_id.map(TEMP_M)
print(f'--- (1) Acciones con balón con puesto: {len(AC):,} · partidos {AC.match_id.nunique()} · sin puesto conocido {int(AC.grupo.isna().sum())}')
print('Acciones por grupo (%):', (100 * AC.grupo.value_counts(normalize=True)).round(1).to_dict())

# ---------- 2. Norma de la liga por puesto y temporada; rasgos por equipo-partido ----------
NORMA = AC.groupby(['temporada', 'puesto'])[['x', 'y']].mean()
print('Norma de la liga (x media de 0 a 120 hacia el arco rival; y de 0 izquierda a 80 derecha), última temporada:')
print(NORMA.loc[TEMPS[-1]].round(1).T.to_string())
AC['amp'] = (AC.y - 40).abs()
jp = AC.groupby(['match_id', 'team', 'player_id', 'puesto', 'temporada']).agg(n=('x', 'size'), x=('x', 'mean'), y=('y', 'mean')).reset_index()
jp = jp.join(NORMA.rename(columns={'x': 'nx', 'y': 'ny'}), on=['temporada', 'puesto'])
jp['dist'] = np.sqrt((jp.x - jp.nx) ** 2 + (jp.y - jp.ny) ** 2)
campo = jp[(jp.n >= MIN_ACC) & ~jp.puesto.str.startswith('POR')]
k = ['match_id', 'team']
def media(d, g, col): return d[d.grupo.isin(g)].groupby(k)[col].mean()
R = pd.DataFrame({
    'altura_laterales': media(AC, ['LAT'], 'x'), 'amplitud_bandas': media(AC, ['EXT'], 'amp'),
    'altura_contenciones': media(AC, ['MCD'], 'x'), 'altura_interiores_mp': media(AC, ['INT', 'MP'], 'x'),
    'altura_delanteros': media(AC, ['DEL'], 'x'), 'portero_libero': media(AC, ['POR'], 'x'),
    'libertad_posicional': campo.groupby(k)['dist'].mean()})
fr = AC[(AC.x >= X_CARGA) & (AC.grupo != 'POR')]
iz = fr[fr.y < CARRIL[0]].groupby(k).size(); de = fr[fr.y > CARRIL[1]].groupby(k).size()
lr = pd.concat([iz.rename('i'), de.rename('d')], axis=1).fillna(0)
R['carga_acciones'] = 100 * (lr.i - lr.d).abs() / (lr.i + lr.d).replace(0, np.nan)
R['lado_carga_acciones'] = np.sign(lr.d - lr.i)                     # descriptivo: +1 derecha, −1 izquierda
pi = campo[campo.y < CARRIL[0]].groupby(k).size(); pd_ = campo[campo.y > CARRIL[1]].groupby(k).size()
pp = pd.concat([pi.rename('i'), pd_.rename('d')], axis=1).fillna(0)
R['carga_estructural'] = (pp.i - pp.d).abs()
R = R.reset_index()
RASGOS = ['altura_laterales', 'amplitud_bandas', 'altura_contenciones', 'altura_interiores_mp', 'altura_delanteros', 'portero_libero',
          'libertad_posicional', 'carga_acciones', 'carga_estructural']
print('\n--- (2) Rasgos por equipo-partido (media, p10, p90 y nulos) ---')
print(R[RASGOS].describe(percentiles=[.1, .9]).T[['count', 'mean', '10%', '90%']].round(2).assign(nulos=R[RASGOS].isna().sum().values).to_string())
print('Lado de la carga (acciones, % de partidos): izquierda', round(100 * (R.lado_carga_acciones < 0).mean(), 1), '· derecha', round(100 * (R.lado_carga_acciones > 0).mean(), 1))
filas = [pd.DataFrame({'metrica': m, 'match_id': R.match_id, 'team': R.team, 'e5': 0, 'e5b': 0, 'cuarto': 1, 'num': R[m], 'den': 1.0}).dropna(subset=['num']) for m in RASGOS]
TRC = pd.concat(filas, ignore_index=True); TRC.to_parquet(os.path.join(FW5, 'r7c_tramos.parquet'), index=False)
R.to_parquet(os.path.join(FW5, 'r7c_equipo_partido.parquet'), index=False)
print('Correlaciones entre rasgos (equipo-partido; |r| alto = miden casi lo mismo):')
print(R[RASGOS].corr().round(2).to_string())

# ---------- 3. Verificación espacial de la tipología de cambios (S10-26) ----------
SUB = pd.read_parquet(os.path.join(FW5, 'r7_subs.parquet')); SUB = SUB[~SUB['lesion'].astype(bool) & SUB.period.isin([1, 2])]
T2 = pd.read_parquet(os.path.join(FW5, 'r7_tramos.parquet'), columns=['match_id', 'team', 'player_id', 'position_id', 'from_clk', 'from_period', 'start_reason'])
T2 = T2.sort_values(['match_id', 'player_id', 'from_clk']); T2['pos_prev'] = T2.groupby(['match_id', 'player_id'])['position_id'].shift()
mv = T2[T2.start_reason.astype(str).str.contains('Tactical', case=False)]
mv = mv.assign(real=[mueve(a, b) for a, b in zip(mv.pos_prev, mv.position_id)])
ACm = {m: g for m, g in AC.groupby('match_id')}
filas = []
for r in SUB.itertuples(index=False):
    a = ACm.get(r.match_id)
    if a is None: continue
    kk = K(r.period, r.clk); a = a[a.team == r.team]
    antes, desp = a[(a.k >= kk - VENT_V) & (a.k < kk)], a[(a.k > kk) & (a.k <= kk + VENT_V)]
    m_ = mv[(mv.match_id == r.match_id) & (mv.team == r.team) & ((mv.from_clk - r.clk).abs() <= 1.0)]
    reales = set(m_.loc[m_.real, 'player_id']) - {r.in_id, r.out_id}; etiqueta = set(m_.loc[~m_.real, 'player_id']) - {r.in_id, r.out_id}
    for pid in set(antes.player_id) & set(desp.player_id) - {r.in_id, r.out_id}:
        x0, x1 = antes[antes.player_id == pid], desp[desp.player_id == pid]
        if len(x0) < 3 or len(x1) < 3: continue
        clase = 'movido (real)' if pid in reales else ('solo etiqueta' if pid in etiqueta else 'no movido')
        filas.append((clase, np.hypot(x1.x.mean() - x0.x.mean(), x1.y.mean() - x0.y.mean()), abs(x1.y.mean() - x0.y.mean())))
VV = pd.DataFrame(filas, columns=['clase', 'desplazamiento', 'cambio_y'])
print('\n--- (3) Verificación espacial: desplazamiento de la posición media (yd), 10 min antes frente a 10 min después de un cambio ---')
print(VV.groupby('clase').agg(jugadores=('desplazamiento', 'size'), mediana=('desplazamiento', 'median'), media=('desplazamiento', 'mean'),
                              cambio_y_mediana=('cambio_y', 'median')).round(2).to_string())
rng = np.random.default_rng(2026)
a_, b_ = VV.loc[VV.clase == 'movido (real)', 'desplazamiento'].values, VV.loc[VV.clase == 'no movido', 'desplazamiento'].values
if len(a_) and len(b_):
    bs = [np.median(rng.choice(a_, len(a_))) - np.median(rng.choice(b_, len(b_))) for _ in range(2000)]
    print(f'Movido (real) − no movido, diferencia de medianas: {np.median(a_) - np.median(b_):+.2f} yd [{np.percentile(bs, 2.5):+.2f}, {np.percentile(bs, 97.5):+.2f}]')
VV.to_parquet(os.path.join(FW5, 'r7c_verificacion.parquet'), index=False)
print(f'\nR7c-0 terminó en {time.time() - t0:.0f} s. Guardado: fw5/r7c_tramos, r7c_equipo_partido, r7c_acciones, r7c_verificacion.')


R7c-r0

In [ ]:

#[R7c-r0 v1.0] Carpeta espejo para R6r-1 y R6r-2 con PREFIJO = 'r7c' (copia de R7b-r0 con otro prefijo)
import os, shutil
DRIVE = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
LOCAL = globals().get('LOCAL_R7C', '/content/isac_r7c/')
D_DP, D_FW1, D_FW5 = os.path.join(DRIVE, 'datos_parquet'), os.path.join(DRIVE, 'datos_parquet', 'fw1'), os.path.join(DRIVE, 'datos_parquet', 'fw5')
L_DP, L_FW = os.path.join(LOCAL, 'datos_parquet'), os.path.join(LOCAL, 'datos_parquet', 'fw1')
shutil.rmtree(LOCAL, ignore_errors=True); os.makedirs(L_FW, exist_ok=True); os.makedirs(os.path.join(LOCAL, 'salidas'), exist_ok=True)
def enlazar(src, dst):
    if os.path.exists(src): os.symlink(src, dst); return True
    return False
enl = {'partido_tecnico_periodo.parquet': enlazar(os.path.join(D_DP, 'partido_tecnico_periodo.parquet'), os.path.join(L_DP, 'partido_tecnico_periodo.parquet'))}
for f in ['r4_partidos.parquet', 'r4_minutos_v21.parquet', 'parametros_fw1.json', 'r4_perfiles_etapa.parquet', 'r6_perfiles_etapa.parquet',
          'r6b_perfiles_etapa.parquet', 'r6c_perfiles_etapa.parquet', 'r7c_tramos.parquet']:
    enl[f] = enlazar(os.path.join(D_FW5, f), os.path.join(L_FW, f))
enl['elo (fw1)'] = enlazar(os.path.join(D_FW1, 'elo_equipo_partido_e2.parquet'), os.path.join(L_FW, 'elo_equipo_partido_e2.parquet'))
print('--- (R7c-r0) Carpeta espejo', LOCAL); print({k: ('ok' if v else 'FALTA') for k, v in enl.items()})
assert all(enl[k] for k in ['partido_tecnico_periodo.parquet', 'r4_partidos.parquet', 'r4_minutos_v21.parquet', 'parametros_fw1.json',
                            'r4_perfiles_etapa.parquet', 'r7c_tramos.parquet', 'elo (fw1)']), 'Falta un insumo obligatorio: avisar a Claude'
BASE = LOCAL; PREFIJO = 'r7c'
print("Listo → corre R6r-1 v1.1 y R6r-2 v1.1 sin cambios; luego R7c-r9 y R7c-9.")

R7c-r9

In [ ]:

# [R7c-r9 v1.0] Copiar resultados r7c_* a Drive y regresar BASE a Drive
import os, shutil
DRIVE = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
LOCAL = globals().get('LOCAL_R7C', '/content/isac_r7c/')
L_FW = os.path.join(LOCAL, 'datos_parquet', 'fw1')
D_FW5, D_SAL5 = os.path.join(DRIVE, 'datos_parquet', 'fw5'), os.path.join(DRIVE, 'salidas', 'fw5'); os.makedirs(D_SAL5, exist_ok=True)
n = []
for src_dir, dst_dir in [(L_FW, D_FW5), (os.path.join(LOCAL, 'salidas'), D_SAL5)]:
    for f in os.listdir(src_dir):
        s = os.path.join(src_dir, f)
        if f.startswith('r7c_') and os.path.isfile(s) and not os.path.islink(s):
            shutil.copy2(s, os.path.join(dst_dir, f)); n.append(f)
print('--- (R7c-r9) Copiados a Drive:', sorted(n)); BASE = DRIVE; print('BASE =', BASE)



R7c9

In [ ]:

#[R7c-9 v1.0] Arquetipos (U6, S10-25): nombre solo si el rasgo es confiable y el técnico se aparta ≥ 1 desviación estándar
import os
import numpy as np
import pandas as pd
RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
FW5, SAL5 = os.path.join(RAIZ, 'datos_parquet', 'fw5'), os.path.join(RAIZ, 'salidas', 'fw5')
CONF_MIN, Z_MIN, ETAPA_MIN = 0.5, 1.0, 15
NOMBRES = {'altura_laterales': ('laterales altos', 'laterales que cuidan'), 'amplitud_bandas': ('extremos abiertos', 'extremos por dentro'),
           'altura_contenciones': ('contención adelantada', 'contención que espera'), 'altura_interiores_mp': ('interiores que pisan área', 'interiores que bajan'),
           'altura_delanteros': ('9 de área', '9 que baja'), 'portero_libero': ('portero líbero', 'portero de línea'),
           'libertad_posicional': ('roles libres', 'roles fijos'), 'carga_acciones': ('carga un costado', 'ataque equilibrado'),
           'carga_estructural': ('sobrecarga estructural', 'estructura simétrica')}
CF = pd.read_csv(os.path.join(SAL5, 'r7c_confiabilidad.csv'))
PC = pd.read_parquet(os.path.join(FW5, 'r7c_perfiles_contraidos.parquet'))
PC = PC[PC.partidos >= ETAPA_MIN]
ok = CF.loc[CF.conf >= CONF_MIN, 'metrica'].tolist()
print('Rasgos con confiabilidad ≥ 0.5 (pueden dar arquetipo):', ok)
print('Rasgos sin arquetipo por poca confiabilidad:', CF.loc[CF.conf < CONF_MIN, ['metrica', 'conf']].round(2).values.tolist())
filas = []
for met in ok:
    d = PC[PC.metrica == met].copy(); sd = d.perfil_mostrado.std(); d['z'] = d.perfil_mostrado / sd
    for r in d[d.z.abs() >= Z_MIN].itertuples(index=False):
        filas.append(dict(etapa_id=r.etapa_id, tecnico=r.tecnico, partidos=r.partidos, rasgo=met,
                          arquetipo=NOMBRES[met][0] if r.z > 0 else NOMBRES[met][1], z=r.z, valor=r.liga + r.perfil_mostrado, liga=r.liga))
AQ = pd.DataFrame(filas)
if AQ.empty:
    print('Ningún técnico se aparta ≥ 1 desviación estándar en rasgos confiables.')
else:
    print('\nArquetipos: cuántos técnicos (etapas de 15+) reciben cada uno')
    print(AQ.groupby(['rasgo', 'arquetipo']).size().rename('etapas').to_string())
    print('\nTécnicos con 3 o más arquetipos (los perfiles más marcados):')
    m = AQ.groupby(['etapa_id', 'tecnico']).arquetipo.apply(lambda s: ', '.join(sorted(s))).reset_index()
    m['n'] = m.arquetipo.str.count(',') + 1
    print(m[m.n >= 3].sort_values('n', ascending=False).head(15).to_string(index=False))
    print('\nAmérica:'); print(AQ[AQ.etapa_id.str.startswith('Am')][['tecnico', 'rasgo', 'arquetipo', 'z', 'valor', 'liga']].round(2).to_string(index=False))
    AQ.to_csv(os.path.join(SAL5, 'r7c_arquetipos.csv'), index=False)
print('\nR7c-9 terminó. Guardado: salidas/fw5/r7c_arquetipos.csv')


Sesión 11

R8-00

In [ ]:
#[R8-00 v1.0]
import os, re, json, glob, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5 = os.path.join(DP, 'fw5'); FW1 = os.path.join(DP, 'fw1'); LIGA = os.path.join(DP, 'liga')
N_POR_TEMP, SEMILLA = 60, 2026
AREA_X, AREA_Y, CHICA_X, CHICA_Y = 102, (18, 62), 114, (30, 50)          # G-4
t0 = time.time()
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
def en_area(x, y): return (x >= AREA_X) & (y >= AREA_Y[0]) & (y <= AREA_Y[1])
def en_chica(x, y): return (x >= CHICA_X) & (y >= CHICA_Y[0]) & (y <= CHICA_Y[1])
def pct(s): return round(100 * float(np.mean(s)), 1) if len(s) else np.nan

# ---------- (a) Inventario ----------
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet'))
print('=== (a) Inventario ===')
print('Columnas de fw5/r4_partidos:', list(Mm.columns))
if 'temporada' not in Mm:
    ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
    Mm = Mm.merge(ptp[['match_id', 'temporada']].drop_duplicates('match_id'), on='match_id', how='left')
    print('  (temporada tomada de partido_tecnico_periodo)')
carp = {}
for sub in ['eventos', 'frames360', 'jugadores360', 'lineups']:
    d = os.path.join(LIGA, sub)
    carp[sub] = {int(f.split('.')[0]) for f in os.listdir(d) if f.endswith('.parquet') and f.split('.')[0].isdigit()} if os.path.isdir(d) else set()
inv = Mm.groupby('temporada')['match_id'].apply(lambda s: pd.Series({'partidos': len(s), **{k: int(s.isin(v).sum()) for k, v in carp.items()}})).unstack()
print('Partidos de R4 (fw5) con archivo por carpeta de datos_parquet/liga/:'); print(inv.to_string())
r1 = sorted(glob.glob(os.path.join(FW1, 'eventos_*.parquet')))
print('R1 (fw1, solo lectura):', [os.path.basename(r) for r in r1])
if r1:
    nombres = pq.read_schema(r1[-1]).names
    print('  columnas de R1 útiles para R8:', [c for c in nombres if re.search(r'fase|reanud|t_pos|nace|trans|contra|obv|xg|jugador|player|set_piece|tecnica|technique', c)])

# ---------- (b) Esquema de los eventos crudos (muestra) ----------
rng = np.random.default_rng(SEMILLA)
mues = []
for temp, g in Mm[Mm.match_id.isin(carp['eventos'])].groupby('temporada'):
    ids = np.sort(g.match_id.values); mues += list(rng.choice(ids, size=min(N_POR_TEMP, len(ids)), replace=False))
TEMP = Mm.set_index('match_id')['temporada']
union = set()
for m in mues: union |= set(pq.read_schema(os.path.join(LIGA, 'eventos', f'{m}.parquet')).names)
GRUPOS = {'balón parado': r'set_piece', 'técnica / curva': r'technique|swing|straight', 'penal': r'penalty', 'OBV': r'^obv',
          'aéreo': r'aerial', 'relacionados': r'related', 'freeze frame': r'freeze', 'xG del tiro': r'statsbomb_xg',
          'id / orden / reloj': r'^(id|index|timestamp|minute|second|period)$', 'jugador': r'^player\.'}
print(f'\n=== (b) Columnas en la muestra ({len(mues)} partidos; {len(union)} columnas en total) ===')
for k, p in GRUPOS.items(): print(f'  {k:20s}:', sorted(c for c in union if re.search(p, c))[:14])
BASICAS = ['id', 'index', 'period', 'minute', 'second', 'type.name', 'team.name', 'player.id', 'possession', 'possession_team.name',
           'play_pattern.name', 'loc_x', 'loc_y', 'pass_end_x', 'pass_end_y', 'pass.type.name', 'pass.length', 'pass.height.name',
           'pass.outcome.name', 'pass.body_part.name', 'pass.cross', 'shot.type.name', 'shot.outcome.name', 'duel.type.name',
           'dribble.outcome.name', 'related_events', 'shot.freeze_frame']
EXTRA = [c for c in union if re.search(r'set_piece|technique|swing|straight|penalty|aerial_won|statsbomb_xg', c) or re.fullmatch(r'obv_(total|for|against)_net', c)]
USAR = [c for c in dict.fromkeys(BASICAS + EXTRA) if c in union]
piezas = []
for m in mues:
    r = os.path.join(LIGA, 'eventos', f'{m}.parquet'); disp = set(pq.read_schema(r).names)
    d = pd.read_parquet(r, columns=[c for c in USAR if c in disp]); d['match_id'] = m; piezas.append(d)
E = pd.concat(piezas, ignore_index=True)
for c in USAR:
    if c not in E: E[c] = np.nan
E['temporada'] = E.match_id.map(TEMP); E['t'] = E['minute'] * 60 + E['second']
E = E.sort_values(['match_id', 'index']).reset_index(drop=True)
SP = next((c for c in E.columns if 'set_piece' in c), None)
TEC = next((c for c in E.columns if c in ['pass.technique.name', 'pass.technique']), None)
OBV = next((c for c in ['obv_total_net'] if c in E.columns), None)
print(f'Eventos cargados: {len(E):,} · columna de balón parado: {SP} · técnica: {TEC} · OBV: {OBV} ({time.time() - t0:.0f} s)')
if SP: print(f'  valores de {SP} (todos los eventos):', E[SP].value_counts(dropna=False).head(6).to_dict())

# ---------- (c) Córners, tiros libres y saques de banda ----------
tn, pt = E['type.name'], E['pass.type.name']
co = E[pt == 'Corner'].copy()
co['directo'] = en_area(co.pass_end_x, co.pass_end_y)
co['yrel'] = np.where(co.loc_y < 40, co.pass_end_y, 80 - co.pass_end_y)    # 0 = banda del saque; postes en 36 y 44
print(f'\n=== (c) Córners: {len(co):,} ({len(co) / len(mues):.1f} por partido) · marcados {pct(co[SP].notna()) if SP else "s/c"}% · directos al área {pct(co.directo)}%')
print('  largo de los no directos (yd): ', co.loc[~co.directo, 'pass.length'].quantile([.1, .25, .5, .75, .9]).round(1).to_dict())
xb = pd.cut(co.pass_end_x, [-1, 96, 102, 108, 114, 121], labels=['<96', '96-102', '102-108', '108-114', '114-120'])
yb = pd.cut(co.yrel, [-100, 18, 30, 36, 44, 50, 62, 200], labels=['<18', '18-30', '30-36', '36-44', '44-50', '50-62', '>62'])
print('  Destino de TODOS los córners, % del total (filas: x del destino; columnas: y relativa al lado del saque):')
print(pd.crosstab(xb, yb, normalize='all').mul(100).round(1).to_string())
if TEC:
    print(f'  Técnica ({TEC}), % de córners:', co[TEC].value_counts(dropna=False, normalize=True).mul(100).round(1).to_dict())
    print('  Técnica por franja y de los directos (% por fila):')
    dd = co[co.directo]; print(pd.crosstab(dd[TEC].fillna('sin dato'), yb[co.directo], normalize='index').mul(100).round(1).to_string())
for f in [c for c in E.columns if re.search(r'swing|straight', c)]: print(f'  bandera {f}: {int(co[f].eq(True).sum())} córners')
fk = E[(pt == 'Free Kick') & (E.loc_x >= 80)]; fs = E[(E['shot.type.name'] == 'Free Kick')]
print(f'Tiros libres en el último tercio: pases {len(fk):,} (marcados {pct(fk[SP].notna()) if SP else "s/c"}%; al área {pct(en_area(fk.pass_end_x, fk.pass_end_y))}%;'
      f' cortos < 15 yd {pct(fk["pass.length"] < 15)}%) · tiros directos {len(fs):,} (mediana x {fs.loc_x.median():.1f})')
th = E[(pt == 'Throw-in') & (E.loc_x >= 80)].copy(); th['area'] = en_area(th.pass_end_x, th.pass_end_y)
print(f'Saques de banda en el último tercio: {len(th):,} · marcados {pct(th[SP].notna()) if SP else "s/c"}% · terminan en el área {pct(th.area)}%'
      f' · largo (yd) {th["pass.length"].quantile([.5, .75, .9, .95]).round(1).to_dict()}')
tt = th.groupby('team.name')['area'].agg(['size', 'mean']).query('size >= 30').sort_values('mean')
print(f'  % al área por equipo (≥ 30 saques): mínimo {100 * tt["mean"].min():.1f}, mediana {100 * tt["mean"].median():.1f}, máximo {100 * tt["mean"].max():.1f}')

# ---------- (d) 360 en córners ----------
print('\n=== (d) 360 ===')
m0 = next(m for m in mues if m in carp['frames360'] and m in carp['jugadores360'])
fcols = pq.read_schema(os.path.join(LIGA, 'frames360', f'{m0}.parquet')).names
jcols = pq.read_schema(os.path.join(LIGA, 'jugadores360', f'{m0}.parquet')).names
print('  columnas frames360:', fcols); print('  columnas jugadores360:', jcols)
KF = ('event_uuid', 'id') if 'event_uuid' in fcols and 'id' in E else (('event_index', 'index') if 'event_index' in fcols else None)
KJ = next((k for k in ['event_uuid', 'event_index'] if k in jcols), None)
print(f'  enlace frame→evento: {KF} · enlace jugador→evento: {KJ}')
assert KF and KJ, 'No encontré la llave de enlace del 360: mandar esta salida a Claude'
co['key'] = co[KF[1]]; E['key'] = E[KF[1]]
# primer contacto = siguiente evento del partido que no sea una presión
sig = E[E['type.name'] != 'Pressure'][['match_id', 'index', 'type.name', 'loc_x', 'loc_y', 'key']]
co = pd.merge_asof(co.sort_values('index'), sig.sort_values('index').rename(columns={'type.name': 'c_tipo', 'loc_x': 'c_x', 'loc_y': 'c_y', 'key': 'c_key'}),
                   on='index', by='match_id', direction='forward', allow_exact_matches=False)
con_f, tot_f, fil = set(), {}, []
for m in mues:
    if m not in carp['frames360']: continue
    f = pd.read_parquet(os.path.join(LIGA, 'frames360', f'{m}.parquet'), columns=[KF[0]])
    con_f |= {(m, k) for k in f[KF[0]]}; tot_f[m] = len(f)
    kc = set(co.loc[co.match_id == m, 'index' if KJ == 'event_index' else 'id'])
    if m in carp['jugadores360'] and kc:
        j = pd.read_parquet(os.path.join(LIGA, 'jugadores360', f'{m}.parquet'))
        j = j[j[KJ].isin(kc)]; j['match_id'] = m; fil.append(j)
co['frame'] = [(m, k) in con_f for m, k in zip(co.match_id, co.key)]
co['frame_contacto'] = [(m, k) in con_f for m, k in zip(co.match_id, co.c_key)]
E_cov = E.assign(fr=[(m, k) in con_f for m, k in zip(E.match_id, E.key)])
print('  Cobertura (% con foto): todos los eventos / córners / directos / cortos, por temporada:')
print(pd.DataFrame({'eventos': E_cov.groupby('temporada').fr.mean(), 'corners': co.groupby('temporada').frame.mean(),
                    'directos': co[co.directo].groupby('temporada').frame.mean(), 'cortos': co[~co.directo].groupby('temporada').frame.mean()}).mul(100).round(1).to_string())
ce = co.groupby('team.name')['frame'].agg(['size', 'mean']).query('size >= 20')
print(f'  Por equipo (≥ 20 córners en la muestra, {len(ce)} equipos): mínimo {100 * ce["mean"].min():.1f}%, mediana {100 * ce["mean"].median():.1f}%, máximo {100 * ce["mean"].max():.1f}%')
print('  Por minuto del saque (1T / 2T):', co.groupby('period').frame.mean().mul(100).round(1).to_dict())
print(f'  Primer contacto (siguiente evento): tipos {co.c_tipo.value_counts(normalize=True).mul(100).round(1).head(6).to_dict()}')
print(f'  Córners directos con foto del saque Y del primer contacto: {pct(co.loc[co.directo, "frame"] & co.loc[co.directo, "frame_contacto"])}%')
if fil:
    J = pd.concat(fil, ignore_index=True)
    for b in ['teammate', 'actor', 'keeper']: J[b] = J[b].fillna(False).astype(bool)
    J['area'] = en_area(J.x, J.y); J['chica'] = en_chica(J.x, J.y)
    att = J[J.teammate & ~J.actor]; de = J[~J.teammate & ~J.keeper]
    po = pd.DataFrame({'atac_area': att[att.area].groupby(['match_id', KJ]).size(), 'atac_visibles': att.groupby(['match_id', KJ]).size(),
                       'def_area': de[de.area].groupby(['match_id', KJ]).size(), 'def_chica': de[de.chica].groupby(['match_id', KJ]).size()}).fillna(0)
    print(f'  Fotos de córner con jugadores: {len(po):,}. Medianas: atacantes en el área {po.atac_area.median():.0f} · atacantes visibles {po.atac_visibles.median():.0f}'
          f' · defensores en el área {po.def_area.median():.0f} · en el área chica {po.def_chica.median():.0f}')
    print(f'  % de fotos con los 10 atacantes de campo visibles (vigilantes contables directamente): {pct(po.atac_visibles >= 10)}%')
    dist = []
    for k, g in J[J.area].groupby(['match_id', KJ]):
        a = g[g.teammate & ~g.actor][['x', 'y']].values; d = g[~g.teammate & ~g.keeper][['x', 'y']].values
        if len(a) and len(d): dist.append(np.sqrt(((d[:, None, :] - a[None, :, :]) ** 2).sum(-1)).min(1))
    if dist:
        dd_ = np.concatenate(dist)
        print(f'  Marcaje: distancia de cada defensor en el área a su atacante más cercano (yd) {np.quantile(dd_, [.1, .25, .5, .75, .9]).round(1)}'
              f' · % a ≤ 2 yd {pct(dd_ <= 2)} · ≤ 3 yd {pct(dd_ <= 3)}')

# ---------- (e) Estatura ----------
print('\n=== (e) Alineaciones ===')
lus = [m for m in mues if m in carp['lineups']]
L = pd.concat([pd.read_parquet(os.path.join(LIGA, 'lineups', f'{m}.parquet')).assign(match_id=m) for m in lus], ignore_index=True)
print('  columnas:', list(L.columns)[:30])
hc = next((c for c in L.columns if 'height' in c), None)
if hc:
    h = pd.to_numeric(L[hc], errors='coerce')
    print(f'  {hc}: con dato {pct(h.notna() & (h > 0))}% de {len(L):,} filas · cuantiles (cm) {h[h > 0].quantile([.05, .5, .95]).round(0).to_dict()}')
    if 'temporada' not in L: L['temporada'] = L.match_id.map(TEMP)
    print('  con dato por temporada (%):', L.assign(ok=(h > 0)).groupby('temporada').ok.mean().mul(100).round(1).to_dict())
else: print('  No hay columna de estatura')

# ---------- (f) Penales ----------
print('\n=== (f) Penales ===')
pc = next((c for c in E.columns if c in ['foul_won.penalty']), None)
ps = E[E['shot.type.name'] == 'Penalty']
print(f'  Tiros de penal: {len(ps)} ({len(ps) / len(mues):.2f} por partido) · xG medio {ps["shot.statsbomb_xg"].mean():.3f} · gol {pct(ps["shot.outcome.name"] == "Goal")}%')
if pc:
    fw = E[(E['type.name'] == 'Foul Won') & E[pc].eq(True)]
    print(f'  Faltas ganadas con penal: {len(fw)}')
    prev = []
    for _, r in fw.iterrows():
        q = E[(E.match_id == r.match_id) & (E.period == r.period) & (E['player.id'] == r['player.id']) & (E['index'] < r['index'])
              & (E.t >= r.t - 5) & (E['type.name'] != 'Foul Won')]
        q = q.iloc[-1] if len(q) else None
        prev.append(dict(tipo='ninguna en 5 s' if q is None else q['type.name'], regate=None if q is None else q['dribble.outcome.name'],
                         obv_prev=np.nan if (q is None or not OBV) else q[OBV], obv_falta=r[OBV] if OBV else np.nan,
                         en_area=bool(q is not None and en_area(q.loc_x, q.loc_y))))
    P = pd.DataFrame(prev)
    print('  Acción previa (≤ 5 s) del jugador que recibe el penal, %:', P.tipo.value_counts(normalize=True).mul(100).round(1).to_dict())
    print('  Resultado del regate previo:', P.regate.value_counts(dropna=True).to_dict(), '· acción previa dentro del área:', f'{pct(P.en_area)}%')
    if OBV:
        print(f'  OBV ({OBV}): falta ganada con penal → con dato {pct(P.obv_falta.notna())}%, media {P.obv_falta.mean():.3f}'
              f' · acción previa → media {P.obv_prev.mean():.3f} · tiro de penal → media {ps[OBV].mean():.3f}')
        dr = E[(E['type.name'] == 'Dribble') & en_area(E.loc_x, E.loc_y)]
        print(f'  Referencia: regates en el área {len(dr):,}, OBV medio {dr[OBV].mean():.4f} (completos {dr.loc[dr["dribble.outcome.name"] == "Complete", OBV].mean():.4f})')
else: print('  No encontré la bandera de penal en la falta ganada')

# ---------- (g) Duelos aéreos ----------
print('\n=== (g) Duelos aéreos ===')
aw = [c for c in E.columns if c.endswith('aerial_won')]
E['gana_aereo'] = E[aw].eq(True).any(axis=1) if aw else False
al = E[(E['type.name'] == 'Duel') & (E['duel.type.name'] == 'Aerial Lost')]
gana = E.set_index('id')['gana_aereo'].to_dict() if 'id' in E else {}
equipo = E.set_index('id')['team.name'].to_dict() if 'id' in E else {}
def rel(v):
    if isinstance(v, str):
        try: return json.loads(v)
        except Exception: return []
    return list(v) if isinstance(v, (list, np.ndarray)) else []
par = [any(gana.get(x, False) and equipo.get(x) != t for x in rel(v)) for v, t in zip(al['related_events'], al['team.name'])]
print(f'  Columnas aerial_won: {aw}')
print(f'  Aéreos perdidos {len(al):,} ({len(al) / len(mues):.1f} por partido) · ganados marcados {int(E.gana_aereo.sum()):,} · perdidos emparejados con un ganador rival por related_events: {pct(par)}%')
inv_a = pd.concat([al['player.id'], E.loc[E.gana_aereo, 'player.id']]).value_counts()
print(f'  Duelos aéreos por jugador en la muestra: mediana {inv_a.median():.0f}, p90 {inv_a.quantile(.9):.0f}, jugadores con ≥ 30: {int((inv_a >= 30).sum())}')

# ---------- (h) Freeze frame de tiros y reparto de goles ----------
print('\n=== (h) Tiros ===')
sh = E[E['type.name'] == 'Shot']
ff = sh['shot.freeze_frame']
print(f'  Tiros {len(sh):,} · con freeze frame {pct(ff.notna())}%')
ej = next((v for v in ff if isinstance(v, str) and v.startswith('[')), None)
if ej: print('  Campos de un jugador del freeze frame:', list(json.loads(ej)[0].keys()))
sh = sh.assign(tipo=np.where(sh['shot.type.name'] == 'Penalty', 'penal', np.where(sh[SP].notna() if SP else False, 'balón parado (marca)', 'resto (juego abierto + transición)')))
rep = sh.groupby('tipo').agg(tiros=('tipo', 'size'), xg=('shot.statsbomb_xg', 'sum'), goles=('shot.outcome.name', lambda s: (s == 'Goal').sum()))
rep['% xG'] = 100 * rep.xg / rep.xg.sum(); rep['% goles'] = 100 * rep.goles / rep.goles.sum()
print(rep.round(1).to_string()); print(f'  Autogoles a favor: {int((E["type.name"] == "Own Goal For").sum())} (la transición se separa con R1 en la celda base)')
print(f'\nR8-00 terminó en {time.time() - t0:.0f} s. No escribió nada. Pega toda la salida.')


R8a-0

In [ ]:
#[R8a-0 v1.0] Funciones de R8 y prueba con datos inventados (un escenario sin efecto y otro con efecto)
import numpy as np
import pandas as pd

AREA_X, AREA_Y, CHICA_X, CHICA_Y = 102, (18, 62), 114, (30, 50)      # G-4
FRONTAL_X = 90                                                       # S11: frontal x 90-102
T_ENVIO_CORTO, T_CONTACTO, T_VENTANA, T_PRIMERA = 10.0, 5.0, 10.0, 1.0
NO_CONTACTO = {'Pressure'}
NO_CAIDA = {'Pressure', 'Duel', 'Block'}
CONTROL = {'Ball Receipt*', 'Carry', 'Pass', 'Ball Recovery', 'Dribble'}
ZONAS_ENVIO = ['primer_palo', 'punto_penal', 'area_chica', 'segundo_palo']

def en_area(x, y): return (x >= AREA_X) & (y >= AREA_Y[0]) & (y <= AREA_Y[1])
def en_chica(x, y): return (x >= CHICA_X) & (y >= CHICA_Y[0]) & (y <= CHICA_Y[1])
def zona_envio(x, yr):
    if not en_area(x, yr): return 'fuera'
    if en_chica(x, yr): return 'area_chica'
    if yr < 36: return 'primer_palo'
    if yr <= 44: return 'punto_penal'
    return 'segundo_palo'
def zona_caida(x, y):
    if en_area(x, y): return 'area'
    if x >= FRONTAL_X and AREA_Y[0] <= y <= AREA_Y[1]: return 'frontal'
    if x >= FRONTAL_X: return 'banda'
    return 'lejos'

def foto(P, lado_bajo, girar=False, sin_actor=True):
    """Conteos de una foto 360 en el marco del equipo que ejecuta el córner. girar=True si el actor de la foto es del rival."""
    tm = P['teammate'].fillna(False).astype(bool).values; ac = P['actor'].fillna(False).astype(bool).values
    kp = P['keeper'].fillna(False).astype(bool).values
    px, py = P['x'].astype(float).values, P['y'].astype(float).values
    if girar: px, py, tm = 120 - px, 80 - py, ~tm
    yr = py if lado_bajo else 80 - py
    a = tm & ~kp & (~ac if sin_actor else True); d = ~tm & ~kp
    ar, ch = en_area(px, yr), en_chica(px, yr)
    fr = (px >= FRONTAL_X) & (px < AREA_X) & (yr >= AREA_Y[0]) & (yr <= AREA_Y[1])
    out = {'atac_area': int((a & ar).sum()), 'atac_chica': int((a & ch).sum()), 'atac_frontal': int((a & fr).sum()),
           'atac_vis': int(a.sum()), 'def_area': int((d & ar).sum()), 'def_chica': int((d & ch).sum())}
    za = np.array([zona_envio(u, v) for u, v in zip(px, yr)]) if len(px) else np.array([])
    for z in ZONAS_ENVIO: out['atac_' + z] = int((a & (za == z)).sum()) if len(px) else 0
    A_, D_ = np.c_[px[a & ar], yr[a & ar]], np.c_[px[d & ar], yr[d & ar]]
    if len(A_) and len(D_):
        dmin = np.sqrt(((D_[:, None, :] - A_[None, :, :]) ** 2).sum(-1)).min(1)
        out['marc2'], out['marc_den'] = int((dmin <= 2).sum()), int(len(D_))
    else:
        out['marc2'], out['marc_den'] = 0, int(len(D_))
    hx = np.clip(np.digitize(px[d & ar], [108, 114]), 0, 2); hy = np.clip(np.digitize(yr[d & ar], [30, 50]), 0, 2)
    for i in range(9): out[f'h{i}'] = int(((hx * 3 + hy) == i).sum())
    return out

def bp_de_partido(M, J=None):
    """Una fila por balón parado (córner, tiro libre en el último tercio, saque de banda en el último tercio)."""
    M = M.sort_values('index').reset_index(drop=True); n = len(M)
    col = lambda c: M[c].values if c in M else np.full(n, np.nan, dtype=object)
    idx, per, t = col('index'), col('period'), col('t').astype(float)
    typ, team, poss = col('type.name'), col('team.name'), col('possession')
    x, y, ex, ey = (col(c).astype(float) for c in ['loc_x', 'loc_y', 'pass_end_x', 'pass_end_y'])
    ptype, tech, sht, sho = col('pass.type.name'), col('pass.technique.name'), col('shot.type.name'), col('shot.outcome.name')
    xg, rec, fcp = col('shot.statsbomb_xg').astype(float), col('ball_receipt.outcome.name'), col('foul_committed.penalty')
    mh, mhb, cu, evl = col('m_home'), col('m_home_b'), col('cuarto'), col('ev_local')
    filas = []
    for c in range(n):
        kind = None
        if typ[c] == 'Pass' and ptype[c] == 'Corner' or typ[c] == 'Shot' and sht[c] == 'Corner': kind = 'corner'
        elif typ[c] == 'Pass' and ptype[c] == 'Free Kick' and x[c] >= 80 or typ[c] == 'Shot' and sht[c] == 'Free Kick': kind = 'tiro_libre'
        elif typ[c] == 'Pass' and ptype[c] == 'Throw-in' and x[c] >= 80: kind = 'saque_banda'
        if kind is None: continue
        A, bajo, es_tiro = team[c], y[c] < 40, typ[c] == 'Shot'
        yr = (lambda v: v) if bajo else (lambda v: 80 - v)
        aA = lambda k: (x[k], y[k]) if team[k] == A else (120 - x[k], 80 - y[k])        # al marco del ejecutor
        directo = bool(es_tiro or en_area(ex[c], ey[c]))
        env, ruta = None, 'directo'
        if es_tiro: env, ruta = c, 'tiro_directo'
        elif directo: env = c
        else:
            ruta = 'reciclado'
            for k in range(c + 1, n):
                if per[k] != per[c] or t[k] > t[c] + T_ENVIO_CORTO or poss[k] != poss[c]: break
                if team[k] == A and (typ[k] == 'Pass' and en_area(ex[k], ey[k]) or typ[k] == 'Shot'):
                    env, ruta = k, 'via_corto'; break
        zona = (zona_envio(ex[env], yr(ey[env])) if typ[env] == 'Pass' else 'tiro') if env is not None else None
        con = None
        if env is not None and typ[env] == 'Shot': con = env
        elif env is not None:
            for k in range(env + 1, n):
                if per[k] != per[env] or t[k] > t[env] + T_CONTACTO: break
                if typ[k] in NO_CONTACTO or (typ[k] == 'Ball Receipt*' and isinstance(rec[k], str)): continue
                con = k; break
        cai = None
        if con is not None:
            for k in range(con + 1, n):
                if per[k] != per[con] or t[k] > t[con] + T_VENTANA: break
                if typ[k] in NO_CAIDA or (typ[k] == 'Ball Receipt*' and isinstance(rec[k], str)): continue
                cai = k; break
        t_ref = t[con] if con is not None else (t[env] if env is not None else t[c])
        fin = t_ref + T_VENTANA
        xg1 = xg2 = 0.0; tiros = goles = autog = 0; penal = False
        for k in range(c, n):
            if per[k] != per[c] or t[k] > fin or poss[k] != poss[c]: break
            if team[k] == A and typ[k] == 'Shot' and sht[k] != 'Penalty':
                v = 0.0 if np.isnan(xg[k]) else xg[k]; tiros += 1; goles += int(sho[k] == 'Goal')
                if con is None or t[k] <= t[con] + T_PRIMERA: xg1 += v
                else: xg2 += v
            if team[k] == A and typ[k] == 'Own Goal For': autog += 1
            if team[k] != A and typ[k] == 'Foul Committed' and fcp[k] == True: penal = True
        gana_con = None if con is None else bool((team[con] != A) if typ[con] == 'Duel' else (team[con] == A))
        clase = None if con is None else ('remate_atac' if team[con] == A and typ[con] == 'Shot' else
                                          'control_atac' if team[con] == A and typ[con] in CONTROL else
                                          'gana_atac' if gana_con else 'gana_def')
        fila = dict(index=idx[c], kind=kind, team=A, period=per[c], t=t[c], m_home=mh[c], m_home_b=mhb[c], cuarto=cu[c], ev_local=evl[c],
                    lado_bajo=bool(bajo), directo=directo, ruta=ruta, tecnica=tech[c] if isinstance(tech[c], str) else None, zona=zona,
                    t_envio=(t[env] - t[c]) if env is not None else np.nan,
                    contacto_tipo=typ[con] if con is not None else None, contacto_clase=clase,
                    contacto_x=aA(con)[0] if con is not None else np.nan, contacto_y=aA(con)[1] if con is not None else np.nan,
                    caida_zona=zona_caida(*aA(cai)) if cai is not None else ('sin_juego' if con is not None else None),
                    caida_gana=bool(team[cai] == A) if cai is not None else None,
                    xg1=xg1, xg2=xg2, tiros=tiros, goles=goles, autogoles=autog, penal=penal)
        if J is not None and kind == 'corner' and directo and not es_tiro:
            fila['s_frame'] = idx[c] in J
            if fila['s_frame']: fila.update({'s_' + k: v for k, v in foto(J[idx[c]], bajo).items()})
            fila['c_frame'] = con is not None and idx[con] in J
            if fila['c_frame']: fila.update({'c_' + k: v for k, v in foto(J[idx[con]], bajo, girar=team[con] != A, sin_actor=False).items()})
        filas.append(fila)
    return filas

def penales_de_partido(M):
    """Una fila por falta cometida con penal (periodos 1-4): beneficiado y acción previa (5 s) de su equipo."""
    M = M.sort_values('index').reset_index(drop=True)
    eqs = [e for e in M['team.name'].dropna().unique()]
    filas = []
    for k in np.where((M['type.name'] == 'Foul Committed').values & (M['foul_committed.penalty'] == True).values & M['period'].between(1, 4).values)[0]:
        B = next((e for e in eqs if e != M.at[k, 'team.name']), None)
        prev = M.iloc[:k][(M['team.name'].iloc[:k] == B) & (M['period'].iloc[:k] == M.at[k, 'period']) & (M['t'].iloc[:k] >= M.at[k, 't'] - 5)
                           & ~M['type.name'].iloc[:k].isin(['Pressure', 'Foul Won'])]
        p = prev.iloc[-1] if len(prev) else None
        filas.append(dict(index=M.at[k, 'index'], team=B, period=M.at[k, 'period'], fase=M.at[k, 'fase'] if 'fase' in M else None,
                          jugador=p['player.id'] if p is not None else np.nan, accion=p['type.name'] if p is not None else 'ninguna',
                          encare=bool(p is not None and p['type.name'] in ('Carry', 'Dribble')),
                          en_area=bool(p is not None and en_area(p['loc_x'], p['loc_y']))))
    return filas

# ---------------- Prueba con datos inventados ----------------
CP = ['index', 'period', 'minute', 't', 'type.name', 'team.name', 'possession', 'loc_x', 'loc_y', 'pass_end_x', 'pass_end_y',
      'pass.type.name', 'pass.technique.name', 'ball_receipt.outcome.name', 'shot.type.name', 'shot.outcome.name', 'shot.statsbomb_xg',
      'foul_committed.penalty', 'player.id', 'fase', 'm_home', 'm_home_b', 'cuarto', 'ev_local']
def ev(rows, t, tipo, team, poss, x=np.nan, y=np.nan, **kw):
    r = {c: np.nan for c in CP}; r.update({'index': len(rows) + 1, 'period': 1, 'minute': int(t // 60), 't': float(t), 'type.name': tipo,
                                           'team.name': team, 'possession': poss, 'loc_x': x, 'loc_y': y, 'm_home': 0, 'm_home_b': 0,
                                           'cuarto': 1, 'ev_local': team == 'A'})
    r.update(kw); rows.append(r); return r['index']
R = []
# T1 · corto + pase al área a los 6 s (primer palo) · despeje del rival (marco propio) · rebote ganado en la frontal · tiro a los +2.8 s
ev(R, 100, 'Pass', 'A', 1, 120, 0.5, pass_end_x=114, pass_end_y=6, **{'pass.type.name': 'Corner'})
ev(R, 101, 'Ball Receipt*', 'A', 1, 114, 6)
ev(R, 106, 'Pass', 'A', 1, 112, 8, pass_end_x=110, pass_end_y=33)
ev(R, 106.8, 'Ball Receipt*', 'A', 1, 110, 33, **{'ball_receipt.outcome.name': 'Incomplete'})
ev(R, 107.2, 'Clearance', 'B', 1, 12, 47)
ev(R, 108, 'Ball Recovery', 'A', 1, 95, 40)
ev(R, 110, 'Shot', 'A', 1, 100, 40, **{'shot.type.name': 'Open Play', 'shot.statsbomb_xg': 0.1, 'shot.outcome.name': 'Off T'})
# T2 · corto reciclado: el pase al área llega a los 12 s (fuera de la regla de 10 s)
ev(R, 200, 'Pass', 'A', 2, 120, 0.5, pass_end_x=115, pass_end_y=5, **{'pass.type.name': 'Corner'})
ev(R, 201, 'Ball Receipt*', 'A', 2, 115, 5)
ev(R, 203, 'Pass', 'A', 2, 115, 5, pass_end_x=85, pass_end_y=20)
ev(R, 212, 'Pass', 'A', 2, 85, 20, pass_end_x=108, pass_end_y=40)
# T3 · directo desde la banda alta al área chica · remate (contacto) · el portero rival la atrapa (nueva posesión) · tiro posterior no cuenta
i3 = ev(R, 300, 'Pass', 'A', 3, 120, 79.5, pass_end_x=116, pass_end_y=42, **{'pass.type.name': 'Corner', 'pass.technique.name': 'Inswinging'})
ev(R, 301, 'Shot', 'A', 3, 116, 42, **{'shot.type.name': 'Open Play', 'shot.statsbomb_xg': 0.3, 'shot.outcome.name': 'Saved'})
ev(R, 301.5, 'Goal Keeper', 'B', 4, 4, 38)
ev(R, 305, 'Shot', 'A', 5, 100, 40, **{'shot.type.name': 'Open Play', 'shot.statsbomb_xg': 0.2, 'shot.outcome.name': 'Off T'})
# T4 · directo al punto de penal · falta con penal del rival a los 1.5 s
ev(R, 400, 'Pass', 'A', 6, 120, 0.5, pass_end_x=110, pass_end_y=40, **{'pass.type.name': 'Corner'})
ev(R, 401, 'Carry', 'A', 6, 108, 40)
ev(R, 401.5, 'Foul Committed', 'B', 6, 12, 40, **{'foul_committed.penalty': True})
MT = pd.DataFrame(R)
J3 = {i3: pd.DataFrame({'teammate': [True, True, True, False, False, False, False], 'actor': [True, False, False, False, False, False, False],
                        'keeper': [False] * 6 + [True], 'x': [120, 110, 116, 110.5, 116.5, 117, 119.5], 'y': [79.5, 43, 40, 43.5, 40.5, 38, 40]})}
F = pd.DataFrame(bp_de_partido(MT, J3)).set_index('t')
esperado = {
    100: dict(ruta='via_corto', zona='primer_palo', contacto_tipo='Clearance', contacto_clase='gana_def', caida_zona='frontal', caida_gana=True, xg1=0.0, xg2=0.1),
    200: dict(ruta='reciclado', zona=None),
    300: dict(ruta='directo', zona='area_chica', contacto_clase='remate_atac', caida_zona='area', caida_gana=False, xg1=0.3, xg2=0.0, tiros=1,
              s_atac_area=2, s_def_area=3, s_def_chica=2, s_marc2=2, s_marc_den=3),
    400: dict(ruta='directo', zona='punto_penal', penal=True)}
fallas = []
for tt, d in esperado.items():
    for k, v in d.items():
        r = F.at[tt, k]
        ok = (r is None or (isinstance(r, float) and np.isnan(r))) if v is None else (abs(r - v) < 1e-9 if isinstance(v, float) else r == v)
        if not ok: fallas.append(f'córner t={tt}: {k} = {r} (esperado {v})')
print('--- (T1-T4) Casos deterministas:', 'TODOS BIEN' if not fallas else fallas)
print(F[['ruta', 'zona', 'contacto_tipo', 'contacto_clase', 'contacto_x', 'contacto_y', 'caida_zona', 'caida_gana', 'xg1', 'xg2', 'penal']].to_string())
PEN = pd.DataFrame(penales_de_partido(MT.assign(**{'team.name': MT['team.name']})))
print('--- Penal de prueba: beneficiado', PEN.team.tolist(), '· acción previa', PEN.accion.tolist(), '· encare', PEN.encare.tolist(), '(esperado A · Carry · True)')

# Escenarios estadísticos: 400 córners directos por equipo, lado del saque al azar (prueba también el espejo de y)
rng0 = np.random.default_rng(2026)
def simula(p_pp, eq, t0):
    rows = []
    for i in range(400):
        bajo = rng0.random() < 0.5; pp = rng0.random() < p_pp
        xx = rng0.uniform(103, 113); yrr = rng0.uniform(19, 35) if pp else rng0.uniform(37, 43)
        rows.append({c: np.nan for c in CP} | {'index': i + 1, 'period': 1, 'minute': 0, 't': t0 + 60.0 * i, 'type.name': 'Pass', 'team.name': eq,
                                              'possession': i + 1, 'loc_x': 120, 'loc_y': 0.5 if bajo else 79.5, 'pass_end_x': xx,
                                              'pass_end_y': yrr if bajo else 80 - yrr, 'pass.type.name': 'Corner'})
    return pd.DataFrame(bp_de_partido(pd.DataFrame(rows)))
for nombre, (pa, pb) in {'SIN efecto (0.40 vs 0.40)': (0.4, 0.4), 'CON efecto (0.70 vs 0.20)': (0.7, 0.2)}.items():
    a, b = simula(pa, 'X', 0), simula(pb, 'Y', 0)
    da, db = 100 * (a.zona == 'primer_palo').mean(), 100 * (b.zona == 'primer_palo').mean()
    print(f'--- {nombre}: % primer palo X {da:.1f} · Y {db:.1f} · diferencia {da - db:+.1f} puntos (esperado ≈ {100 * (pa - pb):+.0f})')
print('Si los casos salen TODOS BIEN y las diferencias se parecen a lo esperado, corre R8a-1.')



R8a-1

In [ ]:
#[R8a-1 v1.0] Corrida real: cinco temporadas (escribe fw5/r8_bp, r8_penales, r8_reparto y cachés r8_extra_*)
import os, json, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
assert 'bp_de_partido' in globals(), 'Corre primero R8a-0'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1 = os.path.join(DP, 'fw1'); FW5 = os.path.join(DP, 'fw5'); LIGA = os.path.join(DP, 'liga')
REHACER = False
CORTE_C1, CORTE_C3 = 25, 70                                          # cuartos de R4 (A2)
t_ini = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
TEMPS = sorted(Mm['temporada'].unique())
COLS_R1 = ['match_id', 'index', 'period', 'minute', 't', 'type.name', 'team.name', 'possession', 'possession_team.name', 'player.id',
           'loc_x', 'loc_y', 'pass_end_x', 'pass_end_y', 'pass.type.name', 'pass.length', 'ball_receipt.outcome.name', 'shot.type.name',
           'shot.outcome.name', 'shot.statsbomb_xg', 'set_piece_phase', 'fase', 'en_transicion', 'pass.technique.name', 'foul_committed.penalty']
OPC_CRUDOS = ['pass.technique.name', 'foul_committed.penalty', 'pass.length', 'ball_receipt.outcome.name', 'second']
RUTAS = {k: os.path.join(FW5, f'r8_{k}.parquet') for k in ['bp', 'penales', 'reparto']}

# HOPS: ¿viene en los eventos? (nombres reales en 20 partidos crudos)
nom = set()
for m in Mm['match_id'].iloc[::85][:20]: nom |= set(pq.read_schema(os.path.join(LIGA, 'eventos', f'{m}.parquet')).names)
print('--- (0) Columnas tipo HOPS o probabilidad de duelo en los crudos:', sorted(c for c in nom if any(s in c.lower() for s in ['hops', 'win_prob', 'rating'])) or 'ninguna')

def extra_crudos(temp, falt):
    ruta = os.path.join(FW5, f"r8_extra_{temp.replace('/', '-')}.parquet")
    if os.path.exists(ruta) and not REHACER:
        X = pd.read_parquet(ruta)
        if all(c in X for c in falt): return X
    piezas = []
    for mid in Mm.loc[Mm.temporada == temp, 'match_id']:
        r = os.path.join(LIGA, 'eventos', f'{mid}.parquet')
        if not os.path.exists(r): continue
        disp = set(pq.read_schema(r).names)
        d = pd.read_parquet(r, columns=['index'] + [c for c in falt if c in disp])
        for c in falt:
            if c not in d: d[c] = np.nan
        d['match_id'] = mid; piezas.append(d)
    X = pd.concat(piezas, ignore_index=True); X.to_parquet(ruta, index=False); return X

if all(os.path.exists(r) for r in RUTAS.values()) and not REHACER:
    BP, PE, RE = (pd.read_parquet(RUTAS[k]) for k in ['bp', 'penales', 'reparto'])
    print('Cargado de caché:', len(BP), 'balones parados')
else:
    bps, pes, res, chk = [], [], [], []
    for temp in TEMPS:
        t0 = time.time()
        ruta_e = os.path.join(FW1, f"eventos_{temp.replace('/', '-')}.parquet"); cols_r1 = set(pq.read_schema(ruta_e).names)
        E = pd.read_parquet(ruta_e, columns=[c for c in COLS_R1 if c in cols_r1])
        E = E[E['match_id'].isin(loc.index)]
        falt = [c for c in OPC_CRUDOS if c not in E.columns and (c != 'second' or 't' not in E.columns)]
        if falt:
            E = E.merge(extra_crudos(temp, falt)[['match_id', 'index'] + falt], on=['match_id', 'index'], how='left')
        if 't' not in E.columns: E['t'] = E['minute'] * 60 + E['second']
        for c in COLS_R1:
            if c not in E.columns: E[c] = np.nan
        E = E.sort_values(['match_id', 'index']).reset_index(drop=True)
        ok_t = ((E['t'] - 60 * E['minute']).between(-0.01, 60.01)).mean()
        # Marcador: copia literal de R6a (goles de los periodos 1-4: tiro convertido o autogol a favor; offset de liguilla)
        E['local'] = E['match_id'].map(loc['local']); E['ev_local'] = E['team.name'] == E['local']
        es_gol = E['period'].between(1, 4) & (((E['type.name'] == 'Shot') & (E['shot.outcome.name'] == 'Goal')) | (E['type.name'] == 'Own Goal For'))
        gh = (es_gol & E['ev_local']).astype(int); ga = (es_gol & ~E['ev_local'] & E['team.name'].notna()).astype(int)
        E['dif_home'] = (gh.groupby(E['match_id']).cumsum() - gh) - (ga.groupby(E['match_id']).cumsum() - ga)
        E['m_home'] = E['dif_home'] + E['match_id'].map(loc['offset']); E['m_home_b'] = E['dif_home'] + E['match_id'].map(loc['offset_b'])
        E['cuarto'] = np.where(E['period'] == 1, np.where(E['minute'] < CORTE_C1, 1, 2), np.where(E['minute'] < CORTE_C3, 3, 4))
        gg = pd.DataFrame({'gh': gh.groupby(E['match_id']).sum(), 'ga': ga.groupby(E['match_id']).sum()})
        cuadra = ((gg.gh == loc.loc[gg.index, 'gl']) & (gg.ga == loc.loc[gg.index, 'gv'])).mean()
        E = E[E['period'].between(1, 4)]
        # Reparto de tiros, xG y goles por tipo de jugada (periodos 1-4; sin tandas)
        sh = E[E['type.name'] == 'Shot'].copy()
        sh['tipo'] = np.select([sh['shot.type.name'] == 'Penalty', sh['fase'] == 'bp', sh['fase'] == 'transicion'], ['penal', 'balon_parado', 'transicion'], 'establecido')
        r = sh.groupby(['match_id', 'team.name', 'tipo']).agg(tiros=('tipo', 'size'), xg=('shot.statsbomb_xg', 'sum'),
                                                              goles=('shot.outcome.name', lambda s: int((s == 'Goal').sum()))).reset_index()
        og = E[E['type.name'] == 'Own Goal For'].groupby(['match_id', 'team.name']).size().rename('goles').reset_index().assign(tipo='autogol', tiros=0, xg=0.0)
        res.append(pd.concat([r, og], ignore_index=True).rename(columns={'team.name': 'team'}))
        n360 = 0
        for mid, M in E.groupby('match_id', sort=False):
            J = None; rj = os.path.join(LIGA, 'jugadores360', f'{mid}.parquet')
            ic = M.loc[(M['pass.type.name'] == 'Corner'), 'index'].values
            if os.path.exists(rj) and len(ic):
                nec = set(np.concatenate([ic + k for k in range(16)]))
                jj = pd.read_parquet(rj, columns=['event_index', 'teammate', 'actor', 'keeper', 'x', 'y'])
                jj = jj[jj.event_index.isin(nec)]; J = {k: g for k, g in jj.groupby('event_index')}; n360 += 1
            for f in bp_de_partido(M, J): f['match_id'] = mid; bps.append(f)
            for f in penales_de_partido(M): f['match_id'] = mid; pes.append(f)
        chk.append(dict(temporada=temp, partidos=E.match_id.nunique(), con_360=n360, t_coherente_pct=round(100 * ok_t, 2),
                        marcador_cuadra_pct=round(100 * cuadra, 2), segundos=round(time.time() - t0)))
        print(f'  {temp}: {chk[-1]}')
    BP, PE, RE = pd.DataFrame(bps), pd.DataFrame(pes), pd.concat(res, ignore_index=True)
    for d_, k in [(BP, 'bp'), (PE, 'penales'), (RE, 'reparto')]: d_.to_parquet(RUTAS[k], index=False)
    print('Chequeos por temporada (t_coherente y marcador_cuadra deben ser ~100):'); print(pd.DataFrame(chk).to_string(index=False))

BP['temporada'] = BP.match_id.map(loc['temporada']); RE['temporada'] = RE.match_id.map(loc['temporada'])
pc = lambda s: round(100 * float(np.mean(s)), 1) if len(s) else np.nan
co = BP[BP.kind == 'corner']
print(f'\n--- (1) Balones parados: {BP.kind.value_counts().to_dict()} · córners por partido {len(co) / BP.match_id.nunique():.1f}')
print('  Ruta del córner (%):', co.ruta.value_counts(normalize=True).mul(100).round(1).to_dict())
cor = co[co.ruta.isin(['via_corto', 'reciclado'])]
print(f'  De los cortos: terminan en envío al área ≤ 10 s {pc(cor.ruta == "via_corto")}% · segundos hasta el envío (mediana) {co.loc[co.ruta == "via_corto", "t_envio"].median():.1f}')
print('  Zona del envío, directos frente a vía corto (% por fila):')
print(pd.crosstab(co.loc[co.ruta.isin(['directo', 'via_corto']), 'ruta'], co.loc[co.ruta.isin(['directo', 'via_corto']), 'zona'], normalize='index').mul(100).round(1).to_string())
print('  Técnica × zona de los directos (% por fila):')
dd = co[co.ruta == 'directo']; print(pd.crosstab(dd.tecnica.fillna('sin dato'), dd.zona, normalize='index').mul(100).round(1).to_string())

print('\n--- (2) Contacto, segunda jugada y xG (córners con envío) ---')
ce = co[co.ruta.isin(['directo', 'via_corto'])]
print('  Clase del primer contacto (%):', ce.contacto_clase.value_counts(normalize=True, dropna=False).mul(100).round(1).to_dict())
print('  Dónde cae la segunda jugada (%):', ce.caida_zona.value_counts(normalize=True, dropna=False).mul(100).round(1).to_dict())
cc = ce[ce.caida_gana.notna() & (ce.contacto_clase != 'control_atac')]
print(f'  Segunda jugada ganada por el atacante (sin contactos controlados por él): {pc(cc.caida_gana.astype(bool))}% de {len(cc):,}')
print('  % ganada por zona de caída:', cc.groupby('caida_zona').caida_gana.apply(lambda s: round(100 * s.astype(bool).mean(), 1)).to_dict())
print(f'  xG por córner (todos los córners): primera jugada {co.xg1.mean():.4f} · segunda {co.xg2.mean():.4f} · total {(co.xg1 + co.xg2).mean():.4f}'
      f' · % del xG en la segunda {100 * co.xg2.sum() / (co.xg1 + co.xg2).sum():.1f}')
print(f'  Goles por 100 córners {100 * co.goles.mean():.2f} · autogoles {100 * co.autogoles.mean():.2f} · penal provocado {pc(co.penal)}% · con tiro {pc(co.tiros > 0)}%')
print('  xG por córner según ruta:', co.groupby('ruta').apply(lambda d: round((d.xg1 + d.xg2).mean(), 4), include_groups=False).to_dict())
print('  xG por córner según zona del envío:', ce.groupby('zona').apply(lambda d: round((d.xg1 + d.xg2).mean(), 4), include_groups=False).to_dict())

print('\n--- (3) 360 (córners directos) ---')
dd = co[co.ruta == 'directo']
if 's_frame' in dd:
    s = dd[dd.s_frame.fillna(False).astype(bool)]
    print(f'  Con foto del saque {pc(dd.s_frame.fillna(False).astype(bool))}% · con foto del contacto {pc(dd.c_frame.fillna(False).astype(bool))}%')
    print('  Medianas en el saque:', s[['s_atac_area', 's_atac_frontal', 's_def_area', 's_def_chica']].median().to_dict())
    m_ = s[s.s_marc_den > 0]; print(f'  Marcaje (defensores del área a ≤ 2 yd de un atacante): {100 * m_.s_marc2.sum() / m_.s_marc_den.sum():.1f}%')
    sc = s[s.c_frame.fillna(False).astype(bool)]
    print('  Atacantes por zona, saque → contacto (media):', {z: f"{sc['s_atac_' + z].mean():.2f} → {sc['c_atac_' + z].mean():.2f}" for z in ZONAS_ENVIO})

print('\n--- (4) Tiros libres y saques de banda del último tercio ---')
for k in ['tiro_libre', 'saque_banda']:
    b = BP[BP.kind == k]
    print(f'  {k}: {len(b):,} · ruta {b.ruta.value_counts(normalize=True).mul(100).round(1).to_dict()} · xG por jugada {(b.xg1 + b.xg2).mean():.4f}')
sb = BP[BP.kind == 'saque_banda'].assign(largo=lambda d: d.ruta == 'directo').groupby('team').largo.mean().mul(100)
print(f'  Saque de banda largo por equipo (% al área): p10 {sb.quantile(.1):.1f} · mediana {sb.median():.1f} · p90 {sb.quantile(.9):.1f}')

print('\n--- (5) Penales (falta cometida con penal, periodos 1-4) ---')
PE['temporada'] = PE.match_id.map(loc['temporada'])
print(f'  {len(PE)} penales ({len(PE) / loc.shape[0]:.2f} por partido) · por temporada {PE.temporada.value_counts().sort_index().to_dict()}')
print('  Acción previa del beneficiado (%):', PE.accion.value_counts(normalize=True).mul(100).round(1).head(6).to_dict())
print(f'  Encare (conducción o regate) {pc(PE.encare)}% · acción previa en el área {pc(PE.en_area)}% · penales en balón parado (fase bp) {pc(PE.fase == "bp")}%')
tp = RE[RE.tipo == 'penal'].tiros.sum(); print(f'  Coherencia: tiros de penal en periodos 1-4 {tp} frente a faltas con penal {len(PE)} (deben ser casi iguales)')

print('\n--- (6) Reparto de la liga (periodos 1-4) ---')
rp = RE.groupby('tipo')[['tiros', 'xg', 'goles']].sum(); rp['% xG'] = 100 * rp.xg / rp.xg.sum(); rp['% goles'] = 100 * rp.goles / rp.goles.sum()
print(rp.round(1).to_string())
sp = RE[RE.tipo != 'penal'].groupby('tipo').xg.sum(); print(f'  Balón parado sin penales: {100 * sp.get("balon_parado", 0) / sp.drop("autogol", errors="ignore").sum():.1f}% del xG')
print('  % del xG de balón parado por temporada:', RE.groupby('temporada').apply(lambda d: round(100 * d.loc[d.tipo == 'balon_parado', 'xg'].sum() / d.xg.sum(), 1), include_groups=False).to_dict())
xbp = RE.loc[RE.tipo == 'balon_parado', 'xg'].sum()
print(f'  Coherencia: xG de nuestras ventanas (córners + tiros libres + saques de banda) / xG de balón parado de R1 = {(BP.xg1 + BP.xg2).sum() / xbp:.2f}'
      f' · por tipo { {k: round((g.xg1 + g.xg2).sum() / xbp, 2) for k, g in BP.groupby("kind")} }')
print(f'\nR8a-1 terminó en {time.time() - t_ini:.0f} s. Guardado: fw5/r8_bp, r8_penales, r8_reparto (+ r8_extra_* si hizo falta).')


R8p-0 Playbooks

In [ ]:
# R8p v1.1 · Playbooks de córner: ofensivos (acomodo en el saque + movimiento al contacto) y defensivos (acomodo en el saque)
# v1.1 (2026-10-17, antes de correr en Colab): k se elige entre los casi igual de reproducibles por silueta (la prueba con libretos
#   limpios dio ARI = 1 para todo k y la v1.0 elegía el más chico); el placebo se lee por tamaño del exceso, no solo por p.
# =====================================================================
# Sesión 11 · 2026-10-17 · Hackathon ISAC 2026, Track 2 · Cinco temporadas (fw5)
# Decisiones (Arturo, sesión 11):
#   - Grilla fina de 20 celdas ANIDADA en las 5 zonas del destino (BP-2): profundidad x 102-108 | 108-114 | 114-117 | 117-120
#     × ancho (y relativa al lado del saque) 18-30 | 30-36 | 36-44 | 44-50 | 50-62. Cada celda pertenece a una sola zona:
#     área chica (x ≥ 114, y 30-50) · primer palo (y < 36) · punto de penal (y 36-44) · segundo palo (y > 44).
#     "Pegados al portero" = franja 117-120 dentro del área chica; además, atacantes a ≤ 2 yd del portero rival.
#   - Playbook ofensivo = k-medias sobre [ocupación de atacantes en el saque (20) + ocupación en el contacto (20)].
#     Playbook defensivo = k-medias sobre la ocupación de defensores en el saque (20). El envío NO entra al agrupamiento:
#     se describe después (qué playbook termina en qué zona).
#   - k se elige por reproducibilidad (dos mitades de partidos al azar; acuerdo ARI), entre 4 y 8 (ofensa) y 3 y 6 (defensa).
#   - Placebo: concentración de cada equipo en sus playbooks frente a etiquetas de equipo revueltas dentro de la temporada.
#   - Sin regla no hay nombre: los nombres se ponen con Arturo a partir de los centroides.
# Orden: celda 1 y R0 → R8a-0 (funciones) → R8p-0 (prueba con datos inventados) → R8p-1 (corrida real; escribe fw5 y salidas/fw5).
# Insumos: fw5/r8_bp (R8a), R1 (fw1, solo lectura) para el primer contacto, jugadores360 por partido.
# =====================================================================

# [R8p-0 v1.1] Funciones y prueba con datos inventados
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

SEMILLA = 2026
XB, YB = [102, 108, 114, 117, 120.0001], [18, 30, 36, 44, 50, 62.0001]
ZONA_CELDA = []
for xi in range(4):
    for yi in range(5):
        ZONA_CELDA.append('area_chica' if xi >= 2 and 1 <= yi <= 3 else 'primer_palo' if yi <= 1 else 'punto_penal' if yi == 2 else 'segundo_palo')
ZONA_CELDA = np.array(ZONA_CELDA)
PEGADO = np.array([xi == 3 and 1 <= yi <= 3 for xi in range(4) for yi in range(5)])

def celda(px, yr):
    xi = np.digitize(px, XB) - 1; yi = np.digitize(yr, YB) - 1
    ok = (xi >= 0) & (xi <= 3) & (yi >= 0) & (yi <= 4)
    return np.where(ok, xi * 5 + yi, -1)

def huella(P, lado_bajo, girar=False, sin_actor=True):
    """20 celdas de atacantes, 20 de defensores y atacantes a ≤ 2 yd del portero rival, en el marco del que cobra."""
    tm = P['teammate'].astype('boolean').fillna(False).to_numpy(bool); ac = P['actor'].astype('boolean').fillna(False).to_numpy(bool)
    kp = P['keeper'].astype('boolean').fillna(False).to_numpy(bool)
    px, py = P['x'].to_numpy(float), P['y'].to_numpy(float)
    if girar: px, py, tm = 120 - px, 80 - py, ~tm
    yr = py if lado_bajo else 80 - py
    a = tm & ~kp & (~ac if sin_actor else True); d = ~tm & ~kp
    ce = celda(px, yr)
    A = np.bincount(ce[a & (ce >= 0)], minlength=20); D = np.bincount(ce[d & (ce >= 0)], minlength=20)
    gk = ~tm & kp
    peg = np.nan
    if gk.any():
        g = np.c_[px[gk], yr[gk]][0]; peg = float((np.hypot(px[a] - g[0], yr[a] - g[1]) <= 2).sum())
    return A, D, peg

def elige_k(X, grupos, ks, rng_seed=SEMILLA):
    """Reproducibilidad: se ajusta en dos mitades de partidos y se compara cómo asignan TODOS los córners (ARI)."""
    rng = np.random.default_rng(rng_seed); u = np.unique(grupos); mitad = set(rng.choice(u, len(u) // 2, replace=False))
    m = np.array([g in mitad for g in grupos]); filas = []
    for k in ks:
        a = KMeans(k, n_init=10, random_state=rng_seed).fit(X[m]); b = KMeans(k, n_init=10, random_state=rng_seed).fit(X[~m])
        full = KMeans(k, n_init=10, random_state=rng_seed).fit(X)
        sub = rng.choice(len(X), min(len(X), 5000), replace=False)
        filas.append(dict(k=k, ari=adjusted_rand_score(a.predict(X), b.predict(X)), silueta=silhouette_score(X[sub], full.labels_[sub]),
                          inercia_por_corner=full.inertia_ / len(X)))
    T = pd.DataFrame(filas)
    cand = T[T.ari >= 0.9 * T.ari.max()]                     # casi igual de reproducibles → gana la mejor separación (silueta)
    kbest = int(cand.loc[cand.silueta.idxmax(), 'k'])
    return kbest, T

def concentracion(lab, equipo, temporada, n_placebo=200, rng_seed=SEMILLA):
    """Media entre equipo-temporadas de Σp² (uso de playbooks), real frente a etiquetas de equipo revueltas dentro de la temporada."""
    df = pd.DataFrame({'l': lab, 'e': equipo, 's': temporada})
    def stat(d):
        p = d.groupby(['s', 'e'])['l'].value_counts(normalize=True)
        return (p ** 2).groupby(level=[0, 1]).sum().mean()
    real = stat(df); rng = np.random.default_rng(rng_seed); pl = []
    for _ in range(n_placebo):
        d2 = df.copy(); d2['e'] = d2.groupby('s')['e'].transform(lambda v: rng.permutation(v.values)); pl.append(stat(d2))
    pl = np.array(pl); return real, pl.mean(), float((pl >= real).mean())
# Lectura: importa el TAMAÑO (real − placebo), no solo p: con miles de córners una diferencia mínima puede salir 'significativa'.

# ---------------- Prueba con datos inventados ----------------
rng0 = np.random.default_rng(SEMILLA)
LIBRETOS = {'pegados': ([(118.5, 38), (118.5, 41), (118, 44)], [(115, 38), (118, 41), (115, 44)]),          # se quedan en la línea del portero
            'carrera': ([(110, 40), (110, 41), (111, 39), (110, 42)], [(112, 32), (112, 34), (115, 33), (110, 41)]),  # punto de penal → primer palo
            'segundo': ([(106, 46), (106, 48), (110, 47)], [(110, 52), (112, 55), (115, 48)])}                         # cargan el segundo palo
def corner_falso(libreto, ruido=1.2):
    s, c = LIBRETOS[libreto]
    sa = np.array(s) + rng0.normal(0, ruido, (len(s), 2)); ca = np.array(c) + rng0.normal(0, ruido, (len(c), 2))
    return celda(sa[:, 0], sa[:, 1]), celda(ca[:, 0], ca[:, 1])
def corner_azar(n=4):
    sa = np.c_[rng0.uniform(102, 120, n), rng0.uniform(18, 62, n)]; ca = np.c_[rng0.uniform(102, 120, n), rng0.uniform(18, 62, n)]
    return celda(sa[:, 0], sa[:, 1]), celda(ca[:, 0], ca[:, 1])
def vec(cs, cc): return np.r_[np.bincount(cs[cs >= 0], minlength=20), np.bincount(cc[cc >= 0], minlength=20)]

# Prueba de la huella: un atacante pegado al portero (lado alto: y se refleja) y un defensor al primer palo
P_ = pd.DataFrame({'teammate': [True, True, False, False], 'actor': [True, False, False, False], 'keeper': [False, False, False, True],
                   'x': [120, 118.5, 110, 119.5], 'y': [79.5, 40.5, 47, 40]})
A_, D_, peg_ = huella(P_, lado_bajo=False)
print('--- (H) Huella de prueba: atacante en celda', int(np.argmax(A_)), f'({ZONA_CELDA[np.argmax(A_)]}, pegado={PEGADO[np.argmax(A_)]})',
      '· defensor en', ZONA_CELDA[np.argmax(D_)], f'· pegados al portero {peg_:.0f}', '(esperado: área chica, pegado=True · primer palo · 1)')
assert ZONA_CELDA[np.argmax(A_)] == 'area_chica' and PEGADO[np.argmax(A_)] and ZONA_CELDA[np.argmax(D_)] == 'primer_palo' and peg_ == 1
# Escenario SIN efecto: 20 equipos, córners al azar · CON efecto: cada equipo usa sobre todo un libreto (80%)
for nombre, con in [('SIN efecto', False), ('CON efecto', True)]:
    X, eq, part = [], [], []
    for e in range(20):
        prefer = list(LIBRETOS)[e % 3]
        for i in range(60):
            if con: lib = prefer if rng0.random() < 0.8 else rng0.choice(list(LIBRETOS)); cs, cc = corner_falso(lib)
            else: cs, cc = corner_azar()
            X.append(vec(cs, cc)); eq.append(f'E{e}'); part.append(f'E{e}-{i // 3}')
    X = np.array(X, float)
    kb, T = elige_k(X, np.array(part), [2, 3, 4, 5])
    lab = KMeans(kb, n_init=10, random_state=SEMILLA).fit_predict(X)
    real, plac, p = concentracion(lab, np.array(eq), np.zeros(len(eq)), n_placebo=100)
    print(f'--- {nombre}: ARI por k {dict(zip(T.k, T.ari.round(2)))} · silueta {dict(zip(T.k, T.silueta.round(2)))} → k elegido {kb} · concentración real {real:.3f} frente a placebo {plac:.3f} (exceso {real - plac:+.3f}; p = {p:.2f})')
print('Esperado: SIN efecto → ARI bajo y exceso de concentración ≈ 0. CON efecto → k = 3, ARI ≈ 1 y concentración real >> placebo (p ≈ 0).')



In [ ]:
#[R8p-1 v1.1] Corrida real
import os, time
import pyarrow.parquet as pq
RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5, LIGA = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5'), os.path.join(DP, 'liga')
SAL5 = os.path.join(RAIZ, 'salidas', 'fw5'); os.makedirs(SAL5, exist_ok=True)
REHACER = False
t_ini = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
BP = pd.read_parquet(os.path.join(FW5, 'r8_bp.parquet'))
C = BP[(BP.kind == 'corner') & (BP.ruta == 'directo') & BP['s_frame'].astype('boolean').fillna(False)].copy()
C['temporada'] = C.match_id.map(loc['temporada'])
C['rival'] = np.where(C.team == C.match_id.map(loc['local']), C.match_id.map(loc['visita']), C.match_id.map(loc['local']))
ruta_h = os.path.join(FW5, 'r8p_huellas.parquet')
if os.path.exists(ruta_h) and not REHACER:
    H = pd.read_parquet(ruta_h); print('Huellas de caché:', len(H))
else:
    # 1. Primer contacto de cada córner directo (misma definición que R8a: siguiente evento sin presiones ni recepciones fallidas, ≤ 5 s)
    piezas = []
    for temp, g in C.groupby('temporada'):
        E = pd.read_parquet(os.path.join(FW1, f"eventos_{temp.replace('/', '-')}.parquet"),
                            columns=['match_id', 'index', 'period', 't', 'type.name', 'team.name', 'ball_receipt.outcome.name'])
        E = E[E.match_id.isin(set(g.match_id))]
        F = E[(E['type.name'] != 'Pressure') & ~((E['type.name'] == 'Ball Receipt*') & E['ball_receipt.outcome.name'].notna())]
        F = F.rename(columns={'index': 'c_index', 'period': 'c_period', 't': 'c_t', 'type.name': 'c_tipo', 'team.name': 'c_team'})
        F['index'] = F['c_index']
        x = pd.merge_asof(g.sort_values('index'), F.sort_values('index')[['match_id', 'index', 'c_index', 'c_period', 'c_t', 'c_tipo', 'c_team']],
                          on='index', by='match_id', direction='forward', allow_exact_matches=False)
        val = (x.c_period == x.period) & (x.c_t <= x.t + 5)
        x.loc[~val, ['c_index', 'c_tipo', 'c_team']] = np.nan
        piezas.append(x)
    C = pd.concat(piezas, ignore_index=True)
    igual = (C.c_tipo.fillna('—') == C.contacto_tipo.fillna('—')).mean()
    print(f'--- (0) Coherencia con R8a: tipo de contacto idéntico en {100 * igual:.2f}% de {len(C):,} córners (debe ser 100)')
    # 2. Huellas desde las fotos 360
    filas = []
    for mid, g in C.groupby('match_id'):
        r = os.path.join(LIGA, 'jugadores360', f'{mid}.parquet')
        if not os.path.exists(r): continue
        nec = set(g['index']) | set(g['c_index'].dropna().astype(int))
        jj = pd.read_parquet(r, columns=['event_index', 'teammate', 'actor', 'keeper', 'x', 'y']); jj = jj[jj.event_index.isin(nec)]
        JJ = {k: v for k, v in jj.groupby('event_index')}
        for _, c in g.iterrows():
            if c['index'] not in JJ: continue
            A, D, peg = huella(JJ[c['index']], bool(c.lado_bajo))
            f = dict(match_id=mid, index=c['index'], pegados_portero=peg, **{f'sa{i}': A[i] for i in range(20)}, **{f'sd{i}': D[i] for i in range(20)})
            ci = c['c_index']
            if pd.notna(ci) and int(ci) in JJ:
                A2, _, _ = huella(JJ[int(ci)], bool(c.lado_bajo), girar=c['c_team'] != c['team'], sin_actor=False)
                f.update({f'ca{i}': A2[i] for i in range(20)})
            filas.append(f)
    H = pd.DataFrame(filas).merge(C[['match_id', 'index', 'team', 'rival', 'temporada', 'zona', 'tecnica', 'xg1', 'xg2', 'goles', 'caida_gana',
                                     'contacto_clase', 's_marc2', 's_marc_den']], on=['match_id', 'index'])
    H.to_parquet(ruta_h, index=False)
SA, SD, CA = [f'sa{i}' for i in range(20)], [f'sd{i}' for i in range(20)], [f'ca{i}' for i in range(20)]
print(f'--- (1) Córners directos con foto del saque: {len(H):,} · con foto del contacto {100 * H[CA[0]].notna().mean():.1f}%'
      f' · con portero visible {100 * H.pegados_portero.notna().mean():.1f}% ({time.time() - t_ini:.0f} s)')
print('  Atacantes y defensores por zona en el saque (media liga):',
      {z: f"atac {H[[c for c, zz in zip(SA, ZONA_CELDA) if zz == z]].sum(1).mean():.2f} · def {H[[c for c, zz in zip(SD, ZONA_CELDA) if zz == z]].sum(1).mean():.2f}"
       for z in ['primer_palo', 'punto_penal', 'area_chica', 'segundo_palo']})
print(f'  Pegados al portero (≤ 2 yd): media {H.pegados_portero.mean():.2f} · % de córners con 1 o más {100 * (H.pegados_portero >= 1).mean():.1f}')

def describe(lab, cols_s, cols_c=None, nombre='ofensivo'):
    d = H.assign(pb=lab); out = []
    for k, g in d.groupby('pb'):
        f = {'playbook': k, 'corners': len(g), '% liga': round(100 * len(g) / len(d), 1)}
        for z in ['primer_palo', 'punto_penal', 'area_chica', 'segundo_palo']:
            cs = [c for c, zz in zip(cols_s, ZONA_CELDA) if zz == z]
            f[z[:6] + '_saque'] = round(g[cs].sum(1).mean(), 2)
            if cols_c: f[z[:6] + '_contacto'] = round(g[[c for c, zz in zip(cols_c, ZONA_CELDA) if zz == z]].sum(1).mean(), 2)
        f['pegados_franja117'] = round(g[[c for c, p in zip(cols_s, PEGADO) if p]].sum(1).mean(), 2)
        if nombre == 'ofensivo':
            f['pegados_portero_2yd'] = round(g.pegados_portero.mean(), 2)
            for z in ['primer_palo', 'punto_penal', 'area_chica', 'segundo_palo']: f['envío_' + z[:6] + '%'] = round(100 * (g.zona == z).mean(), 1)
            f['cerrado%'] = round(100 * (g.tecnica == 'Inswinging').mean(), 1)
        else:
            m_ = g[g.s_marc_den > 0]; f['marcaje_2yd%'] = round(100 * m_.s_marc2.sum() / m_.s_marc_den.sum(), 1)
        f['xG_por_corner'] = round((g.xg1 + g.xg2).mean(), 4)
        out.append(f)
    return pd.DataFrame(out).set_index('playbook')

# 3. Playbooks ofensivos
Ho = H[H[CA[0]].notna()].reset_index(drop=True)
Xo = Ho[SA + CA].to_numpy(float)
ko, To = elige_k(Xo, Ho.match_id.values, range(4, 9))
print('\n--- (2) Playbooks OFENSIVOS · reproducibilidad por k (ARI entre mitades; 1 = idénticos):'); print(To.round(3).to_string(index=False))
if To.ari.max() < 0.5: print('  AVISO: ningún k se reproduce (ARI < 0.5): los grupos no son playbooks estables. NO nombrarlos; avisar a Claude.')
km_o = KMeans(ko, n_init=20, random_state=SEMILLA).fit(Xo); Ho['pb_of'] = km_o.labels_
H_ = H; H = Ho; Do = describe(Ho.pb_of.values, SA, CA, 'ofensivo'); H = H_
print(f'  k elegido = {ko}. Centroides (atacantes medios por zona en el saque y en el contacto; envío y xG):'); print(Do.T.to_string())
real, plac, p = concentracion(Ho.pb_of.values, Ho.team.values, Ho.temporada.values)
print(f'  Placebo de equipo: concentración Σp² real {real:.3f} frente a placebo {plac:.3f} · exceso {real - plac:+.3f} (p = {p:.3f})')
for k in range(ko):
    g = km_o.cluster_centers_[k]
    print(f'\n  Playbook {k} · saque (filas: x 102-108 / 108-114 / 114-117 / 117-120; columnas: y 18-30 / 30-36 / 36-44 / 44-50 / 50-62):')
    print('   ', np.round(g[:20].reshape(4, 5), 1).tolist()); print('    contacto:', np.round(g[20:].reshape(4, 5), 1).tolist())

# 4. Playbooks defensivos
Xd = H[SD].to_numpy(float)
kd, Td = elige_k(Xd, H.match_id.values, range(3, 7))
print('\n--- (3) Playbooks DEFENSIVOS · reproducibilidad por k:'); print(Td.round(3).to_string(index=False))
if Td.ari.max() < 0.5: print('  AVISO: ningún k defensivo se reproduce (ARI < 0.5). NO nombrarlos.')
km_d = KMeans(kd, n_init=20, random_state=SEMILLA).fit(Xd); H['pb_def'] = km_d.labels_
Dd = describe(H.pb_def.values, SD, None, 'defensivo')
print(f'  k elegido = {kd}. Centroides (defensores medios por zona; marcaje; xG concedido por córner):'); print(Dd.T.to_string())
real, plac, p = concentracion(H.pb_def.values, H.rival.values, H.temporada.values)
print(f'  Placebo de equipo (el que defiende): concentración real {real:.3f} frente a placebo {plac:.3f} · exceso {real - plac:+.3f} (p = {p:.3f})')
for k in range(kd): print(f'  Playbook defensivo {k} · saque:', np.round(km_d.cluster_centers_[k].reshape(4, 5), 1).tolist())

# 5. Guardar
H = H.merge(Ho[['match_id', 'index', 'pb_of']], on=['match_id', 'index'], how='left')
H.to_parquet(os.path.join(FW5, 'r8p_corners.parquet'), index=False)
Do.to_csv(os.path.join(SAL5, 'r8p_centroides_ofensivos.csv')); Dd.to_csv(os.path.join(SAL5, 'r8p_centroides_defensivos.csv'))
pd.DataFrame(km_o.cluster_centers_, columns=SA + CA).to_csv(os.path.join(SAL5, 'r8p_km_ofensivo.csv'), index=False)
pd.DataFrame(km_d.cluster_centers_, columns=SD).to_csv(os.path.join(SAL5, 'r8p_km_defensivo.csv'), index=False)
print(f'\nR8p-1 terminó en {time.time() - t_ini:.0f} s. Guardado: fw5/r8p_huellas, r8p_corners; salidas/fw5/r8p_*.csv. Pega toda la salida.')


R8z- Marcaje Ambas celdas

In [ ]:
# R8z v1.1 · Tipo de marcaje en córners defensivos: emparejamiento en el saque + respuesta al acomodo del rival
# =====================================================================
# Sesión 11 · 2026-10-17 · Hackathon ISAC 2026, Track 2 · Cinco temporadas (fw5)
# Decisiones (Arturo, sesión 11; metodología externa aportada por Arturo, adaptada a lo que sí tenemos):
#   1. Emparejamiento en T = 0 (foto 360 del saque): asignación óptima (algoritmo húngaro) entre defensores en el área (sin portero)
#      y atacantes en el área (sin el lanzador). Emparejado: su pareja a ≤ 1.3 yd (≈ 1.2 m). Zonal: ningún atacante a ≤ 2.2 yd (≈ 2 m).
#      Sensibilidad: (1.0, 2.0) y (1.6, 3.0) yd. Rasgos continuos: % de atacantes con marca · defensores zonales.
#   2. Respuesta al rival: dentro de cada equipo defensor (por temporada), ¿su reparto de defensores por zona sigue al reparto de
#      atacantes del rival? Pendiente ≈ 1 = sigue al rival (al hombre); ≈ 0 = forma fija (zona). Se usan PROPORCIONES por zona
#      (no conteos) para que la visibilidad de la cámara no cree una pendiente falsa.
#   3. ¿El acomodo defensivo (playbook de R8p) es reacción al playbook ofensivo del rival? Cramér's V de su cruce.
#   Chequeo secundario: cobertura en el contacto (% de atacantes con un defensor a ≤ 1.3 yd en la foto del contacto).
#   Límites: sin series de tiempo ni identidad de jugadores no se correlacionan trayectorias. Etiqueta hombre/zona/mixto: se fija
#   con Arturo viendo la distribución. Voronoi: banco de ideas para R9 (idea de Arturo).
# Orden: celda 1 y R0 → R8z-0 (prueba con datos inventados) → R8z-1 (corrida real; escribe fw5/r8z_marcaje y salidas/fw5).
# =====================================================================

# [R8z-0 v1.0] Funciones y prueba con datos inventados
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from scipy.stats import spearmanr, chi2_contingency

SEMILLA = 2026
UMBRALES = {'a': (1.0, 2.0), 'b': (1.3, 2.2), 'c': (1.6, 3.0)}            # b = principal (fuente de Arturo, en yardas)
AREA_X, AREA_Y, CHICA_X, CHICA_Y = 102, (18, 62), 114, (30, 50)
ZONAS = ['primer_palo', 'punto_penal', 'area_chica', 'segundo_palo']
def en_area(x, y): return (x >= AREA_X) & (y >= AREA_Y[0]) & (y <= AREA_Y[1])
def zona_vec(x, yr):
    ch = (x >= CHICA_X) & (yr >= CHICA_Y[0]) & (yr <= CHICA_Y[1])
    return np.where(ch, 'area_chica', np.where(yr < 36, 'primer_palo', np.where(yr <= 44, 'punto_penal', 'segundo_palo')))

def marcaje(ax, ay, dx, dy):
    """ax, ay: atacantes en el área; dx, dy: defensores en el área. Devuelve, por juego de umbrales, (% atacantes con marca, zonales)."""
    out = {}
    if len(ax) == 0 or len(dx) == 0:
        for k in UMBRALES: out[f'pct_marca_{k}'], out[f'zonales_{k}'] = np.nan, float(len(dx))
        return out
    D = np.hypot(dx[:, None] - ax[None, :], dy[:, None] - ay[None, :])
    fi, co = linear_sum_assignment(D); dist = D[fi, co]; dmin = D.min(1)
    for k, (u1, u2) in UMBRALES.items():
        out[f'pct_marca_{k}'] = 100.0 * (dist <= u1).sum() / len(ax)
        out[f'zonales_{k}'] = float((dmin > u2).sum())
    return out

def jugadores(P, lado_bajo, girar=False, sin_actor=True):
    tm = P['teammate'].astype('boolean').fillna(False).to_numpy(bool); ac = P['actor'].astype('boolean').fillna(False).to_numpy(bool)
    kp = P['keeper'].astype('boolean').fillna(False).to_numpy(bool)
    px, py = P['x'].to_numpy(float), P['y'].to_numpy(float)
    if girar: px, py, tm = 120 - px, 80 - py, ~tm
    yr = py if lado_bajo else 80 - py
    a = tm & ~kp & (~ac if sin_actor else np.ones_like(tm)) & en_area(px, yr); d = ~tm & ~kp & en_area(px, yr)
    return px[a], yr[a], px[d], yr[d]

def reparto(x, yr):
    if len(x) == 0: return np.full(4, np.nan)
    z = zona_vec(x, yr); return np.array([(z == q).mean() for q in ZONAS])

def pendiente(PA, PD):
    """PA, PD: matrices córner × 4 zonas (proporciones) de UN equipo defensor. Se centran por zona dentro del equipo."""
    ok = ~np.isnan(PA).any(1) & ~np.isnan(PD).any(1); A, D = PA[ok], PD[ok]
    if len(A) < 10: return np.nan
    A, D = A - A.mean(0), D - D.mean(0); a, d = A.ravel(), D.ravel()
    return float((a * d).sum() / (a * a).sum()) if (a * a).sum() > 0 else np.nan

# ---------------- Prueba con datos inventados ----------------
rng0 = np.random.default_rng(SEMILLA)
FIJOS = np.array([[116, 33], [116, 38], [116, 42], [116, 47], [111, 36], [111, 44], [107, 40], [113, 30]])   # zona: 8 puestos fijos
ACOMODOS = [np.array([[110, 40], [111, 38], [110, 42], [109, 41], [111, 39]]),     # paquete en el punto
            np.array([[116, 38], [117, 41], [116, 44], [110, 40], [109, 46]]),     # carga al portero
            np.array([[106, 38], [106, 42], [107, 40], [112, 33], [112, 48]])]     # carrera desde atrás
def corner_inv(tipo):
    A = ACOMODOS[rng0.integers(3)] + rng0.normal(0, 1.2, (5, 2))
    if tipo == 'hombre': D = np.r_[A + rng0.normal(0, 0.5, (5, 2)), FIJOS[[0, 3]] + rng0.normal(0, 0.4, (2, 2))]
    elif tipo == 'zona': D = FIJOS + rng0.normal(0, 0.4, FIJOS.shape)
    else: D = np.r_[A[:3] + rng0.normal(0, 0.5, (3, 2)), FIJOS[:5] + rng0.normal(0, 0.4, (5, 2))]
    return A, D
for nombre, tipos in [('SIN efecto (todos mixtos)', ['mixto'] * 20), ('CON efecto (10 al hombre, 10 en zona)', ['hombre'] * 10 + ['zona'] * 10)]:
    filas = []
    for e, tp in enumerate(tipos):
        PA, PD, pm = [], [], []
        for _ in range(60):
            A, D = corner_inv(tp); m = marcaje(A[:, 0], A[:, 1], D[:, 0], D[:, 1])
            pm.append(m['pct_marca_b']); PA.append(reparto(A[:, 0], A[:, 1])); PD.append(reparto(D[:, 0], D[:, 1]))
        filas.append(dict(equipo=e, tipo=tp, pct_marca=np.mean(pm), pendiente=pendiente(np.array(PA), np.array(PD))))
    T = pd.DataFrame(filas)
    rho = spearmanr(T.pct_marca, T.pendiente).correlation
    print(f'--- {nombre}:', T.groupby('tipo')[['pct_marca', 'pendiente']].mean().round(2).to_dict('index'),
          f'· correlación entre equipos (marca vs pendiente) {rho:+.2f}')
print('Esperado: SIN efecto → todos parecidos (la correlación puede ser cualquiera: no hay variación real).'
      ' CON efecto → al hombre: % de marca alto y pendiente ≈ 1; en zona: % bajo y pendiente ≈ 0; correlación alta y positiva.')


In [ ]:
# [R8z-1 v1.1] Corrida real (v1.1: freno si no se corrió R8z-0)
import os, time
RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
assert all(f in globals() for f in ['jugadores', 'marcaje', 'reparto', 'pendiente']), 'Corre primero R8z-0 (define las funciones)'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5, LIGA = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5'), os.path.join(DP, 'liga')
SAL5 = os.path.join(RAIZ, 'salidas', 'fw5'); os.makedirs(SAL5, exist_ok=True)
REHACER = False
t_ini = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
BP = pd.read_parquet(os.path.join(FW5, 'r8_bp.parquet'))
C = BP[(BP.kind == 'corner') & (BP.ruta == 'directo') & BP['s_frame'].astype('boolean').fillna(False)].copy()
C['temporada'] = C.match_id.map(loc['temporada'])
C['rival'] = np.where(C.team == C.match_id.map(loc['local']), C.match_id.map(loc['visita']), C.match_id.map(loc['local']))
ruta_z = os.path.join(FW5, 'r8z_marcaje.parquet')
if os.path.exists(ruta_z) and not REHACER:
    Z = pd.read_parquet(ruta_z); print('Marcaje de caché:', len(Z))
else:
    # Primer contacto: copia literal de R8p-1 (misma definición que R8a)
    piezas = []
    for temp, g in C.groupby('temporada'):
        E = pd.read_parquet(os.path.join(FW1, f"eventos_{temp.replace('/', '-')}.parquet"),
                            columns=['match_id', 'index', 'period', 't', 'type.name', 'team.name', 'ball_receipt.outcome.name'])
        E = E[E.match_id.isin(set(g.match_id))]
        F = E[(E['type.name'] != 'Pressure') & ~((E['type.name'] == 'Ball Receipt*') & E['ball_receipt.outcome.name'].notna())]
        F = F.rename(columns={'index': 'c_index', 'period': 'c_period', 't': 'c_t', 'type.name': 'c_tipo', 'team.name': 'c_team'})
        F['index'] = F['c_index']
        x = pd.merge_asof(g.sort_values('index'), F.sort_values('index')[['match_id', 'index', 'c_index', 'c_period', 'c_t', 'c_tipo', 'c_team']],
                          on='index', by='match_id', direction='forward', allow_exact_matches=False)
        val = (x.c_period == x.period) & (x.c_t <= x.t + 5)
        x.loc[~val, ['c_index', 'c_tipo', 'c_team']] = np.nan
        piezas.append(x)
    C = pd.concat(piezas, ignore_index=True)
    print(f'--- (0) Coherencia con R8a: tipo de contacto idéntico en {100 * (C.c_tipo.fillna("—") == C.contacto_tipo.fillna("—")).mean():.2f}% (debe ser 100)')
    filas = []
    for mid, g in C.groupby('match_id'):
        r = os.path.join(LIGA, 'jugadores360', f'{mid}.parquet')
        if not os.path.exists(r): continue
        nec = set(g['index']) | set(g['c_index'].dropna().astype(int))
        jj = pd.read_parquet(r, columns=['event_index', 'teammate', 'actor', 'keeper', 'x', 'y']); jj = jj[jj.event_index.isin(nec)]
        JJ = {k: v for k, v in jj.groupby('event_index')}
        for _, c in g.iterrows():
            if c['index'] not in JJ: continue
            ax, ay, dx, dy = jugadores(JJ[c['index']], bool(c.lado_bajo))
            f = dict(match_id=mid, index=c['index'], team=c.team, rival=c.rival, temporada=c.temporada, n_atac=len(ax), n_def=len(dx),
                     **marcaje(ax, ay, dx, dy), **{f'pa_{z}': v for z, v in zip(ZONAS, reparto(ax, ay))},
                     **{f'pd_{z}': v for z, v in zip(ZONAS, reparto(dx, dy))})
            ci = c['c_index']
            if pd.notna(ci) and int(ci) in JJ:
                bx, by, ex_, ey_ = jugadores(JJ[int(ci)], bool(c.lado_bajo), girar=c['c_team'] != c['team'], sin_actor=False)
                f['c_pct_marca_b'] = marcaje(bx, by, ex_, ey_)['pct_marca_b']
            filas.append(f)
    Z = pd.DataFrame(filas); Z.to_parquet(ruta_z, index=False)

print(f'--- (1) Córners: {len(Z):,} · atacantes en el área (mediana) {Z.n_atac.median():.0f} · defensores {Z.n_def.median():.0f} ({time.time() - t_ini:.0f} s)')
for k, (u1, u2) in UMBRALES.items():
    print(f'  Umbrales {u1}/{u2} yd: % de atacantes con marca · cuantiles 10/25/50/75/90 {Z[f"pct_marca_{k}"].quantile([.1, .25, .5, .75, .9]).round(0).tolist()}'
          f' · defensores zonales {Z[f"zonales_{k}"].quantile([.1, .25, .5, .75, .9]).tolist()}')
print('  Histograma del % de atacantes con marca (umbral principal, % de córners por tramo de 20 puntos):',
      pd.cut(Z.pct_marca_b, [-1, 20, 40, 60, 80, 101]).value_counts(normalize=True).sort_index().mul(100).round(1).to_dict())
zc = Z.dropna(subset=['c_pct_marca_b'])
print(f'  Contacto: % de atacantes con marca saque → contacto {zc.pct_marca_b.mean():.1f} → {zc.c_pct_marca_b.mean():.1f}'
      f' · correlación entre córners {zc[["pct_marca_b", "c_pct_marca_b"]].corr().iloc[0, 1]:.2f}')

# 2. Por equipo defensor y temporada: marca media, zonales y pendiente de respuesta al rival (con mitades para su confiabilidad)
rng = np.random.default_rng(SEMILLA)
PA_c, PD_c = [f'pa_{z}' for z in ZONAS], [f'pd_{z}' for z in ZONAS]
filas = []
for (eq, temp), g in Z.groupby(['rival', 'temporada']):
    if len(g) < 40: continue
    m = rng.random(len(g)) < 0.5
    filas.append(dict(equipo=eq, temporada=temp, corners=len(g), pct_marca=g.pct_marca_b.mean(), zonales=g.zonales_b.mean(),
                      pendiente=pendiente(g[PA_c].to_numpy(), g[PD_c].to_numpy()),
                      pend_m1=pendiente(g[m][PA_c].to_numpy(), g[m][PD_c].to_numpy()), pend_m2=pendiente(g[~m][PA_c].to_numpy(), g[~m][PD_c].to_numpy()),
                      marca_m1=g[m].pct_marca_b.mean(), marca_m2=g[~m].pct_marca_b.mean()))
T = pd.DataFrame(filas)
sb = lambda r: 2 * r / (1 + r)
print(f'\n--- (2) Equipos defensores (equipo-temporada con ≥ 40 córners): {len(T)}')
print('  % de atacantes con marca: cuantiles 10/50/90', T.pct_marca.quantile([.1, .5, .9]).round(1).tolist(),
      f'· confiabilidad entre mitades {sb(T[["marca_m1", "marca_m2"]].corr().iloc[0, 1]):.2f}')
print('  Pendiente de respuesta al rival: cuantiles 10/50/90', T.pendiente.quantile([.1, .5, .9]).round(2).tolist(),
      f'· confiabilidad entre mitades {sb(T[["pend_m1", "pend_m2"]].corr().iloc[0, 1]):.2f}')
print(f'  Pendiente de toda la liga (todos los córners juntos, centrados por equipo): '
      f'{pendiente(np.vstack([g[PA_c].to_numpy() - g[PA_c].mean().to_numpy() for _, g in Z.groupby("rival")]), np.vstack([g[PD_c].to_numpy() - g[PD_c].mean().to_numpy() for _, g in Z.groupby("rival")])):.2f}')
r_ = spearmanr(T.pct_marca, T.pendiente)
print(f'  VALIDACIÓN: correlación (Spearman) entre % de marca y pendiente {r_.correlation:+.2f} (p = {r_.pvalue:.3f}) · con zonales'
      f' {spearmanr(T.zonales, T.pendiente).correlation:+.2f} (esperado: positiva con la marca, negativa con los zonales)')
print('  Los 5 equipo-temporadas que más marcan y los 5 que menos (marca, zonales, pendiente):')
print(pd.concat([T.nlargest(5, 'pct_marca'), T.nsmallest(5, 'pct_marca')])[['equipo', 'temporada', 'corners', 'pct_marca', 'zonales', 'pendiente']].round(2).to_string(index=False))

# 3. ¿El acomodo defensivo es reacción al playbook ofensivo del rival?
rp = os.path.join(FW5, 'r8p_corners.parquet')
if os.path.exists(rp):
    P = pd.read_parquet(rp, columns=['match_id', 'index', 'pb_of', 'pb_def']).dropna()
    ct = pd.crosstab(P.pb_def, P.pb_of); chi2 = chi2_contingency(ct)[0]
    v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
    print(f'\n--- (3) Playbook defensivo × playbook ofensivo del rival: Cramér V = {v:.2f} (0 = independientes; > 0.3 = el acomodo defensivo es sobre todo reacción)')
    print('  % de cada playbook defensivo según el ofensivo que enfrenta (columnas suman 100):'); print(pd.crosstab(P.pb_def, P.pb_of, normalize='columns').mul(100).round(1).to_string())
    PZ = P.merge(Z[['match_id', 'index', 'pct_marca_b', 'zonales_b']], on=['match_id', 'index'])
    print('  Marca y zonales por playbook defensivo:'); print(PZ.groupby('pb_def')[['pct_marca_b', 'zonales_b']].mean().round(1).to_string())
T.to_csv(os.path.join(SAL5, 'r8z_equipos.csv'), index=False)
print(f'\nR8z-1 terminó en {time.time() - t_ini:.0f} s. Guardado: fw5/r8z_marcaje · salidas/fw5/r8z_equipos.csv. Pega toda la salida.')

R8u-0

In [ ]:
#[R8u-0 v1.0] Prueba con datos inventados y tramos de los 27 rasgos
import os, time
import numpy as np
import pandas as pd

def e5(m):                                  # copia de R4a
    m = np.asarray(m, dtype=float)
    return np.clip(np.sign(m) * np.ceil(np.abs(m) - 1e-9), -2, 2).astype(int)
def estado(df, defensa=False):
    """e5, e5b desde el equipo que cobra (defensa=False) o desde el que defiende (defensa=True). m_home viene de R8a (marcador local + offset)."""
    s = np.where(df['ev_local'].astype(bool), 1.0, -1.0) * (-1.0 if defensa else 1.0)
    return e5(s * df['m_home'].astype(float)), e5(s * df['m_home_b'].astype(float))
ZON = ['primer_palo', 'punto_penal', 'area_chica', 'segundo_palo']
def pendiente_partes(Z):
    """Numerador y denominador por córner, centrando las proporciones por equipo defensor y temporada."""
    PA, PD = Z[[f'pa_{z}' for z in ZON]], Z[[f'pd_{z}' for z in ZON]]
    ok = PA.notna().all(axis=1) & PD.notna().all(axis=1); Z = Z[ok].copy()
    g = Z.groupby(['rival', 'temporada'])
    A = Z[[f'pa_{z}' for z in ZON]] - g[[f'pa_{z}' for z in ZON]].transform('mean')
    D = Z[[f'pd_{z}' for z in ZON]] - g[[f'pd_{z}' for z in ZON]].transform('mean')
    Z['num'] = (A.values * D.values).sum(1); Z['den'] = (A.values ** 2).sum(1)
    return Z

# --- Prueba con datos inventados ---
# (1) Estado: el local cobra un córner yendo 1-0 arriba → atacante +1, defensor −1; con offset de liguilla −1 → 0 y 0
T_ = pd.DataFrame({'ev_local': [True, True], 'm_home': [1, 0], 'm_home_b': [1, 0]})
a_, _ = estado(T_); d_, _ = estado(T_, defensa=True)
print('--- (T1) Estado atacante', a_.tolist(), '· defensor', d_.tolist(), '(esperado [1, 0] y [-1, 0])'); assert a_.tolist() == [1, 0] and d_.tolist() == [-1, 0]
# (2) Pendiente: un defensor que copia al rival (pendiente 1) y uno con forma fija (pendiente 0); suma/suma debe dar 1 y 0
rng0 = np.random.default_rng(2026)
filas = []
for eq, copia in [('Hombre', True), ('Zona', False)]:
    for i in range(80):
        p = rng0.dirichlet(np.ones(4)); q = p if copia else np.array([0.25, 0.25, 0.25, 0.25])
        filas.append({'rival': eq, 'temporada': 'x', **{f'pa_{z}': v for z, v in zip(ZON, p)}, **{f'pd_{z}': v for z, v in zip(ZON, q)}})
Pz = pendiente_partes(pd.DataFrame(filas)); pz = Pz.groupby('rival').apply(lambda d: d.num.sum() / d.den.sum(), include_groups=False)
print('--- (T2) Pendiente como cociente de sumas:', pz.round(3).to_dict(), '(esperado Hombre 1, Zona 0)'); assert abs(pz['Hombre'] - 1) < 1e-9 and abs(pz['Zona']) < 1e-9
# (3) Sin efecto / con efecto en un rasgo de proporción: dos equipos con 30% y 30% frente a 60% y 20% de córners cortos
for nombre, (pa, pb) in {'SIN efecto': (0.3, 0.3), 'CON efecto': (0.6, 0.2)}.items():
    d = pd.DataFrame({'team': ['X'] * 400 + ['Y'] * 400, 'num': 100.0 * np.r_[rng0.random(400) < pa, rng0.random(400) < pb], 'den': 1.0})
    s = d.groupby('team')[['num', 'den']].sum(); r = (s.num / s.den).round(1)
    print(f'--- (T3) {nombre}: corto_pct X {r["X"]} · Y {r["Y"]} (esperado {100 * pa:.0f} y {100 * pb:.0f})')

# --- Tramos reales ---
RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5')
t0 = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
BP = pd.read_parquet(os.path.join(FW5, 'r8_bp.parquet'))
BP['rival'] = np.where(BP.team == BP.match_id.map(loc['local']), BP.match_id.map(loc['visita']), BP.match_id.map(loc['local']))
BP['temporada'] = BP.match_id.map(loc['temporada'])
B = BP[BP.period.isin([1, 2]) & BP.match_id.isin(loc.index)].copy()                     # D-9
B['e5'], B['e5b'] = estado(B); B['e5_d'], B['e5b_d'] = estado(B, defensa=True)
piezas = []
def agrega(d, num, nombre, den=1.0, defensa=False):
    d = d.assign(num=np.asarray(num, float), den=np.asarray(den, float) if np.ndim(den) else den)
    t = pd.DataFrame({'metrica': nombre, 'match_id': d.match_id.values, 'team': (d.rival if defensa else d.team).values,
                      'e5': (d.e5_d if defensa else d.e5).values, 'e5b': (d.e5b_d if defensa else d.e5b).values,
                      'cuarto': d.cuarto.astype(int).values, 'num': d.num.values, 'den': d.den.values})
    piezas.append(t.dropna(subset=['num', 'den']))

co = B[B.kind == 'corner']; cor = co[co.ruta.isin(['via_corto', 'reciclado'])]
env = co[co.ruta.isin(['directo', 'via_corto']) & co.zona.isin(ZON)]
dtec = co[(co.ruta == 'directo') & co.tecnica.notna()]
seg = co[co.caida_gana.notna() & (co.contacto_clase != 'control_atac')]
s360 = co[(co.ruta == 'directo') & co['s_frame'].astype('boolean').fillna(False)]
agrega(co, 100.0 * co.ruta.isin(['via_corto', 'reciclado']), 'corto_pct')
agrega(cor, 100.0 * (cor.ruta == 'via_corto'), 'corto_a_envio')
for z in ZON: agrega(env, 100.0 * (env.zona == z), f'envio_{z}')
agrega(dtec, 100.0 * (dtec.tecnica == 'Inswinging'), 'cerrado_pct')
agrega(co, 100.0 * (co.xg1 + co.xg2), 'xg_corner'); agrega(co, 100.0 * co.xg2, 'xg2_corner')
agrega(seg, 100.0 * seg.caida_gana.astype(bool), 'segunda_ganada')
agrega(s360, s360.s_atac_frontal, 'atac_frontal_360'); agrega(s360, s360.s_atac_area, 'atac_area_360')
sb = B[B.kind == 'saque_banda']; tl = B[B.kind == 'tiro_libre']
agrega(sb, 100.0 * (sb.ruta == 'directo'), 'saque_banda_largo')
agrega(tl, 100.0 * (tl.ruta == 'tiro_directo'), 'tl_directo'); agrega(tl, 100.0 * tl.ruta.isin(['via_corto', 'reciclado']), 'tl_corto')
# Defensa (córner del rival, asignado al que defiende)
agrega(co, 100.0 * (co.xg1 + co.xg2), 'xg_concedido_corner', defensa=True)
agrega(seg, 100.0 * (1 - seg.caida_gana.astype(bool)), 'segunda_recuperada', defensa=True)
agrega(s360, s360.s_def_area, 'def_area_360', defensa=True)
# Playbooks (R8p) y marcaje (R8z): se les pega el estado desde r8_bp
llave = ['match_id', 'index']
P = pd.read_parquet(os.path.join(FW5, 'r8p_corners.parquet'), columns=llave + ['pb_of', 'pegados_portero']).merge(B[B.kind == 'corner'], on=llave)
po = P[P.pb_of.notna()]
for k in range(int(po.pb_of.max()) + 1): agrega(po, 100.0 * (po.pb_of == k), f'pb_of_{k}')
pp = P[P.pegados_portero.notna()]; agrega(pp, pp.pegados_portero, 'pegados_portero')
Z = pd.read_parquet(os.path.join(FW5, 'r8z_marcaje.parquet')).drop(columns=['team', 'rival', 'temporada'], errors='ignore').merge(B[B.kind == 'corner'], on=llave)
zm = Z[Z.pct_marca_b.notna()]
agrega(zm, zm.pct_marca_b, 'pct_marca', defensa=True); agrega(zm, zm.zonales_b, 'zonales', defensa=True)
Zp = pendiente_partes(Z); agrega(Zp, Zp.num, 'pendiente_marca', den=Zp.den.values, defensa=True)
print(f'\n--- (0) Enlaces: playbooks {len(P):,} de {len(pd.read_parquet(os.path.join(FW5, "r8p_corners.parquet"), columns=["index"])):,} · marcaje {len(Z):,} ({time.time() - t0:.0f} s)')

# Peso del balón parado (por tiro, sin penales): marcador reconstruido con la copia literal de R6a sobre tiros y autogoles
pesos, ch_t, ch_p = [], [], []
OFF = {1: 0, 2: 45, 3: 90, 4: 105}
PE = pd.read_parquet(os.path.join(FW5, 'r8_penales.parquet'))
cor_t = BP[BP.kind == 'corner'][['match_id', 'index', 'period', 't', 'team']]
for temp in sorted(Mm.temporada.unique()):
    E = pd.read_parquet(os.path.join(FW1, f"eventos_{temp.replace('/', '-')}.parquet"),
                        columns=['match_id', 'index', 'period', 'minute', 't', 'type.name', 'team.name', 'shot.type.name', 'shot.statsbomb_xg',
                                 'shot.outcome.name', 'fase'])
    E = E[E.match_id.isin(loc.index)].sort_values(['match_id', 'index'])
    ok_t = ((E.t - 60 * (E.minute - E.period.map(OFF))).between(-0.01, 60.01) | E.period.eq(5)).mean()
    ch_t.append((temp, round(100 * ok_t, 2)))
    pm = PE[PE.match_id.isin(set(E.match_id))].merge(E[['match_id', 'index', 't']], on=['match_id', 'index'], how='left')
    ch_p.append(pm)
    E = E[E['type.name'].isin(['Shot', 'Own Goal For'])].copy()
    E['local'] = E['match_id'].map(loc['local']); E['ev_local'] = E['team.name'] == E['local']
    es_gol = E['period'].between(1, 4) & (((E['type.name'] == 'Shot') & (E['shot.outcome.name'] == 'Goal')) | (E['type.name'] == 'Own Goal For'))
    gh = (es_gol & E['ev_local']).astype(int); ga = (es_gol & ~E['ev_local'] & E['team.name'].notna()).astype(int)
    E['dif_home'] = (gh.groupby(E['match_id']).cumsum() - gh) - (ga.groupby(E['match_id']).cumsum() - ga)
    E['m_home'] = E['dif_home'] + E['match_id'].map(loc['offset']); E['m_home_b'] = E['dif_home'] + E['match_id'].map(loc['offset_b'])
    E['cuarto'] = np.where(E['period'] == 1, np.where(E['minute'] < 25, 1, 2), np.where(E['minute'] < 70, 3, 4))
    sh = E[(E['type.name'] == 'Shot') & (E['shot.type.name'] != 'Penalty') & E.period.isin([1, 2]) & (E['shot.statsbomb_xg'] > 0)]
    sh = sh.rename(columns={'team.name': 'team'}); sh['e5'], sh['e5b'] = estado(sh)
    agrega(sh, 100.0 * sh['shot.statsbomb_xg'] * (sh.fase == 'bp'), 'peso_bp', den=sh['shot.statsbomb_xg'].values)
print('--- (1) Chequeo de t: % de eventos con t = segundos desde el inicio del PERIODO (minuto − 0/45/90/105):', dict(ch_t), '(debe ser ~100)')
PM = pd.concat(ch_p, ignore_index=True)
x = PM.merge(cor_t.rename(columns={'index': 'c_index', 't': 'c_t', 'team': 'c_team'}), on=['match_id', 'period'])
x = x[(x.c_team == x.team) & (x.t - x.c_t).between(0, 15)]
print(f'--- (2) Penales cometidos ≤ 15 s después de un córner del beneficiado: {x[["match_id", "index"]].drop_duplicates().shape[0]} de {len(PM)}'
      f' ({100 * x[["match_id", "index"]].drop_duplicates().shape[0] / len(PM):.1f}%) · R8a había detectado {int(BP.loc[BP.kind == "corner", "penal"].sum())} dentro de la ventana')

TR = pd.concat(piezas, ignore_index=True).groupby(['metrica', 'match_id', 'team', 'e5', 'e5b', 'cuarto'], as_index=False)[['num', 'den']].sum()
TR = TR[TR.den != 0]
TR.to_parquet(os.path.join(FW5, 'r8_tramos.parquet'), index=False)
res = TR.groupby('metrica').apply(lambda d: pd.Series({'equipo_partidos': d[['match_id', 'team']].drop_duplicates().shape[0],
                                                       'eventos_por_ep': d.groupby(['match_id', 'team']).size().median(),
                                                       'media_liga': d.num.sum() / d.den.sum()}), include_groups=False)
print(f'--- (3) Rasgos ({TR.metrica.nunique()}). Media de la liga en su unidad (% · xG×100 por córner · jugadores · pendiente):'); print(res.round(3).to_string())
print(f'R8u-0 terminó en {time.time() - t0:.0f} s. Guardado: fw5/r8_tramos.parquet. Siguiente: R8r-0.')



R8r-0

In [ ]:

# [R8r-0 v1.0] Carpeta espejo para correr R6r-1 y R6r-2 sin cambios sobre fw5 (copia de R7r-0 con prefijo r8)
import os, json, shutil
DRIVE = globals().get('DRIVE_R8', '/content/drive/MyDrive/isac2026/')
LOCAL = globals().get('LOCAL_R8', '/content/isac_r8/')
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
D_DP, D_FW1, D_FW5 = os.path.join(DRIVE, 'datos_parquet'), os.path.join(DRIVE, 'datos_parquet', 'fw1'), os.path.join(DRIVE, 'datos_parquet', 'fw5')
L_DP, L_FW = os.path.join(LOCAL, 'datos_parquet'), os.path.join(LOCAL, 'datos_parquet', 'fw1')
shutil.rmtree(LOCAL, ignore_errors=True)
os.makedirs(L_FW, exist_ok=True); os.makedirs(os.path.join(LOCAL, 'salidas'), exist_ok=True)
def enlazar(src, dst):
    if os.path.exists(src): os.symlink(src, dst); return True
    return False
enl = {'partido_tecnico_periodo.parquet': enlazar(os.path.join(D_DP, 'partido_tecnico_periodo.parquet'), os.path.join(L_DP, 'partido_tecnico_periodo.parquet'))}
for f in ['r4_partidos.parquet', 'r4_minutos_v21.parquet', 'parametros_fw1.json', 'r4_perfiles_etapa.parquet',
          'r6_perfiles_etapa.parquet', 'r6b_perfiles_etapa.parquet', 'r6c_perfiles_etapa.parquet', 'r8_tramos.parquet']:
    enl[f] = enlazar(os.path.join(D_FW5, f), os.path.join(L_FW, f))
enl['elo_equipo_partido_e2.parquet (fw1, solo lectura)'] = enlazar(os.path.join(D_FW1, 'elo_equipo_partido_e2.parquet'), os.path.join(L_FW, 'elo_equipo_partido_e2.parquet'))
print('--- (R8r-0) Carpeta espejo', LOCAL); print({k: ('ok' if v else 'FALTA') for k, v in enl.items()})
oblig = ['partido_tecnico_periodo.parquet', 'r4_partidos.parquet', 'r4_minutos_v21.parquet', 'parametros_fw1.json', 'r4_perfiles_etapa.parquet',
         'r8_tramos.parquet', 'elo_equipo_partido_e2.parquet (fw1, solo lectura)']
assert all(enl[k] for k in oblig), 'Falta un insumo obligatorio: avisar a Claude'
with open(os.path.join(L_FW, 'parametros_fw1.json')) as f:
    print('Temporadas de ajuste:', json.load(f)['temporadas']['entrenamiento'])
BASE = LOCAL; PREFIJO = 'r8'
print("BASE y PREFIJO listos → corre R6r-1 v1.1 y R6r-2 v1.1 (claude/codigo/sesion8/R6r_receta_confiabilidad.py) sin cambiarles nada; luego R8r-9.")




R8r-9

In [ ]:
#[R8r-9 v1.0] Copiar resultados a Drive (fw5 y salidas/fw5) y regresar BASE a Drive (copia de R7r-9 con prefijo r8)
import os, shutil
DRIVE = globals().get('DRIVE_R8', '/content/drive/MyDrive/isac2026/')
LOCAL = globals().get('LOCAL_R8', '/content/isac_r8/')
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
L_FW = os.path.join(LOCAL, 'datos_parquet', 'fw1')
D_FW5, D_SAL5 = os.path.join(DRIVE, 'datos_parquet', 'fw5'), os.path.join(DRIVE, 'salidas', 'fw5')
os.makedirs(D_SAL5, exist_ok=True); n = []
for src_dir, dst_dir in [(L_FW, D_FW5), (os.path.join(LOCAL, 'salidas'), D_SAL5)]:
    for f in os.listdir(src_dir):
        s = os.path.join(src_dir, f)
        if f.startswith('r8_') and os.path.isfile(s) and not os.path.islink(s):
            shutil.copy2(s, os.path.join(dst_dir, f)); n.append(f)
print('--- (R8r-9) Copiados a Drive:', sorted(n))
BASE = DRIVE
print('BASE =', BASE)


R8m (cruces playbooks, ambos)

In [ ]:
# R8m v1.0 · (a) Playbook ofensivo frente al tipo de marcaje del rival · (b) ¿Técnico o plantilla? Control con HOPS y estatura
# =====================================================================
# Sesión 11 · 2026-10-17 · Hackathon ISAC 2026, Track 2 · Cinco temporadas (fw5)
# (a) Unidad: córner directo con playbook (R8p). Resultado: xG de la ventana × 100 (R8a). Tipo de marcaje del RIVAL = tercio de la liga
#     según sus defensores zonales medios en la temporada (R8z; equipo-temporadas con ≥ 40 córners): más al hombre · mixto · más zonal.
#     Se usa el marcaje típico del rival, no el del córner, porque el del córner reacciona al playbook (Cramér V 0.35).
#     Comparación DENTRO del equipo atacante (se resta su media de temporada; regla 14) y en diferencia de diferencias:
#     ventaja del playbook X contra la zona (más zonal − más al hombre) MENOS la misma ventaja del resto de playbooks.
#     Así se quita que los rivales zonales sean, en general, más débiles o más fuertes. IC por remuestreo de partidos (500).
#     Asociación, no causa: el técnico elige el playbook sabiendo quién defiende.
# (b) Capacidad aérea de la plantilla en cada partido = media de los 5 mejores (entre los que jugaron) en:
#     HOPS (duel.hops_rating de StatsBomb, media por jugador y temporada) y estatura (alineaciones). Se correlaciona, entre
#     equipo-temporadas, con los rasgos de "cargar el área chica" y con el xG por córner. Saque de banda largo: ¿lo hace un solo jugador?
# Orden: celda 1 y R0 → R8m-0 (prueba) → R8m-1 (corrida real; escribe fw5/r8m_* y salidas/fw5/r8m_*). Primera vez ~10-15 min (HOPS de los crudos).
# =====================================================================

# [R8m-0 v1.0] Funciones y prueba con datos inventados
import numpy as np
import pandas as pd

SEMILLA = 2026
TERCIOS = ['mas_al_hombre', 'mixto', 'mas_zonal']

def cruce(d, B=500, seed=SEMILLA):
    """d: columnas y, pb, tercil, ats (equipo atacante-temporada), match_id. Devuelve tabla de celdas y contrastes con IC."""
    d = d.dropna(subset=['y', 'pb', 'tercil']).copy()
    d['yc'] = d['y'] - d.groupby('ats')['y'].transform('mean')
    pbs = sorted(d.pb.unique())
    def stats(x):
        m = x.groupby(['pb', 'tercil'])['yc'].mean().unstack()
        c = m['mas_zonal'] - m['mas_al_hombre']
        did = pd.Series({p: c[p] - c.drop(p).mean() for p in c.index})
        return c, did
    c0, did0 = stats(d)
    rng = np.random.default_rng(seed); mids = d.match_id.unique(); g = {m: i for m, i in d.groupby('match_id').indices.items()}
    bc, bd = [], []
    for _ in range(B):
        s = rng.choice(mids, len(mids), replace=True); idx = np.concatenate([g[m] for m in s])
        c, did = stats(d.iloc[idx]); bc.append(c.reindex(pbs)); bd.append(did.reindex(pbs))
    bc, bd = pd.DataFrame(bc), pd.DataFrame(bd)
    T = pd.DataFrame({'zona_menos_hombre': c0.reindex(pbs), 'ic_lo': bc.quantile(.025), 'ic_hi': bc.quantile(.975),
                      'frente_resto_DiD': did0.reindex(pbs), 'did_lo': bd.quantile(.025), 'did_hi': bd.quantile(.975)})
    celdas = d.groupby(['pb', 'tercil']).agg(corners=('y', 'size'), xg_x100=('y', 'mean'), xg_x100_dentro=('yc', 'mean')).round(2)
    return celdas, T

# Prueba: 30 equipos atacantes de calidad distinta, 4 playbooks, rivales de 3 tercios. SIN efecto: nada. CON efecto: playbook 2 rinde +3
# contra la zona. Además, en ambos escenarios los rivales zonales conceden +1 a TODOS (para comprobar que la DiD lo quita).
rng0 = np.random.default_rng(SEMILLA)
for nombre, ef in [('SIN efecto', 0.0), ('CON efecto (+3 del playbook 2 contra la zona)', 3.0)]:
    filas = []
    for e in range(30):
        cal = rng0.normal(0, 1)
        for i in range(300):
            pb, ter = int(rng0.integers(4)), TERCIOS[rng0.integers(3)]
            y = 3.2 + cal + (1.0 if ter == 'mas_zonal' else 0) + (ef if (pb == 2 and ter == 'mas_zonal') else 0) + rng0.normal(0, 8)
            filas.append(dict(y=y, pb=pb, tercil=ter, ats=f'E{e}', match_id=f'E{e}-{i // 3}'))
    _, T = cruce(pd.DataFrame(filas), B=200)
    print(f'--- {nombre}:'); print(T.round(2).to_string())
print('Esperado: SIN efecto → zona_menos_hombre ≈ +1 en todos (efecto general de los rivales) y DiD ≈ 0 con IC que incluye 0.'
      ' CON efecto → DiD del playbook 2 ≈ +3 (IC sin 0) y los demás ≈ −1 (reparten el contraste).')


#[R8m-1 v1.0] Corrida real
import os, json, time
import pyarrow.parquet as pq
from scipy.stats import spearmanr
RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
assert 'cruce' in globals(), 'Corre primero R8m-0'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5, LIGA = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5'), os.path.join(DP, 'liga')
SAL5 = os.path.join(RAIZ, 'salidas', 'fw5'); os.makedirs(SAL5, exist_ok=True)
REHACER = False
t_ini = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')

# ---------- (a) Cruce ----------
P = pd.read_parquet(os.path.join(FW5, 'r8p_corners.parquet'))
Z = pd.read_parquet(os.path.join(FW5, 'r8z_marcaje.parquet'))
zt = Z.groupby(['rival', 'temporada']).agg(n=('zonales_b', 'size'), zonales=('zonales_b', 'mean')).reset_index()
zt = zt[zt.n >= 40].copy(); zt['tercil'] = pd.qcut(zt.zonales, 3, labels=TERCIOS).astype(str)
print('--- (a0) Tercios de marcaje (equipo-temporadas defensoras): zonales medios por tercio',
      zt.groupby('tercil').zonales.agg(['size', 'min', 'max']).round(2).to_dict('index'))
D = P[P.pb_of.notna()].merge(zt[['rival', 'temporada', 'tercil']], on=['rival', 'temporada'], how='left')
D = D.assign(y=100 * (D.xg1 + D.xg2), pb=D.pb_of.astype(int), ats=D.team + ' · ' + D.temporada)
print(f'  Córners con playbook y tercio del rival: {D.tercil.notna().sum():,} de {len(D):,}')
celdas, T = cruce(D)
NOMBRE_PB = {0: 'paquete que se abre', 1: 'carrera desde atrás', 2: 'carga al portero', 3: 'ataque al punto'}
print('\n--- (a1) xG×100 por córner en cada cruce (bruto y dentro del equipo atacante):'); print(celdas.to_string())
T.index = [f'{k} {NOMBRE_PB.get(k, "")}' for k in T.index]
print('\n--- (a2) Contra un rival MÁS ZONAL frente a uno MÁS AL HOMBRE (xG×100 por córner, dentro del atacante; IC 95% por partidos):')
print(T.round(2).to_string())
print('  Lectura: zona_menos_hombre > 0 = el playbook rinde más contra la zona. frente_resto_DiD = esa ventaja menos la del resto de playbooks'
      ' (quita que los rivales zonales sean mejores o peores en general). Solo la DiD con IC sin 0 es un hallazgo.')
T.to_csv(os.path.join(SAL5, 'r8m_cruce.csv'))

# ---------- (b) Plantilla: HOPS, estatura y lanzador de saque largo ----------
ruta_h = os.path.join(FW5, 'r8m_hops_jugador.parquet')
if os.path.exists(ruta_h) and not REHACER:
    HJ = pd.read_parquet(ruta_h); print('\nHOPS por jugador de caché:', len(HJ))
else:
    piezas, sin_col = [], 0
    for mid in Mm.match_id:
        r = os.path.join(LIGA, 'eventos', f'{mid}.parquet')
        if not os.path.exists(r): continue
        disp = set(pq.read_schema(r).names)
        cols = [c for c in ['id', 'type.name', 'duel.type.name', 'player.id', 'duel.hops_rating', 'duel.hops_opponent_rating', 'related_events',
                            'clearance.aerial_won', 'shot.aerial_won', 'pass.aerial_won', 'miscontrol.aerial_won'] if c in disp]
        if 'duel.hops_rating' not in cols: sin_col += 1; continue
        e = pd.read_parquet(r, columns=cols)
        aw = e[[c for c in cols if c.endswith('aerial_won')]].eq(True).any(axis=1) if any(c.endswith('aerial_won') for c in cols) else pd.Series(False, index=e.index)
        gan = dict(zip(e.loc[aw, 'id'], e.loc[aw, 'player.id']))
        al = e[(e['type.name'] == 'Duel') & (e['duel.type.name'] == 'Aerial Lost')]
        def ganador(v):
            ids = json.loads(v) if isinstance(v, str) else (list(v) if isinstance(v, (list, np.ndarray)) else [])
            return next((gan[i] for i in ids if i in gan), np.nan)
        w = al['related_events'].map(ganador) if 'related_events' in al else np.nan
        piezas.append(pd.DataFrame({'match_id': mid, 'player_id': np.r_[al['player.id'].values, np.asarray(w, dtype=float)],
                                    'hops': np.r_[al['duel.hops_rating'].values, al.get('duel.hops_opponent_rating', pd.Series(np.nan, index=al.index)).values]}))
    X = pd.concat(piezas, ignore_index=True).dropna()
    X['temporada'] = X.match_id.map(loc['temporada'])
    HJ = X.groupby(['player_id', 'temporada']).agg(hops=('hops', 'mean'), duelos=('hops', 'size')).reset_index()
    HJ.to_parquet(ruta_h, index=False)
    print(f'\nHOPS leído de los crudos (partidos sin la columna: {sin_col}) · observaciones {len(X):,}')
print(f'--- (b0) HOPS por jugador-temporada: {len(HJ):,} · cuantiles {HJ.hops.quantile([.1, .5, .9]).round(1).tolist()} · mediana de duelos {HJ.duelos.median():.0f}')
JP = pd.read_parquet(os.path.join(FW5, 'r7_jugador_partido.parquet'))
print('  columnas de r7_jugador_partido:', list(JP.columns)[:20])
pcol = next(c for c in ['player_id', 'player.id', 'jugador_id', 'jugador'] if c in JP.columns)
mcol = next(c for c in ['minutos', 'min', 'minutes'] if c in JP.columns)
JP = JP[JP[mcol] > 0].rename(columns={pcol: 'player_id'})
JP['temporada'] = JP.match_id.map(loc['temporada'])
JP = JP.merge(HJ[HJ.duelos >= 10][['player_id', 'temporada', 'hops']], on=['player_id', 'temporada'], how='left')
alt = []
for mid in Mm.match_id:
    r = os.path.join(LIGA, 'lineups', f'{mid}.parquet')
    if os.path.exists(r): alt.append(pd.read_parquet(r, columns=['player_id', 'player_height']))
ALT = pd.concat(alt).dropna().query('player_height > 0').drop_duplicates('player_id')
JP = JP.merge(ALT, on='player_id', how='left')
top5 = lambda s: s.dropna().nlargest(5).mean() if s.notna().sum() >= 5 else np.nan
TM = JP.groupby(['match_id', 'team']).agg(aereo_hops=('hops', top5), aereo_altura=('player_height', top5)).reset_index()
TM['temporada'] = TM.match_id.map(loc['temporada'])
print(f'  Equipo-partidos con HOPS de 5+ jugadores {100 * TM.aereo_hops.notna().mean():.1f}% · con estatura {100 * TM.aereo_altura.notna().mean():.1f}%'
      f' · correlación HOPS-estatura (equipo-partido) {TM[["aereo_hops", "aereo_altura"]].corr().iloc[0, 1]:.2f}')
# Rasgos crudos por equipo-temporada
BP = pd.read_parquet(os.path.join(FW5, 'r8_bp.parquet')); BP['temporada'] = BP.match_id.map(loc['temporada'])
co = BP[BP.kind == 'corner']; env = co[co.ruta.isin(['directo', 'via_corto']) & co.zona.notna() & (co.zona != 'tiro')]
R = pd.DataFrame({
    'carga_portero_pb2': P[P.pb_of.notna()].groupby(['team', 'temporada']).pb_of.apply(lambda s: 100 * (s == 2).mean()),
    'envio_area_chica': env.groupby(['team', 'temporada']).zona.apply(lambda s: 100 * (s == 'area_chica').mean()),
    'cerrado': co[(co.ruta == 'directo') & co.tecnica.notna()].groupby(['team', 'temporada']).tecnica.apply(lambda s: 100 * (s == 'Inswinging').mean()),
    'pegados_portero': P.groupby(['team', 'temporada']).pegados_portero.mean(),
    'xg_corner_x100': co.groupby(['team', 'temporada']).apply(lambda d: 100 * (d.xg1 + d.xg2).mean(), include_groups=False),
    'saque_largo': BP[BP.kind == 'saque_banda'].groupby(['team', 'temporada']).ruta.apply(lambda s: 100 * (s == 'directo').mean()),
    'corners': co.groupby(['team', 'temporada']).size()})
R = R[R.corners >= 40].join(TM.groupby(['team', 'temporada'])[['aereo_hops', 'aereo_altura']].mean())
print(f'\n--- (b1) Equipo-temporadas: {len(R)}. ¿La capacidad aérea de la plantilla explica el balón parado? (Spearman; p entre paréntesis)')
filas = []
for c in ['carga_portero_pb2', 'envio_area_chica', 'cerrado', 'pegados_portero', 'xg_corner_x100', 'saque_largo']:
    a, b = spearmanr(R[c], R.aereo_hops, nan_policy='omit'), spearmanr(R[c], R.aereo_altura, nan_policy='omit')
    filas.append(dict(rasgo=c, con_HOPS=f'{a.correlation:+.2f} ({a.pvalue:.3f})', con_estatura=f'{b.correlation:+.2f} ({b.pvalue:.3f})'))
print(pd.DataFrame(filas).to_string(index=False))
print('  Lectura: |r| < 0.3 → el rasgo NO depende de tener cabeceadores (refuerza "técnico"); |r| ≥ 0.5 → la plantilla pesa.')
# Saque de banda largo: ¿un solo lanzador?
th = BP[(BP.kind == 'saque_banda') & (BP.ruta == 'directo')][['match_id', 'index', 'team', 'temporada']]
pj = []
for temp, g in th.groupby('temporada'):
    E = pd.read_parquet(os.path.join(FW1, f"eventos_{temp.replace('/', '-')}.parquet"), columns=['match_id', 'index', 'player.id'])
    pj.append(g.merge(E, on=['match_id', 'index'], how='left'))
TH = pd.concat(pj)
lz = TH.groupby(['team', 'temporada']).agg(largos=('player.id', 'size'), top=('player.id', lambda s: s.value_counts(normalize=True).iloc[0] if s.notna().any() else np.nan))
lz = lz[lz.largos >= 15]
print(f'\n--- (b2) Saque de banda largo: equipo-temporadas con 15+ largos {len(lz)} · % de los largos que cobra el lanzador principal:'
      f' mediana {100 * lz.top.median():.0f}% · p25 {100 * lz.top.quantile(.25):.0f}% · p75 {100 * lz.top.quantile(.75):.0f}%')
R2 = R.join(lz, how='inner')
r_ = spearmanr(R2.saque_largo, R2.top)
print(f'  Correlación entre % de saques largos y concentración en un lanzador: {r_.correlation:+.2f} (p = {r_.pvalue:.3f})'
      ' (positiva = el saque largo existe porque hay un especialista)')
R.reset_index().to_csv(os.path.join(SAL5, 'r8m_plantilla_equipo_temporada.csv'), index=False)
print(f'\nR8m-1 terminó en {time.time() - t_ini:.0f} s. Guardado: fw5/r8m_hops_jugador · salidas/fw5/r8m_cruce.csv, r8m_plantilla_equipo_temporada.csv. Pega toda la salida.')

R8d- Penales, xg en contra y contras

In [ ]:
# R8d v1.0 · Piezas desacotadas de balón parado: (1) zona vulnerable en defensa · (2) penales por jugador y técnico · (3) contra tras córner (BP-6)
# =====================================================================
# Sesión 11 · 2026-10-17 · Hackathon ISAC 2026, Track 2 · Cinco temporadas (fw5) · Descriptivo (sin R4 + R5)
# (1) xG concedido por córner según la zona del envío (R8a), por etapa del técnico DEFENSOR (15+ partidos), frente a la liga.
#     Zona vulnerable = mayor exceso sobre la liga con ≥ 30 envíos. Se mide su estabilidad (mitades de partidos) antes de creerla.
# (2) Chequeo de Arturo: jugadores que más penales provocan (R8a, beneficiado con acción previa). Encares en el área por 90
#     (regate o conducción que empieza o termina en el área, R1) por jugador y etapa. ¿El técnico mueve el encare de sus jugadores?
#     Residuo del jugador (frente a su propia media entre etapas) contra el efecto de la etapa calculado SIN él (dejar uno fuera).
# (3) BP-6 / O-11: "contra tras balón parado" = posesión del rival que nace de recuperación (O-6) y empieza ≤ 20 s después del saque del
#     córner (aproximación a la ventana O-10), y es contragolpe si tiene un evento propio en x ≥ 80 en ≤ 10 s (O-8; copia literal de R4a:
#     tipos Ball Receipt*, Carry, Duel, Shot). Se relaciona con los atacantes comprometidos en el área (360, R8a).
# Orden: celda 1 y R0 → R8d-0 (prueba) → R8d-1 (corrida real; escribe salidas/fw5/r8d_*). ~5-10 min.
# =====================================================================

# [R8d-0 v1.0] Funciones y prueba con datos inventados
import numpy as np
import pandas as pd

SEMILLA = 2026
TIPOS_CG = {'Ball Receipt*', 'Carry', 'Duel', 'Shot'}          # O-8 (R4a)
VENTANA_CONTRA = 20.0
def contra_tras_corner(E, CO):
    """E: eventos con match_id, period, t, possession, possession_team.name, team.name, nace_rec, t_pos, loc_x, type.name, shot.statsbomb_xg.
    CO: córners con match_id, period, t, team, rival. Devuelve CO con salida (posesión rival ≤ 20 s), nace_rec, contragolpe y xG de esa posesión."""
    pos = E.groupby(['match_id', 'possession']).agg(period=('period', 'first'), t0=('t', 'min'), pteam=('possession_team.name', 'first'),
                                                     nace=('nace_rec', lambda s: bool(s.fillna(False).astype(bool).iloc[0]))).reset_index()
    cg = E[(E['team.name'] == E['possession_team.name']) & E['nace_rec'].fillna(False).astype(bool) & E['t_pos'].between(0, 10)
           & (E['loc_x'] >= 80) & E['type.name'].isin(TIPOS_CG)].groupby(['match_id', 'possession']).size().rename('n_cg')
    xg = E[(E['type.name'] == 'Shot') & (E['team.name'] == E['possession_team.name'])].groupby(['match_id', 'possession'])['shot.statsbomb_xg'].sum().rename('xg_rival')
    pos = pos.join(cg, on=['match_id', 'possession']).join(xg, on=['match_id', 'possession']).fillna({'n_cg': 0, 'xg_rival': 0})
    out = []
    for (m, p, r), g in CO.groupby(['match_id', 'period', 'rival']):
        q = pos[(pos.match_id == m) & (pos.period == p) & (pos.pteam == r)].sort_values('t0')
        for i, c in g.iterrows():
            s = q[(q.t0 > c.t) & (q.t0 <= c.t + VENTANA_CONTRA)]
            f = s.iloc[0] if len(s) else None
            out.append(dict(i=i, salida=f is not None, nace_rec=bool(f is not None and f.nace), contragolpe=bool(f is not None and f.nace and f.n_cg > 0),
                            xg_contra=float(f.xg_rival) if (f is not None and f.nace) else 0.0))
    return CO.join(pd.DataFrame(out).set_index('i'))

def efecto_tecnico(D, min_min=450):
    """D: jugador, etapa, tasa (por 90), minutos. Residuo del jugador frente a su media; efecto de la etapa sin él. Spearman entre ambos."""
    D = D[D.minutos >= min_min].copy()
    D = D[D.groupby('jugador').etapa.transform('nunique') >= 2]
    D['res'] = D.tasa - D.groupby('jugador').tasa.transform('mean')
    s, n = D.groupby('etapa').res.transform('sum'), D.groupby('etapa').res.transform('size')
    D['loo'] = np.where(n > 1, (s - D.res) / (n - 1).clip(lower=1), np.nan)
    D = D.dropna(subset=['loo'])
    from scipy.stats import spearmanr
    return spearmanr(D.res, D.loo).correlation, len(D), D.jugador.nunique()

# --- Prueba (3): un córner con contra a los 3 s, uno con posesión rival a los 25 s (fuera) y uno donde el rival sale pero no contragolpea
R = []
def ev(t, tipo, team, poss, x=50, nace=False, tpos=0.0, pteam=None, xg=np.nan):
    R.append({'match_id': 1, 'period': 1, 't': t, 'type.name': tipo, 'team.name': team, 'possession': poss, 'possession_team.name': pteam or team,
              'loc_x': x, 'nace_rec': nace, 't_pos': tpos, 'shot.statsbomb_xg': xg})
ev(100, 'Pass', 'A', 1, 120); ev(101, 'Clearance', 'B', 1, 10, pteam='A')
ev(103, 'Ball Recovery', 'B', 2, 30, nace=True); ev(108, 'Carry', 'B', 2, 85, nace=True, tpos=5); ev(110, 'Shot', 'B', 2, 100, nace=True, tpos=7, xg=0.12)
ev(200, 'Pass', 'A', 3, 120); ev(225, 'Pass', 'B', 4, 20, nace=True)
ev(300, 'Pass', 'A', 5, 120); ev(304, 'Ball Recovery', 'B', 6, 15, nace=True); ev(306, 'Pass', 'B', 6, 40, nace=True, tpos=2)
CO = pd.DataFrame({'match_id': [1, 1, 1], 'period': [1, 1, 1], 't': [100.0, 200.0, 300.0], 'team': ['A'] * 3, 'rival': ['B'] * 3})
X = contra_tras_corner(pd.DataFrame(R), CO)
print('--- (T3) Contra tras córner:', X[['t', 'salida', 'contragolpe', 'xg_contra']].to_dict('records'))
assert X.contragolpe.tolist() == [True, False, False] and X.salida.tolist() == [True, False, True] and abs(X.xg_contra.iloc[0] - 0.12) < 1e-9
# --- Prueba (2): 60 jugadores en 2-3 etapas; SIN efecto (el técnico no mueve el encare) y CON efecto (cada etapa suma un estilo)
rng0 = np.random.default_rng(SEMILLA)
for nombre, ef in [('SIN efecto', 0.0), ('CON efecto', 1.0)]:
    estilo = {f'T{e}': rng0.normal(0, 1) * ef for e in range(20)}; filas = []
    for j in range(60):
        base = rng0.normal(3, 1)
        for e in rng0.choice(list(estilo), rng0.integers(2, 4), replace=False):
            filas.append(dict(jugador=j, etapa=e, minutos=900, tasa=base + estilo[e] + rng0.normal(0, 0.4)))
    r, n, nj = efecto_tecnico(pd.DataFrame(filas))
    print(f'--- (T2) {nombre}: correlación residuo del jugador vs efecto de la etapa sin él = {r:+.2f} ({n} jugador-etapas, {nj} jugadores)')
print('Esperado: T3 [True, False, False] · T2 SIN ≈ 0 (negativa leve por construcción) · CON claramente positiva.')


# [R8d-1 v1.0] Corrida real
import os, time
import pyarrow.parquet as pq
from scipy.stats import spearmanr
RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
assert 'contra_tras_corner' in globals(), 'Corre primero R8d-0'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5')
SAL5 = os.path.join(RAIZ, 'salidas', 'fw5'); os.makedirs(SAL5, exist_ok=True)
t_ini = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
et = ptp[['match_id', 'team', 'tec_p', 'etapa']].drop_duplicates(['match_id', 'team']).copy(); et['etapa_id'] = et.team + ' · ' + et.etapa.astype(str)
n_et = et.groupby('etapa_id').match_id.nunique(); E15 = set(n_et[n_et >= 15].index)
BP = pd.read_parquet(os.path.join(FW5, 'r8_bp.parquet'))
BP['rival'] = np.where(BP.team == BP.match_id.map(loc['local']), BP.match_id.map(loc['visita']), BP.match_id.map(loc['local']))
BP['temporada'] = BP.match_id.map(loc['temporada'])
co = BP[BP.kind == 'corner'].copy()
am = sorted({e for e in E15 if e.startswith('América')}, key=lambda e: et.loc[et.etapa_id == e, 'match_id'].min())
etiq = lambda e: et.loc[et.etapa_id == e, 'tec_p'].iloc[0]

# ---------- (1) Zona vulnerable ----------
ZON = ['primer_palo', 'punto_penal', 'area_chica', 'segundo_palo']
ev_ = co[co.ruta.isin(['directo', 'via_corto']) & co.zona.isin(ZON)].merge(
    et[['match_id', 'team', 'etapa_id']].rename(columns={'team': 'rival', 'etapa_id': 'etapa_def'}), on=['match_id', 'rival'], how='left')
ev_['y'] = 100 * (ev_.xg1 + ev_.xg2)
liga = ev_.groupby('zona').y.mean()
print('--- (1) xG×100 concedido por córner según la zona del envío (liga):', liga.round(2).to_dict())
d = ev_[ev_.etapa_def.isin(E15)]
tab = d.groupby(['etapa_def', 'zona']).y.agg(['size', 'mean']).reset_index()
tab['exceso'] = tab['mean'] - tab.zona.map(liga)
vul = tab[tab['size'] >= 30].sort_values('exceso', ascending=False).groupby('etapa_def').head(1)
print(f'  Etapas defensoras (15+ partidos): {d.etapa_def.nunique()} · zona vulnerable más frecuente:', vul.zona.value_counts().to_dict())
rng = np.random.default_rng(SEMILLA); mm = d.match_id.unique(); mitad = set(rng.choice(mm, len(mm) // 2, replace=False))
h = d.assign(m=d.match_id.isin(mitad)).groupby(['etapa_def', 'zona', 'm']).y.mean().unstack('m')
h = h.sub(h.index.get_level_values('zona').map(liga).values, axis=0).dropna()
print(f'  Estabilidad del exceso por zona entre mitades de partidos (correlación, etapa × zona): {h.corr().iloc[0, 1]:.2f}'
      ' (cerca de 0 = la "zona vulnerable" es ruido y NO se reporta como identidad)')
for e in am:
    t = tab[tab.etapa_def == e].set_index('zona')[['size', 'mean', 'exceso']].round(2)
    print(f'  {etiq(e)}:', t.to_dict('index'))

# ---------- (2) Penales por jugador ----------
PE = pd.read_parquet(os.path.join(FW5, 'r8_penales.parquet')).merge(et[['match_id', 'team', 'etapa_id', 'tec_p']], on=['match_id', 'team'], how='left')
top = PE.dropna(subset=['jugador']).groupby('jugador').agg(penales=('index', 'size'), encare_pct=('encare', 'mean'), etapas=('etapa_id', 'nunique'))
print(f'\n--- (2) Penales con jugador identificado: {PE.jugador.notna().sum()} de {len(PE)} · jugadores con 3+ penales provocados: {(top.penales >= 3).sum()}'
      f' · de ellos con 2+ técnicos: {((top.penales >= 3) & (top.etapas >= 2)).sum()}')
enc = []
for temp in sorted(Mm.temporada.unique()):
    E = pd.read_parquet(os.path.join(FW1, f"eventos_{temp.replace('/', '-')}.parquet"),
                        columns=['match_id', 'team.name', 'player.id', 'type.name', 'loc_x', 'loc_y', 'carry_end_x', 'carry_end_y', 'period'])
    E = E[E.match_id.isin(loc.index) & E.period.between(1, 4) & E['type.name'].isin(['Dribble', 'Carry'])]
    area = lambda x, y: (x >= 102) & (y >= 18) & (y <= 62)
    E = E[area(E.loc_x, E.loc_y) | ((E['type.name'] == 'Carry') & area(E.carry_end_x, E.carry_end_y))]
    enc.append(E.groupby(['match_id', 'team.name', 'player.id']).size().rename('encares').reset_index())
ENC = pd.concat(enc).rename(columns={'team.name': 'team', 'player.id': 'player_id'})
JP = pd.read_parquet(os.path.join(FW5, 'r7_jugador_partido.parquet'), columns=['match_id', 'team', 'player_id', 'minutos', 'etapa_id', 'tec_p'])
JP = JP[JP.minutos > 0].merge(ENC, on=['match_id', 'team', 'player_id'], how='left').fillna({'encares': 0})
PJ = JP.groupby(['player_id', 'etapa_id']).agg(minutos=('minutos', 'sum'), encares=('encares', 'sum'), tec=('tec_p', 'first')).reset_index()
PJ['tasa'] = 90 * PJ.encares / PJ.minutos
pen_pe = PE.dropna(subset=['jugador']).groupby(['jugador', 'etapa_id']).size().rename('penales')
PJ = PJ.join(pen_pe, on=['player_id', 'etapa_id']).fillna({'penales': 0}); PJ['pen90'] = 90 * PJ.penales / PJ.minutos
r_, n_, nj_ = efecto_tecnico(PJ.rename(columns={'player_id': 'jugador', 'etapa_id': 'etapa'}))
print(f'  ¿El técnico mueve el encare en el área de sus jugadores? correlación residuo-efecto de la etapa = {r_:+.2f} ({n_} jugador-etapas con 450+ min, {nj_} jugadores con 2+ técnicos)')
f_ = PJ[PJ.minutos >= 450]
print(f'  Entre jugador-etapas: encares en el área por 90 frente a penales provocados por 90, Spearman {spearmanr(f_.tasa, f_.pen90).correlation:+.2f}')
tp = top[(top.penales >= 3) & (top.etapas >= 2)].sort_values('penales', ascending=False).head(8).index
print('  Mayores provocadores con 2+ técnicos (por etapa: minutos, encares por 90, penales):')
for j in tp:
    g = PJ[(PJ.player_id == j) & (PJ.minutos >= 270)].sort_values('etapa_id')
    print(f'   jugador {int(j)}:', [(r.tec.split()[-1] if isinstance(r.tec, str) else '?', int(r.minutos), round(r.tasa, 2), int(r.penales)) for r in g.itertuples()])

# ---------- (3) Contra tras córner ----------
cs = []
for temp, g in co.groupby('temporada'):
    E = pd.read_parquet(os.path.join(FW1, f"eventos_{temp.replace('/', '-')}.parquet"),
                        columns=['match_id', 'period', 't', 'type.name', 'team.name', 'possession', 'possession_team.name', 'nace_rec', 't_pos', 'loc_x', 'shot.statsbomb_xg'])
    E = E[E.match_id.isin(set(g.match_id)) & E.period.isin([1, 2])]
    cs.append(contra_tras_corner(E, g[g.period.isin([1, 2])][['match_id', 'period', 't', 'team', 'rival', 'index', 'ruta', 's_atac_area', 'xg1', 'xg2']].copy()))
C = pd.concat(cs, ignore_index=True)
pc = lambda s: round(100 * float(np.mean(s)), 1)
print(f'\n--- (3) Córners (periodos 1-2): {len(C):,} · el rival sale con posesión ≤ 20 s: {pc(C.salida)}% · nace de recuperación: {pc(C.nace_rec)}%'
      f' · CONTRAGOLPE tras córner: {pc(C.contragolpe)}% · xG×100 del rival por córner en esas contras: {100 * C.xg_contra.mean():.3f}'
      f' (frente a {100 * (C.xg1 + C.xg2).mean():.2f} que genera el córner)')
s = C[C.s_atac_area.notna()].copy(); s['comprometidos'] = pd.cut(s.s_atac_area, [-1, 3, 4, 5, 6, 20], labels=['≤3', '4', '5', '6', '≥7'])
print('  Por atacantes comprometidos en el área (360): % de contragolpe rival y xG×100 a favor del córner:')
print(s.groupby('comprometidos', observed=True).agg(corners=('contragolpe', 'size'), contragolpe_pct=('contragolpe', lambda v: round(100 * v.mean(), 1)),
                                                    xg_corner_x100=('xg1', lambda v: round(100 * (v + s.loc[v.index, 'xg2']).mean(), 2)),
                                                    xg_contra_x100=('xg_contra', lambda v: round(100 * v.mean(), 3))).to_string())
C = C.merge(et[['match_id', 'team', 'etapa_id']], on=['match_id', 'team'], how='left')
dd = C[C.etapa_id.isin(E15)]; mm = dd.match_id.unique(); mitad = set(rng.choice(mm, len(mm) // 2, replace=False))
hh = dd.assign(m=dd.match_id.isin(mitad)).groupby(['etapa_id', 'm']).contragolpe.mean().unstack()
print(f'  Estabilidad por técnico (contra concedida tras su córner, mitades de partidos): {hh.corr().iloc[0, 1]:.2f}')
for e in am: print(f'  {etiq(e)}: contragolpe concedido tras su córner {pc(C[C.etapa_id == e].contragolpe)}% de {int((C.etapa_id == e).sum())} córners')
C.to_csv(os.path.join(SAL5, 'r8d_contra_tras_corner.csv'), index=False); tab.to_csv(os.path.join(SAL5, 'r8d_zona_vulnerable.csv'), index=False)
PJ.to_csv(os.path.join(SAL5, 'r8d_encare_jugador_etapa.csv'), index=False)
print(f'\nR8d-1 terminó en {time.time() - t_ini:.0f} s. Guardado: salidas/fw5/r8d_*.csv. Pega toda la salida.')

R8d-2

In [ ]:

# [R8d-2 v1.0] Contra tras córner con recuperación por "primer evento que no es reanudación" (opción A de Arturo, sesión 11)
# Motivo: O-6 usa el play_pattern de StatsBomb; tras un córner las posesiones siguientes siguen como "From Corner", así que la regla
# no veía ninguna contra (R8d-1: 1% de salidas "nacen de recuperación"). Para la ventana tras balón parado (O-11 / BP-6):
# recuperación en juego abierto = primer evento de la posesión del rival NO es reanudación (O-2: es_reanudacion de R1).
# Contragolpe = O-8 sin cambio: evento propio (Ball Receipt*, Carry, Duel, Shot) en x ≥ 80 en ≤ 10 s desde el inicio de la posesión.
import os, time
import numpy as np
import pandas as pd
def contra_v2(E, CO, ventana=20.0):
    E = E.sort_values(['match_id', 'index'])
    pos = E.groupby(['match_id', 'possession']).agg(period=('period', 'first'), t0=('t', 'min'), pteam=('possession_team.name', 'first'),
                                                     reanud=('es_reanudacion', lambda s: bool(s.fillna(False).astype(bool).iloc[0])),
                                                     tipo0=('type.name', 'first')).reset_index()
    E = E.merge(pos[['match_id', 'possession', 't0']], on=['match_id', 'possession'])
    cg = E[(E['team.name'] == E['possession_team.name']) & ((E['t'] - E['t0']).between(0, 10)) & (E['loc_x'] >= 80)
           & E['type.name'].isin({'Ball Receipt*', 'Carry', 'Duel', 'Shot'})].groupby(['match_id', 'possession']).size().rename('n_cg')
    xg = E[(E['type.name'] == 'Shot') & (E['team.name'] == E['possession_team.name'])].groupby(['match_id', 'possession'])['shot.statsbomb_xg'].sum().rename('xg_rival')
    pos = pos.join(cg, on=['match_id', 'possession']).join(xg, on=['match_id', 'possession']).fillna({'n_cg': 0, 'xg_rival': 0})
    out = []
    for (m, p, r), g in CO.groupby(['match_id', 'period', 'rival']):
        q = pos[(pos.match_id == m) & (pos.period == p) & (pos.pteam == r)].sort_values('t0')
        for i, c in g.iterrows():
            s = q[(q.t0 > c.t) & (q.t0 <= c.t + ventana)]; f = s.iloc[0] if len(s) else None
            rec = bool(f is not None and not f.reanud)
            out.append(dict(i=i, salida=f is not None, recupera=rec, tipo0=f.tipo0 if f is not None else None,
                            contragolpe=bool(rec and f.n_cg > 0), xg_contra=float(f.xg_rival) if rec else 0.0))
    return CO.join(pd.DataFrame(out).set_index('i'))
# Prueba: recuperación abierta con contra · saque de meta (reanudación) con carrera al último tercio (NO cuenta) · recuperación sin contra
R2 = []
def e2(i, t, tipo, team, poss, x=50, reanud=False, xg=np.nan, pteam=None):
    R2.append({'match_id': 1, 'index': i, 'period': 1, 't': t, 'type.name': tipo, 'team.name': team, 'possession': poss,
               'possession_team.name': pteam or team, 'loc_x': x, 'es_reanudacion': reanud, 'shot.statsbomb_xg': xg})
e2(1, 100, 'Pass', 'A', 1, 120, reanud=True); e2(2, 101, 'Clearance', 'B', 1, 10, pteam='A')
e2(3, 103, 'Ball Recovery', 'B', 2, 30); e2(4, 108, 'Carry', 'B', 2, 85); e2(5, 110, 'Shot', 'B', 2, 100, xg=0.12)
e2(6, 200, 'Pass', 'A', 3, 120, reanud=True); e2(7, 205, 'Pass', 'B', 4, 6, reanud=True); e2(8, 209, 'Carry', 'B', 4, 85)
e2(9, 300, 'Pass', 'A', 5, 120, reanud=True); e2(10, 304, 'Ball Recovery', 'B', 6, 15); e2(11, 306, 'Pass', 'B', 6, 40)
CO2 = pd.DataFrame({'match_id': [1] * 3, 'period': [1] * 3, 't': [100.0, 200.0, 300.0], 'team': ['A'] * 3, 'rival': ['B'] * 3})
X2 = contra_v2(pd.DataFrame(R2), CO2)
print('--- (T) contragolpe', X2.contragolpe.tolist(), '· recupera', X2.recupera.tolist(), '(esperado [True, False, False] y [True, False, True])')
assert X2.contragolpe.tolist() == [True, False, False] and X2.recupera.tolist() == [True, False, True]

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
t_ini = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
BP = pd.read_parquet(os.path.join(FW5, 'r8_bp.parquet'))
BP['rival'] = np.where(BP.team == BP.match_id.map(loc['local']), BP.match_id.map(loc['visita']), BP.match_id.map(loc['local']))
BP['temporada'] = BP.match_id.map(loc['temporada'])
co = BP[(BP.kind == 'corner') & BP.period.isin([1, 2])]
cs = []
for temp, g in co.groupby('temporada'):
    E = pd.read_parquet(os.path.join(FW1, f"eventos_{temp.replace('/', '-')}.parquet"),
                        columns=['match_id', 'index', 'period', 't', 'type.name', 'team.name', 'possession', 'possession_team.name', 'es_reanudacion', 'loc_x', 'shot.statsbomb_xg'])
    E = E[E.match_id.isin(set(g.match_id)) & E.period.isin([1, 2])]
    cs.append(contra_v2(E, g[['match_id', 'period', 't', 'team', 'rival', 'index', 's_atac_area', 'xg1', 'xg2']].copy()))
C = pd.concat(cs, ignore_index=True)
pc = lambda s: round(100 * float(np.mean(s)), 1)
print(f'--- (3 v2) Córners {len(C):,} · salida rival ≤ 20 s {pc(C.salida)}% · de ellas recuperación en juego abierto {pc(C.loc[C.salida, "recupera"])}%'
      f' · CONTRAGOLPE tras córner {pc(C.contragolpe)}% · xG×100 del rival en esas contras por córner {100 * C.xg_contra.mean():.3f}'
      f' (el córner genera {100 * (C.xg1 + C.xg2).mean():.2f})')
print('  Primer evento de la posesión de salida del rival (%):', C.loc[C.salida, 'tipo0'].value_counts(normalize=True).mul(100).round(1).head(6).to_dict())
s = C[C.s_atac_area.notna()].copy(); s['comprometidos'] = pd.cut(s.s_atac_area, [-1, 3, 4, 5, 6, 20], labels=['≤3', '4', '5', '6', '≥7'])
print('  Por atacantes comprometidos en el área (360):')
print(s.groupby('comprometidos', observed=True).apply(lambda d: pd.Series({'corners': len(d), 'contragolpe_pct': round(100 * d.contragolpe.mean(), 1),
      'xg_corner_x100': round(100 * (d.xg1 + d.xg2).mean(), 2), 'xg_contra_x100': round(100 * d.xg_contra.mean(), 3)}), include_groups=False).to_string())
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
et = ptp[['match_id', 'team', 'tec_p', 'etapa']].drop_duplicates(['match_id', 'team']).copy(); et['etapa_id'] = et.team + ' · ' + et.etapa.astype(str)
n_et = et.groupby('etapa_id').match_id.nunique(); E15 = set(n_et[n_et >= 15].index)
C = C.merge(et[['match_id', 'team', 'etapa_id', 'tec_p']], on=['match_id', 'team'], how='left')
dd = C[C.etapa_id.isin(E15)]; rng = np.random.default_rng(2026); mm = dd.match_id.unique(); mitad = set(rng.choice(mm, len(mm) // 2, replace=False))
hh = dd.assign(m=dd.match_id.isin(mitad)).groupby(['etapa_id', 'm']).contragolpe.mean().unstack()
print(f'  Estabilidad por técnico (contra concedida tras su córner, mitades): {hh.corr().iloc[0, 1]:.2f}')
for e in sorted({e for e in E15 if e.startswith('América')}, key=lambda e: et.loc[et.etapa_id == e, 'match_id'].min()):
    print(f'  {et.loc[et.etapa_id == e, "tec_p"].iloc[0]}: contragolpe concedido tras su córner {pc(C[C.etapa_id == e].contragolpe)}% de {int((C.etapa_id == e).sum())}')
C.to_csv(os.path.join(SAL5, 'r8d_contra_tras_corner_v2.csv'), index=False)
print(f'R8d-2 terminó en {time.time() - t_ini:.0f} s. Guardado: salidas/fw5/r8d_contra_tras_corner_v2.csv')


# Sesión 11

C00-Inspección

In [ ]:
import os, json
import numpy as np
import pandas as pd

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5 = os.path.join(DP, 'fw5'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
PREFIJOS = ['r4', 'r6', 'r6b', 'r6c', 'r7u', 'r7b', 'r7c', 'r8']
ETAPA_MIN, ETAPA_SENS = 15, 10
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 120)

with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
    TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
ULT = TEMPS[-1]
print('Temporadas de fw5:', TEMPS)

# ---------- Etapas (misma tabla y mismo identificador que R4b, R5 y R6r: etapa_id = equipo · etapa) ----------
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy(); ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ET = ptp.groupby('etapa_id').agg(team=('team', 'first'), tecnico=('tec_p', 'first'), partidos=('match_id', 'nunique'),
                                 ini=('fecha', 'min'), fin=('fecha', 'max'), ult_temp=('temporada', 'max'),
                                 n_tec=('tec_p', 'nunique')).reset_index()
print(f'\n--- (2) Etapas: {len(ET)} · con {ETAPA_MIN}+ partidos: {(ET.partidos >= ETAPA_MIN).sum()} · '
      f'con {ETAPA_SENS}+: {(ET.partidos >= ETAPA_SENS).sum()} · que llegan a {ULT}: {(ET.ult_temp == ULT).sum()}')
print('Chequeo: etapas con más de un técnico (debe ser 0):', int((ET.n_tec > 1).sum()),
      '· filas equipo-partido repetidas (debe ser 0):', int(ptp.duplicated(['match_id', 'team']).sum()))

# ---------- (1) Archivos por prefijo ----------
def rutas(pre):
    c = 'r5' if pre == 'r4' else pre
    return {'perfiles_etapa': os.path.join(FW5, f'{pre}_perfiles_etapa.parquet'),
            'contraidos': os.path.join(FW5, f'{c}_perfiles_contraidos.parquet'),
            'confiabilidad': os.path.join(SAL5, f'{c}_confiabilidad.csv'),
            'implantacion': os.path.join(SAL5, f'{c}_implantacion.csv')}
filas, RASGOS = [], {}
ult_set = set(ET.loc[ET.ult_temp == ULT, 'etapa_id'])
for pre in PREFIJOS:
    r = rutas(pre); fila = {'prefijo': pre, **{k: ('ok' if os.path.exists(v) else 'FALTA') for k, v in r.items()}}
    if os.path.exists(r['perfiles_etapa']):
        pe = pd.read_parquet(r['perfiles_etapa'])
        RASGOS[pre] = sorted(pe['metrica'].unique())
        fila.update(rasgos=pe['metrica'].nunique(), etapas=pe['etapa_id'].nunique(),
                    etapas_ultima_temp=len(set(pe['etapa_id']) & ult_set),
                    etapas_sin_tabla=len(set(pe['etapa_id']) - set(ET.etapa_id)))
    if os.path.exists(r['confiabilidad']):
        cf = pd.read_csv(r['confiabilidad'])
        fila.update(nucleo=int((cf.conf >= 0.7).sum()), secundario=int(((cf.conf >= 0.5) & (cf.conf < 0.7)).sum()),
                    fuera=int((cf.conf < 0.5).sum()))
    if os.path.exists(r['implantacion']):
        im = pd.read_csv(r['implantacion'])
        rc = im.groupby('metrica').apply(lambda g: ((g.p_valor < 0.05).sum() >= 2) and (g.cociente > 1).all())
        fila.update(regla_C=int(rc.sum()))
    filas.append(fila)
F = pd.DataFrame(filas)
print('\n--- (1) Insumos por prefijo (fw5 y salidas/fw5) ---')
print('  etapas_ultima_temp > 0 = el perfil ya incluye la quinta temporada · etapas_sin_tabla debe ser 0')
print(F.to_string(index=False))
falta = F.loc[(F.contraidos == 'FALTA') | (F.confiabilidad == 'FALTA') | (F.implantacion == 'FALTA'), 'prefijo'].tolist()
print('Prefijos a los que les falta R5/R6r-2 con cinco temporadas:', falta or 'ninguno')

# ---------- (3) Viajeros ----------
def viajeros(minimo):
    e = ET[ET.partidos >= minimo]
    g = e.groupby('tecnico').filter(lambda d: d.team.nunique() >= 2)
    pares = sum(len(d) * (len(d) - 1) // 2 - sum(k * (k - 1) // 2 for k in d.team.value_counts()) for _, d in g.groupby('tecnico'))
    return g.sort_values(['tecnico', 'ini']), g.tecnico.nunique(), pares
V15, n15, p15 = viajeros(ETAPA_MIN); V10, n10, p10 = viajeros(ETAPA_SENS)
print(f'\n--- (3) Viajeros: técnicos con etapas en 2+ clubes · {ETAPA_MIN}+ partidos: {n15} técnicos, {p15} parejas entre clubes '
      f'· {ETAPA_SENS}+ partidos: {n10} técnicos, {p10} parejas')
print(V15.assign(etapa=lambda d: d.team + ' (' + d.partidos.astype(str) + ')')
      .groupby('tecnico')['etapa'].apply(' → '.join).to_string())

# ---------- (4) Clubes que cambian de técnico (etapas consecutivas del mismo club) ----------
def relevos_club(minimo):
    e = ET[ET.partidos >= minimo].sort_values(['team', 'ini']).copy()      # los interinos cortos se saltan
    e['tec_sig'] = e.groupby('team').tecnico.shift(-1)
    return e[e.tec_sig.notna() & (e.tecnico != e.tec_sig)]
C15, C10 = relevos_club(ETAPA_MIN), relevos_club(ETAPA_SENS)
print(f'\n--- (4) Clubes con relevo (dos etapas consecutivas de técnicos distintos) · {ETAPA_MIN}+ ambas: {len(C15)} parejas en '
      f'{C15.team.nunique()} clubes · {ETAPA_SENS}+ ambas: {len(C10)} parejas')
print('Nota: un interino corto entre dos etapas largas se salta (la pareja cuenta como relevo del club).')

# ---------- (5) Relevos con previa: el técnico que llega ya tuvo una etapa de 15+ en otro club ----------
e15 = ET[ET.partidos >= ETAPA_MIN]
previa = []
for r_ in ET[ET.partidos >= ETAPA_SENS].itertuples(index=False):
    ant = e15[(e15.tecnico == r_.tecnico) & (e15.team != r_.team) & (e15.fin < r_.ini)]
    if len(ant): previa.append((r_.etapa_id, r_.tecnico, r_.partidos, ant.sort_values('fin').iloc[-1].etapa_id))
PV = pd.DataFrame(previa, columns=['etapa_nueva', 'tecnico', 'partidos', 'etapa_previa'])
print(f'\n--- (5) Llegadas con etapa previa de {ETAPA_MIN}+ en otro club (la nueva con {ETAPA_SENS}+): {len(PV)}')
print(PV.to_string(index=False))

# ---------- (6) Rasgos por prefijo y América ----------
print('\n--- (6) Rasgos por prefijo (nombres internos) ---')
for pre, l in RASGOS.items(): print(f'{pre} ({len(l)}):', ', '.join(l))
print('\nArquetipos de R7c en salidas/fw5:', 'ok' if os.path.exists(os.path.join(SAL5, 'r7c_arquetipos.csv')) else 'FALTA')
print('\nEtapas de América:')
print(ET[ET.team.str.startswith('Am')].sort_values('ini')[['etapa_id', 'tecnico', 'partidos', 'ini', 'fin']].to_string(index=False))
print('\nC-00 terminó (no escribió nada).')


Cr0

In [ ]:
#Cr0 No escribe en Drive (solo crea enlaces en el disco temporal de Colab). Celda 1 primero.
# =====================================================================
import os, json, shutil
PREFIJO = 'r4'                                    # ← cambiar: 'r6', 'r6b', 'r6c', 'r7b', 'r4'
assert PREFIJO in ['r6', 'r6b', 'r6c', 'r7b', 'r4'], 'Prefijo no previsto en C-r0'
DRIVE = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
LOCAL = f'/content/isac_c_{PREFIJO}/' if 'RAIZ_PRUEBA' not in globals() else os.path.join(DRIVE, f'_local_{PREFIJO}/')
D_DP, D_FW1, D_FW5 = os.path.join(DRIVE, 'datos_parquet'), os.path.join(DRIVE, 'datos_parquet', 'fw1'), os.path.join(DRIVE, 'datos_parquet', 'fw5')
D_SAL5 = os.path.join(DRIVE, 'salidas', 'fw5')
L_DP, L_FW, L_SAL = os.path.join(LOCAL, 'datos_parquet'), os.path.join(LOCAL, 'datos_parquet', 'fw1'), os.path.join(LOCAL, 'salidas')
shutil.rmtree(LOCAL, ignore_errors=True); os.makedirs(L_FW, exist_ok=True); os.makedirs(L_SAL, exist_ok=True)
def enlazar(src, dst):
    if os.path.exists(src): os.symlink(os.path.abspath(src), dst); return True
    return False
enl = {'partido_tecnico_periodo.parquet': enlazar(os.path.join(D_DP, 'partido_tecnico_periodo.parquet'), os.path.join(L_DP, 'partido_tecnico_periodo.parquet'))}
comunes = ['r4_partidos.parquet', 'r4_minutos_v21.parquet', 'parametros_fw1.json', 'r4_perfiles_etapa.parquet',
           'r6_perfiles_etapa.parquet', 'r6b_perfiles_etapa.parquet', 'r6c_perfiles_etapa.parquet']
if PREFIJO == 'r4':
    propios = ['r4_residuos.parquet']
    enl['r4_coeficientes.csv (salidas/fw5)'] = enlazar(os.path.join(D_SAL5, 'r4_coeficientes.csv'), os.path.join(L_SAL, 'r4_coeficientes.csv'))
else:
    propios = [f'{PREFIJO}_tramos.parquet']
for f in comunes + propios:
    if f.startswith(PREFIJO + '_') and f.endswith('_perfiles_etapa.parquet') and PREFIJO != 'r4':
        continue                                  # R6r-1 rehace el perfil propio; no se enlaza para no pisar el de Drive
    enl[f] = enlazar(os.path.join(D_FW5, f), os.path.join(L_FW, f))
enl['elo (fw1, solo lectura)'] = enlazar(os.path.join(D_FW1, 'elo_equipo_partido_e2.parquet'), os.path.join(L_FW, 'elo_equipo_partido_e2.parquet'))
print('--- (C-r0) Carpeta espejo', LOCAL, '· PREFIJO =', PREFIJO)
print({k: ('ok' if v else 'FALTA') for k, v in enl.items()})
oblig = ['partido_tecnico_periodo.parquet', 'r4_partidos.parquet', 'r4_minutos_v21.parquet', 'parametros_fw1.json',
         'elo (fw1, solo lectura)'] + propios + (['r4_perfiles_etapa.parquet', 'r4_coeficientes.csv (salidas/fw5)'] if PREFIJO == 'r4' else [])
falt = [k for k in oblig if not enl.get(k)]
assert not falt, f'Falta un insumo obligatorio {falt}: avisar a Claude'
with open(os.path.join(L_FW, 'parametros_fw1.json')) as f:
    print('Temporadas de ajuste:', json.load(f)['temporadas']['entrenamiento'])
BASE = LOCAL
print('BASE listo →', 'corre R5 sin cambios; luego C-r9.' if PREFIJO == 'r4' else 'corre R6r-1 v1.1 y R6r-2 v1.1 sin cambios; luego C-r9.')

C-r9

In [ ]:
#[C-r9 v1.0] Copiar a Drive lo que generó el prefijo, con chequeo de coherencia contra la versión previa de fw5
import os, shutil
import numpy as np
import pandas as pd
DRIVE = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1 (freno de Drive)'
assert 'PREFIJO' in globals() and 'LOCAL' in globals(), 'Corre primero C-r0 (y R6r o R5) en esta sesión'
L_FW, L_SAL = os.path.join(LOCAL, 'datos_parquet', 'fw1'), os.path.join(LOCAL, 'salidas')
D_FW5, D_SAL5 = os.path.join(DRIVE, 'datos_parquet', 'fw5'), os.path.join(DRIVE, 'salidas', 'fw5'); os.makedirs(D_SAL5, exist_ok=True)
pref_sal = 'r5' if PREFIJO == 'r4' else PREFIJO
# 1. Coherencia (buena práctica 13): el perfil por etapa rehecho debe coincidir con el que ya estaba en fw5
if PREFIJO != 'r4':
    nuevo, viejo = os.path.join(L_FW, f'{PREFIJO}_perfiles_etapa.parquet'), os.path.join(D_FW5, f'{PREFIJO}_perfiles_etapa.parquet')
    assert os.path.exists(nuevo), 'No hay perfil nuevo: ¿corriste R6r-1?'
    if os.path.exists(viejo):
        a = pd.read_parquet(viejo)[['etapa_id', 'metrica', 'perfil']]; b = pd.read_parquet(nuevo)[['etapa_id', 'metrica', 'perfil']]
        j = a.merge(b, on=['etapa_id', 'metrica'], how='outer', suffixes=('_fw5', '_nuevo'), indicator=True)
        dif = (j.perfil_fw5 - j.perfil_nuevo).abs()
        print(f'Coherencia {PREFIJO}_perfiles_etapa: filas solo en uno {int((j._merge != "both").sum())} (debe ser 0) · '
              f'dif. máxima {np.nanmax(dif):.2e} (debe ser ≈ 0)')
        assert (j._merge == 'both').all() and np.nanmax(dif) < 1e-6, 'El perfil rehecho NO coincide con fw5: no se copia nada; avisar a Claude'
# 2. Copia (solo archivos generados aquí; los enlaces no se copian)
n = []
for src_dir, dst_dir in [(L_FW, D_FW5), (L_SAL, D_SAL5)]:
    for f in os.listdir(src_dir):
        s = os.path.join(src_dir, f)
        if (f.startswith(PREFIJO + '_') or f.startswith(pref_sal + '_')) and os.path.isfile(s) and not os.path.islink(s):
            shutil.copy2(s, os.path.join(dst_dir, f)); n.append(f)
print(f'--- (C-r9) {PREFIJO}: copiados a Drive:', sorted(n))
need = [os.path.join(D_FW5, f'{pref_sal}_perfiles_contraidos.parquet'), os.path.join(D_SAL5, f'{pref_sal}_confiabilidad.csv'),
        os.path.join(D_SAL5, f'{pref_sal}_implantacion.csv')]
print('Insumos de la compatibilidad para', PREFIJO, ':', {os.path.basename(p): ('ok' if os.path.exists(p) else 'FALTA') for p in need})
BASE = DRIVE; print('BASE =', BASE)


C0

In [ ]:
#C0 - Prueba compatibilidad
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

SEMILLA = 2026
CORTE = 0.5          # PROVISIONAL: frontera alto/bajo de cada eje en el mapa; se fija al ver el mapa real (como los umbrales de T-11)
PESO_MAX = 1.0       # el peso de un rasgo en la compatibilidad es el eje técnico recortado a [0, 1]

def corr_cols(X, Y):
    """Pearson por columna, ignorando nulos. X, Y: (filas, rasgos)."""
    m = np.isfinite(X) & np.isfinite(Y); n = m.sum(0).astype(float)
    with np.errstate(invalid='ignore', divide='ignore'):
        mx = np.where(m, X, 0).sum(0) / n; my = np.where(m, Y, 0).sum(0) / n
        dx = np.where(m, X - mx, 0.0); dy = np.where(m, Y - my, 0.0)
        r = (dx * dy).sum(0) / np.sqrt((dx ** 2).sum(0) * (dy ** 2).sum(0))
    r = np.asarray(r, float); r[n < 5] = np.nan
    return r

def r_pares(Z, pares):
    """Correlación de cada rasgo entre las dos etapas de cada pareja (apila a→b y b→a para no depender del orden)."""
    pares = np.asarray(pares, int).reshape(-1, 2)
    if len(pares) == 0: return np.full(Z.shape[1], np.nan)
    a, b = Z[pares[:, 0]], Z[pares[:, 1]]
    return corr_cols(np.vstack([a, b]), np.vstack([b, a]))

def _boot(Z, pares, cl, B, rng):
    u = np.unique(cl); idx = {c: np.where(cl == c)[0] for c in u}; out = np.empty((B, Z.shape[1]))
    for i in range(B):
        out[i] = r_pares(Z, pares[np.concatenate([idx[c] for c in rng.choice(u, len(u))])])
    return out

def ejes(Z, conf, pares_t, cl_t, pares_c, cl_c, pool, B=1000, P=200, seed=SEMILLA):
    """Eje técnico (parejas de viajeros, remuestreo por técnico) y eje club (relevos, remuestreo por club).
    Ambos divididos entre la confiabilidad del rasgo (atenuación, buena práctica 17). Placebo: parejas al azar con otro
    técnico y otro club, del mismo tamaño que las de viajeros (debe dar ~0)."""
    rng = np.random.default_rng(seed)
    pares_t, pares_c, pool = (np.asarray(x, int).reshape(-1, 2) for x in (pares_t, pares_c, pool))
    rt, rc = r_pares(Z, pares_t), r_pares(Z, pares_c)
    bt, bc = _boot(Z, pares_t, np.asarray(cl_t), B, rng), _boot(Z, pares_c, np.asarray(cl_c), B, rng)
    pl = np.array([r_pares(Z, pool[rng.choice(len(pool), len(pares_t), replace=False)]) for _ in range(P)])
    c = np.clip(np.asarray(conf, float), 0.05, None)
    q = lambda M, p: np.nanpercentile(M, p, axis=0) / c
    return pd.DataFrame({'conf': conf, 'r_tec_crudo': rt, 'r_club_crudo': rc,
                         'tec': rt / c, 'tec_lo': q(bt, 2.5), 'tec_hi': q(bt, 97.5),
                         'club': rc / c, 'club_lo': q(bc, 2.5), 'club_hi': q(bc, 97.5),
                         'dif': (rt - rc) / c, 'dif_lo': q(bt - bc, 2.5), 'dif_hi': q(bt - bc, 97.5),
                         'placebo_media': np.nanmean(pl, 0) / c, 'placebo_de': np.nanstd(pl, 0) / c})

def clasificar(T, corte=CORTE):
    ta, ca = T['tec'] >= corte, T['club'] >= corte
    cuad = np.select([ta & ~ca, ~ta & ca, ta & ca], ['del técnico', 'del club', 'de los dos'], 'de nadie')
    firme = np.where(cuad == 'del técnico', T['dif_lo'] > 0, np.where(cuad == 'del club', T['dif_hi'] < 0, False))
    return pd.Series(cuad, index=T.index), pd.Series(np.where(cuad.astype(str) == 'de los dos', '—',
                                                    np.where(cuad.astype(str) == 'de nadie', '—', np.where(firme, 'firme', 'tendencia'))), index=T.index)

def perfil_ventana(e, den, res, s2, tau_e, W3v):
    """Perfil ajustado (sin contraer) de un conjunto de partidos: COPIA LITERAL de perfil_sub (R6r-2 v1.1 / R5):
    residuo por estado de marcador, contracción hacia el propio promedio con tau del estado, mezcla de estados de la liga."""
    e3 = np.sign(np.asarray(e)).astype(int); w = np.asarray(den, float); we = w * np.asarray(res, float)
    WE = np.array([we[e3 == s].sum() for s in (-1, 0, 1)]); W = np.array([w[e3 == s].sum() for s in (-1, 0, 1)])
    if W.sum() <= 0: return np.nan
    tau2 = tau_e ** 2
    with np.errstate(divide='ignore', invalid='ignore'):
        rbar = WE.sum() / W.sum()
        k = tau2 / (tau2 + s2 / W) if tau2 > 0 else np.zeros(3)
        rstar = np.where(W > 0, rbar + k * (WE / W - rbar), rbar)
    return float(rstar @ W3v)

def contraer(perfil, n, tau, sigma):
    """P-5: k = τ² / (τ² + σ²/n), con τ y σ de la confiabilidad del rasgo y n = partidos de la ventana con dato."""
    if not np.isfinite(perfil) or n <= 0 or tau <= 0: return np.nan
    return tau ** 2 / (tau ** 2 + sigma ** 2 / n) * perfil

def distancia(zt, zc, w):
    m = np.isfinite(zt) & np.isfinite(zc) & (w > 0)
    return np.sqrt((w[m] * (zt[m] - zc[m]) ** 2).sum() / w[m].sum()) if m.sum() >= 3 else np.nan

def percentil(d, d_pool):
    """Compatibilidad 0-100%: % de candidatos MÁS lejanos al club que este técnico (empates cuentan la mitad)."""
    d_pool = np.asarray(d_pool, float); d_pool = d_pool[np.isfinite(d_pool)]
    return 100 * ((d_pool > d).mean() + 0.5 * (d_pool == d).mean()) if len(d_pool) and np.isfinite(d) else np.nan

def pesos(T):
    return np.clip(T['tec'].fillna(0).values, 0, PESO_MAX)

def choques(zt, zc, w, nombres, top=3):
    m = np.isfinite(zt) & np.isfinite(zc) & (w > 0)
    aporte = np.where(m, w * (zt - zc) ** 2, 0)
    o = np.argsort(-aporte)[:top]
    return '; '.join(f"{nombres[i]} ({'técnico más alto' if zt[i] > zc[i] else 'técnico más bajo'})" for i in o if aporte[i] > 0)

# =====================================================================
# PRUEBAS
# =====================================================================
if __name__ == '__main__' or True:
    rng0 = np.random.default_rng(SEMILLA); ok_total = []
    def chk(nombre, cond, detalle):
        ok_total.append(bool(cond)); print(f"  [{'PASA' if cond else 'FALLA'}] {nombre}: {detalle}")

    # ---------- (1) perfil_ventana frente a perfil_sub de R6r-2, copiado SIN CAMBIOS ----------
    print('--- (1) perfil_ventana = perfil_sub de R6r-2 (mismos partidos, mismos parámetros) ---')
    COLS = ['we_m', 'we_0', 'we_p', 'w_m', 'w_0', 'w_p', 'num', 'den']
    def perfil_sub(A, cod, mask, tau2, s2, NS, W3v):              # ← copia literal de R6r-2 v1.1 (NS y W3v como argumentos)
        c = cod[mask]
        S = {k: np.bincount(c, weights=A[k][mask], minlength=NS) for k in COLS}
        we = np.column_stack([S['we_m'], S['we_0'], S['we_p']]); w = np.column_stack([S['w_m'], S['w_0'], S['w_p']])
        with np.errstate(divide='ignore', invalid='ignore'):
            rbar = we.sum(1) / w.sum(1)
            k = tau2 / (tau2 + s2 / w) if tau2 > 0 else np.zeros_like(w)
            rstar = np.where(w > 0, rbar[:, None] + k * (we / w - rbar[:, None]), rbar[:, None])
            return rstar @ W3v, S['num'] / S['den']
    W3v_ = np.array([0.272, 0.456, 0.272]); n_part = 30
    filas = pd.DataFrame({'p': np.repeat(np.arange(n_part), 4), 'e': rng0.choice([-2, -1, 0, 1, 2], 4 * n_part),
                          'den': rng0.uniform(1, 10, 4 * n_part), 'res': rng0.normal(0.3, 2, 4 * n_part)})
    s2_, taue_ = 4.0, 0.5
    g = filas.assign(e3=np.sign(filas.e).astype(int), we=filas.den * filas.res).groupby(['p', 'e3'])[['we', 'den']].sum().unstack('e3', fill_value=0)
    A = {'we_m': g[('we', -1)].values, 'we_0': g[('we', 0)].values, 'we_p': g[('we', 1)].values,
         'w_m': g[('den', -1)].values, 'w_0': g[('den', 0)].values, 'w_p': g[('den', 1)].values,
         'num': np.ones(n_part), 'den': np.ones(n_part)}
    ref, _ = perfil_sub(A, np.zeros(n_part, int), np.ones(n_part, bool), taue_ ** 2, s2_, 1, W3v_)
    mio = perfil_ventana(filas.e, filas.den, filas.res, s2_, taue_, W3v_)
    chk('reproduce perfil_sub', abs(ref[0] - mio) < 1e-10, f'R6r-2 {ref[0]:.6f} · C-0 {mio:.6f}')

    # ---------- liga inventada ----------
    def liga(efecto, seed, n_club=18, rondas=6, p_viaje=0.30, T3=(8, 8, 8), conf_v=0.8):
        """Etapas por club en rondas; con prob. p_viaje llega un técnico que ya dirigió otro club (viajero).
        Rasgos: T3[0] del técnico, T3[1] del club, T3[2] de nadie. Perfil = efecto × parte real + ruido (confiabilidad conf_v)."""
        r = np.random.default_rng(seed); tecs, filas = [], []
        for t in range(rondas):
            for c in range(n_club):
                prev = [x for x in set(tecs) if all(f['club'] != c for f in filas if f['tec'] == x)]
                tec = r.choice(prev) if (prev and r.random() < p_viaje) else len(set(tecs))
                tecs.append(tec); filas.append(dict(club=c, tec=int(tec), ronda=t))
        E = pd.DataFrame(filas); nT = E.tec.max() + 1; T = sum(T3)
        U, V = r.normal(0, 1, (nT, T)), r.normal(0, 1, (n_club, T))
        real = np.zeros((len(E), T))
        real[:, :T3[0]] = U[E.tec, :T3[0]]; real[:, T3[0]:T3[0] + T3[1]] = V[E.club, T3[0]:T3[0] + T3[1]]
        ruido = np.sqrt((1 - conf_v) / conf_v)                   # con varianza real 1, confiabilidad = conf_v
        Z = efecto * real + r.normal(0, ruido, real.shape)
        tipo = np.array(['tec'] * T3[0] + ['club'] * T3[1] + ['nadie'] * T3[2])
        return E, Z, tipo, np.full(T, conf_v)

    def pares_de(E):
        pt, ct, pc, cc, pool = [], [], [], [], []
        for tec, d in E.groupby('tec'):
            ix = d.index.values
            for i in range(len(ix)):
                for j in range(i + 1, len(ix)):
                    if E.club[ix[i]] != E.club[ix[j]]: pt.append((ix[i], ix[j])); ct.append(tec)
        for c, d in E.sort_values('ronda').groupby('club'):
            ix = d.index.values
            for i in range(len(ix) - 1):
                if E.tec[ix[i]] != E.tec[ix[i + 1]]: pc.append((ix[i], ix[i + 1])); cc.append(c)
        n = len(E)
        for i in range(n):
            for j in range(i + 1, n):
                if E.tec[i] != E.tec[j] and E.club[i] != E.club[j]: pool.append((i, j))
        return np.array(pt), np.array(ct), np.array(pc), np.array(cc), np.array(pool)

    def validar(E, Z, w):
        """Llegadas con previa: técnico (su etapa previa) frente al club (su etapa anterior = 'ventana'); compatibilidad
        como percentil frente a los demás técnicos; cambio del equipo = distancia ventana → etapa nueva."""
        comp, cambio = [], []
        for i in E.index:
            prev_t = E[(E.tec == E.tec[i]) & (E.ronda < E.ronda[i]) & (E.club != E.club[i])]
            prev_c = E[(E.club == E.club[i]) & (E.ronda == E.ronda[i] - 1)]
            if len(prev_t) == 0 or len(prev_c) == 0: continue
            zt, zc = Z[prev_t.index[-1]], Z[prev_c.index[0]]
            cand = E[(E.ronda < E.ronda[i]) & (E.tec != E.tec[i]) & (E.club != E.club[i])].groupby('tec').tail(1).index
            comp.append(percentil(distancia(zt, zc, w), [distancia(Z[k], zc, w) for k in cand]))
            cambio.append(distancia(zc, Z[i], w))
        return np.array(comp), np.array(cambio)

    for nombre, ef in [('SIN efecto', 0.0), ('CON efecto', 1.0)]:
        E, Z, tipo, conf = liga(ef, seed=7)
        pt, ct, pc, cc, pool = pares_de(E)
        T = ejes(Z, conf, pt, ct, pc, cc, pool, B=300, P=100)
        T['cuadrante'], T['firmeza'] = clasificar(T)
        print(f'\n--- ({2 if ef == 0 else 3}) Liga inventada {nombre}: {len(E)} etapas · {len(pt)} parejas de viajeros · {len(pc)} relevos de club ---')
        print(T.assign(tipo=tipo).groupby('tipo')[['tec', 'club', 'placebo_de']].mean().round(2).to_string())
        print('Cuadrantes por tipo real:'); print(pd.crosstab(tipo, T['cuadrante']).to_string())
        comp, cambio = validar(E, Z, pesos(T))
        rho = spearmanr(comp, cambio, nan_policy='omit')[0]
        bs = [spearmanr(*(lambda ix: (comp[ix], cambio[ix]))(rng0.integers(0, len(comp), len(comp))), nan_policy='omit')[0] for _ in range(500)]
        lo, hi = np.nanpercentile(bs, [2.5, 97.5])
        print(f'Validación: llegadas {len(comp)} · Spearman(compatibilidad, cambio) {rho:+.2f} [{lo:+.2f}, {hi:+.2f}]')
        m = T.assign(tipo=tipo).groupby('tipo')[['tec', 'club']].mean()
        if ef == 0:
            chk('ejes ≈ 0 sin efecto', (m.abs() < 0.3).all().all(), m.round(2).to_dict())
            chk('casi todo "de nadie"', (T['cuadrante'] == 'de nadie').mean() >= 0.8, f"{(T['cuadrante'] == 'de nadie').mean():.0%}")
            chk('validación sin señal (IC con 0)', lo <= 0 <= hi, f'{rho:+.2f}')
        else:
            chk('rasgos del técnico: eje técnico alto, club bajo', m.loc['tec', 'tec'] > 0.6 and m.loc['tec', 'club'] < 0.3, m.loc['tec'].round(2).to_dict())
            chk('rasgos del club: eje club alto, técnico bajo', m.loc['club', 'club'] > 0.6 and m.loc['club', 'tec'] < 0.3, m.loc['club'].round(2).to_dict())
            chk('rasgos de nadie: ambos bajos', (m.loc['nadie'].abs() < 0.3).all(), m.loc['nadie'].round(2).to_dict())
            chk('clasificación correcta ≥ 80%', (T['cuadrante'].values == np.select([tipo == 'tec', tipo == 'club'], ['del técnico', 'del club'], 'de nadie')).mean() >= 0.8,
                f"{(T['cuadrante'].values == np.select([tipo == 'tec', tipo == 'club'], ['del técnico', 'del club'], 'de nadie')).mean():.0%}")
            chk('validación: menos compatibilidad → más cambio', hi < 0, f'{rho:+.2f} [{lo:+.2f}, {hi:+.2f}]')
    print(f"\nC-0 terminó: {sum(ok_total)} de {len(ok_total)} pruebas PASAN. Funciones listas para C-1." if all(ok_total)
          else f"\nC-0: {len(ok_total) - sum(ok_total)} prueba(s) FALLAN → NO correr C-1; mandar esta salida a Claude.")
    C0_OK = all(ok_total)


C1 - Compatibilidad Real

In [ ]:
# [C-1 v1.0] Compatibilidad técnico-club · inspección de columnas → mapa de dos ejes → compatibilidad → validación → América
# =====================================================================
# Sesión 12 · 2026-10-18 · Hackathon ISAC 2026, Track 2 · T-11 (reescrito), N-3 · Decisiones S12-01 a S12-08
# Antes: celda 1 (Drive) → C-0 (define funciones; debe terminar con todas las pruebas PASAN). Tarda ~3-8 min.
# Lee una LISTA DE PREFIJOS: el día 19 se agrega 'r9' a PREFIJOS y se vuelve a correr sin tocar nada más.
# Unidades:
#   ejes del mapa: correlación entre parejas de etapas ÷ confiabilidad del rasgo (≈ 0 a 1; 1 = se conserva completo)
#   z: perfil ÷ τ (dispersión real entre técnicos del rasgo) → 1 = una desviación típica entre técnicos
#   compatibilidad: percentil 0-100% frente a los demás técnicos de la liga (100 = el más parecido a lo que el club ya hace)
# Escribe (con freno de Drive) en salidas/fw5: c_mapa.csv, c_llegadas.csv, c_ranking_america.csv.
# =====================================================================
import os, json, time
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
assert globals().get('C0_OK', False), 'Corre primero C-0 y confirma que todas sus pruebas PASAN'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5 = os.path.join(DP, 'fw5'); FW1 = os.path.join(DP, 'fw1'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
PREFIJOS = ['r4', 'r6', 'r6b', 'r6c', 'r7u', 'r7b', 'r7c', 'r8']       # ← el día 19: agregar 'r9'
ETAPA_MIN, NUEVA_MIN = 15, 10          # P-5 para el mapa; la etapa nueva de una llegada necesita 10+ (C-00)
VENTANA, VENT_MIN = 18, 10             # S12-05 (Arturo): perfil del club = sus últimos 18 partidos antes del relevo (mínimo 10 con dato)
CONF_MIN = 0.5                         # P-6: solo rasgos secundarios o núcleo
B, PLAC, B_VAL = 1000, 200, 2000
ESCRIBIR = True
rng = np.random.default_rng(SEMILLA)
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 90)
t0 = time.time()
sal = lambda p: 'r5' if p == 'r4' else p
corto = lambda s: ' '.join(s.split()[:1] + [s.split()[-2] if len(s.split()) >= 3 else s.split()[-1]]) if isinstance(s, str) and s.strip() else s

# ---------- 0. Inspección de columnas (antes de asumir nada) ----------
REQ = {'contraidos': ['metrica', 'etapa_id', 'perfil_r4', 'perfil_mostrado'], 'confiabilidad': ['metrica', 'conf', 'tau', 'sigma_partido'],
       'perfiles_etapa': ['etapa_id', 'metrica', 'tau_estado'], 'residuos': ['metrica', 'match_id', 'team', 'e', 'den', 'res']}
CONT, CF, TAUE, RES, S2 = [], [], {}, {}, {}
print('--- (0) Inspección por prefijo: filas y columnas requeridas ---')
for p in PREFIJOS:
    fuentes = {'contraidos': os.path.join(FW5, f'{sal(p)}_perfiles_contraidos.parquet'), 'confiabilidad': os.path.join(SAL5, f'{sal(p)}_confiabilidad.csv'),
               'perfiles_etapa': os.path.join(FW5, f'{p}_perfiles_etapa.parquet'), 'residuos': os.path.join(FW5, f'{p}_residuos.parquet')}
    D = {k: (pd.read_csv(v) if v.endswith('.csv') else pd.read_parquet(v)) for k, v in fuentes.items()}
    faltan = {k: [c for c in REQ[k] if c not in D[k].columns] for k in D}
    print(f"  {p}: " + ' · '.join(f"{k} {len(D[k]):,}" for k in D) + ('' if not any(faltan.values()) else f'  ← FALTAN {faltan}'))
    assert not any(faltan.values()), f'Columnas faltantes en {p}: avisar a Claude'
    c = D['contraidos'].assign(rasgo=lambda d: p + ':' + d.metrica); CONT.append(c)
    CF.append(D['confiabilidad'].assign(rasgo=lambda d: p + ':' + d.metrica, prefijo=p))
    TAUE.update({p + ':' + m: v for m, v in D['perfiles_etapa'].groupby('metrica')['tau_estado'].first().items()})
    rs = D['residuos'][REQ['residuos']]
    S2.update({p + ':' + m: float((g.den * g.res ** 2).sum() / (len(g) - 19)) for m, g in rs.groupby('metrica')})   # s2 de R6r-2/R5 (P_REG = 19)
    RES[p] = {t: g for t, g in rs.groupby('team')}
CONT = pd.concat(CONT, ignore_index=True); CF = pd.concat(CF, ignore_index=True).set_index('rasgo')
RASGOS = [r for r in CF.index if CF.loc[r, 'conf'] >= CONF_MIN and CF.loc[r, 'tau'] > 0]
print(f'Rasgos: {len(CF)} en total · {len(RASGOS)} con confiabilidad ≥ {CONF_MIN} (entran) · fuera: {len(CF) - len(RASGOS)}')

# ---------- Etapas (copia literal de C-00) ----------
with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
    TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy(); ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ET = ptp.groupby('etapa_id').agg(team=('team', 'first'), tecnico=('tec_p', 'first'), partidos=('match_id', 'nunique'),
                                 ini=('fecha', 'min'), fin=('fecha', 'max')).reset_index()
ETAPAS = ET.etapa_id.tolist(); IX = {e: i for i, e in enumerate(ETAPAS)}
tau = CF.loc[RASGOS, 'tau'].values; conf = CF.loc[RASGOS, 'conf'].values
ZR = (CONT.pivot_table(index='etapa_id', columns='rasgo', values='perfil_r4').reindex(index=ETAPAS, columns=RASGOS) / tau).values
ZM = (CONT.pivot_table(index='etapa_id', columns='rasgo', values='perfil_mostrado').reindex(index=ETAPAS, columns=RASGOS) / tau).values
print(f'Etapas: {len(ETAPAS)} · con perfil en todos los rasgos: {int(np.isfinite(ZM).all(1).sum())} · celdas nulas {np.isnan(ZM).mean():.1%}')

# ---------- 1. Parejas ----------
L = ET[ET.partidos >= ETAPA_MIN]
pt, ct = [], []
for tec, d in L.groupby('tecnico'):
    ix = [IX[e] for e in d.etapa_id]
    for i in range(len(ix)):
        for j in range(i + 1, len(ix)):
            if ET.team[ix[i]] != ET.team[ix[j]]: pt.append((ix[i], ix[j])); ct.append(tec)
pc, cc, pnc, cnc = [], [], [], []
for team, d in L.sort_values('ini').groupby('team'):
    ix = [IX[e] for e in d.etapa_id]
    for i in range(len(ix) - 1):
        if ET.tecnico[ix[i]] != ET.tecnico[ix[i + 1]]: pc.append((ix[i], ix[i + 1])); cc.append(team)
        if i + 2 < len(ix) and ET.tecnico[ix[i]] != ET.tecnico[ix[i + 2]]: pnc.append((ix[i], ix[i + 2])); cnc.append(team)
li = [IX[e] for e in L.etapa_id]
pool = [(a, b) for k, a in enumerate(li) for b in li[k + 1:] if ET.tecnico[a] != ET.tecnico[b] and ET.team[a] != ET.team[b]]
pt, ct, pc, cc = np.array(pt), np.array(ct), np.array(pc), np.array(cc)
print(f'\n--- (1) Parejas (deben coincidir con C-00: 29 de viajeros y 69 relevos) · viajeros {len(pt)} de {len(set(ct))} técnicos · '
      f'relevos {len(pc)} en {len(set(cc))} clubes · club no consecutivo (sensibilidad) {len(pnc)} · parejas al azar {len(pool):,}')

# ---------- 2. Mapa de dos ejes ----------
T = ejes(ZR, conf, pt, ct, pc, cc, pool, B=B, P=PLAC).set_index(pd.Index(RASGOS, name='rasgo'))
T['cuadrante'], T['firmeza'] = clasificar(T)
T['prefijo'] = [r.split(':')[0] for r in RASGOS]
T['club_no_consec'] = r_pares(ZR, pnc) / np.clip(conf, .05, None)
# Sensibilidad 10+ (etapas de 10 a 14 partidos también entran en las parejas de viajeros)
L10 = ET[ET.partidos >= NUEVA_MIN]; pt10 = []
for tec, d in L10.groupby('tecnico'):
    ix = [IX[e] for e in d.etapa_id]
    pt10 += [(ix[i], ix[j]) for i in range(len(ix)) for j in range(i + 1, len(ix)) if ET.team[ix[i]] != ET.team[ix[j]]]
T['tec_10'] = r_pares(ZR, pt10) / np.clip(conf, .05, None)
# Sensibilidad Elo (S12-03): ¿"club" = nivel de plantilla? Se quita a cada perfil la parte que explica el Elo medio de la etapa
elo = pd.read_parquet(os.path.join(FW1, 'elo_equipo_partido_e2.parquet'), columns=['match_id', 'team', 'elo_pre'])
eloE = ptp.merge(elo, on=['match_id', 'team'], how='left').groupby('etapa_id')['elo_pre'].mean().reindex(ETAPAS).values
ZE = ZR.copy()
for j in range(ZR.shape[1]):
    m = np.isfinite(ZR[:, j]) & np.isfinite(eloE)
    if m.sum() > 10:
        b = np.polyfit(eloE[m], ZR[m, j], 1); ZE[m, j] = ZR[m, j] - np.polyval(b, eloE[m])
T['tec_sin_elo'] = r_pares(ZE, pt) / np.clip(conf, .05, None); T['club_sin_elo'] = r_pares(ZE, pc) / np.clip(conf, .05, None)
T['corr_elo'] = [np.corrcoef(eloE[np.isfinite(ZR[:, j]) & np.isfinite(eloE)], ZR[np.isfinite(ZR[:, j]) & np.isfinite(eloE), j])[0, 1] for j in range(len(RASGOS))]
# Opcional (idea de Arturo): legado: ¿las etapas largas dejan más huella en el club? relevos partidos por la duración de la etapa saliente
dur = np.array([ET.partidos[a] for a, _ in pc]); med = np.median(dur)
T['club_tras_larga'] = r_pares(ZR, pc[dur >= med]) / np.clip(conf, .05, None)
T['club_tras_corta'] = r_pares(ZR, pc[dur < med]) / np.clip(conf, .05, None)

V = ['conf', 'tec', 'tec_lo', 'tec_hi', 'club', 'club_lo', 'club_hi', 'dif_lo', 'dif_hi', 'cuadrante', 'firmeza']
print(f'\n--- (2) Mapa: ejes corregidos por confiabilidad (corte provisional {CORTE}) · ruido de un rasgo sin efecto (placebo): ±{2 * T.placebo_de.median():.2f} ---')
print('Placebo (debe ser ≈ 0): media', round(T.placebo_media.mean(), 3), '· rango', round(T.placebo_media.min(), 2), 'a', round(T.placebo_media.max(), 2))
print('\nCuadrantes por prefijo (rasgos):'); print(pd.crosstab(T.prefijo, T.cuadrante, margins=True).to_string())
print('\nFirmes:', T[T.firmeza == 'firme'].groupby('cuadrante').size().to_dict())
print('\nLos 12 más "del técnico" (orden: eje técnico − eje club):')
print(T.sort_values('dif', ascending=False).head(12)[V].round(2).to_string())
print('\nLos 12 más "del club":')
print(T.sort_values('dif').head(12)[V].round(2).to_string())
CAND = ['r8:saque_banda_largo', 'r8:corto_pct', 'r6:banda_izquierda', 'r8:zonales', 'r8:pct_marca', 'r8:pendiente_marca', 'r8:pb_of_1', 'r8:pb_of_2',
        'r8:pegados_portero', 'r8:def_area_360', 'r4:field_tilt', 'r4:posesion', 'r4:saque_meta_corto', 'r4:riesgo_pase', 'r4:presion_alta',
        'r7u:inicio_linea_3_5', 'r7u:min_menores_23', 'r7c:altura_laterales', 'r7c:amplitud_bandas', 'r7c:portero_libero', 'r7c:libertad_posicional']
print('\nCandidatos de la hipótesis central (club: saque largo, córner corto, banda · técnico: marcaje, playbooks, carga al portero, roles):')
print(T.reindex([c for c in CAND if c in T.index])[V + ['club_no_consec', 'tec_10', 'tec_sin_elo', 'club_sin_elo', 'corr_elo']].round(2).to_string())
print('Candidatos fuera por confiabilidad < 0.5:', [c for c in CAND if c not in T.index])
cl_r = T.cuadrante == 'del club'
print(f"\nSensibilidad Elo: en los rasgos del club, eje club {T.loc[cl_r, 'club'].mean():.2f} → sin Elo {T.loc[cl_r, 'club_sin_elo'].mean():.2f} "
      f"(si cae mucho: 'club' = nivel de plantilla) · en los del técnico, eje técnico {T.loc[T.cuadrante == 'del técnico', 'tec'].mean():.2f} → "
      f"sin Elo {T.loc[T.cuadrante == 'del técnico', 'tec_sin_elo'].mean():.2f}")
print(f"Sensibilidad club no consecutivo (separación en el tiempo parecida a la de los viajeros): eje club medio {T.club.mean():.2f} → {T.club_no_consec.mean():.2f}")
print(f"Legado (opcional): en los rasgos del técnico, eje club tras etapa larga {T.loc[T.cuadrante == 'del técnico', 'club_tras_larga'].mean():.2f} "
      f"frente a tras etapa corta {T.loc[T.cuadrante == 'del técnico', 'club_tras_corta'].mean():.2f} (mediana {med:.0f} partidos)")

# ---------- 3. Perfil del club: últimos 18 partidos antes del relevo ----------
MIN_ = pd.read_parquet(os.path.join(FW5, 'r4_minutos_v21.parquet'))
W3 = MIN_.assign(e3=np.sign(MIN_.e5)).groupby('e3')['minutos'].sum(); W3v = (W3 / W3.sum()).reindex([-1, 0, 1]).fillna(0).values   # copia de R6r
def perfil_club(team, partidos, contraido=True):
    z = np.full(len(RASGOS), np.nan)
    for j, r in enumerate(RASGOS):
        p, m = r.split(':', 1); g = RES[p].get(team)
        if g is None: continue
        g = g[(g.metrica == m) & g.match_id.isin(partidos)]
        n = g.match_id.nunique()
        if n == 0: continue
        pf = perfil_ventana(g.e.values, g.den.values, g.res.values, S2[r], TAUE[r], W3v)
        z[j] = (contraer(pf, n, CF.loc[r, 'tau'], CF.loc[r, 'sigma_partido']) if contraido else pf) / CF.loc[r, 'tau']
    return z
# Coherencia (buena práctica 13): con los partidos de una etapa completa, la ventana debe reproducir perfil_r4 de R6r-2/R5
mx = []
for e in L.sample(5, random_state=SEMILLA).etapa_id:
    zc = perfil_club(ET.team[IX[e]], set(ptp.loc[ptp.etapa_id == e, 'match_id']), contraido=False)
    mx.append(np.nanmax(np.abs(zc - ZR[IX[e]])))
print(f'\n--- (3) Coherencia: ventana = etapa completa reproduce el perfil de R6r-2/R5 · diferencia máxima {max(mx):.2e} z (≈ 0; < 0.01 aceptable)')
def ventana(team, antes_de):
    m = ptp[(ptp.team == team) & (ptp.fecha < antes_de)].sort_values('fecha').tail(VENTANA)
    return set(m.match_id), len(m)

# ---------- 4. Compatibilidad en las llegadas con previa (pesos dejando fuera al técnico evaluado) ----------
def pesos_sin(tec):
    keep = ct != tec
    r = r_pares(ZR, pt[keep]) / np.clip(conf, .05, None)
    return np.clip(np.nan_to_num(r), 0, PESO_MAX)
def candidatos(antes_de, excluir_tec, excluir_team):
    c = L[(L.ini < antes_de) & (L.tecnico != excluir_tec) & (L.team != excluir_team)].sort_values('ini').groupby('tecnico').tail(1)
    return [IX[e] for e in c.etapa_id], c.tecnico.tolist()
filas = []
for r_ in ET[ET.partidos >= NUEVA_MIN].itertuples(index=False):
    ant = L[(L.tecnico == r_.tecnico) & (L.team != r_.team) & (L.fin < r_.ini)]
    if not len(ant): continue
    prev = ant.sort_values('fin').etapa_id.iloc[-1]
    vp, nv = ventana(r_.team, r_.ini)
    if nv < VENT_MIN: filas.append(dict(etapa_nueva=r_.etapa_id, tecnico=corto(r_.tecnico), previa=prev, ventana=nv, nota='ventana corta')); continue
    zc = perfil_club(r_.team, vp); zt = ZM[IX[prev]]; w = pesos_sin(r_.tecnico)
    cix, _ = candidatos(r_.ini, r_.tecnico, r_.team)
    d = distancia(zt, zc, w); comp = percentil(d, [distancia(ZM[k], zc, w) for k in cix])
    zn = ZM[IX[r_.etapa_id]]
    filas.append(dict(etapa_nueva=r_.etapa_id, tecnico=corto(r_.tecnico), previa=prev, ventana=nv, partidos_nueva=r_.partidos, candidatos=len(cix),
                      compat=comp, distancia=d, cambio=distancia(zc, zn, w),
                      nueva_a_tecnico=distancia(zn, zt, w), nueva_a_club=distancia(zn, zc, w),
                      choques=choques(zt, zc, w, RASGOS), v_zc=zc, v_zt=zt, v_zn=zn, v_w=w, v_cix=cix, nota=''))
LL = pd.DataFrame(filas)
ok = LL[LL.nota == ''].copy()
print(f'\n--- (4) Compatibilidad en llegadas con previa: {len(LL)} llegadas · evaluables {len(ok)} (ventana ≥ {VENT_MIN}) ---')
print(ok[['etapa_nueva', 'tecnico', 'previa', 'ventana', 'candidatos', 'compat', 'distancia', 'cambio', 'choques']].round(2).sort_values('compat').to_string(index=False))

# ---------- 5. Validación ----------
cmp_, cam_ = ok.compat.values, ok.cambio.values
rho = spearmanr(cmp_, cam_)[0]
bs = [spearmanr(cmp_[ix], cam_[ix])[0] for ix in (rng.integers(0, len(ok), len(ok)) for _ in range(B_VAL))]
cerca = (ok.nueva_a_tecnico < ok.nueva_a_club).values
pl = []
for _ in range(PLAC):                                        # placebo: el "técnico" es un candidato al azar de la liga
    pl.append(np.mean([distancia(f['v_zn'], ZM[rng.choice(f['v_cix'])], f['v_w']) < distancia(f['v_zn'], f['v_zc'], f['v_w']) for f in ok.to_dict('records')]))
print(f'\n--- (5) Validación (pesos sin el técnico evaluado) ---')
print(f'(a) Spearman(compatibilidad, cuánto cambió el equipo): {rho:+.2f} [{np.nanpercentile(bs, 2.5):+.2f}, {np.nanpercentile(bs, 97.5):+.2f}] '
      f'· se espera negativo (menos compatible → más cambio)')
print(f'(b) La etapa nueva queda más cerca del técnico que del club en {cerca.mean():.0%} de las llegadas · placebo (técnico al azar) '
      f'{np.mean(pl):.0%} [{np.percentile(pl, 2.5):.0%}, {np.percentile(pl, 97.5):.0%}]')
print('Umbrales de Arturo (T-11) como hipótesis: compatibilidad de las llegadas por tramos:',
      pd.cut(ok.compat, [0, 30, 50, 70, 85, 100], include_lowest=True).value_counts().sort_index().to_dict())

# ---------- 6. América ----------
print('\n--- (6) América ---')
print(ok[ok.etapa_nueva.str.startswith('Am')][['etapa_nueva', 'tecnico', 'previa', 'compat', 'cambio', 'choques']].round(2).to_string(index=False) or 'Sin llegadas con previa a América en los datos')
ult = ptp.loc[ptp.team.str.startswith('Am'), 'fecha'].max(); team_am = ptp.loc[ptp.team.str.startswith('Am'), 'team'].iloc[0]
vp, nv = ventana(team_am, ult + pd.Timedelta(days=1)); zc = perfil_club(team_am, vp); wF = pesos(T)
cix, ctec = candidatos(ult + pd.Timedelta(days=1), '', team_am)
dist = np.array([distancia(ZM[k], zc, wF) for k in cix])
RK = pd.DataFrame({'tecnico': [corto(t) for t in ctec], 'etapa': [ETAPAS[k] for k in cix], 'distancia': dist})
RK['compat'] = [percentil(d, np.delete(dist, i)) for i, d in enumerate(dist)]
RK['choques'] = [choques(ZM[k], zc, wF, RASGOS) for k in cix]
RK = RK.sort_values('compat', ascending=False).reset_index(drop=True); RK.index += 1
print(f'Perfil de América = sus últimos {nv} partidos en los datos (hasta {ult.date()}) · candidatos: {len(RK)} técnicos (última etapa de 15+ fuera de América)')
alm = RK[RK.tecnico.str.contains('Almada', na=False)]
if len(alm):
    k_alm = IX[alm.etapa.iloc[0]]; w_alm = pesos_sin([t for t in ctec if 'Almada' in t][0])
    d_loo = distancia(ZM[k_alm], zc, w_alm); comp_loo = percentil(d_loo, [distancia(ZM[k], zc, w_alm) for k in cix if k != k_alm])
    print(f"CASO EN VIVO · Almada ({alm.etapa.iloc[0]}): compatibilidad {alm.compat.iloc[0]:.0f}% (puesto {alm.index[0]} de {len(RK)}) · "
          f"sin sus propias parejas en los pesos: {comp_loo:.0f}% · choques: {alm.choques.iloc[0]}")
print('Los 5 más compatibles:'); print(RK.head(5)[['tecnico', 'etapa', 'compat', 'choques']].round(1).to_string())
print('Los 3 menos compatibles:'); print(RK.tail(3)[['tecnico', 'etapa', 'compat', 'choques']].round(1).to_string())
print('Ortiz y Jardine en este ranking:'); print(RK[RK.tecnico.str.contains('Ortiz|Soares|Jardine', na=False)][['tecnico', 'etapa', 'compat']].round(1).to_string())

# ---------- 7. Guardar ----------
if ESCRIBIR:
    assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Freno: Drive no está montado; no se escribe nada'
    os.makedirs(SAL5, exist_ok=True)
    T.to_csv(os.path.join(SAL5, 'c_mapa.csv'))
    LL.drop(columns=[c for c in LL.columns if c.startswith('v_')]).to_csv(os.path.join(SAL5, 'c_llegadas.csv'), index=False)
    RK.to_csv(os.path.join(SAL5, 'c_ranking_america.csv'))
    print('\nGuardado: salidas/fw5/c_mapa.csv, c_llegadas.csv, c_ranking_america.csv')
print(f'C-1 terminó en {time.time() - t0:.0f} s · prefijos: {PREFIJOS}')

C-2 Placebo registro

In [ ]:
# [C-2 v1.0] Compatibilidad · (i) placebo de la validación (a) · (ii) números para preregistrar Almada → América antes del día 20
# =====================================================================
# Sesión 12 · 2026-10-18 · Correr DESPUÉS de C-0 y C-1 en la misma sesión de Colab (usa sus variables). Tarda < 2 min.
# (i) ¿La correlación "menos compatible → más cambio" (−0.44) podría salir sola, por cómo se construyen las medidas?
#     Placebo: en cada llegada se sustituye al técnico real por un candidato al azar de la liga y se recalcula la correlación
#     con el cambio REAL del equipo. Si el −0.44 fuera un artefacto, el placebo daría algo parecido; si es real, ≈ 0.
# (ii) Almada frente a los últimos 18 partidos de América: rasgo por rasgo (z y unidades naturales), su peso, cuánto aporta
#     a la distancia y su reloj de medición (partidos para leerlo con confiabilidad 0.7). Con 2026/27 habrá ~10-13 partidos:
#     solo se preregistran rasgos con reloj ≤ RELOJ_MAX. También el cambio esperado según la relación de las 27 llegadas.
# Escribe (con freno de Drive) salidas/fw5/c_preregistro_almada.csv.
# =====================================================================
import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
for v in ['ok', 'T', 'ZM', 'RASGOS', 'CF', 'CONT', 'perfil_club', 'ventana', 'candidatos', 'IX', 'ETAPAS', 'ptp', 'RK', 'pesos_sin']:
    assert v in globals(), f'Falta {v}: corre C-0 y C-1 en esta sesión antes de C-2'
RELOJ_MAX, REP = 12, 500
rng2 = np.random.default_rng(SEMILLA)

# ---------- (i) Placebo de la validación (a) ----------
rec = ok.to_dict('records'); cam = ok.cambio.values
rho_real = spearmanr(ok.compat.values, cam)[0]
plac = []
for _ in range(REP):
    comp_p = []
    for f in rec:
        cix = list(f['v_cix']); k = cix[rng2.integers(len(cix))]
        d_k = distancia(ZM[k], f['v_zc'], f['v_w'])
        comp_p.append(percentil(d_k, [distancia(ZM[j], f['v_zc'], f['v_w']) for j in cix if j != k]))
    plac.append(spearmanr(comp_p, cam)[0])
plac = np.array(plac)
print('--- (i) Placebo de la validación (a): técnico al azar en lugar del real ---')
print(f'Real {rho_real:+.2f} · placebo media {np.nanmean(plac):+.2f} [{np.nanpercentile(plac, 2.5):+.2f}, {np.nanpercentile(plac, 97.5):+.2f}] · '
      f'% de placebos tan negativos como el real: {100 * np.mean(plac <= rho_real):.1f}%')

# ---------- (ii) Almada frente a América (últimos 18 partidos) ----------
alm = RK[RK.tecnico.str.contains('Almada', na=False)]
assert len(alm), 'Almada no está en el ranking de C-1'
k_alm = IX[alm.etapa.iloc[0]]; tec_alm = ETAPAS[k_alm]
team_am = ptp.loc[ptp.team.str.startswith('Am'), 'team'].iloc[0]; ult = ptp.loc[ptp.team == team_am, 'fecha'].max()
vp, nv = ventana(team_am, ult + pd.Timedelta(days=1)); zc = perfil_club(team_am, vp)
nombre_alm = ptp.loc[ptp.etapa_id == tec_alm, 'tec_p'].iloc[0]
wF, wL = pesos(T), pesos_sin(nombre_alm)
zt = ZM[k_alm]
liga = CONT.groupby('rasgo')['liga'].first().reindex(RASGOS).values if 'liga' in CONT.columns else np.full(len(RASGOS), np.nan)
tau_ = CF.loc[RASGOS, 'tau'].values
reloj = CF.loc[RASGOS, 'reloj'].values if 'reloj' in CF.columns else np.full(len(RASGOS), np.nan)
ap = np.where(np.isfinite(zt) & np.isfinite(zc), wF * (zt - zc) ** 2, 0)
P = pd.DataFrame({'rasgo': RASGOS, 'cuadrante': T.loc[RASGOS, 'cuadrante'].values, 'peso': wF, 'peso_sin_almada': wL,
                  'z_almada': zt, 'z_america': zc, 'almada_nat': liga + zt * tau_, 'america_nat': liga + zc * tau_, 'liga': liga,
                  'aporte_pct': 100 * ap / ap.sum(), 'reloj': reloj})
P['direccion_esperada'] = np.where(P.z_almada > P.z_america, 'sube', 'baja')
P['preregistrable'] = (P.peso > 0) & (P.reloj <= RELOJ_MAX)
P = P.sort_values('aporte_pct', ascending=False)
print(f'\n--- (ii) Almada ({tec_alm}) frente a América (últimos {nv} partidos, hasta {ult.date()}) ---')
print(f"Compatibilidad: {alm.compat.iloc[0]:.0f}% · puesto {alm.index[0]} de {len(RK)} · distancia {alm.distancia.iloc[0]:.2f} z")
print('Rasgos con peso > 0, ordenados por cuánto aportan a la distancia (z: desviaciones típicas entre técnicos; nat: unidad del rasgo):')
print(P[P.peso > 0].head(12)[['rasgo', 'cuadrante', 'peso', 'z_almada', 'z_america', 'almada_nat', 'america_nat', 'liga', 'aporte_pct',
                              'reloj', 'direccion_esperada', 'preregistrable']].round(2).to_string(index=False))
# Cambio esperado según la relación compatibilidad → cambio de las llegadas
b = np.polyfit(ok.compat.values, cam, 1); res = cam - np.polyval(b, ok.compat.values)
pred = np.polyval(b, alm.compat.iloc[0]); de = res.std(ddof=2)
print(f'\nCambio de las {len(ok)} llegadas (distancia ventana → etapa nueva, z): mediana {np.median(cam):.2f} · p75 {np.percentile(cam, 75):.2f} · máx {cam.max():.2f}')
print(f'Relación lineal: cambio = {b[1]:.2f} {b[0]:+.4f} × compatibilidad · predicción para Almada (compat {alm.compat.iloc[0]:.0f}%): '
      f'{pred:.2f} (banda ±1 desv. de los residuos: {pred - de:.2f} a {pred + de:.2f})')
print('Nota: el cambio de 2026/27 se medirá con ~10-13 partidos; el perfil contraído de la etapa nueva será más cercano a la liga '
      'que el de una etapa completa, lo que REDUCE la distancia medida (la predicción es conservadora si se cumple).')
ESCRIBIR = True
if ESCRIBIR:
    assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Freno: Drive no está montado; no se escribe nada'
    P.to_csv(os.path.join(SAL5, 'c_preregistro_almada.csv'), index=False)
    print('Guardado: salidas/fw5/c_preregistro_almada.csv')

V-00 Inspección

In [ ]:
# [V-00 v1.0] Inspección de lo que existe de 2026/27 (SOLO LECTURA, SOLO CONTEOS: no calcula ningún perfil ni métrica de juego)
# =====================================================================
# Sesión 12 · 2026-10-18 · Insumo para escribir el procedimiento de evaluación del preregistro de Almada (D-2: nadie mira
# el juego de 2026/27 antes de congelar). Qué cuenta:
#   (1) Temporadas en la tabla de técnicos; partidos de 2026/27 (liga, América, Almada) y fecha del último.
#   (2) Para los partidos de América 2026/27: ¿hay eventos crudos por partido? ¿hay 360? ¿hay Elo antes del partido?
#   (3) Qué archivos de R1 (eventos_*, posesiones_*) y de Elo existen por temporada en fw1 y fw5.
#   (4) Carpetas de datos_parquet y datos_crudos (para ubicar el 360 sin adivinar rutas).
# Orden: celda 1 → V-00. Tarda < 1 min.
# =====================================================================
import os, glob
import pandas as pd

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5')
VIVO = '2026/2027'

ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
print('--- (1) Tabla de técnicos: equipo-partidos por temporada ---')
print(ptp.groupby('temporada').agg(filas=('match_id', 'size'), partidos=('match_id', 'nunique')).to_string())
v = ptp[ptp.temporada == VIVO].copy(); v['fecha'] = pd.to_datetime(v['fecha'])
am = v[v.team.str.startswith('Am')]
print(f"\n{VIVO}: partidos {v.match_id.nunique()} · último {v.fecha.max().date() if len(v) else '—'} · América {am.match_id.nunique()} "
      f"(técnicos: {am.tec_p.value_counts().to_dict()}) · jugados {int(am['jugado'].astype(bool).sum()) if 'jugado' in am else '—'}")

print('\n--- (2) Partidos de América en 2026/27: insumos disponibles ---')
DIR_EV = os.path.join(DP, 'liga', 'eventos')
cand360 = [d for d in glob.glob(os.path.join(DP, '**', '*360*'), recursive=True) if os.path.isdir(d)] + \
          [d for d in glob.glob(os.path.join(RAIZ, 'datos_crudos', '**', '*360*'), recursive=True) if os.path.isdir(d)]
elo_f = {n: p for n, p in [('fw1', os.path.join(FW1, 'elo_equipo_partido_e2.parquet')), ('fw5', os.path.join(FW5, 'elo_equipo_partido_e2.parquet'))]
         if os.path.exists(p)}
elo = {n: set(pd.read_parquet(p, columns=['match_id'])['match_id']) for n, p in elo_f.items()}
mids = sorted(am.match_id.unique())
fila = dict(partidos=len(mids), eventos_crudos=sum(os.path.exists(os.path.join(DIR_EV, f'{m}.parquet')) for m in mids))
for d in cand360:
    fila[f'360 en {os.path.relpath(d, RAIZ)}'] = sum(bool(glob.glob(os.path.join(d, f'{m}*'))) for m in mids)
for n, s in elo.items(): fila[f'Elo ({n})'] = sum(m in s for m in mids)
print(pd.Series(fila).to_string())
print('Partidos de liga 2026/27 con Elo:', {n: sum(m in s for m in v.match_id.unique()) for n, s in elo.items()}, 'de', v.match_id.nunique())

print('\n--- (3) Archivos de R1 y Elo por carpeta ---')
for n, d in [('fw1', FW1), ('fw5', FW5)]:
    print(f"{n}: eventos {sorted(os.path.basename(x) for x in glob.glob(os.path.join(d, 'eventos_*.parquet')))} · "
          f"posesiones {len(glob.glob(os.path.join(d, 'posesiones_*.parquet')))} · Elo {[os.path.basename(x) for x in glob.glob(os.path.join(d, 'elo_*'))]}")

print('\n--- (4) Carpetas ---')
for base in [DP, os.path.join(RAIZ, 'datos_crudos')]:
    if os.path.isdir(base):
        print(os.path.relpath(base, RAIZ) + ':', sorted(e for e in os.listdir(base) if os.path.isdir(os.path.join(base, e))))
print('Carpetas con "360" en el nombre:', [os.path.relpath(d, RAIZ) for d in cand360] or 'ninguna')
print('\nV-00 terminó (no escribió nada ni calculó perfiles).')

C-3

In [ ]:
# [C-3 v1.0] Ensayo del preregistro de Almada con las 27 llegadas históricas (calibra P1-P4 antes de congelar)
# =====================================================================
# Sesión 12 · 2026-10-18 · Correr DESPUÉS de C-0, C-1 (y C-2) en la misma sesión de Colab. No toca 2026/27. Tarda ~2-5 min.
# Idea: el día 20 mediremos a Almada con solo ~K partidos. Antes de fijar umbrales, se simula EXACTAMENTE esa medición en las
# 27 llegadas con previa: se toma el perfil (contraído, P-5) de los PRIMEROS K partidos de cada etapa nueva y se calcula:
#   (a) P1 · cambio con K partidos: distancia ventana del club (18) → primeros K, con los pesos sin el técnico evaluado.
#       Es la escala correcta para el umbral de P1 (con K partidos la contracción acerca el perfil a la liga y achica el cambio).
#   (b) P2/P3 · cobertura de la brecha por rasgo: (club_K − club_antes) / (técnico − club_antes). 1 = copia al técnico; 0 = no se movió.
#       Solo brechas de ≥ 0.5 z (las chicas dan cocientes inestables). Se separa por cuadrante del mapa (técnico / club / otros).
#   (c) P4 · identificación a ciegas: con el perfil de los primeros K partidos, ¿qué tan arriba queda el técnico REAL entre todos
#       los candidatos de la liga (los mismos de C-1 + él) por parecido? Solo rasgos con reloj ≤ RELOJ_P4 (legibles con K partidos).
#       Azar: el técnico real cae en promedio en el percentil 50.
# Escribe (con freno de Drive) salidas/fw5/c_ensayo_llegadas.csv.
# =====================================================================
import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
for v in ['ok', 'T', 'ZM', 'RASGOS', 'CF', 'perfil_club', 'IX', 'ET', 'ptp', 'pesos_sin', 'distancia', 'percentil']:
    assert v in globals(), f'Falta {v}: corre C-0 y C-1 en esta sesión antes de C-3'
K_LIST, RELOJ_P4, BRECHA_MIN = [9, 12, 15], 20, 0.5
reloj = CF.loc[RASGOS, 'reloj'].values; legible = reloj <= RELOJ_P4
cuad = T.loc[RASGOS, 'cuadrante'].values
FOCO = ['r8:def_area_360', 'r8:zonales', 'r6:seg_hasta_ultimo', 'r6:pases_hasta_ultimo']
filas, cob = [], []
for f in ok.to_dict('records'):
    e_new = f['etapa_nueva']; k_new = IX[e_new]; team = ET.team[k_new]; tec_full = ET.tecnico[k_new]
    w = pesos_sin(tec_full); zc, zt = f['v_zc'], f['v_zt']; k_prev = IX[f['previa']]
    mids = ptp[ptp.etapa_id == e_new].sort_values('fecha').match_id.tolist()
    for K in K_LIST:
        if len(mids) < K: continue
        zk = perfil_club(team, set(mids[:K]))
        # (a) cambio con K partidos
        cam_k = distancia(zc, zk, w)
        # (c) identificación a ciegas (rasgos legibles con K partidos)
        wl = w * legible
        d_real = distancia(zk, zt, wl)
        d_cand = [distancia(zk, ZM[j], wl) for j in f['v_cix']]
        pct_id = percentil(d_real, d_cand)
        puesto = 1 + int(np.sum(np.array(d_cand) < d_real))
        filas.append(dict(etapa_nueva=e_new, tecnico=f['tecnico'], K=K, compat=f['compat'], cambio_completo=f['cambio'], cambio_K=cam_k,
                          id_percentil=pct_id, id_puesto=puesto, candidatos=len(d_cand) + 1))
        # (b) cobertura de la brecha por rasgo
        brecha = zt - zc
        for j in np.where(np.isfinite(brecha) & (np.abs(brecha) >= BRECHA_MIN) & np.isfinite(zk))[0]:
            cob.append(dict(etapa_nueva=e_new, K=K, rasgo=RASGOS[j], cuadrante=cuad[j], peso=w[j], reloj=reloj[j],
                            cobertura=(zk[j] - zc[j]) / brecha[j]))
EN, CB = pd.DataFrame(filas), pd.DataFrame(cob)
CB['grupo'] = np.where(CB.cuadrante == 'del técnico', 'del técnico', np.where(CB.cuadrante == 'del club', 'del club', 'otros'))

print('--- (a) P1 · cambio del equipo con los primeros K partidos (z; mismos pesos que C-1, sin el técnico evaluado) ---')
for K, g in EN.groupby('K'):
    r = spearmanr(g.compat, g.cambio_K)[0]
    print(f"K = {K:>2}: llegadas {len(g)} · cambio mediana {g.cambio_K.median():.2f} · p25 {g.cambio_K.quantile(.25):.2f} · p75 {g.cambio_K.quantile(.75):.2f} · "
          f"p90 {g.cambio_K.quantile(.9):.2f} · % > 1.00: {100 * (g.cambio_K > 1).mean():.0f}% · Spearman con compatibilidad {r:+.2f} "
          f"(etapa completa: mediana {g.cambio_completo.median():.2f})")

print(f'\n--- (b) P2/P3 · cobertura de la brecha (brechas ≥ {BRECHA_MIN} z; 0 = no se movió, 1 = igual al técnico; mediana y % que cubre ≥ la mitad) ---')
t = CB.groupby(['K', 'grupo']).cobertura.agg(casos='size', mediana='median', p25=lambda s: s.quantile(.25), p75=lambda s: s.quantile(.75),
                                               mitad_o_mas=lambda s: 100 * (s >= .5).mean())
print(t.round(2).to_string())
print('\nRasgos del preregistro (K = 12):')
print(CB[(CB.K == 12) & CB.rasgo.isin(FOCO)].groupby('rasgo').cobertura.agg(casos='size', mediana='median', mitad_o_mas=lambda s: 100 * (s >= .5).mean())
      .round(2).to_string())
print('Rasgos del técnico con reloj ≤ 12 (K = 12), juntos:',
      CB[(CB.K == 12) & (CB.grupo == 'del técnico') & (CB.reloj <= 12)].cobertura.agg(['size', 'median']).round(2).to_dict())

print(f'\n--- (c) P4 · identificación a ciegas: puesto del técnico real entre los candidatos por parecido (rasgos con reloj ≤ {RELOJ_P4}; azar = percentil 50) ---')
for K, g in EN.groupby('K'):
    print(f"K = {K:>2}: llegadas {len(g)} · percentil mediano del real {g.id_percentil.median():.0f} · media {g.id_percentil.mean():.0f} · "
          f"en el 10% más parecido: {100 * (g.id_percentil >= 90).mean():.0f}% · en el 25%: {100 * (g.id_percentil >= 75).mean():.0f}% · "
          f"puesto mediano {g.id_puesto.median():.0f} de {g.candidatos.median():.0f}")
print('\nDetalle K = 12 (de mejor a peor identificación):')
print(EN[EN.K == 12].sort_values('id_percentil', ascending=False)[['etapa_nueva', 'tecnico', 'compat', 'cambio_K', 'id_percentil', 'id_puesto', 'candidatos']]
      .round(2).to_string(index=False))
print(f'Rasgos legibles con reloj ≤ {RELOJ_P4} y peso > 0 (pesos completos): {int((legible & (pesos(T) > 0)).sum())} de {int((pesos(T) > 0).sum())}')

ESCRIBIR = True
if ESCRIBIR:
    assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Freno: Drive no está montado; no se escribe nada'
    EN.to_csv(os.path.join(SAL5, 'c_ensayo_llegadas.csv'), index=False); CB.to_csv(os.path.join(SAL5, 'c_ensayo_cobertura.csv'), index=False)
    print('Guardado: salidas/fw5/c_ensayo_llegadas.csv, c_ensayo_cobertura.csv')

F-1

In [ ]:
# [F-1 v1.0] Familias de rasgos · se definen SIN mirar la portabilidad (S12-02): correlación entre perfiles + lógica de fútbol
# =====================================================================
# Sesión 12 · 2026-10-18 · Correr DESPUÉS de C-0 y C-1 (usa RASGOS, ZR, CF, ET, IX). No imprime ni usa los ejes del mapa.
# Dos fuentes, para comparar (la decisión de cuál manda la toma Arturo con esta salida):
#   (A) Datos: agrupamiento jerárquico (enlace promedio) con distancia 1 − |r| entre perfiles ajustados de etapas de 15+.
#       Se corta en |r| = 0.5, 0.6 y 0.7 (una familia necesita ≥ 2 rasgos). Mismo criterio de redundancia de R5/R6r (|r| ≥ 0.7).
#   (B) Fútbol: familias propuestas por Claude con nombre táctico (lista fija abajo).
# Por familia: integrantes, |r| medio interno, signo de cada integrante (para que todos apunten igual) y confiabilidad del
# compuesto (fórmula de Mosier: 1 − ruido de los integrantes / varianza del promedio). Una familia gana confiabilidad si sus
# integrantes miden lo mismo con ruidos distintos: es lo que da poder a los rasgos "sin dueño demostrable".
# Escribe (con freno de Drive) salidas/fw5/f_familias.csv. La portabilidad por familia va en F-2, después de fijar las familias.
# =====================================================================
import os
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
for v in ['RASGOS', 'ZR', 'CF', 'ET', 'IX']:
    assert v in globals(), f'Falta {v}: corre C-0 y C-1 en esta sesión antes de F-1'
CORTES = [0.5, 0.6, 0.7]
TACTICAS = {
    'presión y bloque alto': ['r4:presion_alta', 'r4:intensidad_ppda', 'r4:contrapresion', 'r4:recuperacion_tras_perdida', 'r7c:altura_laterales',
                              'r7c:altura_contenciones', 'r7c:altura_interiores_mp', 'r7c:altura_delanteros', 'r7c:portero_libero'],
    'control con balón': ['r4:posesion', 'r4:riesgo_pase', 'r4:pase_largo', 'r4:directness', 'r4:saque_meta_corto', 'r4:field_tilt',
                          'r6:pases_hasta_ultimo', 'r6:seg_hasta_ultimo'],
    'transición': ['r6b:contragolpe_propio', 'r6b:mezcla_transicion', 'r6b:jugadores_contra', 'r6b:contra_a_area', 'r4:contragolpe_rival',
                   'r6c:obv_transicion_pct'],
    'banda y centros': ['r6:centros', 'r6:centro_alto', 'r6:entrada_central', 'r6:banda_izquierda', 'r7c:amplitud_bandas'],
    'encare': ['r6b:libertad_encarar', 'r6b:conduccion_larga', 'r6b:centro_tras_regate', 'r6c:reparto_conduccion'],
    'marcaje en córner': ['r8:pct_marca', 'r8:zonales', 'r8:pendiente_marca', 'r8:def_area_360'],
    'cargar el área chica': ['r8:envio_area_chica', 'r8:pb_of_2', 'r8:cerrado_pct', 'r8:pegados_portero'],
    'paciencia del banquillo': ['r7b:minuto_primera_decision', 'r7u:rot_fuera_base', 'r7u:cambio_familia_90'],
}
L15 = [IX[e] for e in ET.loc[ET.partidos >= 15, 'etapa_id']]
X = pd.DataFrame(ZR[L15], columns=RASGOS)
Xs = (X - X.mean()) / X.std()
R = Xs.corr(min_periods=20)
conf = CF.loc[RASGOS, 'conf']

def resumen(miembros):
    m = [r for r in miembros if r in R.index]
    if len(m) < 2: return None
    sub = R.loc[m, m].fillna(0).values
    v = np.linalg.eigh(sub)[1][:, -1]; s = np.sign(v); s[s == 0] = 1
    if s.sum() < 0: s = -s
    comp = (Xs[m] * s).mean(1)
    err = ((1 - conf[m].clip(upper=0.99)).sum()) / len(m) ** 2
    rel = 1 - err / comp.var()
    iu = np.triu_indices(len(m), 1)
    return dict(n=len(m), r_medio=np.abs(sub[iu]).mean(), r_min=np.abs(sub[iu]).min(), conf_compuesto=rel,
                conf_media_integrantes=conf[m].mean(), integrantes=', '.join(('' if si > 0 else '−') + r for r, si in zip(m, s)))

D = 1 - R.abs().fillna(0).values; np.fill_diagonal(D, 0)
Z = linkage(squareform(D, checks=False), method='average')
salida = pd.DataFrame(index=pd.Index(RASGOS, name='rasgo'))
print(f'--- (A) Familias por datos: {len(RASGOS)} rasgos, {len(L15)} etapas de 15+ ---')
for c in CORTES:
    lab = fcluster(Z, t=1 - c, criterion='distance')
    s = pd.Series(lab, index=RASGOS); tam = s.map(s.value_counts())
    salida[f'familia_datos_{c}'] = np.where(tam >= 2, s.astype(str), '—')
    filas = [dict(familia=k, **resumen(g.index.tolist())) for k, g in s[tam >= 2].groupby(s[tam >= 2])]
    print(f'\nCorte |r| ≥ {c}: {len(filas)} familias · rasgos agrupados {int((tam >= 2).sum())} · sueltos {int((tam < 2).sum())}')
    if filas:
        print(pd.DataFrame(filas).sort_values('n', ascending=False).round(2).to_string(index=False))

print('\n--- (B) Familias tácticas propuestas (integrante con "−" = va en sentido contrario al resto) ---')
filas = []
for k, m in TACTICAS.items():
    r = resumen(m)
    fuera = [x for x in m if x not in R.index]
    if r: filas.append(dict(familia=k, **r, fuera_por_confiabilidad=', '.join(fuera) or '—'))
    for x in m:
        if x in salida.index: salida.loc[x, 'familia_tactica'] = k
FT = pd.DataFrame(filas)
print(FT.round(2).to_string(index=False))
print('\nCruce: ¿en qué familias por datos (corte 0.5) caen los integrantes de cada familia táctica?')
for k, m in TACTICAS.items():
    mm = [x for x in m if x in salida.index]
    if mm: print(f"  {k}: {salida.loc[mm, 'familia_datos_0.5'].value_counts().to_dict()}")
print('\nRasgos sin familia táctica:', [r for r in RASGOS if pd.isna(salida.loc[r].get('familia_tactica', np.nan))])
print('\nPares con |r| ≥ 0.7 entre familias tácticas distintas (posibles fusiones):')
ft = salida['familia_tactica'].fillna('—') if 'familia_tactica' in salida else pd.Series('—', index=salida.index)
pares = [(a, ft[a], b, ft[b], round(R.loc[a, b], 2)) for i, a in enumerate(RASGOS) for b in RASGOS[i + 1:]
         if abs(R.loc[a, b]) >= 0.7 and ft[a] != ft[b]]
print(pares or 'ninguno')

ESCRIBIR = True
if ESCRIBIR:
    assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Freno: Drive no está montado; no se escribe nada'
    salida.to_csv(os.path.join(SAL5, 'f_familias.csv'))
    print('\nGuardado: salidas/fw5/f_familias.csv')

F2

In [ ]:
# [F-2 v1.0] Portabilidad por familia · el mapa de dos ejes (C-1) aplicado a las familias fijadas en F-1 (corte |r| ≥ 0.5)
# =====================================================================
# Sesión 12 · 2026-10-19 · Decisiones S12-13 (familias por datos, corte 0.5, nombres tácticos). Correr DESPUÉS de C-0 y C-1
# (usa ejes, clasificar, ZR, ZE, RASGOS, CF, ET, IX, pt, ct, pc, cc, pool). Tarda ~1-2 min.
# Integrantes copiados de la salida de F-1 (corte 0.5). El signo de cada integrante se recalcula igual que en F-1 (primer
# componente principal), para que todos apunten en el mismo sentido.
# Puntaje de familia por etapa = promedio de los integrantes estandarizados (con signo). Confiabilidad = Mosier (como F-1).
# Luego, los mismos dos ejes que C-1 (viaja con el técnico / se queda con el club), corregidos por esa confiabilidad, con IC
# (remuestreo por técnico o por club), placebo, cuadrante con corte 0.5 y firmeza; y la sensibilidad sin Elo.
# Escribe (con freno de Drive) salidas/fw5/f_mapa_familias.csv.
# =====================================================================
import os
import numpy as np
import pandas as pd
for v in ['ejes', 'clasificar', 'ZR', 'ZE', 'RASGOS', 'CF', 'ET', 'IX', 'pt', 'ct', 'pc', 'cc', 'pool', 'T']:
    assert v in globals(), f'Falta {v}: corre C-0 y C-1 en esta sesión antes de F-2'
FAMILIAS = {
    'Dominio territorial': ['r4:posesion', 'r4:field_tilt', 'r4:presion_alta', 'r6:toque_area', 'r6b:jugadores_contra', 'r6b:contra_a_area',
                            'r6c:obv_ultimo_tercio', 'r6c:obv_transicion_pct'],
    'Salida elaborada': ['r4:pase_largo', 'r4:saque_meta_corto', 'r4:riesgo_pase', 'r4:directness', 'r6:pases_hasta_ultimo', 'r6:seg_hasta_ultimo'],
    'Bloque alto y verticalidad': ['r4:contragolpe_rival', 'r6:paso_ultimo', 'r6b:contragolpe_propio', 'r7c:altura_contenciones',
                                   'r7c:altura_delanteros', 'r7c:altura_interiores_mp'],
    'Presión tras pérdida': ['r4:intensidad_ppda', 'r4:contrapresion', 'r4:recuperacion_tras_perdida'],
    'Córner en contra: marcaje': ['r8:pct_marca', 'r8:pendiente_marca', 'r8:zonales'],
    'Córner a favor: cargar el área chica': ['r8:envio_area_chica', 'r8:envio_punto_penal', 'r8:pb_of_2'],
    'Pareja · encare': ['r6b:libertad_encarar', 'r6b:centro_tras_regate'],
    'Pareja · mezcla de progresión y transición': ['r6b:mezcla_transicion', 'r6b:mezcla_progresion'],
    'Pareja · balón parado en corto': ['r8:corto_pct', 'r8:tl_corto'],
    'Pareja · línea de 3 con laterales altos': ['r7u:inicio_linea_3_5', 'r7c:altura_laterales'],
    'Pareja · amplitud': ['r6:entrada_central', 'r7c:amplitud_bandas'],
    'Pareja · progresar conduciendo': ['r6:paso_medio', 'r6c:reparto_conduccion'],
}
L15 = np.array([IX[e] for e in ET.loc[ET.partidos >= 15, 'etapa_id']])
col = {r: j for j, r in enumerate(RASGOS)}
conf_r = CF.loc[RASGOS, 'conf']

def compuesto(Zmat, m, s):
    X = Zmat[:, [col[r] for r in m]]
    mu, sd = np.nanmean(X[L15], 0), np.nanstd(X[L15], 0)
    return np.nanmean((X - mu) / sd * s, 1)

filas, Zf, Zfe, conf_f, nombres = [], [], [], [], []
for nom, m0 in FAMILIAS.items():
    m = [r for r in m0 if r in col]
    if len(m) < 2: print(f'  {nom}: menos de 2 integrantes disponibles, se omite'); continue
    X = pd.DataFrame(ZR[L15][:, [col[r] for r in m]], columns=m)
    R = X.corr().fillna(0).values
    v = np.linalg.eigh(R)[1][:, -1]; s = np.sign(v); s[s == 0] = 1
    if s.sum() < 0: s = -s
    z = compuesto(ZR, m, s)
    err = (1 - conf_r[m].clip(upper=0.99)).sum() / len(m) ** 2
    rel = float(1 - err / np.nanvar(z[L15]))
    Zf.append(z); Zfe.append(compuesto(ZE, m, s)); conf_f.append(rel); nombres.append(nom)
    filas.append(dict(familia=nom, n=len(m), integrantes=', '.join(('' if si > 0 else '−') + r for r, si in zip(m, s)),
                      tec_medio_integrantes=T.loc[m, 'tec'].mean(), club_medio_integrantes=T.loc[m, 'club'].mean()))
Zf, Zfe, conf_f = np.column_stack(Zf), np.column_stack(Zfe), np.array(conf_f)
TF = ejes(Zf, conf_f, pt, ct, pc, cc, pool, B=1000, P=200).set_index(pd.Index(nombres, name='familia'))
TF['cuadrante'], TF['firmeza'] = clasificar(TF)
c_ = np.clip(conf_f, .05, None)
TF['tec_sin_elo'] = r_pares(Zfe, pt) / c_; TF['club_sin_elo'] = r_pares(Zfe, pc) / c_
TF = TF.join(pd.DataFrame(filas).set_index('familia'))
print(f'--- F-2 · Mapa por familia ({len(TF)} familias; corte 0.5; placebo ≈ 0: media {TF.placebo_media.mean():+.2f}) ---')
print('ruido de una familia sin efecto (2 × desv. del placebo): ±' + f'{2 * TF.placebo_de.median():.2f}')
print(TF[['n', 'conf', 'tec', 'tec_lo', 'tec_hi', 'club', 'club_lo', 'club_hi', 'dif_lo', 'dif_hi', 'cuadrante', 'firmeza',
          'tec_sin_elo', 'club_sin_elo', 'tec_medio_integrantes', 'club_medio_integrantes']].round(2).to_string())
print('\nIntegrantes y signos (− = va en sentido contrario al nombre de la familia):')
for k, r in TF.iterrows(): print(f'  {k}: {r.integrantes}')
print('\nResumen por cuadrante:', TF.groupby('cuadrante').size().to_dict(), '· firmes:', TF[TF.firmeza == 'firme'].groupby('cuadrante').size().to_dict())
ESCRIBIR = True
if ESCRIBIR:
    assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Freno: Drive no está montado; no se escribe nada'
    TF.to_csv(os.path.join(SAL5, 'f_mapa_familias.csv'))
    print('Guardado: salidas/fw5/f_mapa_familias.csv')

# Sesión 12

In [ ]:
# R8h v1.0 · Balón parado: emparejamiento aéreo (HOPS) por córner
# =====================================================================
# Sesión 12 (continuación) · 2026-10-19 · Hackathon ISAC 2026, Track 2 · Cinco temporadas (fw5)
# Idea de Arturo: quizá no haya "zonas" de vulnerabilidad, pero sí desventajas de JUGADORES. R8m probó HOPS por equipo-temporada contra
#   el xG por córner y salió ~0, pero esa medida es casi puro ruido (confiabilidad −0.03; buena práctica 17). Aquí se baja al córner
#   individual (16 mil) y se mide el EMPAREJAMIENTO, no el nivel del equipo.
# Unidad: córner (fw5/r8_bp, kind == 'corner'). Diferencia aérea = HOPS aéreo del atacante − HOPS aéreo del defensor en ese partido.
#   HOPS aéreo de un equipo en un partido = media de los 5 mejores HOPS (jugador-temporada, ≥ 10 duelos) entre los que jugaron:
#   COPIA LITERAL de R8m-1 (b), verificada contra sus números (buena práctica 13).
#   Límite declarado: el 360 no identifica jugadores; es la capacidad aérea de los que jugaron ese día, no de los que están en el área.
# Cuartos de la diferencia (cortes de la liga): Q1 = el atacante en gran desventaja aérea … Q4 = en gran ventaja.
# Preguntas fijadas ANTES de correr (aprobadas por Arturo, 2026-10-19):
#   P1 (validación): ¿la diferencia predice quién gana el primer contacto (R8a)? Criterio: Q4 − Q1 > 0 con IC sin 0.
#      Si NO se cumple, la medida no sirve y P2-P3 no se calculan.
#   P2 (ataque, hipótesis de Arturo): con desventaja aérea (Q1), ¿el córner corto vale más frente al directo que con ventaja (Q4)?
#      xG×100 por córner (ventana de R8a), dentro del equipo atacante-temporada. DiD = (corto − directo | Q1) − (corto − directo | Q4).
#   P3 (defensa, hipótesis de Arturo): frente a atacantes más fuertes en el aire, ¿el defensor cambia su organización?
#      (defensores en el área y en el área chica, marca a ≤ 2 yd, zonales de R8z), dentro del equipo defensor-temporada.
#      ¿Le sirve? DiD del xG concedido: (más defensores en el área − menos | atacante mucho más fuerte, Q4) − (lo mismo | parejo, Q2-Q3).
#      Asociación, no causa.
# Apuestas (antes de correr):
#   P2 · Arturo: el corto vale ≥ 0.2 de xG más que el directo en Q1 [unidad por confirmar: el xG medio por córner es 0.032].
#        Claude: DiD +0.5 de xG×100 (0.005 de xG), con IC que incluye 0.
#   P1 · Claude: Q4 − Q1 = +5 puntos porcentuales.  P3 · Claude: defensores en el área Q4 − Q1 = +0.2.
# IC 95% por remuestreo de partidos (500). Semilla 2026.
# Orden: celda 1 → R8h-0 (prueba; no toca Drive) → R8h-1 (corrida real; escribe salidas/fw5/r8h_resultados.csv). ~3-5 min.
# =====================================================================

# [R8h-0 v1.0] Funciones y prueba con datos inventados (un escenario SIN efecto y otro CON efecto)
import numpy as np
import pandas as pd

SEMILLA = 2026
GANA = {'remate_atac', 'control_atac', 'gana_atac'}          # R8a: clases en que el atacante gana el primer contacto

def cuartos(gap, cortes=None):
    gap = np.asarray(gap, dtype=float)
    if cortes is None: cortes = np.nanquantile(gap, [.25, .5, .75])
    q = (np.digitize(gap, cortes) + 1).astype(float); q[np.isnan(gap)] = np.nan
    return q, cortes

def centra(d, col, grupo):
    return d[col] - d.groupby(grupo)[col].transform('mean')

def boot(d, fn, B=500, seed=SEMILLA):
    d = d.reset_index(drop=True)
    rng = np.random.default_rng(seed); g = d.groupby('match_id').indices; mids = np.array(list(g.keys()))
    est = pd.Series(fn(d)); bs = []
    for _ in range(B):
        s = rng.choice(len(mids), len(mids), replace=True)
        bs.append(pd.Series(fn(d.iloc[np.concatenate([g[mids[i]] for i in s])])))
    bs = pd.DataFrame(bs)
    return pd.DataFrame({'valor': est, 'ic_lo': bs.quantile(.025), 'ic_hi': bs.quantile(.975)})

def p1(d, col='gana'):
    m = 100 * d.groupby('q')[col].mean()
    r = {f'Q{k}': m.get(float(k), np.nan) for k in (1, 2, 3, 4)}
    r['Q4 - Q1'] = r['Q4'] - r['Q1']; return r

def p2(d):
    m = d.groupby(['q', 'corto'])['yc'].mean()
    r = {f'Q{k}: corto - directo': m.get((float(k), True), np.nan) - m.get((float(k), False), np.nan) for k in (1, 2, 3, 4)}
    r['DiD (Q1 - Q4)'] = r['Q1: corto - directo'] - r['Q4: corto - directo']; return r

def p3_org(d, cols):
    r = {}
    for c in cols:
        m = d.groupby('q')[c + '_c'].mean()
        for k in (1, 4): r[f'{c} Q{k}'] = m.get(float(k), np.nan)
        r[f'{c} Q4 - Q1'] = r[f'{c} Q4'] - r[f'{c} Q1']
    return r

def p3_sirve(d):
    def dif(m):
        x = d[m]; return x.loc[x.alto, 'yc'].mean() - x.loc[~x.alto, 'yc'].mean()
    r = {'atacante fuerte (Q4): alto - bajo': dif(d.q == 4), 'parejo (Q2-Q3): alto - bajo': dif(d.q.isin([2, 3]))}
    r['DiD (fuerte - parejo)'] = r['atacante fuerte (Q4): alto - bajo'] - r['parejo (Q2-Q3): alto - bajo']; return r

# Prueba: 60 equipos (calidad y tamaño distintos), 250 córners cada uno, 5 córners por partido, diferencia aérea ~ N(0, 10).
# CON efecto: (i) ganar el contacto sube con la diferencia; (ii) el corto vale +2 solo con gran desventaja (gap < −6.7 ≈ Q1);
# (iii) el defensor pone más gente en el área cuando el atacante es más fuerte (+0.05 por punto). "Sirve" no tiene efecto en ninguno.
rng0 = np.random.default_rng(SEMILLA)
def simula(con):
    filas = []
    for e in range(60):
        cal, tam = rng0.normal(0, 1), rng0.normal(0, 1)
        for i in range(250):
            gap = rng0.normal(0, 10); corto = rng0.random() < 0.2
            gana = rng0.random() < 1 / (1 + np.exp(-(-0.1 + (0.06 * gap if con else 0.0))))
            y = 3.3 + cal + (2.0 if (con and corto and gap < -6.7) else 0.0) + rng0.normal(0, 8)
            da = 7 + tam + (0.05 * gap if con else 0.0) + rng0.normal(0, 1)
            filas.append(dict(match_id=f'E{e}-{i // 5}', ats=f'E{e}', dts=f'E{e}', gap=gap, gana=float(gana), corto=corto, y=y, s_def_area=da))
    d = pd.DataFrame(filas); d['q'], _ = cuartos(d.gap)
    d['yc'] = centra(d, 'y', 'ats'); d['s_def_area_c'] = centra(d, 's_def_area', 'dts'); d['alto'] = d.s_def_area_c > 0
    return d
for nombre, con in [('SIN efecto', False), ('CON efecto', True)]:
    d = simula(con)
    print(f'=== {nombre}')
    print('P1:', boot(d, p1, B=200).round(2).loc[['Q1', 'Q4', 'Q4 - Q1']].to_dict('index'))
    print('P2:', boot(d, p2, B=200).round(2).loc[['DiD (Q1 - Q4)']].to_dict('index'))
    print('P3 org:', boot(d, lambda x: p3_org(x, ['s_def_area']), B=200).round(2).loc[['s_def_area Q4 - Q1']].to_dict('index'))
    print('P3 sirve:', boot(d, p3_sirve, B=200).round(2).loc[['DiD (fuerte - parejo)']].to_dict('index'))
print('Esperado · SIN: los cuatro ≈ 0 con IC que incluye 0. · CON: P1 Q4 − Q1 ≈ +50 puntos; P2 DiD ≈ +2; P3 org ≈ +1.3; P3 sirve ≈ 0 (IC con 0).')
print('Si sale así, corre R8h-1.')


# [R8h-1 v1.0] Corrida real
import os, time
RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
assert 'p1' in globals() and 'boot' in globals(), 'Corre primero R8h-0'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5, LIGA = os.path.join(DP, 'fw5'), os.path.join(DP, 'liga')
SAL5 = os.path.join(RAIZ, 'salidas', 'fw5'); os.makedirs(SAL5, exist_ok=True)
t_ini = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
print('Temporadas en r4_partidos (fw5):', sorted(Mm.temporada.unique()))

# ---------- Capacidad aérea por equipo-partido: COPIA LITERAL de R8m-1 (b) ----------
HJ = pd.read_parquet(os.path.join(FW5, 'r8m_hops_jugador.parquet'))
JP = pd.read_parquet(os.path.join(FW5, 'r7_jugador_partido.parquet'))
pcol = next(c for c in ['player_id', 'player.id', 'jugador_id', 'jugador'] if c in JP.columns)
mcol = next(c for c in ['minutos', 'min', 'minutes'] if c in JP.columns)
JP = JP[JP[mcol] > 0].rename(columns={pcol: 'player_id'})
JP['temporada'] = JP.match_id.map(loc['temporada'])
JP = JP.merge(HJ[HJ.duelos >= 10][['player_id', 'temporada', 'hops']], on=['player_id', 'temporada'], how='left')
alt = []
for mid in Mm.match_id:
    r = os.path.join(LIGA, 'lineups', f'{mid}.parquet')
    if os.path.exists(r): alt.append(pd.read_parquet(r, columns=['player_id', 'player_height']))
ALT = pd.concat(alt).dropna().query('player_height > 0').drop_duplicates('player_id')
JP = JP.merge(ALT, on='player_id', how='left')
top5 = lambda s: s.dropna().nlargest(5).mean() if s.notna().sum() >= 5 else np.nan
TM = JP.groupby(['match_id', 'team']).agg(aereo_hops=('hops', top5), aereo_altura=('player_height', top5)).reset_index()
r_ha = TM[['aereo_hops', 'aereo_altura']].corr().iloc[0, 1]
ok = abs(r_ha - 0.35) <= 0.01 and len(HJ) == 2386
print(f'--- (0) Chequeo contra R8m: jugador-temporadas con HOPS {len(HJ):,} (R8m: 2,386) · correlación HOPS-estatura por equipo-partido'
      f' {r_ha:.2f} (R8m: 0.35) → {"REPRODUCE" if ok else "NO REPRODUCE: pega esta línea antes de leer lo demás"}')

# ---------- Córners con diferencia aérea ----------
BP = pd.read_parquet(os.path.join(FW5, 'r8_bp.parquet'))
co = BP[BP.kind == 'corner'].copy()
co['temporada'] = co.match_id.map(loc['temporada'])
lo, vi = co.match_id.map(loc['local']), co.match_id.map(loc['visita'])
co['rival'] = np.where(co.team == lo, vi, lo)
h = TM.set_index(['match_id', 'team'])['aereo_hops']
co['h_at'] = h.reindex(pd.MultiIndex.from_arrays([co.match_id, co.team])).values
co['h_def'] = h.reindex(pd.MultiIndex.from_arrays([co.match_id, co.rival])).values
co['gap'] = co.h_at - co.h_def
co['q'], CORTES = cuartos(co.gap)
print(f'--- (1) Córners {len(co):,} · con diferencia aérea {100 * co.gap.notna().mean():.1f}% · diferencia de HOPS: p10 {co.gap.quantile(.1):.1f}'
      f' · mediana {co.gap.median():.1f} · p90 {co.gap.quantile(.9):.1f} · cortes de los cuartos {np.round(CORTES, 1).tolist()}')
co = co[co.gap.notna()].copy()
co['ats'] = co.team + ' · ' + co.temporada; co['dts'] = co.rival + ' · ' + co.temporada
RES = {}

# ---------- P1: validación ----------
ce = co[co.ruta.isin(['directo', 'via_corto']) & co.contacto_clase.notna()].copy()
ce['gana'] = ce.contacto_clase.isin(GANA).astype(float)
ce['gana_c'] = centra(ce, 'gana', 'ats') + ce.gana.mean()
RES['P1 bruto'] = boot(ce, p1); RES['P1 dentro del atacante'] = boot(ce, lambda d: p1(d, 'gana_c'))
print(f'\n--- P1 · % de primeros contactos que gana el atacante, por cuarto de diferencia aérea ({len(ce):,} córners con envío y contacto)')
print('  Bruto:'); print(RES['P1 bruto'].round(1).to_string())
print('  Dentro del equipo atacante-temporada (quita que los equipos altos sean distintos en todo):'); print(RES['P1 dentro del atacante'].round(1).to_string())
VALIDA = RES['P1 bruto'].loc['Q4 - Q1', 'ic_lo'] > 0
print(f'  → P1 {"SE CUMPLE: la medida distingue quién gana el aire; se leen P2 y P3" if VALIDA else "NO se cumple: la medida no sirve; P2 y P3 no se calculan (preregistro)"}')

if VALIDA:
    # ---------- P2: ataque ----------
    c2 = co[co.ruta.isin(['directo', 'via_corto', 'reciclado'])].copy()
    c2['corto'] = c2.ruta != 'directo'; c2['y'] = 100 * (c2.xg1 + c2.xg2); c2['yc'] = centra(c2, 'y', 'ats')
    desc = c2.groupby('q').apply(lambda d: pd.Series({
        'corners': len(d), 'pct_corto': 100 * d.corto.mean(),
        'xg100_directo': d.loc[~d.corto, 'y'].mean(), 'xg100_corto': d.loc[d.corto, 'y'].mean(),
        'pct_xg_en_segunda_directos': 100 * d.loc[~d.corto, 'xg2'].sum() / (d.loc[~d.corto, 'xg1'] + d.loc[~d.corto, 'xg2']).sum()}),
        include_groups=False)
    print(f'\n--- P2 · Ataque ({len(c2):,} córners sin tiro directo). Descriptivo por cuarto (Q1 = atacante en gran desventaja):')
    print(desc.round(2).to_string())
    RES['P2'] = boot(c2, p2)
    print('  Corto − directo, xG×100 por córner, dentro del equipo atacante-temporada (IC 95% por partidos):')
    print(RES['P2'].round(2).to_string())
    print('  Lectura: DiD > 0 con IC sin 0 = con desventaja aérea el corto gana valor relativo (hipótesis de Arturo).')

    # ---------- P3: defensa ----------
    sf = co['s_frame'] if 's_frame' in co else pd.Series(False, index=co.index)
    d3 = co[(co.ruta == 'directo') & sf.fillna(False).astype(bool)].copy()
    d3['marca2'] = np.where(d3.s_marc_den > 0, 100 * d3.s_marc2 / d3.s_marc_den.where(d3.s_marc_den > 0), np.nan)
    cols = ['s_def_area', 's_def_chica', 'marca2']
    Z = pd.read_parquet(os.path.join(FW5, 'r8z_marcaje.parquet'))
    if {'match_id', 'index', 'zonales_b'} <= set(Z.columns):
        d3 = d3.merge(Z[['match_id', 'index', 'zonales_b']].drop_duplicates(['match_id', 'index']), on=['match_id', 'index'], how='left')
        cols.append('zonales_b'); print(f'\n  zonales de R8z unidos: {100 * d3.zonales_b.notna().mean():.1f}% de los córners con foto')
    else:
        print('\n  aviso: r8z_marcaje no tiene match_id + index + zonales_b; P3 sigue sin zonales. Columnas:', list(Z.columns)[:15])
    for c in cols: d3[c + '_c'] = d3[c] - d3.groupby('dts')[c].transform('mean')
    print(f'--- P3 · Organización defensiva frente a la diferencia aérea ({len(d3):,} córners directos con foto; Q4 = atacante mucho más fuerte).')
    print('  Medias crudas por cuarto:', d3.groupby('q')[cols].mean().round(2).to_dict('index'))
    RES['P3 organizacion'] = boot(d3, lambda d: p3_org(d, cols))
    print('  Dentro del equipo defensor-temporada (cuánto se aparta de su costumbre):'); print(RES['P3 organizacion'].round(2).to_string())
    d3['y'] = 100 * (d3.xg1 + d3.xg2); d3['yc'] = centra(d3, 'y', 'dts'); d3['alto'] = d3.s_def_area_c > 0
    RES['P3 sirve'] = boot(d3, p3_sirve)
    print('  ¿Le sirve? xG×100 concedido por córner, defensores en el área por encima (alto) o por debajo (bajo) de su costumbre:')
    print(RES['P3 sirve'].round(2).to_string())
    print('  Lectura: DiD < 0 con IC sin 0 = poner más gente en el área ayuda más justo contra atacantes fuertes. Asociación.')

pd.concat(RES, names=['bloque', 'estadistico']).to_csv(os.path.join(SAL5, 'r8h_resultados.csv'))
print(f'\nR8h-1 terminó en {time.time() - t_ini:.0f} s. Guardado: salidas/fw5/r8h_resultados.csv. Pega toda la salida.')

In [ ]:
# K1 v1.0 · Concordancia con las estadísticas de temporada de StatsBomb (team-stats, API v4.0.0)
# =====================================================================
# Sesión 12 (continuación) · 2026-10-19 · Hackathon ISAC 2026, Track 2
# Para qué: verificación cruzada, prevista desde el inicio (statsbomb_api_1.md, sección 5: "los agregados sirven para verificación
#   cruzada"). Nuestras métricas salen de eventos con definiciones propias (framework) y se separan por etapa de técnico; las del
#   proveedor son por equipo y temporada y con sus definiciones. Si coinciden por equipo-temporada, el reporte puede decir:
#   "nuestras medidas coinciden con las del proveedor, y además separan a los técnicos, cosa que el proveedor no hace".
# NO entra al análisis: solo compara. 2026/2027 queda EXCLUIDA (preregistro de Almada, limpieza D-2).
# Credenciales: Secretos de Colab, nunca en el código. Si los tuyos se llaman distinto, agrega sus nombres en NOMBRES_SECRETO.
# Apuesta (antes de correr), Spearman entre nuestra intensidad de presión y el PPDA del proveedor (signo negativo esperado,
#   porque nosotros contamos acciones por 100 pases y ellos pases por acción, en la misma zona del 40% del campo):
#   Claude −0.92 · Arturo: ___
# Orden: celda 1 → K1. < 1 min. Escribe datos_crudos/team_stats_73_<season>.json, fw5/k_team_stats.parquet, salidas/fw5/k_concordancia.csv.
# =====================================================================

# [K1 v1.0] Descarga de team-stats y concordancia por equipo-temporada
import os, json, time, requests
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from google.colab import userdata

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5')
DC = os.path.join(RAIZ, 'datos_crudos'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5'); os.makedirs(SAL5, exist_ok=True)
EXCLUIR = {'2026/2027'}
t_ini = time.time()

# ---------- Credenciales ----------
NOMBRES_SECRETO = {'usuario': ['SB_USER', 'SB_USERNAME', 'STATSBOMB_USER', 'STATSBOMB_USERNAME', 'SB_EMAIL'],
                   'clave': ['SB_PASS', 'SB_PASSWORD', 'STATSBOMB_PASS', 'STATSBOMB_PASSWORD']}
def secreto(ops):
    for n in ops:
        try:
            v = userdata.get(n)
            if v: return n, v
        except Exception:
            pass
    return None, None
nu, USER = secreto(NOMBRES_SECRETO['usuario']); nc, PASS = secreto(NOMBRES_SECRETO['clave'])
assert USER and PASS, f'No encontré credenciales en los Secretos de Colab. Probé {NOMBRES_SECRETO}. Agrega los nombres que usaste en R0.'
print('Credenciales leídas de los secretos:', nu, '/', nc)
AUTH, API = (USER, PASS), 'https://data.statsbombservices.com/api/'

# ---------- Temporadas ----------
comp = None
for v in ['v4', 'v2', 'v1']:
    r = requests.get(f'{API}{v}/competitions', auth=AUTH, timeout=60)
    print(f'  competitions {v}: HTTP {r.status_code}')
    if r.status_code == 200: comp = pd.DataFrame(r.json()); break
assert comp is not None, 'No se pudo leer competitions'
L = comp[comp.competition_id == 73][['season_id', 'season_name']].drop_duplicates().sort_values('season_name')
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
NUESTRAS = sorted(set(Mm.temporada) - EXCLUIR)
SEAS = L[L.season_name.isin(NUESTRAS)]
print('Liga MX en la API:', L.to_dict('records'))
print('Nuestras temporadas:', NUESTRAS, '· sin season_id en la API:', sorted(set(NUESTRAS) - set(SEAS.season_name)) or 'ninguna')

# ---------- Descarga (caché en datos_crudos) ----------
piezas = []
for sid, sname in zip(SEAS.season_id, SEAS.season_name):
    ruta = os.path.join(DC, f'team_stats_73_{sid}.json')
    if os.path.exists(ruta):
        with open(ruta) as f: js = json.load(f)
    else:
        r = requests.get(f'{API}v4/competitions/73/seasons/{sid}/team-stats', auth=AUTH, timeout=120)
        print(f'  team-stats {sname}: HTTP {r.status_code}'); r.raise_for_status(); js = r.json()
        with open(ruta, 'w') as f: json.dump(js, f)
    d = pd.json_normalize(js); d['temporada'] = sname; piezas.append(d)
TS = pd.concat(piezas, ignore_index=True)
assert not TS.temporada.isin(EXCLUIR).any(), '2026/2027 no debe entrar (preregistro)'
TS.to_parquet(os.path.join(FW5, 'k_team_stats.parquet'), index=False)
print(f'--- (0) team-stats: {len(TS)} equipo-temporadas · {TS.shape[1]} columnas · por temporada {TS.temporada.value_counts().sort_index().to_dict()}')

# ---------- Nuestras métricas por equipo-temporada (crudas, desde los tramos de R4) ----------
ruta_tr = next((r for r in [os.path.join(FW5, 'r4_tramos_v21.parquet'), os.path.join(FW1, 'r4_tramos_v21.parquet')] if os.path.exists(r)), None)
assert ruta_tr, 'No encontré r4_tramos_v21 en fw5 ni en fw1'
print('Tramos de R4 leídos de:', ruta_tr, '(si es fw1, solo hay 4 temporadas)')
TR = pd.read_parquet(ruta_tr, columns=['metrica', 'match_id', 'team', 'num', 'den'])
TR['temporada'] = TR.match_id.map(loc['temporada']); TR = TR[TR.temporada.isin(NUESTRAS)]
N = TR.groupby(['metrica', 'team', 'temporada'])[['num', 'den']].sum()
N = (N.num / N.den).unstack('metrica')
lado = pd.concat([Mm[['match_id', 'local', 'temporada']].rename(columns={'local': 'team'}),
                  Mm[['match_id', 'visita', 'temporada']].rename(columns={'visita': 'team'})])
pj = lado.groupby(['team', 'temporada']).match_id.nunique()
BP = pd.read_parquet(os.path.join(FW5, 'r8_bp.parquet')); co = BP[BP.kind == 'corner'].copy()
co['temporada'] = co.match_id.map(loc['temporada'])
N['corners_pg'] = co.groupby(['team', 'temporada']).size() / pj
N['xg_por_corner'] = co.groupby(['team', 'temporada']).apply(lambda d: (d.xg1 + d.xg2).mean(), include_groups=False)

S = TS.rename(columns={'team_name': 'team'}).set_index(['team', 'temporada'])
dp, dpc = 'team_season_deep_progressions_pg', 'team_season_deep_progressions_conceded_pg'
if dp in S and dpc in S: S['tilt_sb'] = 100 * S[dp] / (S[dp] + S[dpc])
J = N.join(S, how='inner')
sin_par_n = sorted(set(N.index.get_level_values(0)) - set(S.index.get_level_values(0)))
sin_par_s = sorted(set(S.index.get_level_values(0)) - set(N.index.get_level_values(0)))
print(f'--- (1) Equipo-temporadas unidas: {len(J)} de {len(N)} nuestras y {len(S)} del proveedor')
if sin_par_n or sin_par_s: print('  Nombres sin pareja · nuestros:', sin_par_n, '· proveedor:', sin_par_s)

# ---------- Concordancia ----------
PARES = [  # (nuestra, proveedor, signo esperado, qué se compara)
    ('posesion', 'team_season_possession', +1, 'misma idea; nosotros por tiempo de posesión'),
    ('intensidad_ppda', 'team_season_ppda', -1, 'misma zona (40%); nosotros acciones por 100 pases, ellos pases por acción'),
    ('presion_alta', 'team_season_fhalf_pressures_ratio', +1, 'nosotros fases que arrancan en el último tercio; ellos presiones en campo rival'),
    ('contrapresion', 'team_season_counterpressures_pg', +1, 'nosotros % de pérdidas con contrapresión; ellos conteo por partido'),
    ('recuperacion_tras_perdida', 'team_season_counterpressure_regains_pg', +1, 'nosotros % de pérdidas recuperadas ≤ 10 s; ellos conteo'),
    ('contragolpe_rival', 'team_season_counter_attacking_shots_conceded_pg', +1, 'nosotros % de pérdidas que acaban en contra; ellos tiros'),
    ('directness', 'team_season_avg_team_possession_directness', +1, 'misma idea, todas las posesiones'),
    ('directness', 'team_season_directness', +1, 'ellos solo posesiones que terminan en tiro'),
    ('progresion', 'team_season_deep_progressions_pg', +1, 'nosotros por 10 min de posesión; ellos por partido'),
    ('field_tilt', 'tilt_sb', +1, 'ellos: progresiones al último tercio propias / (propias + concedidas)'),
    ('saque_meta_corto', 'team_season_gk_pass_distance', -1, 'más saque corto → pase del portero más corto'),
    ('corners_pg', 'team_season_corners_pg', +1, 'conteo: debe ser casi idéntico'),
    ('xg_por_corner', 'team_season_xg_per_corner', +1, 'nosotros ventana de 10 s tras el contacto; ellos tiros de córner'),
]
filas = []
for a, b, s, nota in PARES:
    if a not in J or b not in J:
        filas.append(dict(nuestra=a, proveedor=b, n=0, spearman=np.nan, veredicto='falta columna', nota=nota)); continue
    x = J[[a, b]].dropna(); rho = spearmanr(x[a], x[b]).correlation
    v = s * rho
    filas.append(dict(nuestra=a, proveedor=b, n=len(x), spearman=round(rho, 2),
                      veredicto='coincide' if v >= 0.8 else 'parcial (definición distinta)' if v >= 0.5 else 'REVISAR', nota=nota))
K = pd.DataFrame(filas)
print('\n--- (2) Concordancia por equipo-temporada (Spearman; coincide ≥ 0.8 en la dirección esperada · parcial 0.5-0.8 · revisar < 0.5)')
print(K.to_string(index=False))
K.to_csv(os.path.join(SAL5, 'k_concordancia.csv'), index=False)

if 'team_season_ppda' in J and 'intensidad_ppda' in J:
    x = J[['intensidad_ppda', 'team_season_ppda']].dropna()
    dif = (x.intensidad_ppda.rank() - (-x.team_season_ppda).rank()).abs().sort_values(ascending=False)
    print('\n  Presión: las 3 equipo-temporadas donde más difiere el orden (para revisar a mano):')
    print(x.loc[dif.index[:3]].round(2).assign(dif_de_puesto=dif.iloc[:3].values).to_string())

# ---------- Variable poco usada (banco): tiempo de balón en juego ----------
c = 'team_season_ball_in_play_time'
if c in TS:
    print(f'\n--- (3) Tiempo de balón en juego (proveedor; solo descriptivo): p10 {TS[c].quantile(.1):.1f} · mediana {TS[c].median():.1f}'
          f' · p90 {TS[c].quantile(.9):.1f}')
    am = TS[TS.team_name.str.contains('Am[eé]rica', regex=True, na=False)]
    print('  América por temporada:', dict(zip(am.temporada, am[c].round(1))))
print(f'\nK1 terminó en {time.time() - t_ini:.0f} s. Guardado: fw5/k_team_stats, salidas/fw5/k_concordancia.csv. Pega toda la salida.')

In [ ]:
# K1b v1.0 · Concordancia: ¿las tres "REVISAR" de K1 son tasa frente a volumen?
# =====================================================================
# Sesión 12 (continuación) · 2026-10-19 · Hackathon ISAC 2026, Track 2
# K1 dejó tres pares bajos: recuperación tras pérdida (0.02), contragolpe rival (0.26) y directness del proveedor solo en
#   posesiones con tiro (−0.10; explicado por diseño: la versión equivalente da 0.93, no se repite aquí).
# Hipótesis: las nuestras son TASAS (% de nuestras pérdidas) y las del proveedor son VOLÚMENES por partido.
# Prueba, comparando lo comparable:
#   (a) nuestro volumen por partido (conteo de casos / partidos jugados) contra su conteo por partido;
#   (b) su tasa (su conteo / sus contrapresiones o / sus posesiones) contra nuestra tasa.
# Criterio fijado ANTES de correr: "explicado por definición" si (a) ≥ 0.8 o alguna tasa de (b) ≥ 0.5 en la dirección esperada;
#   si no, "investigar la definición" (y no se reporta como concordante).
# Apuesta: Claude · volumen de recuperación 0.75, volumen de contragolpe 0.55 · Arturo: ___
# Orden: celda 1 → K1b (usa la caché de K1: fw5/k_team_stats.parquet). < 30 s. No descarga nada.
# =====================================================================

# [K1b v1.0]
import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5 = os.path.join(DP, 'fw5'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
TS = pd.read_parquet(os.path.join(FW5, 'k_team_stats.parquet'))
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
TEMPS = sorted(TS.temporada.unique())
TR = pd.read_parquet(os.path.join(FW5, 'r4_tramos_v21.parquet'), columns=['metrica', 'match_id', 'team', 'num', 'den'])
TR['temporada'] = TR.match_id.map(loc['temporada']); TR = TR[TR.temporada.isin(TEMPS)]
lado = pd.concat([Mm[['match_id', 'local', 'temporada']].rename(columns={'local': 'team'}),
                  Mm[['match_id', 'visita', 'temporada']].rename(columns={'visita': 'team'})])
pj = lado.groupby(['team', 'temporada']).match_id.nunique()
G = TR.groupby(['metrica', 'team', 'temporada'])[['num', 'den']].sum()
vol = lambda m: (G.loc[m, 'num'] / 100) / pj.reindex(G.loc[m].index)     # num = 100 × caso (R4a)
tasa = lambda m: G.loc[m, 'num'] / G.loc[m, 'den']

S = TS.rename(columns={'team_name': 'team'}).set_index(['team', 'temporada'])
pos = S['team_season_possessions']
print(f'Posesiones del proveedor: mediana {pos.median():.1f} → se toman como', 'totales de temporada (se dividen entre partidos)' if pos.median() > 500 else 'por partido')
pos_pg = pos / S['team_season_matches'] if pos.median() > 500 else pos
reg, cp, cac = S['team_season_counterpressure_regains_pg'], S['team_season_counterpressures_pg'], S['team_season_counter_attacking_shots_conceded_pg']

PRUEBAS = [  # (métrica nuestra, versión, nuestra serie, serie del proveedor, tipo)
    ('recuperacion_tras_perdida', '(a) volumen: nuestras recuperaciones por partido vs sus recuperaciones tras contrapresión', vol('recuperacion_tras_perdida'), reg, 'a'),
    ('recuperacion_tras_perdida', '(b) tasa: nuestra % vs sus recuperaciones / sus contrapresiones', tasa('recuperacion_tras_perdida'), reg / cp, 'b'),
    ('recuperacion_tras_perdida', '(b) tasa: nuestra % vs sus recuperaciones / sus posesiones', tasa('recuperacion_tras_perdida'), reg / pos_pg, 'b'),
    ('contragolpe_rival', '(a) volumen: contragolpes rivales por partido vs sus tiros de contra concedidos', vol('contragolpe_rival'), cac, 'a'),
    ('contragolpe_rival', '(b) tasa: nuestra % vs sus tiros de contra concedidos / sus posesiones', tasa('contragolpe_rival'), cac / pos_pg, 'b'),
    ('contrapresion', '(a) volumen (control: en K1 dio 0.78 como tasa)', vol('contrapresion'), cp, 'a'),
]
filas = []
for m, v, a, b, t in PRUEBAS:
    x = pd.concat([a.rename('nuestra'), b.rename('proveedor')], axis=1).dropna()
    filas.append(dict(metrica=m, version=v, n=len(x), spearman=round(spearmanr(x.nuestra, x.proveedor).correlation, 2), tipo=t))
K = pd.DataFrame(filas)
print('\n--- (1) Concordancia comparando lo comparable (Spearman por equipo-temporada)')
print(K.drop(columns='tipo').to_string(index=False))
print('\n--- (2) Veredicto por métrica (criterio fijado antes de correr)')
for m, g in K.groupby('metrica', sort=False):
    ok = (g.loc[g.tipo == 'a', 'spearman'] >= 0.8).any() or (g.loc[g.tipo == 'b', 'spearman'] >= 0.5).any()
    print(f'  {m}: {"EXPLICADO POR DEFINICIÓN (tasa frente a volumen)" if ok else "INVESTIGAR LA DEFINICIÓN"}')
K.to_csv(os.path.join(SAL5, 'k1b_concordancia_tasas.csv'), index=False)
print('\nK1b terminó. Guardado: salidas/fw5/k1b_concordancia_tasas.csv. Pega toda la salida.')

In [ ]:
# D1 v1.0 · ¿Cuánto dura un técnico en la Liga MX y cuántos alcanzan el reloj de su idea?
# =====================================================================
# Sesión 12 (continuación) · 2026-10-19 · Hackathon ISAC 2026, Track 2
# Gancho de E2 (Arturo): si muchos técnicos salen antes de que su idea pueda confirmarse con números, la directiva decide con ruido.
#   ESPN reporta ~39 partidos de media (12 para técnicos mexicanos). Aquí se calcula con NUESTROS datos (trazable); ESPN queda como contexto.
#   La división por nacionalidad no se puede replicar: no tenemos ese dato.
# Unidad: etapa (equipo + etapa de partido_tecnico_periodo, D-5). Solo cuenta partidos jugados por etapa: no lee eventos ni ninguna
#   conducta, así que usar las fechas de 2026/27 no toca el preregistro de Almada (mismo criterio que D5V).
# Censura (las etapas cortadas por los bordes de los datos no dicen cuánto duraron):
#   izquierda = la primera etapa de cada equipo en los datos (empezó antes de 2021/22);
#   derecha  = la última etapa de cada equipo (sigue en curso). "Completas" = ninguna de las dos.
# Interinos (rol == 'interino', D-5) fuera de la cuenta principal; se reportan aparte.
# Relojes de lectura (partidos para confiabilidad 0.7; R5 y R8u con cinco temporadas): marca del córner 5 · presión alta 10 ·
#   intensidad de presión 21 · libertad posicional 30 · cambios de sistema 38.
# Apuesta (antes de correr): Claude · mediana de las etapas completas 24 partidos; 45% terminan antes de 21 · Arturo: ___
# Orden: celda 1 → D1. < 10 s.
# =====================================================================

# [D1 v1.0]
import os
import numpy as np
import pandas as pd

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5'); os.makedirs(SAL5, exist_ok=True)
RELOJES = {'marca del córner': 5, 'presión alta': 10, 'intensidad de presión': 21, 'libertad posicional': 30, 'cambios de sistema': 38}

ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
print('Columnas:', list(ptp.columns))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)].copy()
ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
E = ptp.groupby('etapa_id').agg(team=('team', 'first'), tecnico=('tec_p', 'first'), partidos=('match_id', 'nunique'),
                                inicio=('fecha', 'min'), fin=('fecha', 'max'),
                                interino=('rol', lambda s: (s == 'interino').all()) if 'rol' in ptp else ('team', lambda s: False))
lim = ptp.groupby('team').fecha.agg(['min', 'max'])
E['censura_izq'] = E.inicio == E.team.map(lim['min'])
E['censura_der'] = E.fin == E.team.map(lim['max'])
E['completa'] = ~E.censura_izq & ~E.censura_der
print(f'--- (0) Etapas: {len(E)} · interinas {int(E.interino.sum())} · censura izquierda {int(E.censura_izq.sum())} · '
      f'derecha (en curso) {int(E.censura_der.sum())} · completas no interinas {int((E.completa & ~E.interino).sum())}')

def resumen(x, nombre):
    p = x.partidos
    fila = dict(grupo=nombre, etapas=len(x), media=round(p.mean(), 1), mediana=p.median(), p25=p.quantile(.25), p75=p.quantile(.75))
    for k, r in RELOJES.items(): fila[f'% antes de {r} ({k})'] = round(100 * (p < r).mean(), 1)
    return fila
T = pd.DataFrame([resumen(E[E.completa & ~E.interino], 'completas, sin interinos (principal)'),
                  resumen(E[~E.interino], 'todas sin interinos (subestima: incluye cortadas)'),
                  resumen(E[E.completa], 'completas, con interinos')])
print('\n--- (1) Duración de las etapas (partidos jugados, todas las competiciones de liga)')
print(T.T.to_string(header=False))
am = E[E.team.str.contains('Am[eé]rica', regex=True)].sort_values('inicio')
print('\n--- (2) América:', am[['tecnico', 'partidos', 'completa', 'interino']].to_dict('records'))
E.reset_index().to_csv(os.path.join(SAL5, 'd1_duracion_etapas.csv'), index=False)
print('\nD1 terminó. Guardado: salidas/fw5/d1_duracion_etapas.csv. Pega toda la salida.')

In [ ]:
# D1b v1.0 · Duración de las etapas por nacionalidad del técnico (réplica propia del dato de ESPN: ~39 frente a ~12 partidos)
# =====================================================================
# Sesión 12 (continuación) · 2026-10-19 · Hackathon ISAC 2026, Track 2
# Requiere D1 (salidas/fw5/d1_duracion_etapas.csv). La nacionalidad sale de los técnicos (managers) en las tablas de partidos
#   (datos_crudos/matches_73_*.json* o datos_parquet/matches_73_*.parquet, misma lectura que R4a) y se une por tec_id_p.
#   Si la estructura no trae `managers`, la celda lo dice y se detiene (Arturo indica de dónde sale).
# Apuesta (antes de correr): Claude · mediana de etapas completas: mexicanos 18, extranjeros 28 · Arturo: ___
# Orden: celda 1 → D1 → D1b. < 30 s.
# =====================================================================

#[D1b v1.0]
import os, glob, gzip, json, re
import numpy as np
import pandas as pd
RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP, DC, SAL5 = os.path.join(RAIZ, 'datos_parquet'), os.path.join(RAIZ, 'datos_crudos'), os.path.join(RAIZ, 'salidas', 'fw5')
RELOJES = {'presión alta': 10, 'intensidad de presión': 21, 'libertad posicional': 30, 'cambios de sistema': 38}

filas = []
for ruta in sorted(glob.glob(os.path.join(DC, 'matches_73_*.json*')) + glob.glob(os.path.join(DP, 'matches_73_*.parquet'))):
    if ruta.endswith('.parquet'): recs = pd.read_parquet(ruta).to_dict('records')
    else:
        with (gzip.open if ruta.endswith('.gz') else open)(ruta, 'rt', encoding='utf-8') as f: recs = json.load(f)
    for r in recs:
        for lado in ['home_team', 'away_team']:
            ms = r.get(f'{lado}.managers', None) if f'{lado}.managers' in r else (r.get(lado) or {}).get('managers') if isinstance(r.get(lado), dict) else None
            if isinstance(ms, str): ms = json.loads(ms)
            for m in (list(ms) if ms is not None and not isinstance(ms, float) else []):
                c = m.get('country') or {}
                filas.append(dict(tec_id=m.get('id'), nombre=m.get('name'), pais=c.get('name') if isinstance(c, dict) else c))
MG = pd.DataFrame(filas).dropna(subset=['tec_id']).drop_duplicates('tec_id')
assert len(MG), 'Las tablas de partidos no traen managers: dime de dónde sale la nacionalidad de los técnicos'
print(f'--- (0) Técnicos con nacionalidad: {len(MG)} · sin país {int(MG.pais.isna().sum())} · países más comunes {MG.pais.value_counts().head(6).to_dict()}')

ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
tid = ptp.groupby('etapa_id').tec_id_p.first()
E = pd.read_csv(os.path.join(SAL5, 'd1_duracion_etapas.csv'))
E['tec_id'] = E.etapa_id.map(tid)
E = E.merge(MG[['tec_id', 'pais']], on='tec_id', how='left')
E['grupo'] = np.where(E.pais.isna(), 'sin dato', np.where(E.pais == 'Mexico', 'mexicano', 'extranjero'))
P = E[E.completa & ~E.interino]
def res(x):
    s = dict(etapas=len(x), media=round(x.partidos.mean(), 1), mediana=x.partidos.median())
    for k, r in RELOJES.items(): s[f'% antes de {r} ({k})'] = round(100 * (x.partidos < r).mean(), 1)
    return pd.Series(s)
print(f'\n--- (1) Etapas completas sin interinos por nacionalidad del técnico ({len(P)})')
print(P.groupby('grupo').apply(res, include_groups=False).T.to_string())
print('\n  Con interinos y etapas cortadas (para comparar con un conteo tipo prensa):')
print(E.groupby('grupo').partidos.agg(['size', 'mean', 'median']).round(1).to_string())
E.to_csv(os.path.join(SAL5, 'd1b_duracion_nacionalidad.csv'), index=False)
print('\nD1b terminó. Guardado: salidas/fw5/d1b_duracion_nacionalidad.csv. Pega toda la salida.')

In [ ]:
# L1 v1.0 · ¿"Quién juega" mueve los rasgos de la plantilla? Dos palancas: la alineación (técnico) y la renovación (mercado)
# =====================================================================
# Sesión 12 (continuación) · 2026-10-19 · Hackathon ISAC 2026, Track 2
# Origen (Arturo): el mapa dice que territorio y posesión "se quedan con el club" y que sin Elo caen a ~0 (nivel de plantilla).
#   Si es así, deben moverse con QUIÉN JUEGA. Eso no está medido. Dos pruebas, fijadas antes de correr:
#   (a1) Palanca del técnico (alineación): dentro de una misma etapa y torneo, en los partidos en que el once se aleja del XI base
#        (titulares fuera de los 11 con más titularidades: `fuera_base`, R7a), ¿cambian field tilt y posesión?
#        Residuo de R4 por partido (ya ajustado por rival, Elo del rival, localía, marcador y cuarto), centrado dentro de etapa × torneo,
#        con favorito y descanso corto como controles. Pendiente con IC 95% por remuestreo de partidos (500).
#   (a2) Palanca del mercado (renovación): en cada relevo, continuidad = % de los minutos de los primeros 10 partidos del técnico
#        nuevo jugados por quienes tuvieron minutos en los últimos 18 del anterior (ventana S12-05). Cambio = residuo medio de los
#        primeros 10 − residuo medio de los últimos 18. ¿Menos continuidad → más cambio de territorio? Spearman con IC.
#        Descriptivo de Arturo: relevos con mercado abierto (llegada en jun-jul o dic-ene) frente a relevos a media temporada.
# Criterio: la palanca "existe" si el IC excluye 0 en la dirección esperada (a1: pendiente < 0; a2: Spearman < 0).
#   Asociación, no causa: se rota más contra rivales débiles y con calendario cargado (por eso los controles).
# Apuestas (antes de correr): Claude · a1 field tilt −0.5 puntos por titular fuera del XI base · a2 Spearman −0.30 ·
#   continuidad con mercado abierto 8 puntos menor que a media temporada · Arturo: ___
# Orden: celda 1 → L1-0 (prueba con datos inventados) → L1-1. ~1-2 min. Escribe salidas/fw5/l1_*.csv.
# =====================================================================

# [L1-0 v1.0] Funciones y prueba con datos inventados
import numpy as np
import pandas as pd
SEMILLA = 2026

def dentro(d, cols, grupo):
    return d[cols] - d.groupby(grupo)[cols].transform('mean')

def pendiente(d, y, x, ctrl):
    X = np.column_stack([d[x].values] + [d[c].values for c in ctrl]); m = ~np.isnan(X).any(1) & ~np.isnan(d[y].values)
    return np.linalg.lstsq(X[m], d[y].values[m], rcond=None)[0][0]

def boot(d, fn, B=500, seed=SEMILLA):
    d = d.reset_index(drop=True); rng = np.random.default_rng(seed)
    g = d.groupby('match_id').indices; mids = np.array(list(g))
    est = fn(d); bs = [fn(d.iloc[np.concatenate([g[mids[i]] for i in rng.choice(len(mids), len(mids))])]) for _ in range(B)]
    return est, np.quantile(bs, .025), np.quantile(bs, .975)

rng0 = np.random.default_rng(SEMILLA)
for nombre, ef in [('SIN efecto', 0.0), ('CON efecto (−1 punto por titular fuera)', -1.0)]:
    filas = []
    for e in range(80):
        nivel = rng0.normal(0, 5)
        for i in range(30):
            fb = rng0.poisson(1.5); fav = rng0.random() < .3
            filas.append(dict(match_id=f'{e}-{i}', grupo=e, fuera_base=fb, favorito=float(fav), descanso_corto=0.0,
                              y=nivel + ef * fb + 2 * fav + rng0.normal(0, 6)))
    d = pd.DataFrame(filas); d[['yc', 'xc', 'fc', 'dc']] = dentro(d, ['y', 'fuera_base', 'favorito', 'descanso_corto'], 'grupo').values
    est, lo, hi = boot(d, lambda z: pendiente(z, 'yc', 'xc', ['fc', 'dc']), B=200)
    print(f'{nombre}: pendiente {est:+.2f} [{lo:+.2f}, {hi:+.2f}]')
print('Esperado: SIN ≈ 0 con IC que incluye 0; CON ≈ −1 con IC sin 0. Si sale así, corre L1-1.')

# [L1-1 v1.0] Corrida real
import os
from scipy.stats import spearmanr
RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
assert 'pendiente' in globals(), 'Corre primero L1-0'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5 = os.path.join(DP, 'fw5'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
MET = ['field_tilt', 'posesion']
R = pd.read_parquet(os.path.join(FW5, 'r4_residuos.parquet'), columns=['metrica', 'match_id', 'team', 'den', 'res'])
R = R[R.metrica.isin(MET)].assign(wr=lambda d: d.den * d.res).groupby(['metrica', 'match_id', 'team'])[['wr', 'den']].sum()
R = (R.wr / R.den).unstack('metrica').reset_index()
EP = pd.read_parquet(os.path.join(FW5, 'r7_equipo_partido.parquet'))
print('Columnas usadas de r7_equipo_partido:', [c for c in ['etapa_id', 'torneo', 'fecha', 'rol', 'fuera_base', 'favorito', 'descanso_corto'] if c in EP])
D = EP.merge(R, on=['match_id', 'team'], how='inner')
D = D[D.fuera_base.notna() & (D.rol != 'interino')].copy()
D['fecha'] = pd.to_datetime(D.fecha); D['favorito'] = D.favorito.astype(float); D['descanso_corto'] = D.descanso_corto.astype(float)
D['grupo'] = D.etapa_id + ' | ' + D.torneo
print(f'--- (0) Equipo-partidos con residuo y XI base: {len(D):,} · titulares fuera del XI base: media {D.fuera_base.mean():.2f} · '
      f'p90 {D.fuera_base.quantile(.9):.0f} · desv. est. del residuo: field tilt {D.field_tilt.std():.1f}, posesión {D.posesion.std():.1f}')

# ---------- (a1) Alineación ----------
filas = []
for m in MET:
    d = D[D[m].notna()].copy()
    d[['yc', 'xc', 'fc', 'dc']] = dentro(d, [m, 'fuera_base', 'favorito', 'descanso_corto'], 'grupo').values
    est, lo, hi = boot(d, lambda z: pendiente(z, 'yc', 'xc', ['fc', 'dc']))
    filas.append(dict(rasgo=m, n=len(d), pendiente_por_titular=round(est, 2), ic_lo=round(lo, 2), ic_hi=round(hi, 2),
                      por_3_titulares=round(3 * est, 2), palanca='EXISTE' if hi < 0 else ('opuesta' if lo > 0 else 'no se ve')))
A1 = pd.DataFrame(filas)
print('\n--- (a1) Palanca del técnico: puntos de cada rasgo por cada titular fuera de su XI base (dentro de etapa × torneo)')
print(A1.to_string(index=False))

# ---------- (a2) Renovación en los relevos ----------
JP = pd.read_parquet(os.path.join(FW5, 'r7_jugador_partido.parquet'), columns=['match_id', 'team', 'player_id', 'minutos', 'etapa_id', 'fecha'])
JP['fecha'] = pd.to_datetime(JP.fecha)
orden = D.drop_duplicates(['team', 'etapa_id']).sort_values(['team', 'fecha'])[['team', 'etapa_id', 'fecha']]
filas = []
for team, g in EP[EP.rol != 'interino'].assign(fecha=lambda x: pd.to_datetime(x.fecha)).sort_values('fecha').groupby('team'):
    ets = list(dict.fromkeys(g.etapa_id))
    for a, b in zip(ets[:-1], ets[1:]):
        pa = g[g.etapa_id == a].sort_values('fecha').tail(18); pb = g[g.etapa_id == b].sort_values('fecha').head(10)
        if len(pa) < 10 or len(pb) < 10: continue
        ant = set(JP.loc[JP.match_id.isin(pa.match_id) & (JP.team == team) & (JP.minutos > 0), 'player_id'])
        jb = JP[JP.match_id.isin(pb.match_id) & (JP.team == team)]
        fila = dict(team=team, sale=a, llega=b, llegada=pb.fecha.min(), continuidad=100 * jb.loc[jb.player_id.isin(ant), 'minutos'].sum() / jb.minutos.sum())
        for m in MET:
            ra = D.loc[D.match_id.isin(pa.match_id) & (D.team == team), m].mean(); rb = D.loc[D.match_id.isin(pb.match_id) & (D.team == team), m].mean()
            fila[f'cambio_{m}'] = rb - ra
        filas.append(fila)
A2 = pd.DataFrame(filas)
A2['mercado_abierto'] = A2.llegada.dt.month.isin([6, 7, 12, 1])
print(f'\n--- (a2) Palanca del mercado: {len(A2)} relevos (etapas de 10+ partidos a cada lado, sin interinos)')
print(f'  Continuidad: mediana {A2.continuidad.median():.0f}% · p10 {A2.continuidad.quantile(.1):.0f}% · p90 {A2.continuidad.quantile(.9):.0f}%')
print('  Por momento de la llegada:', A2.groupby('mercado_abierto').continuidad.agg(['size', 'median']).round(1).rename(index={True: 'mercado abierto', False: 'media temporada'}).to_dict('index'))
rng = np.random.default_rng(SEMILLA)
for m in MET:
    x = A2[['continuidad', f'cambio_{m}']].dropna(); y = x[f'cambio_{m}'].abs()
    rho = spearmanr(x.continuidad, y).correlation
    bs = [spearmanr(*x.assign(a=y).iloc[rng.integers(0, len(x), len(x))][['continuidad', 'a']].values.T).correlation for _ in range(500)]
    lo, hi = np.nanquantile(bs, [.025, .975])
    print(f'  {m}: Spearman(continuidad, |cambio|) {rho:+.2f} [{lo:+.2f}, {hi:+.2f}] → {"EXISTE" if hi < 0 else "no se ve"}'
          f' · |cambio| medio con continuidad < mediana {y[x.continuidad < x.continuidad.median()].mean():.1f} frente a ≥ mediana {y[x.continuidad >= x.continuidad.median()].mean():.1f}')
A1.to_csv(os.path.join(SAL5, 'l1_alineacion.csv'), index=False); A2.to_csv(os.path.join(SAL5, 'l1_relevos.csv'), index=False)
print('\nL1-1 terminó. Guardado: salidas/fw5/l1_alineacion.csv, l1_relevos.csv. Pega toda la salida.')


In [ ]:
# R9-00 v1.0 · Inspección para la organización defensiva 360 (solo lectura: no escribe nada)
# =====================================================================
# Sesión 12 (continuación) · 2026-10-19 · Hackathon ISAC 2026, Track 2 · Framework M11 (definiciones "con la evidencia del recálculo")
# Orden: celda 1 → R9-00. ~3-5 min (30 partidos por temporada, semilla fija).
# Responde, sin asumir nada:
#   (a) columnas disponibles: pases que rompen líneas, recepciones en espacio, responsabilidad defensiva, área visible del 360
#   (b) fotos útiles: acciones del RIVAL con balón en juego abierto (Pass, Carry, Ball Receipt*), sin balón parado ni reanudación,
#       a partir de 5 s de iniciada la posesión; cuántas tienen foto y cuántos defensores de campo se ven, por zona del balón
#   (c) medidas candidatas (en yardas, desde la propia portería del equipo que defiende):
#       altura A = media de los 4 defensores de campo más retrasados · altura B = el último defensor de campo (línea del fuera de juego)
#       largo del bloque = del defensor más adelantado al más retrasado · ancho = rango en y de los visibles · orden = desv. est. en x de los 4 más retrasados
#   (d) ¿cuánto depende la altura de dónde está el balón? (justifica ajustarla por zona)
#   (e) ¿hay diferencias entre equipos? reparto de la altura media por equipo-temporada con el balón en el tercio medio
# Coordenadas: el 360 viene en el marco del equipo que ejecuta la acción (ataca hacia x = 120). Los defensores son teammate = False
#   y su portería está en x = 120, así que su altura es 120 − x.
# =====================================================================

# [R9-00 v1.0]
import os, re, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5, LIGA = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5'), os.path.join(DP, 'liga')
N_POR_TEMP, SEMILLA = 30, 2026
t0 = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet'))
rng = np.random.default_rng(SEMILLA)
con360 = {int(f.split('.')[0]) for f in os.listdir(os.path.join(LIGA, 'jugadores360')) if f.split('.')[0].isdigit()}
mues = {t: list(rng.choice(np.sort(g.match_id[g.match_id.isin(con360)].values), N_POR_TEMP, replace=False)) for t, g in Mm.groupby('temporada')}

# ---------- (a) Columnas ----------
m0 = mues[max(mues)][0]
ev_cols = set()
for t, ms in mues.items():
    for m in ms[:5]: ev_cols |= set(pq.read_schema(os.path.join(LIGA, 'eventos', f'{m}.parquet')).names)
print('=== (a) Columnas ===')
for k, p in {'rompe líneas': r'line_?break', 'recepción en espacio': r'in_space|space', 'responsabilidad defensiva': r'defensive_resp',
             'control / transición (fases de juego)': r'control|is_transition|directness'}.items():
    print(f'  {k:38s}:', sorted(c for c in ev_cols if re.search(p, c, re.I))[:10] or 'ninguna')
print('  jugadores360:', pq.read_schema(os.path.join(LIGA, 'jugadores360', f'{m0}.parquet')).names)
if os.path.isdir(os.path.join(LIGA, 'frames360')):
    print('  frames360:', pq.read_schema(os.path.join(LIGA, 'frames360', f'{m0}.parquet')).names)

# ---------- (b) Fotos útiles ----------
COLS = ['match_id', 'index', 'period', 't', 'type.name', 'team.name', 'possession_team.name', 'loc_x', 'loc_y', 'fase', 'es_reanudacion', 't_pos']
filas = []
for temp, ms in mues.items():
    ruta = os.path.join(FW1, f"eventos_{temp.replace('/', '-')}.parquet"); disp = set(pq.read_schema(ruta).names)
    E = pd.read_parquet(ruta, columns=[c for c in COLS if c in disp], filters=[('match_id', 'in', [int(m) for m in ms])])
    for c in COLS:
        if c not in E: E[c] = np.nan
    E = E[(E['team.name'] == E['possession_team.name']) & E['type.name'].isin(['Pass', 'Carry', 'Ball Receipt*']) & (E.fase != 'bp')
          & ~E.es_reanudacion.fillna(False).astype(bool) & (E.t_pos.fillna(0) >= 5) & E.loc_x.notna()]
    for m, g in E.groupby('match_id'):
        J = pd.read_parquet(os.path.join(LIGA, 'jugadores360', f'{m}.parquet'), columns=['event_index', 'teammate', 'keeper', 'x', 'y'])
        J = J[J.event_index.isin(set(g['index']))]
        for b in ['teammate', 'keeper']: J[b] = J[b].fillna(False).astype(bool)
        D = J[~J.teammate & ~J.keeper]
        gk = J[~J.teammate & J.keeper].groupby('event_index').size()
        for k, d in D.groupby('event_index'):
            x, y = d.x.values, d.y.values; xs = np.sort(x)[::-1]
            filas.append(dict(temporada=temp, match_id=m, index=k, n_def=len(x), portero_visible=k in gk.index,
                              altura_A=120 - xs[:4].mean() if len(x) >= 4 else np.nan, altura_B=120 - xs[0],
                              largo=xs[0] - xs[-1] if len(x) >= 8 else np.nan, ancho=y.max() - y.min() if len(x) >= 8 else np.nan,
                              orden=xs[:4].std() if len(x) >= 4 else np.nan))
    bal = E.set_index(['match_id', 'index'])
    print(f'  {temp}: {len(ms)} partidos · acciones del rival elegibles {len(E):,} ({time.time() - t0:.0f} s)')
    filas_b = bal  # solo para depurar
F = pd.DataFrame(filas)
EL = pd.concat([pd.read_parquet(os.path.join(FW1, f"eventos_{t.replace('/', '-')}.parquet"), columns=['match_id', 'index', 'loc_x', 'team.name'],
                                filters=[('match_id', 'in', [int(m) for m in ms])]) for t, ms in mues.items()])
F = F.merge(EL, on=['match_id', 'index'], how='left')
F['zona_balon'] = pd.cut(F.loc_x, [-1, 40, 80, 121], labels=['tercio propio del rival (presión)', 'tercio medio', 'nuestro tercio'])
print(f'\n=== (b) Fotos con defensores visibles: {len(F):,} · defensores de campo visibles (mediana) {F.n_def.median():.0f}')
print('  % de fotos con ≥ 4 / ≥ 8 / los 10 defensores de campo visibles, por zona del balón:')
print(F.groupby('zona_balon', observed=True).n_def.agg(n='size', al_menos_4=lambda s: 100 * (s >= 4).mean(),
                                                        al_menos_8=lambda s: 100 * (s >= 8).mean(), los_10=lambda s: 100 * (s >= 10).mean()).round(1).to_string())

# ---------- (c) y (d) Medidas y dependencia del balón ----------
print('\n=== (c) Medidas candidatas (yd desde la propia portería), mediana [p10, p90] por zona del balón ===')
for c in ['altura_A', 'altura_B', 'largo', 'ancho', 'orden']:
    q = F.groupby('zona_balon', observed=True)[c].quantile([.1, .5, .9]).unstack().round(1)
    print(f'  {c}:', {z: f'{r[.5]} [{r[.1]}, {r[.9]}]' for z, r in q.iterrows()})
ok = F.altura_A.notna()
b = np.polyfit(120 - F.loc[ok, 'loc_x'], F.loc[ok, 'altura_A'], 1)
print(f'\n=== (d) Altura A contra posición del balón (desde nuestra portería): pendiente {b[0]:.2f} yd por yd · '
      f'correlación {np.corrcoef(120 - F.loc[ok, "loc_x"], F.loc[ok, "altura_A"])[0, 1]:.2f} · correlación A-B {F[["altura_A", "altura_B"]].corr().iloc[0, 1]:.2f}')

# ---------- (e) ¿Hay señal entre equipos? ----------
F['defensor'] = F['team.name']  # equipo con balón; el que defiende es el otro
loc = Mm.set_index('match_id')
F['defensor'] = np.where(F['team.name'] == F.match_id.map(loc['local']), F.match_id.map(loc['visita']), F.match_id.map(loc['local']))
tm = F[(F.zona_balon == 'tercio medio') & ok].groupby(['defensor', 'temporada']).altura_A.agg(['size', 'mean']).query('size >= 200')
print(f'\n=== (e) Altura A con el balón en el tercio medio, por equipo-temporada (≥ 200 fotos; {len(tm)} equipo-temporadas en la muestra):'
      f' p10 {tm["mean"].quantile(.1):.1f} · mediana {tm["mean"].median():.1f} · p90 {tm["mean"].quantile(.9):.1f} yd')
am = tm[tm.index.get_level_values(0).str.contains('Am[eé]rica', regex=True)]
print('  América en la muestra:', am['mean'].round(1).to_dict())
print(f'\nR9-00 terminó en {time.time() - t0:.0f} s. No escribió nada. Pega toda la salida.')

In [ ]:
# A1 v1.1 · ¿El choque de Almada en córner defensivo (−3.5 z en defensores en el área) es real o es la cámara?
# v1.1 (2026-10-19): la v1.0 leía el 360 de los ~1,700 partidos desde Drive y era demasiado lenta (se interrumpió sin resultados).
#   Ahora lee solo los partidos de Almada en Pachuca (todos) y una muestra fija de la liga (50 por temporada, semilla 2026).
#   La prueba se reformula sobre la BRECHA (Almada − liga) con IC por remuestreo de partidos, que es lo que decide.
# =====================================================================
# Sesión 12 (continuación) · Hackathon ISAC 2026, Track 2 · Chequeo abierto del relevo de la sesión 12
# Si la cámara ve peor el área en los córners que defendió Pachuca con Almada, R8a contaría menos defensores sin que los hubiera.
# Prueba: para cada córner directo rival con foto, qué parte del área grande cae dentro del área visible del 360 (frames360.visible_area,
#   marco del atacante: área en x ≥ 102, 18 ≤ y ≤ 62). Se compara Almada en Pachuca con la liga:
#   (1) cobertura: % con foto, % con el área completa visible (≥ 98%);
#   (2) brecha de defensores en el área (Almada − liga) en TODAS las fotos y solo en las de área completa.
#   Si la brecha se mantiene con el área completa, el choque es real; si se cierra, era la cámara.
# Solo lee fw5 (r8_bp) y los 360; no toca 2026/27 ni el preregistro (solo valida la medición).
# Apuestas (antes de correr): Claude y Arturo · la brecha se mantiene (real).
# Orden: celda 1 → A1. ~4-6 min.
# =====================================================================

# [A1 v1.1]
import os, json, time
import numpy as np
import pandas as pd
from shapely.geometry import Polygon, box
RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5, LIGA = os.path.join(DP, 'fw5'), os.path.join(DP, 'liga')
SEMILLA, N_LIGA = 2026, 50
t0 = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
ptp['etapa_id'] = ptp.team + ' · ' + ptp.etapa.astype(str)
ALM = ptp[ptp.tec_p.astype(str).str.contains('Almada') & ptp.team.str.contains('Pachuca')]
mid_alm = set(ALM.match_id) & set(loc.index)
rng = np.random.default_rng(SEMILLA)
mid_liga = set()
for t, g in Mm[~Mm.match_id.isin(mid_alm)].groupby('temporada'):
    ids = np.sort(g.match_id.values); mid_liga |= set(rng.choice(ids, min(N_LIGA, len(ids)), replace=False))
print(f'Partidos: Almada en Pachuca {len(mid_alm)} · muestra de la liga {len(mid_liga)}')

BP = pd.read_parquet(os.path.join(FW5, 'r8_bp.parquet'))
co = BP[(BP.kind == 'corner') & (BP.ruta == 'directo') & BP.match_id.isin(mid_alm | mid_liga)].copy()
lo, vi = co.match_id.map(loc['local']), co.match_id.map(loc['visita'])
co['defensor'] = np.where(co.team == lo, vi, lo)
co['grupo'] = np.where(co.match_id.isin(mid_alm) & co.defensor.str.contains('Pachuca'), 'almada',
                       np.where(co.match_id.isin(mid_liga), 'liga', 'otro'))
co = co[co.grupo != 'otro'].copy()
co['foto'] = co.s_frame.astype('boolean').fillna(False).astype(bool) if 's_frame' in co else False
co['de_local'] = co.defensor == co.match_id.map(loc['local'])
AREA = box(102, 18, 120, 62)
def poligono(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return None
    if isinstance(v, str): v = json.loads(v)
    a = np.asarray(v, dtype=float).ravel()
    if len(a) < 6: return None
    p = Polygon(a.reshape(-1, 2))
    return p if p.is_valid else p.buffer(0)
frac = {}
for i_, (m, g) in enumerate(co[co.foto].groupby('match_id')):
    rf = os.path.join(LIGA, 'frames360', f'{m}.parquet')
    if not os.path.exists(rf): continue
    fr = pd.read_parquet(rf, columns=['event_index', 'visible_area'])
    fr = fr[fr.event_index.isin(set(g['index']))].drop_duplicates('event_index').set_index('event_index').visible_area
    for i in g['index']:
        p = poligono(fr.get(i)) if i in fr.index else None
        frac[(m, i)] = p.intersection(AREA).area / AREA.area if p is not None else np.nan
    if (i_ + 1) % 100 == 0: print(f'  {i_ + 1} partidos leídos · {time.time() - t0:.0f} s')
co['frac_area'] = [frac.get((m, i), np.nan) for m, i in zip(co.match_id, co['index'])]
co['area_completa'] = co.frac_area >= 0.98

def resumen(d, nombre):
    f = d[d.foto]
    return dict(grupo=nombre, corners=len(d), pct_con_foto=round(100 * d.foto.mean(), 1), pct_area_completa=round(100 * f.area_completa.mean(), 1),
                frac_area_media=round(f.frac_area.mean(), 3), def_area_todas=round(f.s_def_area.mean(), 2),
                def_area_area_completa=round(f.loc[f.area_completa, 's_def_area'].mean(), 2), n_area_completa=int(f.area_completa.sum()))
A, L = co[co.grupo == 'almada'], co[co.grupo == 'liga']
T = pd.DataFrame([resumen(A, 'Pachuca con Almada'), resumen(L, 'liga (muestra)'),
                  resumen(A[A.de_local], 'Almada, Pachuca de local'), resumen(A[~A.de_local], 'Almada, Pachuca de visita')])
print('\n--- (1) Cobertura y defensores en el área (córners directos rivales):'); print(T.to_string(index=False))

def brecha(d, solo_completa):
    f = d[d.foto & (d.area_completa if solo_completa else True)]
    return f.loc[f.grupo == 'almada', 's_def_area'].mean() - f.loc[f.grupo == 'liga', 's_def_area'].mean()
g_ = co.groupby('match_id').indices; mids = np.array(list(g_)); bs = {False: [], True: []}
for _ in range(500):
    idx = np.concatenate([g_[mids[i]] for i in rng.choice(len(mids), len(mids))]); d = co.iloc[idx]
    for k in bs: bs[k].append(brecha(d, k))
print('\n--- (2) Brecha de defensores en el área, Almada − liga (IC 95% por partidos):')
for k, nombre in [(False, 'todas las fotos'), (True, 'solo área completa visible')]:
    print(f'  {nombre}: {brecha(co, k):+.2f} [{np.nanquantile(bs[k], .025):+.2f}, {np.nanquantile(bs[k], .975):+.2f}]')
print('  Lectura: si la brecha con el área completa sigue negativa y con IC sin 0, el choque es real; si se acerca a 0, era la cámara.')
print(f'\nA1 terminó en {time.time() - t0:.0f} s. No escribió nada. Pega toda la salida.')

In [ ]:
# A2 v1.0 · Intervalos del mapa por familia SIN Elo (¿la salida elaborada viaja con el técnico una vez descontado el nivel de la plantilla?)
# =====================================================================
# Sesión 12 (continuación) · 2026-10-19 · Hackathon ISAC 2026, Track 2 · Reabre S12-23 porque la narrativa central se apoya en él (S12-34)
# F-2 dio la salida elaborada sin Elo en 0.63 (técnico) sin intervalo. Aquí se corre la MISMA función `ejes` de C-1 (remuestreo por técnico
#   o por club, placebo y firmeza) sobre los puntajes de familia sin Elo (Zfe) que F-2 ya calculó. Nada se redefine.
# Limitación declarada (la misma de F-2): la confiabilidad de cada familia es la de la versión con Elo.
# Criterio fijado ANTES de correr: la salida elaborada "viaja con el técnico sin Elo" si su eje técnico sin Elo tiene IC que no incluye 0
#   y queda ≥ 0.5 (corte del mapa). Si el IC incluye 0, se reporta como tendencia, nunca como hallazgo.
# Apuesta: Claude · 0.63 [0.15, 1.05] (IC sin 0, tendencia) · Arturo: ___
# Orden: celda 1 → C-0 → C-1 → F-2 → A2 (en la misma sesión de Colab). ~2-3 min.
# =====================================================================

# [A2 v1.0]
import os
for v in ['ejes', 'clasificar', 'Zfe', 'conf_f', 'nombres', 'pt', 'ct', 'pc', 'cc', 'pool', 'TF']:
    assert v in globals(), f'Falta {v}: corre C-0, C-1 y F-2 en esta sesión antes de A2'
TFe = ejes(Zfe, conf_f, pt, ct, pc, cc, pool, B=1000, P=200).set_index(pd.Index(nombres, name='familia'))
TFe['cuadrante'], TFe['firmeza'] = clasificar(TFe)
chk = (TFe['tec'] - TF['tec_sin_elo']).abs().max()
print(f'Chequeo contra F-2 (buena práctica 13): diferencia máxima del eje técnico sin Elo {chk:.3f} (debe ser ≈ 0)')
print('--- A2 · Mapa por familia SIN Elo, con intervalos (mismo método que C-1 y F-2) ---')
print(TFe[['conf', 'tec', 'tec_lo', 'tec_hi', 'club', 'club_lo', 'club_hi', 'cuadrante', 'firmeza']].round(2).to_string())
s = TFe.loc['Salida elaborada']
print(f"\nSalida elaborada sin Elo: técnico {s.tec:.2f} [{s.tec_lo:.2f}, {s.tec_hi:.2f}] · club {s.club:.2f} [{s.club_lo:.2f}, {s.club_hi:.2f}] → "
      f"{'VIAJA CON EL TÉCNICO (criterio cumplido)' if (s.tec_lo > 0 and s.tec >= 0.5) else 'TENDENCIA (IC con 0 o bajo el corte): no se afirma'}")
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Freno: Drive no está montado; no se escribe nada'
TFe.to_csv(os.path.join(SAL5, 'a2_mapa_familias_sin_elo.csv'))
print('Guardado: salidas/fw5/a2_mapa_familias_sin_elo.csv. Pega toda la salida.')

In [ ]:
# R9a v1.1 · Organización defensiva 360: una fila por foto del rival con balón · contragolpes propios (B6) · Voronoi recortado al área visible
# v1.1 (2026-10-19): (1) guarda e5b (sensibilidad de liguilla, como R4/R8u); (2) agrega el Voronoi recortado al área visible (S12-36,
#   Arturo) en la misma lectura del 360, para no leer otra vez ~1,700 partidos. La caché de la v1.0 se rehace sola (no tiene vor_def).
# =====================================================================
# Sesión 12 (continuación) · Hackathon ISAC 2026, Track 2 · Cinco temporadas (fw5) · Decisiones S12-36 a S12-39
# Fotos de defensa (S12-38): acciones del rival con balón (Pass, Carry, Ball Receipt*) en juego abierto, sin balón parado ni reanudación,
#   desde 5 s de posesión, periodos 1-2. Defensores = teammate False y no portero (el 360 está en el marco del atacante; nuestra
#   portería en x = 120, altura = 120 − x). Medidas (R9-00):
#   altura_A = media de los 4 defensores de campo más retrasados (≥ 4 visibles) · altura_B = último defensor (sensibilidad)
#   largo = del más retrasado al más adelantado y ancho = rango en y (≥ 8 visibles) · orden = desv. est. en x de los 4 más retrasados
#   Del 360 (frames360): line_breaking_pass, ball_receipt_in_space, distance_to_nearest_defender.
#   balon_def = distancia del balón a NUESTRA portería (120 − loc_x): las medidas se ajustan por esto en R9u.
# Voronoi recortado (control del espacio visible): en fotos con el balón en el tercio medio (40 ≤ balon_def < 80) y ≥ 8 defensores
#   visibles, una de cada 3: regiones de Voronoi de TODOS los jugadores visibles (porteros incluidos), recortadas al polígono visible de la
#   cámara y al campo. vor_def = % del área visible que controla la defensa; vor_def_cerca = lo mismo dentro de las 35 yd de nuestra
#   portería (x ≥ 85 en el marco del atacante). Límite: solo lo que ve la cámara.
# Contragolpes propios (B6): acciones propias con balón (Pass, Carry) en fase 'transicion' de R1; compañeros de campo por delante del balón.
# Marcador y cuarto: copia literal de R8a (goles de los periodos 1-4; offset y offset_b de liguilla; cuartos 0-25 | 25-45+ | 45-70 | 70-90+).
# Orden: celda 1 → R9a-0 (prueba) → R9a-1 (~60-75 min; caché por temporada). Escribe fw5/r9_fotos_*, r9_contras_*.
# =====================================================================

# [R9a-0 v1.1] Funciones y prueba con fotos inventadas
import json
import numpy as np
import pandas as pd
from scipy.spatial import Voronoi
from shapely.geometry import Polygon, box

def medidas_defensa(J):
    """J: jugadores 360 de varias fotos (event_index, teammate, keeper, x, y). Una fila por foto con las medidas del bloque."""
    D = J[~J.teammate & ~J.keeper].sort_values(['event_index', 'x'], ascending=[True, False]).copy()
    D['r'] = D.groupby('event_index').cumcount()
    g = D.groupby('event_index')
    out = pd.DataFrame({'n_def': g.size(), 'x_max': g.x.max(), 'x_min': g.x.min(), 'y_rango': g.y.max() - g.y.min()})
    t4 = D[D.r < 4].groupby('event_index').x
    out['x4'], out['sd4'] = t4.mean(), t4.std(ddof=0)
    out['altura_A'] = np.where(out.n_def >= 4, 120 - out.x4, np.nan)
    out['altura_B'] = 120 - out.x_max
    out['largo'] = np.where(out.n_def >= 8, out.x_max - out.x_min, np.nan)
    out['ancho'] = np.where(out.n_def >= 8, out.y_rango, np.nan)
    out['orden'] = np.where(out.n_def >= 4, out.sd4, np.nan)
    return out[['n_def', 'altura_A', 'altura_B', 'largo', 'ancho', 'orden']]

def adelante_del_balon(J, xb):
    T = J[J.teammate & ~J.keeper].copy(); T['xb'] = T.event_index.map(xb)
    g = T.assign(ad=T.x > T.xb).groupby('event_index')
    return pd.DataFrame({'adelante': g.ad.sum(), 'comp_visibles': g.size()})

CAMPO, CERCA = box(0, 0, 120, 80), box(85, 0, 120, 80)
FAR = np.array([[-1000, -1000], [-1000, 1080], [1120, -1000], [1120, 1080]])
def poligono(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return None
    if isinstance(v, str): v = json.loads(v)
    a = np.asarray(v, dtype=float).ravel()
    if len(a) < 6: return None
    p = Polygon(a.reshape(-1, 2)); p = p if p.is_valid else p.buffer(0)
    return p.intersection(CAMPO)

def voronoi_control(pts, es_def, vis):
    """pts: posiciones (n, 2) de todos los visibles; es_def: booleano (n); vis: polígono visible. Devuelve (% defensa, % defensa cerca)."""
    if vis is None or vis.area < 1 or len(pts) < 4: return np.nan, np.nan
    pts = pts + np.random.default_rng(0).normal(0, 1e-6, pts.shape)          # evita puntos repetidos
    vor = Voronoi(np.vstack([pts, FAR])); a_def = a_tot = c_def = c_tot = 0.0
    vis_c = vis.intersection(CERCA)
    for i in range(len(pts)):
        reg = vor.regions[vor.point_region[i]]
        if not reg or -1 in reg: continue
        cel = Polygon(vor.vertices[reg]); a = cel.intersection(vis).area; c = cel.intersection(vis_c).area if vis_c.area > 0 else 0.0
        a_tot += a; c_tot += c
        if es_def[i]: a_def += a; c_def += c
    return (100 * a_def / a_tot if a_tot > 0 else np.nan), (100 * c_def / c_tot if c_tot > 1 else np.nan)

# Prueba 1 (medidas): foto 1 = 4 defensores en línea a x = 100 + 4 a x = 80 → altura_A 20, orden 0, largo 20, ancho 40
J = pd.DataFrame({'event_index': [1] * 9 + [2] * 3 + [3] * 6,
                  'teammate': [False] * 8 + [False] + [False] * 3 + [True] * 6,
                  'keeper': [False] * 8 + [True] + [False] * 3 + [False] * 5 + [True],
                  'x': [100, 100, 100, 100, 80, 80, 80, 80, 119, 90, 95, 99, 80, 75, 72, 60, 50, 5],
                  'y': [20, 33, 47, 60, 20, 33, 47, 60, 40, 30, 40, 50, 10, 20, 30, 40, 50, 40]})
M = medidas_defensa(J); A = adelante_del_balon(J, pd.Series({3: 70.0}))
esperado = {(1, 'altura_A'): 20.0, (1, 'orden'): 0.0, (1, 'largo'): 20.0, (1, 'ancho'): 40.0, (1, 'n_def'): 8, (1, 'altura_B'): 20.0,
            (2, 'n_def'): 3, (2, 'altura_B'): 21.0}
fallas = [f'{k}: {M.at[k[0], k[1]]} (esperado {v})' for k, v in esperado.items() if abs(M.at[k[0], k[1]] - v) > 1e-9]
if not np.isnan(M.at[2, 'altura_A']): fallas.append('foto 2 debería no tener altura_A (3 defensores)')
if A.at[3, 'adelante'] != 3 or A.at[3, 'comp_visibles'] != 5: fallas.append(f"contragolpe: {A.loc[3].to_dict()} (esperado 3 adelante, 5 visibles)")
# Prueba 2 (Voronoi): defensores en (90, 20) y (90, 60), atacantes en (30, 20) y (30, 60) → la frontera es x = 60.
#   Campo completo visible → defensa 50% · cerca de su portería (x ≥ 85) → 100% · visible recortado a x ≤ 60 → 0%.
P2, D2 = np.array([[90., 20.], [90., 60.], [30., 20.], [30., 60.]]), np.array([True, True, False, False])
v1, c1 = voronoi_control(P2, D2, box(0, 0, 120, 80)); d2 = voronoi_control(P2, D2, box(0, 0, 60, 80))[0]
for nombre, val, esp in [('Voronoi campo completo', v1, 50.0), ('Voronoi cerca de la portería', c1, 100.0), ('Voronoi recortado a x ≤ 60', d2, 0.0)]:
    if abs(val - esp) > 1e-3: fallas.append(f'{nombre}: {val:.3f} (esperado {esp})')
print('--- R9a-0:', 'TODOS BIEN' if not fallas else fallas)
print(f'  Voronoi de prueba: campo completo {v1:.1f}% · cerca {c1:.1f}% · recortado {d2:.1f}%')

# [R9a-1 v1.1] Corrida real
import os, time
import pyarrow.parquet as pq
RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
assert 'voronoi_control' in globals(), 'Corre primero R9a-0 v1.1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5, LIGA = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5'), os.path.join(DP, 'liga')
REHACER = False
CORTE_C1, CORTE_C3, CADA = 25, 70, 3
t_ini = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
TEMPS = sorted(Mm.temporada.unique())
COLS = ['match_id', 'index', 'period', 'minute', 't', 'type.name', 'team.name', 'possession_team.name', 'loc_x', 'loc_y',
        'fase', 'es_reanudacion', 't_pos', 'shot.outcome.name']
FCOLS = ['event_index', 'line_breaking_pass', 'ball_receipt_in_space', 'distance_to_nearest_defender', 'visible_area']
def e5(m):
    m = np.asarray(m, dtype=float); return np.clip(np.sign(m) * np.ceil(np.abs(m) - 1e-9), -2, 2)

for temp in TEMPS:
    s_ = temp.replace('/', '-'); r_f, r_c = os.path.join(FW5, f'r9_fotos_{s_}.parquet'), os.path.join(FW5, f'r9_contras_{s_}.parquet')
    if os.path.exists(r_f) and os.path.exists(r_c) and not REHACER and 'vor_def' in pq.read_schema(r_f).names:
        print(f'  {temp}: caché v1.1'); continue
    t0 = time.time()
    ruta = os.path.join(FW1, f'eventos_{s_}.parquet'); disp = set(pq.read_schema(ruta).names)
    E = pd.read_parquet(ruta, columns=[c for c in COLS if c in disp])
    for c in COLS:
        if c not in E: E[c] = np.nan
    E = E[E.match_id.isin(loc.index)].sort_values(['match_id', 'index']).reset_index(drop=True)
    # Marcador: copia literal de R8a
    E['local'] = E.match_id.map(loc['local']); E['ev_local'] = E['team.name'] == E['local']
    es_gol = E['period'].between(1, 4) & (((E['type.name'] == 'Shot') & (E['shot.outcome.name'] == 'Goal')) | (E['type.name'] == 'Own Goal For'))
    gh = (es_gol & E['ev_local']).astype(int); ga = (es_gol & ~E['ev_local'] & E['team.name'].notna()).astype(int)
    E['dif_home'] = (gh.groupby(E['match_id']).cumsum() - gh) - (ga.groupby(E['match_id']).cumsum() - ga)
    E['m_home'] = E['dif_home'] + E['match_id'].map(loc['offset']); E['m_home_b'] = E['dif_home'] + E['match_id'].map(loc['offset_b'])
    E['cuarto'] = np.where(E['period'] == 1, np.where(E['minute'] < CORTE_C1, 1, 2), np.where(E['minute'] < CORTE_C3, 3, 4))
    E = E[E.period.isin([1, 2]) & E.loc_x.notna()]
    con_balon = (E['team.name'] == E['possession_team.name'])
    DEF = E[con_balon & E['type.name'].isin(['Pass', 'Carry', 'Ball Receipt*']) & (E.fase != 'bp')
            & ~E.es_reanudacion.fillna(False).astype(bool) & (E.t_pos.fillna(0) >= 5)]
    CON = E[con_balon & E['type.name'].isin(['Pass', 'Carry']) & (E.fase == 'transicion')]
    pf, pc, nvor = [], [], 0
    for m in sorted(set(DEF.match_id) | set(CON.match_id)):
        rj = os.path.join(LIGA, 'jugadores360', f'{m}.parquet')
        if not os.path.exists(rj): continue
        J = pd.read_parquet(rj, columns=['event_index', 'teammate', 'keeper', 'x', 'y'])
        for b in ['teammate', 'keeper']: J[b] = J[b].fillna(False).astype(bool)
        rf = os.path.join(LIGA, 'frames360', f'{m}.parquet')
        FR = pd.read_parquet(rf, columns=[c for c in FCOLS if c in set(pq.read_schema(rf).names)]).drop_duplicates('event_index').set_index('event_index') if os.path.exists(rf) else None
        g = DEF[DEF.match_id == m]
        if len(g):
            Md = medidas_defensa(J[J.event_index.isin(set(g['index']))])
            out = g.set_index('index').join(Md, how='inner')
            if FR is not None: out = out.join(FR.drop(columns=['visible_area'], errors='ignore'), how='left')
            out['defensor'] = np.where(out.ev_local, out.match_id.map(loc['visita']), out.match_id.map(loc['local']))
            s = np.where(out.ev_local, -1.0, 1.0)                                    # marcador visto desde el que defiende
            out['e5'], out['e5b'] = e5(s * out.m_home), e5(s * out.m_home_b)
            out['balon_def'] = 120 - out.loc_x
            out['vor_def'] = np.nan; out['vor_def_cerca'] = np.nan
            if FR is not None and 'visible_area' in FR:
                eleg = out[(out.balon_def >= 40) & (out.balon_def < 80) & (out.n_def >= 8)].index[::CADA]
                Jv = J[J.event_index.isin(set(eleg))]
                for k, jj in Jv.groupby('event_index'):
                    v, c = voronoi_control(jj[['x', 'y']].values.astype(float), (~jj.teammate).values, poligono(FR.visible_area.get(k)))
                    out.at[k, 'vor_def'], out.at[k, 'vor_def_cerca'] = v, c; nvor += 1
            pf.append(out.reset_index())
        g = CON[CON.match_id == m]
        if len(g):
            Ad = adelante_del_balon(J[J.event_index.isin(set(g['index']))], g.set_index('index').loc_x)
            oc = g.set_index('index').join(Ad, how='inner')
            s = np.where(oc.ev_local, 1.0, -1.0)
            oc['e5'], oc['e5b'] = e5(s * oc.m_home), e5(s * oc.m_home_b)
            pc.append(oc.reset_index())
    KF = ['match_id', 'index', 'period', 'minute', 'cuarto', 'e5', 'e5b', 'type.name', 'team.name', 'defensor', 'loc_y', 'balon_def', 'n_def',
          'altura_A', 'altura_B', 'largo', 'ancho', 'orden', 'line_breaking_pass', 'ball_receipt_in_space', 'distance_to_nearest_defender',
          'vor_def', 'vor_def_cerca']
    F = pd.concat(pf, ignore_index=True); F = F[[c for c in KF if c in F]]
    C = pd.concat(pc, ignore_index=True)[['match_id', 'index', 'period', 'minute', 'cuarto', 'e5', 'e5b', 'type.name', 'team.name',
                                          'loc_x', 't_pos', 'adelante', 'comp_visibles']]
    F.to_parquet(r_f, index=False); C.to_parquet(r_c, index=False)
    print(f'  {temp}: fotos de defensa {len(F):,} (con altura {100 * F.altura_A.notna().mean():.0f}%) · Voronoi {nvor:,} · '
          f'contragolpe {len(C):,} · {time.time() - t0:.0f} s')

F = pd.concat([pd.read_parquet(os.path.join(FW5, f"r9_fotos_{t.replace('/', '-')}.parquet")) for t in TEMPS], ignore_index=True)
C = pd.concat([pd.read_parquet(os.path.join(FW5, f"r9_contras_{t.replace('/', '-')}.parquet")) for t in TEMPS], ignore_index=True)
print(f'\n--- (1) Fotos de defensa: {len(F):,} en {F.match_id.nunique():,} partidos · con altura_A {100 * F.altura_A.notna().mean():.1f}% · '
      f'con largo y ancho {100 * F.largo.notna().mean():.1f}%')
for c, tipo in [('line_breaking_pass', 'Pass'), ('ball_receipt_in_space', 'Ball Receipt*')]:
    if c in F:
        s = F.loc[F['type.name'] == tipo, c]
        print(f'  {c}: con dato {100 * s.notna().mean():.1f}% de las {tipo} del rival · % verdadero {100 * s.eq(True).mean():.1f}')
print('  Altura A por zona del balón (mediana, yd):', F.groupby(pd.cut(F.balon_def, [0, 40, 80, 121]), observed=True).altura_A.median().round(1).to_dict())
v = F.vor_def.dropna()
print(f'  Voronoi: {len(v):,} fotos · defensa controla del área visible: mediana {v.median():.1f}% [p10 {v.quantile(.1):.1f}, p90 {v.quantile(.9):.1f}]'
      f' · cerca de su portería: mediana {F.vor_def_cerca.median():.1f}%')
print(f'--- (2) Contragolpes propios: {len(C):,} fotos · compañeros por delante del balón: mediana {C.adelante.median():.0f} · '
      f'p90 {C.adelante.quantile(.9):.0f} · compañeros visibles (mediana) {C.comp_visibles.median():.0f}')
print(f'\nR9a-1 terminó en {time.time() - t_ini:.0f} s. Guardado: fw5/r9_fotos_*, r9_contras_*. Pega toda la salida.')



In [ ]:
# R9u v1.0 · Rasgos de organización defensiva 360 en formato R4 y paso por R6r (receta y confiabilidad) con el patrón espejo (prefijo r9)
# =====================================================================
# Sesión 12 (continuación) · 2026-10-19 · Hackathon ISAC 2026, Track 2 · Cinco temporadas (fw5) · Decisiones S12-36 a S12-39
# Insumo: fw5/r9_fotos_* y r9_contras_* (R9a v1.1). Una fila de tramo = (métrica, partido, equipo, e5, e5b, cuarto, num, den), igual que R8u.
# Ajuste por la posición del balón (S12-37): cada medida de una foto se expresa como desviación frente a la media de la LIGA en su franja
#   de 5 yd de distancia del balón a la portería del que defiende (cinco temporadas). Así el rasgo no mide dónde juega el rival.
# Rasgos (equipo que defiende, salvo el último):
#   altura (A; balón a < 80 yd de su portería, ≥ 4 defensores) · altura_presion (balón en el tercio del rival; sensibilidad: la cámara a
#   veces no ve la última línea) · largo, ancho (≥ 8 defensores; balón a < 80) · orden (≥ 4; balón a < 80)
#   · rompe_lineas_conc (% de pases del rival que rompen líneas) · espacio_conc (% de recepciones del rival en espacio)
#   · dist_defensor (distancia del receptor rival a su defensor más cercano, ajustada por franja)
#   · vor_def, vor_def_cerca (Voronoi recortado, tercio medio, ajustados por franja)
#   · adelante_contra (equipo que ATACA: compañeros por delante del balón en sus contragolpes; fotos con ≥ 8 compañeros visibles;
#     ajustado por franja de la posición del balón). Hipótesis de Arturo (B6).
# Orden: celda 1 → R9u-0 (prueba + tramos; escribe fw5/r9_tramos) → R9r-0 → R6r-1 v1.1 y R6r-2 v1.1 SIN CAMBIOS (sesión 8) → R9r-9.
# =====================================================================

# [R9u-0 v1.0] Prueba con datos inventados y tramos
import os, time
import numpy as np
import pandas as pd

def ajusta_franja(v, pos, ancho=5.0):
    """Desviación de cada valor frente a la media de su franja de posición del balón (franjas de `ancho` yd)."""
    f = np.floor(np.asarray(pos, float) / ancho)
    s = pd.Series(np.asarray(v, float)); return (s - s.groupby(f).transform('mean')).values

# Prueba: dos equipos con la MISMA línea real (altura = 0.8 × balón + 5); el equipo X defiende con el balón más lejos.
# Sin ajuste parecen distintos; con ajuste, iguales (SIN efecto). CON efecto: X defiende 4 yd más arriba a igualdad de balón.
rng0 = np.random.default_rng(2026)
for nombre, ef in [('SIN efecto', 0.0), ('CON efecto (+4 yd)', 4.0)]:
    bx, by = rng0.uniform(40, 80, 3000), rng0.uniform(20, 60, 3000)
    ax = 0.8 * bx + 5 + ef + rng0.normal(0, 4, 3000); ay = 0.8 * by + 5 + rng0.normal(0, 4, 3000)
    adj = ajusta_franja(np.r_[ax, ay], np.r_[bx, by])
    print(f'--- {nombre}: crudo X − Y {ax.mean() - ay.mean():+.1f} yd · ajustado X − Y {adj[:3000].mean() - adj[3000:].mean():+.1f} yd (esperado {ef:+.1f})')

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5 = os.path.join(DP, 'fw5')
t0 = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet'))
TEMPS = sorted(Mm.temporada.unique())
F = pd.concat([pd.read_parquet(os.path.join(FW5, f"r9_fotos_{t.replace('/', '-')}.parquet")) for t in TEMPS], ignore_index=True)
C = pd.concat([pd.read_parquet(os.path.join(FW5, f"r9_contras_{t.replace('/', '-')}.parquet")) for t in TEMPS], ignore_index=True)
assert 'e5b' in F and 'vor_def' in F, 'r9_fotos no es de la v1.1: corre R9a v1.1'
piezas = []
def agrega(d, num, nombre, team_col='defensor', den=1.0):
    t = pd.DataFrame({'metrica': nombre, 'match_id': d.match_id.values, 'team': d[team_col].values, 'e5': d.e5.astype(int).values,
                      'e5b': d.e5b.astype(int).values, 'cuarto': d.cuarto.astype(int).values, 'num': np.asarray(num, float),
                      'den': np.asarray(den, float) if np.ndim(den) else den})
    piezas.append(t.dropna(subset=['num']))

base = F[F.balon_def < 80]
for c, req in [('altura_A', 4), ('largo', 8), ('ancho', 8), ('orden', 4)]:
    d = base[base[c].notna() & (base.n_def >= req)]
    agrega(d, ajusta_franja(d[c], d.balon_def), 'altura' if c == 'altura_A' else c)
pz = F[(F.balon_def >= 80) & F.altura_A.notna()]
agrega(pz, ajusta_franja(pz.altura_A, pz.balon_def), 'altura_presion')
if 'line_breaking_pass' in F:
    d = F[(F['type.name'] == 'Pass') & F.line_breaking_pass.notna()]; agrega(d, 100.0 * d.line_breaking_pass.astype(bool), 'rompe_lineas_conc')
if 'ball_receipt_in_space' in F:
    d = F[(F['type.name'] == 'Ball Receipt*') & F.ball_receipt_in_space.notna()]; agrega(d, 100.0 * d.ball_receipt_in_space.astype(bool), 'espacio_conc')
if 'distance_to_nearest_defender' in F:
    d = F[(F['type.name'] == 'Ball Receipt*') & F.distance_to_nearest_defender.notna()]
    agrega(d, ajusta_franja(d.distance_to_nearest_defender, d.balon_def), 'dist_defensor')
for c in ['vor_def', 'vor_def_cerca']:
    d = F[F[c].notna()]; agrega(d, ajusta_franja(d[c], d.balon_def), c)
c8 = C[C.comp_visibles >= 8]
agrega(c8, ajusta_franja(c8.adelante, c8.loc_x), 'adelante_contra', team_col='team.name')

TR = pd.concat(piezas, ignore_index=True).groupby(['metrica', 'match_id', 'team', 'e5', 'e5b', 'cuarto'], as_index=False)[['num', 'den']].sum()
TR = TR[TR.den != 0]
TR.to_parquet(os.path.join(FW5, 'r9_tramos.parquet'), index=False)
res = TR.groupby('metrica').apply(lambda d: pd.Series({'equipo_partidos': d[['match_id', 'team']].drop_duplicates().shape[0],
                                                       'fotos_por_ep': d.groupby(['match_id', 'team']).den.sum().median(),
                                                       'media_liga': d.num.sum() / d.den.sum()}), include_groups=False)
print(f'\n--- Rasgos r9 ({TR.metrica.nunique()}). Media de la liga en su unidad (yd de desviación ≈ 0 en los ajustados · % en los de tasa):')
print(res.round(3).to_string())
print(f'Contragolpes con ≥ 8 compañeros visibles: {len(c8):,} de {len(C):,} ({100 * len(c8) / len(C):.0f}%)')
print(f'R9u-0 terminó en {time.time() - t0:.0f} s. Guardado: fw5/r9_tramos.parquet. Siguiente: R9r-0.')



In [ ]:

# [R9r-0 v1.0] Carpeta espejo para correr R6r-1 y R6r-2 sin cambios sobre fw5 (copia literal de R8r-0 con prefijo r9)
import os, json, shutil
DRIVE = '/content/drive/MyDrive/isac2026/'
LOCAL = '/content/isac_r9/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
D_DP, D_FW1, D_FW5 = os.path.join(DRIVE, 'datos_parquet'), os.path.join(DRIVE, 'datos_parquet', 'fw1'), os.path.join(DRIVE, 'datos_parquet', 'fw5')
L_DP, L_FW = os.path.join(LOCAL, 'datos_parquet'), os.path.join(LOCAL, 'datos_parquet', 'fw1')
shutil.rmtree(LOCAL, ignore_errors=True)
os.makedirs(L_FW, exist_ok=True); os.makedirs(os.path.join(LOCAL, 'salidas'), exist_ok=True)
def enlazar(src, dst):
    if os.path.exists(src): os.symlink(src, dst); return True
    return False
enl = {'partido_tecnico_periodo.parquet': enlazar(os.path.join(D_DP, 'partido_tecnico_periodo.parquet'), os.path.join(L_DP, 'partido_tecnico_periodo.parquet'))}
for f in ['r4_partidos.parquet', 'r4_minutos_v21.parquet', 'parametros_fw1.json', 'r4_perfiles_etapa.parquet',
          'r6_perfiles_etapa.parquet', 'r6b_perfiles_etapa.parquet', 'r6c_perfiles_etapa.parquet', 'r9_tramos.parquet']:
    enl[f] = enlazar(os.path.join(D_FW5, f), os.path.join(L_FW, f))
enl['elo_equipo_partido_e2.parquet (fw1, solo lectura)'] = enlazar(os.path.join(D_FW1, 'elo_equipo_partido_e2.parquet'), os.path.join(L_FW, 'elo_equipo_partido_e2.parquet'))
print('--- (R9r-0) Carpeta espejo', LOCAL); print({k: ('ok' if v else 'FALTA') for k, v in enl.items()})
oblig = ['partido_tecnico_periodo.parquet', 'r4_partidos.parquet', 'r4_minutos_v21.parquet', 'parametros_fw1.json', 'r4_perfiles_etapa.parquet',
         'r9_tramos.parquet', 'elo_equipo_partido_e2.parquet (fw1, solo lectura)']
assert all(enl[k] for k in oblig), 'Falta un insumo obligatorio: avisar a Claude'
with open(os.path.join(L_FW, 'parametros_fw1.json')) as f:
    print('Temporadas de ajuste:', json.load(f)['temporadas']['entrenamiento'])
BASE = LOCAL; PREFIJO = 'r9'
print("BASE y PREFIJO listos → corre R6r-1 v1.1 y R6r-2 v1.1 (claude/codigo/sesion8/R6r_receta_confiabilidad.py) sin cambiarles nada; luego R9r-9.")



In [ ]:

# [R9r-9 v1.0] Copiar resultados a Drive (fw5 y salidas/fw5) y regresar BASE a Drive (copia literal de R8r-9 con prefijo r9)
import os, shutil
DRIVE = '/content/drive/MyDrive/isac2026/'
LOCAL = '/content/isac_r9/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
L_FW = os.path.join(LOCAL, 'datos_parquet', 'fw1')
D_FW5, D_SAL5 = os.path.join(DRIVE, 'datos_parquet', 'fw5'), os.path.join(DRIVE, 'salidas', 'fw5')
os.makedirs(D_SAL5, exist_ok=True); n = []
for src_dir, dst_dir in [(L_FW, D_FW5), (os.path.join(LOCAL, 'salidas'), D_SAL5)]:
    for f in os.listdir(src_dir):
        s = os.path.join(src_dir, f)
        if f.startswith('r9_') and os.path.isfile(s) and not os.path.islink(s):
            shutil.copy2(s, os.path.join(dst_dir, f)); n.append(f)
print('--- (R9r-9) Copiados a Drive:', sorted(n))
need = [os.path.join(D_FW5, 'r9_perfiles_contraidos.parquet'), os.path.join(D_SAL5, 'r9_confiabilidad.csv'), os.path.join(D_SAL5, 'r9_implantacion.csv'),
        os.path.join(D_FW5, 'r9_perfiles_etapa.parquet'), os.path.join(D_FW5, 'r9_residuos.parquet')]
print('Insumos de C-1b:', {os.path.basename(p): ('ok' if os.path.exists(p) else 'FALTA') for p in need})
BASE = DRIVE
print('BASE =', BASE)

In [ ]:
# [C-1b v1.0] Compatibilidad técnico-club CON organización defensiva 360 (r9) · copia literal de C-1 v1.0 con tres cambios:
#   (1) PREFIJOS agrega 'r9' (como estaba previsto); (2) guarda en c1b_*.csv para NO pisar c_*.csv, que usa el preregistro de Almada
#   congelado (S12-39); (3) la lista de candidatos de la sección 2 agrega los rasgos de r9.
# =====================================================================
# Sesión 12 (continuación) · 2026-10-19 · Hackathon ISAC 2026, Track 2
# Antes: celda 1 → C-0 (todas las pruebas PASAN) → R9u-0 → R9r-0 → R6r-1 → R6r-2 → R9r-9 → C-1b. Tarda ~3-8 min.
# La evaluación de Almada (V-1) usa C-1 (sin r9), nunca esta celda.
# =====================================================================
import os, json, time
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
assert globals().get('C0_OK', False), 'Corre primero C-0 y confirma que todas sus pruebas PASAN'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5 = os.path.join(DP, 'fw5'); FW1 = os.path.join(DP, 'fw1'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
PREFIJOS = ['r4', 'r6', 'r6b', 'r6c', 'r7u', 'r7b', 'r7c', 'r8', 'r9']      # C-1b: se agrega 'r9'
ETAPA_MIN, NUEVA_MIN = 15, 10
VENTANA, VENT_MIN = 18, 10
CONF_MIN = 0.5
B, PLAC, B_VAL = 1000, 200, 2000
ESCRIBIR = True
rng = np.random.default_rng(SEMILLA)
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 90)
t0 = time.time()
sal = lambda p: 'r5' if p == 'r4' else p
corto = lambda s: ' '.join(s.split()[:1] + [s.split()[-2] if len(s.split()) >= 3 else s.split()[-1]]) if isinstance(s, str) and s.strip() else s

# ---------- 0. Inspección de columnas ----------
REQ = {'contraidos': ['metrica', 'etapa_id', 'perfil_r4', 'perfil_mostrado'], 'confiabilidad': ['metrica', 'conf', 'tau', 'sigma_partido'],
       'perfiles_etapa': ['etapa_id', 'metrica', 'tau_estado'], 'residuos': ['metrica', 'match_id', 'team', 'e', 'den', 'res']}
CONT, CF, TAUE, RES, S2 = [], [], {}, {}, {}
print('--- (0) Inspección por prefijo: filas y columnas requeridas ---')
for p in PREFIJOS:
    fuentes = {'contraidos': os.path.join(FW5, f'{sal(p)}_perfiles_contraidos.parquet'), 'confiabilidad': os.path.join(SAL5, f'{sal(p)}_confiabilidad.csv'),
               'perfiles_etapa': os.path.join(FW5, f'{p}_perfiles_etapa.parquet'), 'residuos': os.path.join(FW5, f'{p}_residuos.parquet')}
    D = {k: (pd.read_csv(v) if v.endswith('.csv') else pd.read_parquet(v)) for k, v in fuentes.items()}
    faltan = {k: [c for c in REQ[k] if c not in D[k].columns] for k in D}
    print(f"  {p}: " + ' · '.join(f"{k} {len(D[k]):,}" for k in D) + ('' if not any(faltan.values()) else f'  ← FALTAN {faltan}'))
    assert not any(faltan.values()), f'Columnas faltantes en {p}: avisar a Claude'
    c = D['contraidos'].assign(rasgo=lambda d: p + ':' + d.metrica); CONT.append(c)
    CF.append(D['confiabilidad'].assign(rasgo=lambda d: p + ':' + d.metrica, prefijo=p))
    TAUE.update({p + ':' + m: v for m, v in D['perfiles_etapa'].groupby('metrica')['tau_estado'].first().items()})
    rs = D['residuos'][REQ['residuos']]
    S2.update({p + ':' + m: float((g.den * g.res ** 2).sum() / (len(g) - 19)) for m, g in rs.groupby('metrica')})
    RES[p] = {t: g for t, g in rs.groupby('team')}
CONT = pd.concat(CONT, ignore_index=True); CF = pd.concat(CF, ignore_index=True).set_index('rasgo')
RASGOS = [r for r in CF.index if CF.loc[r, 'conf'] >= CONF_MIN and CF.loc[r, 'tau'] > 0]
print(f'Rasgos: {len(CF)} en total · {len(RASGOS)} con confiabilidad ≥ {CONF_MIN} (entran) · fuera: {len(CF) - len(RASGOS)}')
print('Rasgos r9 y su confiabilidad:', CF[CF.prefijo == 'r9']['conf'].round(2).to_dict())

# ---------- Etapas (copia literal de C-00) ----------
with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
    TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy(); ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ET = ptp.groupby('etapa_id').agg(team=('team', 'first'), tecnico=('tec_p', 'first'), partidos=('match_id', 'nunique'),
                                 ini=('fecha', 'min'), fin=('fecha', 'max')).reset_index()
ETAPAS = ET.etapa_id.tolist(); IX = {e: i for i, e in enumerate(ETAPAS)}
tau = CF.loc[RASGOS, 'tau'].values; conf = CF.loc[RASGOS, 'conf'].values
ZR = (CONT.pivot_table(index='etapa_id', columns='rasgo', values='perfil_r4').reindex(index=ETAPAS, columns=RASGOS) / tau).values
ZM = (CONT.pivot_table(index='etapa_id', columns='rasgo', values='perfil_mostrado').reindex(index=ETAPAS, columns=RASGOS) / tau).values
print(f'Etapas: {len(ETAPAS)} · con perfil en todos los rasgos: {int(np.isfinite(ZM).all(1).sum())} · celdas nulas {np.isnan(ZM).mean():.1%}')

# ---------- 1. Parejas ----------
L = ET[ET.partidos >= ETAPA_MIN]
pt, ct = [], []
for tec, d in L.groupby('tecnico'):
    ix = [IX[e] for e in d.etapa_id]
    for i in range(len(ix)):
        for j in range(i + 1, len(ix)):
            if ET.team[ix[i]] != ET.team[ix[j]]: pt.append((ix[i], ix[j])); ct.append(tec)
pc, cc, pnc, cnc = [], [], [], []
for team, d in L.sort_values('ini').groupby('team'):
    ix = [IX[e] for e in d.etapa_id]
    for i in range(len(ix) - 1):
        if ET.tecnico[ix[i]] != ET.tecnico[ix[i + 1]]: pc.append((ix[i], ix[i + 1])); cc.append(team)
        if i + 2 < len(ix) and ET.tecnico[ix[i]] != ET.tecnico[ix[i + 2]]: pnc.append((ix[i], ix[i + 2])); cnc.append(team)
li = [IX[e] for e in L.etapa_id]
pool = [(a, b) for k, a in enumerate(li) for b in li[k + 1:] if ET.tecnico[a] != ET.tecnico[b] and ET.team[a] != ET.team[b]]
pt, ct, pc, cc = np.array(pt), np.array(ct), np.array(pc), np.array(cc)
print(f'\n--- (1) Parejas (deben coincidir con C-1: 29 de viajeros y 69 relevos) · viajeros {len(pt)} de {len(set(ct))} técnicos · '
      f'relevos {len(pc)} en {len(set(cc))} clubes · parejas al azar {len(pool):,}')

# ---------- 2. Mapa de dos ejes ----------
T = ejes(ZR, conf, pt, ct, pc, cc, pool, B=B, P=PLAC).set_index(pd.Index(RASGOS, name='rasgo'))
T['cuadrante'], T['firmeza'] = clasificar(T)
T['prefijo'] = [r.split(':')[0] for r in RASGOS]
T['club_no_consec'] = r_pares(ZR, pnc) / np.clip(conf, .05, None)
L10 = ET[ET.partidos >= NUEVA_MIN]; pt10 = []
for tec, d in L10.groupby('tecnico'):
    ix = [IX[e] for e in d.etapa_id]
    pt10 += [(ix[i], ix[j]) for i in range(len(ix)) for j in range(i + 1, len(ix)) if ET.team[ix[i]] != ET.team[ix[j]]]
T['tec_10'] = r_pares(ZR, pt10) / np.clip(conf, .05, None)
elo = pd.read_parquet(os.path.join(FW1, 'elo_equipo_partido_e2.parquet'), columns=['match_id', 'team', 'elo_pre'])
eloE = ptp.merge(elo, on=['match_id', 'team'], how='left').groupby('etapa_id')['elo_pre'].mean().reindex(ETAPAS).values
ZE = ZR.copy()
for j in range(ZR.shape[1]):
    m = np.isfinite(ZR[:, j]) & np.isfinite(eloE)
    if m.sum() > 10:
        b = np.polyfit(eloE[m], ZR[m, j], 1); ZE[m, j] = ZR[m, j] - np.polyval(b, eloE[m])
T['tec_sin_elo'] = r_pares(ZE, pt) / np.clip(conf, .05, None); T['club_sin_elo'] = r_pares(ZE, pc) / np.clip(conf, .05, None)
T['corr_elo'] = [np.corrcoef(eloE[np.isfinite(ZR[:, j]) & np.isfinite(eloE)], ZR[np.isfinite(ZR[:, j]) & np.isfinite(eloE), j])[0, 1] for j in range(len(RASGOS))]

V = ['conf', 'tec', 'tec_lo', 'tec_hi', 'club', 'club_lo', 'club_hi', 'dif_lo', 'dif_hi', 'cuadrante', 'firmeza']
print(f'\n--- (2) Mapa con r9 · ruido de un rasgo sin efecto (placebo): ±{2 * T.placebo_de.median():.2f} ---')
print('Placebo (debe ser ≈ 0): media', round(T.placebo_media.mean(), 3))
print('\nCuadrantes por prefijo (rasgos):'); print(pd.crosstab(T.prefijo, T.cuadrante, margins=True).to_string())
print('\nFirmes:', T[T.firmeza == 'firme'].groupby('cuadrante').size().to_dict())
print('\nRasgos de organización defensiva 360 (r9):')
print(T[T.prefijo == 'r9'][V + ['tec_sin_elo', 'club_sin_elo', 'corr_elo']].round(2).to_string())
print('Rasgos r9 fuera por confiabilidad < 0.5:', [r for r in CF[CF.prefijo == 'r9'].index if r not in T.index])
# Coherencia con C-1: los rasgos que ya estaban deben conservar su cuadrante (cambian solo los pesos de la compatibilidad, no el mapa)
c1 = pd.read_csv(os.path.join(SAL5, 'c_mapa.csv')).set_index('rasgo')
com = T.index.intersection(c1.index)
print(f'Coherencia con C-1: {len(com)} rasgos en común · mismo cuadrante {100 * (T.loc[com, "cuadrante"] == c1.loc[com, "cuadrante"]).mean():.0f}% '
      f'(los ejes de cada rasgo no dependen de los demás: debe ser ~100%)')

# ---------- 3. Perfil del club ----------
MIN_ = pd.read_parquet(os.path.join(FW5, 'r4_minutos_v21.parquet'))
W3 = MIN_.assign(e3=np.sign(MIN_.e5)).groupby('e3')['minutos'].sum(); W3v = (W3 / W3.sum()).reindex([-1, 0, 1]).fillna(0).values
def perfil_club(team, partidos, contraido=True):
    z = np.full(len(RASGOS), np.nan)
    for j, r in enumerate(RASGOS):
        p, m = r.split(':', 1); g = RES[p].get(team)
        if g is None: continue
        g = g[(g.metrica == m) & g.match_id.isin(partidos)]
        n = g.match_id.nunique()
        if n == 0: continue
        pf = perfil_ventana(g.e.values, g.den.values, g.res.values, S2[r], TAUE[r], W3v)
        z[j] = (contraer(pf, n, CF.loc[r, 'tau'], CF.loc[r, 'sigma_partido']) if contraido else pf) / CF.loc[r, 'tau']
    return z
def ventana(team, antes_de):
    m = ptp[(ptp.team == team) & (ptp.fecha < antes_de)].sort_values('fecha').tail(VENTANA)
    return set(m.match_id), len(m)

# ---------- 4. Compatibilidad en las llegadas ----------
def pesos_sin(tec):
    keep = ct != tec
    r = r_pares(ZR, pt[keep]) / np.clip(conf, .05, None)
    return np.clip(np.nan_to_num(r), 0, PESO_MAX)
def candidatos(antes_de, excluir_tec, excluir_team):
    c = L[(L.ini < antes_de) & (L.tecnico != excluir_tec) & (L.team != excluir_team)].sort_values('ini').groupby('tecnico').tail(1)
    return [IX[e] for e in c.etapa_id], c.tecnico.tolist()
filas = []
for r_ in ET[ET.partidos >= NUEVA_MIN].itertuples(index=False):
    ant = L[(L.tecnico == r_.tecnico) & (L.team != r_.team) & (L.fin < r_.ini)]
    if not len(ant): continue
    prev = ant.sort_values('fin').etapa_id.iloc[-1]
    vp, nv = ventana(r_.team, r_.ini)
    if nv < VENT_MIN: filas.append(dict(etapa_nueva=r_.etapa_id, tecnico=corto(r_.tecnico), previa=prev, ventana=nv, nota='ventana corta')); continue
    zc = perfil_club(r_.team, vp); zt = ZM[IX[prev]]; w = pesos_sin(r_.tecnico)
    cix, _ = candidatos(r_.ini, r_.tecnico, r_.team)
    d = distancia(zt, zc, w); comp = percentil(d, [distancia(ZM[k], zc, w) for k in cix])
    zn = ZM[IX[r_.etapa_id]]
    filas.append(dict(etapa_nueva=r_.etapa_id, tecnico=corto(r_.tecnico), previa=prev, ventana=nv, partidos_nueva=r_.partidos, candidatos=len(cix),
                      compat=comp, distancia=d, cambio=distancia(zc, zn, w),
                      nueva_a_tecnico=distancia(zn, zt, w), nueva_a_club=distancia(zn, zc, w),
                      choques=choques(zt, zc, w, RASGOS), v_zc=zc, v_zt=zt, v_zn=zn, v_w=w, v_cix=cix, nota=''))
LL = pd.DataFrame(filas)
ok = LL[LL.nota == ''].copy()

# ---------- 5. Validación ----------
cmp_, cam_ = ok.compat.values, ok.cambio.values
rho = spearmanr(cmp_, cam_)[0]
bs = [spearmanr(cmp_[ix], cam_[ix])[0] for ix in (rng.integers(0, len(ok), len(ok)) for _ in range(B_VAL))]
cerca = (ok.nueva_a_tecnico < ok.nueva_a_club).values
pl = []
for _ in range(PLAC):
    pl.append(np.mean([distancia(f['v_zn'], ZM[rng.choice(f['v_cix'])], f['v_w']) < distancia(f['v_zn'], f['v_zc'], f['v_w']) for f in ok.to_dict('records')]))
print(f'\n--- (5) Validación con r9 (C-1 sin r9: Spearman −0.44 [−0.73, −0.07]; más cerca del técnico 52%, placebo 26%) ---')
print(f'(a) Spearman(compatibilidad, cambio): {rho:+.2f} [{np.nanpercentile(bs, 2.5):+.2f}, {np.nanpercentile(bs, 97.5):+.2f}]')
print(f'(b) Más cerca del técnico que del club en {cerca.mean():.0%} · placebo {np.mean(pl):.0%} [{np.percentile(pl, 2.5):.0%}, {np.percentile(pl, 97.5):.0%}]')

# ---------- 6. América ----------
ult = ptp.loc[ptp.team.str.startswith('Am'), 'fecha'].max(); team_am = ptp.loc[ptp.team.str.startswith('Am'), 'team'].iloc[0]
vp, nv = ventana(team_am, ult + pd.Timedelta(days=1)); zc = perfil_club(team_am, vp); wF = pesos(T)
cix, ctec = candidatos(ult + pd.Timedelta(days=1), '', team_am)
dist = np.array([distancia(ZM[k], zc, wF) for k in cix])
RK = pd.DataFrame({'tecnico': [corto(t) for t in ctec], 'etapa': [ETAPAS[k] for k in cix], 'distancia': dist})
RK['compat'] = [percentil(d, np.delete(dist, i)) for i, d in enumerate(dist)]
RK['choques'] = [choques(ZM[k], zc, wF, RASGOS) for k in cix]
RK = RK.sort_values('compat', ascending=False).reset_index(drop=True); RK.index += 1
alm = RK[RK.tecnico.str.contains('Almada', na=False)]
print('\n--- (6) América con r9 (solo para el reporte; el preregistro usa C-1) ---')
if len(alm): print(f"Almada: compatibilidad {alm.compat.iloc[0]:.0f}% (puesto {alm.index[0]} de {len(RK)}) · choques: {alm.choques.iloc[0]}")
c1r = pd.read_csv(os.path.join(SAL5, 'c_ranking_america.csv'))
j = RK.merge(c1r[['tecnico', 'etapa', 'compat']], on=['tecnico', 'etapa'], suffixes=('_c1b', '_c1'))
print(f'Ranking de América C-1b frente a C-1: Spearman {spearmanr(j.compat_c1b, j.compat_c1)[0]:+.2f} ({len(j)} técnicos)')

# ---------- 7. Guardar (nombres c1b_*: no pisa lo del preregistro) ----------
if ESCRIBIR:
    assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Freno: Drive no está montado; no se escribe nada'
    T.to_csv(os.path.join(SAL5, 'c1b_mapa.csv'))
    LL.drop(columns=[c for c in LL.columns if c.startswith('v_')]).to_csv(os.path.join(SAL5, 'c1b_llegadas.csv'), index=False)
    RK.to_csv(os.path.join(SAL5, 'c1b_ranking_america.csv'))
    print('\nGuardado: salidas/fw5/c1b_mapa.csv, c1b_llegadas.csv, c1b_ranking_america.csv')
print(f'C-1b terminó en {time.time() - t0:.0f} s · prefijos: {PREFIJOS}')

In [ ]:
# M1 v1.0 · Mapa de disputa (S4-30, idea de Arturo): "por acá presionas y recuperas, por acá presionas y te superan"
# =====================================================================
# Sesión 12 (continuación) · 2026-10-19 · Hackathon ISAC 2026, Track 2 · Cinco temporadas (fw1 R1 + posesiones; solo lectura)
# Unidad: fase de presión de R4 (copia literal de R4a: cadena de presiones del mismo equipo en la misma posesión rival con pausas ≤ 5 s;
#   fuera si su primera presión es de balón parado o cae en los primeros 5 s de una posesión que nace de reanudación, P-14).
# Desenlace de cada fase, desde su primera presión (fijado antes de correr):
#   recupera = la siguiente posesión es del equipo que presiona y empieza ≤ 5 s después (mismo periodo)
#   falta    = falta (no ofensiva) del que presiona en la misma posesión ≤ 5 s
#   superada = el rival conserva y avanza ≥ 15 yd más allá del punto de la presión en ≤ 10 s (marco del rival)
#   neutra   = lo demás (el rival conserva sin avanzar)
# Zona: 9 zonas en el marco del que presiona: tercio (x < 40 propio · 40-80 medio · ≥ 80 alto) × carril (y < 26.7 · centro · > 53.3).
# Salidas: mapa de la liga, rasgos por etapa (recupera y superada en presión alta y media) con confiabilidad entre mitades
#   (Spearman-Brown, 200 repartos; etapas de 15+), y mapas de América por técnico. Descriptivo: el ajuste por contexto queda fuera.
# Apuesta (antes de correr): Claude · liga, presión alta: recupera 22%, superada 18% · confiabilidad de "recupera en alta" 0.6 · Arturo: ___
# Orden: celda 1 → M1. ~6-10 min. Escribe salidas/fw5/m1_mapa_liga.csv, m1_etapas.csv, m1_america.csv.
# =====================================================================

# [M1 v1.0]
import os, json, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
SEMILLA, PAUSA, EXCL_REANUD = 2026, 5, 5
t_ini = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
TEMPS = sorted(Mm.temporada.unique())
COLS = ['match_id', 'index', 'period', 't', 'type.name', 'team.name', 'possession', 'possession_team.name', 'loc_x', 'loc_y', 'fase',
        'foul_committed.offensive']
fases = []
for temp in TEMPS:
    s_ = temp.replace('/', '-'); t0 = time.time()
    ruta = os.path.join(FW1, f'eventos_{s_}.parquet'); disp = set(pq.read_schema(ruta).names)
    E = pd.read_parquet(ruta, columns=[c for c in COLS if c in disp])
    for c in COLS:
        if c not in E: E[c] = np.nan
    E = E[E.match_id.isin(loc.index) & E.period.isin([1, 2])].sort_values(['match_id', 'index']).reset_index(drop=True)
    Pz0 = pd.read_parquet(os.path.join(FW1, f'posesiones_{s_}.parquet')).sort_values(['match_id', 'possession'])
    for c_, s2 in [('sig_eq', 'equipo'), ('sig_t0', 't0'), ('sig_per', 'period')]:
        Pz0[c_] = Pz0.groupby('match_id')[s2].shift(-1)
    Pz = Pz0[Pz0.match_id.isin(loc.index)]
    # ---- Fases de presión: copia literal de R4a ----
    ps = E[(E['type.name'] == 'Pressure') & (E['possession_team.name'] != E['team.name'])]
    ps = ps.merge(Pz[['match_id', 'possession', 't0', 'primer_reanudacion', 'sig_eq', 'sig_t0', 'sig_per']], on=['match_id', 'possession'], how='left')
    ps = ps.sort_values(['match_id', 'possession', 'team.name', 'index']).reset_index(drop=True)
    ps['gap'] = ps.groupby(['match_id', 'possession', 'team.name'])['t'].diff()
    ps['sobre'] = ps['primer_reanudacion'].eq(True) & ((ps['t'] - ps['t0']) <= EXCL_REANUD)
    ps['fid'] = (ps['gap'].isna() | (ps['gap'] > PAUSA)).cumsum()
    ini = ps.drop_duplicates('fid', keep='first')
    ini = ini[(ini['fase'] != 'bp') & ~ini['sobre']].copy()
    # ---- Desenlace ----
    ini['recupera'] = (ini.sig_eq == ini['team.name']) & (ini.sig_per == ini.period) & ((ini.sig_t0 - ini.t) <= 5)
    fo = E[(E['type.name'] == 'Foul Committed') & ~E['foul_committed.offensive'].eq(True)][['match_id', 'possession', 'team.name', 't']]
    j = ini[['fid', 'match_id', 'possession', 'team.name', 't']].merge(fo, on=['match_id', 'possession', 'team.name'], suffixes=('', '_f'))
    ini['falta'] = ini.fid.isin(j.loc[(j.t_f >= j.t) & (j.t_f - j.t <= 5), 'fid'])
    rv = E[E['team.name'] == E['possession_team.name']][['match_id', 'possession', 't', 'loc_x']]
    k = ini[['fid', 'match_id', 'possession', 't', 'loc_x']].merge(rv, on=['match_id', 'possession'], suffixes=('', '_r'))
    k = k[(k.t_r >= k.t) & (k.t_r - k.t <= 10)]
    avance = k.groupby('fid').loc_x_r.max() - (120 - k.groupby('fid').loc_x.first())        # en el marco del rival
    ini['superada'] = ini.fid.map(avance).fillna(-99) >= 15
    ini['desenlace'] = np.select([ini.recupera, ini.falta, ini.superada], ['recupera', 'falta', 'superada'], 'neutra')
    ini['tercio'] = pd.cut(ini.loc_x, [-1, 40, 80, 121], labels=['propio', 'medio', 'alto'])
    ini['carril'] = pd.cut(ini.loc_y, [-1, 80 / 3, 160 / 3, 81], labels=['izq', 'centro', 'der'])
    ini['temporada'] = temp
    fases.append(ini[['match_id', 'team.name', 'temporada', 'tercio', 'carril', 'desenlace']])
    print(f'  {temp}: fases {len(ini):,} · {time.time() - t0:.0f} s')
P = pd.concat(fases, ignore_index=True).rename(columns={'team.name': 'team'})
for d_ in ['recupera', 'falta', 'superada', 'neutra']: P[d_] = 100.0 * (P.desenlace == d_)

ML = P.groupby(['tercio', 'carril'], observed=True)[['recupera', 'falta', 'superada', 'neutra']].mean().round(1)
ML['fases'] = P.groupby(['tercio', 'carril'], observed=True).size()
print(f'\n--- (1) Mapa de la liga ({len(P):,} fases): % de cada desenlace por zona (marco del que presiona)'); print(ML.to_string())
print('  Por tercio:', P.groupby('tercio', observed=True)[['recupera', 'superada']].mean().round(1).to_dict('index'))

# ---- Rasgos por etapa y confiabilidad entre mitades ----
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
ptp['etapa_id'] = ptp.team + ' · ' + ptp.etapa.astype(str)
P = P.merge(ptp[['match_id', 'team', 'etapa_id', 'tec_p']], on=['match_id', 'team'], how='left')
n_et = P.groupby('etapa_id').match_id.nunique(); E15 = n_et[n_et >= 15].index
RAS = {'recupera_alta': ('alto', 'recupera'), 'superada_alta': ('alto', 'superada'), 'recupera_media': ('medio', 'recupera'), 'superada_media': ('medio', 'superada')}
rng = np.random.default_rng(SEMILLA); conf = {}
for r, (ter, col) in RAS.items():
    d = P[(P.tercio == ter) & P.etapa_id.isin(E15)]
    mp = d.groupby(['etapa_id', 'match_id'])[col].agg(['sum', 'size']).reset_index()
    rs = []
    for _ in range(200):
        mp['h'] = rng.random(len(mp)) < 0.5
        a = mp[mp.h].groupby('etapa_id')[['sum', 'size']].sum(); b = mp[~mp.h].groupby('etapa_id')[['sum', 'size']].sum()
        x = (a['sum'] / a['size']).align(b['sum'] / b['size'], join='inner'); rr = np.corrcoef(*x)[0, 1]; rs.append(2 * rr / (1 + rr))
    conf[r] = float(np.median(rs))
print('\n--- (2) Confiabilidad entre mitades (Spearman-Brown, mediana de 200 repartos; etapas de 15+):', {k: round(v, 2) for k, v in conf.items()})
ET_ = pd.DataFrame({r: P[P.tercio == ter].groupby('etapa_id')[col].mean() for r, (ter, col) in RAS.items()})
ET_['fases_altas'] = P[P.tercio == 'alto'].groupby('etapa_id').size(); ET_['partidos'] = n_et
print('  Dispersión entre etapas de 15+ (p10 · mediana · p90):', {r: [round(ET_.loc[E15, r].quantile(q), 1) for q in (.1, .5, .9)] for r in RAS})

# ---- América por técnico ----
am = P[P.team.str.contains('Am[eé]rica', regex=True)]
MA = am.groupby(['tec_p', 'tercio', 'carril'], observed=True)[['recupera', 'superada']].mean().round(1).reset_index()
print('\n--- (3) América por técnico, presión alta (% recupera / % superada):')
print(am[am.tercio == 'alto'].groupby('tec_p')[['recupera', 'superada']].agg(['mean', 'size']).round(1).to_string())
ML.to_csv(os.path.join(SAL5, 'm1_mapa_liga.csv')); ET_.to_csv(os.path.join(SAL5, 'm1_etapas.csv')); MA.to_csv(os.path.join(SAL5, 'm1_america.csv'), index=False)
print(f'\nM1 terminó en {time.time() - t_ini:.0f} s. Guardado: salidas/fw5/m1_mapa_liga.csv, m1_etapas.csv, m1_america.csv. Pega toda la salida.')

In [ ]:
# B7 v1.0 · ¿Jardine moderó la salida corta porque se la castigaban? (hipótesis abierta de Arturo y Claude, plan maestro v6, sección 4)
# =====================================================================
# Sesión 12 (continuación) · 2026-10-19 · Hackathon ISAC 2026, Track 2 · Cinco temporadas (fw1 R1 + posesiones; solo lectura)
# Antecedente (R5b): Jardine llegó con sobrecorrección en el saque de meta corto (+22 → +29) y lo moderó (+3).
# Pregunta: ¿la moderación vino después de perder balones en la salida? Medidas por partido de América con Jardine (bloques de 5):
#   corto = saque de meta que termina antes de x = 40 (M3, copia de R4a: pass_end_x < tercio_medio_x)
#   perdida_salida = saque corto cuya posesión termina en el rival SIN que América haya llegado a x ≥ 40 (marco propio)
#   tiro_rival = la posesión siguiente a esa pérdida termina en tiro rival ≤ 15 s
#   presión rival = presión alta del rival en la temporada (crudo de R4, sin contar ese partido)
# Pruebas (descriptivas; 26 bloques, poco poder): tendencia del % corto; ¿un bloque con más pérdidas precede a uno con menos cortos?
#   (Spearman entre pérdidas del bloque k y cambio del % corto de k a k+1); referencia de la liga para las mismas tasas.
# Apuesta (antes de correr): Claude · Spearman pérdidas → cambio −0.30 (IC con 0) · Arturo: ___
# Orden: celda 1 → B7. ~2-3 min. Escribe salidas/fw5/b7_jardine_bloques.csv.
# =====================================================================

# [B7 v1.0]
import os, json, time
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
with open(os.path.join(FW1, 'parametros_fw1.json')) as f: X_MED = json.load(f)['P']['tercio_medio_x']
t_ini = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet')); ptp['fecha'] = pd.to_datetime(ptp.fecha)
JAR = ptp[ptp.tec_p.astype(str).str.contains('Jardine') & ptp.team.str.contains('Am[eé]rica', regex=True)].sort_values('fecha')
JAR = JAR[JAR.match_id.isin(loc.index)]
print(f'Partidos de Jardine en América con datos: {len(JAR)} (del {JAR.fecha.min().date()} al {JAR.fecha.max().date()})')
filas = []
for temp in sorted(Mm.temporada.unique()):
    s_ = temp.replace('/', '-')
    E = pd.read_parquet(os.path.join(FW1, f'eventos_{s_}.parquet'), columns=['match_id', 'index', 'period', 't', 'type.name', 'team.name', 'possession',
                                                                              'possession_team.name', 'loc_x', 'pass.type.name', 'pass_end_x'])
    E = E[E.match_id.isin(loc.index) & E.period.isin([1, 2])]
    Pz = pd.read_parquet(os.path.join(FW1, f'posesiones_{s_}.parquet'), columns=['match_id', 'possession', 'equipo', 't0', 'period']).sort_values(['match_id', 'possession'])
    Pz['sig_eq'] = Pz.groupby('match_id').equipo.shift(-1); Pz['sig_pos'] = Pz.groupby('match_id').possession.shift(-1)
    sk = E[(E['type.name'] == 'Pass') & (E['pass.type.name'] == 'Goal Kick')].copy()
    sk['corto'] = sk.pass_end_x < X_MED
    mx = E[E['team.name'] == E['possession_team.name']].groupby(['match_id', 'possession']).loc_x.max().rename('x_max')
    sk = sk.join(mx, on=['match_id', 'possession']).merge(Pz[['match_id', 'possession', 'sig_eq', 'sig_pos']], on=['match_id', 'possession'], how='left')
    sk['perdida_salida'] = sk.corto & (sk.sig_eq != sk['team.name']) & sk.sig_eq.notna() & (sk.x_max < X_MED)
    tiros = E[E['type.name'] == 'Shot'][['match_id', 'possession', 'team.name', 't']].rename(columns={'team.name': 'tirador', 't': 't_tiro', 'possession': 'sig_pos'})
    t0p = Pz[['match_id', 'possession', 't0']].rename(columns={'possession': 'sig_pos', 't0': 't0_sig'})
    x = sk[sk.perdida_salida][['match_id', 'index', 'team.name', 'sig_pos']].merge(t0p, on=['match_id', 'sig_pos']).merge(tiros, on=['match_id', 'sig_pos'])
    x = x[(x.tirador != x['team.name']) & ((x.t_tiro - x.t0_sig) <= 15)]
    sk['tiro_rival'] = sk.set_index(['match_id', 'index']).index.isin(x.set_index(['match_id', 'index']).index)
    filas.append(sk[['match_id', 'team.name', 'corto', 'perdida_salida', 'tiro_rival']])
S = pd.concat(filas, ignore_index=True).rename(columns={'team.name': 'team'})
co = S[S.corto]
print(f'\n--- (1) Liga: saques de meta {len(S):,} · % corto {100 * S.corto.mean():.1f} · de los cortos, perdidos en la salida {100 * co.perdida_salida.mean():.1f}%'
      f' · y con tiro rival ≤ 15 s {100 * co.tiro_rival.mean():.2f}%')
# Presión alta del rival en la temporada (crudo de R4, sin contar el partido)
TR = pd.read_parquet(os.path.join(FW5, 'r4_tramos_v21.parquet'), columns=['metrica', 'match_id', 'team', 'num', 'den'])
pa = TR[TR.metrica == 'presion_alta'].groupby(['match_id', 'team'])[['num', 'den']].sum().reset_index()
pa['temporada'] = pa.match_id.map(loc['temporada'])
tot = pa.groupby(['team', 'temporada'])[['num', 'den']].transform('sum')
pa['pres_rival_loo'] = (tot.num - pa.num) / (tot.den - pa.den)
J = S[S.team.str.contains('Am[eé]rica', regex=True) & S.match_id.isin(set(JAR.match_id))]
pm = J.groupby('match_id').agg(saques=('corto', 'size'), pct_corto=('corto', 'mean'), cortos=('corto', 'sum'),
                               perdidas=('perdida_salida', 'sum'), tiros_rival=('tiro_rival', 'sum')).reset_index()
pm = pm.merge(JAR[['match_id', 'fecha']], on='match_id').sort_values('fecha').reset_index(drop=True)
riv = lambda m: loc.at[m, 'visita'] if 'Am' in str(loc.at[m, 'local']) else loc.at[m, 'local']
pm['rival'] = pm.match_id.map(riv)
pm = pm.merge(pa[['match_id', 'team', 'pres_rival_loo']].rename(columns={'team': 'rival'}), on=['match_id', 'rival'], how='left')
pm['bloque'] = pm.index // 5 + 1
BL = pm.groupby('bloque').agg(desde=('fecha', 'min'), partidos=('match_id', 'size'), saques=('saques', 'sum'), cortos=('cortos', 'sum'),
                              perdidas=('perdidas', 'sum'), tiros_rival=('tiros_rival', 'sum'), pres_rival=('pres_rival_loo', 'mean'))
BL['pct_corto'] = 100 * BL.cortos / BL.saques; BL['pct_perdida'] = 100 * BL.perdidas / BL.cortos.replace(0, np.nan)
print(f'\n--- (2) Jardine en América por bloques de 5 partidos (% corto · % de cortos perdidos en la salida · presión alta media de los rivales):')
print(BL[['desde', 'partidos', 'saques', 'pct_corto', 'pct_perdida', 'tiros_rival', 'pres_rival']].round(1).assign(desde=lambda d: d.desde.dt.date).to_string())
tr = spearmanr(BL.index, BL.pct_corto)
d_ = BL.pct_corto.diff().shift(-1); ok = d_.notna() & BL.pct_perdida.notna()
rr = spearmanr(BL.pct_perdida[ok], d_[ok]); rp = spearmanr(pm.pct_corto, pm.pres_rival_loo, nan_policy='omit')
rng = np.random.default_rng(2026); xs = np.c_[BL.pct_perdida[ok].values, d_[ok].values]
bs = [spearmanr(*xs[rng.integers(0, len(xs), len(xs))].T)[0] for _ in range(1000)]
print(f'\n--- (3) Pruebas (descriptivas, {len(BL)} bloques):')
print(f'  Tendencia del % corto a lo largo de la etapa: Spearman {tr[0]:+.2f} (p = {tr[1]:.3f})')
print(f'  Pérdidas del bloque k → cambio del % corto de k a k+1: Spearman {rr[0]:+.2f} [{np.nanpercentile(bs, 2.5):+.2f}, {np.nanpercentile(bs, 97.5):+.2f}]'
      ' (negativo = tras perder más balones en la salida, sale menos en corto)')
print(f'  Por partido: % corto frente a la presión alta del rival: Spearman {rp[0]:+.2f} (p = {rp[1]:.3f}) (negativo = sale menos en corto contra quien presiona)')
print(f'  Referencia de la liga: % de cortos perdidos en la salida {100 * co.perdida_salida.mean():.1f}% · Jardine {100 * J[J.corto].perdida_salida.mean():.1f}%')
BL.to_csv(os.path.join(SAL5, 'b7_jardine_bloques.csv'))
print(f'\nB7 terminó en {time.time() - t_ini:.0f} s. Guardado: salidas/fw5/b7_jardine_bloques.csv. Pega toda la salida.')

#Sesión 13

In [ ]:
# [E-00 v1.0] Inspección previa a la re-descarga de 2026/27 (SOLO LECTURA: no escribe nada, no llama a la API, no calcula juego)
# =====================================================================
# Sesión 13 · 2026-10-19 · Preregistro de Almada, sección 4, paso 1 · Buena práctica: inspeccionar antes de asumir
# El código de descarga y conversión de las sesiones 1-3 (celdas D1, conversión y E3) y el de la tabla de técnicos
# (procesar) no están en el proyecto. Antes de escribir la re-descarga hay que saber EXACTAMENTE qué existe hoy:
#   (1) tablas de partidos (crudo y parquet): fechas de foto, columnas, cuántos de 2026/27 están jugados
#   (2) tabla de técnicos (equipo_partido_tecnico y partido_tecnico_periodo): columnas, tipos, filas por temporada
#   (3) calendario de América en 2026/27 según la tabla de técnicos (fechas e ids; NUNCA marcadores ni juego: D-2)
#   (4) qué partidos de 2026/27 tienen crudo (json.gz) y parquet (eventos, alineaciones, 360) y cuáles están en ptp
#   (5) registros de conversión (validacion_*.jsonl) y tablas derivadas con filas de 2026/27 (Elo, Elo de xG, marcador)
#   (6) esquema de un partido de 2026/27 ya convertido (plantilla para que los nuevos queden idénticos)
# Orden: celda 1 (Drive) → E-00. Tarda < 2 min.
# =====================================================================
import os, re, glob, json, gzip
import pandas as pd
import pyarrow.parquet as pq

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP, DC, SAL = os.path.join(RAIZ, 'datos_parquet'), os.path.join(RAIZ, 'datos_crudos'), os.path.join(RAIZ, 'salidas')
FW1, FW5, FW6 = (os.path.join(DP, f) for f in ('fw1', 'fw5', 'fw6'))
VIVO, SID_VIVO = '2026/2027', 351
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 60)
mid_de = lambda r: int(re.findall(r'\d+', os.path.basename(r))[0])
def titulo(t): print('\n' + '=' * 78 + f'\n{t}\n' + '=' * 78)

# ---------- (1) Tablas de partidos ----------
titulo('(1) Tablas de partidos (fotos guardadas)')
fotos = sorted(glob.glob(os.path.join(DC, 'matches_73_*')) + glob.glob(os.path.join(DP, 'matches_73_*')))
for f in fotos:
    print(f'  {os.path.relpath(f, RAIZ):60s} {os.path.getsize(f) / 1e6:7.2f} MB')
def leer_foto(f):
    if f.endswith('.parquet'):
        return pd.read_parquet(f)
    with (gzip.open if f.endswith('.gz') else open)(f, 'rt', encoding='utf-8') as h:
        return pd.json_normalize(json.load(h))
pq_fotos = [f for f in fotos if f.endswith('.parquet')]
if pq_fotos:
    M = leer_foto(pq_fotos[-1])
    print(f'\n  Última foto parquet: {os.path.basename(pq_fotos[-1])} · filas {len(M)} · columnas ({M.shape[1]}):')
    print('   ', list(M.columns))
    if 'season_label' in M:
        print('  season_id ↔ season_label:', M.drop_duplicates('season_id').set_index('season_id')['season_label'].to_dict())
v_json = [f for f in fotos if not f.endswith('.parquet') and f'_{SID_VIVO}_' in os.path.basename(f)]
if v_json:
    M3 = leer_foto(v_json[-1])
    jug = M3[M3['home_score'].notna()]
    print(f'\n  Última foto cruda de {VIVO}: {os.path.basename(v_json[-1])} · partidos {len(M3)} · jugados {len(jug)} · '
          f'último jugado {jug.match_date.max() if len(jug) else "—"}')
    print('  match_status:', M3['match_status'].value_counts().to_dict() if 'match_status' in M3 else '—',
          '· match_status_360 (jugados):', jug['match_status_360'].value_counts().to_dict() if 'match_status_360' in jug else '—')
    print('  ¿columnas de managers?', [c for c in M3.columns if 'managers' in c])
    print('  ¿columnas de actualización?', [c for c in M3.columns if 'updated' in c or 'last_' in c])
else:
    M3 = None
    print(f'\n  No hay foto cruda separada de {VIVO} (season {SID_VIVO}).')

# ---------- (2) Tabla de técnicos ----------
titulo('(2) Tabla de técnicos')
for nom in ['equipo_partido_tecnico.parquet', 'partido_tecnico_periodo.parquet', 'etapas_tecnico.parquet']:
    p = os.path.join(DP, nom)
    if not os.path.exists(p):
        print(f'  {nom}: NO EXISTE'); continue
    t = pd.read_parquet(p)
    print(f'\n  {nom}: {len(t)} filas · tipos: {dict(t.dtypes.astype(str))}')
    if 'temporada' in t:
        print('  filas por temporada:', t.groupby('temporada').size().to_dict())
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
v = ptp[ptp.temporada == VIVO].copy(); v['fecha_dt'] = pd.to_datetime(v['fecha'])
print(f'\n  ptp {VIVO}: partidos {v.match_id.nunique()} · primero {v.fecha_dt.min().date()} · último {v.fecha_dt.max().date()} · '
      f'roles {v.rol.value_counts().to_dict()}')
print('  Ejemplo de 2 filas (sin marcador):'); print(v.drop(columns='fecha_dt').head(2).to_string(index=False))

# ---------- (3) Calendario de América en 2026/27 (solo fechas e ids) ----------
titulo(f'(3) América en {VIVO} según ptp (calendario; sin marcador ni juego)')
am = v[v.team.str.startswith('Am')].sort_values('fecha_dt')
print(am[['match_id', 'fecha', 'lado', 'tec_p', 'etapa', 'rol']].to_string(index=False))
print(f'  Partidos de América: {len(am)} · técnicos: {am.tec_p.value_counts().to_dict()} · etapa_id: '
      f'{sorted(set(am.team + " · " + am.etapa.astype(str)))}')
if M3 is not None:
    es_am = M3['home_team.home_team_name'].str.startswith('Am') | M3['away_team.away_team_name'].str.startswith('Am')
    a3 = M3[es_am].sort_values('match_date')
    a3 = a3.assign(jugado=a3['home_score'].notna(), en_ptp=a3['match_id'].isin(set(ptp.match_id)))
    print(f'\n  América en la foto cruda {os.path.basename(v_json[-1])} (calendario completo de la foto):')
    print(a3[['match_id', 'match_date', 'jugado', 'en_ptp'] + [c for c in ['match_status_360', 'competition_stage.name'] if c in a3]]
          .to_string(index=False))

# ---------- (4) Cobertura de archivos por partido de 2026/27 ----------
titulo(f'(4) Archivos por partido de {VIVO}')
ids_ptp = set(v.match_id)
ids_foto = set(M3.loc[M3['home_score'].notna(), 'match_id'].astype(int)) if M3 is not None else set()
crudo = {s: {mid_de(f) for f in glob.glob(os.path.join(DC, 'liga', s, '*.json.gz'))} for s in ['eventos', 'lineups', 'frames360']}
parq = {s: {mid_de(f) for f in glob.glob(os.path.join(DP, 'liga', s, '*.parquet'))} for s in ['eventos', 'lineups', 'frames360', 'jugadores360']}
filas = []
for nom, ids in [('en ptp', ids_ptp), ('jugados en la foto cruda', ids_foto), ('en la foto y NO en ptp', ids_foto - ids_ptp),
                 ('América en ptp', set(am.match_id))]:
    f = {'grupo': nom, 'partidos': len(ids)}
    f.update({f'crudo_{s}': len(ids & c) for s, c in crudo.items()})
    f.update({f'parquet_{s}': len(ids & c) for s, c in parq.items()})
    filas.append(f)
print(pd.DataFrame(filas).to_string(index=False))
print('  Totales en carpetas (todas las temporadas): crudo', {s: len(c) for s, c in crudo.items()},
      '· parquet', {s: len(c) for s, c in parq.items()})

# ---------- (5) Registros y tablas derivadas ----------
titulo('(5) Registros de conversión y tablas derivadas con filas de 2026/27')
for f in sorted(glob.glob(os.path.join(SAL, 'validacion_*.jsonl')) + glob.glob(os.path.join(SAL, 'fallos_descarga_*.csv')) +
                glob.glob(os.path.join(SAL, 'ids_etapa3_*.csv')) + glob.glob(os.path.join(DP, 'resumen_partido_etapa*.parquet'))):
    try:
        t = pd.read_json(f, lines=True) if f.endswith('.jsonl') else (pd.read_csv(f) if f.endswith('.csv') else pd.read_parquet(f))
        n_v = int(t['match_id'].astype(int).isin(ids_ptp | ids_foto).sum()) if 'match_id' in t else '—'
        print(f'  {os.path.relpath(f, RAIZ):55s} filas {len(t):5d} · de 2026/27: {n_v}' +
              (f" · season_id {t['season_id'].value_counts().to_dict()}" if 'season_id' in t else ''))
    except Exception as e:
        print(f'  {os.path.relpath(f, RAIZ)}: no se pudo leer ({type(e).__name__})')
for carpeta in [FW1, FW5]:
    for nom in ['elo_partido_e2.parquet', 'elo_equipo_partido_e2.parquet', 'elo_xg_partido.parquet', 'tiros_xg.parquet']:
        p = os.path.join(carpeta, nom)
        if os.path.exists(p):
            t = pd.read_parquet(p)
            fv = t['match_id'].isin(ids_ptp)
            ult = pd.to_datetime(t.loc[fv, 'fecha']).max().date() if 'fecha' in t and fv.any() else '—'
            print(f'  {os.path.relpath(p, RAIZ):45s} filas {len(t):6d} · partidos de 2026/27 {t.loc[fv, "match_id"].nunique():3d} · último {ult}')
p = os.path.join(DP, 'equipo_partido_marcador.parquet')
t = pd.read_parquet(p, columns=['match_id'])
print(f'  equipo_partido_marcador.parquet: partidos de 2026/27 {t.match_id.isin(ids_ptp).sum() // 2}')
with open(os.path.join(FW5, 'parametros_fw1.json')) as h:
    print('  fw5/parametros_fw1.json temporadas:', json.load(h)['temporadas'])
print('  ¿Existe fw6?', os.path.isdir(FW6), '· ¿salidas/fw6?', os.path.isdir(os.path.join(SAL, 'fw6')))
for s in ['eventos', 'posesiones']:
    print(f'  R1 {s}:', {os.path.basename(d): sorted(os.path.basename(x) for x in glob.glob(os.path.join(d, f'{s}_*.parquet')))
                         for d in [FW1, FW5] if os.path.isdir(d)})

# ---------- (6) Esquema de un partido de 2026/27 ya convertido ----------
titulo('(6) Esquema de un partido ya convertido (plantilla)')
ej = sorted(ids_ptp & parq['eventos'] & parq['frames360'])
ej_prev = sorted((set(ptp.loc[ptp.temporada == '2025/2026', 'match_id'])) & parq['eventos'] & parq['frames360'])
for etiqueta, lst in [(VIVO, ej), ('2025/2026', ej_prev)]:
    if not lst:
        print(f'  {etiqueta}: sin partidos con eventos y 360'); continue
    m = lst[-1]
    print(f'\n  {etiqueta} · match_id {m}')
    for s in ['eventos', 'lineups', 'frames360', 'jugadores360']:
        a = os.path.join(DP, 'liga', s, f'{m}.parquet')
        if os.path.exists(a):
            sch = pq.read_schema(a)
            tipos = dict(zip(sch.names, map(str, sch.types)))
            txt = str(tipos) if s != 'eventos' else f'{len(sch.names)} columnas · v11: ' + str({c: c in tipos for c in ['control_degree', 'is_controlled_possession', 'possession_directness', 'set_piece_phase', 'obv_total_net']})
            print(f'    {s:13s} {txt[:600]}')
print('\nE-00 terminó (no escribió nada, no llamó a la API, no calculó juego).')

In [ ]:
# [E-1 v1.0] Re-descarga de 2026/27 con corte fijo 2026-10-19 (preregistro de Almada, sección 4, paso 1)
# =====================================================================
# Sesión 13 · 2026-10-19 · Base: celda E3 de la sesión 3 (descarga de la etapa 3), misma función bajar() sin cambios.
# Qué hace:
#   (1) Foto de la tabla de partidos de 2026/27 (season 351) con fecha 2026-10-19 en datos_crudos/. Si ya existe, NO se vuelve a
#       pedir: se lee la guardada (la foto queda congelada aunque se corra dos veces).
#   (2) Objetivo = partidos jugados (marcador no nulo) con fecha ≤ 2026-10-19 y match_status 'available' (procesados por el
#       proveedor). Los jugados que aún no están procesados NO entran y se listan.
#   (3) Calendario de América hasta el corte: cuáles entran y cuál falta (y por qué).
#   (4) Descarga crudos (json.gz) en datos_crudos/liga/{lineups,eventos,frames360}/ solo de lo que falta (bajar() salta lo que
#       ya existe: los 88 de la etapa 3 no se tocan). Incluye 360 que antes estaba 'unscheduled' y hoy está disponible.
#   (5) Avisa si el proveedor actualizó partidos que ya teníamos (last_updated posterior al 2026-10-07): NO se re-descargan
#       (se conserva la foto con fecha documentada, regla de la sesión 3); solo se reporta.
# Modo: SOLO_CONTAR = True primero (pide solo la tabla de partidos y muestra el plan); luego SOLO_CONTAR = False para bajar.
# Orden: celda 1 (Drive) → E-1. No escribe en fw1 ni en fw5. Tarda ~1 min contando; ~5-10 min descargando.
# =====================================================================
import os, gzip, json, time, requests
import pandas as pd
from google.colab import userdata

SOLO_CONTAR = False                                   # ← primero True; después False

URL_360     = "https://data.statsbomb.com/api/v2/360-frames/{match_id}"
URL_EVENTOS = "https://data.statsbomb.com/api/v11/events/{match_id}"
URL_MATCHES = "https://data.statsbomb.com/api/v6/competitions/{competition_id}/seasons/{season_id}/matches"
URL_LINEUPS = "https://data.statsbomb.com/api/v5/lineups/{match_id}"

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado; corre la celda 1'
assert 'PEGAR' not in URL_MATCHES + URL_LINEUPS + URL_EVENTOS + URL_360, 'Pega las cuatro URLs (las mismas de E3)'
AUTH = (userdata.get('SB_USER'), userdata.get('SB_PASS'))
SID, CORTE, HOY, FOTO_PREV = 351, '2026-10-19', '2026-10-19', '2026-10-07'
CRUDO = os.path.join(RAIZ, 'datos_crudos', 'liga'); SAL = os.path.join(RAIZ, 'salidas')

def bajar(url, ruta):                                # ← copia literal de E3 (sesión 3)
    if os.path.exists(ruta):
        return 'ya existía'
    estado = 'sin intento'
    for _ in range(3):
        try:
            r = requests.get(url, auth=AUTH, timeout=120)
        except Exception as e:
            estado = type(e).__name__; time.sleep(5); continue
        if r.status_code == 200:
            os.makedirs(os.path.dirname(ruta), exist_ok=True)
            with gzip.open(ruta, 'wb') as f:
                f.write(r.content)
            return 'ok'
        estado = f'HTTP {r.status_code}'
        if r.status_code == 404:
            break
        time.sleep(5)
    return estado

# ---------- (1) Foto de la tabla de partidos ----------
ruta_foto = os.path.join(RAIZ, 'datos_crudos', f'matches_73_{SID}_{HOY}.json.gz')
if os.path.exists(ruta_foto):
    origen = 'leída de Drive (ya congelada)'
else:
    r = requests.get(URL_MATCHES.format(competition_id=73, season_id=SID), auth=AUTH, timeout=120)
    r.raise_for_status()
    with gzip.open(ruta_foto, 'wb') as f:
        f.write(r.content)
    origen = 'pedida a la API y guardada'
with gzip.open(ruta_foto, 'rt', encoding='utf-8') as f:
    M = pd.json_normalize(json.load(f))
M['match_id'] = M['match_id'].astype(int)
print(f'--- (1) Foto {os.path.basename(ruta_foto)}: {origen} · partidos {len(M)} · jugados {int(M.home_score.notna().sum())} · '
      f'último jugado {M.loc[M.home_score.notna(), "match_date"].max()}')

# ---------- (2) Objetivo ----------
jug = M[M['home_score'].notna() & (M['match_date'] <= CORTE)].copy()
obj = jug[jug['match_status'] == 'available'].copy()
fuera = jug[jug['match_status'] != 'available']
existe = lambda t, m: os.path.exists(os.path.join(CRUDO, t, f'{m}.json.gz'))
obj['ya_eventos'] = [existe('eventos', m) for m in obj.match_id]
obj['ya_360'] = [existe('frames360', m) for m in obj.match_id]
nuevos = obj[~obj['ya_eventos']]
f360 = obj[obj['ya_eventos'] & ~obj['ya_360'] & (obj['match_status_360'] == 'available')]
print(f'--- (2) Jugados hasta el {CORTE}: {len(jug)} · procesados (entran): {len(obj)} · jugados sin procesar (no entran): {len(fuera)}')
print(f'    Ya descargados: {int(obj.ya_eventos.sum())} · nuevos a bajar: {len(nuevos)} (fechas {nuevos.match_date.min()} a {nuevos.match_date.max()}) · '
      f'360 de los nuevos: {nuevos.match_status_360.value_counts().to_dict()}')
print(f'    Partidos viejos cuyo 360 ahora está disponible (se agrega): {len(f360)}')
if len(fuera):
    print('    Sin procesar:', fuera[['match_id', 'match_date', 'match_status']].to_dict('records'))

# ---------- (3) América ----------
es_am = M['home_team.home_team_name'].str.startswith('Am') | M['away_team.away_team_name'].str.startswith('Am')
am = M[es_am & (M['match_date'] <= CORTE)].sort_values('match_date').copy()
am['jugado'] = am['home_score'].notna()
am['entra'] = am['match_id'].isin(obj['match_id'])
am['nota'] = ['entra' if e else ('no jugado / reprogramado' if not j else 'jugado, sin procesar') for e, j in zip(am.entra, am.jugado)]
print(f'--- (3) América hasta el {CORTE} en el calendario: {len(am)} · entran {int(am.entra.sum())} · faltan {int((~am.entra).sum())}')
print(am[['match_id', 'match_date', 'match_status', 'match_status_360', 'nota']].to_string(index=False))

# ---------- (5) Actualizaciones del proveedor sobre lo que ya teníamos ----------
col_upd = 'last_updated' if 'last_updated' in obj else None
if col_upd:
    viejos = obj[obj['ya_eventos']]
    n_upd = int((viejos[col_upd].astype(str) > FOTO_PREV).sum())
    print(f'--- (5) Ya descargados con {col_upd} posterior al {FOTO_PREV}: {n_upd} de {len(viejos)} (no se re-descargan; se reporta)')

# ---------- (4) Descarga ----------
if SOLO_CONTAR:
    print('\nSOLO_CONTAR = True: no se bajó nada (salvo la foto). Revisa el plan y vuelve a correr con SOLO_CONTAR = False.')
else:
    log = []
    for i, row in enumerate(nuevos.itertuples(), 1):
        mid = int(row.match_id)
        log.append((mid, 'lineups', bajar(URL_LINEUPS.format(match_id=mid), os.path.join(CRUDO, 'lineups', f'{mid}.json.gz'))))
        log.append((mid, 'eventos', bajar(URL_EVENTOS.format(match_id=mid), os.path.join(CRUDO, 'eventos', f'{mid}.json.gz'))))
        if row.match_status_360 == 'available':
            log.append((mid, 'frames360', bajar(URL_360.format(match_id=mid), os.path.join(CRUDO, 'frames360', f'{mid}.json.gz'))))
        if i % 25 == 0: print(f'   {i}/{len(nuevos)}')
    for mid in f360.match_id:
        log.append((int(mid), 'frames360', bajar(URL_360.format(match_id=mid), os.path.join(CRUDO, 'frames360', f'{mid}.json.gz'))))
    lg = pd.DataFrame(log, columns=['match_id', 'tipo', 'estado'])
    print('\n--- (4) Resultado de la descarga:'); print(pd.crosstab(lg['tipo'], lg['estado']).to_string() if len(lg) else '  nada que bajar')
    fallos = lg[~lg['estado'].isin(['ok', 'ya existía'])]
    fallos.to_csv(os.path.join(SAL, f'fallos_descarga_{HOY}.csv'), index=False)
    obj.assign(nuevo=~obj['ya_eventos'])[['match_id', 'match_date', 'match_status_360', 'nuevo']].to_csv(
        os.path.join(SAL, f'ids_e1_corte_{CORTE}.csv'), index=False)
    print(f'Fallos: {len(fallos)} · guardado salidas/ids_e1_corte_{CORTE}.csv ({len(obj)} partidos) y salidas/fallos_descarga_{HOY}.csv')
    am_ok = [m for m in am.loc[am.entra, 'match_id'] if existe('eventos', m)]
    print(f'América: partidos con eventos crudos en Drive hasta el corte: {len(am_ok)} de {int(am.entra.sum())} que entran')
print('\nE-1 terminó.')

In [ ]:
# [E-2 v1.0] Conversión a parquet de los partidos nuevos de 2026/27 (preregistro de Almada, sección 4, paso 1)
# =====================================================================
# Sesión 13 · 2026-10-19 · Funciones de conversión: COPIA LITERAL de la celda de conversión de la sesión 1 (etapa 1/2):
# leer, a_texto, anidado_a_texto, separar_coordenadas, aplanar_eventos, guardar, convertir_360, convertir_lineups, goles_equipos.
# Qué hace:
#   (A) PRUEBA DE REPRODUCCIÓN (buena práctica 13): reconvierte desde el crudo 2 partidos de 2026/27 que YA están en parquet
#       (incluido uno de América) en una carpeta temporal de Colab y exige que salgan IDÉNTICOS a los de Drive en las cuatro
#       tablas. Si no, se detiene sin escribir nada.
#   (B) Convierte SOLO los partidos nuevos de E-1 (salidas/ids_e1_corte_2026-10-19.csv, nuevo = True) a
#       datos_parquet/liga/{eventos,lineups,frames360,jugadores360}/. Nunca sobrescribe un parquet existente.
#       Registro propio: salidas/validacion_e1_2026-10-19.jsonl (no se mezcla con los de las etapas 1 y 2).
#   (C) Chequeos: goles de eventos = marcador, 2 Starting XI, frames 360 enlazados, columnas nuevas frente a las ya existentes.
# Orden: celda 1 (Drive) → E-2. No usa la API. Tarda ~1-2 min.
# =====================================================================
import os, gzip, json, time, glob
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado; corre la celda 1'
BASE_ = Path(RAIZ)
RAW = BASE_ / 'datos_crudos' / 'liga'
OUT = BASE_ / 'datos_parquet' / 'liga'
CORTE, SID = '2026-10-19', 351
LOG = BASE_ / 'salidas' / f'validacion_e1_{CORTE}.jsonl'
PRUEBA = Path('/content/e2_prueba')
PRUEBA_IDS = [4063611, 4063612]                     # 4063611 = América 28-sep (con 360); 4063612 = otro de 2026/27 ya convertido

# ---------- Funciones: copia literal de la sesión 1 (solo leer() recibe RAW igual que antes) ----------
def leer(tipo, mid):
    with gzip.open(RAW / tipo / f'{mid}.json.gz', 'rt', encoding='utf-8') as f:
        return json.load(f)

def a_texto(v):
    if v is None or (isinstance(v, float) and v != v):
        return None
    return json.dumps(v, ensure_ascii=False)

def anidado_a_texto(df):
    for c in df.columns:
        if pd.api.types.is_object_dtype(df[c]):
            if df[c].apply(lambda v: isinstance(v, (list, dict))).any():
                df[c] = df[c].apply(a_texto)
    return df

def separar_coordenadas(df, col, pref):
    if col not in df.columns:
        return
    for i, eje in enumerate(['x', 'y', 'z']):
        vals = df[col].apply(lambda a: a[i] if isinstance(a, list) and len(a) > i else np.nan)
        if eje == 'z' and vals.isna().all():
            continue
        df[f'{pref}_{eje}'] = vals.astype('float32')
    df.drop(columns=[col], inplace=True)

def aplanar_eventos(raw, mid):
    df = pd.json_normalize(raw)
    df.insert(0, 'match_id', mid)
    for col, pref in [('location', 'loc'), ('pass.end_location', 'pass_end'),
                      ('carry.end_location', 'carry_end'), ('shot.end_location', 'shot_end')]:
        separar_coordenadas(df, col, pref)
    return anidado_a_texto(df)

def guardar(df, ruta):
    try:
        df.to_parquet(ruta, index=False, compression='zstd')
        return 'ok'
    except Exception:
        d = df.copy()
        for c in d.columns:
            if pd.api.types.is_object_dtype(d[c]):
                d[c] = d[c].apply(lambda v: None if v is None or (isinstance(v, float) and v != v) else str(v))
        d.to_parquet(ruta, index=False, compression='zstd')
        return 'respaldo_texto'

def convertir_360(raw, mapa, mid):
    fr, jg = [], []
    for f in raw:
        uuid = f.get('event_uuid')
        idx = mapa.get(uuid, -1)
        ff = f.get('freeze_frame') or []
        va = f.get('visible_area') or []
        mates = sum(1 for p in ff if p.get('teammate'))
        fr.append({
            'match_id': mid, 'event_index': idx, 'event_uuid': uuid,
            'line_breaking_pass': f.get('line_breaking_pass'),
            'num_defenders_on_goal_side_of_actor': f.get('num_defenders_on_goal_side_of_actor'),
            'distance_to_nearest_defender': f.get('distance_to_nearest_defender'),
            'ball_receipt_in_space': f.get('ball_receipt_in_space'),
            'ball_receipt_exceeds_distance': f.get('ball_receipt_exceeds_distance'),
            'n_visibles': len(ff), 'n_visibles_companeros': mates,
            'n_visibles_rivales': len(ff) - mates,
            'tiene_actor': any(p.get('actor') for p in ff),
            'area_visible_vacia': len(va) == 0,
            'visible_area': a_texto(va) if va else None,
            'visible_player_counts': a_texto(f.get('visible_player_counts')),
        })
        for p in ff:
            loc = p.get('location') or [np.nan, np.nan]
            jg.append((mid, idx, bool(p.get('teammate')), bool(p.get('actor')),
                       bool(p.get('keeper')), loc[0], loc[1]))
    frames = pd.DataFrame(fr)
    jug = pd.DataFrame(jg, columns=['match_id', 'event_index', 'teammate', 'actor', 'keeper', 'x', 'y'])
    jug = jug.astype({'match_id': 'int32', 'event_index': 'int32', 'x': 'float32', 'y': 'float32'})
    return frames, jug

def convertir_lineups(raw, mid):
    filas = []
    for eq in raw:
        for p in eq.get('lineup') or []:
            r = {'match_id': mid, 'team_id': eq.get('team_id'), 'team_name': eq.get('team_name'),
                 'equipo_formations': a_texto(eq.get('formations')),
                 'equipo_events': a_texto(eq.get('events'))}
            r.update(p)
            filas.append(r)
    return anidado_a_texto(pd.DataFrame(filas))

def goles_equipos(ev):
    e = ev[ev['period'] < 5]
    g = {}
    if 'shot.outcome.name' in e.columns:
        s = e[(e['type.name'] == 'Shot') & (e['shot.outcome.name'] == 'Goal')]
        for t, n in s['team.id'].value_counts().items():
            g[int(t)] = g.get(int(t), 0) + int(n)
    og = e[e['type.name'] == 'Own Goal For']
    for t, n in og['team.id'].value_counts().items():
        g[int(t)] = g.get(int(t), 0) + int(n)
    return g

def convertir_partido(mid, dest):
    """Mismo flujo de la sesión 1: eventos → mapa id→index → alineaciones → 360 (si hay crudo)."""
    ev = aplanar_eventos(leer('eventos', mid), mid)
    r_ev = guardar(ev, dest / 'eventos' / f'{mid}.parquet')
    mapa = dict(zip(ev['id'], ev['index']))
    lu = convertir_lineups(leer('lineups', mid), mid)
    guardar(lu, dest / 'lineups' / f'{mid}.parquet')
    tiene360 = (RAW / 'frames360' / f'{mid}.json.gz').exists()
    frames = None
    if tiene360:
        frames, jug = convertir_360(leer('frames360', mid), mapa, mid)
        guardar(frames, dest / 'frames360' / f'{mid}.parquet')
        guardar(jug, dest / 'jugadores360' / f'{mid}.parquet')
    return ev, lu, frames, r_ev

# ---------- (A) Prueba de reproducción ----------
print('--- (A) Reproducción: reconvertir partidos ya convertidos y comparar con Drive (deben ser idénticos) ---')
for c in ['eventos', 'frames360', 'jugadores360', 'lineups']:
    (PRUEBA / c).mkdir(parents=True, exist_ok=True)
ok_rep = True
for mid in PRUEBA_IDS:
    convertir_partido(mid, PRUEBA)
    for c in ['eventos', 'lineups', 'frames360', 'jugadores360']:
        a, b = OUT / c / f'{mid}.parquet', PRUEBA / c / f'{mid}.parquet'
        if not a.exists() and not b.exists():
            print(f'  {mid} {c:13s} sin archivo en ninguno (sin 360): ok'); continue
        A, Bn = pd.read_parquet(a), pd.read_parquet(b)
        try:
            pd.testing.assert_frame_equal(A, Bn, check_dtype=False, check_like=True)
            print(f'  {mid} {c:13s} idéntico · {A.shape[0]} filas × {A.shape[1]} columnas')
        except AssertionError as e:
            ok_rep = False
            print(f'  {mid} {c:13s} DISTINTO · columnas solo en Drive {sorted(set(A.columns) - set(Bn.columns))[:5]} · '
                  f'solo en la nueva {sorted(set(Bn.columns) - set(A.columns))[:5]} · {str(e)[:200]}')
assert ok_rep, 'La reconversión NO reproduce los parquet existentes: no se convierte nada. Manda esta salida a Claude.'

# ---------- (B) Conversión de los nuevos ----------
ids = pd.read_csv(BASE_ / 'salidas' / f'ids_e1_corte_{CORTE}.csv')
nuevos = ids.loc[ids['nuevo'].astype(bool), 'match_id'].astype(int).tolist()
with gzip.open(BASE_ / 'datos_crudos' / f'matches_73_{SID}_{CORTE}.json.gz', 'rt', encoding='utf-8') as f:
    M = pd.json_normalize(json.load(f)).set_index('match_id')
hechos = set()
if LOG.exists():
    hechos = {json.loads(l)['match_id'] for l in open(LOG) if l.strip()}
pend = [m for m in nuevos if m not in hechos and not (OUT / 'eventos' / f'{m}.parquet').exists()]
print(f'\n--- (B) Nuevos en E-1: {len(nuevos)} · ya convertidos: {len(nuevos) - len(pend)} · a convertir ahora: {len(pend)}')
errores, respaldo = [], 0
t0 = time.time()
with open(LOG, 'a') as flog:
    for mid in pend:
        try:
            ev, lu, frames, r_ev = convertir_partido(mid, OUT)
            respaldo += r_ev != 'ok'
            m = M.loc[mid]
            g = goles_equipos(ev)
            gl, gv = g.get(int(m['home_team.home_team_id']), 0), g.get(int(m['away_team.away_team_id']), 0)
            fila = {'match_id': mid, 'season_id': SID, 'n_eventos': int(len(ev)), 'n_columnas': int(ev.shape[1]),
                    'n_starting_xi': int((ev['type.name'] == 'Starting XI').sum()),
                    'periodo_max': int(ev['period'].max()),
                    'n_lineup_min_equipo': int(lu.groupby('team_id').size().min()) if len(lu) else 0,
                    'tiene_360': frames is not None, 'n_frames': 0 if frames is None else int(len(frames)),
                    'frames_sin_evento': 0 if frames is None else int((frames['event_index'] < 0).sum()),
                    'coincide_marcador': bool(gl == int(m['home_score']) and gv == int(m['away_score']))}
            flog.write(json.dumps(fila) + '\n'); flog.flush()
        except Exception as e:
            errores.append((mid, type(e).__name__, str(e)[:150]))
print(f'  convertidos en {time.time() - t0:.0f} s · errores {len(errores)} · eventos con respaldo de texto {respaldo}')
for e in errores[:10]: print('   ', e)

# ---------- (C) Chequeos ----------
val = pd.read_json(LOG, lines=True).drop_duplicates('match_id', keep='last') if LOG.exists() else pd.DataFrame()
print(f'\n--- (C) Registro {LOG.name}: {len(val)} partidos')
if len(val):
    print(f"  goles de eventos ≠ marcador: {int((~val.coincide_marcador).sum())} · Starting XI ≠ 2: {int((val.n_starting_xi != 2).sum())} · "
          f"periodo máximo {val.periodo_max.value_counts().to_dict()} · con 360: {int(val.tiene_360.sum())} · "
          f"frames 360 sin evento enlazado: {int(val.frames_sin_evento.sum())} de {int(val.n_frames.sum())} · mín. jugadores por equipo {int(val.n_lineup_min_equipo.min())}")
viejos = sorted(set(ids.loc[~ids['nuevo'].astype(bool), 'match_id'].astype(int)))[:40]
col_v = set().union(*[set(pq.read_schema(OUT / 'eventos' / f'{m}.parquet').names) for m in viejos])
tip_v = {}
for m in viejos:
    s = pq.read_schema(OUT / 'eventos' / f'{m}.parquet'); tip_v.update({n: str(t) for n, t in zip(s.names, s.types) if str(t) != 'null'})
col_n, conf = set(), {}
for m in nuevos:
    p = OUT / 'eventos' / f'{m}.parquet'
    if not p.exists(): continue
    s = pq.read_schema(p); col_n |= set(s.names)
    for n, t in zip(s.names, s.types):
        if str(t) != 'null' and n in tip_v and tip_v[n] != str(t): conf[n] = (tip_v[n], str(t))
print(f'  Columnas de eventos: nuevas que no estaban en 40 partidos previos de 2026/27: {sorted(col_n - col_v)[:10]} · '
      f'conflictos de tipo: {conf if conf else "ninguno"}')
am = [4063616, 4063629]
print('  América nuevos (11-oct, 18-oct):', {m: {c: (OUT / c / f'{m}.parquet').exists() for c in ['eventos', 'lineups', 'frames360']} for m in am})
print('\nE-2 terminó.')

In [ ]:
# [E-3 v1.0] Tabla de técnicos con 2026/27 al corte 2026-10-19, en datos_parquet/fw6/ (preregistro de Almada, sección 4, paso 2)
# =====================================================================
# Sesión 13 · 2026-10-19 · Funciones: COPIA LITERAL de la sesión 1: lado() (equipo_partido_tecnico) y procesar() (D-5: suspensión
# ≤ 5 partidos A-B-A → titular; interino ≤ 3 entre dos técnicos distintos; la última racha nunca es interina).
# Qué hace:
#   (1) Filas de 2021/22 a 2025/26: las de equipo_partido_tecnico.parquet tal cual. Filas de 2026/27: rehechas con lado() desde la
#       foto del 2026-10-19 (E-1). Compara las de 2026/27 que ya existían (técnico registrado) con las nuevas.
#   (2) procesar() sobre todo, igual que en la sesión 1.
#   (3) VALIDACIÓN OBLIGATORIA: en 2021/22-2025/26 la tabla nueva debe ser IDÉNTICA a la de Drive (cada columna, cada equipo-partido).
#       En los 88 partidos de 2026/27 ya existentes se reporta cualquier cambio (puede pasar: una racha que era la última deja de serlo).
#   (4) Escribe SOLO en datos_parquet/fw6/: partido_tecnico_periodo.parquet, etapas_tecnico.parquet, equipo_partido_tecnico.parquet.
#       La tabla de Drive (datos_parquet/partido_tecnico_periodo.parquet) no se toca.
# Orden: celda 1 (Drive) → E-1 ya corrido → E-3. No usa la API. Tarda < 30 s.
# =====================================================================
import os, gzip, json
import numpy as np
import pandas as pd

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado; corre la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW6 = os.path.join(DP, 'fw6')
VIVO, SID, CORTE = '2026/2027', 351, '2026-10-19'
UMBRAL_SUSP, UMBRAL_INT = 5, 3                    # D-5 (sesión 1)
pd.set_option('display.width', 220)

# ---------- lado(): copia literal de la sesión 1 ----------
def n_tec(m):
    try: return len(m)
    except Exception: return 0

def campo(m, k):
    try: return m[0][k] if len(m) > 0 else None
    except Exception: return None

def lado(df, s):
    o = pd.DataFrame({
        'match_id': df['match_id'], 'fecha': df['match_date'],
        'temporada': df['season_label'], 'etapa': df['competition_stage.name'],
        'jugado': df['home_score'].notna(),
        'team': df[f'{s}_team.{s}_team_name'],
        'n_tec': df[f'{s}_team.managers'].apply(n_tec),
        'tec_id': df[f'{s}_team.managers'].apply(lambda m: campo(m, 'id')),
        'tec': df[f'{s}_team.managers'].apply(lambda m: campo(m, 'name')),
    })
    o['lado'] = s
    return o

# ---------- procesar(): copia literal de la sesión 1 ----------
def procesar(g):
    g = g.sort_values(['fecha', 'match_id']).reset_index(drop=True)
    g['tec_id_p'] = g['tec_id']
    g['tec_p'] = g['tec']
    g['rol'] = 'titular'
    while True:
        g['etapa'] = (g['tec_id_p'] != g['tec_id_p'].shift()).cumsum()
        r = g.groupby('etapa').agg(n=('match_id', 'count'), tid=('tec_id_p', 'first'))
        ids = r.index.tolist()
        movido = False
        for k in range(1, len(ids) - 1):
            a, b, c = r.iloc[k - 1], r.iloc[k], r.iloc[k + 1]
            if a['tid'] == c['tid'] and b['tid'] != a['tid'] and b['n'] <= UMBRAL_SUSP:
                m = g['etapa'] == ids[k]
                nombre = g.loc[g['etapa'] == ids[k - 1], 'tec_p'].iloc[0]
                g.loc[m, 'rol'] = 'sustituto_suspension'
                g.loc[m, 'tec_id_p'] = a['tid']
                g.loc[m, 'tec_p'] = nombre
                movido = True
                break
        if not movido:
            break
    tam = g.groupby('etapa')['match_id'].transform('count')
    ultima = g['etapa'].max()
    g.loc[(tam <= UMBRAL_INT) & (g['etapa'] < ultima) & (g['rol'] == 'titular'), 'rol'] = 'interino'
    return g

# ---------- (1) Filas ----------
tm_old = pd.read_parquet(os.path.join(DP, 'equipo_partido_tecnico.parquet'))
with gzip.open(os.path.join(RAIZ, 'datos_crudos', f'matches_73_{SID}_{CORTE}.json.gz'), 'rt', encoding='utf-8') as f:
    M = pd.json_normalize(json.load(f))
M['season_label'] = VIVO
M = M[M['match_date'] <= CORTE]                                       # corte fijo (los programados después no entran)
tm_new = pd.concat([lado(M, 'home'), lado(M, 'away')], ignore_index=True)
tm_new['match_id'] = tm_new['match_id'].astype('int64'); tm_new['tec_id'] = tm_new['tec_id'].astype(float)
prev = tm_old[tm_old.temporada == VIVO]
cmp = prev[prev.jugado].merge(tm_new, on=['match_id', 'team'], how='left', suffixes=('_ant', '_nue'))
dif = cmp[(cmp.tec_id_ant != cmp.tec_id_nue) | (cmp.n_tec_ant != cmp.n_tec_nue)]
print(f'--- (1) 2026/27: filas jugadas en la tabla anterior {int(prev.jugado.sum())} · en la nueva (corte {CORTE}) {int(tm_new.jugado.sum())} · '
      f'técnico registrado distinto en filas que ya existían: {len(dif)}')
if len(dif): print(dif[['match_id', 'fecha_ant', 'team', 'tec_ant', 'tec_nue']].to_string(index=False))
tm = pd.concat([tm_old[tm_old.temporada != VIVO], tm_new], ignore_index=True)

# ---------- (2) procesar ----------
j = tm[tm['jugado'] & (tm['n_tec'] > 0)].copy()
jj = pd.concat([procesar(g) for _, g in j.groupby('team')], ignore_index=True)
ep = (jj.groupby(['team', 'etapa'])
        .agg(tec=('tec_p', 'first'), partidos=('match_id', 'count'),
             de_sustituto=('rol', lambda s: (s == 'sustituto_suspension').sum()),
             desde=('fecha', 'min'), hasta=('fecha', 'max'),
             rol=('rol', lambda s: 'interino' if (s == 'interino').any() else 'titular'))
        .reset_index())

# ---------- (3) Validación ----------
ptp_d = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
k = ['match_id', 'team']; cols = [c for c in ptp_d.columns if c not in k]
J = ptp_d.merge(jj, on=k, how='outer', suffixes=('_d', '_n'), indicator=True)
hist = J['temporada_d'].fillna(J['temporada_n']) != VIVO
dist = pd.Series(False, index=J.index)
for c in cols:
    a, b = J[f'{c}_d'], J[f'{c}_n']
    dist |= ~((a == b) | (a.isna() & b.isna()))
h = J[hist]
print(f'\n--- (3) Validación 2021/22-2025/26 · filas Drive {int((ptp_d.temporada != VIVO).sum())} · nueva {int((jj.temporada != VIVO).sum())} · '
      f'solo en una {int((h._merge != "both").sum())} · con alguna columna distinta {int(dist[hist & (J._merge == "both")].sum())} (las dos deben ser 0)')
OK = bool((h._merge == 'both').all() and not dist[hist & (J._merge == 'both')].any())
v88 = J[~hist & (J._merge == 'both')]
cam = v88[dist[~hist & (J._merge == 'both')]]
print(f'2026/27 · equipo-partidos ya existentes {len(v88)} · con cambios {len(cam)} · nuevos {int((J[~hist]._merge == "right_only").sum())} · '
      f'desaparecidos {int((J[~hist]._merge == "left_only").sum())}')
if len(cam): print(cam[['match_id', 'team', 'fecha_d', 'tec_p_d', 'tec_p_n', 'etapa_d', 'etapa_n', 'rol_d', 'rol_n']].to_string(index=False))
v = jj[jj.temporada == VIVO]
print(f'2026/27 nueva · partidos {v.match_id.nunique()} · último {v.fecha.max()} · roles {v.rol.value_counts().to_dict()}')
am = v[v.team.str.startswith('Am')].sort_values('fecha')
print(f"América 2026/27 · partidos {len(am)} · {am.tec_p.value_counts().to_dict()} · etapa_id {sorted(set(am.team + ' · ' + am.etapa.astype(str)))} · "
      f"roles {am.rol.value_counts().to_dict()} · fechas {am.fecha.min()} a {am.fecha.max()}")

# ---------- (4) Guardar en fw6 ----------
assert OK, 'La tabla nueva NO reproduce la histórica: no se escribe nada. Manda esta salida a Claude.'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado'
os.makedirs(FW6, exist_ok=True)
jj.to_parquet(os.path.join(FW6, 'partido_tecnico_periodo.parquet'))
ep.to_parquet(os.path.join(FW6, 'etapas_tecnico.parquet'))
tm.to_parquet(os.path.join(FW6, 'equipo_partido_tecnico.parquet'))
print(f'\nVALIDACIÓN OK · guardado en datos_parquet/fw6/: partido_tecnico_periodo ({len(jj)} filas), etapas_tecnico ({len(ep)}), '
      f'equipo_partido_tecnico ({len(tm)}). La tabla de Drive no se tocó.')

In [ ]:
# [E-4 v1.0] R1 (objetos base) de 2026/27 al corte 2026-10-19, escrito en datos_parquet/fw6/ (preregistro, sección 4, paso 2)
# =====================================================================
# Sesión 13 · 2026-10-19 · procesar_partido(), COLS y BOOLS: COPIA LITERAL de claude/R1_objetos_base_framework_v1.py.
# Parámetros: se LEEN de datos_parquet/fw1/parametros_fw1.json (lo que escribió R0), sin volver a correr R0 (R0 escribe en fw1).
# Qué hace:
#   (A) PRUEBA DE REPRODUCCIÓN: procesa 2 partidos de 2026/27 que ya están en fw1/eventos_2026-2027 y exige que eventos y
#       posesiones salgan IDÉNTICOS. Si no, se detiene sin escribir.
#   (B) Procesa los 106 partidos de 2026/27 de la tabla de técnicos de fw6 y escribe fw6/eventos_2026-2027.parquet y
#       fw6/posesiones_2026-2027.parquet. fw1 no se toca.
#   (C) Validación completa: en los 88 partidos que ya existían, fw6 debe ser idéntico a fw1. Resumen compacto de los 18 nuevos.
# Orden: celda 1 (Drive) → E-2 y E-3 ya corridos → E-4. Tarda ~2-4 min.
# =====================================================================
import os, glob, re, json, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado; corre la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); LIGA = os.path.join(DP, 'liga')
FW1, FW6 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw6')
VIVO = '2026/2027'; T_ = VIVO.replace('/', '-')
PRUEBA_IDS = [4063611, 4063612]
assert os.path.exists(os.path.join(FW6, 'partido_tecnico_periodo.parquet')), 'Falta fw6/partido_tecnico_periodo: corre E-3 primero'

with open(os.path.join(FW1, 'parametros_fw1.json')) as f:          # lo que fijó R0 (sesión 4)
    CFG = json.load(f)
P = CFG['P']
REANUD_PASE, REANUD_TIRO = CFG['reanud_pase'], CFG['reanud_tiro']
PATRON_RECUPERACION, TIPOS_CONTRAGOLPE = CFG['patron_recuperacion'], CFG['tipos_contragolpe']

# ---------- COPIA LITERAL DE R1 (COLS, BOOLS, procesar_partido) ----------
COLS = ["index", "period", "timestamp", "minute", "second", "duration", "type.name",
        "possession", "possession_team.id", "possession_team.name", "play_pattern.name",
        "team.id", "team.name", "player.id", "position.id", "loc_x", "loc_y",
        "under_pressure", "counterpress", "set_piece_phase", "location_bucket",
        "control_degree", "is_controlled_possession", "possession_directness",
        "pass.length", "pass.angle", "pass.type.name", "pass.outcome.name", "pass.height.name",
        "pass.recipient.id", "pass.cross", "pass.switch", "pass.pass_success_probability",
        "pass_end_x", "pass_end_y", "carry_end_x", "carry_end_y", "ball_receipt.outcome.name",
        "shot.statsbomb_xg", "shot.type.name", "shot.outcome.name",
        "duel.type.name", "duel.outcome.name", "interception.outcome.name",
        "ball_recovery.recovery_failure",
        "foul_committed.offensive", "foul_committed.card.name", "bad_behaviour.card.name",
        "substitution.replacement.id", "substitution.outcome.name", "tactics.formation",
        "out", "off_camera", "obv_total_net"]
BOOLS = ["under_pressure", "counterpress", "pass.cross", "pass.switch",
         "ball_recovery.recovery_failure", "foul_committed.offensive", "out", "off_camera"]


def procesar_partido(mid, ruta):
    nombres = set(pq.read_schema(ruta).names)
    ev = pd.read_parquet(ruta, columns=[c for c in COLS if c in nombres])
    for c in COLS:
        if c not in ev.columns:
            ev[c] = np.nan
    ev = ev.sort_values("index").reset_index(drop=True)
    ev.insert(0, "match_id", mid)
    ev["v11"] = "control_degree" in nombres
    for c in BOOLS:
        ev[c] = ev[c].eq(True)
    ev["t"] = pd.to_timedelta(ev["timestamp"]).dt.total_seconds()   # segundos dentro del periodo
    ev = ev.drop(columns="timestamp")
    tipo = ev["type.name"]

    # O-2 · reanudaciones (pases y tiros)
    ev["es_reanudacion"] = ev["pass.type.name"].isin(REANUD_PASE) | ev["shot.type.name"].isin(REANUD_TIRO)

    # O-1 + S4-05 · el balón parado es del equipo que lo ejecuta y acaba cuando el rival controla
    marca = ev["set_piece_phase"].notna()
    dueno = ev["team.id"].where(ev["es_reanudacion"] & marca)
    ev["bp_equipo"] = dueno.groupby(ev["period"]).ffill()
    ev["bp_sin_dueno"] = marca & ev["bp_equipo"].isna()
    ev["es_bp"] = marca & ((ev["possession_team.id"] == ev["bp_equipo"]) | ev["bp_sin_dueno"])

    # O-3 / O-12 · tabla de posesiones y su tiempo (B decidido; C sensibilidad)
    cb = ev[ev["loc_x"].notna()]
    g = cb.groupby("possession")
    pos = pd.DataFrame({
        "period": g["period"].first(),
        "equipo_id": g["possession_team.id"].first(),
        "equipo": g["possession_team.name"].first(),
        "patron": g["play_pattern.name"].first(),
        "primer_tipo": g["type.name"].first(),
        "primer_reanudacion": g["es_reanudacion"].first().astype(bool),
        "inicia_bp": g["es_bp"].first().astype(bool),
        "t0": g["t"].min(),
        "t1": (cb["t"] + cb["duration"].fillna(0)).groupby(cb["possession"]).max(),
        "controlada": g["is_controlled_possession"].first(),
        "directness": g["possession_directness"].first(),
    })
    pos["dur_B"] = (pos["t1"] - pos["t0"]).clip(lower=0)
    t_sig = g["t"].shift(-1)
    dt = (t_sig - cb["t"]).clip(lower=0)
    ultimo = t_sig.isna()
    dt = dt.where(~ultimo, cb["duration"].fillna(0))
    dt = dt.where(ultimo | (dt <= P["tope_intervalo_C_s"]), 0)
    pos["dur_C"] = dt.groupby(cb["possession"]).sum()

    # O-6 · posesión que nace de una recuperación en juego abierto
    pos["nace_rec"] = (pos["patron"].isin(PATRON_RECUPERACION)
                       & ~pos["primer_reanudacion"] & (pos["period"] <= 4))

    # O-7 · transición (se guarda t_pos para poder probar 15 s sin recalcular)
    ev = ev.join(pos[["t0", "nace_rec"]], on="possession")
    ev["nace_rec"] = ev["nace_rec"].eq(True)
    ev["t_pos"] = ev["t"] - ev["t0"]
    ev["en_transicion"] = ev["nace_rec"] & ev["t_pos"].between(0, P["transicion_s"])

    # O-11 · jerarquía de fases: balón parado > transición > establecido
    ev["fase"] = np.select([ev["es_bp"], ev["en_transicion"]], ["bp", "transicion"], default="establecido")

    # O-8 · contragolpe
    propio = ev["team.id"] == ev["possession_team.id"]
    m_cg = (ev["en_transicion"] & propio & (ev["loc_x"] >= P["tercio_ultimo_x"])
            & tipo.isin(TIPOS_CONTRAGOLPE))
    pos["contragolpe"] = pos.index.isin(ev.loc[m_cg, "possession"].unique()) & pos["nace_rec"]

    # xG sin penales y tiros por posesión
    tiro = (tipo == "Shot") & (ev["shot.type.name"] != "Penalty")
    pos["xg_np"] = ev.loc[tiro].groupby("possession")["shot.statsbomb_xg"].sum().reindex(pos.index).fillna(0)
    pos["tiros"] = ev.loc[tiro].groupby("possession").size().reindex(pos.index).fillna(0).astype(int)

    # O-9 · pausa desde la presión anterior del mismo equipo en la misma posesión rival
    pr = ev.loc[tipo == "Pressure"]
    ev["presion_gap_prev"] = pr.groupby(["possession", "team.id"])["t"].diff()

    # O-13 · diferencia de goles ANTES del evento, vista desde el equipo del evento
    gol = (((tipo == "Shot") & (ev["shot.outcome.name"] == "Goal") & (ev["period"] <= 4))
           | (tipo == "Own Goal For"))
    equipos = list(ev["team.id"].dropna().unique())
    if len(equipos) == 2:
        a, b = equipos
        prev = {}
        for e in equipos:
            ge = (gol & (ev["team.id"] == e)).astype(int)
            prev[e] = ge.cumsum() - ge
        ev["dif_antes"] = np.where(ev["team.id"] == a, prev[a] - prev[b],
                                   np.where(ev["team.id"] == b, prev[b] - prev[a], np.nan))
    else:
        ev["dif_antes"] = np.nan

    # P-9 · ventana de 5 minutos (0-8 primer tiempo con añadido en la 8; 9-17 segundo; 18 = añadido 2T)
    m = ev["minute"]
    ev["ventana"] = np.select(
        [ev["period"] == 1, ev["period"] == 2],
        [(m // 5).clip(upper=8), np.where(m >= 90, 18, ((m - 45) // 5).clip(lower=0, upper=8) + 9)],
        default=-1)

    pos = pos.reset_index()
    pos.insert(0, "match_id", mid)
    ev = ev.drop(columns=["t0"])
    return ev, pos
# ---------- FIN DE LA COPIA LITERAL ----------

def igual(A, B):
    try:
        pd.testing.assert_frame_equal(A.reset_index(drop=True), B.reset_index(drop=True), check_dtype=False, check_like=True)
        return True, ''
    except AssertionError as e:
        return False, str(e)[:250]

ruta = lambda m: os.path.join(LIGA, 'eventos', f'{m}.parquet')
E1 = pd.read_parquet(os.path.join(FW1, f'eventos_{T_}.parquet'))
P1 = pd.read_parquet(os.path.join(FW1, f'posesiones_{T_}.parquet'))

# ---------- (A) Reproducción ----------
print('--- (A) Reproducción contra fw1 (deben ser idénticos) ---')
ok = True
for mid in PRUEBA_IDS:
    e, p = procesar_partido(mid, ruta(mid))
    r1, m1 = igual(E1[E1.match_id == mid].sort_values('index'), e.sort_values('index'))
    r2, m2 = igual(P1[P1.match_id == mid].sort_values('possession'), p.sort_values('possession'))
    ok &= r1 and r2
    print(f'  {mid}: eventos {"idénticos" if r1 else "DISTINTOS " + m1} · posesiones {"idénticas" if r2 else "DISTINTAS " + m2}')
assert ok, 'R1 copiado NO reproduce fw1: no se escribe nada. Manda esta salida a Claude.'

# ---------- (B) Todos los partidos de 2026/27 de fw6 ----------
ptp6 = pd.read_parquet(os.path.join(FW6, 'partido_tecnico_periodo.parquet'))
ids = sorted(ptp6.loc[ptp6.temporada == VIVO, 'match_id'].unique())
evs, poss, fallos = [], [], []
t0 = time.time()
for mid in ids:
    try:
        e, p = procesar_partido(int(mid), ruta(int(mid))); evs.append(e); poss.append(p)
    except Exception as err:
        fallos.append((mid, repr(err)[:160]))
E6, P6 = pd.concat(evs, ignore_index=True), pd.concat(poss, ignore_index=True)
print(f'\n--- (B) 2026/27: {len(evs)} de {len(ids)} partidos · fallos {len(fallos)} · {len(E6):,} eventos · {len(P6):,} posesiones · {time.time() - t0:.0f} s')
for f_ in fallos[:5]: print('   FALLO', f_)
assert not fallos, 'Hubo fallos: no se escribe nada. Manda esta salida a Claude.'

# ---------- (C) Validación completa contra fw1 y resumen ----------
viejos = sorted(set(E1.match_id) & set(ids))
faltan = sorted(set(E1.match_id) - set(ids))
print(f'Partidos de fw1 que NO están en la tabla de fw6: {len(faltan)} {faltan[:5]} (debe ser 0)')
rE, mE = igual(E1[E1.match_id.isin(viejos)].sort_values(['match_id', 'index']), E6[E6.match_id.isin(viejos)].sort_values(['match_id', 'index']))
rP, mP = igual(P1[P1.match_id.isin(viejos)].sort_values(['match_id', 'possession']), P6[P6.match_id.isin(viejos)].sort_values(['match_id', 'possession']))
print(f'--- (C) Los {len(viejos)} partidos que ya estaban en fw1: eventos {"idénticos" if rE else "DISTINTOS " + mE} · '
      f'posesiones {"idénticas" if rP else "DISTINTAS " + mP}')
assert rE and rP and not faltan, 'fw6 no coincide con fw1 en los partidos viejos: no se escribe nada. Manda esta salida a Claude.'
nv = E6[~E6.match_id.isin(viejos)]; pn = P6[~P6.match_id.isin(viejos)]
est = nv[(nv.period <= 2) & nv.loc_x.notna()]
fases = (est.fase.value_counts(normalize=True) * 100).round(1).to_dict()
pm = pn[pn.period <= 2].groupby('match_id').dur_B.sum() / 60
print(f'  Nuevos: {nv.match_id.nunique()} partidos · eventos por partido {len(nv) / max(1, nv.match_id.nunique()):.0f} · % eventos por fase {fases} · '
      f'posesión B por partido {pm.mean():.1f} min · v11 en todos: {bool(nv.groupby("match_id").v11.first().all())} · '
      f'bp sin dueño {int(nv.bp_sin_dueno.sum())}')
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado'
E6.to_parquet(os.path.join(FW6, f'eventos_{T_}.parquet'), index=False)
P6.to_parquet(os.path.join(FW6, f'posesiones_{T_}.parquet'), index=False)
print(f'\nVALIDACIÓN OK · guardado fw6/eventos_{T_}.parquet y fw6/posesiones_{T_}.parquet. fw1 no se tocó.')

In [ ]:
# [E-5 v1.0] Elo (E-2) y Elo de xG extendidos a los partidos de 2026/27 hasta el 2026-10-19, en datos_parquet/fw6/
# =====================================================================
# Sesión 13 · 2026-10-19 · Preregistro de Almada, sección 4, paso 3 · Base: R3 (sesión 5) y E11b (sesión 9).
# COPIAS LITERALES de R3: _leer_matches, _col, CANDIDATAS, cargar_matches (R3-0); armado de la tabla de partidos (R3a);
# correr_elo (R3-0, versión de E11b con arranque como argumento); lectura de tiros, _dist_goles y correr_elo_s (R3e).
# Configuración congelada (bitácora v8 §3): K 20 · H 75 · R verano 0.25 · R invierno 0 · arranque por tabla.
# El arranque se toma del propio archivo de fw1 (Elo con el que cada equipo de 2021/22 jugó su primer partido), como en E11b.
# Qué hace:
#   (A) Arma la tabla de partidos con la tabla de técnicos de fw6 (1,807 partidos) y la foto de partidos más reciente.
#   (B) REPRODUCCIÓN: el Elo y el Elo de xG recalculados deben ser IDÉNTICOS a los de fw1 en los 1,789 partidos que ya estaban
#       (diferencia máxima < 1e-6). Si no, se detiene sin escribir.
#   (C) Escribe en fw6: elo_partido_e2, elo_equipo_partido_e2, elo_xg_partido, tiros_xg. fw1 no se toca.
# REGLA DE HOLDOUT (R3): no imprime resultados de 2025/26 ni de 2026/27; solo conteos y diferencias.
# Orden: celda 1 (Drive) → E-3 ya corrido → E-5. Tarda ~1-2 min.
# =====================================================================
import os, re, glob, json, gzip, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado; corre la celda 1'
BASE = RAIZ
DP = os.path.join(BASE, 'datos_parquet'); DC = os.path.join(BASE, 'datos_crudos')
FW1, FW6 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw6')
E2 = dict(K=20, H=75, R_ver=0.25, R_inv=0.0)
INICIO = 1500.0
CALENTAMIENTO = pd.Timestamp('2021-10-01')
VIVO = '2026/2027'
t0 = time.time()

# ---------- Copia literal de R3-0: carga de partidos ----------
def _leer_matches(ruta):
    if ruta.endswith('.parquet'):
        df = pd.read_parquet(ruta)
        obj = [c for c in df.columns if df[c].dtype == object]
        if any(df[c].map(lambda v: isinstance(v, dict)).any() for c in obj):
            df = pd.json_normalize(df.to_dict('records'))
    else:
        abrir = gzip.open if ruta.endswith('.gz') else open
        with abrir(ruta, 'rt', encoding='utf-8') as f:
            df = pd.json_normalize(json.load(f))
    m = re.search(r'(\d{4}-\d{2}-\d{2})', os.path.basename(ruta))
    df['_fecha_archivo'] = m.group(1) if m else '0000-00-00'
    df['_archivo'] = os.path.basename(ruta)
    return df

def _col(df, opciones):
    for o in opciones:
        if o in df.columns:
            return o
    return None

CANDIDATAS = {
    'match_id': ['match_id'],
    'fecha_m': ['match_date'],
    'hora': ['kick_off'],
    'local_id': ['home_team.home_team_id', 'home_team_id'],
    'local': ['home_team.home_team_name', 'home_team_name'],
    'visita_id': ['away_team.away_team_id', 'away_team_id'],
    'visita': ['away_team.away_team_name', 'away_team_name'],
    'gl': ['home_score'], 'gv': ['away_score'],
    'estadio': ['stadium.name', 'stadium_name', 'stadium'],
}

def cargar_matches():
    rutas = sorted(glob.glob(os.path.join(DC, 'matches_73_*.json*')) +
                   glob.glob(os.path.join(DP, 'matches_73_*.parquet')))
    piezas = []
    for r in rutas:
        try:
            df = _leer_matches(r)
        except Exception as e:
            print(f'  aviso: no se pudo leer {os.path.basename(r)}: {e}')
            continue
        cols = {k: _col(df, v) for k, v in CANDIDATAS.items()}
        faltan = [k for k in ['match_id', 'local', 'visita', 'gl', 'gv'] if cols[k] is None]
        if faltan:
            print(f'  aviso: {os.path.basename(r)} sin columnas {faltan}; se omite')
            continue
        out = pd.DataFrame({k: (df[c] if c else np.nan) for k, c in cols.items()})
        if out['estadio'].map(lambda v: isinstance(v, dict)).any():
            out['estadio'] = out['estadio'].map(lambda v: v.get('name') if isinstance(v, dict) else v)
        out['_fecha_archivo'] = df['_fecha_archivo']; out['_archivo'] = df['_archivo']
        piezas.append(out)
    M = pd.concat(piezas, ignore_index=True)
    M = (M.sort_values('_fecha_archivo', ascending=False)
           .drop_duplicates('match_id', keep='first'))
    return M, rutas

# ---------- Copia literal de E11b: correr_elo con arranque como argumento (devuelve también post) ----------
def correr_elo(P, K, H, R_ver, R_inv, ini):
    rating = dict(ini); temp_act = torneo_act = None
    pre_l = np.empty(len(P)); pre_v = np.empty(len(P)); post_l = np.empty(len(P)); post_v = np.empty(len(P))
    for i, r in enumerate(P.itertuples(index=False)):
        if temp_act is not None and r.temporada != temp_act:
            for t in rating: rating[t] = INICIO + (1 - R_ver) * (rating[t] - INICIO)
        elif torneo_act is not None and r.torneo != torneo_act:
            for t in rating: rating[t] = INICIO + (1 - R_inv) * (rating[t] - INICIO)
        temp_act, torneo_act = r.temporada, r.torneo
        rl = rating.get(r.local_id, INICIO); rv = rating.get(r.visita_id, INICIO)
        E = 1.0 / (1.0 + 10 ** (-(rl + H - rv) / 400))
        dif = r.gl - r.gv
        S = 1.0 if dif > 0 else (0.5 if dif == 0 else 0.0)
        n = abs(dif)
        G = 1.0 if n <= 1 else (1.5 if n == 2 else (11 + n) / 8)
        delta = K * G * (S - E)
        rating[r.local_id] = rl + delta; rating[r.visita_id] = rv - delta
        pre_l[i], pre_v[i], post_l[i], post_v[i] = rl, rv, rl + delta, rv - delta
    return pre_l, pre_v, post_l, post_v

# ---------- (A) Tabla de partidos: copia literal de R3a con la tabla de técnicos de fw6 ----------
ptp = pd.read_parquet(os.path.join(FW6, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp.columns:
    ptp = ptp[ptp['jugado'].astype(bool)]
ptp['fecha'] = pd.to_datetime(ptp['fecha'])
uni = (ptp.groupby('match_id').agg(fecha=('fecha', 'first'), temporada=('temporada', 'first'),
                                   n_filas=('team', 'size')).reset_index())
loc_ptp = ptp[ptp['lado'] == 'home'][['match_id', 'team']].rename(columns={'team': 'local_ptp'})
M, rutas = cargar_matches()
P = uni.merge(M, on='match_id', how='left').merge(loc_ptp, on='match_id', how='left')
P['hora'] = P['hora'].fillna('00:00:00.000')
P = P.sort_values(['fecha', 'hora', 'match_id']).reset_index(drop=True)
P['torneo'] = np.where(P['fecha'].dt.month >= 7, 'Apertura', 'Clausura')
orden = P.groupby('temporada')['fecha'].min().sort_values().index.tolist()
P['sidx'] = P['temporada'].map({s: i for i, s in enumerate(orden)})
P['t'] = (P['fecha'] - CALENTAMIENTO).dt.days / 365.25
P['y'] = np.where(P.gl > P.gv, 2, np.where(P.gl == P.gv, 1, 0))
for c in ['local_id', 'visita_id']:
    if P[c].isna().all():
        P[c] = P[c.replace('_id', '')]
P['estadio'] = P['estadio'].fillna('(sin estadio)')
print(f'--- (A) Fotos leídas: {len(rutas)} · partidos {len(P)} · de {VIVO}: {int((P.temporada == VIVO).sum())} · '
      f'sin marcador: {int(P[["gl", "gv"]].isna().any(axis=1).sum())} · filas de técnicos ≠ 2: {int((P.n_filas != 2).sum())} · '
      f'local distinto entre partidos y técnicos: {int((P["local"] != P["local_ptp"]).sum())}')
assert not P[['gl', 'gv']].isna().any().any(), 'Hay partidos sin marcador: avisar a Claude'

# ---------- (B) Reproducción contra fw1 ----------
F1 = pd.read_parquet(os.path.join(FW1, 'elo_partido_e2.parquet'))
viejo = P['match_id'].isin(set(F1['match_id']))
orden_ok = P.loc[viejo, 'match_id'].tolist() == F1['match_id'].tolist()
lado = pd.concat([F1[['sidx', 'local_id', 'elo_local_pre']].set_axis(['sidx', 'id', 'pre'], axis=1),
                  F1[['sidx', 'visita_id', 'elo_visita_pre']].set_axis(['sidx', 'id', 'pre'], axis=1)])
lado = lado.reset_index().sort_values(['index'])
prim = lado.groupby('id').first()
INI = prim.loc[prim['sidx'] == 0, 'pre'].to_dict()                      # arranque por tabla (E-14), como en E11b
pl, pv, ql, qv = correr_elo(P, **E2, ini=INI)
P['elo_local_pre'], P['elo_visita_pre'], P['elo_local_post'], P['elo_visita_post'] = pl, pv, ql, qv
P['H'] = float(E2['H'])
P['d'] = P['elo_local_pre'] + P['H'] - P['elo_visita_pre']
cmp = P[viejo].merge(F1, on='match_id', suffixes=('', '_f1'))
dif = max((cmp[c] - cmp[c + '_f1']).abs().max() for c in ['elo_local_pre', 'elo_visita_pre', 'elo_local_post', 'elo_visita_post'])
print(f'--- (B) Elo · partidos que ya estaban: {int(viejo.sum())} de {len(F1)} · mismo orden de partidos: {orden_ok} · '
      f'diferencia máxima con fw1: {dif:.2e} (debe ser < 1e-6) · arranque por tabla: {len(INI)} equipos')

# Elo de xG (R3e, copia literal de la lectura de tiros y del cálculo)
T1 = pd.read_parquet(os.path.join(FW1, 'tiros_xg.parquet'))
DIR_EV = os.path.join(DP, 'liga', 'eventos')
COLS_T = ['type.name', 'team.name', 'period', 'possession', 'shot.statsbomb_xg', 'shot.type.name']
piezas = []
for mid in sorted(set(P['match_id']) - set(T1['match_id'])):
    ruta = os.path.join(DIR_EV, f'{mid}.parquet')
    disp = set(pq.read_schema(ruta).names)
    d = pd.read_parquet(ruta, columns=[c for c in COLS_T if c in disp])
    for c in COLS_T:
        if c not in d.columns: d[c] = np.nan
    d = d[(d['type.name'] == 'Shot') & d['period'].isin([1, 2, 3, 4])]
    piezas.append(pd.DataFrame({'match_id': mid, 'team': d['team.name'].values, 'possession': d['possession'].values,
                                'xg': d['shot.statsbomb_xg'].astype(float).values,
                                'penal': (d['shot.type.name'] == 'Penalty').values}))
T = pd.concat([T1] + piezas, ignore_index=True)
Tp = (T.dropna(subset=['xg']).assign(q=lambda d: np.log1p(-d.xg.clip(0, 0.999)))
        .groupby(['match_id', 'team', 'possession'])['q'].sum().reset_index())
Tp['p_gol'] = 1 - np.exp(Tp['q'])
def _dist_goles(ps):
    d = np.array([1.0])
    for p in ps: d = np.convolve(d, [1 - p, p])
    return d
grupos = {k: g['p_gol'].values for k, g in Tp.groupby(['match_id', 'team'])}
xs = []
for r in P.itertuples(index=False):
    dl = _dist_goles(grupos.get((r.match_id, r.local), []))
    dv = _dist_goles(grupos.get((r.match_id, r.visita), []))
    M_ = np.outer(dl, dv)
    p_l = np.tril(M_, -1).sum(); p_e = np.trace(M_)
    xs.append((p_l, p_e, 1 - p_l - p_e,
               sum(grupos.get((r.match_id, r.local), [])), sum(grupos.get((r.match_id, r.visita), []))))
X3 = pd.DataFrame(xs, columns=['xp_local_gana', 'xp_empate', 'xp_visita_gana', 'xg_local', 'xg_visita'])
for c in X3: P[c] = X3[c].values
P['S_xg'] = P['xp_local_gana'] + 0.5 * P['xp_empate']
P.loc[~P['match_id'].isin(T['match_id'].unique()), 'S_xg'] = np.nan
def correr_elo_s(P, S, K, H, R_ver, R_inv, ini):
    rating = dict(ini); temp_act = torneo_act = None
    pre_l = np.empty(len(P)); pre_v = np.empty(len(P)); post_l = np.empty(len(P)); post_v = np.empty(len(P))
    for i, r in enumerate(P.itertuples(index=False)):
        if temp_act is not None and r.temporada != temp_act:
            for t in rating: rating[t] = INICIO + (1 - R_ver) * (rating[t] - INICIO)
        elif torneo_act is not None and r.torneo != torneo_act:
            for t in rating: rating[t] = INICIO + (1 - R_inv) * (rating[t] - INICIO)
        temp_act, torneo_act = r.temporada, r.torneo
        rl = rating.get(r.local_id, INICIO); rv = rating.get(r.visita_id, INICIO)
        E = 1.0 / (1.0 + 10 ** (-(rl + H - rv) / 400))
        delta = 0.0 if np.isnan(S[i]) else K * (S[i] - E)
        rating[r.local_id] = rl + delta; rating[r.visita_id] = rv - delta
        pre_l[i], pre_v[i], post_l[i], post_v[i] = rl, rv, rl + delta, rv - delta
    return pre_l, pre_v, post_l, post_v
xl, xv, xlp, xvp = correr_elo_s(P, P['S_xg'].values, **E2, ini=INI)
P['eloxg_local_pre'], P['eloxg_visita_pre'], P['eloxg_local_post'], P['eloxg_visita_post'] = xl, xv, xlp, xvp
X1 = pd.read_parquet(os.path.join(FW1, 'elo_xg_partido.parquet'))
cx = P.merge(X1, on='match_id', suffixes=('', '_f1'))
dx = max((cx[c] - cx[c + '_f1']).abs().max() for c in ['eloxg_local_pre', 'eloxg_visita_pre', 'S_xg'])
print(f'    Elo de xG · partidos comparados {len(cx)} · diferencia máxima con fw1 (pre y S_xg): {dx:.2e} (debe ser < 1e-6) · '
      f'tiros leídos de partidos nuevos: {sum(len(p) for p in piezas)} en {len(piezas)} partidos')
OK = orden_ok and dif < 1e-6 and dx < 1e-6 and len(cx) == len(X1)
nuevos = P[~viejo]
am = nuevos[nuevos['local'].str.startswith('Am') | nuevos['visita'].str.startswith('Am')]
print(f'    Partidos nuevos con Elo: {len(nuevos)} ({nuevos.fecha.min().date()} a {nuevos.fecha.max().date()}) · de América: {len(am)} · '
      f'nulos en Elo o Elo de xG: {int(nuevos[["elo_local_pre", "elo_visita_pre", "eloxg_local_pre", "eloxg_visita_pre"]].isna().sum().sum())}')

# ---------- (C) Guardar en fw6 (mismas columnas que R3a y R3e) ----------
assert OK, 'El Elo recalculado NO reproduce fw1: no se escribe nada. Manda esta salida a Claude.'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado'
cols_out = ['match_id', 'fecha', 'temporada', 'sidx', 'torneo', 'local', 'visita', 'local_id', 'visita_id',
            'estadio', 'gl', 'gv', 'elo_local_pre', 'elo_visita_pre', 'elo_local_post', 'elo_visita_post', 'H', 'd']
P[cols_out].to_parquet(os.path.join(FW6, 'elo_partido_e2.parquet'), index=False)
largo = pd.concat([
    P.assign(team=P.local, lado='home', elo_pre=P.elo_local_pre, elo_post=P.elo_local_post, elo_rival_pre=P.elo_visita_pre),
    P.assign(team=P.visita, lado='away', elo_pre=P.elo_visita_pre, elo_post=P.elo_visita_post, elo_rival_pre=P.elo_local_pre)
])[['match_id', 'fecha', 'temporada', 'sidx', 'team', 'lado', 'elo_pre', 'elo_post', 'elo_rival_pre']]
largo.to_parquet(os.path.join(FW6, 'elo_equipo_partido_e2.parquet'), index=False)
cols_x = ['match_id', 'fecha', 'temporada', 'sidx', 'local', 'visita', 'xg_local', 'xg_visita', 'xp_local_gana',
          'xp_empate', 'xp_visita_gana', 'S_xg', 'eloxg_local_pre', 'eloxg_visita_pre', 'eloxg_local_post', 'eloxg_visita_post']
P[cols_x].to_parquet(os.path.join(FW6, 'elo_xg_partido.parquet'), index=False)
T.to_parquet(os.path.join(FW6, 'tiros_xg.parquet'), index=False)
print(f'\nVALIDACIÓN OK · guardado en fw6: elo_partido_e2 ({len(P)}), elo_equipo_partido_e2 ({len(largo)}), elo_xg_partido ({len(P)}), '
      f'tiros_xg ({len(T)}) · {time.time() - t0:.0f} s. fw1 no se tocó.')

# Validación Almada (S.13.5)

In [ ]:
# [X6-0 v1.1] Carpeta espejo para reajustar todo el pipeline con 2026/27 agregada (preregistro de Almada, §4 paso 4; patrón X5 / D-11)
# =====================================================================
# Sesión 13 · 2026-10-20 · Preregistro, Aclaración 1 (2026-10-20 11:35).
# v1.1 (14:30): (1) YA NO BORRA la carpeta espejo: se puede volver a correr en cualquier momento (p. ej. tras un reinicio de Colab) y
#   solo rehace lo que falta. (2) Recupera de Drive los resultados respaldados con X6-g (datos_parquet/fw6_x6/), así un reinicio no
#   cuesta nada. (3) Incluye los reintentos ante errores de lectura de Drive (antes en X6-r). (4) Dice en qué paso vas.
# Idea (igual que X5): NO se modifica ninguna celda validada. La carpeta /content/isac_fw6/ tiene ENLACES (solo lectura) a Drive:
#   · tabla de técnicos, R1 de 2026/27, Elo y Elo de xG → los de datos_parquet/fw6/ (sesión 13: E-3, E-4, E-5)
#   · parametros_fw1.json → el de fw5 con 2026/27 agregada a las temporadas de ajuste
#   · datos_parquet/fw5 local = MISMA carpeta que datos_parquet/fw1 local (enlace)
#   · ninguna caché de resultados de fw1/fw5 se enlaza; solo cachés de columnas crudas por temporada (r6*_extra_*, r8_extra_*) hasta 2025/26
#   · NINGÚN archivo con "2026-2027" en el nombre se toma de fw1 o fw5 (allí solo hay 88 partidos): solo de fw6.
# Fija BASE, RAIZ_PRUEBA y RAIZ_X6 = carpeta espejo. X6-9 los borra al final.
# =====================================================================
import os, json, glob, shutil, time
import pandas as pd
import pyarrow.parquet as pq

DRIVE = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado; corre la celda 1'
LOCAL = '/content/isac_fw6/'
VIVO, T_ = '2026/2027', '2026-2027'
D_DP = os.path.join(DRIVE, 'datos_parquet')
D_FW1, D_FW5, D_FW6 = (os.path.join(D_DP, f) for f in ('fw1', 'fw5', 'fw6'))
CK = os.path.join(D_DP, 'fw6_x6')                                   # respaldo de X6-g
L_DP = os.path.join(LOCAL, 'datos_parquet'); L_FW = os.path.join(L_DP, 'fw1')
L_SAL, L_SAL5 = os.path.join(LOCAL, 'salidas'), os.path.join(LOCAL, 'salidas', 'fw5')
nueva = not os.path.isdir(L_FW)
os.makedirs(L_FW, exist_ok=True); os.makedirs(L_SAL5, exist_ok=True)
if not os.path.lexists(os.path.join(L_DP, 'fw5')): os.symlink(L_FW, os.path.join(L_DP, 'fw5'))   # fw5 local = fw1 local

# 0. Reintentos de lectura ante fallos momentáneos de Drive (OSError, Errno 5). No cambian ningún cálculo.
def _con_reintentos(f, nombre):
    if getattr(f, '_reintentos', False): return f
    def g(*a, **k):
        for i in range(6):
            try:
                return f(*a, **k)
            except OSError as e:
                if i == 5: raise
                print(f'  [reintento {i + 1}/5 de {nombre}: {type(e).__name__} {str(e)[:60]}]'); time.sleep(5 * (i + 1))
    g._reintentos = True
    return g
pd.read_parquet = _con_reintentos(pd.read_parquet, 'read_parquet')
pq.read_schema = _con_reintentos(pq.read_schema, 'read_schema')
pq.read_table = _con_reintentos(pq.read_table, 'read_table')

def enlazar(src, dst):
    if os.path.exists(src) and not os.path.lexists(dst):
        os.symlink(src, dst); return True
    return os.path.islink(dst) and os.path.realpath(dst) == os.path.realpath(src)

# 1. Raíz y datos_parquet: todo menos lo que se reemplaza
TABLAS_FW6 = ['partido_tecnico_periodo.parquet', 'etapas_tecnico.parquet', 'equipo_partido_tecnico.parquet']
for e in os.listdir(DRIVE):
    if e not in {'datos_parquet', 'salidas'}: enlazar(os.path.join(DRIVE, e), os.path.join(LOCAL, e))
for e in os.listdir(D_DP):
    if e not in {'fw1', 'fw5', 'fw6', 'fw6_x6'} | set(TABLAS_FW6): enlazar(os.path.join(D_DP, e), os.path.join(L_DP, e))
for f in TABLAS_FW6: assert enlazar(os.path.join(D_FW6, f), os.path.join(L_DP, f)), f'Falta fw6/{f}: corre E-3'

# 2. fw1 local: entradas
enl = {}
for f in ['elo_equipo_partido_e2.parquet', 'elo_partido_e2.parquet', 'elo_xg_partido.parquet', 'tiros_xg.parquet',
          f'eventos_{T_}.parquet', f'posesiones_{T_}.parquet']:
    enl[f] = 'fw6' if enlazar(os.path.join(D_FW6, f), os.path.join(L_FW, f)) else 'FALTA'
PATRONES = ['eventos_*.parquet', 'posesiones_*.parquet', 'componentes_*.parquet', 'presion_*.parquet',
            'r6_extra_*.parquet', 'r6b_extra_*.parquet', 'r6c_extra_*.parquet', 'r8_extra_*.parquet']
for origen, carpeta in [('fw1', D_FW1), ('fw5', D_FW5)]:
    for p in PATRONES:
        for r in sorted(glob.glob(os.path.join(carpeta, p))):
            f = os.path.basename(r)
            if T_ in f or os.path.islink(r) or f in enl: continue
            if enlazar(r, os.path.join(L_FW, f)): enl[f] = origen
# 3. Parámetros: los de fw5 con 2026/27 agregada al ajuste
with open(os.path.join(D_FW5, 'parametros_fw1.json')) as f:
    PRM_ = json.load(f)
PRM_['temporadas']['entrenamiento'] = sorted(set(PRM_['temporadas']['entrenamiento']) | {VIVO})
PRM_['_fw6'] = 'copia de fw5 con 2026/27 agregada (sesión 13, X6; preregistro de Almada §4 paso 4)'
with open(os.path.join(L_FW, 'parametros_fw1.json'), 'w') as f:
    json.dump(PRM_, f, indent=1)
TEMPS = PRM_['temporadas']['entrenamiento']
# 4. R2a y R2b (componentes_, presion_) solo se usan en chequeos de coherencia: vacíos donde no existan
for t in TEMPS:
    for tipo in ['componentes', 'presion']:
        p = os.path.join(L_FW, f"{tipo}_{t.replace('/', '-')}.parquet")
        if not os.path.lexists(p):
            pd.DataFrame({'match_id': pd.Series(dtype='int64'), 'equipo': pd.Series(dtype='object')}).to_parquet(p, index=False)

# 5. Recuperar lo respaldado con X6-g (solo archivos que no estén ya en la carpeta local)
rec = 0
for sub, dst in [('fw1', L_FW), ('salidas', L_SAL), ('salidas_fw5', L_SAL5)]:
    src = os.path.join(CK, sub)
    if not os.path.isdir(src): continue
    for f in os.listdir(src):
        d = os.path.join(dst, f)
        if not os.path.lexists(d): shutil.copy2(os.path.join(src, f), d); rec += 1

# 6. Chequeos y en qué paso vas
ptp_ = pd.read_parquet(os.path.join(L_DP, 'partido_tecnico_periodo.parquet'))
ev26 = pd.read_parquet(os.path.join(L_FW, f'eventos_{T_}.parquet'), columns=['match_id'])['match_id'].nunique()
elo_ = pd.read_parquet(os.path.join(L_FW, 'elo_equipo_partido_e2.parquet'), columns=['match_id'])['match_id']
m26 = set(ptp_.loc[ptp_.temporada == VIVO, 'match_id'])
falta = [f for t in TEMPS for f in [f"eventos_{t.replace('/', '-')}.parquet", f"posesiones_{t.replace('/', '-')}.parquet"]
         if not os.path.exists(os.path.join(L_FW, f))] + [k for k, v in enl.items() if v == 'FALTA']
# las cachés de resultados solo pueden venir del respaldo de X6 (nunca enlaces a fw1/fw5)
mal = [f for f in os.listdir(L_FW) if os.path.islink(os.path.join(L_FW, f)) and f.split('_')[0] in
       {'r4', 'r5', 'r6', 'r6b', 'r6c', 'r7', 'r7u', 'r7b', 'r7c', 'r8', 'r8p', 'r8z'} and 'extra' not in f]
print(f'--- (X6-0 v1.1) Carpeta espejo {"CREADA" if nueva else "YA EXISTÍA (no se borró nada)"} en {LOCAL} · archivos recuperados del respaldo: {rec}')
print(f'Temporadas de ajuste: {TEMPS}')
print(f'Tabla de técnicos (fw6): partidos de {VIVO} {len(m26)} · R1 {VIVO}: {ev26} · con Elo: {elo_.isin(m26).sum() // 2} (los tres deben ser 106)')
print('Faltantes:', falta or 'ninguno', '· resultados enlazados a fw1/fw5 (debe ser ninguno):', mal or 'ninguno')
assert not falta and not mal and len(m26) == ev26 == 106, 'Algo no cuadra: NO seguir; manda esta salida a Claude'
hay = lambda *fs: all(os.path.exists(os.path.join(L_FW, f)) or os.path.exists(os.path.join(L_SAL, f)) or os.path.exists(os.path.join(L_SAL5, f)) for f in fs)
PASOS = [('2 · R4a', ['r4_tramos_v21.parquet', 'r4_partidos.parquet', 'r4_minutos_v21.parquet']),
         ('2 · R4b', ['r4_residuos.parquet', 'r4_perfiles_etapa.parquet', 'r4_coeficientes.csv']),
         ('3 · R5', ['r5_perfiles_contraidos.parquet', 'r5_confiabilidad.csv']),
         ('4 · R6a', ['r6_tramos.parquet']), ('4 · R6b', ['r6b_tramos.parquet']), ('4 · R6c', ['r6c_tramos.parquet'])] + \
        [(f'5 · R6r {p}', [f'{p}_perfiles_contraidos.parquet', f'{p}_confiabilidad.csv']) for p in ['r6', 'r6b', 'r6c']] + \
        [('6 · R7a', ['r7_equipo_partido.parquet', 'r7_jugador_partido.parquet', 'r7_subs.parquet']),
         ('7 · R7u-0', ['r7u_tramos.parquet']), ('7 · R6r r7u', ['r7u_perfiles_contraidos.parquet', 'r7u_confiabilidad.csv']),
         ('8 · R7b-0', ['r7b_tramos.parquet']), ('8 · R6r r7b', ['r7b_perfiles_contraidos.parquet', 'r7b_confiabilidad.csv']),
         ('9 · R7c-0', ['r7c_tramos.parquet']), ('9 · R6r r7c', ['r7c_perfiles_contraidos.parquet', 'r7c_confiabilidad.csv']),
         ('10 · X6-R8a-1', ['r8_bp.parquet', 'r8_penales.parquet']), ('10 · X6-R8p-1', ['r8p_corners.parquet']),
         ('10 · X6-R8z-1', ['r8z_marcaje.parquet']), ('10 · X6-R8u-0', ['r8_tramos.parquet']),
         ('10 · R6r r8', ['r8_perfiles_contraidos.parquet', 'r8_confiabilidad.csv'])]
hechos = [n for n, fs in PASOS if hay(*fs)]
sig = next((n for n, fs in PASOS if not hay(*fs)), '11 · X6-9')
print('Pasos con sus resultados presentes:', hechos or 'ninguno')
print('→ SIGUIENTE PASO:', sig, '(los pasos 2, 4, 6 y las celdas de R8 cargan su caché si ya existe y terminan rápido)')
BASE = LOCAL; RAIZ_PRUEBA = LOCAL; RAIZ_X6 = LOCAL
print('BASE = RAIZ_PRUEBA = RAIZ_X6 =', LOCAL, '· reintentos de lectura activos')

# =====================================================================
# HOJA DE RUTA X6 (misma sesión de Colab; no correr R0 ni ninguna celda *-r0 / *-r9 / X5)
#  1. celda 1 (Drive) → X6-0   (si Colab se reinicia: celda 1 → X6-0 y seguir desde el "SIGUIENTE PASO" que imprime)
#  2. R4a y R4b → X6-g          3. R5 → X6-g          4. R6a, R6b, R6c → X6-g
#  5. Para cada PREFIJO en r6, r6b, r6c: X6-p → R6r-1 → R6r-2 → X6-g
#  6. R7a → X6-g
#  7. R7u-0 → X6-p 'r7u' → R6r-1 → R6r-2 → X6-g
#  8. R7b-0 → X6-p 'r7b' → R6r-1 → R6r-2 → X6-g
#  9. R7c-0 → X6-p 'r7c' → R6r-1 → R6r-2 → X6-g          (no correr R7c-9)
# 10. R8a-0 → X6-R8a-1 · R8p-0 → X6-R8p-1 · R8z-0 → X6-R8z-1 · X6-R8u-0 → X6-p 'r8' → R6r-1 → R6r-2 → X6-g
# 11. X6-9
# X6-p:   assert globals().get('BASE') == '/content/isac_fw6/', 'BASE no apunta al espejo: corre X6-0'
#         PREFIJO = 'r6'
# =====================================================================

In [ ]:
#X6-p
assert globals().get('BASE') == '/content/isac_fw6/', 'BASE no apunta al espejo: corre X6-0'
PREFIJO = 'r8'

In [ ]:
# [X6-r v1.0] Reintentos de lectura ante errores de Drive (OSError / Errno 5 "Input/output error")
# =====================================================================
# Sesión 13 · 2026-10-20 · Correr UNA vez en la sesión de X6 (después de X6-0, antes de volver a correr R7a).
# Qué hace: si una lectura de parquet desde Drive falla por un error de entrada/salida (fallo momentáneo de la conexión de Drive,
# no de los datos), la repite hasta 6 veces con espera creciente. NO cambia ningún cálculo: el archivo leído es el mismo.
# Aplica a pd.read_parquet y a pq.read_schema en TODAS las celdas siguientes de esta sesión.
# =====================================================================
import time
import pandas as pd
import pyarrow.parquet as pq

def _con_reintentos(f, nombre):
    if getattr(f, '_reintentos', False): return f
    def g(*a, **k):
        for i in range(6):
            try:
                return f(*a, **k)
            except OSError as e:
                if i == 5: raise
                print(f'  [reintento {i + 1}/5 de {nombre}: {type(e).__name__} {str(e)[:60]}]'); time.sleep(5 * (i + 1))
    g._reintentos = True
    return g
pd.read_parquet = _con_reintentos(pd.read_parquet, 'read_parquet')
pq.read_schema = _con_reintentos(pq.read_schema, 'read_schema')
pq.read_table = _con_reintentos(pq.read_table, 'read_table')
assert globals().get('BASE') == '/content/isac_fw6/', 'BASE no apunta al espejo: corre X6-0'
print('Reintentos activos para read_parquet, read_schema y read_table · BASE =', BASE, '→ vuelve a correr R7a.')

In [ ]:
# [X6-g v1.0] Respaldo en Drive de lo que X6 lleva hecho (para que un reinicio de Colab no cueste nada)
# =====================================================================
# Sesión 13 · 2026-10-20 · Correr después de cada bloque de la hoja de ruta X6 (ver X6-0). Tarda segundos a ~2 min.
# Copia a Drive, en datos_parquet/fw6_x6/ (carpeta de trabajo, NO es fw6), solo los archivos que X6 generó (no los enlaces) y solo
# los nuevos o cambiados. X6-0 v1.1 los recupera si la carpeta local se pierde. fw1, fw5 y fw6 no se tocan.
# =====================================================================
import os, shutil
DRIVE, LOCAL = '/content/drive/MyDrive/isac2026/', '/content/isac_fw6/'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado'
assert os.path.isdir(os.path.join(LOCAL, 'datos_parquet', 'fw1')), 'No hay carpeta espejo: corre X6-0'
CK = os.path.join(DRIVE, 'datos_parquet', 'fw6_x6')
nuevos = []
for sub, src in [('fw1', os.path.join(LOCAL, 'datos_parquet', 'fw1')), ('salidas', os.path.join(LOCAL, 'salidas')),
                 ('salidas_fw5', os.path.join(LOCAL, 'salidas', 'fw5'))]:
    dst = os.path.join(CK, sub); os.makedirs(dst, exist_ok=True)
    for f in os.listdir(src):
        s, d = os.path.join(src, f), os.path.join(dst, f)
        if not os.path.isfile(s) or os.path.islink(s): continue
        if os.path.exists(d) and os.path.getsize(d) == os.path.getsize(s) and os.path.getmtime(d) >= os.path.getmtime(s): continue
        shutil.copy2(s, d); nuevos.append(f)
tot = sum(len(os.listdir(os.path.join(CK, s))) for s in ['fw1', 'salidas', 'salidas_fw5'])
print(f'--- (X6-g) Respaldados ahora: {len(nuevos)} · total en datos_parquet/fw6_x6: {tot}')
print('  nuevos:', sorted(nuevos)[:30], '…' if len(nuevos) > 30 else '')

In [ ]:
# [X6-R8a-1] = R8a-1 v1.0 (claude/codigo/sesion11/R8a_base.py) con UNA línea cambiada para X6 (preregistro, Aclaración 1 punto 5):
#   original:  RAIZ = '/content/drive/MyDrive/isac2026/'
#   X6:        RAIZ = RAIZ_X6   (carpeta espejo /content/isac_fw6/; se agrega un freno que lo verifica)
# Todo lo demás es copia literal hasta el bloque (1). Se omiten las impresiones descriptivas (2)-(6) del original: no escriben nada
# y mostrarían cifras de 2026/27. Antes: X6-0 y R8a-0 (sin cambios) en esta sesión.
import os, json, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RAIZ = RAIZ_X6                                                       # X6: única línea cambiada
assert RAIZ == '/content/isac_fw6/', 'Freno X6: RAIZ no es la carpeta espejo; corre X6-0'   # X6: freno agregado
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
assert 'bp_de_partido' in globals(), 'Corre primero R8a-0'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1 = os.path.join(DP, 'fw1'); FW5 = os.path.join(DP, 'fw5'); LIGA = os.path.join(DP, 'liga')
REHACER = False
CORTE_C1, CORTE_C3 = 25, 70                                          # cuartos de R4 (A2)
t_ini = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
TEMPS = sorted(Mm['temporada'].unique())
COLS_R1 = ['match_id', 'index', 'period', 'minute', 't', 'type.name', 'team.name', 'possession', 'possession_team.name', 'player.id',
           'loc_x', 'loc_y', 'pass_end_x', 'pass_end_y', 'pass.type.name', 'pass.length', 'ball_receipt.outcome.name', 'shot.type.name',
           'shot.outcome.name', 'shot.statsbomb_xg', 'set_piece_phase', 'fase', 'en_transicion', 'pass.technique.name', 'foul_committed.penalty']
OPC_CRUDOS = ['pass.technique.name', 'foul_committed.penalty', 'pass.length', 'ball_receipt.outcome.name', 'second']
RUTAS = {k: os.path.join(FW5, f'r8_{k}.parquet') for k in ['bp', 'penales', 'reparto']}

# HOPS: ¿viene en los eventos? (nombres reales en 20 partidos crudos)
nom = set()
for m in Mm['match_id'].iloc[::85][:20]: nom |= set(pq.read_schema(os.path.join(LIGA, 'eventos', f'{m}.parquet')).names)
print('--- (0) Columnas tipo HOPS o probabilidad de duelo en los crudos:', sorted(c for c in nom if any(s in c.lower() for s in ['hops', 'win_prob', 'rating'])) or 'ninguna')

def extra_crudos(temp, falt):
    ruta = os.path.join(FW5, f"r8_extra_{temp.replace('/', '-')}.parquet")
    if os.path.exists(ruta) and not REHACER:
        X = pd.read_parquet(ruta)
        if all(c in X for c in falt): return X
    piezas = []
    for mid in Mm.loc[Mm.temporada == temp, 'match_id']:
        r = os.path.join(LIGA, 'eventos', f'{mid}.parquet')
        if not os.path.exists(r): continue
        disp = set(pq.read_schema(r).names)
        d = pd.read_parquet(r, columns=['index'] + [c for c in falt if c in disp])
        for c in falt:
            if c not in d: d[c] = np.nan
        d['match_id'] = mid; piezas.append(d)
    X = pd.concat(piezas, ignore_index=True); X.to_parquet(ruta, index=False); return X

if all(os.path.exists(r) for r in RUTAS.values()) and not REHACER:
    BP, PE, RE = (pd.read_parquet(RUTAS[k]) for k in ['bp', 'penales', 'reparto'])
    print('Cargado de caché:', len(BP), 'balones parados')
else:
    bps, pes, res, chk = [], [], [], []
    for temp in TEMPS:
        t0 = time.time()
        ruta_e = os.path.join(FW1, f"eventos_{temp.replace('/', '-')}.parquet"); cols_r1 = set(pq.read_schema(ruta_e).names)
        E = pd.read_parquet(ruta_e, columns=[c for c in COLS_R1 if c in cols_r1])
        E = E[E['match_id'].isin(loc.index)]
        falt = [c for c in OPC_CRUDOS if c not in E.columns and (c != 'second' or 't' not in E.columns)]
        if falt:
            E = E.merge(extra_crudos(temp, falt)[['match_id', 'index'] + falt], on=['match_id', 'index'], how='left')
        if 't' not in E.columns: E['t'] = E['minute'] * 60 + E['second']
        for c in COLS_R1:
            if c not in E.columns: E[c] = np.nan
        E = E.sort_values(['match_id', 'index']).reset_index(drop=True)
        ok_t = ((E['t'] - 60 * E['minute']).between(-0.01, 60.01)).mean()
        # Marcador: copia literal de R6a (goles de los periodos 1-4: tiro convertido o autogol a favor; offset de liguilla)
        E['local'] = E['match_id'].map(loc['local']); E['ev_local'] = E['team.name'] == E['local']
        es_gol = E['period'].between(1, 4) & (((E['type.name'] == 'Shot') & (E['shot.outcome.name'] == 'Goal')) | (E['type.name'] == 'Own Goal For'))
        gh = (es_gol & E['ev_local']).astype(int); ga = (es_gol & ~E['ev_local'] & E['team.name'].notna()).astype(int)
        E['dif_home'] = (gh.groupby(E['match_id']).cumsum() - gh) - (ga.groupby(E['match_id']).cumsum() - ga)
        E['m_home'] = E['dif_home'] + E['match_id'].map(loc['offset']); E['m_home_b'] = E['dif_home'] + E['match_id'].map(loc['offset_b'])
        E['cuarto'] = np.where(E['period'] == 1, np.where(E['minute'] < CORTE_C1, 1, 2), np.where(E['minute'] < CORTE_C3, 3, 4))
        gg = pd.DataFrame({'gh': gh.groupby(E['match_id']).sum(), 'ga': ga.groupby(E['match_id']).sum()})
        cuadra = ((gg.gh == loc.loc[gg.index, 'gl']) & (gg.ga == loc.loc[gg.index, 'gv'])).mean()
        E = E[E['period'].between(1, 4)]
        # Reparto de tiros, xG y goles por tipo de jugada (periodos 1-4; sin tandas)
        sh = E[E['type.name'] == 'Shot'].copy()
        sh['tipo'] = np.select([sh['shot.type.name'] == 'Penalty', sh['fase'] == 'bp', sh['fase'] == 'transicion'], ['penal', 'balon_parado', 'transicion'], 'establecido')
        r = sh.groupby(['match_id', 'team.name', 'tipo']).agg(tiros=('tipo', 'size'), xg=('shot.statsbomb_xg', 'sum'),
                                                              goles=('shot.outcome.name', lambda s: int((s == 'Goal').sum()))).reset_index()
        og = E[E['type.name'] == 'Own Goal For'].groupby(['match_id', 'team.name']).size().rename('goles').reset_index().assign(tipo='autogol', tiros=0, xg=0.0)
        res.append(pd.concat([r, og], ignore_index=True).rename(columns={'team.name': 'team'}))
        n360 = 0
        for mid, M in E.groupby('match_id', sort=False):
            J = None; rj = os.path.join(LIGA, 'jugadores360', f'{mid}.parquet')
            ic = M.loc[(M['pass.type.name'] == 'Corner'), 'index'].values
            if os.path.exists(rj) and len(ic):
                nec = set(np.concatenate([ic + k for k in range(16)]))
                jj = pd.read_parquet(rj, columns=['event_index', 'teammate', 'actor', 'keeper', 'x', 'y'])
                jj = jj[jj.event_index.isin(nec)]; J = {k: g for k, g in jj.groupby('event_index')}; n360 += 1
            for f in bp_de_partido(M, J): f['match_id'] = mid; bps.append(f)
            for f in penales_de_partido(M): f['match_id'] = mid; pes.append(f)
        chk.append(dict(temporada=temp, partidos=E.match_id.nunique(), con_360=n360, t_coherente_pct=round(100 * ok_t, 2),
                        marcador_cuadra_pct=round(100 * cuadra, 2), segundos=round(time.time() - t0)))
        print(f'  {temp}: {chk[-1]}')
    BP, PE, RE = pd.DataFrame(bps), pd.DataFrame(pes), pd.concat(res, ignore_index=True)
    for d_, k in [(BP, 'bp'), (PE, 'penales'), (RE, 'reparto')]: d_.to_parquet(RUTAS[k], index=False)
    print('Chequeos por temporada (t_coherente y marcador_cuadra deben ser ~100):'); print(pd.DataFrame(chk).to_string(index=False))

BP['temporada'] = BP.match_id.map(loc['temporada']); RE['temporada'] = RE.match_id.map(loc['temporada'])
pc = lambda s: round(100 * float(np.mean(s)), 1) if len(s) else np.nan
co = BP[BP.kind == 'corner']
print(f'\n--- (1) Balones parados: {BP.kind.value_counts().to_dict()} · córners por partido {len(co) / BP.match_id.nunique():.1f}')
print('  Ruta del córner (%):', co.ruta.value_counts(normalize=True).mul(100).round(1).to_dict())
print(f'\nX6-R8a-1 terminó en {time.time() - t_ini:.0f} s. Guardado (espejo): r8_bp, r8_penales, r8_reparto (+ r8_extra_2026-2027).')

In [ ]:
# [X6-R8p-1] = R8p-1 v1.1 (claude/codigo/sesion11/R8p_playbooks.py) para X6, con los cambios de la Aclaración 1 (puntos 3 y 5):
#   (a) RAIZ = RAIZ_X6 (carpeta espejo) en lugar de '/content/drive/MyDrive/isac2026/' + freno.
#   (b) NO se vuelve a agrupar: cada córner se asigna al playbook más cercano de los CENTROIDES de fw5 (salidas/fw5/r8p_km_*.csv,
#       lectura de Drive). Es lo mismo que KMeans.predict con el modelo de la sesión 11.
#   (c) Se omiten la elección de k, los placebos y las tablas descriptivas (no escriben nada que use R8u).
# Copia literal del original: bloque 1 (primer contacto) y bloque 2 (huellas 360). Mismo archivo de salida: r8p_corners.
# Validación obligatoria: en los córners de 2021/22-2025/26, huellas idénticas a fw5/r8p_huellas y etiquetas idénticas a fw5/r8p_corners.
# Antes: X6-0, X6-R8a-1 y R8p-0 (sin cambios) en esta sesión.
import os, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
RAIZ = RAIZ_X6                                                       # X6 (a)
assert RAIZ == '/content/isac_fw6/', 'Freno X6: RAIZ no es la carpeta espejo; corre X6-0'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
assert 'huella' in globals() and 'ZONA_CELDA' in globals(), 'Corre primero R8p-0 (define huella)'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5, LIGA = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5'), os.path.join(DP, 'liga')
SAL5 = os.path.join(RAIZ, 'salidas', 'fw5'); os.makedirs(SAL5, exist_ok=True)
D_FW5, D_SAL5 = '/content/drive/MyDrive/isac2026/datos_parquet/fw5', '/content/drive/MyDrive/isac2026/salidas/fw5'   # X6 (b): solo lectura
REHACER = False
t_ini = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
BP = pd.read_parquet(os.path.join(FW5, 'r8_bp.parquet'))
C = BP[(BP.kind == 'corner') & (BP.ruta == 'directo') & BP['s_frame'].astype('boolean').fillna(False)].copy()
C['temporada'] = C.match_id.map(loc['temporada'])
C['rival'] = np.where(C.team == C.match_id.map(loc['local']), C.match_id.map(loc['visita']), C.match_id.map(loc['local']))
ruta_h = os.path.join(FW5, 'r8p_huellas.parquet')
if os.path.exists(ruta_h) and not REHACER:
    H = pd.read_parquet(ruta_h); print('Huellas de caché:', len(H))
else:
    # 1. Primer contacto de cada córner directo (misma definición que R8a: siguiente evento sin presiones ni recepciones fallidas, ≤ 5 s)
    piezas = []
    for temp, g in C.groupby('temporada'):
        E = pd.read_parquet(os.path.join(FW1, f"eventos_{temp.replace('/', '-')}.parquet"),
                            columns=['match_id', 'index', 'period', 't', 'type.name', 'team.name', 'ball_receipt.outcome.name'])
        E = E[E.match_id.isin(set(g.match_id))]
        F = E[(E['type.name'] != 'Pressure') & ~((E['type.name'] == 'Ball Receipt*') & E['ball_receipt.outcome.name'].notna())]
        F = F.rename(columns={'index': 'c_index', 'period': 'c_period', 't': 'c_t', 'type.name': 'c_tipo', 'team.name': 'c_team'})
        F['index'] = F['c_index']
        x = pd.merge_asof(g.sort_values('index'), F.sort_values('index')[['match_id', 'index', 'c_index', 'c_period', 'c_t', 'c_tipo', 'c_team']],
                          on='index', by='match_id', direction='forward', allow_exact_matches=False)
        val = (x.c_period == x.period) & (x.c_t <= x.t + 5)
        x.loc[~val, ['c_index', 'c_tipo', 'c_team']] = np.nan
        piezas.append(x)
    C = pd.concat(piezas, ignore_index=True)
    igual = (C.c_tipo.fillna('—') == C.contacto_tipo.fillna('—')).mean()
    print(f'--- (0) Coherencia con R8a: tipo de contacto idéntico en {100 * igual:.2f}% de {len(C):,} córners (debe ser 100)')
    # 2. Huellas desde las fotos 360
    filas = []
    for mid, g in C.groupby('match_id'):
        r = os.path.join(LIGA, 'jugadores360', f'{mid}.parquet')
        if not os.path.exists(r): continue
        nec = set(g['index']) | set(g['c_index'].dropna().astype(int))
        jj = pd.read_parquet(r, columns=['event_index', 'teammate', 'actor', 'keeper', 'x', 'y']); jj = jj[jj.event_index.isin(nec)]
        JJ = {k: v for k, v in jj.groupby('event_index')}
        for _, c in g.iterrows():
            if c['index'] not in JJ: continue
            A, D, peg = huella(JJ[c['index']], bool(c.lado_bajo))
            f = dict(match_id=mid, index=c['index'], pegados_portero=peg, **{f'sa{i}': A[i] for i in range(20)}, **{f'sd{i}': D[i] for i in range(20)})
            ci = c['c_index']
            if pd.notna(ci) and int(ci) in JJ:
                A2, _, _ = huella(JJ[int(ci)], bool(c.lado_bajo), girar=c['c_team'] != c['team'], sin_actor=False)
                f.update({f'ca{i}': A2[i] for i in range(20)})
            filas.append(f)
    H = pd.DataFrame(filas).merge(C[['match_id', 'index', 'team', 'rival', 'temporada', 'zona', 'tecnica', 'xg1', 'xg2', 'goles', 'caida_gana',
                                     'contacto_clase', 's_marc2', 's_marc_den']], on=['match_id', 'index'])
    H.to_parquet(ruta_h, index=False)
SA, SD, CA = [f'sa{i}' for i in range(20)], [f'sd{i}' for i in range(20)], [f'ca{i}' for i in range(20)]
print(f'--- (1) Córners directos con foto del saque: {len(H):,} · con foto del contacto {100 * H[CA[0]].notna().mean():.1f}%'
      f' · con portero visible {100 * H.pegados_portero.notna().mean():.1f}% ({time.time() - t_ini:.0f} s) · por temporada {H.temporada.value_counts().sort_index().to_dict()}')

# X6 (b) · Asignación a los centroides congelados de fw5 (playbook más cercano, distancia euclidiana = KMeans.predict)
def asigna(X, cen):
    return ((X[:, None, :] - cen[None, :, :]) ** 2).sum(-1).argmin(1)
cen_o = pd.read_csv(os.path.join(D_SAL5, 'r8p_km_ofensivo.csv'))[SA + CA].to_numpy(float)
cen_d = pd.read_csv(os.path.join(D_SAL5, 'r8p_km_defensivo.csv'))[SD].to_numpy(float)
Ho = H[H[CA[0]].notna()].reset_index(drop=True)
Ho['pb_of'] = asigna(Ho[SA + CA].to_numpy(float), cen_o)
H['pb_def'] = asigna(H[SD].to_numpy(float), cen_d)
H = H.merge(Ho[['match_id', 'index', 'pb_of']], on=['match_id', 'index'], how='left')
print(f'--- (2) Playbooks con centroides de fw5: ofensivos k = {len(cen_o)} · defensivos k = {len(cen_d)}')

# Validación contra fw5 (Drive, solo lectura)
k_ = ['match_id', 'index']
h5 = pd.read_parquet(os.path.join(D_FW5, 'r8p_huellas.parquet'))
c5 = pd.read_parquet(os.path.join(D_FW5, 'r8p_corners.parquet'), columns=k_ + ['pb_of', 'pb_def'])
num = ['pegados_portero'] + SA + SD + CA
j = h5[k_ + num].merge(H[k_ + num], on=k_, how='left', suffixes=('_5', '_6'), indicator=True)
dif_h = max(float(np.nanmax(np.abs(j[c + '_5'] - j[c + '_6']))) if j[c + '_5'].notna().any() else 0.0 for c in num)
nul_h = int(sum(((j[c + '_5'].isna()) != (j[c + '_6'].isna())).sum() for c in num))
jc = c5.merge(H[k_ + ['pb_of', 'pb_def']], on=k_, how='left', suffixes=('_5', '_6'))
ig_of = ((jc.pb_of_5 == jc.pb_of_6) | (jc.pb_of_5.isna() & jc.pb_of_6.isna())).mean()
ig_def = (jc.pb_def_5 == jc.pb_def_6).mean()
print(f'--- (3) Validación (córners de fw5: {len(h5):,}) · en X6: {int((j._merge == "both").sum()):,} · huellas: diferencia máxima {dif_h:.2e} · '
      f'nulos distintos {nul_h} · etiqueta ofensiva igual {100 * ig_of:.2f}% · defensiva igual {100 * ig_def:.2f}% (todo debe ser 0 / 100)')
nuevos = H[~H.set_index(k_).index.isin(h5.set_index(k_).index)]
print(f'  Córners nuevos (2026/27): {len(nuevos)} · con contacto (tienen playbook ofensivo) {int(nuevos.pb_of.notna().sum())}')
assert (j._merge == 'both').all() and dif_h < 1e-9 and nul_h == 0 and ig_of == 1 and ig_def == 1, \
    'X6-R8p no reproduce fw5: NO seguir; manda esta salida a Claude'
H.to_parquet(os.path.join(FW5, 'r8p_corners.parquet'), index=False)
print(f'\nX6-R8p-1 terminó en {time.time() - t_ini:.0f} s. VALIDACIÓN OK · guardado (espejo): r8p_huellas, r8p_corners.')


In [ ]:
# [X6-R8z-1] = R8z-1 v1.1 (claude/codigo/sesion11/R8z_marcaje.py) con UNA línea cambiada para X6 (preregistro, Aclaración 1 punto 5):
#   original:  RAIZ = '/content/drive/MyDrive/isac2026/'
#   X6:        RAIZ = RAIZ_X6   (carpeta espejo; se agrega un freno que lo verifica)
# Todo lo demás es copia literal, salvo que se omiten las impresiones descriptivas finales del bloque (2) y el bloque (3)
# (no escriben nada que use R8u). Antes: X6-0, X6-R8a-1, X6-R8p-1 y R8z-0 (sin cambios) en esta sesión.
import os, time
RAIZ = RAIZ_X6                                                       # X6: única línea cambiada
assert RAIZ == '/content/isac_fw6/', 'Freno X6: RAIZ no es la carpeta espejo; corre X6-0'   # X6: freno agregado
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
assert all(f in globals() for f in ['jugadores', 'marcaje', 'reparto', 'pendiente']), 'Corre primero R8z-0 (define las funciones)'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5, LIGA = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5'), os.path.join(DP, 'liga')
SAL5 = os.path.join(RAIZ, 'salidas', 'fw5'); os.makedirs(SAL5, exist_ok=True)
REHACER = False
t_ini = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
BP = pd.read_parquet(os.path.join(FW5, 'r8_bp.parquet'))
C = BP[(BP.kind == 'corner') & (BP.ruta == 'directo') & BP['s_frame'].astype('boolean').fillna(False)].copy()
C['temporada'] = C.match_id.map(loc['temporada'])
C['rival'] = np.where(C.team == C.match_id.map(loc['local']), C.match_id.map(loc['visita']), C.match_id.map(loc['local']))
ruta_z = os.path.join(FW5, 'r8z_marcaje.parquet')
if os.path.exists(ruta_z) and not REHACER:
    Z = pd.read_parquet(ruta_z); print('Marcaje de caché:', len(Z))
else:
    # Primer contacto: copia literal de R8p-1 (misma definición que R8a)
    piezas = []
    for temp, g in C.groupby('temporada'):
        E = pd.read_parquet(os.path.join(FW1, f"eventos_{temp.replace('/', '-')}.parquet"),
                            columns=['match_id', 'index', 'period', 't', 'type.name', 'team.name', 'ball_receipt.outcome.name'])
        E = E[E.match_id.isin(set(g.match_id))]
        F = E[(E['type.name'] != 'Pressure') & ~((E['type.name'] == 'Ball Receipt*') & E['ball_receipt.outcome.name'].notna())]
        F = F.rename(columns={'index': 'c_index', 'period': 'c_period', 't': 'c_t', 'type.name': 'c_tipo', 'team.name': 'c_team'})
        F['index'] = F['c_index']
        x = pd.merge_asof(g.sort_values('index'), F.sort_values('index')[['match_id', 'index', 'c_index', 'c_period', 'c_t', 'c_tipo', 'c_team']],
                          on='index', by='match_id', direction='forward', allow_exact_matches=False)
        val = (x.c_period == x.period) & (x.c_t <= x.t + 5)
        x.loc[~val, ['c_index', 'c_tipo', 'c_team']] = np.nan
        piezas.append(x)
    C = pd.concat(piezas, ignore_index=True)
    print(f'--- (0) Coherencia con R8a: tipo de contacto idéntico en {100 * (C.c_tipo.fillna("—") == C.contacto_tipo.fillna("—")).mean():.2f}% (debe ser 100)')
    filas = []
    for mid, g in C.groupby('match_id'):
        r = os.path.join(LIGA, 'jugadores360', f'{mid}.parquet')
        if not os.path.exists(r): continue
        nec = set(g['index']) | set(g['c_index'].dropna().astype(int))
        jj = pd.read_parquet(r, columns=['event_index', 'teammate', 'actor', 'keeper', 'x', 'y']); jj = jj[jj.event_index.isin(nec)]
        JJ = {k: v for k, v in jj.groupby('event_index')}
        for _, c in g.iterrows():
            if c['index'] not in JJ: continue
            ax, ay, dx, dy = jugadores(JJ[c['index']], bool(c.lado_bajo))
            f = dict(match_id=mid, index=c['index'], team=c.team, rival=c.rival, temporada=c.temporada, n_atac=len(ax), n_def=len(dx),
                     **marcaje(ax, ay, dx, dy), **{f'pa_{z}': v for z, v in zip(ZONAS, reparto(ax, ay))},
                     **{f'pd_{z}': v for z, v in zip(ZONAS, reparto(dx, dy))})
            ci = c['c_index']
            if pd.notna(ci) and int(ci) in JJ:
                bx, by, ex_, ey_ = jugadores(JJ[int(ci)], bool(c.lado_bajo), girar=c['c_team'] != c['team'], sin_actor=False)
                f['c_pct_marca_b'] = marcaje(bx, by, ex_, ey_)['pct_marca_b']
            filas.append(f)
    Z = pd.DataFrame(filas); Z.to_parquet(ruta_z, index=False)

print(f'--- (1) Córners: {len(Z):,} · atacantes en el área (mediana) {Z.n_atac.median():.0f} · defensores {Z.n_def.median():.0f} ({time.time() - t_ini:.0f} s)')
for k, (u1, u2) in UMBRALES.items():
    print(f'  Umbrales {u1}/{u2} yd: % de atacantes con marca · cuantiles 10/25/50/75/90 {Z[f"pct_marca_{k}"].quantile([.1, .25, .5, .75, .9]).round(0).tolist()}'
          f' · defensores zonales {Z[f"zonales_{k}"].quantile([.1, .25, .5, .75, .9]).tolist()}')
print('  Histograma del % de atacantes con marca (umbral principal, % de córners por tramo de 20 puntos):',
      pd.cut(Z.pct_marca_b, [-1, 20, 40, 60, 80, 101]).value_counts(normalize=True).sort_index().mul(100).round(1).to_dict())
zc = Z.dropna(subset=['c_pct_marca_b'])
print(f'  Contacto: % de atacantes con marca saque → contacto {zc.pct_marca_b.mean():.1f} → {zc.c_pct_marca_b.mean():.1f}'
      f' · correlación entre córners {zc[["pct_marca_b", "c_pct_marca_b"]].corr().iloc[0, 1]:.2f}')

# 2. Por equipo defensor y temporada: marca media, zonales y pendiente de respuesta al rival (con mitades para su confiabilidad)
rng = np.random.default_rng(SEMILLA)
PA_c, PD_c = [f'pa_{z}' for z in ZONAS], [f'pd_{z}' for z in ZONAS]
filas = []
for (eq, temp), g in Z.groupby(['rival', 'temporada']):
    if len(g) < 40: continue
    m = rng.random(len(g)) < 0.5
    filas.append(dict(equipo=eq, temporada=temp, corners=len(g), pct_marca=g.pct_marca_b.mean(), zonales=g.zonales_b.mean(),
                      pendiente=pendiente(g[PA_c].to_numpy(), g[PD_c].to_numpy()),
                      pend_m1=pendiente(g[m][PA_c].to_numpy(), g[m][PD_c].to_numpy()), pend_m2=pendiente(g[~m][PA_c].to_numpy(), g[~m][PD_c].to_numpy()),
                      marca_m1=g[m].pct_marca_b.mean(), marca_m2=g[~m].pct_marca_b.mean()))
T = pd.DataFrame(filas)
sb = lambda r: 2 * r / (1 + r)
print(f'\n--- (2) Equipos defensores (equipo-temporada con ≥ 40 córners): {len(T)}')
print('  % de atacantes con marca: cuantiles 10/50/90', T.pct_marca.quantile([.1, .5, .9]).round(1).tolist(),
      f'· confiabilidad entre mitades {sb(T[["marca_m1", "marca_m2"]].corr().iloc[0, 1]):.2f}')
print('  Pendiente de respuesta al rival: cuantiles 10/50/90', T.pendiente.quantile([.1, .5, .9]).round(2).tolist(),
      f'· confiabilidad entre mitades {sb(T[["pend_m1", "pend_m2"]].corr().iloc[0, 1]):.2f}')
T.to_csv(os.path.join(SAL5, 'r8z_equipos.csv'), index=False)
print(f'\nX6-R8z-1 terminó en {time.time() - t_ini:.0f} s. Guardado (espejo): r8z_marcaje · salidas/fw5/r8z_equipos.csv.')


In [ ]:
# [X6-R8u-0] = R8u-0 v1.0 (claude/codigo/sesion11/R8u_rasgos.py), parte "Tramos reales", con UNA línea cambiada para X6
# (preregistro, Aclaración 1 punto 5):
#   original:  RAIZ = '/content/drive/MyDrive/isac2026/'
#   X6:        RAIZ = RAIZ_X6   (carpeta espejo; se agrega un freno que lo verifica)
# Se incluyen sin cambios las funciones del original (e5, estado, ZON, pendiente_partes). Las pruebas T1-T3 del original no cambian
# nada y no se repiten. Todo lo demás es copia literal. Antes: X6-0, X6-R8a-1, X6-R8p-1, X6-R8z-1 en esta sesión.
import os, time
import numpy as np
import pandas as pd

def e5(m):                                  # copia de R4a
    m = np.asarray(m, dtype=float)
    return np.clip(np.sign(m) * np.ceil(np.abs(m) - 1e-9), -2, 2).astype(int)
def estado(df, defensa=False):
    """e5, e5b desde el equipo que cobra (defensa=False) o desde el que defiende (defensa=True). m_home viene de R8a (marcador local + offset)."""
    s = np.where(df['ev_local'].astype(bool), 1.0, -1.0) * (-1.0 if defensa else 1.0)
    return e5(s * df['m_home'].astype(float)), e5(s * df['m_home_b'].astype(float))
ZON = ['primer_palo', 'punto_penal', 'area_chica', 'segundo_palo']
def pendiente_partes(Z):
    """Numerador y denominador por córner, centrando las proporciones por equipo defensor y temporada."""
    PA, PD = Z[[f'pa_{z}' for z in ZON]], Z[[f'pd_{z}' for z in ZON]]
    ok = PA.notna().all(axis=1) & PD.notna().all(axis=1); Z = Z[ok].copy()
    g = Z.groupby(['rival', 'temporada'])
    A = Z[[f'pa_{z}' for z in ZON]] - g[[f'pa_{z}' for z in ZON]].transform('mean')
    D = Z[[f'pd_{z}' for z in ZON]] - g[[f'pd_{z}' for z in ZON]].transform('mean')
    Z['num'] = (A.values * D.values).sum(1); Z['den'] = (A.values ** 2).sum(1)
    return Z

# --- Tramos reales ---
RAIZ = RAIZ_X6                                                       # X6: única línea cambiada
assert RAIZ == '/content/isac_fw6/', 'Freno X6: RAIZ no es la carpeta espejo; corre X6-0'   # X6: freno agregado
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5')
t0 = time.time()
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
BP = pd.read_parquet(os.path.join(FW5, 'r8_bp.parquet'))
BP['rival'] = np.where(BP.team == BP.match_id.map(loc['local']), BP.match_id.map(loc['visita']), BP.match_id.map(loc['local']))
BP['temporada'] = BP.match_id.map(loc['temporada'])
B = BP[BP.period.isin([1, 2]) & BP.match_id.isin(loc.index)].copy()                     # D-9
B['e5'], B['e5b'] = estado(B); B['e5_d'], B['e5b_d'] = estado(B, defensa=True)
piezas = []
def agrega(d, num, nombre, den=1.0, defensa=False):
    d = d.assign(num=np.asarray(num, float), den=np.asarray(den, float) if np.ndim(den) else den)
    t = pd.DataFrame({'metrica': nombre, 'match_id': d.match_id.values, 'team': (d.rival if defensa else d.team).values,
                      'e5': (d.e5_d if defensa else d.e5).values, 'e5b': (d.e5b_d if defensa else d.e5b).values,
                      'cuarto': d.cuarto.astype(int).values, 'num': d.num.values, 'den': d.den.values})
    piezas.append(t.dropna(subset=['num', 'den']))

co = B[B.kind == 'corner']; cor = co[co.ruta.isin(['via_corto', 'reciclado'])]
env = co[co.ruta.isin(['directo', 'via_corto']) & co.zona.isin(ZON)]
dtec = co[(co.ruta == 'directo') & co.tecnica.notna()]
seg = co[co.caida_gana.notna() & (co.contacto_clase != 'control_atac')]
s360 = co[(co.ruta == 'directo') & co['s_frame'].astype('boolean').fillna(False)]
agrega(co, 100.0 * co.ruta.isin(['via_corto', 'reciclado']), 'corto_pct')
agrega(cor, 100.0 * (cor.ruta == 'via_corto'), 'corto_a_envio')
for z in ZON: agrega(env, 100.0 * (env.zona == z), f'envio_{z}')
agrega(dtec, 100.0 * (dtec.tecnica == 'Inswinging'), 'cerrado_pct')
agrega(co, 100.0 * (co.xg1 + co.xg2), 'xg_corner'); agrega(co, 100.0 * co.xg2, 'xg2_corner')
agrega(seg, 100.0 * seg.caida_gana.astype(bool), 'segunda_ganada')
agrega(s360, s360.s_atac_frontal, 'atac_frontal_360'); agrega(s360, s360.s_atac_area, 'atac_area_360')
sb = B[B.kind == 'saque_banda']; tl = B[B.kind == 'tiro_libre']
agrega(sb, 100.0 * (sb.ruta == 'directo'), 'saque_banda_largo')
agrega(tl, 100.0 * (tl.ruta == 'tiro_directo'), 'tl_directo'); agrega(tl, 100.0 * tl.ruta.isin(['via_corto', 'reciclado']), 'tl_corto')
# Defensa (córner del rival, asignado al que defiende)
agrega(co, 100.0 * (co.xg1 + co.xg2), 'xg_concedido_corner', defensa=True)
agrega(seg, 100.0 * (1 - seg.caida_gana.astype(bool)), 'segunda_recuperada', defensa=True)
agrega(s360, s360.s_def_area, 'def_area_360', defensa=True)
# Playbooks (R8p) y marcaje (R8z): se les pega el estado desde r8_bp
llave = ['match_id', 'index']
P = pd.read_parquet(os.path.join(FW5, 'r8p_corners.parquet'), columns=llave + ['pb_of', 'pegados_portero']).merge(B[B.kind == 'corner'], on=llave)
po = P[P.pb_of.notna()]
for k in range(int(po.pb_of.max()) + 1): agrega(po, 100.0 * (po.pb_of == k), f'pb_of_{k}')
pp = P[P.pegados_portero.notna()]; agrega(pp, pp.pegados_portero, 'pegados_portero')
Z = pd.read_parquet(os.path.join(FW5, 'r8z_marcaje.parquet')).drop(columns=['team', 'rival', 'temporada'], errors='ignore').merge(B[B.kind == 'corner'], on=llave)
zm = Z[Z.pct_marca_b.notna()]
agrega(zm, zm.pct_marca_b, 'pct_marca', defensa=True); agrega(zm, zm.zonales_b, 'zonales', defensa=True)
Zp = pendiente_partes(Z); agrega(Zp, Zp.num, 'pendiente_marca', den=Zp.den.values, defensa=True)
print(f'\n--- (0) Enlaces: playbooks {len(P):,} de {len(pd.read_parquet(os.path.join(FW5, "r8p_corners.parquet"), columns=["index"])):,} · marcaje {len(Z):,} ({time.time() - t0:.0f} s)')

# Peso del balón parado (por tiro, sin penales): marcador reconstruido con la copia literal de R6a sobre tiros y autogoles
pesos, ch_t, ch_p = [], [], []
OFF = {1: 0, 2: 45, 3: 90, 4: 105}
PE = pd.read_parquet(os.path.join(FW5, 'r8_penales.parquet'))
cor_t = BP[BP.kind == 'corner'][['match_id', 'index', 'period', 't', 'team']]
for temp in sorted(Mm.temporada.unique()):
    E = pd.read_parquet(os.path.join(FW1, f"eventos_{temp.replace('/', '-')}.parquet"),
                        columns=['match_id', 'index', 'period', 'minute', 't', 'type.name', 'team.name', 'shot.type.name', 'shot.statsbomb_xg',
                                 'shot.outcome.name', 'fase'])
    E = E[E.match_id.isin(loc.index)].sort_values(['match_id', 'index'])
    ok_t = ((E.t - 60 * (E.minute - E.period.map(OFF))).between(-0.01, 60.01) | E.period.eq(5)).mean()
    ch_t.append((temp, round(100 * ok_t, 2)))
    pm = PE[PE.match_id.isin(set(E.match_id))].merge(E[['match_id', 'index', 't']], on=['match_id', 'index'], how='left')
    ch_p.append(pm)
    E = E[E['type.name'].isin(['Shot', 'Own Goal For'])].copy()
    E['local'] = E['match_id'].map(loc['local']); E['ev_local'] = E['team.name'] == E['local']
    es_gol = E['period'].between(1, 4) & (((E['type.name'] == 'Shot') & (E['shot.outcome.name'] == 'Goal')) | (E['type.name'] == 'Own Goal For'))
    gh = (es_gol & E['ev_local']).astype(int); ga = (es_gol & ~E['ev_local'] & E['team.name'].notna()).astype(int)
    E['dif_home'] = (gh.groupby(E['match_id']).cumsum() - gh) - (ga.groupby(E['match_id']).cumsum() - ga)
    E['m_home'] = E['dif_home'] + E['match_id'].map(loc['offset']); E['m_home_b'] = E['dif_home'] + E['match_id'].map(loc['offset_b'])
    E['cuarto'] = np.where(E['period'] == 1, np.where(E['minute'] < 25, 1, 2), np.where(E['minute'] < 70, 3, 4))
    sh = E[(E['type.name'] == 'Shot') & (E['shot.type.name'] != 'Penalty') & E.period.isin([1, 2]) & (E['shot.statsbomb_xg'] > 0)]
    sh = sh.rename(columns={'team.name': 'team'}); sh['e5'], sh['e5b'] = estado(sh)
    agrega(sh, 100.0 * sh['shot.statsbomb_xg'] * (sh.fase == 'bp'), 'peso_bp', den=sh['shot.statsbomb_xg'].values)
print('--- (1) Chequeo de t: % de eventos con t = segundos desde el inicio del PERIODO (minuto − 0/45/90/105):', dict(ch_t), '(debe ser ~100)')

TR = pd.concat(piezas, ignore_index=True).groupby(['metrica', 'match_id', 'team', 'e5', 'e5b', 'cuarto'], as_index=False)[['num', 'den']].sum()
TR = TR[TR.den != 0]
TR.to_parquet(os.path.join(FW5, 'r8_tramos.parquet'), index=False)
res = TR.groupby('metrica').apply(lambda d: pd.Series({'equipo_partidos': d[['match_id', 'team']].drop_duplicates().shape[0],
                                                       'media_liga': d.num.sum() / d.den.sum()}), include_groups=False)
print(f'--- (3) Rasgos ({TR.metrica.nunique()}). Media de la liga en su unidad:'); print(res.round(3).to_string())
print(f'X6-R8u-0 terminó en {time.time() - t0:.0f} s. Guardado (espejo): r8_tramos. Siguiente: X6-p con PREFIJO = "r8" → R6r-1 → R6r-2.')


In [ ]:
# [X6-9 v1.0] Validación de X6 contra fw5 y copia a Drive (preregistro de Almada, §4 paso 4; Aclaración 1)
# =====================================================================
# Sesión 13 · 2026-10-20 · Correr AL FINAL de la hoja de ruta X6, en la misma sesión de Colab. Tarda ~1-3 min.
# (1) Copia a Drive todo lo que X6 generó (no los enlaces): datos_parquet/fw6/ y salidas/fw6/. fw1 y fw5 no se tocan.
# (2) Reproducción (como X5-1): en los partidos de 2021/22-2025/26, los TRAMOS de los ocho bloques deben ser idénticos a fw5
#     (mismo código, mismos datos; solo cambia que hay una temporada más).
# (3) Validación obligatoria del preregistro: perfiles de las etapas ya existentes (mismos partidos que en fw5) con correlación
#     ≥ 0.99 por rasgo frente a fw5, en los 65 rasgos de C-1. Las etapas que ganaron partidos en 2026/27 se reportan aparte.
# (4) Escribe datos_parquet/fw6/validacion_x6.json (V-1 no corre si ok = false) y regresa BASE a Drive.
# NO imprime ningún perfil de equipo ni de técnico: solo conteos y correlaciones.
# =====================================================================
import os, json, shutil
import numpy as np
import pandas as pd

DRIVE = '/content/drive/MyDrive/isac2026/'
LOCAL = '/content/isac_fw6/'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado'
assert globals().get('RAIZ_X6') == LOCAL, 'Corre X6-9 en la misma sesión que X6-0 y toda la hoja de ruta'
L_FW = os.path.join(LOCAL, 'datos_parquet', 'fw1')
D_DP = os.path.join(DRIVE, 'datos_parquet'); D_FW5, D_FW6 = os.path.join(D_DP, 'fw5'), os.path.join(D_DP, 'fw6')
D_SAL5, D_SAL6 = os.path.join(DRIVE, 'salidas', 'fw5'), os.path.join(DRIVE, 'salidas', 'fw6')
os.makedirs(D_FW6, exist_ok=True); os.makedirs(D_SAL6, exist_ok=True)
PREF = ['r4', 'r6', 'r6b', 'r6c', 'r7u', 'r7b', 'r7c', 'r8']
sal = lambda p: 'r5' if p == 'r4' else p

# ---------- (1) Copia ----------
n = []
for src_dir, dst_dir in [(L_FW, D_FW6), (os.path.join(LOCAL, 'salidas'), D_SAL6), (os.path.join(LOCAL, 'salidas', 'fw5'), D_SAL6)]:
    for f in os.listdir(src_dir):
        s = os.path.join(src_dir, f)
        if os.path.isfile(s) and not os.path.islink(s):
            shutil.copy2(s, os.path.join(dst_dir, f)); n.append(f)
print(f'--- (1) Copiados a Drive: {len(n)} archivos → datos_parquet/fw6/ y salidas/fw6/')
falta = [f'{p}_perfiles_contraidos' for p in [sal(x) for x in PREF] if not os.path.exists(os.path.join(D_FW6, f'{p}_perfiles_contraidos.parquet'))]
falta += [f'{sal(p)}_confiabilidad.csv' for p in PREF if not os.path.exists(os.path.join(D_SAL6, f'{sal(p)}_confiabilidad.csv'))]
print('Insumos de la compatibilidad que faltan en fw6:', falta or 'ninguno')

# ---------- (2) Reproducción de tramos en 2021/22-2025/26 ----------
mid5 = set(pd.read_parquet(os.path.join(D_FW5, 'r4_partidos.parquet'))['match_id'])
filas = []
for arch in ['r4_tramos_v21', 'r6_tramos', 'r6b_tramos', 'r6c_tramos', 'r7u_tramos', 'r7b_tramos', 'r7c_tramos', 'r8_tramos']:
    a = pd.read_parquet(os.path.join(D_FW5, f'{arch}.parquet'))
    b = pd.read_parquet(os.path.join(D_FW6, f'{arch}.parquet'))
    k = [c for c in ['metrica', 'match_id', 'team', 'e5', 'e5b', 'cuarto'] if c in a.columns and c in b.columns]
    j = a.merge(b[b['match_id'].isin(mid5)], on=k, how='outer', suffixes=('_5', '_6'), indicator=True)
    igual = np.isclose(j['num_5'].fillna(-1), j['num_6'].fillna(-1)) & np.isclose(j['den_5'].fillna(-1), j['den_6'].fillna(-1))
    filas.append(dict(archivo=arch, tramos_fw5=len(a), solo_en_uno=int((j['_merge'] != 'both').sum()), iguales_pct=100 * igual.mean(),
                      partidos_2026_27=int(b.loc[~b['match_id'].isin(mid5), 'match_id'].nunique())))
V = pd.DataFrame(filas)
print('\n--- (2) Reproducción de tramos en las cinco temporadas previas (solo_en_uno = 0 e iguales_pct = 100 en los ocho):')
print(V.round(3).to_string(index=False))
OK_TR = bool((V['solo_en_uno'] == 0).all() and (V['iguales_pct'] > 99.999).all() and (V['partidos_2026_27'] > 90).all())

# ---------- (3) Perfiles de las etapas existentes frente a fw5 ----------
RAS = pd.read_csv(os.path.join(D_SAL5, 'c_mapa.csv'))['rasgo'].tolist()           # los 65 de C-1 (congelados)
ptp5 = pd.read_parquet(os.path.join(D_DP, 'partido_tecnico_periodo.parquet'))
ptp6 = pd.read_parquet(os.path.join(D_FW6, 'partido_tecnico_periodo.parquet'))
with open(os.path.join(D_FW5, 'parametros_fw1.json')) as f:
    T5 = set(json.load(f)['temporadas']['entrenamiento'])
eid = lambda d: d['team'] + ' · ' + d['etapa'].astype(str)
s5 = ptp5[ptp5.temporada.isin(T5) & ptp5.jugado.astype(bool)].assign(e=eid).groupby('e')['match_id'].apply(frozenset)
s6 = ptp6[ptp6.jugado.astype(bool)].assign(e=eid).groupby('e')['match_id'].apply(frozenset)
igual_e = [e for e in s5.index if e in s6.index and s5[e] == s6[e]]
crece = [e for e in s5.index if e in s6.index and s5[e] != s6[e]]
print(f'\n--- (3) Etapas en fw5: {len(s5)} · con los mismos partidos en X6: {len(igual_e)} · que ganaron partidos en 2026/27: {len(crece)} · '
      f'etapas nuevas en 2026/27: {len(set(s6.index) - set(s5.index))}')
res = []
for p in PREF:
    a = pd.read_parquet(os.path.join(D_FW5, f'{sal(p)}_perfiles_contraidos.parquet'))
    b = pd.read_parquet(os.path.join(D_FW6, f'{sal(p)}_perfiles_contraidos.parquet'))
    j = a.merge(b, on=['metrica', 'etapa_id'], suffixes=('_5', '_6'))
    for met, g in j.groupby('metrica'):
        r = f'{p}:{met}'
        if r not in RAS: continue
        def c(gg, col):
            m = np.isfinite(gg[col + '_5']) & np.isfinite(gg[col + '_6'])
            return np.corrcoef(gg.loc[m, col + '_5'], gg.loc[m, col + '_6'])[0, 1] if m.sum() >= 5 else np.nan
        gi, g15 = g[g.etapa_id.isin(igual_e)], g[g.etapa_id.isin(igual_e) & (g.partidos_5 >= 15)]
        gc = g[g.etapa_id.isin(crece)]
        res.append(dict(rasgo=r, etapas=len(gi), corr_perfil=c(gi, 'perfil_r4'), corr_mostrado=c(gi, 'perfil_mostrado'),
                        corr_15=c(g15, 'perfil_r4'), dif_max_z=np.nanmax(np.abs(gi.perfil_mostrado_5 - gi.perfil_mostrado_6)) /
                        max(1e-12, float(pd.read_csv(os.path.join(D_SAL5, f'{sal(p)}_confiabilidad.csv')).set_index('metrica').loc[met, 'tau'])),
                        corr_etapas_que_crecen=c(gc, 'perfil_r4') if len(gc) >= 5 else np.nan))
R = pd.DataFrame(res)
cf6 = pd.concat([pd.read_csv(os.path.join(D_SAL6, f'{sal(p)}_confiabilidad.csv')).assign(rasgo=lambda d: p + ':' + d.metrica) for p in PREF])
cf5 = pd.concat([pd.read_csv(os.path.join(D_SAL5, f'{sal(p)}_confiabilidad.csv')).assign(rasgo=lambda d: p + ':' + d.metrica) for p in PREF])
cc = cf5.set_index('rasgo')[['conf', 'tau']].join(cf6.set_index('rasgo')[['conf', 'tau']], lsuffix='_5', rsuffix='_6').reindex(RAS)
print(f'Rasgos de C-1 encontrados en X6: {len(R)} de {len(RAS)} (deben ser {len(RAS)})')
print(f'Correlación por rasgo (etapas con los mismos partidos), perfil ajustado: mínima {R.corr_perfil.min():.4f} · mediana {R.corr_perfil.median():.4f} · '
      f'rasgos < 0.99: {int((R.corr_perfil < 0.99).sum())}')
print(f'  perfil mostrado (tras P-5): mínima {R.corr_mostrado.min():.4f} · solo etapas de 15+: mínima {R.corr_15.min():.4f} · '
      f'cambio máximo en z (perfil mostrado ÷ τ de fw5): mediana {R.dif_max_z.median():.3f} · máximo {R.dif_max_z.max():.3f}')
print(f'  etapas que ganaron partidos (informativo): correlación mínima {R.corr_etapas_que_crecen.min():.3f}')
if (R.corr_perfil < 0.99).any(): print(R[R.corr_perfil < 0.99].round(4).to_string(index=False))
print(f'Confiabilidad X6 frente a fw5 en los 65: cambio máximo {np.nanmax(np.abs(cc.conf_6 - cc.conf_5)):.3f} · '
      f'rasgos que en X6 quedarían < 0.5: {int((cc.conf_6 < 0.5).sum())} (la lista sigue siendo la congelada; solo se reporta) · '
      f'τ X6/fw5: mediana {np.nanmedian(cc.tau_6 / cc.tau_5):.3f}')
OK_PF = bool(len(R) == len(RAS) and (R.corr_perfil >= 0.99).all())

# ---------- (4) Marca y regreso a Drive ----------
OK = OK_TR and OK_PF and not falta
with open(os.path.join(D_FW6, 'validacion_x6.json'), 'w') as f:
    json.dump(dict(ok=OK, tramos=OK_TR, perfiles=OK_PF, faltan=falta, tabla_tramos=V.to_dict('records'),
                   corr_min=float(R.corr_perfil.min()), rasgos_bajo_099=R.loc[R.corr_perfil < 0.99, 'rasgo'].tolist()), f, indent=1, default=str)
R.to_csv(os.path.join(D_SAL6, 'x6_validacion_perfiles.csv'), index=False)
for v in ['RAIZ_PRUEBA', 'RAIZ_X6']: globals().pop(v, None)
BASE = DRIVE
print('\nVALIDACIÓN X6:', 'OK → se puede correr V-1' if OK else 'FALLÓ → NO correr V-1; manda esta salida a Claude',
      '· BASE =', BASE, '(RAIZ_PRUEBA y RAIZ_X6 borrados)')


In [ ]:
# [V-1 v1.0] Evaluación preregistrada de Almada en América: P1-P4 (preregistro §3-§4 y Aclaraciones 1 y 2)
# =====================================================================
# Sesión 13 · 2026-10-20 · Antes: celda 1 (Drive) → C-0 (claude/codigo/sesion12/C_0_prueba.py, SIN CAMBIOS: define las funciones y
# debe terminar con todas sus pruebas PASAN) → X6 completo con "VALIDACIÓN X6: OK" → V-1. Si X6-9 se corrió en otra sesión, basta
# con celda 1 → C-0 → V-1. Tarda ~3-6 min. No usa C-1 ni C-3 de la sesión 12 en memoria: los rehace aquí.
# Código: las secciones 0, 1, 3 y 4 de C-1 y el bucle de C-3 se copian SIN CAMBIOS DE LÓGICA dentro de construir() y ensayo(); lo único
# distinto es de dónde leen (fw5 o fw6) y que la lista de rasgos es la congelada de C-1 (Aclaración 1, punto 1).
# Pasos (cada uno se detiene si falla):
#   (A) REPRODUCCIÓN sobre fw5: las 27 llegadas de C-3 con K = 9, 12 y 15 deben dar los mismos números que salidas/fw5/
#       c_ensayo_llegadas.csv (compatibilidad, cambio, percentil y puesto de identificación). Si no, no se evalúa a Almada.
#   (B) X6: mismas 27 llegadas (las de 2026/27 no entran al ensayo) → umbral de P1 = percentil 75 del cambio con K = 11.
#   (C) Almada: zt (Pachuca · 2), zc (últimos 18 de América antes de su llegada), zn (sus K partidos de 2026/27), pesos sin sus parejas.
#   (D) P1 a P4 por separado (cumple / no cumple / no evaluado) con su valor · cambios de zt y zc > 0.1 z frente a fw5 · sensibilidad fw5.
# Escribe (con freno de Drive) salidas/fw6/v1_resultado.csv, v1_rasgos.csv, v1_p4_ranking.csv, v1_reproduccion.csv.
# =====================================================================
import os, json, time
from types import SimpleNamespace
import numpy as np
import pandas as pd

DRIVE = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado; corre la celda 1'
assert globals().get('C0_OK', False), 'Corre primero C-0 y confirma que todas sus pruebas PASAN'
assert 'RAIZ_PRUEBA' not in globals() and 'RAIZ_X6' not in globals(), 'Quedan variables de X6: corre X6-9 (las borra) o reinicia y corre celda 1 → C-0'
DP = os.path.join(DRIVE, 'datos_parquet')
VERS = {'fw5': dict(FW=os.path.join(DP, 'fw5'), SAL=os.path.join(DRIVE, 'salidas', 'fw5'), PTP=os.path.join(DP, 'partido_tecnico_periodo.parquet')),
        'fw6': dict(FW=os.path.join(DP, 'fw6'), SAL=os.path.join(DRIVE, 'salidas', 'fw6'), PTP=os.path.join(DP, 'fw6', 'partido_tecnico_periodo.parquet'))}
with open(os.path.join(VERS['fw6']['FW'], 'validacion_x6.json')) as f:
    VX6 = json.load(f)
assert VX6.get('ok'), 'X6 no pasó su validación (validacion_x6.json): no se evalúa'
PREFIJOS = ['r4', 'r6', 'r6b', 'r6c', 'r7u', 'r7b', 'r7c', 'r8']
ETAPA_MIN, NUEVA_MIN = 15, 10                      # C-1
VENTANA, VENT_MIN = 18, 10                         # S12-05
CONF_MIN = 0.5
RELOJ_P4 = 20                                      # C-3
FOCO = {'P2a': 'r8:def_area_360', 'P2b': 'r8:zonales', 'P3': 'r6:seg_hasta_ultimo'}
INICIO_VIVO = pd.Timestamp('2026-07-01')           # llegadas de 2026/27: no entran al ensayo (las 27 son históricas)
RAS = pd.read_csv(os.path.join(VERS['fw5']['SAL'], 'c_mapa.csv'))['rasgo'].tolist()           # los 65 congelados de C-1
CUAD = pd.read_csv(os.path.join(VERS['fw5']['SAL'], 'c_mapa.csv')).set_index('rasgo')['cuadrante']
sal = lambda p: 'r5' if p == 'r4' else p
corto = lambda s: ' '.join(s.split()[:1] + [s.split()[-2] if len(s.split()) >= 3 else s.split()[-1]]) if isinstance(s, str) and s.strip() else s
t0 = time.time(); pd.set_option('display.width', 250)


def construir(nombre):
    """C-1 secciones 0, 1, 3 y 4 (sin cambios de lógica) sobre la versión indicada."""
    FW5, SAL5, PTPF = VERS[nombre]['FW'], VERS[nombre]['SAL'], VERS[nombre]['PTP']
    REQ = {'contraidos': ['metrica', 'etapa_id', 'perfil_r4', 'perfil_mostrado'], 'confiabilidad': ['metrica', 'conf', 'tau', 'sigma_partido'],
           'perfiles_etapa': ['etapa_id', 'metrica', 'tau_estado'], 'residuos': ['metrica', 'match_id', 'team', 'e', 'den', 'res']}
    CONT, CF, TAUE, RES, S2 = [], [], {}, {}, {}
    for p in PREFIJOS:
        fuentes = {'contraidos': os.path.join(FW5, f'{sal(p)}_perfiles_contraidos.parquet'), 'confiabilidad': os.path.join(SAL5, f'{sal(p)}_confiabilidad.csv'),
                   'perfiles_etapa': os.path.join(FW5, f'{p}_perfiles_etapa.parquet'), 'residuos': os.path.join(FW5, f'{p}_residuos.parquet')}
        D = {k: (pd.read_csv(v) if v.endswith('.csv') else pd.read_parquet(v)) for k, v in fuentes.items()}
        faltan = {k: [c for c in REQ[k] if c not in D[k].columns] for k in D}
        assert not any(faltan.values()), f'{nombre}: columnas faltantes en {p}: {faltan}'
        CONT.append(D['contraidos'].assign(rasgo=lambda d: p + ':' + d.metrica))
        CF.append(D['confiabilidad'].assign(rasgo=lambda d: p + ':' + d.metrica, prefijo=p))
        TAUE.update({p + ':' + m: v for m, v in D['perfiles_etapa'].groupby('metrica')['tau_estado'].first().items()})
        rs = D['residuos'][REQ['residuos']]
        S2.update({p + ':' + m: float((g.den * g.res ** 2).sum() / (len(g) - 19)) for m, g in rs.groupby('metrica')})
        RES[p] = {t: g for t, g in rs.groupby('team')}
    CONT = pd.concat(CONT, ignore_index=True); CF = pd.concat(CF, ignore_index=True).set_index('rasgo')
    RASGOS_SEL = [r for r in CF.index if CF.loc[r, 'conf'] >= CONF_MIN and CF.loc[r, 'tau'] > 0]
    faltan = [r for r in RAS if r not in CF.index or not CF.loc[r, 'tau'] > 0]
    assert not faltan, f'{nombre}: rasgos congelados sin dato: {faltan}'
    RASGOS = list(RAS)                                                       # Aclaración 1 (1): lista congelada
    with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
        TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
    ptp = pd.read_parquet(PTPF)
    if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
    ptp = ptp[ptp['temporada'].isin(TEMPS)].copy(); ptp['fecha'] = pd.to_datetime(ptp['fecha'])
    ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
    ET = ptp.groupby('etapa_id').agg(team=('team', 'first'), tecnico=('tec_p', 'first'), partidos=('match_id', 'nunique'),
                                     ini=('fecha', 'min'), fin=('fecha', 'max')).reset_index()
    ETAPAS = ET.etapa_id.tolist(); IX = {e: i for i, e in enumerate(ETAPAS)}
    tau = CF.loc[RASGOS, 'tau'].values; conf = CF.loc[RASGOS, 'conf'].values
    ZR = (CONT.pivot_table(index='etapa_id', columns='rasgo', values='perfil_r4').reindex(index=ETAPAS, columns=RASGOS) / tau).values
    ZM = (CONT.pivot_table(index='etapa_id', columns='rasgo', values='perfil_mostrado').reindex(index=ETAPAS, columns=RASGOS) / tau).values
    L = ET[ET.partidos >= ETAPA_MIN]
    pt, ct = [], []
    for tec, d in L.groupby('tecnico'):
        ix = [IX[e] for e in d.etapa_id]
        for i in range(len(ix)):
            for j in range(i + 1, len(ix)):
                if ET.team[ix[i]] != ET.team[ix[j]]: pt.append((ix[i], ix[j])); ct.append(tec)
    pt, ct = np.array(pt), np.array(ct)
    MIN_ = pd.read_parquet(os.path.join(FW5, 'r4_minutos_v21.parquet'))
    W3 = MIN_.assign(e3=np.sign(MIN_.e5)).groupby('e3')['minutos'].sum(); W3v = (W3 / W3.sum()).reindex([-1, 0, 1]).fillna(0).values

    def perfil_club(team, partidos, contraido=True, rasgos=None):
        rasgos = RASGOS if rasgos is None else rasgos
        z = np.full(len(rasgos), np.nan)
        for j, r in enumerate(rasgos):
            p, m = r.split(':', 1); g = RES[p].get(team)
            if g is None: continue
            g = g[(g.metrica == m) & g.match_id.isin(partidos)]
            n = g.match_id.nunique()
            if n == 0: continue
            pf = perfil_ventana(g.e.values, g.den.values, g.res.values, S2[r], TAUE[r], W3v)
            z[j] = (contraer(pf, n, CF.loc[r, 'tau'], CF.loc[r, 'sigma_partido']) if contraido else pf) / CF.loc[r, 'tau']
        return z
    def n_con_dato(team, partidos, r):
        p, m = r.split(':', 1); g = RES[p].get(team)
        return 0 if g is None else int(g[(g.metrica == m) & g.match_id.isin(partidos)].match_id.nunique())
    def ventana(team, antes_de):
        m = ptp[(ptp.team == team) & (ptp.fecha < antes_de)].sort_values('fecha').tail(VENTANA)
        return set(m.match_id), len(m)
    def pesos_sin(tec):
        keep = ct != tec
        r = r_pares(ZR, pt[keep]) / np.clip(conf, .05, None)
        return np.clip(np.nan_to_num(r), 0, PESO_MAX)
    def candidatos(antes_de, excluir_tec, excluir_team):
        c = L[(L.ini < antes_de) & (L.tecnico != excluir_tec) & (L.team != excluir_team)].sort_values('ini').groupby('tecnico').tail(1)
        return [IX[e] for e in c.etapa_id], c.tecnico.tolist()
    filas = []
    for r_ in ET[ET.partidos >= NUEVA_MIN].itertuples(index=False):
        ant = L[(L.tecnico == r_.tecnico) & (L.team != r_.team) & (L.fin < r_.ini)]
        if not len(ant): continue
        prev = ant.sort_values('fin').etapa_id.iloc[-1]
        vp, nv = ventana(r_.team, r_.ini)
        if nv < VENT_MIN: filas.append(dict(etapa_nueva=r_.etapa_id, tecnico=corto(r_.tecnico), previa=prev, ventana=nv, nota='ventana corta')); continue
        zc = perfil_club(r_.team, vp); zt = ZM[IX[prev]]; w = pesos_sin(r_.tecnico)
        cix, _ = candidatos(r_.ini, r_.tecnico, r_.team)
        d = distancia(zt, zc, w); comp = percentil(d, [distancia(ZM[k], zc, w) for k in cix])
        zn = ZM[IX[r_.etapa_id]]
        filas.append(dict(etapa_nueva=r_.etapa_id, tecnico=corto(r_.tecnico), tec_full=r_.tecnico, ini=r_.ini, previa=prev, ventana=nv,
                          partidos_nueva=r_.partidos, candidatos=len(cix), compat=comp, distancia=d, cambio=distancia(zc, zn, w),
                          v_zc=zc, v_zt=zt, v_zn=zn, v_w=w, v_cix=cix, nota=''))
    LL = pd.DataFrame(filas)
    ok = LL[LL.nota == ''].copy()
    return SimpleNamespace(**{k: v for k, v in locals().items() if k != 'filas'})


def ensayo(S, K_LIST):
    """Bucle de C-3 (sin cambios de lógica): cambio con los primeros K partidos e identificación a ciegas. Solo llegadas históricas."""
    reloj = S.CF.loc[S.RASGOS, 'reloj'].values; legible = reloj <= RELOJ_P4
    filas = []
    for f in S.ok[S.ok.ini < INICIO_VIVO].to_dict('records'):
        e_new = f['etapa_nueva']; k_new = S.IX[e_new]; team = S.ET.team[k_new]; tec_full = S.ET.tecnico[k_new]
        w = S.pesos_sin(tec_full); zc, zt = f['v_zc'], f['v_zt']
        mids = S.ptp[S.ptp.etapa_id == e_new].sort_values('fecha').match_id.tolist()
        for K in K_LIST:
            if len(mids) < K: continue
            zk = S.perfil_club(team, set(mids[:K]))
            cam_k = distancia(zc, zk, w)
            wl = w * legible
            d_real = distancia(zk, zt, wl)
            d_cand = [distancia(zk, S.ZM[j], wl) for j in f['v_cix']]
            filas.append(dict(etapa_nueva=e_new, tecnico=f['tecnico'], K=K, compat=f['compat'], cambio_completo=f['cambio'], cambio_K=cam_k,
                              id_percentil=percentil(d_real, d_cand), id_puesto=1 + int(np.sum(np.array(d_cand) < d_real)), candidatos=len(d_cand) + 1))
    return pd.DataFrame(filas)


# ---------- (A) Reproducción sobre fw5 ----------
S5 = construir('fw5')
assert S5.RASGOS == [r for r in S5.RASGOS_SEL], 'fw5: la lista congelada no coincide con la selección de C-1 (orden o contenido)'
EN5 = ensayo(S5, [9, 12, 15])
REF = pd.read_csv(os.path.join(VERS['fw5']['SAL'], 'c_ensayo_llegadas.csv'))
J = REF.merge(EN5, on=['etapa_nueva', 'K'], how='outer', suffixes=('_ref', '_v1'), indicator=True)
COLS_R = ['compat', 'cambio_completo', 'cambio_K', 'id_percentil', 'id_puesto', 'candidatos']
difs = {c: float(np.nanmax(np.abs(J[c + '_ref'] - J[c + '_v1']))) for c in COLS_R}
OK_REP = bool((J._merge == 'both').all() and max(difs.values()) < 1e-6)
print(f'--- (A) Reproducción de C-3 sobre fw5: llegadas {EN5.etapa_nueva.nunique()} · filas (llegada × K) {len(EN5)} de {len(REF)} · '
      f'solo en una {int((J._merge != "both").sum())} · diferencia máxima {max(difs.values()):.2e} (debe ser < 1e-6) · {time.time() - t0:.0f} s')
print('  por columna:', {k: f'{v:.1e}' for k, v in difs.items()})
assert OK_REP, 'V-1 NO reproduce C-3: no se evalúa a Almada. Manda esta salida a Claude.'

# ---------- (B) X6: ensayo con K = 11 → umbral de P1 ----------
S6 = construir('fw6')
lleg5, lleg6 = set(S5.ok.etapa_nueva), set(S6.ok[S6.ok.ini < INICIO_VIVO].etapa_nueva)
EN6 = ensayo(S6, [11]); EN5_11 = ensayo(S5, [11])
UMBRAL_P1, UMBRAL_P1_FW5 = float(EN6.cambio_K.quantile(.75)), float(EN5_11.cambio_K.quantile(.75))
print(f'\n--- (B) X6 · llegadas históricas {len(lleg6)} (iguales a las de fw5: {lleg6 == lleg5}) · llegadas de 2026/27 fuera del ensayo: '
      f'{int((S6.ok.ini >= INICIO_VIVO).sum())} · con 11+ partidos para K = 11: {len(EN6)}')
print(f'  Cambio con K = 11 (z): mediana {EN6.cambio_K.median():.2f} · p75 {UMBRAL_P1:.3f} ← UMBRAL DE P1 · p90 {EN6.cambio_K.quantile(.9):.2f} '
      f'· (fw5 con K = 11: p75 {UMBRAL_P1_FW5:.3f}; indicativos del preregistro K = 9 → 1.01, K = 12 → 1.06)')
print(f'  Identificación a ciegas en las llegadas, K = 11: en el 10% más parecido {100 * (EN6.id_percentil >= 90).mean():.0f}% (ensayo K = 12: 37%) · '
      f'percentil del técnico real: p25 {EN6.id_percentil.quantile(.25):.0f} · mediana {EN6.id_percentil.median():.0f} · p75 {EN6.id_percentil.quantile(.75):.0f} (azar = 50)')
assert lleg6 == lleg5, 'Las llegadas históricas de X6 no son las mismas 27 de fw5: avisar a Claude'

# ---------- (C) Almada ----------
def almada(S):
    et = S.ET
    e_am = et[et.team.str.startswith('Am') & et.tecnico.str.contains('Almada', na=False)]
    e_pa = et[et.team.str.startswith('Pachuca') & et.tecnico.str.contains('Almada', na=False)].sort_values('partidos')
    return (e_am.etapa_id.iloc[0] if len(e_am) else None), e_pa.etapa_id.iloc[-1], et
e_am, e_pa, ET6 = almada(S6)
assert e_am is not None, 'No está la etapa de Almada en América en X6'
r_am = ET6.set_index('etapa_id').loc[e_am]; team_am, tec_alm, ini_alm = r_am.team, r_am.tecnico, r_am.ini
mids_alm = S6.ptp[S6.ptp.etapa_id == e_am].sort_values('fecha').match_id.tolist(); K = len(mids_alm)
vp, nv = ventana_ = S6.ventana(team_am, ini_alm)
fin_v = S6.ptp[S6.ptp.match_id.isin(vp) & (S6.ptp.team == team_am)].fecha.max().date()
zt, zc, zn = S6.ZM[S6.IX[e_pa]], S6.perfil_club(team_am, vp), S6.perfil_club(team_am, set(mids_alm))
w = S6.pesos_sin(tec_alm)
print(f'\n--- (C) Almada · etapa nueva {e_am}: K = {K} partidos ({pd.Timestamp(ini_alm).date()} a {S6.ptp[S6.ptp.etapa_id == e_am].fecha.max().date()}) · '
      f'técnico: {e_pa} ({int(ET6.set_index("etapa_id").loc[e_pa, "partidos"])} partidos; preregistro: Pachuca · 2, 139) · '
      f'club: últimos {nv} de {team_am} hasta {fin_v} (preregistro: 18, hasta 2026-05-11) · rasgos con peso > 0: {int((w > 0).sum())}')

# ---------- (D) Predicciones ----------
R = S6.RASGOS; ix = {r: i for i, r in enumerate(R)}
liga6 = S6.CONT.groupby('rasgo')['liga'].first().reindex(R).values; tau6 = S6.CF.loc[R, 'tau'].values
def cob(i): return (zn[i] - zc[i]) / (zt[i] - zc[i]) if np.isfinite(zt[i] - zc[i]) and zt[i] != zc[i] else np.nan
filas = []
# P1
d1 = distancia(zc, zn, w)
filas.append(dict(id='P1', prediccion='América cambia más que una llegada típica', valor=round(d1, 3), umbral=f'> {UMBRAL_P1:.3f} (p75, K = {K})',
                  resultado='cumple' if d1 > UMBRAL_P1 else 'no cumple'))
# P2 y P3
det = []
for clave, r in FOCO.items():
    i = ix[r]; c = cob(i); n = S6.n_con_dato(team_am, set(mids_alm), r)
    det.append(dict(pred=clave, rasgo=r, partidos_con_dato=n, z_america_antes=zc[i], z_almada=zt[i], z_nuevo=zn[i], cobertura=c,
                    nat_america_antes=liga6[i] + zc[i] * tau6[i], nat_almada=liga6[i] + zt[i] * tau6[i], nat_nuevo=liga6[i] + zn[i] * tau6[i],
                    liga=liga6[i], peso=w[i]))
DET = pd.DataFrame(det)
c2 = DET.set_index('pred').cobertura
p2_ok = c2[['P2a', 'P2b']].notna().all()
filas.append(dict(id='P2', prediccion='Lo del técnico viaja (defensores en el área en córner y zonales)',
                  valor=f"{c2['P2a']:.2f} y {c2['P2b']:.2f}", umbral='≥ 0.5 en los dos',
                  resultado=('cumple' if (c2[['P2a', 'P2b']] >= 0.5).all() else 'no cumple') if p2_ok else 'no evaluado'))
filas.append(dict(id='P3', prediccion='Lo del club no viaja (segundos hasta el último tercio)', valor=f"{c2['P3']:.2f}", umbral='< 0.5',
                  resultado=('cumple' if c2['P3'] < 0.5 else 'no cumple') if np.isfinite(c2['P3']) else 'no evaluado'))
# P4: candidatos del ranking de C-1 (incluye a Almada en Pachuca), perfiles con partidos anteriores a su llegada (Aclaraciones 1 y 2)
RK = pd.read_csv(os.path.join(VERS['fw5']['SAL'], 'c_ranking_america.csv'), index_col=0)
assert e_pa in set(RK.etapa), 'Almada en Pachuca no está en el ranking de C-1'
reloj6 = S6.CF.loc[R, 'reloj'].values; reloj5 = S5.CF.loc[R, 'reloj'].values
wl = w * (reloj6 <= RELOJ_P4)
def perfil_antes(S, e):
    tm = S.ET.set_index('etapa_id').loc[e, 'team']
    m = set(S.ptp[(S.ptp.etapa_id == e) & (S.ptp.fecha < ini_alm)].match_id)
    return S.perfil_club(tm, m), len(m), int(S.ET.set_index('etapa_id').loc[e, 'partidos'])
P4 = []
for e in RK.etapa:
    z_e, n_antes, n_total = perfil_antes(S6, e)
    P4.append(dict(etapa=e, tecnico=corto(S6.ET.set_index('etapa_id').loc[e, 'tecnico']), partidos_antes=n_antes, partidos_X6=n_total,
                   real=(e == e_pa), distancia=distancia(zn, z_e, wl)))
P4 = pd.DataFrame(P4).sort_values('distancia').reset_index(drop=True); P4.index += 1
d_real = float(P4.loc[P4.real, 'distancia'].iloc[0]); d_cand = P4.loc[~P4.real, 'distancia'].values
pct4, puesto4 = percentil(d_real, d_cand), 1 + int(np.sum(d_cand < d_real))
filas.append(dict(id='P4', prediccion='Identificación a ciegas: los datos señalan a Almada', valor=f'percentil {pct4:.0f} (puesto {puesto4} de {len(P4)})',
                  umbral='percentil ≥ 90', resultado='cumple' if pct4 >= 90 else 'no cumple'))
RES_ = pd.DataFrame(filas)
print(f'\n--- (D) RESULTADO (lectura literal, E-16) · K = {K} · versión X6 ---')
print(RES_.to_string(index=False))
print('\nDetalle P2-P3 (z = desviaciones típicas entre técnicos; nat = unidad del rasgo; cobertura 0 = no se movió, 1 = igual a Almada):')
print(DET.round(2).to_string(index=False))
print(f'\nP4 · rasgos usados: {int((wl > 0).sum())} (reloj ≤ {RELOJ_P4} y peso > 0; con el reloj de fw5 serían {int(((w > 0) & (reloj5 <= RELOJ_P4)).sum())}) · '
      f'candidatos con partidos después del 19-jul recortados: {int((P4.partidos_X6 > P4.partidos_antes).sum())} · los 8 más parecidos:')
print(P4.head(8)[['tecnico', 'etapa', 'distancia', 'real']].round(3).to_string())

# Cambios de zt y zc frente a fw5 (preregistro §4 paso 4: > 0.1 z en un rasgo de P2-P4 se reporta)
zt5, zc5 = S5.ZM[S5.IX[e_pa]], S5.perfil_club(team_am, vp)
usados = sorted(set(FOCO.values()) | {R[i] for i in np.where(wl > 0)[0]})
CH = pd.DataFrame([dict(rasgo=r, dz_almada=zt[ix[r]] - zt5[ix[r]], dz_america=zc[ix[r]] - zc5[ix[r]]) for r in usados])
grandes = CH[(CH.dz_almada.abs() > 0.1) | (CH.dz_america.abs() > 0.1)]
print(f'\nCambios de zt y zc (X6 − fw5) en los {len(usados)} rasgos de P2-P4: máximo |Δ| Almada {CH.dz_almada.abs().max():.3f} · América {CH.dz_america.abs().max():.3f} · '
      f'rasgos con > 0.1 z: {len(grandes)}')
if len(grandes): print(grandes.round(3).to_string(index=False))
# Sensibilidad fw5 (pesos, umbral, zt y zc de fw5; zn solo existe en X6)
w5 = S5.pesos_sin(tec_alm); d1_5 = distancia(zc5, zn, w5)
cob5 = {k: (zn[ix[r]] - zc5[ix[r]]) / (zt5[ix[r]] - zc5[ix[r]]) for k, r in FOCO.items()}
print(f'\nSensibilidad fw5 · P1: {d1_5:.3f} frente a {UMBRAL_P1_FW5:.3f} → {"cumple" if d1_5 > UMBRAL_P1_FW5 else "no cumple"} · '
      f'P2: {cob5["P2a"]:.2f} y {cob5["P2b"]:.2f} → {"cumple" if min(cob5["P2a"], cob5["P2b"]) >= 0.5 else "no cumple"} · '
      f'P3: {cob5["P3"]:.2f} → {"cumple" if cob5["P3"] < 0.5 else "no cumple"}')

# ---------- Guardar ----------
assert os.path.ismount('/content/drive'), 'Freno: Drive no está montado; no se escribe nada'
S6_SAL = VERS['fw6']['SAL']
RES_.assign(K=K, umbral_p1=UMBRAL_P1, umbral_p1_fw5=UMBRAL_P1_FW5).to_csv(os.path.join(S6_SAL, 'v1_resultado.csv'), index=False)
DET.to_csv(os.path.join(S6_SAL, 'v1_rasgos.csv'), index=False); P4.to_csv(os.path.join(S6_SAL, 'v1_p4_ranking.csv'))
J.to_csv(os.path.join(S6_SAL, 'v1_reproduccion.csv'), index=False); CH.to_csv(os.path.join(S6_SAL, 'v1_cambios_zt_zc.csv'), index=False)
print(f'\nV-1 terminó en {time.time() - t0:.0f} s. Guardado: salidas/fw6/v1_resultado, v1_rasgos, v1_p4_ranking, v1_reproduccion, v1_cambios_zt_zc.')

# Sección 14 (Entregables)

In [ ]:
# [G-0 v1.0] Cadena de relevos de los técnicos del selector (SOLO LECTURA: no escribe nada)
# =====================================================================
# Sesión 14 · 2026-10-20 · Chat del reporte · Decisiones S14-04 (selector) y S14-06 (cadena de relevos)
# Para cada etapa de Jardine, Larcamón y Ortiz en las cinco temporadas (fw5) imprime:
#   (1) quiénes son en la tabla de técnicos (nombres exactos) y sus etapas (club, partidos, fechas, rol);
#   (2) quién dirigía antes en ese club y si la ventana del club (últimos 18 partidos antes de llegar, mínimo 10; S12-05)
#       cabe dentro de los datos; qué técnicos componen esa ventana;
#   (3) nivel y resultados como CONTEXTO (no se interpretan como efecto del técnico): Elo medio de la etapa y su percentil
#       entre las etapas de 15+ partidos, Elo medio de la ventana del club, puntos por partido y diferencia de goles por partido;
#   (4) si la etapa tiene perfil en fw5 (r4_perfiles_etapa) y si entra en el mapa de C-1 como etapa de 15+.
# Orden: celda 1 (Drive) → G-0. Tarda < 1 min. No toca fw1, fw5 ni fw6.
# =====================================================================
import os, json
import numpy as np
import pandas as pd

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5')
VENTANA, VENTANA_MIN, ETAPA_MIN = 18, 10, 15                      # S12-05 y P-5
BUSCAR = {'Jardine': 'Jardine', 'Larcamón': 'Larcam', 'Ortiz': 'Ortiz'}
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 80)

with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
    TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
print('Temporadas de fw5:', TEMPS)

# ---------- Tabla de técnicos (mismo identificador que C-00, R4b, R5: etapa_id = equipo · etapa) ----------
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy(); ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ET = ptp.groupby('etapa_id').agg(team=('team', 'first'), tecnico=('tec_p', 'first'), partidos=('match_id', 'nunique'),
                                 ini=('fecha', 'min'), fin=('fecha', 'max'),
                                 rol=('rol', lambda s: s.mode().iat[0])).reset_index()
PRIMERA = ptp.groupby('team')['fecha'].min()                       # primer partido de cada club en los datos

# ---------- (1) Nombres y etapas ----------
print('\n--- (1) Nombres en la tabla de técnicos que contienen cada búsqueda (revisar que sean las personas correctas) ---')
SEL = {}
for nombre, patron in BUSCAR.items():
    cand = ET[ET.tecnico.astype(str).str.contains(patron, case=False)]
    print(f'  {nombre}: ' + (' | '.join(f"{t} → {', '.join(sorted(g.team))}" for t, g in cand.groupby('tecnico')) or 'NINGUNO'))
    if nombre == 'Ortiz':                                          # el Ortiz de América (puede haber homónimos)
        ok = cand.groupby('tecnico').team.apply(lambda s: s.str.startswith('Am').any())
        cand = cand[cand.tecnico.isin(ok[ok].index)]
    if cand.tecnico.nunique() != 1:
        print(f'  AVISO: {nombre} coincide con {cand.tecnico.nunique()} técnicos distintos; se listan todos y Claude revisa.')
    SEL[nombre] = cand.sort_values('ini')

# ---------- Elo y resultados por equipo-partido (fw1 tiene las cinco temporadas: 1,789 partidos) ----------
elo = pd.read_parquet(os.path.join(FW1, 'elo_equipo_partido_e2.parquet'))[['match_id', 'team', 'elo_pre']]
par = pd.read_parquet(os.path.join(FW1, 'elo_partido_e2.parquet'))[['match_id', 'local', 'visita', 'gl', 'gv']]
res = pd.concat([par.assign(team=par.local, gf=par.gl, gc=par.gv), par.assign(team=par.visita, gf=par.gv, gc=par.gl)])
res['pts'] = np.select([res.gf > res.gc, res.gf == res.gc], [3, 1], 0)
EP = ptp[['match_id', 'team', 'fecha', 'etapa_id', 'tec_p']].merge(elo, on=['match_id', 'team'], how='left') \
        .merge(res[['match_id', 'team', 'gf', 'gc', 'pts']], on=['match_id', 'team'], how='left')
print(f'\nChequeo de cruce: equipo-partidos {len(EP)} · sin Elo {int(EP.elo_pre.isna().sum())} · sin marcador {int(EP.pts.isna().sum())} '
      f'(ambos deben ser 0 o casi 0; si no, los nombres de equipo no coinciden entre tablas)')
ELO_ET = EP.groupby('etapa_id').elo_pre.mean()
ref15 = ELO_ET[ELO_ET.index.isin(ET.loc[ET.partidos >= ETAPA_MIN, 'etapa_id'])]

perf_ok = set()
ruta_p = os.path.join(FW5, 'r4_perfiles_etapa.parquet')
if os.path.exists(ruta_p): perf_ok = set(pd.read_parquet(ruta_p, columns=['etapa_id']).etapa_id)

# ---------- (2) y (3) Cadena por etapa ----------
filas = []
for nombre, etapas in SEL.items():
    for e in etapas.itertuples(index=False):
        club = EP[(EP.team == e.team) & (EP.fecha < e.ini)].sort_values('fecha')
        ven = club.tail(VENTANA)
        prev = ET[(ET.team == e.team) & (ET.fin < e.ini)].sort_values('ini')
        ant = prev.iloc[-1] if len(prev) else None
        ant_largo = prev[(prev.partidos > 3) & (prev.rol != 'interino')]
        ant_largo = ant_largo.iloc[-1] if len(ant_largo) else None
        d = EP[EP.etapa_id == e.etapa_id]
        filas.append({
            'técnico': nombre, 'etapa_id': e.etapa_id, 'partidos': e.partidos, 'rol': e.rol,
            'del': e.ini.date(), 'al': e.fin.date(),
            'llega con datos previos del club': 'sí' if e.ini > PRIMERA[e.team] else 'NO (empieza con los datos)',
            'antecesor inmediato': '—' if ant is None else f'{ant.tecnico} ({ant.partidos}, {ant.rol})',
            'último antecesor no interino': '—' if ant_largo is None else f'{ant_largo.tecnico} ({ant_largo.partidos})',
            'partidos del club en la ventana': len(ven),
            'ventana válida (≥ 10)': len(ven) >= VENTANA_MIN,
            'técnicos en la ventana': ', '.join(f'{t} ({n})' for t, n in ven.tec_p.value_counts().items()) or '—',
            'Elo medio etapa': round(d.elo_pre.mean(), 0),
            'percentil Elo (etapas 15+)': round(100 * (ref15 < d.elo_pre.mean()).mean(), 0),
            'Elo medio ventana del club': round(ven.elo_pre.mean(), 0) if len(ven) else np.nan,
            'puntos por partido': round(d.pts.mean(), 2),
            'dif. goles por partido': round((d.gf - d.gc).mean(), 2),
            'perfil en fw5': e.etapa_id in perf_ok, 'en el mapa (15+)': e.partidos >= ETAPA_MIN})
G = pd.DataFrame(filas)
print('\n--- (2) Cadena de relevos: quién estaba antes y si la ventana del club cabe en los datos ---')
print(G[['técnico', 'etapa_id', 'partidos', 'rol', 'del', 'al', 'llega con datos previos del club', 'antecesor inmediato',
         'último antecesor no interino', 'partidos del club en la ventana', 'ventana válida (≥ 10)']].to_string(index=False))
print('\nTécnicos que componen la ventana del club (últimos 18 partidos antes de llegar):')
print(G[['etapa_id', 'técnicos en la ventana']].to_string(index=False))
print('\n--- (3) Nivel y resultados (CONTEXTO, no efecto del técnico) ---')
print(G[['técnico', 'etapa_id', 'Elo medio etapa', 'percentil Elo (etapas 15+)', 'Elo medio ventana del club',
         'puntos por partido', 'dif. goles por partido']].to_string(index=False))
print(f'Referencia: Elo medio de las {len(ref15)} etapas de 15+: mediana {ref15.median():.0f} · p10 {ref15.quantile(.1):.0f} · p90 {ref15.quantile(.9):.0f}')
print('\n--- (4) ¿Tiene perfil en fw5 y entra al mapa? ---')
print(G[['etapa_id', 'perfil en fw5', 'en el mapa (15+)']].to_string(index=False))
print('\nG-0 terminó (no escribió nada). Pega la salida completa en el chat del reporte.')


In [ ]:
# [G-1 v1.0] Catálogo de rasgos para elegir la ficha del retrato (SOLO LECTURA: no escribe nada)
# =====================================================================
# Sesión 14 · 2026-10-20 · Chat del reporte · Insumo para fijar la ficha por componente (S14-01, S14-02)
# Junta en una sola tabla, por rasgo, lo que ya está calculado en salidas/fw5 (nada se recalcula):
#   confiabilidad y reloj (R5 / R6r-2: *_confiabilidad.csv) · regla C de implantación (*_implantacion.csv) ·
#   dueño en el mapa de dos ejes: cuadrante, firmeza, eje técnico, eje club y sus versiones sin Elo (C-1: c_mapa.csv;
#   para r9, C-1b: c1b_mapa.csv).
# Rasgos con confiabilidad < 0.5 no están en el mapa: se listan aparte (solo podrían mostrarse como "no se puede leer").
# Orden: celda 1 (Drive) → G-1. Tarda segundos. Pega la salida completa en el chat del reporte.
# =====================================================================
import os
import numpy as np
import pandas as pd

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
PREFIJOS = ['r4', 'r6', 'r6b', 'r6c', 'r7u', 'r7b', 'r7c', 'r8', 'r9']
sal = lambda p: 'r5' if p == 'r4' else p
pd.set_option('display.width', 250); pd.set_option('display.max_rows', 300); pd.set_option('display.max_colwidth', 40)

# ---------- Confiabilidad, reloj y regla C ----------
CF, IM = [], []
for p in PREFIJOS:
    rc, ri = os.path.join(SAL5, f'{sal(p)}_confiabilidad.csv'), os.path.join(SAL5, f'{sal(p)}_implantacion.csv')
    if not os.path.exists(rc):
        print(f'  AVISO: falta {os.path.basename(rc)}'); continue
    c = pd.read_csv(rc); c['rasgo'] = p + ':' + c['metrica']; c['prefijo'] = p
    CF.append(c[['rasgo', 'prefijo', 'conf'] + [k for k in ['reloj'] if k in c.columns]])
    if os.path.exists(ri):
        im = pd.read_csv(ri)
        rC = im.groupby('metrica').apply(lambda g: bool(((g.p_valor < 0.05).sum() >= 2) and (g.cociente > 1).all()))
        IM.append(pd.DataFrame({'rasgo': p + ':' + rC.index, 'regla_C': rC.values}))
CF = pd.concat(CF, ignore_index=True)
IM = pd.concat(IM, ignore_index=True) if IM else pd.DataFrame(columns=['rasgo', 'regla_C'])

# ---------- Dueño (mapa de dos ejes) ----------
cols = ['cuadrante', 'firmeza', 'tec', 'club', 'tec_sin_elo', 'club_sin_elo']
M = pd.read_csv(os.path.join(SAL5, 'c_mapa.csv'), index_col=0)
rb = os.path.join(SAL5, 'c1b_mapa.csv')
if os.path.exists(rb):
    Mb = pd.read_csv(rb, index_col=0)
    M = pd.concat([M, Mb[Mb.index.str.startswith('r9:')]])
    print(f'Mapa: c_mapa.csv ({(~M.index.str.startswith("r9:")).sum()} rasgos) + r9 de c1b_mapa.csv ({M.index.str.startswith("r9:").sum()})')
else:
    print('AVISO: falta c1b_mapa.csv; r9 queda sin dueño en este catálogo')
M = M[[c for c in cols if c in M.columns]]

T = CF.merge(IM, on='rasgo', how='left').merge(M, left_on='rasgo', right_index=True, how='left')
T['regla_C'] = T['regla_C'].map({True: 'sí', False: 'no'}).fillna('—')
num = [c for c in ['conf', 'reloj', 'tec', 'club', 'tec_sin_elo', 'club_sin_elo'] if c in T.columns]
T[num] = T[num].astype(float).round(2)
if 'reloj' in T.columns: T['reloj'] = T['reloj'].round(0)

print(f'\nRasgos: {len(T)} · con confiabilidad ≥ 0.5: {(T.conf >= 0.5).sum()} · en el mapa: {T.cuadrante.notna().sum()} '
      f'(deben ser 65 de C-1 + 11 de r9 = 76)')
print('Chequeo: rasgos del mapa sin confiabilidad (debe ser 0):', len(set(M.index) - set(CF.rasgo)))

print('\n--- (1) Rasgos que se pueden leer (confiabilidad ≥ 0.5), por prefijo ---')
print('    reloj = partidos para leer el rasgo con confiabilidad 0.7 · regla_C = el técnico nuevo lo instala al llegar')
for p, g in T[T.conf >= 0.5].groupby('prefijo', sort=False):
    print(f'\n[{p}]')
    print(g.drop(columns='prefijo').sort_values('conf', ascending=False).to_string(index=False))

print('\n--- (2) Rasgos que NO se pueden leer (confiabilidad < 0.5) ---')
print(T[T.conf < 0.5][['rasgo', 'conf']].sort_values('conf', ascending=False).to_string(index=False))

print('\n--- (3) Resumen por dueño (rasgos legibles) ---')
print(pd.crosstab(T.loc[T.conf >= 0.5, 'cuadrante'].fillna('fuera del mapa'), T.loc[T.conf >= 0.5, 'firmeza'].fillna('—')).to_string())
print('\nG-1 terminó (no escribió nada).')

In [ ]:
# [G-2 v1.0] Sensibilidad del mapa de dos ejes: "sin Elo" con el NIVEL HEREDADO (no circular) · solo lectura
# =====================================================================
# Sesión 14 · 2026-10-20 · Chat del reporte · Decisión S14-11 (Arturo aprueba 4a)
# Objeción que responde: el "sin Elo" de C-1 quita a cada perfil lo que explica el Elo MEDIO DE LA ETAPA, y ese Elo sale de los
#   resultados del propio técnico: podría estar quitando parte del efecto del técnico (sobrecontrol).
# Qué hace (copia literal de C-1b, secciones 0-2, sin compatibilidad ni escritura):
#   (A) REPRODUCCIÓN: recalcula el "sin Elo" con el Elo medio de la etapa y lo compara con c1b_mapa.csv (diferencia < 1e-6).
#   (B) H2 (principal): quita a cada perfil lo que explica el NIVEL HEREDADO = Elo medio del club en sus últimos 18 partidos
#       antes de que llegara el técnico (mínimo 10; misma ventana que S12-05). Etapas sin ventana → fuera de esta versión.
#   (C) H1 (sensibilidad): Elo del club en el primer partido de la etapa (cubre todas las etapas).
#   (D) Resumen por dueño (etiquetas de C-1/C-1b): eje técnico de lo que viaja y eje club de lo que depende de la plantilla,
#       crudo · sin Elo de la etapa · sin nivel heredado (H2, H1); y el detalle de los rasgos de la ficha, con IC para H2.
# Criterio (propuesto por Claude, aprobado por Arturo ANTES de correr; ver decisiones_entrega_s14.md):
#   "se sostiene" si con H2 (i) el eje técnico medio de lo que viaja queda ≥ 0.6 y (ii) el eje club medio de lo que depende de la
#   plantilla cae al menos un tercio frente al crudo. (i) sí y (ii) no → lo que viaja es robusto, pero que "plantilla = nivel" se
#   apoya en el Elo de la etapa y en L1, no en el nivel heredado (se redacta así).
# Orden: celda 1 → C-0 (todas las pruebas PASAN) → G-2. Tarda ~2-5 min. No escribe nada.
# =====================================================================
import os, json, time
import numpy as np
import pandas as pd

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
assert globals().get('C0_OK', False), 'Corre primero C-0 (claude/codigo/sesion12/C_0_prueba.py) y confirma que todas sus pruebas PASAN'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5 = os.path.join(DP, 'fw5'); FW1 = os.path.join(DP, 'fw1'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
PREFIJOS = ['r4', 'r6', 'r6b', 'r6c', 'r7u', 'r7b', 'r7c', 'r8', 'r9']
ETAPA_MIN, VENTANA, VENT_MIN, CONF_MIN = 15, 18, 10, 0.5
B, PLAC = 500, 100
pd.set_option('display.width', 250); pd.set_option('display.max_rows', 120)
t0 = time.time()
sal = lambda p: 'r5' if p == 'r4' else p

# ---------- 0. Perfiles y confiabilidad (copia de C-1b, sin residuos: no se calcula compatibilidad) ----------
CONT, CF = [], []
for p in PREFIJOS:
    c = pd.read_parquet(os.path.join(FW5, f'{sal(p)}_perfiles_contraidos.parquet'))
    f = pd.read_csv(os.path.join(SAL5, f'{sal(p)}_confiabilidad.csv'))
    for k, cols in [('contraidos', ['metrica', 'etapa_id', 'perfil_r4']), ('confiabilidad', ['metrica', 'conf', 'tau'])]:
        d = c if k == 'contraidos' else f
        assert all(x in d.columns for x in cols), f'Faltan columnas en {p} {k}: avisar a Claude'
    CONT.append(c.assign(rasgo=lambda d: p + ':' + d.metrica)); CF.append(f.assign(rasgo=lambda d: p + ':' + d.metrica, prefijo=p))
CONT = pd.concat(CONT, ignore_index=True); CF = pd.concat(CF, ignore_index=True).set_index('rasgo')
RASGOS = [r for r in CF.index if CF.loc[r, 'conf'] >= CONF_MIN and CF.loc[r, 'tau'] > 0]

with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
    TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy(); ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ET = ptp.groupby('etapa_id').agg(team=('team', 'first'), tecnico=('tec_p', 'first'), partidos=('match_id', 'nunique'),
                                 ini=('fecha', 'min'), fin=('fecha', 'max')).reset_index()
ETAPAS = ET.etapa_id.tolist(); IX = {e: i for i, e in enumerate(ETAPAS)}
tau = CF.loc[RASGOS, 'tau'].values.astype(float); conf = CF.loc[RASGOS, 'conf'].values.astype(float)
ZR = (CONT.pivot_table(index='etapa_id', columns='rasgo', values='perfil_r4').reindex(index=ETAPAS, columns=RASGOS).astype(float) / tau).values
print(f'Rasgos en el mapa: {len(RASGOS)} (deben ser 76) · etapas: {len(ETAPAS)}')

# ---------- 1. Parejas (copia literal de C-1b) ----------
L = ET[ET.partidos >= ETAPA_MIN]
pt, ct = [], []
for tec, d in L.groupby('tecnico'):
    ix = [IX[e] for e in d.etapa_id]
    for i in range(len(ix)):
        for j in range(i + 1, len(ix)):
            if ET.team[ix[i]] != ET.team[ix[j]]: pt.append((ix[i], ix[j])); ct.append(tec)
pc, cc = [], []
for team, d in L.sort_values('ini').groupby('team'):
    ix = [IX[e] for e in d.etapa_id]
    for i in range(len(ix) - 1):
        if ET.tecnico[ix[i]] != ET.tecnico[ix[i + 1]]: pc.append((ix[i], ix[i + 1])); cc.append(team)
li = [IX[e] for e in L.etapa_id]
pool = [(a, b) for k, a in enumerate(li) for b in li[k + 1:] if ET.tecnico[a] != ET.tecnico[b] and ET.team[a] != ET.team[b]]
pt, ct, pc, cc, pool = np.array(pt), np.array(ct), np.array(pc), np.array(cc), np.array(pool)
print(f'Parejas: viajeros {len(pt)} de {len(set(ct))} técnicos (deben ser 29 de 20) · relevos {len(pc)} en {len(set(cc))} clubes (69 en 18)')

# ---------- 2. Tres medidas de nivel por etapa ----------
elo = pd.read_parquet(os.path.join(FW1, 'elo_equipo_partido_e2.parquet'), columns=['match_id', 'team', 'elo_pre'])
EP = ptp.merge(elo, on=['match_id', 'team'], how='left')
nivel_etapa = EP.groupby('etapa_id')['elo_pre'].mean().reindex(ETAPAS).values                    # el de C-1 (circular)
h1 = EP.sort_values('fecha').groupby('etapa_id')['elo_pre'].first().reindex(ETAPAS).values       # Elo al llegar
h2 = np.full(len(ETAPAS), np.nan)                                                                  # nivel heredado (ventana de 18)
for i, r_ in enumerate(ET.itertuples(index=False)):
    v = EP[(EP.team == r_.team) & (EP.fecha < r_.ini)].sort_values('fecha').tail(VENTANA)
    if len(v) >= VENT_MIN: h2[i] = v.elo_pre.mean()
en15 = np.isin(np.arange(len(ETAPAS)), li)
print(f'Etapas de 15+ con nivel heredado (H2): {int(np.isfinite(h2[en15]).sum())} de {int(en15.sum())} · '
      f'correlación entre medidas (etapas 15+): etapa–H2 {np.corrcoef(nivel_etapa[en15 & np.isfinite(h2)], h2[en15 & np.isfinite(h2)])[0, 1]:.2f} · '
      f'etapa–H1 {np.corrcoef(nivel_etapa[en15], h1[en15])[0, 1]:.2f}')

def quitar(nivel, nan_fuera):
    """Copia del bloque 'sin Elo' de C-1: a cada rasgo le quita la recta contra el nivel. Si nan_fuera, las etapas sin nivel
    quedan fuera (NaN) en vez de conservar su perfil crudo."""
    Z = ZR.copy() if not nan_fuera else np.where(np.isfinite(nivel)[:, None], ZR, np.nan)
    for j in range(ZR.shape[1]):
        m = np.isfinite(ZR[:, j]) & np.isfinite(nivel)
        if m.sum() > 10:
            b = np.polyfit(nivel[m], ZR[m, j], 1); Z[m, j] = ZR[m, j] - np.polyval(b, nivel[m])
    return Z
cdiv = np.clip(conf, .05, None)
def ejes_pt(Z): return r_pares(Z, pt) / cdiv, r_pares(Z, pc) / cdiv

# ---------- (A) Reproducción del "sin Elo" de C-1b ----------
ref = pd.read_csv(os.path.join(SAL5, 'c1b_mapa.csv')).set_index('rasgo').reindex(RASGOS)
te, ce = ejes_pt(quitar(nivel_etapa, False))
dif = np.nanmax(np.abs(np.r_[te - ref.tec_sin_elo.values, ce - ref.club_sin_elo.values]))
tr, cr = ejes_pt(ZR)
dif0 = np.nanmax(np.abs(np.r_[tr - ref.tec.values, cr - ref.club.values]))
print(f'\n--- (A) Reproducción · ejes crudos frente a c1b_mapa: {dif0:.2e} · "sin Elo" frente a c1b_mapa: {dif:.2e} (ambos deben ser < 1e-6) ---')
assert dif0 < 1e-6 and dif < 1e-6, 'No reproduce C-1b: no se interpreta nada. Manda esta salida a Claude.'

# ---------- (B) y (C) Sin nivel heredado ----------
t2, c2 = ejes_pt(quitar(h2, True)); t1, c1 = ejes_pt(quitar(h1, False))
dueño = ref.cuadrante.map({'del técnico': 'viaja con el técnico', 'del club': 'depende de la plantilla',
                           'de nadie': 'depende de la mezcla', 'de los dos': 'de los dos'}).values
R = pd.DataFrame({'dueño': dueño, 'firmeza': ref.firmeza.values, 'conf': conf, 'tec_crudo': tr, 'tec_sin_elo_etapa': te,
                  'tec_sin_heredado': t2, 'tec_sin_elo_llegar': t1, 'club_crudo': cr, 'club_sin_elo_etapa': ce,
                  'club_sin_heredado': c2, 'club_sin_elo_llegar': c1}, index=pd.Index(RASGOS, name='rasgo'))

print('\n--- (D) Resumen por dueño (media de los rasgos) ---')
print('    tec = viaja con el técnico (parejas de viajeros) · club = se queda con el club (relevos) · ambos ÷ confiabilidad')
S = R.groupby('dueño')[[c for c in R.columns if c.startswith(('tec_', 'club_'))]].mean().round(2)
S.insert(0, 'rasgos', R.groupby('dueño').size())
print(S.T.to_string())
v, p_ = R[R.dueño == 'viaja con el técnico'], R[R.dueño == 'depende de la plantilla']
i_ok = v.tec_sin_heredado.mean() >= 0.6
ii_ok = p_.club_sin_heredado.mean() <= p_.club_crudo.mean() * 2 / 3
print(f'\nCriterio (H2): (i) eje técnico de lo que viaja {v.tec_sin_heredado.mean():.2f} (≥ 0.60) → {"CUMPLE" if i_ok else "NO CUMPLE"} · '
      f'(ii) eje club de lo de la plantilla {p_.club_crudo.mean():.2f} → {p_.club_sin_heredado.mean():.2f} '
      f'(debe quedar ≤ {p_.club_crudo.mean() * 2 / 3:.2f}) → {"CUMPLE" if ii_ok else "NO CUMPLE"}')
print('Lectura preescrita:', 'SE SOSTIENE: lo que viaja no depende del nivel, y lo de la plantilla es sobre todo nivel heredado.' if i_ok and ii_ok else
      'LO QUE VIAJA ES ROBUSTO; "plantilla = nivel" se apoya en el Elo de la etapa y en L1, no en el nivel heredado.' if i_ok else
      'NO SE SOSTIENE con el nivel heredado: revisar la redacción de la tesis con Claude antes de seguir.')

# ---------- IC con H2 para los rasgos de la ficha (remuestreo por técnico / por club, como C-1) ----------
ZH2 = quitar(h2, True)
TH = ejes(ZH2, conf, pt, ct, pc, cc, pool, B=B, P=PLAC).set_index(pd.Index(RASGOS, name='rasgo'))
FICHA = ['r4:posesion', 'r4:field_tilt', 'r4:presion_alta', 'r4:intensidad_ppda', 'r6:paso_ultimo', 'r6:toque_area', 'r6:seg_hasta_ultimo',
         'r6:pases_hasta_ultimo', 'r6c:obv_transicion_pct', 'r7u:inicio_linea_3_5', 'r7u:cambio_familia_90', 'r7c:altura_laterales',
         'r7c:libertad_posicional', 'r7c:portero_libero', 'r8:zonales', 'r8:pct_marca', 'r8:def_area_360', 'r8:pegados_portero',
         'r8:envio_segundo_palo', 'r9:espacio_conc', 'r9:rompe_lineas_conc', 'r9:dist_defensor', 'r9:adelante_contra', 'r9:altura_presion']
FICHA = [f for f in FICHA if f in TH.index]
D = R.loc[FICHA, ['dueño', 'firmeza', 'tec_crudo', 'tec_sin_elo_etapa', 'club_crudo', 'club_sin_elo_etapa']].join(
    TH.loc[FICHA, ['tec', 'tec_lo', 'tec_hi', 'club', 'club_lo', 'club_hi']].add_suffix('_H2'))
print(f'\n--- Detalle: rasgos de la ficha (H2 con IC 95%, {B} remuestreos) ---')
print(D.round(2).to_string())
print(f'\nPlacebo con H2 (debe ser ≈ 0): media {TH.placebo_media.mean():+.3f} · ruido de un rasgo sin efecto ±{2 * TH.placebo_de.median():.2f}')
print(f'\nG-2 terminó en {time.time() - t0:.0f} s (no escribió nada).')

In [ ]:
# [G-3 v1.1] Exportación de AGREGADOS para el reporte HTML → salidas/fw5/reporte_datos.json (no toca nada más)
# =====================================================================
# Sesión 14 · 2026-10-20 · Chat del reporte · Decisiones S14-01 a S14-16 · v1.1: agrega la tabla de arquetipos (R7c-9)
# Regla de datos del reto: solo agregados (perfiles de etapa, tablas de liga, resúmenes). Freno: cualquier tabla con columna
#   match_id, index o id de evento se descarta y se avisa. Nada por partido sale de Drive.
# Qué exporta:
#   (1) rasgos: confiabilidad, τ, σ por partido, reloj, regla C, dueño (C-1 / C-1b) con ejes, IC y sin Elo; media de la liga
#       en unidades naturales cuando el archivo de tramos lo permite.
#   (2) etapas: las 9 del selector (Jardine, Larcamón, Ortiz), América de Solari, y las 3 de referencia de E1 (anónimas en el
#       HTML): perfil contraído en unidades naturales, banda de 95% y partidos con dato, por rasgo.
#   (3) cadena de relevos: antecesor, ventana del club, nivel heredado / al llegar / final (Elo), puntos y diferencia de goles,
#       xG a favor y en contra por partido (contexto, no rasgo); perfil del club antes de llegar (zc), del técnico en su etapa
#       previa (zt) y de la etapa (zn), por rasgo (copia literal de perfil_club de C-1b).
#   (4) tablas ya calculadas (familias, validación, duración, L1, K1, R8h, M1, B7, Almada en vivo...): se copian tal cual.
# Orden: celda 1 → C-0 (todas las pruebas PASAN) → G-3. Tarda ~5-10 min (lee los residuos, como C-1b).
# Después: descargar salidas/fw5/reporte_datos.json de Drive y subirlo al chat del reporte junto con la salida de esta celda.
# =====================================================================
import os, json, glob, time
import numpy as np
import pandas as pd

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
assert globals().get('C0_OK', False), 'Corre primero C-0 (claude/codigo/sesion12/C_0_prueba.py) y confirma que todas sus pruebas PASAN'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5, FW1 = os.path.join(DP, 'fw5'), os.path.join(DP, 'fw1')
SAL5, SAL6 = os.path.join(RAIZ, 'salidas', 'fw5'), os.path.join(RAIZ, 'salidas', 'fw6')
PREFIJOS = ['r4', 'r6', 'r6b', 'r6c', 'r7u', 'r7b', 'r7c', 'r8', 'r9']
ETAPA_MIN, VENTANA, VENT_MIN, CONF_MIN = 15, 18, 10, 0.5
SELECTOR = {'Jardine': 'Jardine', 'Larcamón': 'Larcam', 'Ortiz': 'Fernando Ortiz'}
E1_REF = {'A': 'Reynoso', 'B': 'Ferretti', 'C': 'Almada'}            # perfiles de E1: en el HTML NO se nombran (salvo C al final)
PROHIBIDAS = {'match_id', 'index', 'id', 'event_id', 'player_id'}
sal = lambda p: 'r5' if p == 'r4' else p
t0 = time.time(); avisos = []
pd.set_option('display.width', 250); pd.set_option('display.max_rows', 120)

def limpio(v):
    if isinstance(v, (np.floating, float)): return None if not np.isfinite(v) else round(float(v), 4)
    if isinstance(v, (np.integer,)): return int(v)
    if isinstance(v, (np.bool_,)): return bool(v)
    if isinstance(v, (pd.Timestamp,)): return str(v.date())
    return v
def registros(df):
    return [{k: limpio(v) for k, v in r.items()} for r in df.to_dict('records')]

# ---------- (1) Rasgos ----------
CONT, CF, TAUE, RES, S2, TRM = [], [], {}, {}, {}, {}
for p in PREFIJOS:
    c = pd.read_parquet(os.path.join(FW5, f'{sal(p)}_perfiles_contraidos.parquet'))
    f = pd.read_csv(os.path.join(SAL5, f'{sal(p)}_confiabilidad.csv'))
    pe = pd.read_parquet(os.path.join(FW5, f'{p}_perfiles_etapa.parquet'))
    rs = pd.read_parquet(os.path.join(FW5, f'{p}_residuos.parquet'), columns=['metrica', 'match_id', 'team', 'e', 'den', 'res'])
    CONT.append(c.assign(rasgo=lambda d: p + ':' + d.metrica)); CF.append(f.assign(rasgo=lambda d: p + ':' + d.metrica, prefijo=p))
    TAUE.update({p + ':' + m: v for m, v in pe.groupby('metrica')['tau_estado'].first().items()})
    S2.update({p + ':' + m: float((g.den * g.res ** 2).sum() / (len(g) - 19)) for m, g in rs.groupby('metrica')})   # copia de C-1b
    RES[p] = {t: g for t, g in rs.groupby('team')}
    # media de la liga en unidades naturales: tramos con num/den si existen (si no, queda vacía y se avisa)
    rt = os.path.join(FW5, f'{p}_tramos.parquet')
    if os.path.exists(rt):
        tr = pd.read_parquet(rt)
        if {'metrica', 'num', 'den'} <= set(tr.columns):
            g = tr.groupby('metrica')[['num', 'den']].sum(); TRM.update({p + ':' + m: float(v) for m, v in (g.num / g.den).items()})
        else: avisos.append(f'{p}_tramos sin num/den (columnas: {list(tr.columns)[:12]})')
    else: avisos.append(f'falta {p}_tramos.parquet en fw5')
CONT = pd.concat(CONT, ignore_index=True); CF = pd.concat(CF, ignore_index=True).set_index('rasgo')
for c_ in ['conf', 'tau', 'sigma_partido']: CF[c_] = CF[c_].astype(float)
RASGOS = [r for r in CF.index if CF.loc[r, 'conf'] >= CONF_MIN and CF.loc[r, 'tau'] > 0]
TODOS = CF.index.tolist()

im = []
for p in PREFIJOS:
    ri = os.path.join(SAL5, f'{sal(p)}_implantacion.csv')
    if os.path.exists(ri):
        d = pd.read_csv(ri)
        for m, g in d.groupby('metrica'): im.append((p + ':' + m, bool(((g.p_valor < 0.05).sum() >= 2) and (g.cociente > 1).all())))
IMP = dict(im)
M1 = pd.read_csv(os.path.join(SAL5, 'c_mapa.csv')).set_index('rasgo')
Mb = pd.read_csv(os.path.join(SAL5, 'c1b_mapa.csv')).set_index('rasgo')
MAPA = pd.concat([M1, Mb[Mb.index.str.startswith('r9:')]])
cols_mapa = [c for c in ['tec', 'tec_lo', 'tec_hi', 'club', 'club_lo', 'club_hi', 'dif_lo', 'dif_hi', 'cuadrante', 'firmeza',
                         'tec_sin_elo', 'club_sin_elo', 'corr_elo', 'placebo_de'] if c in MAPA.columns]
rasgos_out = {}
for r in TODOS:
    d = {'prefijo': r.split(':')[0], 'metrica': r.split(':', 1)[1], 'conf': CF.loc[r, 'conf'], 'tau': CF.loc[r, 'tau'],
         'sigma_partido': CF.loc[r, 'sigma_partido'], 'reloj': CF.loc[r, 'reloj'] if 'reloj' in CF.columns else None,
         'regla_C': IMP.get(r), 'liga': TRM.get(r), 'en_mapa': r in MAPA.index}
    if r in MAPA.index: d.update({c: MAPA.loc[r, c] for c in cols_mapa})
    rasgos_out[r] = {k: limpio(v) for k, v in d.items()}
print(f'(1) Rasgos: {len(TODOS)} · legibles (≥ {CONF_MIN}) {len(RASGOS)} · en el mapa {int(sum(v["en_mapa"] for v in rasgos_out.values()))} (76) · '
      f'con media de liga {sum(v["liga"] is not None for v in rasgos_out.values())}')

# ---------- Etapas (copia literal de C-00 / C-1b) ----------
with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
    TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy(); ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ET = ptp.groupby('etapa_id').agg(team=('team', 'first'), tecnico=('tec_p', 'first'), partidos=('match_id', 'nunique'),
                                 ini=('fecha', 'min'), fin=('fecha', 'max'), rol=('rol', lambda s: s.mode().iat[0])).reset_index()
ETI = ET.set_index('etapa_id')
etapa_de = ptp.set_index(['match_id', 'team'])['etapa_id']
tau_v = CF.loc[RASGOS, 'tau'].values

# selección de etapas
SEL = {}
for nombre, patron in SELECTOR.items():
    for e in ET[ET.tecnico.astype(str).str.contains(patron, case=False)].sort_values('ini').itertuples(index=False):
        SEL[e.etapa_id] = nombre
am = ET[ET.team.str.startswith('Am')].sort_values('ini')
solari = am[am.tecnico.str.contains('Solari', case=False)]
EXTRA = {e: 'América (contexto)' for e in solari.etapa_id}
rk = pd.read_csv(os.path.join(SAL5, 'c_ranking_america.csv'))
E1 = {}
for k, patron in E1_REF.items():
    h = rk[rk.tecnico.astype(str).str.contains(patron, case=False)]
    if len(h): E1[h.etapa.iloc[0]] = k
    else: avisos.append(f'E1: no se encontró {patron} en c_ranking_america')
print(f'(2) Etapas del selector: {len(SEL)} (deben ser 9) → {list(SEL)}')
print(f'    Contexto: {list(EXTRA)} · referencias de E1: {E1}')

# partidos con dato por rasgo y etapa (desde los residuos: mismo conteo que usa la contracción)
def n_por_etapa(etapas):
    out = {}
    for r in RASGOS:
        p, m = r.split(':', 1)
        for e in etapas:
            g = RES[p].get(ETI.loc[e, 'team'])
            if g is None: out[(r, e)] = 0; continue
            ids = set(ptp.loc[ptp.etapa_id == e, 'match_id'])
            out[(r, e)] = int(g.loc[(g.metrica == m) & g.match_id.isin(ids), 'match_id'].nunique())
    return out
ETAPAS_OUT = list(SEL) + list(EXTRA) + list(E1)
NN = n_por_etapa(ETAPAS_OUT)
CONTp = CONT.set_index(['rasgo', 'etapa_id'])
def perfil_etapa(e):
    filas = {}
    for r in RASGOS:
        if (r, e) not in CONTp.index: continue
        x = CONTp.loc[(r, e)]
        n = NN.get((r, e), 0); tau_, sg = CF.loc[r, 'tau'], CF.loc[r, 'sigma_partido']
        k = tau_ ** 2 / (tau_ ** 2 + sg ** 2 / n) if n > 0 else np.nan
        sd = np.sqrt(k) * sg / np.sqrt(n) if n > 0 else np.nan              # sd posterior del perfil contraído
        filas[r] = {k_: limpio(v) for k_, v in {'perfil': x['perfil_mostrado'], 'perfil_sin_contraer': x['perfil_r4'],
                                                'lo': x['perfil_mostrado'] - 1.96 * sd, 'hi': x['perfil_mostrado'] + 1.96 * sd,
                                                'z': x['perfil_mostrado'] / tau_, 'n': n}.items()}
    return filas

# ---------- (3) Cadena de relevos y contexto ----------
elo = pd.read_parquet(os.path.join(FW1, 'elo_equipo_partido_e2.parquet'), columns=['match_id', 'team', 'elo_pre', 'elo_post'])
par = pd.read_parquet(os.path.join(FW1, 'elo_partido_e2.parquet'), columns=['match_id', 'local', 'visita', 'gl', 'gv'])
xg = pd.read_parquet(os.path.join(FW1, 'elo_xg_partido.parquet'), columns=['match_id', 'xg_local', 'xg_visita'])
pr = par.merge(xg, on='match_id', how='left')
res = pd.concat([pr.assign(team=pr.local, gf=pr.gl, gc=pr.gv, xgf=pr.xg_local, xgc=pr.xg_visita),
                 pr.assign(team=pr.visita, gf=pr.gv, gc=pr.gl, xgf=pr.xg_visita, xgc=pr.xg_local)])
res['pts'] = np.select([res.gf > res.gc, res.gf == res.gc], [3, 1], 0)
EP = ptp[['match_id', 'team', 'fecha', 'etapa_id', 'tec_p']].merge(elo, on=['match_id', 'team'], how='left') \
        .merge(res[['match_id', 'team', 'gf', 'gc', 'xgf', 'xgc', 'pts']], on=['match_id', 'team'], how='left')
print(f'(3) Cruce: equipo-partidos {len(EP)} · sin Elo {int(EP.elo_pre.isna().sum())} · sin marcador {int(EP.pts.isna().sum())} · '
      f'sin xG {int(EP.xgf.isna().sum())}')
ELO_ET = EP.groupby('etapa_id').elo_pre.mean(); ref15 = ELO_ET[ELO_ET.index.isin(ET.loc[ET.partidos >= ETAPA_MIN, 'etapa_id'])]
pct = lambda x: None if not np.isfinite(x) else round(100 * float((ref15 < x).mean()), 0)

# copia literal de C-1b (sección 3): perfil del club en una ventana de partidos
MIN_ = pd.read_parquet(os.path.join(FW5, 'r4_minutos_v21.parquet'))
W3 = MIN_.assign(e3=np.sign(MIN_.e5)).groupby('e3')['minutos'].sum(); W3v = (W3 / W3.sum()).reindex([-1, 0, 1]).fillna(0).values
def perfil_club(team, partidos, contraido=True):
    z = np.full(len(RASGOS), np.nan)
    for j, r in enumerate(RASGOS):
        p, m = r.split(':', 1); g = RES[p].get(team)
        if g is None: continue
        g = g[(g.metrica == m) & g.match_id.isin(partidos)]
        n = g.match_id.nunique()
        if n == 0: continue
        pf = perfil_ventana(g.e.values, g.den.values, g.res.values, S2[r], TAUE[r], W3v)
        z[j] = (contraer(pf, n, CF.loc[r, 'tau'], CF.loc[r, 'sigma_partido']) if contraido else pf) / CF.loc[r, 'tau']
    return z
ZM = CONT.pivot_table(index='etapa_id', columns='rasgo', values='perfil_mostrado').reindex(columns=RASGOS).astype(float) / tau_v
# coherencia (buena práctica 13): ventana = etapa completa sin contraer reproduce perfil_r4
ZR = CONT.pivot_table(index='etapa_id', columns='rasgo', values='perfil_r4').reindex(columns=RASGOS).astype(float) / tau_v
e_ = list(SEL)[0]
dmax = np.nanmax(np.abs(perfil_club(ETI.loc[e_, 'team'], set(ptp.loc[ptp.etapa_id == e_, 'match_id']), contraido=False) - ZR.loc[e_].values))
print(f'    Coherencia de perfil_club con C-1b (etapa completa {e_}): diferencia máxima {dmax:.2e} z (≈ 0; < 0.01 aceptable)')
if dmax > 0.01: avisos.append(f'perfil_club no reproduce perfil_r4 ({dmax:.3f} z): revisar antes de usar zc')

cadena = {}
for e in list(SEL) + list(EXTRA):
    r_ = ETI.loc[e]; d = EP[EP.etapa_id == e].sort_values('fecha')
    ven = EP[(EP.team == r_.team) & (EP.fecha < r_.ini)].sort_values('fecha').tail(VENTANA)
    prev = ET[(ET.team == r_.team) & (ET.fin < r_.ini)].sort_values('ini')
    ant = prev.iloc[-1] if len(prev) else None
    previa = ET[(ET.tecnico == r_.tecnico) & (ET.team != r_.team) & (ET.fin < r_.ini) & (ET.partidos >= ETAPA_MIN)].sort_values('fin')
    siguiente = ET[(ET.tecnico == r_.tecnico) & (ET.team != r_.team) & (ET.ini > r_.fin)].sort_values('ini')
    ok_v = len(ven) >= VENT_MIN
    zc = perfil_club(r_.team, set(ven.match_id)) if ok_v else np.full(len(RASGOS), np.nan)
    zt = ZM.loc[previa.etapa_id.iloc[-1]].values if len(previa) else np.full(len(RASGOS), np.nan)
    zn = ZM.loc[e].values
    cadena[e] = {k: limpio(v) for k, v in {
        'tecnico_grupo': SEL.get(e, EXTRA.get(e)), 'tecnico': r_.tecnico, 'club': r_.team, 'partidos': r_.partidos, 'rol': r_.rol,
        'del': r_.ini, 'al': r_.fin, 'antecesor': None if ant is None else ant.tecnico, 'antecesor_partidos': None if ant is None else ant.partidos,
        'antecesor_rol': None if ant is None else ant.rol, 'ventana_partidos': len(ven), 'ventana_valida': ok_v,
        'ventana_tecnicos': ', '.join(f'{t} ({n})' for t, n in ven.tec_p.value_counts().items()),
        'etapa_previa': previa.etapa_id.iloc[-1] if len(previa) else None, 'etapa_siguiente': siguiente.etapa_id.iloc[0] if len(siguiente) else None,
        'elo_heredado': ven.elo_pre.mean() if ok_v else np.nan, 'elo_al_llegar': d.elo_pre.iloc[0], 'elo_final': d.elo_post.iloc[-1],
        'elo_medio_etapa': d.elo_pre.mean(), 'pct_heredado': pct(ven.elo_pre.mean()) if ok_v else None, 'pct_final': pct(d.elo_post.iloc[-1]),
        'pts_por_partido': d.pts.mean(), 'dg_por_partido': (d.gf - d.gc).mean(), 'xg_favor_pp': d.xgf.mean(), 'xg_contra_pp': d.xgc.mean(),
        'pts_ventana': ven.pts.mean() if ok_v else np.nan}.items()}
    cadena[e]['rasgos'] = {r: {'zc': limpio(zc[j]), 'zt': limpio(zt[j]), 'zn': limpio(zn[j])} for j, r in enumerate(RASGOS)}
ref_liga = {'elo_mediana_15': limpio(ref15.median()), 'elo_p10_15': limpio(ref15.quantile(.1)), 'elo_p90_15': limpio(ref15.quantile(.9)),
            'xg_favor_pp_liga': limpio(EP.xgf.mean()), 'pts_pp_liga': limpio(EP.pts.mean()), 'etapas_15': int(len(ref15))}

etapas_out = {}
for e in ETAPAS_OUT:
    etapas_out[e] = {'grupo': SEL.get(e) or EXTRA.get(e) or f'E1-{E1[e]}', 'partidos': int(ETI.loc[e, 'partidos']),
                     'del': str(ETI.loc[e, 'ini'].date()), 'al': str(ETI.loc[e, 'fin'].date()), 'rasgos': perfil_etapa(e)}
    if e in E1: etapas_out[e].update({'club': None, 'tecnico': None})          # anónimas en el archivo (E1 a ciegas)
    else: etapas_out[e].update({'club': ETI.loc[e, 'team'], 'tecnico': ETI.loc[e, 'tecnico']})
E1_out = {E1[e]: {'etapa_key': e} for e in E1}
for k_ in E1_out:
    h = rk[rk.etapa == E1_out[k_]['etapa_key']]
    E1_out[k_].update({c: limpio(h[c].iloc[0]) for c in ['compat', 'distancia', 'choques'] if c in h.columns})

# ---------- (4) Tablas ya calculadas (copia tal cual, con freno de datos) ----------
TABLAS = {'familias': 'f_familias*', 'mapa_familias': 'f_mapa_familias*', 'familias_sin_elo': 'a2_mapa_familias_sin_elo*',
          'llegadas': 'c_llegadas*', 'ranking_america': 'c_ranking_america*', 'ensayo': 'c_ensayo_*', 'preregistro': 'c_preregistro_almada*',
          'duracion': 'd1_duracion_etapas*', 'duracion_nac': 'd1b_duracion_nacionalidad*', 'l1_alineacion': 'l1_alineacion*',
          'l1_relevos': 'l1_relevos*', 'concordancia': 'k_concordancia*', 'concordancia_tasas': 'k1b_concordancia_tasas*',
          'aereo': 'r8h_resultados*', 'disputa_liga': 'm1_mapa_liga*', 'disputa_etapas': 'm1_etapas*', 'disputa_america': 'm1_america*',
          'jardine_salida': 'b7_jardine_bloques*', 'r9_coeficientes': 'r9_coeficientes*', 'r9_implantacion': 'r9_implantacion*',
          'r5_implantacion': 'r5_implantacion*', 'arquetipos': 'r7c_arquetipos*'}
tablas_out = {}
def cargar(ruta):
    return pd.read_csv(ruta) if ruta.endswith('.csv') else pd.read_parquet(ruta)
for k, patron in TABLAS.items():
    for ruta in sorted(glob.glob(os.path.join(SAL5, patron))):
        if not ruta.endswith(('.csv', '.parquet')): continue
        df = cargar(ruta)
        clave = k if len([x for x in glob.glob(os.path.join(SAL5, patron)) if x.endswith(('.csv', '.parquet'))]) == 1 else os.path.splitext(os.path.basename(ruta))[0]
        if PROHIBIDAS & {c.lower() for c in df.columns}: avisos.append(f'DESCARTADA por tener columnas por partido/evento: {os.path.basename(ruta)}'); continue
        if len(df) > 5000: avisos.append(f'DESCARTADA por tamaño ({len(df)} filas): {os.path.basename(ruta)}'); continue
        tablas_out[clave] = registros(df)
for ruta in sorted(glob.glob(os.path.join(SAL6, 'v1_*'))):                  # Almada en vivo (V-1)
    if not ruta.endswith(('.csv', '.parquet')): continue
    df = cargar(ruta)
    if PROHIBIDAS & {c.lower() for c in df.columns}: avisos.append(f'DESCARTADA (fw6): {os.path.basename(ruta)}'); continue
    if len(df) > 5000: avisos.append(f'DESCARTADA por tamaño (fw6): {os.path.basename(ruta)}'); continue
    tablas_out['almada_' + os.path.splitext(os.path.basename(ruta))[0]] = registros(df)
for k, patron in TABLAS.items():
    if not glob.glob(os.path.join(SAL5, patron)): avisos.append(f'no se encontró {patron} en salidas/fw5')

# ---------- Guardar ----------
OUT = {'version': 'G-3 v1.1', 'generado': time.strftime('%Y-%m-%d %H:%M'), 'temporadas': TEMPS, 'rasgos': rasgos_out,
       'etapas': etapas_out, 'cadena': cadena, 'referencia_liga': ref_liga, 'e1': E1_out, 'tablas': tablas_out}
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Freno: Drive no está montado; no se escribe nada'
ruta_out = os.path.join(SAL5, 'reporte_datos.json')
with open(ruta_out, 'w', encoding='utf-8') as f: json.dump(OUT, f, ensure_ascii=False)

# ---------- Chequeos para pegar en el chat ----------
print(f'\n--- Chequeos ---')
print(f'Archivo: {ruta_out} · {os.path.getsize(ruta_out) / 1e6:.2f} MB · tablas copiadas: {len(tablas_out)} → {sorted(tablas_out)}')
print('\nEscala (para fijar unidades): media de liga y dispersión entre técnicos (τ) de algunos rasgos')
muestra = ['r4:posesion', 'r4:field_tilt', 'r4:intensidad_ppda', 'r6:paso_ultimo', 'r6:seg_hasta_ultimo', 'r6:pases_hasta_ultimo',
           'r7c:libertad_posicional', 'r7u:cambio_familia_90', 'r8:zonales', 'r8:pct_marca', 'r8:def_area_360', 'r9:espacio_conc', 'r9:altura']
print(pd.DataFrame({r: {'liga': rasgos_out[r]['liga'], 'tau': rasgos_out[r]['tau'], 'sigma': rasgos_out[r]['sigma_partido'],
                         'reloj': rasgos_out[r]['reloj']} for r in muestra if r in rasgos_out}).T.round(3).to_string())
ej = next(e for e in SEL if 'Jardine' in str(ETI.loc[e, 'tecnico']) and ETI.loc[e, 'team'].startswith('Am'))
print(f'\nEjemplo · {ej}: perfil contraído [banda 95%] y partidos con dato')
print(pd.DataFrame({r: etapas_out[ej]['rasgos'].get(r, {}) for r in muestra}).T[['perfil', 'lo', 'hi', 'z', 'n']].round(2).to_string())
print('\nCadena (contexto):')
print(pd.DataFrame({e: {k: cadena[e][k] for k in ['antecesor', 'etapa_previa', 'elo_heredado', 'pct_heredado', 'elo_final', 'pct_final',
                                                  'pts_por_partido', 'xg_favor_pp', 'xg_contra_pp']} for e in cadena}).T.to_string())
cub = {e: int(sum(v['zc'] is not None and v['zt'] is not None for v in cadena[e]['rasgos'].values())) for e in cadena}
print('\nRasgos con zc y zt (cobertura calculable) por etapa:', cub)
print('\nAVISOS:' if avisos else '\nSin avisos.'); [print('  ·', a) for a in avisos]
print(f'\nG-3 terminó en {time.time() - t0:.0f} s. Sube reporte_datos.json y esta salida al chat del reporte.')

In [ ]:
# R7d v1.0 · Huella DEFENSIVA por rol: dónde defiende cada puesto (eventos defensivos con jugador y ubicación)
# =====================================================================
# Sesión 14 · 2026-10-20 · Chat del reporte · Decisión S14-21 (Arturo aprueba). Gemela de R7c (huella con balón).
# Por qué eventos y no 360: las fotos 360 no identifican jugadores; los eventos defensivos sí (quién, dónde y en qué puesto).
# Límite declarado: solo se ve dónde ACTÚA cada rol (presiona, entra, intercepta, recupera, bloquea, despeja), no dónde se para
#   cuando no interviene.
# Acciones (juego abierto, sin balón parado ni reanudaciones, periodos 1-2, durante posesión del RIVAL, como R4):
#   Pressure · Duel (solo Tackle si existe la columna) · Interception · Ball Recovery · Block · Clearance.
#   Coordenadas en el marco del que defiende (x = 0 su portería, 120 la del rival), como todos los eventos.
# Rasgos por equipo-partido (formato R4, una fila por partido; pasan por R6r con PREFIJO 'r7d'):
#   def_altura_delanteros · def_altura_interiores_mp · def_altura_extremos · def_altura_contenciones · def_altura_laterales ·
#   def_altura_centrales (x media de sus acciones defensivas) · def_amplitud_centrales (|y − 40| media: centrales que salen a banda)
#   · def_reparto_adelante (% de las acciones defensivas del equipo hechas por delanteros, extremos, interiores y mediapuntas)
#   · def_medios_ultima_linea (% de las acciones defensivas de contenciones e interiores a < 30 yd de su portería y en el carril
#     central: medios que bajan a defender la última línea; idea de Arturo: el interior que se incrusta entre centrales).
# Orden: celda 1 → R7d-0 → R7d-r0 → R6r-1 v1.1 y R6r-2 v1.1 SIN CAMBIOS (claude/codigo/sesion8/R6r_receta_confiabilidad.py)
#        → R7d-r9 → C-0 → R7d-9 (mapa de dos ejes y exportación). Tarda ~10-25 min en total.
# =====================================================================

#[R7d-0 v1.0] Acciones defensivas con puesto y rasgos r7d ------------------------------------------------------
import os, json, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5'); DIR_EV = os.path.join(DP, 'liga', 'eventos')
REHACER = False
CARRIL = (80 / 3, 160 / 3); X_ULT = 30
TIPOS = ['Pressure', 'Duel', 'Interception', 'Ball Recovery', 'Block', 'Clearance']
pd.set_option('display.width', 250)
t0 = time.time()
GRUPO = {1: 'POR', 3: 'CEN', 4: 'CEN', 5: 'CEN', 2: 'LAT', 6: 'LAT', 7: 'LAT', 8: 'LAT', 9: 'MCD', 10: 'MCD', 11: 'MCD',
         13: 'INT', 14: 'INT', 15: 'INT', 18: 'MP', 19: 'MP', 20: 'MP', 12: 'EXT', 16: 'EXT', 17: 'EXT', 21: 'EXT',
         22: 'DEL', 23: 'DEL', 24: 'DEL', 25: 'DEL'}                                                   # copia de R7c
K = lambda p, c: p * 1000.0 + c
with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
    TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); MIDS = set(Mm.match_id); TEMP_M = Mm.set_index('match_id')['temporada']

# ---------- 1. Acciones defensivas con jugador y ubicación ----------
ruta_a = os.path.join(FW5, 'r7d_acciones.parquet')
if os.path.exists(ruta_a) and not REHACER:
    AC = pd.read_parquet(ruta_a)
else:
    piezas = []
    for temp in TEMPS:
        t_ = temp.replace('/', '-'); r1 = os.path.join(FW1, f'eventos_{t_}.parquet'); disp = set(pq.read_schema(r1).names)
        cols = [c for c in ['match_id', 'index', 'period', 'minute', 'second', 'type.name', 'team.name', 'possession_team.name', 'loc_x', 'loc_y',
                            'fase', 'es_reanudacion', 'duel.type.name', 'player.id'] if c in disp]
        E = pd.read_parquet(r1, columns=cols); E = E[E.match_id.isin(MIDS) & E.period.isin([1, 2])]
        if 'second' not in E: E['second'] = 30
        tn = E['type.name']
        m = (E['fase'] != 'bp') & ~E['es_reanudacion'].fillna(False).astype(bool) & E.loc_x.notna() & E.loc_y.notna() & tn.isin(TIPOS)
        if 'possession_team.name' in E: m &= E['possession_team.name'] != E['team.name']
        if 'duel.type.name' in E: m &= ~((tn == 'Duel') & (E['duel.type.name'] != 'Tackle'))
        E = E[m]
        fuente = 'R1'
        if 'player.id' not in E:                         # los eventos defensivos no están en la caché de R6b: se leen los crudos
            xs = [pd.read_parquet(os.path.join(DIR_EV, f'{mm}.parquet'), columns=['index', 'player.id']).assign(match_id=mm)
                  for mm in E.match_id.unique() if os.path.exists(os.path.join(DIR_EV, f'{mm}.parquet'))]
            E = E.merge(pd.concat(xs), on=['match_id', 'index'], how='inner'); fuente = 'eventos crudos'
        piezas.append(pd.DataFrame({'match_id': E.match_id.values, 'team': E['team.name'].values, 'player_id': E['player.id'].values,
                                    'tipo': E['type.name'].values, 'k': K(E.period, E.minute + E.second / 60).values,
                                    'x': E.loc_x.values, 'y': E.loc_y.values}))
        print(f'  {temp}: {E.match_id.nunique()} partidos · {len(E):,} acciones defensivas · jugador desde {fuente} ({time.time() - t0:.0f} s)')
    AC = pd.concat(piezas, ignore_index=True).dropna(subset=['player_id']); AC['player_id'] = AC.player_id.astype('int64')
    TRA = pd.read_parquet(os.path.join(FW5, 'r7_tramos.parquet'), columns=['match_id', 'player_id', 'position_id', 'from_clk', 'from_period', 'to_clk', 'to_period'])
    TRA = TRA[TRA.from_period.isin([1, 2])].copy()                                                 # copia de R7c
    TRA['k0'] = K(TRA['from_period'], TRA['from_clk']); TRA['k1'] = np.where(TRA['to_clk'].notna(), K(TRA['to_period'].fillna(2), TRA['to_clk']), np.inf)
    AC = AC.merge(TRA[['match_id', 'player_id', 'position_id', 'k0', 'k1']], on=['match_id', 'player_id'], how='inner')
    AC = AC[(AC.k >= AC.k0 - 0.02) & (AC.k < AC.k1 + 0.02)].drop_duplicates(['match_id', 'player_id', 'k', 'x', 'y', 'tipo']).drop(columns=['k0', 'k1'])
    AC.to_parquet(ruta_a, index=False)
AC['grupo'] = AC.position_id.map(GRUPO); AC['temporada'] = AC.match_id.map(TEMP_M)
print(f'--- (1) Acciones defensivas con puesto: {len(AC):,} · partidos {AC.match_id.nunique()} · sin puesto {int(AC.grupo.isna().sum())}')
print('Por tipo (%):', (100 * AC.tipo.value_counts(normalize=True)).round(1).to_dict())
print('Por grupo (%):', (100 * AC.grupo.value_counts(normalize=True)).round(1).to_dict())
print('Altura media (x, desde su portería) por grupo, liga:', AC.groupby('grupo').x.mean().round(1).to_dict())

# ---------- 2. Rasgos por equipo-partido ----------
k = ['match_id', 'team']
def media(g, col='x'): return AC[AC.grupo.isin(g)].groupby(k)[col].mean()
AC['amp'] = (AC.y - 40).abs()
tot = AC.groupby(k).size()
adel = AC[AC.grupo.isin(['DEL', 'EXT', 'INT', 'MP'])].groupby(k).size()
med = AC[AC.grupo.isin(['MCD', 'INT'])]
med_ult = med[(med.x < X_ULT) & (med.y >= CARRIL[0]) & (med.y <= CARRIL[1])].groupby(k).size()
R = pd.DataFrame({
    'def_altura_delanteros': media(['DEL']), 'def_altura_interiores_mp': media(['INT', 'MP']), 'def_altura_extremos': media(['EXT']),
    'def_altura_contenciones': media(['MCD']), 'def_altura_laterales': media(['LAT']), 'def_altura_centrales': media(['CEN']),
    'def_amplitud_centrales': media(['CEN'], 'amp'),
    'def_reparto_adelante': 100 * adel.reindex(tot.index).fillna(0) / tot,
    'def_medios_ultima_linea': 100 * med_ult.reindex(med.groupby(k).size().index).fillna(0) / med.groupby(k).size()}).reset_index()
RASGOS = [c for c in R.columns if c.startswith('def_')]
print('\n--- (2) Rasgos por equipo-partido (media, p10, p90 y nulos) ---')
print(R[RASGOS].describe(percentiles=[.1, .9]).T[['count', 'mean', '10%', '90%']].round(2).assign(nulos=R[RASGOS].isna().sum().values).to_string())
print('Correlaciones entre rasgos (|r| alto = miden casi lo mismo):'); print(R[RASGOS].corr().round(2).to_string())
r7c = os.path.join(FW5, 'r7c_equipo_partido.parquet')
if os.path.exists(r7c):
    C = R.merge(pd.read_parquet(r7c), on=k)
    pares = [('def_altura_laterales', 'altura_laterales'), ('def_altura_contenciones', 'altura_contenciones'),
             ('def_altura_interiores_mp', 'altura_interiores_mp'), ('def_altura_delanteros', 'altura_delanteros')]
    print('Mismo rol con y sin balón (correlación por equipo-partido; si es ~1, la huella defensiva no agrega nada):',
          {a: round(C[a].corr(C[b]), 2) for a, b in pares})
filas = [pd.DataFrame({'metrica': m, 'match_id': R.match_id, 'team': R.team, 'e5': 0, 'e5b': 0, 'cuarto': 1, 'num': R[m], 'den': 1.0}).dropna(subset=['num']) for m in RASGOS]
pd.concat(filas, ignore_index=True).to_parquet(os.path.join(FW5, 'r7d_tramos.parquet'), index=False)
R.to_parquet(os.path.join(FW5, 'r7d_equipo_partido.parquet'), index=False)
print(f'\nR7d-0 terminó en {time.time() - t0:.0f} s. Guardado: fw5/r7d_tramos, r7d_equipo_partido, r7d_acciones.')


In [ ]:
# [R7d-r0 v1.0] Carpeta espejo para R6r-1 y R6r-2 con PREFIJO = 'r7d' (copia de R7c-r0 con otro prefijo)
import os, shutil
DRIVE = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
LOCAL = globals().get('LOCAL_R7D', '/content/isac_r7d/')
D_DP, D_FW1, D_FW5 = os.path.join(DRIVE, 'datos_parquet'), os.path.join(DRIVE, 'datos_parquet', 'fw1'), os.path.join(DRIVE, 'datos_parquet', 'fw5')
L_DP, L_FW = os.path.join(LOCAL, 'datos_parquet'), os.path.join(LOCAL, 'datos_parquet', 'fw1')
shutil.rmtree(LOCAL, ignore_errors=True); os.makedirs(L_FW, exist_ok=True); os.makedirs(os.path.join(LOCAL, 'salidas'), exist_ok=True)
def enlazar(src, dst):
    if os.path.exists(src): os.symlink(src, dst); return True
    return False
enl = {'partido_tecnico_periodo.parquet': enlazar(os.path.join(D_DP, 'partido_tecnico_periodo.parquet'), os.path.join(L_DP, 'partido_tecnico_periodo.parquet'))}
for f in ['r4_partidos.parquet', 'r4_minutos_v21.parquet', 'parametros_fw1.json', 'r4_perfiles_etapa.parquet', 'r6_perfiles_etapa.parquet',
          'r6b_perfiles_etapa.parquet', 'r6c_perfiles_etapa.parquet', 'r7c_perfiles_etapa.parquet', 'r7d_tramos.parquet']:
    enl[f] = enlazar(os.path.join(D_FW5, f), os.path.join(L_FW, f))
enl['elo (fw1)'] = enlazar(os.path.join(D_FW1, 'elo_equipo_partido_e2.parquet'), os.path.join(L_FW, 'elo_equipo_partido_e2.parquet'))
print('--- (R7d-r0) Carpeta espejo', LOCAL); print({k_: ('ok' if v else 'FALTA') for k_, v in enl.items()})
assert all(enl[k_] for k_ in ['partido_tecnico_periodo.parquet', 'r4_partidos.parquet', 'r4_minutos_v21.parquet', 'parametros_fw1.json',
                              'r4_perfiles_etapa.parquet', 'r7d_tramos.parquet', 'elo (fw1)']), 'Falta un insumo obligatorio: avisar a Claude'
BASE = LOCAL; PREFIJO = 'r7d'
print("Listo → corre R6r-1 v1.1 y R6r-2 v1.1 sin cambios; luego R7d-r9.")




In [ ]:
# [R7d-r9 v1.0] Copiar resultados r7d_* a Drive y regresar BASE a Drive (copia de R7c-r9)
import os, shutil
DRIVE = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
LOCAL = globals().get('LOCAL_R7D', '/content/isac_r7d/')
L_FW = os.path.join(LOCAL, 'datos_parquet', 'fw1')
D_FW5, D_SAL5 = os.path.join(DRIVE, 'datos_parquet', 'fw5'), os.path.join(DRIVE, 'salidas', 'fw5'); os.makedirs(D_SAL5, exist_ok=True)
n = []
for src_dir, dst_dir in [(L_FW, D_FW5), (os.path.join(LOCAL, 'salidas'), D_SAL5)]:
    for f in os.listdir(src_dir):
        s = os.path.join(src_dir, f)
        if f.startswith('r7d_') and os.path.isfile(s) and not os.path.islink(s):
            shutil.copy2(s, os.path.join(dst_dir, f)); n.append(f)
print('--- (R7d-r9) Copiados a Drive:', sorted(n)); BASE = DRIVE; print('BASE =', BASE)
print('Siguiente: corre C-0 (claude/codigo/sesion12/C_0_prueba.py) y después R7d-9.')

In [ ]:
# [R7d-9 v1.0] Huella defensiva por rol: mapa de dos ejes (dueño) y exportación de agregados → salidas/fw5/reporte_r7d.json
# =====================================================================
# Sesión 14 · 2026-10-20 · Chat del reporte · Después de R7d-0 → R7d-r0 → R6r-1 → R6r-2 → R7d-r9 → C-0 (todas las pruebas PASAN).
# (1) Confiabilidad y reloj de los rasgos r7d (de R6r-2). (2) Mapa de dos ejes con las MISMAS parejas que C-1b (29 viajeros,
#     69 relevos) y las mismas reglas (÷ confiabilidad, corte 0.5, firmeza por IC de la diferencia), más sin Elo de la etapa y sin
#     nivel heredado (G-2). (3) Exporta, para las etapas del selector, perfil contraído, banda de 95%, partidos con dato, perfil
#     del club antes de llegar (zc), del técnico en su etapa previa (zt) y de la etapa (zn): copia de G-3. Solo agregados.
# Tarda ~2-4 min. Escribe solo salidas/fw5/r7d_mapa.csv y salidas/fw5/reporte_r7d.json.
# =====================================================================
import os, json, time
import numpy as np
import pandas as pd

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
assert globals().get('C0_OK', False), 'Corre primero C-0 (claude/codigo/sesion12/C_0_prueba.py) y confirma que todas sus pruebas PASAN'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5, FW1 = os.path.join(DP, 'fw5'), os.path.join(DP, 'fw1'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
ETAPA_MIN, VENTANA, VENT_MIN, CONF_MIN, B, PLAC = 15, 18, 10, 0.5, 1000, 200
SELECTOR = {'Jardine': 'Jardine', 'Larcamón': 'Larcam', 'Fernando Ortiz': 'Fernando Ortiz'}
t0 = time.time(); pd.set_option('display.width', 250)

c = pd.read_parquet(os.path.join(FW5, 'r7d_perfiles_contraidos.parquet')); f = pd.read_csv(os.path.join(SAL5, 'r7d_confiabilidad.csv'))
pe = pd.read_parquet(os.path.join(FW5, 'r7d_perfiles_etapa.parquet'))
rs = pd.read_parquet(os.path.join(FW5, 'r7d_residuos.parquet'), columns=['metrica', 'match_id', 'team', 'e', 'den', 'res'])
tr = pd.read_parquet(os.path.join(FW5, 'r7d_tramos.parquet'))
for col_ in ['conf', 'tau', 'sigma_partido']: f[col_] = f[col_].astype(float)
f = f.set_index('metrica')
print('--- (1) Confiabilidad y reloj (partidos para leer el rasgo con confiabilidad 0.7) ---')
print(f[[x for x in ['conf', 'reloj', 'tau', 'sigma_partido'] if x in f.columns]].sort_values('conf', ascending=False).round(2).to_string())
RASGOS = [m for m in f.index if f.loc[m, 'conf'] >= CONF_MIN and f.loc[m, 'tau'] > 0]
print('Entran al mapa (confiabilidad ≥ 0.5):', RASGOS)

# ---------- Etapas y parejas (copia literal de C-1b / G-2) ----------
with open(os.path.join(FW5, 'parametros_fw1.json')) as fh:
    TEMPS = sorted(json.load(fh)['temporadas']['entrenamiento'])
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy(); ptp['fecha'] = pd.to_datetime(ptp['fecha'])
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ET = ptp.groupby('etapa_id').agg(team=('team', 'first'), tecnico=('tec_p', 'first'), partidos=('match_id', 'nunique'),
                                 ini=('fecha', 'min'), fin=('fecha', 'max')).reset_index()
ETAPAS = ET.etapa_id.tolist(); IX = {e: i for i, e in enumerate(ETAPAS)}; ETI = ET.set_index('etapa_id')
tau = f.loc[RASGOS, 'tau'].values; conf = f.loc[RASGOS, 'conf'].values
ZR = (c.pivot_table(index='etapa_id', columns='metrica', values='perfil_r4').reindex(index=ETAPAS, columns=RASGOS).astype(float) / tau).values
ZM = (c.pivot_table(index='etapa_id', columns='metrica', values='perfil_mostrado').reindex(index=ETAPAS, columns=RASGOS).astype(float) / tau).values
L = ET[ET.partidos >= ETAPA_MIN]
pt, ct = [], []
for tec, d in L.groupby('tecnico'):
    ix = [IX[e] for e in d.etapa_id]
    for i in range(len(ix)):
        for j in range(i + 1, len(ix)):
            if ET.team[ix[i]] != ET.team[ix[j]]: pt.append((ix[i], ix[j])); ct.append(tec)
pc, cc = [], []
for team, d in L.sort_values('ini').groupby('team'):
    ix = [IX[e] for e in d.etapa_id]
    for i in range(len(ix) - 1):
        if ET.tecnico[ix[i]] != ET.tecnico[ix[i + 1]]: pc.append((ix[i], ix[i + 1])); cc.append(team)
li = [IX[e] for e in L.etapa_id]
pool = [(a, b) for k_, a in enumerate(li) for b in li[k_ + 1:] if ET.tecnico[a] != ET.tecnico[b] and ET.team[a] != ET.team[b]]
pt, ct, pc, cc, pool = np.array(pt), np.array(ct), np.array(pc), np.array(cc), np.array(pool)
print(f'\nParejas: viajeros {len(pt)} de {len(set(ct))} técnicos (deben ser 29 de 20) · relevos {len(pc)} en {len(set(cc))} clubes (69 en 18)')

# ---------- (2) Mapa ----------
T = ejes(ZR, conf, pt, ct, pc, cc, pool, B=B, P=PLAC).set_index(pd.Index(RASGOS, name='rasgo'))
T['cuadrante'], T['firmeza'] = clasificar(T)
elo = pd.read_parquet(os.path.join(FW1, 'elo_equipo_partido_e2.parquet'), columns=['match_id', 'team', 'elo_pre'])
EP = ptp.merge(elo, on=['match_id', 'team'], how='left')
nivel = EP.groupby('etapa_id')['elo_pre'].mean().reindex(ETAPAS).values
h2 = np.full(len(ETAPAS), np.nan)
for i, r_ in enumerate(ET.itertuples(index=False)):
    v = EP[(EP.team == r_.team) & (EP.fecha < r_.ini)].sort_values('fecha').tail(VENTANA)
    if len(v) >= VENT_MIN: h2[i] = v.elo_pre.mean()
def quitar(niv, nan_fuera):                                                      # copia de G-2
    Z = ZR.copy() if not nan_fuera else np.where(np.isfinite(niv)[:, None], ZR, np.nan)
    for j in range(ZR.shape[1]):
        m = np.isfinite(ZR[:, j]) & np.isfinite(niv)
        if m.sum() > 10:
            b = np.polyfit(niv[m], ZR[m, j], 1); Z[m, j] = ZR[m, j] - np.polyval(b, niv[m])
    return Z
cd = np.clip(conf, .05, None)
for nom, Z in [('sin_elo', quitar(nivel, False)), ('sin_heredado', quitar(h2, True))]:
    T[f'tec_{nom}'] = r_pares(Z, pt) / cd; T[f'club_{nom}'] = r_pares(Z, pc) / cd
T['conf'] = conf; T['reloj'] = f.loc[RASGOS, 'reloj'].values if 'reloj' in f.columns else np.nan
V = ['conf', 'reloj', 'tec', 'tec_lo', 'tec_hi', 'club', 'club_lo', 'club_hi', 'cuadrante', 'firmeza', 'tec_sin_elo', 'club_sin_elo', 'tec_sin_heredado', 'club_sin_heredado']
print(f'\n--- (2) Mapa de dos ejes · placebo (≈ 0): {T.placebo_media.mean():+.3f} · ruido de un rasgo sin efecto ±{2 * T.placebo_de.median():.2f} ---')
print(T[V].round(2).to_string())
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Freno: Drive no está montado'
T.to_csv(os.path.join(SAL5, 'r7d_mapa.csv'))

# ---------- (3) Exportación (copia de G-3) ----------
S2 = {m: float((g.den * g.res ** 2).sum() / (len(g) - 19)) for m, g in rs.groupby('metrica')}
TAUE = pe.groupby('metrica')['tau_estado'].first().to_dict(); RES = {t: g for t, g in rs.groupby('team')}
MIN_ = pd.read_parquet(os.path.join(FW5, 'r4_minutos_v21.parquet'))
W3 = MIN_.assign(e3=np.sign(MIN_.e5)).groupby('e3')['minutos'].sum(); W3v = (W3 / W3.sum()).reindex([-1, 0, 1]).fillna(0).values
def perfil_club(team, partidos, contraido=True):
    z = np.full(len(RASGOS), np.nan)
    for j, m_ in enumerate(RASGOS):
        g = RES.get(team)
        if g is None: continue
        g = g[(g.metrica == m_) & g.match_id.isin(partidos)]; n = g.match_id.nunique()
        if n == 0: continue
        pf = perfil_ventana(g.e.values, g.den.values, g.res.values, S2[m_], TAUE[m_], W3v)
        z[j] = (contraer(pf, n, f.loc[m_, 'tau'], f.loc[m_, 'sigma_partido']) if contraido else pf) / f.loc[m_, 'tau']
    return z
e0 = L.etapa_id.iloc[0]
dmax = np.nanmax(np.abs(perfil_club(ETI.loc[e0, 'team'], set(ptp.loc[ptp.etapa_id == e0, 'match_id']), contraido=False) - ZR[IX[e0]]))
print(f'\nCoherencia de perfil_club (etapa completa {e0}): diferencia máxima {dmax:.2e} z (≈ 0)')
SEL = [e for p in SELECTOR.values() for e in ET[ET.tecnico.astype(str).str.contains(p, case=False)].etapa_id]
liga = tr.groupby('metrica').apply(lambda g: (g.num * g.den).sum() / g.den.sum()).to_dict()
cp = c.set_index(['metrica', 'etapa_id'])
lim = lambda v: None if v is None or not np.isfinite(float(v)) else round(float(v), 4)
out = {'rasgos': {}, 'etapas': {}}
for j, m_ in enumerate(RASGOS):
    out['rasgos'][m_] = {k_: (lim(T.loc[m_, k_]) if isinstance(T.loc[m_, k_], (int, float, np.integer, np.floating)) else str(T.loc[m_, k_])) for k_ in V}
    out['rasgos'][m_].update({'tau': lim(f.loc[m_, 'tau']), 'sigma_partido': lim(f.loc[m_, 'sigma_partido']), 'liga': lim(liga.get(m_))})
for e in SEL:
    r_ = ETI.loc[e]; ids = set(ptp.loc[ptp.etapa_id == e, 'match_id'])
    ven = ptp[(ptp.team == r_.team) & (ptp.fecha < r_.ini)].sort_values('fecha').tail(VENTANA)
    previa = ET[(ET.tecnico == r_.tecnico) & (ET.team != r_.team) & (ET.fin < r_.ini) & (ET.partidos >= ETAPA_MIN)].sort_values('fin')
    zc = perfil_club(r_.team, set(ven.match_id)) if len(ven) >= VENT_MIN else np.full(len(RASGOS), np.nan)
    zt = ZM[IX[previa.etapa_id.iloc[-1]]] if len(previa) else np.full(len(RASGOS), np.nan)
    fil = {}
    for j, m_ in enumerate(RASGOS):
        if (m_, e) not in cp.index: continue
        x = cp.loc[(m_, e)]; g = RES.get(r_.team); n = int(g.loc[(g.metrica == m_) & g.match_id.isin(ids), 'match_id'].nunique()) if g is not None else 0
        ta, sg = f.loc[m_, 'tau'], f.loc[m_, 'sigma_partido']; kk = ta ** 2 / (ta ** 2 + sg ** 2 / n) if n else np.nan
        sd = np.sqrt(kk) * sg / np.sqrt(n) if n else np.nan
        fil[m_] = {'perfil': lim(x['perfil_mostrado']), 'lo': lim(x['perfil_mostrado'] - 1.96 * sd), 'hi': lim(x['perfil_mostrado'] + 1.96 * sd),
                   'z': lim(x['perfil_mostrado'] / ta), 'n': n, 'zc': lim(zc[j]), 'zt': lim(zt[j]), 'zn': lim(ZM[IX[e], j])}
    out['etapas'][e] = fil
ruta = os.path.join(SAL5, 'reporte_r7d.json')
with open(ruta, 'w', encoding='utf-8') as fh: json.dump(out, fh, ensure_ascii=False)
print(f'Etapas exportadas ({len(SEL)}): {SEL}')
ej = next((e for e in SEL if 'Jardine' in str(ETI.loc[e, 'tecnico']) and e.startswith('Am')), SEL[0])
print(f'\nEjemplo · {ej} (valor = liga + perfil):')
print(pd.DataFrame({m_: {'liga': out['rasgos'][m_]['liga'], 'perfil': out['etapas'][ej].get(m_, {}).get('perfil'), 'n': out['etapas'][ej].get(m_, {}).get('n')}
                    for m_ in RASGOS}).T.round(2).to_string())
print(f'\nR7d-9 terminó en {time.time() - t0:.0f} s. Guardado: salidas/fw5/r7d_mapa.csv y reporte_r7d.json. Sube el JSON y esta salida al chat.')

In [ ]:
# [G-4 v1.0] Los 11 en la cancha (con y sin balón) y percentiles entre técnicos → salidas/fw5/reporte_posiciones.json
# =====================================================================
# Sesión 14 · 2026-10-20 · Chat del reporte · Pedido de Arturo: ver a los 11 en la cancha y leer cada rasgo más rápido.
# (1) Posición media por puesto exacto (los 25 puestos de StatsBomb: lateral derecho, central izquierdo, mediapunta...) para las
#     etapas del selector y la media de la liga por puesto. Con balón: acciones de R7c (r7c_acciones). Sin balón: acciones
#     defensivas de R7d (r7d_acciones; correr R7d-0 antes). Por etapa se toman los 11 puestos con más acciones (1 portero) y solo
#     puestos con 30 acciones o más. Es DESCRIPTIVO (sin ajuste por contexto ni contracción): los rasgos medidos de la ficha
#     siguen siendo los de R7c/R7d con su confiabilidad.
# (2) Percentil de cada etapa del selector entre las etapas de 15+ partidos, en cada rasgo (perfil contraído): "más que el X% de
#     los técnicos". Una cifra fácil de leer para la ficha.
# Orden: celda 1 → (R7d-0 ya corrido) → G-4. Tarda ~1-3 min. Solo agregados; no escribe nada más.
# =====================================================================
import os, json, glob, time
import numpy as np
import pandas as pd

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5 = os.path.join(DP, 'fw5'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
SELECTOR = ['Jardine', 'Larcam', 'Fernando Ortiz']; MIN_ACC, ETAPA_MIN = 30, 15
NOMBRE_PUESTO = {1: 'Portero', 2: 'Lateral derecho', 3: 'Central derecho', 4: 'Central', 5: 'Central izquierdo', 6: 'Lateral izquierdo',
                 7: 'Carrilero derecho', 8: 'Carrilero izquierdo', 9: 'Contención derecha', 10: 'Contención', 11: 'Contención izquierda',
                 12: 'Volante derecho', 13: 'Interior derecho', 14: 'Medio centro', 15: 'Interior izquierdo', 16: 'Volante izquierdo',
                 17: 'Extremo derecho', 18: 'Mediapunta derecho', 19: 'Mediapunta', 20: 'Mediapunta izquierdo', 21: 'Extremo izquierdo',
                 22: 'Delantero derecho', 23: 'Delantero centro', 24: 'Delantero izquierdo', 25: 'Segundo delantero'}
t0 = time.time(); pd.set_option('display.width', 250)
with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
    TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy(); ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ET = ptp.groupby('etapa_id').agg(tecnico=('tec_p', 'first'), partidos=('match_id', 'nunique')).reset_index()
SEL = [e for p in SELECTOR for e in ET[ET.tecnico.astype(str).str.contains(p, case=False)].etapa_id]
print(f'Etapas del selector: {len(SEL)} (deben ser 9) → {SEL}')

def once(ruta, nombre):
    if not os.path.exists(ruta): print(f'  AVISO: falta {os.path.basename(ruta)} ({nombre}); se omite'); return None, None
    A = pd.read_parquet(ruta, columns=['match_id', 'team', 'position_id', 'x', 'y']).merge(ptp[['match_id', 'team', 'etapa_id']], on=['match_id', 'team'])
    A['position_id'] = A.position_id.astype(int)
    liga = A.groupby('position_id').agg(x=('x', 'mean'), y=('y', 'mean'), n=('x', 'size'))
    out = {}
    for e in SEL:
        g = A[A.etapa_id == e].groupby('position_id').agg(x=('x', 'mean'), y=('y', 'mean'), n=('x', 'size'))
        g = g[g.n >= MIN_ACC].sort_values('n', ascending=False)
        por = g[g.index == 1].head(1); campo = g[g.index != 1].head(10)
        sel = pd.concat([por, campo])
        out[e] = [{'puesto': int(p), 'nombre': NOMBRE_PUESTO.get(int(p), str(p)), 'x': round(float(r.x), 2), 'y': round(float(r.y), 2),
                   'n': int(r.n), 'lx': round(float(liga.loc[p, 'x']), 2), 'ly': round(float(liga.loc[p, 'y']), 2),
                   'pct_acciones': round(100 * float(r.n) / float(g.n.sum()), 1)} for p, r in sel.iterrows()]
    print(f'  {nombre}: {len(A):,} acciones · puestos en la liga {len(liga)}')
    return out, {int(p): {'nombre': NOMBRE_PUESTO.get(int(p), str(p)), 'x': round(float(r.x), 2), 'y': round(float(r.y), 2), 'n': int(r.n)} for p, r in liga.iterrows()}
con, liga_con = once(os.path.join(FW5, 'r7c_acciones.parquet'), 'con balón (R7c)')
sin, liga_sin = once(os.path.join(FW5, 'r7d_acciones.parquet'), 'sin balón (R7d)')

# ---------- (2) Percentiles entre etapas de 15+ ----------
L15 = set(ET.loc[ET.partidos >= ETAPA_MIN, 'etapa_id'])
pct = {e: {} for e in SEL}
for ruta in sorted(glob.glob(os.path.join(FW5, '*_perfiles_contraidos.parquet'))):
    pre = os.path.basename(ruta).split('_')[0]; pre = 'r4' if pre == 'r5' else pre
    c = pd.read_parquet(ruta, columns=['metrica', 'etapa_id', 'perfil_mostrado'])
    for m, g in c.groupby('metrica'):
        ref = g[g.etapa_id.isin(L15)].perfil_mostrado.dropna().values
        if len(ref) < 20: continue
        for e in SEL:
            v = g.loc[g.etapa_id == e, 'perfil_mostrado']
            if len(v) and np.isfinite(v.iloc[0]):
                pct[e][f'{pre}:{m}'] = round(100 * ((ref < v.iloc[0]).mean() + 0.5 * (ref == v.iloc[0]).mean()), 0)
print(f'Percentiles: {len(next(iter(pct.values())))} rasgos por etapa (referencia: {len(L15)} etapas de 15+)')

OUT = {'version': 'G-4 v1.0', 'con_balon': con, 'sin_balon': sin, 'liga_con': liga_con, 'liga_sin': liga_sin, 'percentiles': pct}
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Freno: Drive no está montado'
ruta = os.path.join(SAL5, 'reporte_posiciones.json')
with open(ruta, 'w', encoding='utf-8') as f: json.dump(OUT, f, ensure_ascii=False)
ej = next((e for e in SEL if e.startswith('Am') and 'Jardine' in ET.set_index('etapa_id').loc[e, 'tecnico']), SEL[0])
for nom, d in [('con balón', con), ('sin balón', sin)]:
    if d: print(f'\n{ej} · {nom}:'); print(pd.DataFrame(d[ej])[['nombre', 'x', 'lx', 'y', 'ly', 'n', 'pct_acciones']].to_string(index=False))
print(f'\nPercentiles de ejemplo ({ej}):', {k: v for k, v in list(pct[ej].items())[:8]})
print(f'\nG-4 terminó en {time.time() - t0:.0f} s · {os.path.getsize(ruta) / 1e3:.0f} KB. Sube reporte_posiciones.json y esta salida al chat.')

In [ ]:
# [G-5 v1.0] Dónde juega cada etapa (por tercios) y medias de liga de los rasgos de R4 → salidas/fw5/reporte_territorio.json
# =====================================================================
# Sesión 14 · 2026-10-21 · Chat del reporte · Pedido de Arturo: la capa de espacio (posiciones medias) no refleja el territorio;
#   un equipo con field tilt alto juega mucho en el tercio rival, pero sus posiciones medias mezclan todas sus acciones.
# (1) Territorio real por tercios: % de las acciones con balón (R7c) y de las acciones defensivas (R7d) de cada etapa que ocurren
#     en el tercio propio, el medio y el rival; y la media de la liga. Descriptivo, solo agregados.
# (2) Medias de liga en unidades naturales de los rasgos de R4 (intensidad de presión, presión alta, saque de meta corto, etc.),
#     que faltaban en G-3 porque no hay r4_tramos en fw5: se buscan en fw1 y fw5 (tramos con num/den).
# Orden: celda 1 → G-5. Tarda ~1 min. No escribe nada más.
# =====================================================================
import os, json
import numpy as np
import pandas as pd

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
SELECTOR = ['Jardine', 'Larcam', 'Fernando Ortiz']
with open(os.path.join(FW5, 'parametros_fw1.json')) as f:
    TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy(); ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ET = ptp.groupby('etapa_id').agg(tecnico=('tec_p', 'first')).reset_index()
SEL = [e for p in SELECTOR for e in ET[ET.tecnico.astype(str).str.contains(p, case=False)].etapa_id]
print('Etapas:', len(SEL), SEL)

def tercios(ruta, nombre):
    if not os.path.exists(ruta): print('  AVISO: falta', os.path.basename(ruta)); return None
    A = pd.read_parquet(ruta, columns=['match_id', 'team', 'x']).merge(ptp[['match_id', 'team', 'etapa_id']], on=['match_id', 'team'])
    A['t'] = pd.cut(A.x, [-1, 40, 80, 121], labels=['propio', 'medio', 'rival'])
    liga = (100 * A.t.value_counts(normalize=True)).round(1).to_dict()
    out = {'liga': liga}
    for e in SEL:
        out[e] = (100 * A.loc[A.etapa_id == e, 't'].value_counts(normalize=True)).round(1).to_dict()
    print(f'  {nombre}: liga {liga} · ejemplo {SEL[0]} {out[SEL[0]]}')
    return out
TER = {'con_balon': tercios(os.path.join(FW5, 'r7c_acciones.parquet'), 'con balón'),
       'sin_balon': tercios(os.path.join(FW5, 'r7d_acciones.parquet'), 'sin balón')}

LIGA = {}
for d in [FW5, FW1]:
    r = os.path.join(d, 'r4_tramos.parquet')
    if os.path.exists(r) and not LIGA:
        t = pd.read_parquet(r)
        print('  r4_tramos en', os.path.basename(d), '· columnas:', list(t.columns)[:12])
        if {'metrica', 'num', 'den'} <= set(t.columns):
            g = t.groupby('metrica')[['num', 'den']].sum(); LIGA = {m: round(float(v), 4) for m, v in (g.num / g.den).items()}
if not LIGA: print('  AVISO: no se encontró r4_tramos con num/den; las medias de liga de R4 quedan vacías')
print('Medias de liga de R4:', LIGA)

OUT = {'version': 'G-5 v1.0', 'tercios': TER, 'liga_r4': LIGA}
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Freno: Drive no está montado'
with open(os.path.join(SAL5, 'reporte_territorio.json'), 'w', encoding='utf-8') as f: json.dump(OUT, f, ensure_ascii=False)
print('\nG-5 terminó. Sube salidas/fw5/reporte_territorio.json y esta salida al chat.')

In [ ]:
# [G-6 v1.0] Lugar en la tabla de Elo al llegar y al irse (SOLO LECTURA: no escribe nada)
# =====================================================================
# Sesión 15 · 2026-10-21 · Chat del reporte · Corrección 8 de Arturo: el nivel se muestra como lugar entre los 18 clubes, no como percentil.
# Definición (se propone a Arturo; ver el chat):
#   - "Al llegar": orden de los clubes de la temporada por su Elo vigente el día ANTES del primer partido del técnico
#     (Elo vigente = Elo después del último partido jugado antes de esa fecha; si el club aún no jugaba, su Elo de arranque).
#   - "Al irse": el mismo orden el día DESPUÉS de su último partido.
#   - Clubes que cuentan: los que juegan la temporada de esa fecha (18). Lugar 1 = Elo más alto.
#   El Elo es el mismo de siempre (E-2, fw1): K 20 · H 75 · regreso de 25% hacia 1,500 en verano. Ese regreso no cambia el orden.
# Orden: celda 1 (Drive) → G-6. Tarda segundos. Pega TODA la salida en el chat del reporte.
# =====================================================================
import os, json
import numpy as np
import pandas as pd

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1 = os.path.join(DP, 'fw1')
ETAPAS = ['Atlético San Luis · 2', 'América · 3', 'Puebla · 1', 'León · 4', 'Necaxa · 9', 'Cruz Azul · 9',
          'América · 2', 'Monterrey · 3', 'Santos Laguna · 6', 'América · 1']

ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp['fecha'] = pd.to_datetime(ptp['fecha']); ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
elo = pd.read_parquet(os.path.join(FW1, 'elo_equipo_partido_e2.parquet'))
elo['fecha'] = pd.to_datetime(elo['fecha'])

def tabla(fecha, temporada):
    """Elo vigente de cada club de la temporada en `fecha` (antes de jugar ese día) y su lugar."""
    clubes = elo.loc[elo.temporada == temporada, 'team'].unique()
    vals = {}
    for c in clubes:
        e = elo[elo.team == c].sort_values('fecha')
        antes = e[e.fecha < fecha]
        vals[c] = antes.elo_post.iloc[-1] if len(antes) else e.elo_pre.iloc[0]
    s = pd.Series(vals).sort_values(ascending=False)
    return s, {c: i + 1 for i, c in enumerate(s.index)}

out, filas = {}, []
for et in ETAPAS:
    d = ptp[ptp.etapa_id == et]
    if not len(d): print('AVISO: no encontré', et); continue
    club, ini, fin = d.team.iloc[0], d.fecha.min(), d.fecha.max()
    t_ini = elo.loc[(elo.team == club) & (elo.fecha >= ini), 'temporada'].iloc[0]
    t_fin = elo.loc[(elo.team == club) & (elo.fecha <= fin), 'temporada'].iloc[-1]
    s0, r0 = tabla(ini, t_ini)
    s1, r1 = tabla(fin + pd.Timedelta(days=1), t_fin)
    out[et] = {'lugar_llega': r0[club], 'n_llega': len(s0), 'lugar_va': r1[club], 'n_va': len(s1),
               'elo_llega': round(float(s0[club]), 1), 'elo_va': round(float(s1[club]), 1),
               'lider_llega': s0.index[0], 'lider_va': s1.index[0]}
    filas.append({'etapa': et, 'llega': ini.date(), 'se va': fin.date(), 'temporada (llega/va)': f'{t_ini} / {t_fin}',
                  'lugar al llegar': f"{r0[club]} de {len(s0)}", 'lugar al irse': f"{r1[club]} de {len(s1)}",
                  'Elo llegar/irse': f"{s0[club]:.0f} / {s1[club]:.0f}"})
print('--- G-6 · Lugar en la tabla de Elo (contexto; no mide el efecto del técnico) ---')
print(pd.DataFrame(filas).to_string(index=False))
print('\nChequeo: todas las tablas deben tener 18 clubes →', sorted({v['n_llega'] for v in out.values()} | {v['n_va'] for v in out.values()}))
print('\nPEGA ESTO (va a datos_retrato.json):')
print(json.dumps(out, ensure_ascii=False))

In [ ]:
# [G-7 v1.0] Cómo cambia el estilo según el marcador (SOLO LECTURA de fw5/fw1; escribe un JSON chico en salidas/fw5)
# =====================================================================
# Sesión 15 · 2026-10-21 · Chat del reporte · Mejora 3 de la revisión: "¿cómo cambian los patrones según el contexto?" (componente
#   Consistencia y variabilidad del reto). Hoy el marcador solo se usa para AJUSTAR; aquí se MUESTRA.
# Qué hace:
#   (1) Busca los archivos de tramos (formato R4: metrica, match_id, team, e5, num, den) en fw5 y, si falta alguno, en fw1.
#       Imprime cuáles encontró y sus columnas (si algo no cuadra, Claude lo revisa con esta salida).
#   (2) Para cada rasgo y cada estado del marcador (perdiendo, empatando, ganando = signo de e5, marcador efectivo de R4):
#       valor de la liga = Σnum / Σden en ese estado; valor de cada etapa del selector igual, más partidos y peso (den) con ese estado.
#       Es el valor CRUDO (sin ajuste por rival ni contracción): se compara contra la liga en el MISMO estado.
#   (3) Imprime una tabla de control (América · 3 y Jardine en San Luis) y escribe salidas/fw5/reporte_contexto.json.
# Orden: celda 1 (Drive) → G-7. Tarda ~1-2 min. Pega TODA la salida en el chat del reporte y sube el JSON (o pégalo si es corto).
# =====================================================================
import os, re, json, glob
import numpy as np
import pandas as pd

RAIZ = globals().get('RAIZ_PRUEBA', '/content/drive/MyDrive/isac2026/')
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW1, FW5 = os.path.join(DP, 'fw1'), os.path.join(DP, 'fw5'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
ETAPAS = ['Atlético San Luis · 2', 'América · 3', 'Puebla · 1', 'León · 4', 'Necaxa · 9', 'Cruz Azul · 9',
          'América · 2', 'Monterrey · 3', 'Santos Laguna · 6', 'América · 1']
EST = {-1: 'perdiendo', 0: 'empatando', 1: 'ganando'}

ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
MAP = ptp[['match_id', 'team', 'etapa_id']].drop_duplicates(['match_id', 'team'])

# ---------- (1) Archivos de tramos ----------
cand = {}
for d in [FW5, FW1]:                                              # fw5 tiene prioridad (cinco temporadas)
    for f in sorted(glob.glob(os.path.join(d, '*tramos*.parquet'))):
        pre = os.path.basename(f).split('_')[0]
        if not re.fullmatch(r'r\d+[a-z]*', pre): continue
        if pre in cand and os.path.dirname(cand[pre]) == FW5: continue
        if pre in cand and 'v21' not in os.path.basename(f) and 'v21' in os.path.basename(cand[pre]): pass
        cand.setdefault(pre, f)
print('--- (1) Archivos de tramos encontrados ---')
TR = []
for pre, f in sorted(cand.items()):
    t = pd.read_parquet(f)
    ok = {'metrica', 'match_id', 'team', 'e5', 'num', 'den'} <= set(t.columns)
    print(f'  {pre:5s} {os.path.relpath(f, DP)} · filas {len(t):,} · {"OK" if ok else "FALTAN COLUMNAS: " + str(list(t.columns)[:10])}')
    if ok:
        t = t[['metrica', 'match_id', 'team', 'e5', 'num', 'den']].copy(); t['id'] = pre + ':' + t['metrica'].astype(str); TR.append(t)
assert TR, 'No encontré tramos con columnas metrica/match_id/team/e5/num/den: pega esta salida a Claude'
T = pd.concat(TR, ignore_index=True)
T = T[T.den > 0].copy(); T['s'] = np.sign(T['e5']).astype(int)
T = T.merge(MAP, on=['match_id', 'team'], how='left')
print(f'Filas con etapa: {T.etapa_id.notna().mean():.1%} (debe ser ~100%) · rasgos: {T.id.nunique()}')

# ---------- (2) Liga y etapas por estado ----------
g = T.groupby(['id', 's'])[['num', 'den']].sum(); LIGA = (g.num / g.den).unstack('s')
S = T[T.etapa_id.isin(ETAPAS)]
ge = S.groupby(['etapa_id', 'id', 's']).agg(num=('num', 'sum'), den=('den', 'sum'), partidos=('match_id', 'nunique'))
ge['v'] = ge.num / ge.den
OUT = {'version': 'G-7 v1.0', 'liga': {i: {str(s): round(float(v), 4) for s, v in r.dropna().items()} for i, r in LIGA.iterrows()}, 'etapas': {}}
for (e, i), d in ge.groupby(level=[0, 1]):
    OUT['etapas'].setdefault(e, {})[i] = {str(s): [round(float(r.v), 4), int(r.partidos), round(float(r.den), 1)] for (_, _, s), r in d.iterrows()}

# ---------- (3) Control ----------
CTRL = ['r4:posesion', 'r4:intensidad_ppda', 'r4:presion_alta', 'r4:pase_largo', 'r6:seg_hasta_ultimo', 'r7u:cambio_familia_90', 'r9:altura']
print('\n--- (2) Liga por estado del marcador (valor crudo) ---')
print(LIGA.reindex([c for c in CTRL if c in LIGA.index]).rename(columns=EST).round(2).to_string())
for e in ['América · 3', 'Atlético San Luis · 2']:
    if e not in OUT['etapas']: print('AVISO: sin datos para', e); continue
    print(f'\n{e}: valor (partidos con ese estado)')
    for c in CTRL:
        if c in OUT['etapas'][e]:
            print(f'  {c:24s}', ' · '.join(f"{EST[int(s)]} {v[0]:.2f} ({v[1]})" for s, v in sorted(OUT['etapas'][e][c].items(), key=lambda x: int(x[0]))))
print('\nEtapas con datos:', sorted(OUT['etapas']), '· faltan:', [e for e in ETAPAS if e not in OUT['etapas']])
assert 'RAIZ_PRUEBA' in globals() or os.path.ismount('/content/drive'), 'Freno: Drive no está montado'
ruta = os.path.join(SAL5, 'reporte_contexto.json')
with open(ruta, 'w', encoding='utf-8') as f: json.dump(OUT, f, ensure_ascii=False, separators=(',', ':'))
print(f'\nG-7 terminó. Escribió {ruta} ({os.path.getsize(ruta) / 1024:.0f} KB). Súbelo al chat (o pégalo si pesa menos de ~60 KB).')

In [ ]:
# [G-8 v1.0] Emparejamiento aéreo por HOPS frente a por estatura, en partidos reales (SOLO LECTURA de fw5 y liga/lineups; escribe un JSON chico)
# =====================================================================
# Sesión 15 · 2026-10-21 · Chat del reporte · Punto 12 de Arturo: E-3 con datos reales en vez de inventados.
# Qué hace (unidad = un lado de un partido: el equipo que defiende el córner contra los cabeceadores del rival):
#   (0) Reconstruye la capacidad aérea igual que R8h-1 (HOPS jugador-temporada con ≥ 10 duelos + estatura de lineups) y chequea
#       la correlación HOPS-estatura por equipo-partido contra R8m (0.35).
#   (1) En cada partido, los 5 jugadores MÁS ALTOS de cada equipo que jugaron (con HOPS y estatura) son "los que van al área".
#       Defensa: marca por estatura (el más alto al más alto) contra marca por HOPS (método húngaro que minimiza la suma de
#       desventajas al cuadrado). Desventaja de un duelo = max(0, HOPS del atacante − HOPS de su marca).
#       Ataque: si el rival marca por estatura, ¿cuál es nuestro duelo más favorable? ¿Es el más alto, el de mejor HOPS u otro?
#   (2) Resumen de la liga con IC 95% (remuestreo de partidos, 500, semilla 2026) y 3 ejemplos al azar, anónimos.
# Límite: no sabemos quién entró al área ni quién marcó a quién (el 360 no identifica jugadores). Es un ejercicio de asignación
#   sobre jugadores reales, no una prueba de que marcar por HOPS evita goles.
# Orden: celda 1 (Drive) → G-8. Tarda ~2-4 min. Pega TODA la salida y sube salidas/fw5/reporte_hops.json (o pégalo, es chico).
# =====================================================================
import os, json, time
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5, LIGA = os.path.join(DP, 'fw5'), os.path.join(DP, 'liga')
SAL5 = os.path.join(RAIZ, 'salidas', 'fw5'); SEMILLA = 2026; t0 = time.time()

# ---------- (0) Capacidad aérea: igual que R8h-1 ----------
Mm = pd.read_parquet(os.path.join(FW5, 'r4_partidos.parquet')); loc = Mm.set_index('match_id')
HJ = pd.read_parquet(os.path.join(FW5, 'r8m_hops_jugador.parquet'))
JP = pd.read_parquet(os.path.join(FW5, 'r7_jugador_partido.parquet'))
pcol = next(c for c in ['player_id', 'player.id', 'jugador_id', 'jugador'] if c in JP.columns)
mcol = next(c for c in ['minutos', 'min', 'minutes'] if c in JP.columns)
JP = JP[JP[mcol] > 0].rename(columns={pcol: 'player_id'})
JP['temporada'] = JP.match_id.map(loc['temporada'])
JP = JP.merge(HJ[HJ.duelos >= 10][['player_id', 'temporada', 'hops']], on=['player_id', 'temporada'], how='left')
alt = []
for mid in Mm.match_id:
    r = os.path.join(LIGA, 'lineups', f'{mid}.parquet')
    if os.path.exists(r): alt.append(pd.read_parquet(r, columns=['player_id', 'player_height']))
ALT = pd.concat(alt).dropna().query('player_height > 0').drop_duplicates('player_id')
JP = JP.merge(ALT, on='player_id', how='left')
top5 = lambda s: s.dropna().nlargest(5).mean() if s.notna().sum() >= 5 else np.nan
TM = JP.groupby(['match_id', 'team']).agg(h=('hops', top5), a=('player_height', top5))
r_ha = TM[['h', 'a']].corr().iloc[0, 1]
print(f'--- (0) Correlación HOPS-estatura por equipo-partido {r_ha:.2f} (R8m: 0.35) → {"REPRODUCE" if abs(r_ha - .35) <= .01 else "NO REPRODUCE: pega esta línea"}')
J = JP.dropna(subset=['hops', 'player_height'])
r_j = J.drop_duplicates(['player_id', 'temporada'])[['hops', 'player_height']].corr().iloc[0, 1]
print(f'    Correlación HOPS-estatura entre jugadores (jugador-temporada): {r_j:.2f} · rango de HOPS p5 {J.hops.quantile(.05):.1f} · p50 {J.hops.median():.1f} · p95 {J.hops.quantile(.95):.1f}')

# ---------- (1) Asignación por partido ----------
G = {k: g.nlargest(5, 'player_height')[['player_height', 'hops']].to_numpy() for k, g in J.groupby(['match_id', 'team']) if len(g) >= 5}
filas = []
for mid in Mm.match_id:
    eq = [t for (m, t) in [(mid, loc.at[mid, 'local']), (mid, loc.at[mid, 'visita'])] if (m, t) in G]
    if len(eq) < 2: continue
    for de, at in [(eq[0], eq[1]), (eq[1], eq[0])]:
        D, A = G[(mid, de)], G[(mid, at)]                          # ya vienen del más alto al más bajo
        Dh, Ah = D[:, 1], A[:, 1]
        dz_alt = np.maximum(0, Ah - Dh)                             # marca por estatura: i-ésimo más alto con i-ésimo más alto
        C = np.maximum(0, Ah[None, :] - Dh[:, None]) ** 2
        fi, co = linear_sum_assignment(C); dz_h = np.maximum(0, Ah[co] - Dh[fi])
        adv = Ah - Dh                                               # ataque: atacante j contra su marca por estatura (mismo rango)
        b = int(np.argmax(adv))
        filas.append(dict(match_id=mid, tot_alt=dz_alt.sum(), tot_h=dz_h.sum(), peor_alt=dz_alt.max(), peor_h=dz_h.max(),
                          cambia=bool((co != np.arange(5)).any()), mejor_adv=adv[b], adv_alto=adv[0], adv_mejor_hops=adv[int(np.argmax(Ah))],
                          caza_no_alto=b != 0, caza_no_mejor=b != int(np.argmax(Ah)), caza_ninguno=(b != 0) and (b != int(np.argmax(Ah)))))
F = pd.DataFrame(filas)
print(f'\n--- (1) Lados de partido analizados: {len(F):,} ({F.match_id.nunique():,} partidos)')

def resumen(d):
    return {'reduccion_total_pct': 100 * (1 - d.tot_h.sum() / d.tot_alt.sum()), 'reduccion_peor_pct': 100 * (1 - d.peor_h.sum() / d.peor_alt.sum()),
            'tot_alt': d.tot_alt.mean(), 'tot_h': d.tot_h.mean(), 'peor_alt': d.peor_alt.mean(), 'peor_h': d.peor_h.mean(),
            'pct_cambia': 100 * d.cambia.mean(), 'pct_caza_no_alto': 100 * d.caza_no_alto.mean(), 'pct_caza_no_mejor': 100 * d.caza_no_mejor.mean(),
            'pct_caza_ninguno': 100 * d.caza_ninguno.mean(), 'mejor_adv': d.mejor_adv.mean(), 'adv_alto': d.adv_alto.mean(), 'adv_mejor_hops': d.adv_mejor_hops.mean()}
rng = np.random.default_rng(SEMILLA); idx = F.groupby('match_id').indices; mids = np.array(list(idx))
est = pd.Series(resumen(F)); bs = pd.DataFrame([resumen(F.iloc[np.concatenate([idx[m] for m in rng.choice(mids, len(mids))])]) for _ in range(500)])
RES = pd.DataFrame({'valor': est, 'ic_lo': bs.quantile(.025), 'ic_hi': bs.quantile(.975)})
print(RES.round(2).to_string())
print('Lectura: reduccion_total_pct = cuánto baja la desventaja aérea total al marcar por HOPS en vez de por estatura.'
      '\n         pct_caza_ninguno = partidos en que el mejor duelo a favor no es ni el más alto ni el de mejor HOPS.')

# ---------- (2) Ejemplos al azar (anónimos) y JSON ----------
ej = []
for mid in rng.choice(F.match_id.unique(), 3, replace=False):
    eq = [loc.at[mid, 'local'], loc.at[mid, 'visita']]
    D, A = G[(mid, eq[0])], G[(mid, eq[1])]
    ej.append({'def': [[int(round(a)), round(float(h), 1)] for a, h in D], 'ata': [[int(round(a)), round(float(h), 1)] for a, h in A]})
OUT = {'version': 'G-8 v1.0', 'n_lados': int(len(F)), 'n_partidos': int(F.match_id.nunique()), 'r_jugador': round(float(r_j), 2), 'r_equipo': round(float(r_ha), 2),
       'res': {k: [round(float(v), 2) for v in RES.loc[k]] for k in RES.index}, 'ejemplos': ej}
ruta = os.path.join(SAL5, 'reporte_hops.json')
with open(ruta, 'w', encoding='utf-8') as f: json.dump(OUT, f, ensure_ascii=False, separators=(',', ':'))
print('\nEjemplo 1 (defensores / atacantes, [estatura cm, HOPS]):', ej[0])
print(f'G-8 terminó en {time.time() - t0:.0f} s. Escribió {ruta} ({os.path.getsize(ruta)} bytes). Súbelo o pégalo en el chat.')

In [ ]:
# [G-4b v1.0] Los 11 en la cancha, versión corregida: más puestos para que el reporte junte carriles ANTES de elegir a los 10 de campo
# =====================================================================
# Sesión 15 · 2026-10-21 · Punto 4 de Arturo (León de Larcamón: el extremo derecho "aparece" con balón y desaparece sin balón).
# Causa: G-4 elegía los 10 puestos de campo con más acciones ANTES de juntar lateral con carrilero (y volante con extremo). Un técnico
#   que alterna línea de 4 y de 3/5 gasta 4 de esos 10 lugares en dos carriles, y se queda sin un jugador real. No era el portero.
# Arreglo: se exportan hasta 16 puestos de campo (con 30 acciones o más); el reporte junta carriles y luego toma los 10 con más acciones.
# Solo cambia la selección; posiciones, liga y percentiles quedan iguales (no se recalculan). Orden: celda 1 → G-4b. ~1-2 min.
# =====================================================================
import os, json, time
import numpy as np
import pandas as pd
RAIZ = '/content/drive/MyDrive/isac2026/'
assert os.path.ismount('/content/drive'), 'Drive no está montado: corre primero la celda 1'
DP = os.path.join(RAIZ, 'datos_parquet'); FW5 = os.path.join(DP, 'fw5'); SAL5 = os.path.join(RAIZ, 'salidas', 'fw5')
SELECTOR = ['Jardine', 'Larcam', 'Fernando Ortiz']; MIN_ACC, N_CAMPO = 30, 16
NOMBRE_PUESTO = {1: 'Portero', 2: 'Lateral derecho', 3: 'Central derecho', 4: 'Central', 5: 'Central izquierdo', 6: 'Lateral izquierdo',
                 7: 'Carrilero derecho', 8: 'Carrilero izquierdo', 9: 'Contención derecha', 10: 'Contención', 11: 'Contención izquierda',
                 12: 'Volante derecho', 13: 'Interior derecho', 14: 'Medio centro', 15: 'Interior izquierdo', 16: 'Volante izquierdo',
                 17: 'Extremo derecho', 18: 'Mediapunta derecho', 19: 'Mediapunta', 20: 'Mediapunta izquierdo', 21: 'Extremo izquierdo',
                 22: 'Delantero derecho', 23: 'Delantero centro', 24: 'Delantero izquierdo', 25: 'Segundo delantero'}
t0 = time.time()
with open(os.path.join(FW5, 'parametros_fw1.json')) as f: TEMPS = sorted(json.load(f)['temporadas']['entrenamiento'])
ptp = pd.read_parquet(os.path.join(DP, 'partido_tecnico_periodo.parquet'))
if 'jugado' in ptp: ptp = ptp[ptp['jugado'].astype(bool)]
ptp = ptp[ptp['temporada'].isin(TEMPS)].copy(); ptp['etapa_id'] = ptp['team'] + ' · ' + ptp['etapa'].astype(str)
ET = ptp.groupby('etapa_id').agg(tecnico=('tec_p', 'first')).reset_index()
SEL = [e for p in SELECTOR for e in ET[ET.tecnico.astype(str).str.contains(p, case=False)].etapa_id]
print(f'Etapas del selector: {len(SEL)} (deben ser 9)')
def once(ruta):
    A = pd.read_parquet(ruta, columns=['match_id', 'team', 'position_id', 'x', 'y']).merge(ptp[['match_id', 'team', 'etapa_id']], on=['match_id', 'team'])
    A['position_id'] = A.position_id.astype(int)
    liga = A.groupby('position_id').agg(x=('x', 'mean'), y=('y', 'mean'))
    out = {}
    for e in SEL:
        g = A[A.etapa_id == e].groupby('position_id').agg(x=('x', 'mean'), y=('y', 'mean'), n=('x', 'size'))
        g = g[g.n >= MIN_ACC].sort_values('n', ascending=False)
        sel = pd.concat([g[g.index == 1].head(1), g[g.index != 1].head(N_CAMPO)])
        out[e] = [{'puesto': int(p), 'nombre': NOMBRE_PUESTO.get(int(p), str(p)), 'x': round(float(r.x), 2), 'y': round(float(r.y), 2), 'n': int(r.n),
                   'lx': round(float(liga.loc[p, 'x']), 2), 'ly': round(float(liga.loc[p, 'y']), 2)} for p, r in sel.iterrows()]
    return out
OUT = {'version': 'G-4b v1.0', 'con_balon': once(os.path.join(FW5, 'r7c_acciones.parquet')), 'sin_balon': once(os.path.join(FW5, 'r7d_acciones.parquet'))}
for f in ['con_balon', 'sin_balon']:
    print(f'León · 4 · {f}:', [(p['nombre'], p['n']) for p in OUT[f].get('León · 4', [])])
ruta = os.path.join(SAL5, 'reporte_posiciones_v2.json')
with open(ruta, 'w', encoding='utf-8') as f: json.dump(OUT, f, ensure_ascii=False, separators=(',', ':'))
print(f'G-4b terminó en {time.time() - t0:.0f} s · {os.path.getsize(ruta) / 1e3:.0f} KB. Sube reporte_posiciones_v2.json al chat.')